In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 1 PLOS Computational Biology figure package.

This script generates the revised Step 1 figure package:

  Main Figure 1
    Source composition and descriptor heterogeneity of the ACP-oriented OncoPep corpus.

  Supplementary Figure S1
    Pairwise source-level descriptor shifts in the OncoPep corpus.

Primary outputs are saved to separated directories:

  <BASE_DIR>/plos_comp/step_01/main_figure/
  <BASE_DIR>/plos_comp/step_01/supplementary_figures/

For checklist convenience, selected final outputs are also copied to:

  <BASE_DIR>/plos_comp/step_01/

Designed for VS Code, Jupyter, and command-line execution.

Before running:
  1. Confirm BASE_DIR_DEFAULT.
  2. Confirm INPUT_CSV_DEFAULT or pass --input_csv.
  3. Confirm whether the source label should be DCP or DCTPep.
     This script standardizes DCTPep-like labels to DCP by default.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime
from itertools import combinations
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

try:
    from scipy.stats import ks_2samp
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# =============================================================================
# 1. User-editable paths and manuscript constants
# =============================================================================

BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
INPUT_CSV_DEFAULT = BASE_DIR_DEFAULT / "OncoPep_merged_clean_labeled4.csv"

SOURCE_ORDER_DEFAULT = ("dbAMPseq", "APD3", "CancerPPD2", "DCP")

# Manuscript counts used as validation guards. They do not replace computed values.
EXPECTED_SOURCE_COUNTS_DEFAULT = {
    "dbAMPseq": 43641,
    "APD3": 2646,
    "CancerPPD2": 276,
    "DCP": 4442,
}
EXPECTED_TOTAL_DEFAULT = 51005


# =============================================================================
# 2. Unified PLOS-style palette
# =============================================================================

PLOS_PALETTE = {
    "olive_green": "#9AAA63",
    "soft_coral": "#DD705D",
    "charcoal_brown": "#6A5E61",
    "pale_sand": "#F0CF95",
    "muted_mint": "#A8D3B2",
    "teal_green": "#56A98B",
    "blue_teal": "#1F95B8",
    "warm_brown": "#B67D4E",
    "axis_dark": "#333333",
    "light_gray": "#D9D9D9",
    "background": "#FFFFFF",
    "grid": "#EAEAEA",
    "grid_light": "#F3F3F3",
}

SOURCE_FACE = {
    "dbAMPseq": PLOS_PALETTE["blue_teal"],
    "APD3": PLOS_PALETTE["teal_green"],
    "CancerPPD2": PLOS_PALETTE["warm_brown"],
    "DCP": PLOS_PALETTE["soft_coral"],
}

SOURCE_EDGE = {
    "dbAMPseq": "#176F8A",
    "APD3": "#3E806A",
    "CancerPPD2": "#8F5F37",
    "DCP": "#A84F42",
}

S1_DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "plos_cliffs_delta",
    [PLOS_PALETTE["blue_teal"], PLOS_PALETTE["light_gray"], PLOS_PALETTE["soft_coral"]],
    N=256,
)


# =============================================================================
# 3. Configuration
# =============================================================================

@dataclass
class Step1Config:
    base_dir: Path = BASE_DIR_DEFAULT
    input_csv: Path = INPUT_CSV_DEFAULT
    output_root_name: str = "plos_comp/step_01"

    required_columns: Tuple[str, ...] = (
        "sequence",
        "source_db4",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
    )

    source_aliases: Tuple[str, ...] = (
        "source_db4",
        "source",
        "source_db",
        "dataset",
        "db",
        "dataset_source",
    )

    source_order: Tuple[str, ...] = SOURCE_ORDER_DEFAULT
    expected_source_counts: Mapping[str, int] = field(default_factory=lambda: EXPECTED_SOURCE_COUNTS_DEFAULT)
    expected_total_retained: int = EXPECTED_TOTAL_DEFAULT

    min_len: int = 5
    max_len: int = 60

    # Validation controls.
    fail_on_unknown_sources: bool = True
    validate_expected_counts: bool = True
    strict_expected_counts: bool = False

    # Export.
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True

    # Typography.
    font_size: float = 8.3
    title_size: float = 9.2
    label_size: float = 8.8
    tick_size: float = 7.8
    legend_size: float = 7.5
    panel_letter_size: float = 13.0
    annotation_size: float = 7.6
    count_label_size: float = 8.2

    # Figure sizes in inches.
    fig1_size: Tuple[float, float] = (11.4, 6.8)
    fig_s1_size: Tuple[float, float] = (10.9, 5.0)

    # Plot style.
    axis_linewidth: float = 0.8
    violin_alpha: float = 0.38
    violin_width: float = 0.58
    box_width: float = 0.16
    box_alpha: float = 0.98

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def main_dir(self) -> Path:
        return self.output_root / "main_figure"

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"


# =============================================================================
# 4. General utilities
# =============================================================================

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step1Config) -> None:
    for directory in [cfg.output_root, cfg.main_dir, cfg.supp_dir, cfg.reports_dir]:
        ensure_dir(directory)


def sha256_file(path: Path, chunk_size: int = 2**20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        f.write(text)


def save_figure_all_formats(fig: plt.Figure, basepath_no_ext: Path, cfg: Step1Config) -> Dict[str, str]:
    basepath_no_ext.parent.mkdir(parents=True, exist_ok=True)
    output_paths = {
        "pdf": basepath_no_ext.with_suffix(".pdf"),
        "png": basepath_no_ext.with_suffix(".png"),
        "tiff": basepath_no_ext.with_suffix(".tiff"),
    }
    fig.savefig(output_paths["pdf"], bbox_inches="tight", pad_inches=0.06, transparent=False)
    fig.savefig(output_paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06, transparent=False)
    fig.savefig(output_paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06, transparent=False)
    return {k: str(v) for k, v in output_paths.items()}


def find_input_csv(cfg: Step1Config) -> Path:
    candidates = [
        cfg.input_csv,
        cfg.base_dir / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir / "OncoPep_merged_clean_labeled.csv",
        cfg.base_dir / "OncoPep_merged_clean.csv",
        cfg.base_dir / "PepOnco_merged_clean_labeled4.csv",
        cfg.base_dir / "data" / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir / "tables" / "OncoPep_merged_clean_labeled4.csv",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    likely = sorted(cfg.base_dir.glob("**/*OncoPep*clean*.csv"))
    if likely:
        return likely[0]

    raise FileNotFoundError(
        "Could not find the cleaned OncoPep corpus CSV. "
        f"Default attempted path: {cfg.input_csv}. "
        "Edit INPUT_CSV_DEFAULT or pass --input_csv."
    )


def get_script_text() -> Tuple[str, Optional[Path], Optional[str]]:
    """
    Return script text for reproducibility snapshots.

    In normal .py execution, __file__ is available. In pasted Jupyter execution, __file__
    may not exist, so a best-effort source reconstruction is used.
    """
    file_name = globals().get("__file__", None)
    if file_name:
        script_path = Path(file_name).resolve()
        if script_path.exists():
            text = script_path.read_text(encoding="utf-8")
            return text, script_path, sha256_file(script_path)

    try:
        text = inspect.getsource(sys.modules[__name__])
        return text, None, sha256_text(text)
    except Exception:
        text = (
            "# OncoPep Step 1 plotting code snapshot could not be recovered automatically.\n"
            "# If this was run from a notebook cell, export the notebook or save the script manually.\n"
        )
        return text, None, sha256_text(text)


def set_plot_style(cfg: Step1Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": cfg.axis_linewidth,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS_PALETTE["background"],
            "axes.facecolor": PLOS_PALETTE["background"],
            "savefig.facecolor": PLOS_PALETTE["background"],
            "text.color": PLOS_PALETTE["axis_dark"],
            "axes.labelcolor": PLOS_PALETTE["axis_dark"],
            "axes.edgecolor": PLOS_PALETTE["axis_dark"],
            "xtick.color": PLOS_PALETTE["axis_dark"],
            "ytick.color": PLOS_PALETTE["axis_dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, linewidth=0.45, color=PLOS_PALETTE["grid"], zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PLOS_PALETTE["axis_dark"])
        ax.spines[side].set_linewidth(0.75)


def panel_letter(ax: plt.Axes, letter: str, cfg: Step1Config, dx: float = -0.12, dy: float = 1.08) -> None:
    ax.text(
        dx,
        dy,
        letter,
        transform=ax.transAxes,
        fontsize=cfg.panel_letter_size,
        fontweight="bold",
        ha="left",
        va="top",
        color=PLOS_PALETTE["axis_dark"],
    )


def finite_array(x: Sequence[float]) -> np.ndarray:
    arr = np.asarray(x, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_sequence_column(series: pd.Series) -> pd.Series:
    return series.where(series.notna(), "").astype(str).str.strip().str.upper()


def summary_stats(x: Sequence[float]) -> Dict[str, Any]:
    arr = finite_array(x)
    if len(arr) == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "min": np.nan,
            "max": np.nan,
        }
    return {
        "n": int(len(arr)),
        "mean": float(np.mean(arr)),
        "sd": float(np.std(arr, ddof=1)) if len(arr) > 1 else np.nan,
        "median": float(np.median(arr)),
        "q25": float(np.percentile(arr, 25)),
        "q75": float(np.percentile(arr, 75)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
    }


# =============================================================================
# 5. Data preparation and validation
# =============================================================================

AA20_SET = set("ACDEFGHIKLMNPQRSTVWY")


def canonical_only(seq: str) -> bool:
    return isinstance(seq, str) and len(seq) > 0 and set(seq).issubset(AA20_SET)


def max_run_length(seq: str) -> int:
    if not isinstance(seq, str) or len(seq) == 0:
        return 0
    best = cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return int(best)


def shannon_entropy(seq: str) -> float:
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    counts = pd.Series(list(seq)).value_counts().to_numpy(dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def dominant_residue_fraction(seq: str) -> float:
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    return float(pd.Series(list(seq)).value_counts().max() / len(seq))


def source_key(label: Any) -> str:
    """
    Robust key for source-label normalization.
    Removes punctuation/spacing and converts to lowercase.
    """
    raw = "" if pd.isna(label) else str(label)
    cleaned = raw.strip().lower()
    for token in [" ", "_", "-", "/", "\\", ".", "(", ")", "[", "]", "{", "}"]:
        cleaned = cleaned.replace(token, "")
    return cleaned


SOURCE_KEY_TO_CANONICAL = {
    "dbampseq": "dbAMPseq",
    "dbamp": "dbAMPseq",
    "apd3": "APD3",
    "apd": "APD3",
    "cancerppd2": "CancerPPD2",
    "cancerppd": "CancerPPD2",
    "dcp": "DCP",
    "dctpep": "DCP",
    "dctpeptide": "DCP",
    "dcpdctpep": "DCP",
    "dctpepdcp": "DCP",
}


def normalize_source_label(label: Any) -> str:
    key = source_key(label)
    return SOURCE_KEY_TO_CANONICAL.get(key, f"UNKNOWN::{str(label).strip()}")


def harmonize_column_names(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    out = df.copy()
    if "source_db4" not in out.columns:
        for candidate in cfg.source_aliases:
            if candidate in out.columns:
                out = out.rename(columns={candidate: "source_db4"})
                break
    return out


def validate_required_columns(df: pd.DataFrame, required: Iterable[str], context: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{context}: missing required columns: {missing}")


def build_source_label_audit(df: pd.DataFrame, source_col: str = "source_db4") -> pd.DataFrame:
    audit = (
        df[source_col]
        .where(df[source_col].notna(), "")
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
        .reset_index()
    )
    audit.columns = ["original_source_label", "n_rows"]
    audit["normalized_source"] = audit["original_source_label"].apply(normalize_source_label)
    audit["status"] = np.where(audit["normalized_source"].str.startswith("UNKNOWN::"), "unknown", "recognized")
    return audit[["original_source_label", "normalized_source", "status", "n_rows"]]


def prepare_dataframe(raw_df: pd.DataFrame, cfg: Step1Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = harmonize_column_names(df, cfg)
    validate_required_columns(df, cfg.required_columns, "Input corpus")

    source_audit = build_source_label_audit(df, "source_db4")
    unknown_audit = source_audit[source_audit["status"] == "unknown"].copy()
    if not unknown_audit.empty and cfg.fail_on_unknown_sources:
        raise ValueError(
            "Unknown source labels detected. Fix source mapping before plotting:\n"
            + unknown_audit.to_string(index=False)
        )

    df["sequence_raw"] = df["sequence"]
    df["sequence"] = normalize_sequence_column(df["sequence"])
    df["source_original"] = df["source_db4"].where(df["source_db4"].notna(), "").astype(str).str.strip()
    df["source"] = df["source_original"].apply(normalize_source_label)
    df.loc[df["source"].str.startswith("UNKNOWN::"), "source"] = "Other"

    for col in ["length", "net_charge_pH7", "hydrophobicity_KD"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["sequence_id"] = np.arange(1, len(df) + 1)
    df["len_from_seq"] = df["sequence"].str.len()
    df["is_canonical"] = df["sequence"].apply(canonical_only)

    # Computed descriptors.
    df["shannon_entropy"] = df["sequence"].apply(shannon_entropy)
    df["max_homopolymer_run"] = df["sequence"].apply(max_run_length)
    df["dominant_residue_fraction"] = df["sequence"].apply(dominant_residue_fraction)

    # Basic final-retained flag for this Step 1 audit.
    df["length_missing"] = df["length"].isna()
    df["charge_missing"] = df["net_charge_pH7"].isna()
    df["hydrophobicity_missing"] = df["hydrophobicity_KD"].isna()
    df["sequence_missing"] = df["sequence"].eq("")

    length_rounded = pd.Series(pd.NA, index=df.index, dtype="Int64")
    length_present = df["length"].notna()
    length_rounded.loc[length_present] = df.loc[length_present, "length"].round().astype("Int64")
    df["length_rounded_int"] = length_rounded

    df["length_mismatch"] = False
    df.loc[length_present, "length_mismatch"] = (
        df.loc[length_present, "length_rounded_int"].astype("Int64")
        != df.loc[length_present, "len_from_seq"].astype("Int64")
    ).fillna(False)

    df["length_out_of_range"] = np.where(
        df["sequence_missing"],
        False,
        (df["len_from_seq"] < cfg.min_len) | (df["len_from_seq"] > cfg.max_len),
    )

    df["exclude_step1"] = (
        df["sequence_missing"]
        | (~df["is_canonical"])
        | df["length_missing"]
        | df["charge_missing"]
        | df["hydrophobicity_missing"]
        | df["length_mismatch"]
        | df["length_out_of_range"]
    )
    df["retained_step1"] = ~df["exclude_step1"]

    df["source_for_plot"] = df["source"]
    if cfg.fail_on_unknown_sources:
        unexpected_after_norm = sorted(set(df["source_for_plot"]) - set(cfg.source_order))
        unexpected_after_norm = [x for x in unexpected_after_norm if x != "Other"]
        if unexpected_after_norm:
            raise ValueError(f"Unexpected normalized sources after mapping: {unexpected_after_norm}")

    return df, source_audit


# =============================================================================
# 6. Step 1 analysis tables
# =============================================================================

def build_source_counts(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    retained = df[df["retained_step1"]].copy()
    counts = (
        retained["source_for_plot"]
        .value_counts()
        .reindex(list(cfg.source_order))
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    counts.columns = ["source", "n_sequences"]
    total = int(counts["n_sequences"].sum())
    counts["percent_total"] = np.where(total > 0, 100 * counts["n_sequences"] / total, np.nan)
    counts["total_retained"] = total
    return counts


def validate_expected_counts(counts_df: pd.DataFrame, cfg: Step1Config) -> Tuple[bool, pd.DataFrame, list[str]]:
    rows = []
    warnings_list: list[str] = []

    observed = counts_df.set_index("source")["n_sequences"].to_dict()
    observed_total = int(counts_df["n_sequences"].sum())
    expected_total = int(cfg.expected_total_retained)

    rows.append(
        {
            "item": "total_retained",
            "observed": observed_total,
            "expected": expected_total,
            "difference": observed_total - expected_total,
            "matches": observed_total == expected_total,
        }
    )

    for source, expected in cfg.expected_source_counts.items():
        obs = int(observed.get(source, 0))
        rows.append(
            {
                "item": f"source::{source}",
                "observed": obs,
                "expected": int(expected),
                "difference": obs - int(expected),
                "matches": obs == int(expected),
            }
        )

    audit = pd.DataFrame(rows)
    ok = bool(audit["matches"].all())

    if cfg.validate_expected_counts and not ok:
        message = (
            "Computed Step 1 counts do not match the manuscript validation counts. "
            "Inspect count_validation_audit.csv before using the figure."
        )
        warnings_list.append(message)
        if cfg.strict_expected_counts:
            raise ValueError(message + "\n" + audit.to_string(index=False))
        warnings.warn(message, RuntimeWarning)

    return ok, audit, warnings_list


def build_descriptor_distribution_table(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    retained = df[df["retained_step1"] & df["source_for_plot"].isin(cfg.source_order)].copy()
    cols = [
        "sequence_id",
        "source_for_plot",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "shannon_entropy",
        "dominant_residue_fraction",
        "max_homopolymer_run",
    ]
    out = retained[cols].copy()
    out = out.rename(columns={"source_for_plot": "source"})

    if out.empty:
        raise ValueError("No retained rows available for descriptor plotting after source/filter validation.")

    for source in cfg.source_order:
        if (out["source"] == source).sum() == 0:
            warnings.warn(f"Source {source!r} has zero retained rows in descriptor table.", RuntimeWarning)

    return out


def build_source_summary_table(descriptor_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    descriptors = [
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "shannon_entropy",
        "dominant_residue_fraction",
        "max_homopolymer_run",
    ]
    rows = []
    for source in cfg.source_order:
        sub = descriptor_df[descriptor_df["source"] == source]
        row = {"source": source, "n_sequences": int(len(sub))}
        for col in descriptors:
            s = summary_stats(sub[col].to_numpy())
            for key, value in s.items():
                row[f"{col}_{key}"] = value
        rows.append(row)
    return pd.DataFrame(rows)


def cliffs_delta_exact(x: Sequence[float], y: Sequence[float]) -> float:
    """
    Exact Cliff's delta using sorted search.

    δ = [#(x > y) - #(x < y)] / (n_x * n_y)

    Ties contribute 0. Complexity is O((n_x + n_y) log n_y), avoiding
    the memory cost of all pairwise comparisons.
    """
    x_arr = finite_array(x)
    y_arr = finite_array(y)
    if len(x_arr) == 0 or len(y_arr) == 0:
        return np.nan

    y_sorted = np.sort(y_arr)
    less_than_x = np.searchsorted(y_sorted, x_arr, side="left")
    less_or_equal_x = np.searchsorted(y_sorted, x_arr, side="right")
    greater_than_x = len(y_sorted) - less_or_equal_x

    gt = np.sum(less_than_x)
    lt = np.sum(greater_than_x)
    return float((gt - lt) / (len(x_arr) * len(y_sorted)))


def ks_test(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float]:
    if not HAS_SCIPY:
        return np.nan, np.nan
    x_arr = finite_array(x)
    y_arr = finite_array(y)
    if len(x_arr) < 10 or len(y_arr) < 10:
        return np.nan, np.nan
    stat, p_value = ks_2samp(x_arr, y_arr)
    return float(stat), float(p_value)


def build_pairwise_shift_table(descriptor_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    descriptors = [
        ("length", "Length"),
        ("net_charge_pH7", "Net charge"),
        ("hydrophobicity_KD", "Hydrophobicity"),
        ("shannon_entropy", "Shannon entropy"),
        ("max_homopolymer_run", "Max homopolymer run"),
        ("dominant_residue_fraction", "Dominant residue fraction"),
    ]

    rows = []
    for source_a, source_b in combinations(cfg.source_order, 2):
        sub_a = descriptor_df[descriptor_df["source"] == source_a]
        sub_b = descriptor_df[descriptor_df["source"] == source_b]

        for descriptor_col, descriptor_label in descriptors:
            a_values = sub_a[descriptor_col].to_numpy()
            b_values = sub_b[descriptor_col].to_numpy()
            delta = cliffs_delta_exact(a_values, b_values)
            ks_stat, ks_pvalue = ks_test(a_values, b_values)

            rows.append(
                {
                    "panel": "A",
                    "data_type": "pairwise_cliffs_delta",
                    "source_a": source_a,
                    "source_b": source_b,
                    "source_pair": f"{source_a} vs {source_b}",
                    "descriptor": descriptor_label,
                    "descriptor_column": descriptor_col,
                    "cliffs_delta": delta,
                    "abs_cliffs_delta": abs(delta) if np.isfinite(delta) else np.nan,
                    "ks_stat": ks_stat,
                    "ks_pvalue": ks_pvalue,
                    "x_n_input": int(finite_array(a_values).size),
                    "y_n_input": int(finite_array(b_values).size),
                    "x_n_used": int(finite_array(a_values).size),
                    "y_n_used": int(finite_array(b_values).size),
                    "subsampled": False,
                    "cliffs_delta_method": "exact_sorted_search",
                }
            )

    return pd.DataFrame(rows)


def build_descriptor_shift_summary(pairwise_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        pairwise_df.groupby("descriptor", as_index=False)
        .agg(
            mean_abs_cliffs_delta=("abs_cliffs_delta", "mean"),
            max_abs_cliffs_delta=("abs_cliffs_delta", "max"),
            n_pairwise_comparisons=("source_pair", "nunique"),
        )
        .sort_values("mean_abs_cliffs_delta", ascending=False)
    )
    summary.insert(0, "panel", "B")
    summary["data_type"] = "descriptor_shift_summary"
    return summary


def build_figure1_source_data_all(counts_df: pd.DataFrame, descriptor_df: pd.DataFrame) -> pd.DataFrame:
    panel_a = counts_df.copy()
    panel_a.insert(0, "panel", "A")
    panel_a["data_type"] = "source_counts"

    descriptor_long = descriptor_df.melt(
        id_vars=["sequence_id", "source"],
        value_vars=[
            "length",
            "net_charge_pH7",
            "hydrophobicity_KD",
            "shannon_entropy",
            "dominant_residue_fraction",
        ],
        var_name="descriptor",
        value_name="value",
    )
    descriptor_long.insert(
        0,
        "panel",
        descriptor_long["descriptor"].map(
            {
                "length": "B",
                "net_charge_pH7": "C",
                "hydrophobicity_KD": "D",
                "shannon_entropy": "E",
                "dominant_residue_fraction": "F",
            }
        ),
    )
    descriptor_long["data_type"] = "descriptor_distribution"

    return pd.concat([panel_a, descriptor_long], ignore_index=True, sort=False)


def build_s1_source_data_all(pairwise_df: pd.DataFrame, shift_summary_df: pd.DataFrame) -> pd.DataFrame:
    panel_a = pairwise_df.copy()
    panel_b = shift_summary_df.copy()
    return pd.concat([panel_a, panel_b], ignore_index=True, sort=False)


# =============================================================================
# 7. Plotting helpers
# =============================================================================

def plot_violin_box(
    ax: plt.Axes,
    descriptor_df: pd.DataFrame,
    column: str,
    ylabel: str,
    cfg: Step1Config,
    y_limits: Optional[Tuple[float, float]] = None,
    zero_line: bool = False,
) -> None:
    positions = np.arange(1, len(cfg.source_order) + 1)
    data = [
        finite_array(descriptor_df.loc[descriptor_df["source"] == source, column].to_numpy())
        for source in cfg.source_order
    ]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=cfg.violin_width,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, source in zip(vp["bodies"], cfg.source_order):
        body.set_facecolor(SOURCE_FACE[source])
        body.set_edgecolor(SOURCE_EDGE[source])
        body.set_alpha(cfg.violin_alpha)
        body.set_linewidth(0.9)
        body.set_zorder(3)

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=cfg.box_width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color=PLOS_PALETTE["axis_dark"], linewidth=1.15),
        whiskerprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        capprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        boxprops=dict(linewidth=0.75),
    )

    for patch, source in zip(bp["boxes"], cfg.source_order):
        patch.set_facecolor("#FFFFFF")
        patch.set_alpha(cfg.box_alpha)
        patch.set_edgecolor(SOURCE_EDGE[source])
        patch.set_zorder(4)

    if zero_line:
        ax.axhline(0, color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75, linestyle="--", zorder=2)

    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(cfg.source_order, rotation=12, ha="right")

    if y_limits is not None:
        ax.set_ylim(*y_limits)

    beautify_axis(ax, "y")


def source_handles(cfg: Step1Config) -> list[Patch]:
    return [
        Patch(
            facecolor=SOURCE_FACE[source],
            edgecolor=SOURCE_EDGE[source],
            label=source,
            linewidth=0.9,
        )
        for source in cfg.source_order
    ]


def heatmap_annotation_color(value: float) -> str:
    return "#FFFFFF" if np.isfinite(value) and abs(value) >= 0.55 else PLOS_PALETTE["axis_dark"]


# =============================================================================
# 8. Main Figure 1
# =============================================================================

def generate_main_figure1(
    counts_df: pd.DataFrame,
    descriptor_df: pd.DataFrame,
    cfg: Step1Config,
) -> Dict[str, str]:
    total_n = int(counts_df["n_sequences"].sum())

    fig, axes = plt.subplots(2, 3, figsize=cfg.fig1_size)
    fig.subplots_adjust(
        left=0.075,
        right=0.985,
        top=0.925,
        bottom=0.145,
        wspace=0.34,
        hspace=0.40,
    )

    # Panel A: source composition.
    ax = axes[0, 0]
    x = np.arange(len(cfg.source_order))
    counts = counts_df.set_index("source").reindex(cfg.source_order)["n_sequences"].fillna(0).astype(int)
    percentages = counts_df.set_index("source").reindex(cfg.source_order)["percent_total"].fillna(0)

    bars = ax.bar(
        x,
        counts.values,
        width=0.55,
        color=[SOURCE_FACE[source] for source in cfg.source_order],
        edgecolor=[SOURCE_EDGE[source] for source in cfg.source_order],
        linewidth=0.95,
        zorder=3,
    )
    ax.set_yscale("log")
    ax.set_ylabel("Retained sequences (log scale)")
    ax.set_xticks(x)
    ax.set_xticklabels(cfg.source_order, rotation=12, ha="right")
    ax.set_title("Source composition", pad=8)
    beautify_axis(ax, "y")

    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 2.2)

    for bar, source in zip(bars, cfg.source_order):
        count = int(counts.loc[source])
        pct = float(percentages.loc[source])
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            count * 1.10 if count > 0 else 1,
            f"{count:,}\n({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=cfg.count_label_size,
            color=PLOS_PALETTE["axis_dark"],
            linespacing=1.1,
        )

    ax.text(
        0.03,
        0.94,
        f"Retained corpus\nN = {total_n:,} peptides",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor=PLOS_PALETTE["pale_sand"],
            edgecolor=PLOS_PALETTE["charcoal_brown"],
            linewidth=0.6,
        ),
    )
    panel_letter(ax, "a", cfg)

    specs = [
        (axes[0, 1], "length", "Length (aa)", "Length", "b", None, False),
        (axes[0, 2], "net_charge_pH7", "Net charge at pH 7", "Net charge", "c", None, True),
        (axes[1, 0], "hydrophobicity_KD", "Mean hydrophobicity\n(Kyte–Doolittle)", "Hydrophobicity", "d", None, True),
        (axes[1, 1], "shannon_entropy", "Shannon entropy (bits)", "Shannon entropy", "e", None, False),
        (axes[1, 2], "dominant_residue_fraction", "Dominant residue fraction", "Dominant residue fraction", "f", (0.05, 1.0), False),
    ]

    for ax_i, col, ylabel, title, label, ylim, zero_line in specs:
        plot_violin_box(ax_i, descriptor_df, col, ylabel, cfg, y_limits=ylim, zero_line=zero_line)
        ax_i.set_title(title, pad=8)
        panel_letter(ax_i, label, cfg)

    handles = source_handles(cfg)
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=4,
        frameon=True,
        bbox_to_anchor=(0.5, 0.045),
        fontsize=cfg.legend_size,
        edgecolor=PLOS_PALETTE["light_gray"],
        facecolor=PLOS_PALETTE["background"],
        handlelength=1.5,
        handletextpad=0.5,
        columnspacing=1.25,
        borderpad=0.35,
    )

    base = cfg.main_dir / "Figure_1_redesigned"
    paths = save_figure_all_formats(fig, base, cfg)
    plt.close(fig)
    return paths


# =============================================================================
# 9. Supplementary Figure S1
# =============================================================================

def generate_supplementary_figure_s1(
    pairwise_df: pd.DataFrame,
    shift_summary_df: pd.DataFrame,
    cfg: Step1Config,
) -> Dict[str, str]:
    descriptors = [
        "Length",
        "Net charge",
        "Hydrophobicity",
        "Shannon entropy",
        "Max homopolymer run",
        "Dominant residue fraction",
    ]
    source_pairs = [f"{a} vs {b}" for a, b in combinations(cfg.source_order, 2)]

    heatmap = np.full((len(source_pairs), len(descriptors)), np.nan)
    for i, pair in enumerate(source_pairs):
        for j, desc in enumerate(descriptors):
            values = pairwise_df.loc[
                (pairwise_df["source_pair"] == pair) & (pairwise_df["descriptor"] == desc),
                "cliffs_delta",
            ].to_numpy()
            if len(values):
                heatmap[i, j] = values[0]

    fig = plt.figure(figsize=cfg.fig_s1_size)
    gs = fig.add_gridspec(1, 2, width_ratios=[2.25, 1.05], wspace=0.42)
    ax_hm = fig.add_subplot(gs[0, 0])
    ax_bar = fig.add_subplot(gs[0, 1])
    fig.subplots_adjust(left=0.12, right=0.965, top=0.90, bottom=0.25)

    # Panel A: Cliff's delta heatmap.
    im = ax_hm.imshow(
        heatmap,
        aspect="auto",
        cmap=S1_DIVERGING_CMAP,
        vmin=-1.0,
        vmax=1.0,
        interpolation="nearest",
    )
    ax_hm.set_xticks(np.arange(len(descriptors)))
    ax_hm.set_xticklabels(descriptors, rotation=28, ha="right")
    ax_hm.set_yticks(np.arange(len(source_pairs)))
    ax_hm.set_yticklabels(source_pairs)
    ax_hm.tick_params(length=0)
    ax_hm.set_title("Pairwise source shifts", pad=8)

    for i in range(heatmap.shape[0]):
        for j in range(heatmap.shape[1]):
            value = heatmap[i, j]
            if np.isfinite(value):
                ax_hm.text(
                    j,
                    i,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=6.7,
                    color=heatmap_annotation_color(value),
                )

    for spine in ax_hm.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax_hm, fraction=0.04, pad=0.03)
    cbar.set_label("Cliff’s δ", fontsize=cfg.legend_size)
    cbar.outline.set_linewidth(0.6)
    cbar.outline.set_edgecolor(PLOS_PALETTE["light_gray"])

    ax_hm.text(
        0.0,
        -0.25,
        "Positive δ: descriptor is higher in the first source of the pair; negative δ: lower.",
        transform=ax_hm.transAxes,
        ha="left",
        va="top",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
    )
    panel_letter(ax_hm, "a", cfg, dx=-0.13, dy=1.08)

    # Panel B: ranked mean absolute Cliff's delta.
    summary = shift_summary_df.sort_values("mean_abs_cliffs_delta", ascending=True)
    y = np.arange(len(summary))
    max_x = max(float(summary["mean_abs_cliffs_delta"].max()), 0.05)

    ax_bar.barh(
        y,
        summary["mean_abs_cliffs_delta"],
        color=PLOS_PALETTE["charcoal_brown"],
        edgecolor=PLOS_PALETTE["axis_dark"],
        linewidth=0.55,
        zorder=3,
    )
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels(summary["descriptor"])
    ax_bar.set_xlabel("Mean |Cliff’s δ|")
    ax_bar.set_title("Descriptor-level shift", pad=8)
    ax_bar.set_xlim(0, max_x * 1.18)
    beautify_axis(ax_bar, "x")

    for yi, value in zip(y, summary["mean_abs_cliffs_delta"]):
        ax_bar.text(
            value + max_x * 0.025,
            yi,
            f"{value:.2f}",
            ha="left",
            va="center",
            fontsize=cfg.annotation_size,
            color=PLOS_PALETTE["axis_dark"],
        )

    panel_letter(ax_bar, "b", cfg, dx=-0.30, dy=1.08)

    base = cfg.supp_dir / "Supplementary_Figure_S1_redesigned"
    paths = save_figure_all_formats(fig, base, cfg)
    plt.close(fig)
    return paths


# =============================================================================
# 10. Output helpers
# =============================================================================

def save_panel_specific_figure1_data(descriptor_df: pd.DataFrame, cfg: Step1Config) -> None:
    mapping = {
        "B": ("length", "Figure_1_panel_b_source_data.csv"),
        "C": ("net_charge_pH7", "Figure_1_panel_c_source_data.csv"),
        "D": ("hydrophobicity_KD", "Figure_1_panel_d_source_data.csv"),
        "E": ("shannon_entropy", "Figure_1_panel_e_source_data.csv"),
        "F": ("dominant_residue_fraction", "Figure_1_panel_f_source_data.csv"),
    }
    for panel, (col, filename) in mapping.items():
        panel_df = descriptor_df[["sequence_id", "source", col]].copy()
        panel_df.insert(0, "panel", panel)
        panel_df = panel_df.rename(columns={col: "value"})
        panel_df["descriptor"] = col
        save_csv(panel_df, cfg.main_dir / filename)


def save_root_level_copies(cfg: Step1Config) -> list[str]:
    """
    Preserve separated directories while also creating root-level convenience copies
    matching common journal/checklist paths.
    """
    if not cfg.create_root_level_copies:
        return []

    copy_pairs = [
        (cfg.main_dir / "Figure_1_redesigned.png", cfg.output_root / "Figure_1_redesigned.png"),
        (cfg.main_dir / "Figure_1_redesigned.pdf", cfg.output_root / "Figure_1_redesigned.pdf"),
        (cfg.main_dir / "Figure_1_redesigned.tiff", cfg.output_root / "Figure_1_redesigned.tiff"),
        (cfg.main_dir / "Figure_1_source_data_all_panels.csv", cfg.output_root / "Figure_1_source_data_all_panels.csv"),
        (cfg.main_dir / "Figure_1_redesign_code.py", cfg.output_root / "Figure_1_redesign_code.py"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.png", cfg.output_root / "Supplementary_Figure_S1_redesigned.png"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.pdf", cfg.output_root / "Supplementary_Figure_S1_redesigned.pdf"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.tiff", cfg.output_root / "Supplementary_Figure_S1_redesigned.tiff"),
        (cfg.supp_dir / "Supplementary_Figure_S1_source_data_all_panels.csv", cfg.output_root / "Supplementary_Figure_S1_source_data_all_panels.csv"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesign_code.py", cfg.output_root / "Supplementary_Figure_S1_redesign_code.py"),
    ]

    copied = []
    for src, dst in copy_pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def save_code_snapshots(cfg: Step1Config) -> Tuple[str, Optional[str], Optional[str]]:
    script_text, script_path, script_hash = get_script_text()
    save_text(script_text, cfg.main_dir / "Figure_1_redesign_code.py")
    save_text(script_text, cfg.supp_dir / "Supplementary_Figure_S1_redesign_code.py")
    save_text(script_text, cfg.output_root / "Step_01_full_redesign_code.py")
    return script_text, str(script_path) if script_path else None, script_hash


def write_readme(cfg: Step1Config, warnings_list: Sequence[str]) -> Path:
    text = f"""# OncoPep Step 1 PLOS figure package

Generated: {datetime.now().isoformat()}

Primary outputs:

- main_figure/Figure_1_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S1_redesigned.png/.pdf/.tiff

Source-data files:

- main_figure/Figure_1_source_data_all_panels.csv
- supplementary_figures/Supplementary_Figure_S1_source_data_all_panels.csv
- panel-specific source-data CSVs in each figure directory

Notes:

- Figure 1 uses source composition and descriptor distributions.
- Supplementary Figure S1 uses exact Cliff's delta computed by sorted-search counting.
- KS statistics are saved in source data but not plotted.
- Positive Cliff's delta means the descriptor is higher in the first source of the pair.

Warnings:

{chr(10).join('- ' + w for w in warnings_list) if warnings_list else '- None'}
"""
    path = cfg.output_root / "README_step_01_outputs.txt"
    save_text(text, path)
    return path


# =============================================================================
# 11. Main run function
# =============================================================================

def run_step1(cfg: Step1Config) -> None:
    ensure_output_tree(cfg)
    set_plot_style(cfg)

    warnings_list: list[str] = []

    input_csv = find_input_csv(cfg)
    raw_df = pd.read_csv(input_csv, low_memory=False)
    df, source_audit_df = prepare_dataframe(raw_df, cfg)

    save_csv(source_audit_df, cfg.reports_dir / "source_label_audit.csv")
    save_csv(source_audit_df, cfg.output_root / "source_label_audit.csv")

    counts_df = build_source_counts(df, cfg)
    counts_ok, count_audit_df, count_warnings = validate_expected_counts(counts_df, cfg)
    warnings_list.extend(count_warnings)

    descriptor_df = build_descriptor_distribution_table(df, cfg)
    source_summary_df = build_source_summary_table(descriptor_df, cfg)
    pairwise_df = build_pairwise_shift_table(descriptor_df, cfg)
    shift_summary_df = build_descriptor_shift_summary(pairwise_df)
    figure1_all = build_figure1_source_data_all(counts_df, descriptor_df)
    s1_all = build_s1_source_data_all(pairwise_df, shift_summary_df)

    # Save source-data before plotting.
    save_csv(counts_df, cfg.main_dir / "Figure_1_panel_a_source_data.csv")
    save_csv(counts_df, cfg.main_dir / "Figure_1_panel_a_source_counts.csv")
    save_csv(descriptor_df, cfg.main_dir / "Figure_1_panel_bf_descriptor_distributions.csv")
    save_panel_specific_figure1_data(descriptor_df, cfg)
    save_csv(source_summary_df, cfg.main_dir / "Figure_1_descriptor_summary_by_source.csv")
    save_csv(figure1_all, cfg.main_dir / "Figure_1_source_data_all_panels.csv")

    save_csv(pairwise_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_a_source_data.csv")
    save_csv(pairwise_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_a_effect_sizes.csv")
    save_csv(shift_summary_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_b_source_data.csv")
    save_csv(shift_summary_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_b_descriptor_shift_summary.csv")
    save_csv(s1_all, cfg.supp_dir / "Supplementary_Figure_S1_source_data_all_panels.csv")

    save_csv(count_audit_df, cfg.reports_dir / "count_validation_audit.csv")
    save_csv(count_audit_df, cfg.output_root / "count_validation_audit.csv")

    # Generate figures.
    fig1_paths = generate_main_figure1(counts_df, descriptor_df, cfg)
    s1_paths = generate_supplementary_figure_s1(pairwise_df, shift_summary_df, cfg)

    # Save code snapshots after figure generation.
    _, script_path, script_hash = save_code_snapshots(cfg)

    readme_path = write_readme(cfg, warnings_list)
    root_copies = save_root_level_copies(cfg)

    manifest = {
        "step": "step_01",
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python_version": sys.version,
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": mpl.__version__,
            "scipy_available": HAS_SCIPY,
        },
        "input_csv": str(input_csv),
        "input_sha256": sha256_file(input_csv),
        "script_path": script_path,
        "script_sha256": script_hash,
        "base_dir": str(cfg.base_dir),
        "output_root": str(cfg.output_root),
        "main_dir": str(cfg.main_dir),
        "supp_dir": str(cfg.supp_dir),
        "source_order": list(cfg.source_order),
        "total_retained_sequences": int(counts_df["n_sequences"].sum()),
        "source_counts": counts_df.to_dict(orient="records"),
        "expected_count_validation": {
            "enabled": cfg.validate_expected_counts,
            "strict": cfg.strict_expected_counts,
            "passed": counts_ok,
            "audit_file": str(cfg.output_root / "count_validation_audit.csv"),
        },
        "source_label_audit_file": str(cfg.output_root / "source_label_audit.csv"),
        "cliffs_delta": {
            "method": "exact_sorted_search",
            "subsampled": False,
        },
        "outputs": {
            "main_figure": fig1_paths,
            "supplementary_figure_s1": s1_paths,
            "root_level_copies": root_copies,
            "readme": str(readme_path),
        },
        "warnings": warnings_list,
        "config": asdict(cfg),
    }
    save_json(manifest, cfg.output_root / "step_01_manifest.json")
    save_json(manifest, cfg.reports_dir / "step_01_manifest.json")

    print("\n✅ Step 1 PLOS figure package generated successfully.")
    print(f"Input CSV: {input_csv}")
    print(f"Main figure directory: {cfg.main_dir}")
    print(f"Supplementary figure directory: {cfg.supp_dir}")
    print(f"Root output directory: {cfg.output_root}")

    print("\nSaved main figure files:")
    for path in fig1_paths.values():
        print(f"  - {path}")

    print("\nSaved Supplementary Figure S1 files:")
    for path in s1_paths.values():
        print(f"  - {path}")

    print("\nKey reports:")
    print(f"  - {cfg.output_root / 'step_01_manifest.json'}")
    print(f"  - {cfg.output_root / 'source_label_audit.csv'}")
    print(f"  - {cfg.output_root / 'count_validation_audit.csv'}")
    print(f"  - {readme_path}")

    if warnings_list:
        print("\n⚠️ Warnings:")
        for warning in warnings_list:
            print(f"  - {warning}")


# =============================================================================
# 12. Notebook and CLI entry points
# =============================================================================

def main_notebook(
    base_dir: str | Path = BASE_DIR_DEFAULT,
    input_csv: str | Path = INPUT_CSV_DEFAULT,
    min_len: int = 5,
    max_len: int = 60,
    strict_expected_counts: bool = False,
    fail_on_unknown_sources: bool = True,
) -> None:
    cfg = Step1Config(
        base_dir=Path(base_dir),
        input_csv=Path(input_csv),
        min_len=min_len,
        max_len=max_len,
        strict_expected_counts=strict_expected_counts,
        fail_on_unknown_sources=fail_on_unknown_sources,
    )
    run_step1(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    """
    Parse command-line arguments safely.

    Jupyter/IPython kernels inject extra arguments such as:
        --f=/run/user/.../kernel-xxxx.json

    Using parse_known_args prevents those kernel-specific arguments from
    crashing the script when the code is pasted into a notebook cell.
    """
    parser = argparse.ArgumentParser(
        description="Generate OncoPep Step 1 PLOS Computational Biology figures."
    )
    parser.add_argument(
        "--base_dir",
        type=str,
        default=str(BASE_DIR_DEFAULT),
        help="Base project directory containing the PLOS folder.",
    )
    parser.add_argument(
        "--input_csv",
        type=str,
        default=str(INPUT_CSV_DEFAULT),
        help="Path to the cleaned OncoPep corpus CSV.",
    )
    parser.add_argument("--min_len", type=int, default=5, help="Minimum retained peptide length.")
    parser.add_argument("--max_len", type=int, default=60, help="Maximum retained peptide length.")
    parser.add_argument(
        "--strict_counts",
        action="store_true",
        help="Raise an error if computed counts differ from manuscript validation counts.",
    )
    parser.add_argument(
        "--allow_unknown_sources",
        action="store_true",
        help="Do not fail on unknown source labels. Unknown labels are excluded from the four-source plots.",
    )
    parser.add_argument(
        "--no_root_copies",
        action="store_true",
        help="Do not create root-level convenience copies of final outputs.",
    )

    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"ℹ️ Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    cfg = Step1Config(
        base_dir=Path(args.base_dir),
        input_csv=Path(args.input_csv),
        min_len=args.min_len,
        max_len=args.max_len,
        strict_expected_counts=args.strict_counts,
        fail_on_unknown_sources=not args.allow_unknown_sources,
        create_root_level_copies=not args.no_root_copies,
    )
    run_step1(cfg)

In [ ]:
main_notebook(
    base_dir="/home/data3/Moe/nature_computational_peponco/PLOS",
    input_csv="/home/data3/Moe/nature_computational_peponco/OncoPep_merged_clean_labeled4.csv",
    strict_expected_counts=False,
    fail_on_unknown_sources=True,
)

OncoPep Step 1 PLOS Computational Biology figure package.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 1 PLOS Computational Biology figure package.

This script generates the revised Step 1 figure package:

  Main Figure 1
    Source composition and descriptor heterogeneity of the ACP-oriented OncoPep corpus.

  Supplementary Figure S1
    Pairwise source-level descriptor shifts in the OncoPep corpus.

Primary outputs are saved to separated directories:

  <BASE_DIR>/plos_comp/step_01/main_figure/
  <BASE_DIR>/plos_comp/step_01/supplementary_figures/

For checklist convenience, selected final outputs are also copied to:

  <BASE_DIR>/plos_comp/step_01/

Designed for VS Code, Jupyter, and command-line execution.

v4 update: uppercase panel letters; improved Figure 1A label placement; shortened S1 labels; fixed S1 colorbar ticks; removed S1B y-axis label.

Before running:
  1. Confirm BASE_DIR_DEFAULT.
  2. Confirm INPUT_CSV_DEFAULT or pass --input_csv.
  3. Confirm whether the source label should be DCP or DCTPep.
     This script standardizes DCTPep-like labels to DCP by default.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime
from itertools import combinations
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

try:
    from scipy.stats import ks_2samp
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# =============================================================================
# 1. User-editable paths and manuscript constants
# =============================================================================

BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
INPUT_CSV_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/OncoPep_merged_clean_labeled4.csv")

SOURCE_ORDER_DEFAULT = ("dbAMPseq", "APD3", "CancerPPD2", "DCP")

# Manuscript counts used as validation guards. They do not replace computed values.
EXPECTED_SOURCE_COUNTS_DEFAULT = {
    "dbAMPseq": 43641,
    "APD3": 2646,
    "CancerPPD2": 276,
    "DCP": 4442,
}
EXPECTED_TOTAL_DEFAULT = 51005


# =============================================================================
# 2. Unified PLOS-style palette
# =============================================================================

PLOS_PALETTE = {
    "olive_green": "#9AAA63",
    "soft_coral": "#DD705D",
    "charcoal_brown": "#6A5E61",
    "pale_sand": "#F0CF95",
    "muted_mint": "#A8D3B2",
    "teal_green": "#56A98B",
    "blue_teal": "#1F95B8",
    "warm_brown": "#B67D4E",
    "axis_dark": "#333333",
    "light_gray": "#D9D9D9",
    "background": "#FFFFFF",
    "grid": "#EAEAEA",
    "grid_light": "#F3F3F3",
}

SOURCE_FACE = {
    "dbAMPseq": PLOS_PALETTE["blue_teal"],
    "APD3": PLOS_PALETTE["teal_green"],
    "CancerPPD2": PLOS_PALETTE["warm_brown"],
    "DCP": PLOS_PALETTE["soft_coral"],
}

SOURCE_EDGE = {
    "dbAMPseq": "#176F8A",
    "APD3": "#3E806A",
    "CancerPPD2": "#8F5F37",
    "DCP": "#A84F42",
}

S1_DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "plos_cliffs_delta",
    [PLOS_PALETTE["blue_teal"], PLOS_PALETTE["light_gray"], PLOS_PALETTE["soft_coral"]],
    N=256,
)


# =============================================================================
# 3. Configuration
# =============================================================================

@dataclass
class Step1Config:
    base_dir: Path = BASE_DIR_DEFAULT
    input_csv: Path = INPUT_CSV_DEFAULT
    output_root_name: str = "plos_comp/step_01"

    required_columns: Tuple[str, ...] = (
        "sequence",
        "source_db4",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
    )

    source_aliases: Tuple[str, ...] = (
        "source_db4",
        "source",
        "source_db",
        "dataset",
        "db",
        "dataset_source",
    )

    source_order: Tuple[str, ...] = SOURCE_ORDER_DEFAULT
    expected_source_counts: Mapping[str, int] = field(default_factory=lambda: EXPECTED_SOURCE_COUNTS_DEFAULT)
    expected_total_retained: int = EXPECTED_TOTAL_DEFAULT

    min_len: int = 5
    max_len: int = 60

    # Validation controls.
    fail_on_unknown_sources: bool = True
    validate_expected_counts: bool = True
    strict_expected_counts: bool = False

    # Export.
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True

    # Typography.
    font_size: float = 8.3
    title_size: float = 9.2
    label_size: float = 8.8
    tick_size: float = 7.8
    legend_size: float = 7.5
    panel_letter_size: float = 13.0
    annotation_size: float = 7.6
    count_label_size: float = 8.2

    # Figure sizes in inches.
    fig1_size: Tuple[float, float] = (11.4, 6.8)
    fig_s1_size: Tuple[float, float] = (10.9, 5.0)

    # Plot style.
    axis_linewidth: float = 0.8
    violin_alpha: float = 0.38
    violin_width: float = 0.58
    box_width: float = 0.16
    box_alpha: float = 0.98

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def main_dir(self) -> Path:
        return self.output_root / "main_figure"

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"


# =============================================================================
# 4. General utilities
# =============================================================================

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step1Config) -> None:
    for directory in [cfg.output_root, cfg.main_dir, cfg.supp_dir, cfg.reports_dir]:
        ensure_dir(directory)


def sha256_file(path: Path, chunk_size: int = 2**20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        f.write(text)


def save_figure_all_formats(fig: plt.Figure, basepath_no_ext: Path, cfg: Step1Config) -> Dict[str, str]:
    basepath_no_ext.parent.mkdir(parents=True, exist_ok=True)
    output_paths = {
        "pdf": basepath_no_ext.with_suffix(".pdf"),
        "png": basepath_no_ext.with_suffix(".png"),
        "tiff": basepath_no_ext.with_suffix(".tiff"),
    }
    fig.savefig(output_paths["pdf"], bbox_inches="tight", pad_inches=0.06, transparent=False)
    fig.savefig(output_paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06, transparent=False)
    fig.savefig(output_paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06, transparent=False)
    return {k: str(v) for k, v in output_paths.items()}


def find_input_csv(cfg: Step1Config) -> Path:
    candidates = [
        cfg.input_csv,
        cfg.base_dir / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir.parent / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir / "OncoPep_merged_clean_labeled.csv",
        cfg.base_dir / "OncoPep_merged_clean.csv",
        cfg.base_dir / "PepOnco_merged_clean_labeled4.csv",
        cfg.base_dir / "data" / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir / "tables" / "OncoPep_merged_clean_labeled4.csv",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    likely = sorted(cfg.base_dir.glob("**/*OncoPep*clean*.csv"))
    if likely:
        return likely[0]

    raise FileNotFoundError(
        "Could not find the cleaned OncoPep corpus CSV. "
        f"Default attempted path: {cfg.input_csv}. "
        "Edit INPUT_CSV_DEFAULT or pass --input_csv."
    )


def get_script_text() -> Tuple[str, Optional[Path], Optional[str]]:
    """
    Return script text for reproducibility snapshots.

    In normal .py execution, __file__ is available. In pasted Jupyter execution, __file__
    may not exist, so a best-effort source reconstruction is used.
    """
    file_name = globals().get("__file__", None)
    if file_name:
        script_path = Path(file_name).resolve()
        if script_path.exists():
            text = script_path.read_text(encoding="utf-8")
            return text, script_path, sha256_file(script_path)

    try:
        text = inspect.getsource(sys.modules[__name__])
        return text, None, sha256_text(text)
    except Exception:
        text = (
            "# OncoPep Step 1 plotting code snapshot could not be recovered automatically.\n"
            "# If this was run from a notebook cell, export the notebook or save the script manually.\n"
        )
        return text, None, sha256_text(text)


def set_plot_style(cfg: Step1Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": cfg.axis_linewidth,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS_PALETTE["background"],
            "axes.facecolor": PLOS_PALETTE["background"],
            "savefig.facecolor": PLOS_PALETTE["background"],
            "text.color": PLOS_PALETTE["axis_dark"],
            "axes.labelcolor": PLOS_PALETTE["axis_dark"],
            "axes.edgecolor": PLOS_PALETTE["axis_dark"],
            "xtick.color": PLOS_PALETTE["axis_dark"],
            "ytick.color": PLOS_PALETTE["axis_dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, linewidth=0.45, color=PLOS_PALETTE["grid"], zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PLOS_PALETTE["axis_dark"])
        ax.spines[side].set_linewidth(0.75)


def panel_letter(ax: plt.Axes, letter: str, cfg: Step1Config, dx: float = -0.12, dy: float = 1.08) -> None:
    ax.text(
        dx,
        dy,
        str(letter).upper(),
        transform=ax.transAxes,
        fontsize=cfg.panel_letter_size,
        fontweight="bold",
        ha="left",
        va="top",
        color=PLOS_PALETTE["axis_dark"],
    )


def finite_array(x: Sequence[float]) -> np.ndarray:
    arr = np.asarray(x, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_sequence_column(series: pd.Series) -> pd.Series:
    return series.where(series.notna(), "").astype(str).str.strip().str.upper()


def summary_stats(x: Sequence[float]) -> Dict[str, Any]:
    arr = finite_array(x)
    if len(arr) == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "min": np.nan,
            "max": np.nan,
        }
    return {
        "n": int(len(arr)),
        "mean": float(np.mean(arr)),
        "sd": float(np.std(arr, ddof=1)) if len(arr) > 1 else np.nan,
        "median": float(np.median(arr)),
        "q25": float(np.percentile(arr, 25)),
        "q75": float(np.percentile(arr, 75)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
    }


# =============================================================================
# 5. Data preparation and validation
# =============================================================================

AA20_SET = set("ACDEFGHIKLMNPQRSTVWY")


def canonical_only(seq: str) -> bool:
    return isinstance(seq, str) and len(seq) > 0 and set(seq).issubset(AA20_SET)


def max_run_length(seq: str) -> int:
    if not isinstance(seq, str) or len(seq) == 0:
        return 0
    best = cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return int(best)


def shannon_entropy(seq: str) -> float:
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    counts = pd.Series(list(seq)).value_counts().to_numpy(dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def dominant_residue_fraction(seq: str) -> float:
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    return float(pd.Series(list(seq)).value_counts().max() / len(seq))


def source_key(label: Any) -> str:
    """
    Robust key for source-label normalization.
    Removes punctuation/spacing and converts to lowercase.
    """
    raw = "" if pd.isna(label) else str(label)
    cleaned = raw.strip().lower()
    for token in [" ", "_", "-", "/", "\\", ".", "(", ")", "[", "]", "{", "}"]:
        cleaned = cleaned.replace(token, "")
    return cleaned


SOURCE_KEY_TO_CANONICAL = {
    "dbampseq": "dbAMPseq",
    "dbamp": "dbAMPseq",
    "apd3": "APD3",
    "apd": "APD3",
    "cancerppd2": "CancerPPD2",
    "cancerppd": "CancerPPD2",
    "dcp": "DCP",
    "dctpep": "DCP",
    "dctpeptide": "DCP",
    "dcpdctpep": "DCP",
    "dctpepdcp": "DCP",
}


def normalize_source_label(label: Any) -> str:
    key = source_key(label)
    return SOURCE_KEY_TO_CANONICAL.get(key, f"UNKNOWN::{str(label).strip()}")


def harmonize_column_names(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    out = df.copy()
    if "source_db4" not in out.columns:
        for candidate in cfg.source_aliases:
            if candidate in out.columns:
                out = out.rename(columns={candidate: "source_db4"})
                break
    return out


def validate_required_columns(df: pd.DataFrame, required: Iterable[str], context: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{context}: missing required columns: {missing}")


def build_source_label_audit(df: pd.DataFrame, source_col: str = "source_db4") -> pd.DataFrame:
    audit = (
        df[source_col]
        .where(df[source_col].notna(), "")
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
        .reset_index()
    )
    audit.columns = ["original_source_label", "n_rows"]
    audit["normalized_source"] = audit["original_source_label"].apply(normalize_source_label)
    audit["status"] = np.where(audit["normalized_source"].str.startswith("UNKNOWN::"), "unknown", "recognized")
    return audit[["original_source_label", "normalized_source", "status", "n_rows"]]


def prepare_dataframe(raw_df: pd.DataFrame, cfg: Step1Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = harmonize_column_names(df, cfg)
    validate_required_columns(df, cfg.required_columns, "Input corpus")

    source_audit = build_source_label_audit(df, "source_db4")
    unknown_audit = source_audit[source_audit["status"] == "unknown"].copy()
    if not unknown_audit.empty and cfg.fail_on_unknown_sources:
        raise ValueError(
            "Unknown source labels detected. Fix source mapping before plotting:\n"
            + unknown_audit.to_string(index=False)
        )

    df["sequence_raw"] = df["sequence"]
    df["sequence"] = normalize_sequence_column(df["sequence"])
    df["source_original"] = df["source_db4"].where(df["source_db4"].notna(), "").astype(str).str.strip()
    df["source"] = df["source_original"].apply(normalize_source_label)
    df.loc[df["source"].str.startswith("UNKNOWN::"), "source"] = "Other"

    for col in ["length", "net_charge_pH7", "hydrophobicity_KD"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["sequence_id"] = np.arange(1, len(df) + 1)
    df["len_from_seq"] = df["sequence"].str.len()
    df["is_canonical"] = df["sequence"].apply(canonical_only)

    # Computed descriptors.
    df["shannon_entropy"] = df["sequence"].apply(shannon_entropy)
    df["max_homopolymer_run"] = df["sequence"].apply(max_run_length)
    df["dominant_residue_fraction"] = df["sequence"].apply(dominant_residue_fraction)

    # Basic final-retained flag for this Step 1 audit.
    df["length_missing"] = df["length"].isna()
    df["charge_missing"] = df["net_charge_pH7"].isna()
    df["hydrophobicity_missing"] = df["hydrophobicity_KD"].isna()
    df["sequence_missing"] = df["sequence"].eq("")

    length_rounded = pd.Series(pd.NA, index=df.index, dtype="Int64")
    length_present = df["length"].notna()
    length_rounded.loc[length_present] = df.loc[length_present, "length"].round().astype("Int64")
    df["length_rounded_int"] = length_rounded

    df["length_mismatch"] = False
    df.loc[length_present, "length_mismatch"] = (
        df.loc[length_present, "length_rounded_int"].astype("Int64")
        != df.loc[length_present, "len_from_seq"].astype("Int64")
    ).fillna(False)

    df["length_out_of_range"] = np.where(
        df["sequence_missing"],
        False,
        (df["len_from_seq"] < cfg.min_len) | (df["len_from_seq"] > cfg.max_len),
    )

    df["exclude_step1"] = (
        df["sequence_missing"]
        | (~df["is_canonical"])
        | df["length_missing"]
        | df["charge_missing"]
        | df["hydrophobicity_missing"]
        | df["length_mismatch"]
        | df["length_out_of_range"]
    )
    df["retained_step1"] = ~df["exclude_step1"]

    df["source_for_plot"] = df["source"]
    if cfg.fail_on_unknown_sources:
        unexpected_after_norm = sorted(set(df["source_for_plot"]) - set(cfg.source_order))
        unexpected_after_norm = [x for x in unexpected_after_norm if x != "Other"]
        if unexpected_after_norm:
            raise ValueError(f"Unexpected normalized sources after mapping: {unexpected_after_norm}")

    return df, source_audit


# =============================================================================
# 6. Step 1 analysis tables
# =============================================================================

def build_source_counts(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    retained = df[df["retained_step1"]].copy()
    counts = (
        retained["source_for_plot"]
        .value_counts()
        .reindex(list(cfg.source_order))
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    counts.columns = ["source", "n_sequences"]
    total = int(counts["n_sequences"].sum())
    counts["percent_total"] = np.where(total > 0, 100 * counts["n_sequences"] / total, np.nan)
    counts["total_retained"] = total
    return counts


def validate_expected_counts(counts_df: pd.DataFrame, cfg: Step1Config) -> Tuple[bool, pd.DataFrame, list[str]]:
    rows = []
    warnings_list: list[str] = []

    observed = counts_df.set_index("source")["n_sequences"].to_dict()
    observed_total = int(counts_df["n_sequences"].sum())
    expected_total = int(cfg.expected_total_retained)

    rows.append(
        {
            "item": "total_retained",
            "observed": observed_total,
            "expected": expected_total,
            "difference": observed_total - expected_total,
            "matches": observed_total == expected_total,
        }
    )

    for source, expected in cfg.expected_source_counts.items():
        obs = int(observed.get(source, 0))
        rows.append(
            {
                "item": f"source::{source}",
                "observed": obs,
                "expected": int(expected),
                "difference": obs - int(expected),
                "matches": obs == int(expected),
            }
        )

    audit = pd.DataFrame(rows)
    ok = bool(audit["matches"].all())

    if cfg.validate_expected_counts and not ok:
        message = (
            "Computed Step 1 counts do not match the manuscript validation counts. "
            "Inspect count_validation_audit.csv before using the figure."
        )
        warnings_list.append(message)
        if cfg.strict_expected_counts:
            raise ValueError(message + "\n" + audit.to_string(index=False))
        warnings.warn(message, RuntimeWarning)

    return ok, audit, warnings_list


def build_descriptor_distribution_table(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    retained = df[df["retained_step1"] & df["source_for_plot"].isin(cfg.source_order)].copy()
    cols = [
        "sequence_id",
        "source_for_plot",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "shannon_entropy",
        "dominant_residue_fraction",
        "max_homopolymer_run",
    ]
    out = retained[cols].copy()
    out = out.rename(columns={"source_for_plot": "source"})

    if out.empty:
        raise ValueError("No retained rows available for descriptor plotting after source/filter validation.")

    for source in cfg.source_order:
        if (out["source"] == source).sum() == 0:
            warnings.warn(f"Source {source!r} has zero retained rows in descriptor table.", RuntimeWarning)

    return out


def build_source_summary_table(descriptor_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    descriptors = [
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "shannon_entropy",
        "dominant_residue_fraction",
        "max_homopolymer_run",
    ]
    rows = []
    for source in cfg.source_order:
        sub = descriptor_df[descriptor_df["source"] == source]
        row = {"source": source, "n_sequences": int(len(sub))}
        for col in descriptors:
            s = summary_stats(sub[col].to_numpy())
            for key, value in s.items():
                row[f"{col}_{key}"] = value
        rows.append(row)
    return pd.DataFrame(rows)


def cliffs_delta_exact(x: Sequence[float], y: Sequence[float]) -> float:
    """
    Exact Cliff's delta using sorted search.

    δ = [#(x > y) - #(x < y)] / (n_x * n_y)

    Ties contribute 0. Complexity is O((n_x + n_y) log n_y), avoiding
    the memory cost of all pairwise comparisons.
    """
    x_arr = finite_array(x)
    y_arr = finite_array(y)
    if len(x_arr) == 0 or len(y_arr) == 0:
        return np.nan

    y_sorted = np.sort(y_arr)
    less_than_x = np.searchsorted(y_sorted, x_arr, side="left")
    less_or_equal_x = np.searchsorted(y_sorted, x_arr, side="right")
    greater_than_x = len(y_sorted) - less_or_equal_x

    gt = np.sum(less_than_x)
    lt = np.sum(greater_than_x)
    return float((gt - lt) / (len(x_arr) * len(y_sorted)))


def ks_test(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float]:
    if not HAS_SCIPY:
        return np.nan, np.nan
    x_arr = finite_array(x)
    y_arr = finite_array(y)
    if len(x_arr) < 10 or len(y_arr) < 10:
        return np.nan, np.nan
    stat, p_value = ks_2samp(x_arr, y_arr)
    return float(stat), float(p_value)


def build_pairwise_shift_table(descriptor_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    descriptors = [
        ("length", "Length"),
        ("net_charge_pH7", "Net charge"),
        ("hydrophobicity_KD", "Hydrophobicity"),
        ("shannon_entropy", "Shannon entropy"),
        ("max_homopolymer_run", "Max homopolymer run"),
        ("dominant_residue_fraction", "Dominant residue fraction"),
    ]

    rows = []
    for source_a, source_b in combinations(cfg.source_order, 2):
        sub_a = descriptor_df[descriptor_df["source"] == source_a]
        sub_b = descriptor_df[descriptor_df["source"] == source_b]

        for descriptor_col, descriptor_label in descriptors:
            a_values = sub_a[descriptor_col].to_numpy()
            b_values = sub_b[descriptor_col].to_numpy()
            delta = cliffs_delta_exact(a_values, b_values)
            ks_stat, ks_pvalue = ks_test(a_values, b_values)

            rows.append(
                {
                    "panel": "A",
                    "data_type": "pairwise_cliffs_delta",
                    "source_a": source_a,
                    "source_b": source_b,
                    "source_pair": f"{source_a} vs {source_b}",
                    "descriptor": descriptor_label,
                    "descriptor_column": descriptor_col,
                    "cliffs_delta": delta,
                    "abs_cliffs_delta": abs(delta) if np.isfinite(delta) else np.nan,
                    "ks_stat": ks_stat,
                    "ks_pvalue": ks_pvalue,
                    "x_n_input": int(finite_array(a_values).size),
                    "y_n_input": int(finite_array(b_values).size),
                    "x_n_used": int(finite_array(a_values).size),
                    "y_n_used": int(finite_array(b_values).size),
                    "subsampled": False,
                    "cliffs_delta_method": "exact_sorted_search",
                }
            )

    return pd.DataFrame(rows)


def build_descriptor_shift_summary(pairwise_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        pairwise_df.groupby("descriptor", as_index=False)
        .agg(
            mean_abs_cliffs_delta=("abs_cliffs_delta", "mean"),
            max_abs_cliffs_delta=("abs_cliffs_delta", "max"),
            n_pairwise_comparisons=("source_pair", "nunique"),
        )
        .sort_values("mean_abs_cliffs_delta", ascending=False)
    )
    summary.insert(0, "panel", "B")
    summary["data_type"] = "descriptor_shift_summary"
    return summary


def build_figure1_source_data_all(counts_df: pd.DataFrame, descriptor_df: pd.DataFrame) -> pd.DataFrame:
    panel_a = counts_df.copy()
    panel_a.insert(0, "panel", "A")
    panel_a["data_type"] = "source_counts"

    descriptor_long = descriptor_df.melt(
        id_vars=["sequence_id", "source"],
        value_vars=[
            "length",
            "net_charge_pH7",
            "hydrophobicity_KD",
            "shannon_entropy",
            "dominant_residue_fraction",
        ],
        var_name="descriptor",
        value_name="value",
    )
    descriptor_long.insert(
        0,
        "panel",
        descriptor_long["descriptor"].map(
            {
                "length": "B",
                "net_charge_pH7": "C",
                "hydrophobicity_KD": "D",
                "shannon_entropy": "E",
                "dominant_residue_fraction": "F",
            }
        ),
    )
    descriptor_long["data_type"] = "descriptor_distribution"

    return pd.concat([panel_a, descriptor_long], ignore_index=True, sort=False)


def build_s1_source_data_all(pairwise_df: pd.DataFrame, shift_summary_df: pd.DataFrame) -> pd.DataFrame:
    panel_a = pairwise_df.copy()
    panel_b = shift_summary_df.copy()
    return pd.concat([panel_a, panel_b], ignore_index=True, sort=False)


# =============================================================================
# 7. Plotting helpers
# =============================================================================

def plot_violin_box(
    ax: plt.Axes,
    descriptor_df: pd.DataFrame,
    column: str,
    ylabel: str,
    cfg: Step1Config,
    y_limits: Optional[Tuple[float, float]] = None,
    zero_line: bool = False,
) -> None:
    positions = np.arange(1, len(cfg.source_order) + 1)
    data = [
        finite_array(descriptor_df.loc[descriptor_df["source"] == source, column].to_numpy())
        for source in cfg.source_order
    ]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=cfg.violin_width,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, source in zip(vp["bodies"], cfg.source_order):
        body.set_facecolor(SOURCE_FACE[source])
        body.set_edgecolor(SOURCE_EDGE[source])
        body.set_alpha(cfg.violin_alpha)
        body.set_linewidth(0.9)
        body.set_zorder(3)

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=cfg.box_width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color=PLOS_PALETTE["axis_dark"], linewidth=1.15),
        whiskerprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        capprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        boxprops=dict(linewidth=0.75),
    )

    for patch, source in zip(bp["boxes"], cfg.source_order):
        patch.set_facecolor("#FFFFFF")
        patch.set_alpha(cfg.box_alpha)
        patch.set_edgecolor(SOURCE_EDGE[source])
        patch.set_zorder(4)

    if zero_line:
        ax.axhline(0, color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75, linestyle="--", zorder=2)

    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(cfg.source_order, rotation=8, ha="right")

    if y_limits is not None:
        ax.set_ylim(*y_limits)

    beautify_axis(ax, "y")


def source_handles(cfg: Step1Config) -> list[Patch]:
    return [
        Patch(
            facecolor=SOURCE_FACE[source],
            edgecolor=SOURCE_EDGE[source],
            label=source,
            linewidth=0.9,
        )
        for source in cfg.source_order
    ]


def heatmap_annotation_color(value: float) -> str:
    return "#FFFFFF" if np.isfinite(value) and abs(value) >= 0.55 else PLOS_PALETTE["axis_dark"]


# =============================================================================
# 8. Main Figure 1
# =============================================================================

def generate_main_figure1(
    counts_df: pd.DataFrame,
    descriptor_df: pd.DataFrame,
    cfg: Step1Config,
) -> Dict[str, str]:
    total_n = int(counts_df["n_sequences"].sum())

    fig, axes = plt.subplots(2, 3, figsize=cfg.fig1_size)
    fig.subplots_adjust(
        left=0.075,
        right=0.985,
        top=0.925,
        bottom=0.145,
        wspace=0.34,
        hspace=0.40,
    )

    # Panel A: source composition.
    ax = axes[0, 0]
    x = np.arange(len(cfg.source_order))
    counts = counts_df.set_index("source").reindex(cfg.source_order)["n_sequences"].fillna(0).astype(int)
    percentages = counts_df.set_index("source").reindex(cfg.source_order)["percent_total"].fillna(0)

    bars = ax.bar(
        x,
        counts.values,
        width=0.55,
        color=[SOURCE_FACE[source] for source in cfg.source_order],
        edgecolor=[SOURCE_EDGE[source] for source in cfg.source_order],
        linewidth=0.95,
        zorder=3,
    )
    ax.set_yscale("log")
    ax.set_ylabel("Retained sequences (log scale)")
    ax.set_xticks(x)
    ax.set_xticklabels(cfg.source_order, rotation=8, ha="right")
    ax.set_title("Source composition", pad=8)
    beautify_axis(ax, "y")

    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 2.35)

    for bar, source in zip(bars, cfg.source_order):
        count = int(counts.loc[source])
        pct = float(percentages.loc[source])
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            count * 1.10 if count > 0 else 1,
            f"{count:,}\n({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=cfg.count_label_size,
            color=PLOS_PALETTE["axis_dark"],
            linespacing=1.1,
        )

    ax.text(
        0.97,
        0.12,
        f"Retained corpus\nN = {total_n:,} peptides",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor=PLOS_PALETTE["pale_sand"],
            edgecolor=PLOS_PALETTE["charcoal_brown"],
            linewidth=0.6,
        ),
    )
    panel_letter(ax, "a", cfg)

    specs = [
        (axes[0, 1], "length", "Length (aa)", "Length", "b", None, False),
        (axes[0, 2], "net_charge_pH7", "Net charge at pH 7", "Net charge", "c", None, True),
        (axes[1, 0], "hydrophobicity_KD", "Hydrophobicity\n(Kyte–Doolittle)", "Hydrophobicity", "d", None, True),
        (axes[1, 1], "shannon_entropy", "Shannon entropy (bits)", "Shannon entropy", "e", None, False),
        (axes[1, 2], "dominant_residue_fraction", "Dominant residue fraction", "Dominant residue fraction", "f", (0.05, 1.0), False),
    ]

    for ax_i, col, ylabel, title, label, ylim, zero_line in specs:
        plot_violin_box(ax_i, descriptor_df, col, ylabel, cfg, y_limits=ylim, zero_line=zero_line)
        ax_i.set_title(title, pad=8)
        panel_letter(ax_i, label, cfg)

    handles = source_handles(cfg)
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=4,
        frameon=True,
        bbox_to_anchor=(0.5, 0.045),
        fontsize=cfg.legend_size,
        edgecolor=PLOS_PALETTE["light_gray"],
        facecolor=PLOS_PALETTE["background"],
        handlelength=1.5,
        handletextpad=0.5,
        columnspacing=1.25,
        borderpad=0.35,
    )

    base = cfg.main_dir / "Figure_1_redesigned"
    paths = save_figure_all_formats(fig, base, cfg)
    plt.close(fig)
    return paths


# =============================================================================
# 9. Supplementary Figure S1
# =============================================================================

def generate_supplementary_figure_s1(
    pairwise_df: pd.DataFrame,
    shift_summary_df: pd.DataFrame,
    cfg: Step1Config,
) -> Dict[str, str]:
    descriptors = [
        "Length",
        "Net charge",
        "Hydrophobicity",
        "Shannon entropy",
        "Max homopolymer run",
        "Dominant residue fraction",
    ]
    descriptor_display = {
        "Length": "Length",
        "Net charge": "Net charge",
        "Hydrophobicity": "Hydrophobicity",
        "Shannon entropy": "Shannon entropy",
        "Max homopolymer run": "Max run",
        "Dominant residue fraction": "Dominant residue",
    }
    source_pairs = [f"{a} vs {b}" for a, b in combinations(cfg.source_order, 2)]

    heatmap = np.full((len(source_pairs), len(descriptors)), np.nan)
    for i, pair in enumerate(source_pairs):
        for j, desc in enumerate(descriptors):
            values = pairwise_df.loc[
                (pairwise_df["source_pair"] == pair) & (pairwise_df["descriptor"] == desc),
                "cliffs_delta",
            ].to_numpy()
            if len(values):
                heatmap[i, j] = values[0]

    fig = plt.figure(figsize=cfg.fig_s1_size)
    gs = fig.add_gridspec(1, 2, width_ratios=[2.25, 1.05], wspace=0.42)
    ax_hm = fig.add_subplot(gs[0, 0])
    ax_bar = fig.add_subplot(gs[0, 1])
    fig.subplots_adjust(left=0.12, right=0.965, top=0.90, bottom=0.25)

    # Panel A: Cliff's delta heatmap.
    im = ax_hm.imshow(
        heatmap,
        aspect="auto",
        cmap=S1_DIVERGING_CMAP,
        vmin=-1.0,
        vmax=1.0,
        interpolation="nearest",
    )
    ax_hm.set_xticks(np.arange(len(descriptors)))
    ax_hm.set_xticklabels([descriptor_display[d] for d in descriptors], rotation=26, ha="right")
    ax_hm.set_yticks(np.arange(len(source_pairs)))
    ax_hm.set_yticklabels(source_pairs)
    ax_hm.tick_params(length=0)
    ax_hm.set_title("Pairwise source shifts", pad=8)

    for i in range(heatmap.shape[0]):
        for j in range(heatmap.shape[1]):
            value = heatmap[i, j]
            if np.isfinite(value):
                ax_hm.text(
                    j,
                    i,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=6.7,
                    color=heatmap_annotation_color(value),
                )

    for spine in ax_hm.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax_hm, fraction=0.04, pad=0.03)
    cbar.set_label("Cliff’s δ", fontsize=cfg.legend_size)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    cbar.outline.set_linewidth(0.6)
    cbar.outline.set_edgecolor(PLOS_PALETTE["light_gray"])

    ax_hm.text(
        0.0,
        -0.25,
        "Positive δ: descriptor is higher in the first source of the pair; negative δ: lower.",
        transform=ax_hm.transAxes,
        ha="left",
        va="top",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
    )
    panel_letter(ax_hm, "a", cfg, dx=-0.13, dy=1.08)

    # Panel B: ranked mean absolute Cliff's delta.
    summary = shift_summary_df.sort_values("mean_abs_cliffs_delta", ascending=True)
    y = np.arange(len(summary))
    max_x = max(float(summary["mean_abs_cliffs_delta"].max()), 0.05)

    ax_bar.barh(
        y,
        summary["mean_abs_cliffs_delta"],
        color=PLOS_PALETTE["charcoal_brown"],
        edgecolor=PLOS_PALETTE["axis_dark"],
        linewidth=0.55,
        zorder=3,
    )
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels(summary["descriptor"].replace({
        "Max homopolymer run": "Max run",
        "Dominant residue fraction": "Dominant residue",
    }))
    ax_bar.set_ylabel("")
    ax_bar.set_xlabel("Mean |Cliff’s δ|")
    ax_bar.set_title("Descriptor-level shift", pad=8)
    ax_bar.set_xlim(0, max_x * 1.18)
    beautify_axis(ax_bar, "x")

    for yi, value in zip(y, summary["mean_abs_cliffs_delta"]):
        ax_bar.text(
            value + max_x * 0.025,
            yi,
            f"{value:.2f}",
            ha="left",
            va="center",
            fontsize=cfg.annotation_size,
            color=PLOS_PALETTE["axis_dark"],
        )

    panel_letter(ax_bar, "b", cfg, dx=-0.30, dy=1.08)

    base = cfg.supp_dir / "Supplementary_Figure_S1_redesigned"
    paths = save_figure_all_formats(fig, base, cfg)
    plt.close(fig)
    return paths


# =============================================================================
# 10. Output helpers
# =============================================================================

def save_panel_specific_figure1_data(descriptor_df: pd.DataFrame, cfg: Step1Config) -> None:
    mapping = {
        "B": ("length", "Figure_1_panel_b_source_data.csv"),
        "C": ("net_charge_pH7", "Figure_1_panel_c_source_data.csv"),
        "D": ("hydrophobicity_KD", "Figure_1_panel_d_source_data.csv"),
        "E": ("shannon_entropy", "Figure_1_panel_e_source_data.csv"),
        "F": ("dominant_residue_fraction", "Figure_1_panel_f_source_data.csv"),
    }
    for panel, (col, filename) in mapping.items():
        panel_df = descriptor_df[["sequence_id", "source", col]].copy()
        panel_df.insert(0, "panel", panel)
        panel_df = panel_df.rename(columns={col: "value"})
        panel_df["descriptor"] = col
        save_csv(panel_df, cfg.main_dir / filename)


def save_root_level_copies(cfg: Step1Config) -> list[str]:
    """
    Preserve separated directories while also creating root-level convenience copies
    matching common journal/checklist paths.
    """
    if not cfg.create_root_level_copies:
        return []

    copy_pairs = [
        (cfg.main_dir / "Figure_1_redesigned.png", cfg.output_root / "Figure_1_redesigned.png"),
        (cfg.main_dir / "Figure_1_redesigned.pdf", cfg.output_root / "Figure_1_redesigned.pdf"),
        (cfg.main_dir / "Figure_1_redesigned.tiff", cfg.output_root / "Figure_1_redesigned.tiff"),
        (cfg.main_dir / "Figure_1_source_data_all_panels.csv", cfg.output_root / "Figure_1_source_data_all_panels.csv"),
        (cfg.main_dir / "Figure_1_redesign_code.py", cfg.output_root / "Figure_1_redesign_code.py"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.png", cfg.output_root / "Supplementary_Figure_S1_redesigned.png"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.pdf", cfg.output_root / "Supplementary_Figure_S1_redesigned.pdf"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.tiff", cfg.output_root / "Supplementary_Figure_S1_redesigned.tiff"),
        (cfg.supp_dir / "Supplementary_Figure_S1_source_data_all_panels.csv", cfg.output_root / "Supplementary_Figure_S1_source_data_all_panels.csv"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesign_code.py", cfg.output_root / "Supplementary_Figure_S1_redesign_code.py"),
    ]

    copied = []
    for src, dst in copy_pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def save_code_snapshots(cfg: Step1Config) -> Tuple[str, Optional[str], Optional[str]]:
    script_text, script_path, script_hash = get_script_text()
    save_text(script_text, cfg.main_dir / "Figure_1_redesign_code.py")
    save_text(script_text, cfg.supp_dir / "Supplementary_Figure_S1_redesign_code.py")
    save_text(script_text, cfg.output_root / "Step_01_full_redesign_code.py")
    return script_text, str(script_path) if script_path else None, script_hash


def write_readme(cfg: Step1Config, warnings_list: Sequence[str]) -> Path:
    text = f"""# OncoPep Step 1 PLOS figure package

Generated: {datetime.now().isoformat()}

Primary outputs:

- main_figure/Figure_1_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S1_redesigned.png/.pdf/.tiff

Source-data files:

- main_figure/Figure_1_source_data_all_panels.csv
- supplementary_figures/Supplementary_Figure_S1_source_data_all_panels.csv
- panel-specific source-data CSVs in each figure directory

Notes:

- Figure 1 uses source composition and descriptor distributions.
- Panel labels are uppercase: A, B, C...
- Supplementary Figure S1 uses exact Cliff's delta computed by sorted-search counting.
- KS statistics are saved in source data but not plotted.
- Positive Cliff's delta means the descriptor is higher in the first source of the pair.

Warnings:

{chr(10).join('- ' + w for w in warnings_list) if warnings_list else '- None'}
"""
    path = cfg.output_root / "README_step_01_outputs.txt"
    save_text(text, path)
    return path


# =============================================================================
# 11. Main run function
# =============================================================================

def run_step1(cfg: Step1Config) -> None:
    ensure_output_tree(cfg)
    set_plot_style(cfg)

    warnings_list: list[str] = []

    input_csv = find_input_csv(cfg)
    raw_df = pd.read_csv(input_csv, low_memory=False)
    df, source_audit_df = prepare_dataframe(raw_df, cfg)

    save_csv(source_audit_df, cfg.reports_dir / "source_label_audit.csv")
    save_csv(source_audit_df, cfg.output_root / "source_label_audit.csv")

    counts_df = build_source_counts(df, cfg)
    counts_ok, count_audit_df, count_warnings = validate_expected_counts(counts_df, cfg)
    warnings_list.extend(count_warnings)

    descriptor_df = build_descriptor_distribution_table(df, cfg)
    source_summary_df = build_source_summary_table(descriptor_df, cfg)
    pairwise_df = build_pairwise_shift_table(descriptor_df, cfg)
    shift_summary_df = build_descriptor_shift_summary(pairwise_df)
    figure1_all = build_figure1_source_data_all(counts_df, descriptor_df)
    s1_all = build_s1_source_data_all(pairwise_df, shift_summary_df)

    # Save source-data before plotting.
    save_csv(counts_df, cfg.main_dir / "Figure_1_panel_a_source_data.csv")
    save_csv(counts_df, cfg.main_dir / "Figure_1_panel_a_source_counts.csv")
    save_csv(descriptor_df, cfg.main_dir / "Figure_1_panel_bf_descriptor_distributions.csv")
    save_panel_specific_figure1_data(descriptor_df, cfg)
    save_csv(source_summary_df, cfg.main_dir / "Figure_1_descriptor_summary_by_source.csv")
    save_csv(figure1_all, cfg.main_dir / "Figure_1_source_data_all_panels.csv")

    save_csv(pairwise_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_a_source_data.csv")
    save_csv(pairwise_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_a_effect_sizes.csv")
    save_csv(shift_summary_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_b_source_data.csv")
    save_csv(shift_summary_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_b_descriptor_shift_summary.csv")
    save_csv(s1_all, cfg.supp_dir / "Supplementary_Figure_S1_source_data_all_panels.csv")

    save_csv(count_audit_df, cfg.reports_dir / "count_validation_audit.csv")
    save_csv(count_audit_df, cfg.output_root / "count_validation_audit.csv")

    # Generate figures.
    fig1_paths = generate_main_figure1(counts_df, descriptor_df, cfg)
    s1_paths = generate_supplementary_figure_s1(pairwise_df, shift_summary_df, cfg)

    # Save code snapshots after figure generation.
    _, script_path, script_hash = save_code_snapshots(cfg)

    readme_path = write_readme(cfg, warnings_list)
    root_copies = save_root_level_copies(cfg)

    manifest = {
        "step": "step_01",
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python_version": sys.version,
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": mpl.__version__,
            "scipy_available": HAS_SCIPY,
        },
        "input_csv": str(input_csv),
        "input_sha256": sha256_file(input_csv),
        "script_path": script_path,
        "script_sha256": script_hash,
        "base_dir": str(cfg.base_dir),
        "output_root": str(cfg.output_root),
        "main_dir": str(cfg.main_dir),
        "supp_dir": str(cfg.supp_dir),
        "source_order": list(cfg.source_order),
        "total_retained_sequences": int(counts_df["n_sequences"].sum()),
        "source_counts": counts_df.to_dict(orient="records"),
        "expected_count_validation": {
            "enabled": cfg.validate_expected_counts,
            "strict": cfg.strict_expected_counts,
            "passed": counts_ok,
            "audit_file": str(cfg.output_root / "count_validation_audit.csv"),
        },
        "source_label_audit_file": str(cfg.output_root / "source_label_audit.csv"),
        "cliffs_delta": {
            "method": "exact_sorted_search",
            "subsampled": False,
        },
        "outputs": {
            "main_figure": fig1_paths,
            "supplementary_figure_s1": s1_paths,
            "root_level_copies": root_copies,
            "readme": str(readme_path),
        },
        "warnings": warnings_list,
        "config": asdict(cfg),
    }
    save_json(manifest, cfg.output_root / "step_01_manifest.json")
    save_json(manifest, cfg.reports_dir / "step_01_manifest.json")

    print("\n✅ Step 1 PLOS figure package generated successfully.")
    print(f"Input CSV: {input_csv}")
    print(f"Main figure directory: {cfg.main_dir}")
    print(f"Supplementary figure directory: {cfg.supp_dir}")
    print(f"Root output directory: {cfg.output_root}")

    print("\nSaved main figure files:")
    for path in fig1_paths.values():
        print(f"  - {path}")

    print("\nSaved Supplementary Figure S1 files:")
    for path in s1_paths.values():
        print(f"  - {path}")

    print("\nKey reports:")
    print(f"  - {cfg.output_root / 'step_01_manifest.json'}")
    print(f"  - {cfg.output_root / 'source_label_audit.csv'}")
    print(f"  - {cfg.output_root / 'count_validation_audit.csv'}")
    print(f"  - {readme_path}")

    if warnings_list:
        print("\n⚠️ Warnings:")
        for warning in warnings_list:
            print(f"  - {warning}")


# =============================================================================
# 12. Notebook and CLI entry points
# =============================================================================

def main_notebook(
    base_dir: str | Path = BASE_DIR_DEFAULT,
    input_csv: str | Path = INPUT_CSV_DEFAULT,
    min_len: int = 5,
    max_len: int = 60,
    strict_expected_counts: bool = False,
    fail_on_unknown_sources: bool = True,
) -> None:
    cfg = Step1Config(
        base_dir=Path(base_dir),
        input_csv=Path(input_csv),
        min_len=min_len,
        max_len=max_len,
        strict_expected_counts=strict_expected_counts,
        fail_on_unknown_sources=fail_on_unknown_sources,
    )
    run_step1(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    """
    Parse command-line arguments safely.

    Jupyter/IPython kernels inject extra arguments such as:
        --f=/run/user/.../kernel-xxxx.json

    Using parse_known_args prevents those kernel-specific arguments from
    crashing the script when the code is pasted into a notebook cell.
    """
    parser = argparse.ArgumentParser(
        description="Generate OncoPep Step 1 PLOS Computational Biology figures."
    )
    parser.add_argument(
        "--base_dir",
        type=str,
        default=str(BASE_DIR_DEFAULT),
        help="Base project directory containing the PLOS folder.",
    )
    parser.add_argument(
        "--input_csv",
        type=str,
        default=str(INPUT_CSV_DEFAULT),
        help="Path to the cleaned OncoPep corpus CSV.",
    )
    parser.add_argument("--min_len", type=int, default=5, help="Minimum retained peptide length.")
    parser.add_argument("--max_len", type=int, default=60, help="Maximum retained peptide length.")
    parser.add_argument(
        "--strict_counts",
        action="store_true",
        help="Raise an error if computed counts differ from manuscript validation counts.",
    )
    parser.add_argument(
        "--allow_unknown_sources",
        action="store_true",
        help="Do not fail on unknown source labels. Unknown labels are excluded from the four-source plots.",
    )
    parser.add_argument(
        "--no_root_copies",
        action="store_true",
        help="Do not create root-level convenience copies of final outputs.",
    )

    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"ℹ️ Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    cfg = Step1Config(
        base_dir=Path(args.base_dir),
        input_csv=Path(args.input_csv),
        min_len=args.min_len,
        max_len=args.max_len,
        strict_expected_counts=args.strict_counts,
        fail_on_unknown_sources=not args.allow_unknown_sources,
        create_root_level_copies=not args.no_root_copies,
    )
    run_step1(cfg)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 1 PLOS Computational Biology figure package.

This script generates the revised Step 1 figure package:

  Main Figure 1
    Source composition and descriptor heterogeneity of the ACP-oriented OncoPep corpus.

  Supplementary Figure S1
    Pairwise source-level descriptor shifts in the OncoPep corpus.

Primary outputs are saved to separated directories:

  <BASE_DIR>/plos_comp/step_01/main_figure/
  <BASE_DIR>/plos_comp/step_01/supplementary_figures/

For checklist convenience, selected final outputs are also copied to:

  <BASE_DIR>/plos_comp/step_01/

Designed for VS Code, Jupyter, and command-line execution.

v5 update: uppercase panel letters; Figure 1A uses a clean upper-right boxed callout for corpus size; shortened S1 labels; fixed S1 colorbar ticks; removed S1B y-axis label.

Before running:
  1. Confirm BASE_DIR_DEFAULT.
  2. Confirm INPUT_CSV_DEFAULT or pass --input_csv.
  3. Confirm whether the source label should be DCP or DCTPep.
     This script standardizes DCTPep-like labels to DCP by default.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime
from itertools import combinations
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

try:
    from scipy.stats import ks_2samp
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# =============================================================================
# 1. User-editable paths and manuscript constants
# =============================================================================

BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
INPUT_CSV_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/OncoPep_merged_clean_labeled4.csv")

SOURCE_ORDER_DEFAULT = ("dbAMPseq", "APD3", "CancerPPD2", "DCP")

# Manuscript counts used as validation guards. They do not replace computed values.
EXPECTED_SOURCE_COUNTS_DEFAULT = {
    "dbAMPseq": 43641,
    "APD3": 2646,
    "CancerPPD2": 276,
    "DCP": 4442,
}
EXPECTED_TOTAL_DEFAULT = 51005


# =============================================================================
# 2. Unified PLOS-style palette
# =============================================================================

PLOS_PALETTE = {
    "olive_green": "#9AAA63",
    "soft_coral": "#DD705D",
    "charcoal_brown": "#6A5E61",
    "pale_sand": "#F0CF95",
    "muted_mint": "#A8D3B2",
    "teal_green": "#56A98B",
    "blue_teal": "#1F95B8",
    "warm_brown": "#B67D4E",
    "axis_dark": "#333333",
    "light_gray": "#D9D9D9",
    "background": "#FFFFFF",
    "grid": "#EAEAEA",
    "grid_light": "#F3F3F3",
}

SOURCE_FACE = {
    "dbAMPseq": PLOS_PALETTE["blue_teal"],
    "APD3": PLOS_PALETTE["teal_green"],
    "CancerPPD2": PLOS_PALETTE["warm_brown"],
    "DCP": PLOS_PALETTE["soft_coral"],
}

SOURCE_EDGE = {
    "dbAMPseq": "#176F8A",
    "APD3": "#3E806A",
    "CancerPPD2": "#8F5F37",
    "DCP": "#A84F42",
}

S1_DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "plos_cliffs_delta",
    [PLOS_PALETTE["blue_teal"], PLOS_PALETTE["light_gray"], PLOS_PALETTE["soft_coral"]],
    N=256,
)


# =============================================================================
# 3. Configuration
# =============================================================================

@dataclass
class Step1Config:
    base_dir: Path = BASE_DIR_DEFAULT
    input_csv: Path = INPUT_CSV_DEFAULT
    output_root_name: str = "plos_comp/step_01"

    required_columns: Tuple[str, ...] = (
        "sequence",
        "source_db4",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
    )

    source_aliases: Tuple[str, ...] = (
        "source_db4",
        "source",
        "source_db",
        "dataset",
        "db",
        "dataset_source",
    )

    source_order: Tuple[str, ...] = SOURCE_ORDER_DEFAULT
    expected_source_counts: Mapping[str, int] = field(default_factory=lambda: EXPECTED_SOURCE_COUNTS_DEFAULT)
    expected_total_retained: int = EXPECTED_TOTAL_DEFAULT

    min_len: int = 5
    max_len: int = 60

    # Validation controls.
    fail_on_unknown_sources: bool = True
    validate_expected_counts: bool = True
    strict_expected_counts: bool = False

    # Export.
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True

    # Typography.
    font_size: float = 8.3
    title_size: float = 9.2
    label_size: float = 8.8
    tick_size: float = 7.8
    legend_size: float = 7.5
    panel_letter_size: float = 13.0
    annotation_size: float = 7.6
    count_label_size: float = 8.2

    # Figure sizes in inches.
    fig1_size: Tuple[float, float] = (11.4, 6.8)
    fig_s1_size: Tuple[float, float] = (10.9, 5.0)

    # Plot style.
    axis_linewidth: float = 0.8
    violin_alpha: float = 0.38
    violin_width: float = 0.58
    box_width: float = 0.16
    box_alpha: float = 0.98

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def main_dir(self) -> Path:
        return self.output_root / "main_figure"

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"


# =============================================================================
# 4. General utilities
# =============================================================================

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def ensure_output_tree(cfg: Step1Config) -> None:
    for directory in [cfg.output_root, cfg.main_dir, cfg.supp_dir, cfg.reports_dir]:
        ensure_dir(directory)


def sha256_file(path: Path, chunk_size: int = 2**20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=json_default)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        f.write(text)


def save_figure_all_formats(fig: plt.Figure, basepath_no_ext: Path, cfg: Step1Config) -> Dict[str, str]:
    basepath_no_ext.parent.mkdir(parents=True, exist_ok=True)
    output_paths = {
        "pdf": basepath_no_ext.with_suffix(".pdf"),
        "png": basepath_no_ext.with_suffix(".png"),
        "tiff": basepath_no_ext.with_suffix(".tiff"),
    }
    fig.savefig(output_paths["pdf"], bbox_inches="tight", pad_inches=0.06, transparent=False)
    fig.savefig(output_paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06, transparent=False)
    fig.savefig(output_paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06, transparent=False)
    return {k: str(v) for k, v in output_paths.items()}


def find_input_csv(cfg: Step1Config) -> Path:
    candidates = [
        cfg.input_csv,
        cfg.base_dir / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir.parent / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir / "OncoPep_merged_clean_labeled.csv",
        cfg.base_dir / "OncoPep_merged_clean.csv",
        cfg.base_dir / "PepOnco_merged_clean_labeled4.csv",
        cfg.base_dir / "data" / "OncoPep_merged_clean_labeled4.csv",
        cfg.base_dir / "tables" / "OncoPep_merged_clean_labeled4.csv",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    likely = sorted(cfg.base_dir.glob("**/*OncoPep*clean*.csv"))
    if likely:
        return likely[0]

    raise FileNotFoundError(
        "Could not find the cleaned OncoPep corpus CSV. "
        f"Default attempted path: {cfg.input_csv}. "
        "Edit INPUT_CSV_DEFAULT or pass --input_csv."
    )


def get_script_text() -> Tuple[str, Optional[Path], Optional[str]]:
    """
    Return script text for reproducibility snapshots.

    In normal .py execution, __file__ is available. In pasted Jupyter execution, __file__
    may not exist, so a best-effort source reconstruction is used.
    """
    file_name = globals().get("__file__", None)
    if file_name:
        script_path = Path(file_name).resolve()
        if script_path.exists():
            text = script_path.read_text(encoding="utf-8")
            return text, script_path, sha256_file(script_path)

    try:
        text = inspect.getsource(sys.modules[__name__])
        return text, None, sha256_text(text)
    except Exception:
        text = (
            "# OncoPep Step 1 plotting code snapshot could not be recovered automatically.\n"
            "# If this was run from a notebook cell, export the notebook or save the script manually.\n"
        )
        return text, None, sha256_text(text)


def set_plot_style(cfg: Step1Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": cfg.axis_linewidth,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS_PALETTE["background"],
            "axes.facecolor": PLOS_PALETTE["background"],
            "savefig.facecolor": PLOS_PALETTE["background"],
            "text.color": PLOS_PALETTE["axis_dark"],
            "axes.labelcolor": PLOS_PALETTE["axis_dark"],
            "axes.edgecolor": PLOS_PALETTE["axis_dark"],
            "xtick.color": PLOS_PALETTE["axis_dark"],
            "ytick.color": PLOS_PALETTE["axis_dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, linewidth=0.45, color=PLOS_PALETTE["grid"], zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PLOS_PALETTE["axis_dark"])
        ax.spines[side].set_linewidth(0.75)


def panel_letter(ax: plt.Axes, letter: str, cfg: Step1Config, dx: float = -0.12, dy: float = 1.08) -> None:
    ax.text(
        dx,
        dy,
        str(letter).upper(),
        transform=ax.transAxes,
        fontsize=cfg.panel_letter_size,
        fontweight="bold",
        ha="left",
        va="top",
        color=PLOS_PALETTE["axis_dark"],
    )


def finite_array(x: Sequence[float]) -> np.ndarray:
    arr = np.asarray(x, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_sequence_column(series: pd.Series) -> pd.Series:
    return series.where(series.notna(), "").astype(str).str.strip().str.upper()


def summary_stats(x: Sequence[float]) -> Dict[str, Any]:
    arr = finite_array(x)
    if len(arr) == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "min": np.nan,
            "max": np.nan,
        }
    return {
        "n": int(len(arr)),
        "mean": float(np.mean(arr)),
        "sd": float(np.std(arr, ddof=1)) if len(arr) > 1 else np.nan,
        "median": float(np.median(arr)),
        "q25": float(np.percentile(arr, 25)),
        "q75": float(np.percentile(arr, 75)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
    }


# =============================================================================
# 5. Data preparation and validation
# =============================================================================

AA20_SET = set("ACDEFGHIKLMNPQRSTVWY")


def canonical_only(seq: str) -> bool:
    return isinstance(seq, str) and len(seq) > 0 and set(seq).issubset(AA20_SET)


def max_run_length(seq: str) -> int:
    if not isinstance(seq, str) or len(seq) == 0:
        return 0
    best = cur = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return int(best)


def shannon_entropy(seq: str) -> float:
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    counts = pd.Series(list(seq)).value_counts().to_numpy(dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def dominant_residue_fraction(seq: str) -> float:
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    return float(pd.Series(list(seq)).value_counts().max() / len(seq))


def source_key(label: Any) -> str:
    """
    Robust key for source-label normalization.
    Removes punctuation/spacing and converts to lowercase.
    """
    raw = "" if pd.isna(label) else str(label)
    cleaned = raw.strip().lower()
    for token in [" ", "_", "-", "/", "\\", ".", "(", ")", "[", "]", "{", "}"]:
        cleaned = cleaned.replace(token, "")
    return cleaned


SOURCE_KEY_TO_CANONICAL = {
    "dbampseq": "dbAMPseq",
    "dbamp": "dbAMPseq",
    "apd3": "APD3",
    "apd": "APD3",
    "cancerppd2": "CancerPPD2",
    "cancerppd": "CancerPPD2",
    "dcp": "DCP",
    "dctpep": "DCP",
    "dctpeptide": "DCP",
    "dcpdctpep": "DCP",
    "dctpepdcp": "DCP",
}


def normalize_source_label(label: Any) -> str:
    key = source_key(label)
    return SOURCE_KEY_TO_CANONICAL.get(key, f"UNKNOWN::{str(label).strip()}")


def harmonize_column_names(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    out = df.copy()
    if "source_db4" not in out.columns:
        for candidate in cfg.source_aliases:
            if candidate in out.columns:
                out = out.rename(columns={candidate: "source_db4"})
                break
    return out


def validate_required_columns(df: pd.DataFrame, required: Iterable[str], context: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{context}: missing required columns: {missing}")


def build_source_label_audit(df: pd.DataFrame, source_col: str = "source_db4") -> pd.DataFrame:
    audit = (
        df[source_col]
        .where(df[source_col].notna(), "")
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
        .reset_index()
    )
    audit.columns = ["original_source_label", "n_rows"]
    audit["normalized_source"] = audit["original_source_label"].apply(normalize_source_label)
    audit["status"] = np.where(audit["normalized_source"].str.startswith("UNKNOWN::"), "unknown", "recognized")
    return audit[["original_source_label", "normalized_source", "status", "n_rows"]]


def prepare_dataframe(raw_df: pd.DataFrame, cfg: Step1Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = harmonize_column_names(df, cfg)
    validate_required_columns(df, cfg.required_columns, "Input corpus")

    source_audit = build_source_label_audit(df, "source_db4")
    unknown_audit = source_audit[source_audit["status"] == "unknown"].copy()
    if not unknown_audit.empty and cfg.fail_on_unknown_sources:
        raise ValueError(
            "Unknown source labels detected. Fix source mapping before plotting:\n"
            + unknown_audit.to_string(index=False)
        )

    df["sequence_raw"] = df["sequence"]
    df["sequence"] = normalize_sequence_column(df["sequence"])
    df["source_original"] = df["source_db4"].where(df["source_db4"].notna(), "").astype(str).str.strip()
    df["source"] = df["source_original"].apply(normalize_source_label)
    df.loc[df["source"].str.startswith("UNKNOWN::"), "source"] = "Other"

    for col in ["length", "net_charge_pH7", "hydrophobicity_KD"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["sequence_id"] = np.arange(1, len(df) + 1)
    df["len_from_seq"] = df["sequence"].str.len()
    df["is_canonical"] = df["sequence"].apply(canonical_only)

    # Computed descriptors.
    df["shannon_entropy"] = df["sequence"].apply(shannon_entropy)
    df["max_homopolymer_run"] = df["sequence"].apply(max_run_length)
    df["dominant_residue_fraction"] = df["sequence"].apply(dominant_residue_fraction)

    # Basic final-retained flag for this Step 1 audit.
    df["length_missing"] = df["length"].isna()
    df["charge_missing"] = df["net_charge_pH7"].isna()
    df["hydrophobicity_missing"] = df["hydrophobicity_KD"].isna()
    df["sequence_missing"] = df["sequence"].eq("")

    length_rounded = pd.Series(pd.NA, index=df.index, dtype="Int64")
    length_present = df["length"].notna()
    length_rounded.loc[length_present] = df.loc[length_present, "length"].round().astype("Int64")
    df["length_rounded_int"] = length_rounded

    df["length_mismatch"] = False
    df.loc[length_present, "length_mismatch"] = (
        df.loc[length_present, "length_rounded_int"].astype("Int64")
        != df.loc[length_present, "len_from_seq"].astype("Int64")
    ).fillna(False)

    df["length_out_of_range"] = np.where(
        df["sequence_missing"],
        False,
        (df["len_from_seq"] < cfg.min_len) | (df["len_from_seq"] > cfg.max_len),
    )

    df["exclude_step1"] = (
        df["sequence_missing"]
        | (~df["is_canonical"])
        | df["length_missing"]
        | df["charge_missing"]
        | df["hydrophobicity_missing"]
        | df["length_mismatch"]
        | df["length_out_of_range"]
    )
    df["retained_step1"] = ~df["exclude_step1"]

    df["source_for_plot"] = df["source"]
    if cfg.fail_on_unknown_sources:
        unexpected_after_norm = sorted(set(df["source_for_plot"]) - set(cfg.source_order))
        unexpected_after_norm = [x for x in unexpected_after_norm if x != "Other"]
        if unexpected_after_norm:
            raise ValueError(f"Unexpected normalized sources after mapping: {unexpected_after_norm}")

    return df, source_audit


# =============================================================================
# 6. Step 1 analysis tables
# =============================================================================

def build_source_counts(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    retained = df[df["retained_step1"]].copy()
    counts = (
        retained["source_for_plot"]
        .value_counts()
        .reindex(list(cfg.source_order))
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    counts.columns = ["source", "n_sequences"]
    total = int(counts["n_sequences"].sum())
    counts["percent_total"] = np.where(total > 0, 100 * counts["n_sequences"] / total, np.nan)
    counts["total_retained"] = total
    return counts


def validate_expected_counts(counts_df: pd.DataFrame, cfg: Step1Config) -> Tuple[bool, pd.DataFrame, list[str]]:
    rows = []
    warnings_list: list[str] = []

    observed = counts_df.set_index("source")["n_sequences"].to_dict()
    observed_total = int(counts_df["n_sequences"].sum())
    expected_total = int(cfg.expected_total_retained)

    rows.append(
        {
            "item": "total_retained",
            "observed": observed_total,
            "expected": expected_total,
            "difference": observed_total - expected_total,
            "matches": observed_total == expected_total,
        }
    )

    for source, expected in cfg.expected_source_counts.items():
        obs = int(observed.get(source, 0))
        rows.append(
            {
                "item": f"source::{source}",
                "observed": obs,
                "expected": int(expected),
                "difference": obs - int(expected),
                "matches": obs == int(expected),
            }
        )

    audit = pd.DataFrame(rows)
    ok = bool(audit["matches"].all())

    if cfg.validate_expected_counts and not ok:
        message = (
            "Computed Step 1 counts do not match the manuscript validation counts. "
            "Inspect count_validation_audit.csv before using the figure."
        )
        warnings_list.append(message)
        if cfg.strict_expected_counts:
            raise ValueError(message + "\n" + audit.to_string(index=False))
        warnings.warn(message, RuntimeWarning)

    return ok, audit, warnings_list


def build_descriptor_distribution_table(df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    retained = df[df["retained_step1"] & df["source_for_plot"].isin(cfg.source_order)].copy()
    cols = [
        "sequence_id",
        "source_for_plot",
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "shannon_entropy",
        "dominant_residue_fraction",
        "max_homopolymer_run",
    ]
    out = retained[cols].copy()
    out = out.rename(columns={"source_for_plot": "source"})

    if out.empty:
        raise ValueError("No retained rows available for descriptor plotting after source/filter validation.")

    for source in cfg.source_order:
        if (out["source"] == source).sum() == 0:
            warnings.warn(f"Source {source!r} has zero retained rows in descriptor table.", RuntimeWarning)

    return out


def build_source_summary_table(descriptor_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    descriptors = [
        "length",
        "net_charge_pH7",
        "hydrophobicity_KD",
        "shannon_entropy",
        "dominant_residue_fraction",
        "max_homopolymer_run",
    ]
    rows = []
    for source in cfg.source_order:
        sub = descriptor_df[descriptor_df["source"] == source]
        row = {"source": source, "n_sequences": int(len(sub))}
        for col in descriptors:
            s = summary_stats(sub[col].to_numpy())
            for key, value in s.items():
                row[f"{col}_{key}"] = value
        rows.append(row)
    return pd.DataFrame(rows)


def cliffs_delta_exact(x: Sequence[float], y: Sequence[float]) -> float:
    """
    Exact Cliff's delta using sorted search.

    δ = [#(x > y) - #(x < y)] / (n_x * n_y)

    Ties contribute 0. Complexity is O((n_x + n_y) log n_y), avoiding
    the memory cost of all pairwise comparisons.
    """
    x_arr = finite_array(x)
    y_arr = finite_array(y)
    if len(x_arr) == 0 or len(y_arr) == 0:
        return np.nan

    y_sorted = np.sort(y_arr)
    less_than_x = np.searchsorted(y_sorted, x_arr, side="left")
    less_or_equal_x = np.searchsorted(y_sorted, x_arr, side="right")
    greater_than_x = len(y_sorted) - less_or_equal_x

    gt = np.sum(less_than_x)
    lt = np.sum(greater_than_x)
    return float((gt - lt) / (len(x_arr) * len(y_sorted)))


def ks_test(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float]:
    if not HAS_SCIPY:
        return np.nan, np.nan
    x_arr = finite_array(x)
    y_arr = finite_array(y)
    if len(x_arr) < 10 or len(y_arr) < 10:
        return np.nan, np.nan
    stat, p_value = ks_2samp(x_arr, y_arr)
    return float(stat), float(p_value)


def build_pairwise_shift_table(descriptor_df: pd.DataFrame, cfg: Step1Config) -> pd.DataFrame:
    descriptors = [
        ("length", "Length"),
        ("net_charge_pH7", "Net charge"),
        ("hydrophobicity_KD", "Hydrophobicity"),
        ("shannon_entropy", "Shannon entropy"),
        ("max_homopolymer_run", "Max homopolymer run"),
        ("dominant_residue_fraction", "Dominant residue fraction"),
    ]

    rows = []
    for source_a, source_b in combinations(cfg.source_order, 2):
        sub_a = descriptor_df[descriptor_df["source"] == source_a]
        sub_b = descriptor_df[descriptor_df["source"] == source_b]

        for descriptor_col, descriptor_label in descriptors:
            a_values = sub_a[descriptor_col].to_numpy()
            b_values = sub_b[descriptor_col].to_numpy()
            delta = cliffs_delta_exact(a_values, b_values)
            ks_stat, ks_pvalue = ks_test(a_values, b_values)

            rows.append(
                {
                    "panel": "A",
                    "data_type": "pairwise_cliffs_delta",
                    "source_a": source_a,
                    "source_b": source_b,
                    "source_pair": f"{source_a} vs {source_b}",
                    "descriptor": descriptor_label,
                    "descriptor_column": descriptor_col,
                    "cliffs_delta": delta,
                    "abs_cliffs_delta": abs(delta) if np.isfinite(delta) else np.nan,
                    "ks_stat": ks_stat,
                    "ks_pvalue": ks_pvalue,
                    "x_n_input": int(finite_array(a_values).size),
                    "y_n_input": int(finite_array(b_values).size),
                    "x_n_used": int(finite_array(a_values).size),
                    "y_n_used": int(finite_array(b_values).size),
                    "subsampled": False,
                    "cliffs_delta_method": "exact_sorted_search",
                }
            )

    return pd.DataFrame(rows)


def build_descriptor_shift_summary(pairwise_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        pairwise_df.groupby("descriptor", as_index=False)
        .agg(
            mean_abs_cliffs_delta=("abs_cliffs_delta", "mean"),
            max_abs_cliffs_delta=("abs_cliffs_delta", "max"),
            n_pairwise_comparisons=("source_pair", "nunique"),
        )
        .sort_values("mean_abs_cliffs_delta", ascending=False)
    )
    summary.insert(0, "panel", "B")
    summary["data_type"] = "descriptor_shift_summary"
    return summary


def build_figure1_source_data_all(counts_df: pd.DataFrame, descriptor_df: pd.DataFrame) -> pd.DataFrame:
    panel_a = counts_df.copy()
    panel_a.insert(0, "panel", "A")
    panel_a["data_type"] = "source_counts"

    descriptor_long = descriptor_df.melt(
        id_vars=["sequence_id", "source"],
        value_vars=[
            "length",
            "net_charge_pH7",
            "hydrophobicity_KD",
            "shannon_entropy",
            "dominant_residue_fraction",
        ],
        var_name="descriptor",
        value_name="value",
    )
    descriptor_long.insert(
        0,
        "panel",
        descriptor_long["descriptor"].map(
            {
                "length": "B",
                "net_charge_pH7": "C",
                "hydrophobicity_KD": "D",
                "shannon_entropy": "E",
                "dominant_residue_fraction": "F",
            }
        ),
    )
    descriptor_long["data_type"] = "descriptor_distribution"

    return pd.concat([panel_a, descriptor_long], ignore_index=True, sort=False)


def build_s1_source_data_all(pairwise_df: pd.DataFrame, shift_summary_df: pd.DataFrame) -> pd.DataFrame:
    panel_a = pairwise_df.copy()
    panel_b = shift_summary_df.copy()
    return pd.concat([panel_a, panel_b], ignore_index=True, sort=False)


# =============================================================================
# 7. Plotting helpers
# =============================================================================

def plot_violin_box(
    ax: plt.Axes,
    descriptor_df: pd.DataFrame,
    column: str,
    ylabel: str,
    cfg: Step1Config,
    y_limits: Optional[Tuple[float, float]] = None,
    zero_line: bool = False,
) -> None:
    positions = np.arange(1, len(cfg.source_order) + 1)
    data = [
        finite_array(descriptor_df.loc[descriptor_df["source"] == source, column].to_numpy())
        for source in cfg.source_order
    ]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=cfg.violin_width,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, source in zip(vp["bodies"], cfg.source_order):
        body.set_facecolor(SOURCE_FACE[source])
        body.set_edgecolor(SOURCE_EDGE[source])
        body.set_alpha(cfg.violin_alpha)
        body.set_linewidth(0.9)
        body.set_zorder(3)

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=cfg.box_width,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color=PLOS_PALETTE["axis_dark"], linewidth=1.15),
        whiskerprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        capprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        boxprops=dict(linewidth=0.75),
    )

    for patch, source in zip(bp["boxes"], cfg.source_order):
        patch.set_facecolor("#FFFFFF")
        patch.set_alpha(cfg.box_alpha)
        patch.set_edgecolor(SOURCE_EDGE[source])
        patch.set_zorder(4)

    if zero_line:
        ax.axhline(0, color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75, linestyle="--", zorder=2)

    ax.set_ylabel(ylabel)
    ax.set_xticks(positions)
    ax.set_xticklabels(cfg.source_order, rotation=8, ha="right")

    if y_limits is not None:
        ax.set_ylim(*y_limits)

    beautify_axis(ax, "y")


def source_handles(cfg: Step1Config) -> list[Patch]:
    return [
        Patch(
            facecolor=SOURCE_FACE[source],
            edgecolor=SOURCE_EDGE[source],
            label=source,
            linewidth=0.9,
        )
        for source in cfg.source_order
    ]


def heatmap_annotation_color(value: float) -> str:
    return "#FFFFFF" if np.isfinite(value) and abs(value) >= 0.55 else PLOS_PALETTE["axis_dark"]


# =============================================================================
# 8. Main Figure 1
# =============================================================================

def generate_main_figure1(
    counts_df: pd.DataFrame,
    descriptor_df: pd.DataFrame,
    cfg: Step1Config,
) -> Dict[str, str]:
    total_n = int(counts_df["n_sequences"].sum())

    fig, axes = plt.subplots(2, 3, figsize=cfg.fig1_size)
    fig.subplots_adjust(
        left=0.075,
        right=0.985,
        top=0.925,
        bottom=0.145,
        wspace=0.34,
        hspace=0.40,
    )

    # Panel A: source composition.
    ax = axes[0, 0]
    x = np.arange(len(cfg.source_order))
    counts = counts_df.set_index("source").reindex(cfg.source_order)["n_sequences"].fillna(0).astype(int)
    percentages = counts_df.set_index("source").reindex(cfg.source_order)["percent_total"].fillna(0)

    bars = ax.bar(
        x,
        counts.values,
        width=0.55,
        color=[SOURCE_FACE[source] for source in cfg.source_order],
        edgecolor=[SOURCE_EDGE[source] for source in cfg.source_order],
        linewidth=0.95,
        zorder=3,
    )
    ax.set_yscale("log")
    ax.set_ylabel("Retained sequences (log scale)")
    ax.set_xticks(x)
    ax.set_xticklabels(cfg.source_order, rotation=8, ha="right")
    ax.set_title("Source composition", pad=8)
    beautify_axis(ax, "y")

    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 2.45)

    for bar, source in zip(bars, cfg.source_order):
        count = int(counts.loc[source])
        pct = float(percentages.loc[source])
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            count * 1.10 if count > 0 else 1,
            f"{count:,}\n({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=cfg.count_label_size,
            color=PLOS_PALETTE["axis_dark"],
            linespacing=1.1,
        )

    ax.text(
        0.965,
        0.93,
        f"Source composition\nN = {total_n:,} peptides",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
        bbox=dict(
            boxstyle="round,pad=0.26",
            facecolor="#FFFDF8",
            edgecolor=PLOS_PALETTE["charcoal_brown"],
            linewidth=0.7,
        ),
        zorder=6,
    )
    panel_letter(ax, "a", cfg)

    specs = [
        (axes[0, 1], "length", "Length (aa)", "Length", "b", None, False),
        (axes[0, 2], "net_charge_pH7", "Net charge at pH 7", "Net charge", "c", None, True),
        (axes[1, 0], "hydrophobicity_KD", "Hydrophobicity\n(Kyte–Doolittle)", "Hydrophobicity", "d", None, True),
        (axes[1, 1], "shannon_entropy", "Shannon entropy (bits)", "Shannon entropy", "e", None, False),
        (axes[1, 2], "dominant_residue_fraction", "Dominant residue fraction", "Dominant residue fraction", "f", (0.05, 1.0), False),
    ]

    for ax_i, col, ylabel, title, label, ylim, zero_line in specs:
        plot_violin_box(ax_i, descriptor_df, col, ylabel, cfg, y_limits=ylim, zero_line=zero_line)
        ax_i.set_title(title, pad=8)
        panel_letter(ax_i, label, cfg)

    handles = source_handles(cfg)
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=4,
        frameon=True,
        bbox_to_anchor=(0.5, 0.045),
        fontsize=cfg.legend_size,
        edgecolor=PLOS_PALETTE["light_gray"],
        facecolor=PLOS_PALETTE["background"],
        handlelength=1.5,
        handletextpad=0.5,
        columnspacing=1.25,
        borderpad=0.35,
    )

    base = cfg.main_dir / "Figure_1_redesigned"
    paths = save_figure_all_formats(fig, base, cfg)
    plt.close(fig)
    return paths


# =============================================================================
# 9. Supplementary Figure S1
# =============================================================================

def generate_supplementary_figure_s1(
    pairwise_df: pd.DataFrame,
    shift_summary_df: pd.DataFrame,
    cfg: Step1Config,
) -> Dict[str, str]:
    descriptors = [
        "Length",
        "Net charge",
        "Hydrophobicity",
        "Shannon entropy",
        "Max homopolymer run",
        "Dominant residue fraction",
    ]
    descriptor_display = {
        "Length": "Length",
        "Net charge": "Net charge",
        "Hydrophobicity": "Hydrophobicity",
        "Shannon entropy": "Shannon entropy",
        "Max homopolymer run": "Max run",
        "Dominant residue fraction": "Dominant residue",
    }
    source_pairs = [f"{a} vs {b}" for a, b in combinations(cfg.source_order, 2)]

    heatmap = np.full((len(source_pairs), len(descriptors)), np.nan)
    for i, pair in enumerate(source_pairs):
        for j, desc in enumerate(descriptors):
            values = pairwise_df.loc[
                (pairwise_df["source_pair"] == pair) & (pairwise_df["descriptor"] == desc),
                "cliffs_delta",
            ].to_numpy()
            if len(values):
                heatmap[i, j] = values[0]

    fig = plt.figure(figsize=cfg.fig_s1_size)
    gs = fig.add_gridspec(1, 2, width_ratios=[2.25, 1.05], wspace=0.42)
    ax_hm = fig.add_subplot(gs[0, 0])
    ax_bar = fig.add_subplot(gs[0, 1])
    fig.subplots_adjust(left=0.12, right=0.965, top=0.90, bottom=0.25)

    # Panel A: Cliff's delta heatmap.
    im = ax_hm.imshow(
        heatmap,
        aspect="auto",
        cmap=S1_DIVERGING_CMAP,
        vmin=-1.0,
        vmax=1.0,
        interpolation="nearest",
    )
    ax_hm.set_xticks(np.arange(len(descriptors)))
    ax_hm.set_xticklabels([descriptor_display[d] for d in descriptors], rotation=26, ha="right")
    ax_hm.set_yticks(np.arange(len(source_pairs)))
    ax_hm.set_yticklabels(source_pairs)
    ax_hm.tick_params(length=0)
    ax_hm.set_title("Pairwise source shifts", pad=8)

    for i in range(heatmap.shape[0]):
        for j in range(heatmap.shape[1]):
            value = heatmap[i, j]
            if np.isfinite(value):
                ax_hm.text(
                    j,
                    i,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=6.7,
                    color=heatmap_annotation_color(value),
                )

    for spine in ax_hm.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax_hm, fraction=0.04, pad=0.03)
    cbar.set_label("Cliff’s δ", fontsize=cfg.legend_size)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    cbar.outline.set_linewidth(0.6)
    cbar.outline.set_edgecolor(PLOS_PALETTE["light_gray"])

    ax_hm.text(
        0.0,
        -0.25,
        "Positive δ: descriptor is higher in the first source of the pair; negative δ: lower.",
        transform=ax_hm.transAxes,
        ha="left",
        va="top",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
    )
    panel_letter(ax_hm, "a", cfg, dx=-0.13, dy=1.08)

    # Panel B: ranked mean absolute Cliff's delta.
    summary = shift_summary_df.sort_values("mean_abs_cliffs_delta", ascending=True)
    y = np.arange(len(summary))
    max_x = max(float(summary["mean_abs_cliffs_delta"].max()), 0.05)

    ax_bar.barh(
        y,
        summary["mean_abs_cliffs_delta"],
        color=PLOS_PALETTE["charcoal_brown"],
        edgecolor=PLOS_PALETTE["axis_dark"],
        linewidth=0.55,
        zorder=3,
    )
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels(summary["descriptor"].replace({
        "Max homopolymer run": "Max run",
        "Dominant residue fraction": "Dominant residue",
    }))
    ax_bar.set_ylabel("")
    ax_bar.set_xlabel("Mean |Cliff’s δ|")
    ax_bar.set_title("Descriptor-level shift", pad=8)
    ax_bar.set_xlim(0, max_x * 1.18)
    beautify_axis(ax_bar, "x")

    for yi, value in zip(y, summary["mean_abs_cliffs_delta"]):
        ax_bar.text(
            value + max_x * 0.025,
            yi,
            f"{value:.2f}",
            ha="left",
            va="center",
            fontsize=cfg.annotation_size,
            color=PLOS_PALETTE["axis_dark"],
        )

    panel_letter(ax_bar, "b", cfg, dx=-0.30, dy=1.08)

    base = cfg.supp_dir / "Supplementary_Figure_S1_redesigned"
    paths = save_figure_all_formats(fig, base, cfg)
    plt.close(fig)
    return paths


# =============================================================================
# 10. Output helpers
# =============================================================================

def save_panel_specific_figure1_data(descriptor_df: pd.DataFrame, cfg: Step1Config) -> None:
    mapping = {
        "B": ("length", "Figure_1_panel_b_source_data.csv"),
        "C": ("net_charge_pH7", "Figure_1_panel_c_source_data.csv"),
        "D": ("hydrophobicity_KD", "Figure_1_panel_d_source_data.csv"),
        "E": ("shannon_entropy", "Figure_1_panel_e_source_data.csv"),
        "F": ("dominant_residue_fraction", "Figure_1_panel_f_source_data.csv"),
    }
    for panel, (col, filename) in mapping.items():
        panel_df = descriptor_df[["sequence_id", "source", col]].copy()
        panel_df.insert(0, "panel", panel)
        panel_df = panel_df.rename(columns={col: "value"})
        panel_df["descriptor"] = col
        save_csv(panel_df, cfg.main_dir / filename)


def save_root_level_copies(cfg: Step1Config) -> list[str]:
    """
    Preserve separated directories while also creating root-level convenience copies
    matching common journal/checklist paths.
    """
    if not cfg.create_root_level_copies:
        return []

    copy_pairs = [
        (cfg.main_dir / "Figure_1_redesigned.png", cfg.output_root / "Figure_1_redesigned.png"),
        (cfg.main_dir / "Figure_1_redesigned.pdf", cfg.output_root / "Figure_1_redesigned.pdf"),
        (cfg.main_dir / "Figure_1_redesigned.tiff", cfg.output_root / "Figure_1_redesigned.tiff"),
        (cfg.main_dir / "Figure_1_source_data_all_panels.csv", cfg.output_root / "Figure_1_source_data_all_panels.csv"),
        (cfg.main_dir / "Figure_1_redesign_code.py", cfg.output_root / "Figure_1_redesign_code.py"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.png", cfg.output_root / "Supplementary_Figure_S1_redesigned.png"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.pdf", cfg.output_root / "Supplementary_Figure_S1_redesigned.pdf"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesigned.tiff", cfg.output_root / "Supplementary_Figure_S1_redesigned.tiff"),
        (cfg.supp_dir / "Supplementary_Figure_S1_source_data_all_panels.csv", cfg.output_root / "Supplementary_Figure_S1_source_data_all_panels.csv"),
        (cfg.supp_dir / "Supplementary_Figure_S1_redesign_code.py", cfg.output_root / "Supplementary_Figure_S1_redesign_code.py"),
    ]

    copied = []
    for src, dst in copy_pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def save_code_snapshots(cfg: Step1Config) -> Tuple[str, Optional[str], Optional[str]]:
    script_text, script_path, script_hash = get_script_text()
    save_text(script_text, cfg.main_dir / "Figure_1_redesign_code.py")
    save_text(script_text, cfg.supp_dir / "Supplementary_Figure_S1_redesign_code.py")
    save_text(script_text, cfg.output_root / "Step_01_full_redesign_code.py")
    return script_text, str(script_path) if script_path else None, script_hash


def write_readme(cfg: Step1Config, warnings_list: Sequence[str]) -> Path:
    text = f"""# OncoPep Step 1 PLOS figure package

Generated: {datetime.now().isoformat()}

Primary outputs:

- main_figure/Figure_1_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S1_redesigned.png/.pdf/.tiff

Source-data files:

- main_figure/Figure_1_source_data_all_panels.csv
- supplementary_figures/Supplementary_Figure_S1_source_data_all_panels.csv
- panel-specific source-data CSVs in each figure directory

Notes:

- Figure 1 uses source composition and descriptor distributions.
- Panel labels are uppercase: A, B, C...
- Supplementary Figure S1 uses exact Cliff's delta computed by sorted-search counting.
- KS statistics are saved in source data but not plotted.
- Positive Cliff's delta means the descriptor is higher in the first source of the pair.

Warnings:

{chr(10).join('- ' + w for w in warnings_list) if warnings_list else '- None'}
"""
    path = cfg.output_root / "README_step_01_outputs.txt"
    save_text(text, path)
    return path


# =============================================================================
# 11. Main run function
# =============================================================================

def run_step1(cfg: Step1Config) -> None:
    ensure_output_tree(cfg)
    set_plot_style(cfg)

    warnings_list: list[str] = []

    input_csv = find_input_csv(cfg)
    raw_df = pd.read_csv(input_csv, low_memory=False)
    df, source_audit_df = prepare_dataframe(raw_df, cfg)

    save_csv(source_audit_df, cfg.reports_dir / "source_label_audit.csv")
    save_csv(source_audit_df, cfg.output_root / "source_label_audit.csv")

    counts_df = build_source_counts(df, cfg)
    counts_ok, count_audit_df, count_warnings = validate_expected_counts(counts_df, cfg)
    warnings_list.extend(count_warnings)

    descriptor_df = build_descriptor_distribution_table(df, cfg)
    source_summary_df = build_source_summary_table(descriptor_df, cfg)
    pairwise_df = build_pairwise_shift_table(descriptor_df, cfg)
    shift_summary_df = build_descriptor_shift_summary(pairwise_df)
    figure1_all = build_figure1_source_data_all(counts_df, descriptor_df)
    s1_all = build_s1_source_data_all(pairwise_df, shift_summary_df)

    # Save source-data before plotting.
    save_csv(counts_df, cfg.main_dir / "Figure_1_panel_a_source_data.csv")
    save_csv(counts_df, cfg.main_dir / "Figure_1_panel_a_source_counts.csv")
    save_csv(descriptor_df, cfg.main_dir / "Figure_1_panel_bf_descriptor_distributions.csv")
    save_panel_specific_figure1_data(descriptor_df, cfg)
    save_csv(source_summary_df, cfg.main_dir / "Figure_1_descriptor_summary_by_source.csv")
    save_csv(figure1_all, cfg.main_dir / "Figure_1_source_data_all_panels.csv")

    save_csv(pairwise_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_a_source_data.csv")
    save_csv(pairwise_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_a_effect_sizes.csv")
    save_csv(shift_summary_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_b_source_data.csv")
    save_csv(shift_summary_df, cfg.supp_dir / "Supplementary_Figure_S1_panel_b_descriptor_shift_summary.csv")
    save_csv(s1_all, cfg.supp_dir / "Supplementary_Figure_S1_source_data_all_panels.csv")

    save_csv(count_audit_df, cfg.reports_dir / "count_validation_audit.csv")
    save_csv(count_audit_df, cfg.output_root / "count_validation_audit.csv")

    # Generate figures.
    fig1_paths = generate_main_figure1(counts_df, descriptor_df, cfg)
    s1_paths = generate_supplementary_figure_s1(pairwise_df, shift_summary_df, cfg)

    # Save code snapshots after figure generation.
    _, script_path, script_hash = save_code_snapshots(cfg)

    readme_path = write_readme(cfg, warnings_list)
    root_copies = save_root_level_copies(cfg)

    manifest = {
        "step": "step_01",
        "timestamp": datetime.now().isoformat(),
        "platform": platform.platform(),
        "python_version": sys.version,
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": mpl.__version__,
            "scipy_available": HAS_SCIPY,
        },
        "input_csv": str(input_csv),
        "input_sha256": sha256_file(input_csv),
        "script_path": script_path,
        "script_sha256": script_hash,
        "base_dir": str(cfg.base_dir),
        "output_root": str(cfg.output_root),
        "main_dir": str(cfg.main_dir),
        "supp_dir": str(cfg.supp_dir),
        "source_order": list(cfg.source_order),
        "total_retained_sequences": int(counts_df["n_sequences"].sum()),
        "source_counts": counts_df.to_dict(orient="records"),
        "expected_count_validation": {
            "enabled": cfg.validate_expected_counts,
            "strict": cfg.strict_expected_counts,
            "passed": counts_ok,
            "audit_file": str(cfg.output_root / "count_validation_audit.csv"),
        },
        "source_label_audit_file": str(cfg.output_root / "source_label_audit.csv"),
        "cliffs_delta": {
            "method": "exact_sorted_search",
            "subsampled": False,
        },
        "outputs": {
            "main_figure": fig1_paths,
            "supplementary_figure_s1": s1_paths,
            "root_level_copies": root_copies,
            "readme": str(readme_path),
        },
        "warnings": warnings_list,
        "config": asdict(cfg),
    }
    save_json(manifest, cfg.output_root / "step_01_manifest.json")
    save_json(manifest, cfg.reports_dir / "step_01_manifest.json")

    print("\n✅ Step 1 PLOS figure package generated successfully.")
    print(f"Input CSV: {input_csv}")
    print(f"Main figure directory: {cfg.main_dir}")
    print(f"Supplementary figure directory: {cfg.supp_dir}")
    print(f"Root output directory: {cfg.output_root}")

    print("\nSaved main figure files:")
    for path in fig1_paths.values():
        print(f"  - {path}")

    print("\nSaved Supplementary Figure S1 files:")
    for path in s1_paths.values():
        print(f"  - {path}")

    print("\nKey reports:")
    print(f"  - {cfg.output_root / 'step_01_manifest.json'}")
    print(f"  - {cfg.output_root / 'source_label_audit.csv'}")
    print(f"  - {cfg.output_root / 'count_validation_audit.csv'}")
    print(f"  - {readme_path}")

    if warnings_list:
        print("\n⚠️ Warnings:")
        for warning in warnings_list:
            print(f"  - {warning}")


# =============================================================================
# 12. Notebook and CLI entry points
# =============================================================================

def main_notebook(
    base_dir: str | Path = BASE_DIR_DEFAULT,
    input_csv: str | Path = INPUT_CSV_DEFAULT,
    min_len: int = 5,
    max_len: int = 60,
    strict_expected_counts: bool = False,
    fail_on_unknown_sources: bool = True,
) -> None:
    cfg = Step1Config(
        base_dir=Path(base_dir),
        input_csv=Path(input_csv),
        min_len=min_len,
        max_len=max_len,
        strict_expected_counts=strict_expected_counts,
        fail_on_unknown_sources=fail_on_unknown_sources,
    )
    run_step1(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    """
    Parse command-line arguments safely.

    Jupyter/IPython kernels inject extra arguments such as:
        --f=/run/user/.../kernel-xxxx.json

    Using parse_known_args prevents those kernel-specific arguments from
    crashing the script when the code is pasted into a notebook cell.
    """
    parser = argparse.ArgumentParser(
        description="Generate OncoPep Step 1 PLOS Computational Biology figures."
    )
    parser.add_argument(
        "--base_dir",
        type=str,
        default=str(BASE_DIR_DEFAULT),
        help="Base project directory containing the PLOS folder.",
    )
    parser.add_argument(
        "--input_csv",
        type=str,
        default=str(INPUT_CSV_DEFAULT),
        help="Path to the cleaned OncoPep corpus CSV.",
    )
    parser.add_argument("--min_len", type=int, default=5, help="Minimum retained peptide length.")
    parser.add_argument("--max_len", type=int, default=60, help="Maximum retained peptide length.")
    parser.add_argument(
        "--strict_counts",
        action="store_true",
        help="Raise an error if computed counts differ from manuscript validation counts.",
    )
    parser.add_argument(
        "--allow_unknown_sources",
        action="store_true",
        help="Do not fail on unknown source labels. Unknown labels are excluded from the four-source plots.",
    )
    parser.add_argument(
        "--no_root_copies",
        action="store_true",
        help="Do not create root-level convenience copies of final outputs.",
    )

    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"ℹ️ Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    cfg = Step1Config(
        base_dir=Path(args.base_dir),
        input_csv=Path(args.input_csv),
        min_len=args.min_len,
        max_len=args.max_len,
        strict_expected_counts=args.strict_counts,
        fail_on_unknown_sources=not args.allow_unknown_sources,
        create_root_level_copies=not args.no_root_copies,
    )
    run_step1(cfg)

In [ ]:
from pathlib import Path

search_roots = [
    Path("/home/data3/Moe"),
    Path("/home/data3/mohamed"),
    Path("/home/data3"),
]

target_patterns = [
    "step2_split_summary.csv",
    "step2_leakage_audits.csv",
    "step2_nearest_neighbor_audit.csv",
    "step2_threshold_sensitivity.csv",
    "step2_unique_sequences_with_splits.csv",
    "step2_row_level_with_splits.csv",
    "*step2*",
    "*Step2*",
    "*split*summary*.csv",
    "*leakage*audit*.csv",
]

for root in search_roots:
    print(f"\nSearching in: {root}")
    if not root.exists():
        print("  root not found")
        continue

    found_any = False
    for pattern in target_patterns:
        hits = list(root.rglob(pattern))
        if hits:
            found_any = True
            print(f"\nPattern: {pattern}")
            for h in hits[:30]:
                print(" ", h)
            if len(hits) > 30:
                print(f"  ... {len(hits)-30} more")
    if not found_any:
        print("  no Step 2-like files found")

In [ ]:
from pathlib import Path
import pandas as pd

step2_dir = Path("/home/data3/Moe/nature_computational_peponco/step2")

files = {
    "split_summary": step2_dir / "tables/step2_split_summary.csv",
    "leakage_audits": step2_dir / "tables/step2_leakage_audits.csv",
    "nearest_neighbor": step2_dir / "tables/step2_nearest_neighbor_audit.csv",
    "threshold_sensitivity": step2_dir / "tables/step2_threshold_sensitivity.csv",
    "unique_sequences": step2_dir / "splits/step2_unique_sequences_with_splits.csv",
    "row_level": step2_dir / "splits/step2_row_level_with_splits.csv",
}

print("=== File existence check ===")
for name, path in files.items():
    print(f"{name:25s}: {path.exists()}  {path}")

print("\n=== Split summary ===")
split_summary = pd.read_csv(files["split_summary"])
print(split_summary)

print("\n=== Leakage audits ===")
leakage = pd.read_csv(files["leakage_audits"])
print(leakage)

print("\n=== Threshold sensitivity ===")
threshold = pd.read_csv(files["threshold_sensitivity"])
print(threshold)

print("\n=== Nearest-neighbor audit summary ===")
nn = pd.read_csv(files["nearest_neighbor"])
print(nn["nearest_train_jaccard"].describe())

print("\n=== Unique sequence split counts ===")
unique_df = pd.read_csv(files["unique_sequences"])
print(unique_df["assigned_split"].value_counts())

print("\n=== Row-level split counts ===")
row_df = pd.read_csv(files["row_level"])
print(row_df["assigned_split"].value_counts())

In [ ]:
from pathlib import Path
import pandas as pd

step2_dir = Path("/home/data3/Moe/nature_computational_peponco/step2")

split_summary = pd.read_csv(step2_dir / "tables/step2_split_summary.csv")
leakage = pd.read_csv(step2_dir / "tables/step2_leakage_audits.csv")
threshold = pd.read_csv(step2_dir / "tables/step2_threshold_sensitivity.csv")
nn = pd.read_csv(step2_dir / "tables/step2_nearest_neighbor_audit.csv", low_memory=False)

print("=== Split summary ===")
print(split_summary.to_string(index=False))

print("\n=== Exact overlap audit ===")
exact = leakage[leakage["audit_type"] == "exact_sequence_overlap"]
print(exact[["split_a", "split_b", "n_overlap"]].to_string(index=False))

print("\n=== Cluster overlap audit ===")
cluster = leakage[leakage["audit_type"] == "cluster_overlap"]
print(cluster[["split_a", "split_b", "n_overlap"]].to_string(index=False))

print("\n=== Sampled cross-split Jaccard audit ===")
sampled = leakage[leakage["audit_type"] == "sampled_cross_split_jaccard"]
cols = ["split_a", "split_b", "n_sampled_pairs", "max_jaccard", "mean_jaccard", "threshold"]
print(sampled[cols].to_string(index=False))

print("\n=== Threshold sensitivity ===")
print(threshold.to_string(index=False))

print("\n=== Nearest-train Jaccard summary ===")
print(nn["nearest_train_jaccard"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

print("\n=== Nearest-train Jaccard by split ===")
print(
    nn.groupby("query_split")["nearest_train_jaccard"]
      .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
      .to_string()
)

In [ ]:
from pathlib import Path
import pandas as pd

paths = [
    Path("/home/data3/Moe/nature_computational_peponco/step2/tables/step2_split_summary.csv"),
    Path("/home/data3/Moe/nature_computational_peponco/step1/tables/nature_computational2/step2/tables/step2_split_summary.csv"),
]

for p in paths:
    print("\nPATH:", p)
    print("exists:", p.exists())
    if p.exists():
        print(pd.read_csv(p).to_string(index=False))

OncoPep Step 2 — PLOS Computational Biology supplementary-figure redesign.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 2 — PLOS Computational Biology supplementary-figure redesign.

Generates a supplementary-only Step 2 package from recovered Step 2 split/audit
outputs:

  Supplementary Figure S2 — similarity-aware partitioning and split-leakage audit
  Supplementary Figure S3 — nearest-train similarity and threshold sensitivity
  Supplementary Figure S4 — descriptor preservation across partitions

This script does NOT rerun split construction. Main Figure 2 is intentionally
reserved for the OncoPep architecture and mathematical workflow.

Notebook:
    results = main_notebook(
        source_step2_dir="/home/data3/Moe/nature_computational_peponco/step2",
        base_dir="/home/data3/Moe/nature_computational_peponco/PLOS",
        output_root_name="plos_comp/step_02",
        main_jaccard_threshold=0.55,
    )

Terminal:
    python OncoPep_step_02_PLOS_redesign_code_v2.py
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


# =============================================================================
# Configuration
# =============================================================================

SOURCE_STEP2_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/step2")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_02"

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
PAIR_ORDER = (("train", "val"), ("train", "test"), ("val", "test"))
PAIR_DISPLAY = {
    ("train", "val"): "Train–validation",
    ("train", "test"): "Train–test",
    ("val", "test"): "Validation–test",
}
EXPECTED_SPLIT_COUNTS_DEFAULT = {"train": 24485, "val": 13260, "test": 13260}
EXPECTED_TOTAL_DEFAULT = sum(EXPECTED_SPLIT_COUNTS_DEFAULT.values())

PLOS_PALETTE = {
    "olive_green": "#9AAA63",
    "soft_coral": "#DD705D",
    "charcoal_brown": "#6A5E61",
    "pale_sand": "#F0CF95",
    "muted_mint": "#A8D3B2",
    "teal_green": "#56A98B",
    "blue_teal": "#1F95B8",
    "warm_brown": "#B67D4E",
    "axis_dark": "#333333",
    "light_gray": "#D9D9D9",
    "grid": "#EAEAEA",
    "background": "#FFFFFF",
}

SPLIT_FACE = {
    "train": PLOS_PALETTE["blue_teal"],
    "val": PLOS_PALETTE["muted_mint"],
    "test": PLOS_PALETTE["soft_coral"],
}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}
SPLIT_HATCH = {"train": "", "val": "//", "test": "\\\\"}

PAIR_FACE = {
    ("train", "val"): PLOS_PALETTE["blue_teal"],
    ("train", "test"): PLOS_PALETTE["warm_brown"],
    ("val", "test"): PLOS_PALETTE["soft_coral"],
}
PAIR_EDGE = {
    ("train", "val"): "#176F8A",
    ("train", "test"): "#8F5F37",
    ("val", "test"): "#A84F42",
}
OVERLAP_FACE = {
    "exact_sequence_overlap": PLOS_PALETTE["charcoal_brown"],
    "cluster_overlap": PLOS_PALETTE["warm_brown"],
}
OVERLAP_EDGE = {"exact_sequence_overlap": "#4F474A", "cluster_overlap": "#8F5F37"}


@dataclass
class Config:
    source_step2_dir: Path = SOURCE_STEP2_DIR_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT
    main_jaccard_threshold: float = 0.55
    expected_split_counts: Dict[str, int] = field(default_factory=lambda: EXPECTED_SPLIT_COUNTS_DEFAULT.copy())
    expected_total_sequences: int = EXPECTED_TOTAL_DEFAULT
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    git_commit: Optional[str] = None

    # Typography
    font_size: float = 8.3
    axis_label_size: float = 8.9
    tick_label_size: float = 7.9
    title_size: float = 9.3
    legend_size: float = 7.8
    annotation_size: float = 7.7
    panel_letter_size: float = 13.2

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def tables_dir(self) -> Path:
        return self.output_root / "tables"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_note_dir(self) -> Path:
        return self.output_root / "main_figure_note"


# =============================================================================
# Utility functions
# =============================================================================

def ensure_output_tree(cfg: Config) -> None:
    for p in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.tables_dir, cfg.reports_dir, cfg.code_dir, cfg.main_note_dir]:
        p.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"{type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=json_default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def finite_array(values: Any) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_split(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_col(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    if required:
        raise ValueError(f"Could not resolve column. Candidates={list(candidates)}; available={list(df.columns)}")
    return None


def assert_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns {missing}; available={list(df.columns)}")


def validate_splits_present(series: pd.Series, context: str) -> None:
    observed = set(normalize_split(series).dropna().unique())
    missing = [s for s in SPLIT_ORDER if s not in observed]
    if missing:
        raise ValueError(f"{context}: missing split labels {missing}; observed={sorted(observed)}")


def metric_definition(metric_name: str) -> str:
    definitions = {
        "n_unique_sequences": "Number of unique peptide sequences assigned to each split.",
        "n_clusters": "Number of similarity-connected components assigned to each split.",
        "max_sampled_cross_split_3mer_jaccard": "Maximum sampled cross-split 3-mer set Jaccard similarity for a split pair.",
        "exact_sequence_overlap": "Number of identical peptide sequences shared between split pairs.",
        "cluster_overlap": "Number of similarity clusters shared between split pairs.",
        "nearest_train_3mer_jaccard": "3-mer set Jaccard similarity between a held-out peptide and its nearest training peptide.",
        "threshold_exceedance_fraction": "Fraction of held-out peptides with nearest-train Jaccard greater than or equal to a threshold.",
        "retained_similarity_edges": "Number of similarity-graph edges retained at a given Jaccard threshold.",
        "split_size_stability": "Number of unique sequences assigned to each split under threshold-sensitivity reruns.",
        "descriptor_distribution": "Sequence-level descriptor distribution within a split.",
    }
    return definitions.get(metric_name, "")


# =============================================================================
# Plot helpers
# =============================================================================

def set_style(cfg: Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_label_size,
            "ytick.labelsize": cfg.tick_label_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": 0.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS_PALETTE["background"],
            "axes.facecolor": PLOS_PALETTE["background"],
            "savefig.facecolor": PLOS_PALETTE["background"],
            "text.color": PLOS_PALETTE["axis_dark"],
            "axes.labelcolor": PLOS_PALETTE["axis_dark"],
            "axes.edgecolor": PLOS_PALETTE["axis_dark"],
            "xtick.color": PLOS_PALETTE["axis_dark"],
            "ytick.color": PLOS_PALETTE["axis_dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS_PALETTE["grid"], linewidth=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PLOS_PALETTE["axis_dark"])
        ax.spines[side].set_linewidth(0.75)


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.09) -> None:
    ax.text(dx, dy, str(letter).upper(), transform=ax.transAxes, fontsize=cfg.panel_letter_size,
            fontweight="bold", ha="left", va="top", color=PLOS_PALETTE["axis_dark"])


def save_figure(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {"png": base.with_suffix(".png"), "pdf": base.with_suffix(".pdf"), "tiff": base.with_suffix(".tiff")}
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


def split_legend_handles() -> list[Patch]:
    return [
        Patch(facecolor=SPLIT_FACE[s], edgecolor=SPLIT_EDGE[s], hatch=SPLIT_HATCH[s], label=SPLIT_DISPLAY[s], linewidth=0.8)
        for s in SPLIT_ORDER
    ]


# =============================================================================
# Input loading and validation
# =============================================================================

def locate_input_files(cfg: Config) -> Dict[str, Path]:
    root = cfg.source_step2_dir
    paths = {
        "split_summary": root / "tables" / "step2_split_summary.csv",
        "leakage": root / "tables" / "step2_leakage_audits.csv",
        "nearest": root / "tables" / "step2_nearest_neighbor_audit.csv",
        "threshold": root / "tables" / "step2_threshold_sensitivity.csv",
        "unique": root / "splits" / "step2_unique_sequences_with_splits.csv",
        "row": root / "splits" / "step2_row_level_with_splits.csv",
    }
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required Step 2 input files:\n" + "\n".join(missing))
    return paths


def load_inputs(cfg: Config) -> Dict[str, Any]:
    paths = locate_input_files(cfg)
    inputs: Dict[str, Any] = {
        "split_summary": pd.read_csv(paths["split_summary"]),
        "leakage": pd.read_csv(paths["leakage"]),
        "nearest": pd.read_csv(paths["nearest"], low_memory=False),
        "threshold": pd.read_csv(paths["threshold"]),
        "unique": pd.read_csv(paths["unique"], low_memory=False),
        "row": pd.read_csv(paths["row"], low_memory=False),
        "_paths": paths,
    }
    validate_and_normalize_inputs(inputs)
    return inputs


def validate_and_normalize_inputs(inputs: Dict[str, Any]) -> None:
    split_summary = inputs["split_summary"]
    leakage = inputs["leakage"]
    nearest = inputs["nearest"]
    threshold = inputs["threshold"]
    unique = inputs["unique"]
    row = inputs["row"]

    assert_columns(split_summary, ["split"], "step2_split_summary.csv")
    split_count_col = resolve_col(split_summary, ["n_unique_sequences", "n_sequences", "n_rows"])
    cluster_col = resolve_col(split_summary, ["n_clusters"], required=False)
    split_summary["split"] = normalize_split(split_summary["split"])
    validate_splits_present(split_summary["split"], "split summary")
    split_summary[split_count_col] = pd.to_numeric(split_summary[split_count_col], errors="coerce")
    if cluster_col:
        split_summary[cluster_col] = pd.to_numeric(split_summary[cluster_col], errors="coerce")

    assert_columns(leakage, ["audit_type", "split_a", "split_b"], "step2_leakage_audits.csv")
    leakage["split_a"] = normalize_split(leakage["split_a"])
    leakage["split_b"] = normalize_split(leakage["split_b"])
    expected_audits = {"exact_sequence_overlap", "cluster_overlap", "sampled_cross_split_jaccard"}
    observed_audits = set(leakage["audit_type"].astype(str))
    missing = expected_audits - observed_audits
    if missing:
        raise ValueError(f"Missing leakage audit_type values: {sorted(missing)}; observed={sorted(observed_audits)}")
    for col in ["n_overlap", "n_sampled_pairs", "max_jaccard", "mean_jaccard", "threshold"]:
        if col in leakage.columns:
            leakage[col] = pd.to_numeric(leakage[col], errors="coerce")

    nearest_split_col = resolve_col(nearest, ["query_split", "assigned_split", "split"])
    nearest_jaccard_col = resolve_col(nearest, ["nearest_train_jaccard", "nearest_train_similarity", "jaccard"])
    nearest[nearest_split_col] = normalize_split(nearest[nearest_split_col])
    nearest[nearest_jaccard_col] = pd.to_numeric(nearest[nearest_jaccard_col], errors="coerce")
    if nearest_split_col != "query_split":
        nearest.rename(columns={nearest_split_col: "query_split"}, inplace=True)
    if nearest_jaccard_col != "nearest_train_jaccard":
        nearest.rename(columns={nearest_jaccard_col: "nearest_train_jaccard"}, inplace=True)
    if not {"val", "test"}.issubset(set(nearest["query_split"].unique())):
        raise ValueError("Nearest-neighbor audit must include validation and test query_split values.")

    assert_columns(threshold, ["threshold", "n_edges"], "step2_threshold_sensitivity.csv")
    threshold["threshold"] = pd.to_numeric(threshold["threshold"], errors="coerce")
    threshold["n_edges"] = pd.to_numeric(threshold["n_edges"], errors="coerce")
    missing_size_cols = [f"{s}_n_sequences" for s in SPLIT_ORDER if f"{s}_n_sequences" not in threshold.columns]
    if missing_size_cols:
        raise ValueError(f"Threshold sensitivity table missing columns: {missing_size_cols}")
    for col in [f"{s}_n_sequences" for s in SPLIT_ORDER]:
        threshold[col] = pd.to_numeric(threshold[col], errors="coerce")

    for name, df in [("unique", unique), ("row", row)]:
        split_col = resolve_col(df, ["assigned_split", "split"])
        df[split_col] = normalize_split(df[split_col])
        if split_col != "assigned_split":
            df.rename(columns={split_col: "assigned_split"}, inplace=True)
        validate_splits_present(df["assigned_split"], f"{name} split file")

    descriptor_errors = []
    for name, df in [("unique", unique), ("row", row)]:
        try:
            resolve_col(df, ["length", "sequence_length"])
            resolve_col(df, ["net_charge_pH7", "net_charge", "charge"])
            resolve_col(df, ["hydrophobicity_KD", "hydrophobicity", "mean_hydropathy"])
            resolve_col(df, ["shannon_entropy", "entropy", "sequence_entropy"])
            break
        except Exception as exc:
            descriptor_errors.append(f"{name}: {exc}")
    else:
        raise ValueError("Could not resolve descriptor columns:\n" + "\n".join(descriptor_errors))


# =============================================================================
# Source-data builders
# =============================================================================

def complete_pair_table(df: pd.DataFrame, value_cols: Sequence[str]) -> pd.DataFrame:
    base = pd.DataFrame({
        "split_a": [a for a, _ in PAIR_ORDER],
        "split_b": [b for _, b in PAIR_ORDER],
        "pair_display": [PAIR_DISPLAY[p] for p in PAIR_ORDER],
    })
    if df.empty:
        out = base.copy()
    else:
        keep = ["split_a", "split_b"] + [c for c in value_cols if c in df.columns]
        out = base.merge(df[keep], on=["split_a", "split_b"], how="left")
    for col in value_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0)
    return out


def leakage_subset(leakage: pd.DataFrame, audit_type: str, value_cols: Sequence[str]) -> pd.DataFrame:
    sub = leakage.loc[leakage["audit_type"].astype(str) == audit_type].copy()
    if not sub.empty:
        sub["split_a"] = normalize_split(sub["split_a"])
        sub["split_b"] = normalize_split(sub["split_b"])
    return complete_pair_table(sub, value_cols)


def build_s2_source_data(inputs: Dict[str, Any]) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    split_summary = inputs["split_summary"].copy()
    split_count_col = resolve_col(split_summary, ["n_unique_sequences", "n_sequences", "n_rows"])
    cluster_col = resolve_col(split_summary, ["n_clusters"], required=False)

    split = pd.DataFrame({"split": list(SPLIT_ORDER)}).merge(split_summary, on="split", how="left")
    split.rename(columns={split_count_col: "n_unique_sequences"}, inplace=True)
    if cluster_col and cluster_col != "n_clusters":
        split.rename(columns={cluster_col: "n_clusters"}, inplace=True)
    if "n_clusters" not in split.columns:
        split["n_clusters"] = np.nan
    split["split_display"] = split["split"].map(SPLIT_DISPLAY)
    split["metric_name"] = "n_unique_sequences"
    split["metric_definition"] = metric_definition("n_unique_sequences")
    split["units"] = "unique sequences"
    split["panel"] = "A"
    split["data_type"] = "split_unique_sequence_counts"

    sampled = leakage_subset(inputs["leakage"], "sampled_cross_split_jaccard",
                             ["n_sampled_pairs", "max_jaccard", "mean_jaccard", "threshold"])
    sampled["metric_name"] = "max_sampled_cross_split_3mer_jaccard"
    sampled["metric_definition"] = metric_definition("max_sampled_cross_split_3mer_jaccard")
    sampled["units"] = "3-mer Jaccard similarity"
    sampled["panel"] = "B"
    sampled["data_type"] = "sampled_cross_split_jaccard"

    exact = leakage_subset(inputs["leakage"], "exact_sequence_overlap", ["n_overlap"])
    exact["overlap_type"] = "exact_sequence_overlap"
    exact["overlap_type_display"] = "Exact sequence"
    exact["metric_name"] = "exact_sequence_overlap"
    exact["metric_definition"] = metric_definition("exact_sequence_overlap")
    exact["units"] = "overlap count"

    cluster = leakage_subset(inputs["leakage"], "cluster_overlap", ["n_overlap"])
    cluster["overlap_type"] = "cluster_overlap"
    cluster["overlap_type_display"] = "Similarity cluster"
    cluster["metric_name"] = "cluster_overlap"
    cluster["metric_definition"] = metric_definition("cluster_overlap")
    cluster["units"] = "overlap count"

    overlap = pd.concat([exact, cluster], ignore_index=True)
    overlap["panel"] = "C"
    overlap["data_type"] = "overlap_audit"

    cluster_counts = split[["split", "split_display", "n_clusters"]].copy()
    cluster_counts["metric_name"] = "n_clusters"
    cluster_counts["metric_definition"] = metric_definition("n_clusters")
    cluster_counts["units"] = "similarity clusters"
    cluster_counts["panel"] = "D"
    cluster_counts["data_type"] = "similarity_cluster_counts"

    panels = {"A": split, "B": sampled, "C": overlap, "D": cluster_counts}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def prepare_nearest_df(nearest: pd.DataFrame) -> pd.DataFrame:
    df = nearest.copy()
    df["query_split"] = normalize_split(df["query_split"])
    df["nearest_train_jaccard"] = pd.to_numeric(df["nearest_train_jaccard"], errors="coerce")
    df = df[df["query_split"].isin(["val", "test"])].copy()
    df["query_split_display"] = df["query_split"].map(SPLIT_DISPLAY)
    df["metric_name"] = "nearest_train_3mer_jaccard"
    df["metric_definition"] = metric_definition("nearest_train_3mer_jaccard")
    df["units"] = "3-mer Jaccard similarity"
    return df


def ecdf_dataframe(nn: pd.DataFrame) -> pd.DataFrame:
    records = []
    for split in ["val", "test"]:
        vals = np.sort(finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"]))
        n = len(vals)
        for i, v in enumerate(vals, start=1):
            records.append({
                "query_split": split,
                "query_split_display": SPLIT_DISPLAY[split],
                "nearest_train_jaccard": float(v),
                "ecdf": float(i / n) if n else np.nan,
                "n_sequences": int(n),
                "metric_name": "nearest_train_3mer_jaccard_ecdf",
                "metric_definition": "Empirical cumulative distribution of nearest-train 3-mer Jaccard similarity.",
                "units": "fraction of held-out sequences",
            })
    return pd.DataFrame(records)


def nearest_quantiles(nn: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split in ["val", "test"]:
        vals = finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"])
        for name, percentile in [("median", 50), ("p90", 90), ("p95", 95), ("p99", 99)]:
            rows.append({
                "query_split": split,
                "query_split_display": SPLIT_DISPLAY[split],
                "quantile": name,
                "percentile": percentile,
                "nearest_train_jaccard": float(np.percentile(vals, percentile)) if len(vals) else np.nan,
                "n_sequences": int(len(vals)),
            })
    return pd.DataFrame(rows)


def threshold_exceedance(nn: pd.DataFrame, thresholds: Sequence[float] = (0.10, 0.20, 0.30, 0.50, 0.55)) -> pd.DataFrame:
    rows = []
    for split in ["val", "test"]:
        vals = finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"])
        for thr in thresholds:
            rows.append({
                "query_split": split,
                "query_split_display": SPLIT_DISPLAY[split],
                "threshold": float(thr),
                "n_sequences": int(len(vals)),
                "n_exceeding": int(np.sum(vals >= thr)) if len(vals) else 0,
                "fraction_exceeding": float(np.mean(vals >= thr)) if len(vals) else np.nan,
                "metric_name": "threshold_exceedance_fraction",
                "metric_definition": metric_definition("threshold_exceedance_fraction"),
                "units": "fraction",
            })
    return pd.DataFrame(rows)


def build_s3_source_data(inputs: Dict[str, Any]) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    nn = prepare_nearest_df(inputs["nearest"])
    ecdf = ecdf_dataframe(nn)
    ecdf["panel"] = "A"
    ecdf["data_type"] = "nearest_train_jaccard_ecdf"

    quantiles = nearest_quantiles(nn)
    quantiles["panel"] = "A"
    quantiles["data_type"] = "nearest_train_jaccard_quantiles"

    exceed = threshold_exceedance(nn)
    exceed["panel"] = "B"
    exceed["data_type"] = "nearest_train_threshold_exceedance"

    threshold = inputs["threshold"].copy()
    edges = threshold[["threshold", "n_edges"]].copy()
    edges["metric_name"] = "retained_similarity_edges"
    edges["metric_definition"] = metric_definition("retained_similarity_edges")
    edges["units"] = "edges"
    edges["panel"] = "C"
    edges["data_type"] = "retained_similarity_edges"

    size_cols = [f"{s}_n_sequences" for s in SPLIT_ORDER]
    sizes = threshold[["threshold"] + size_cols].melt(id_vars="threshold", var_name="split_column", value_name="n_unique_sequences")
    sizes["split"] = sizes["split_column"].str.replace("_n_sequences", "", regex=False)
    sizes["split_display"] = sizes["split"].map(SPLIT_DISPLAY)
    sizes["metric_name"] = "split_size_stability"
    sizes["metric_definition"] = metric_definition("split_size_stability")
    sizes["units"] = "unique sequences"
    sizes["panel"] = "D"
    sizes["data_type"] = "split_size_stability"

    panels = {"A": ecdf, "A_quantiles": quantiles, "B": exceed, "C": edges, "D": sizes}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def descriptor_dataframe(inputs: Dict[str, Any]) -> pd.DataFrame:
    errors = []
    for name in ["unique", "row"]:
        df = inputs[name]
        try:
            length_col = resolve_col(df, ["length", "sequence_length"])
            charge_col = resolve_col(df, ["net_charge_pH7", "net_charge", "charge"])
            hydro_col = resolve_col(df, ["hydrophobicity_KD", "hydrophobicity", "mean_hydropathy"])
            entropy_col = resolve_col(df, ["shannon_entropy", "entropy", "sequence_entropy"])
            split_col = resolve_col(df, ["assigned_split", "split"])
            out = df[[split_col, length_col, charge_col, hydro_col, entropy_col]].copy()
            out.columns = ["assigned_split", "length", "net_charge_pH7", "hydrophobicity_KD", "shannon_entropy"]
            out["assigned_split"] = normalize_split(out["assigned_split"])
            for col in ["length", "net_charge_pH7", "hydrophobicity_KD", "shannon_entropy"]:
                out[col] = pd.to_numeric(out[col], errors="coerce")
            out = out[out["assigned_split"].isin(SPLIT_ORDER)].copy()
            if len(out) == 0:
                raise ValueError(f"{name} descriptor dataframe is empty after filtering.")
            return out
        except Exception as exc:
            errors.append(f"{name}: {exc}")
    raise ValueError("Failed to construct descriptor dataframe:\n" + "\n".join(errors))


def build_s4_source_data(inputs: Dict[str, Any]) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    desc = descriptor_dataframe(inputs)
    specs = {
        "A": ("length", "Length", "aa", "Peptide sequence length."),
        "B": ("net_charge_pH7", "Net charge at pH 7", "charge units", "Estimated peptide net charge at pH 7."),
        "C": ("hydrophobicity_KD", "Kyte–Doolittle hydrophobicity", "mean KD score", "Mean Kyte–Doolittle hydrophobicity."),
        "D": ("shannon_entropy", "Shannon entropy", "bits", "Shannon entropy of amino-acid composition."),
    }
    panels = {}
    all_rows = []
    for panel, (col, label, units, definition) in specs.items():
        panel_df = desc[["assigned_split", col]].copy()
        panel_df.rename(columns={col: "value"}, inplace=True)
        panel_df["split_display"] = panel_df["assigned_split"].map(SPLIT_DISPLAY)
        panel_df["descriptor_column"] = col
        panel_df["descriptor"] = label
        panel_df["metric_name"] = "descriptor_distribution"
        panel_df["metric_definition"] = definition
        panel_df["units"] = units
        panel_df["panel"] = panel
        panel_df["data_type"] = "split_descriptor_distribution"
        panels[panel] = panel_df
        all_rows.append(panel_df)
    return panels, pd.concat(all_rows, ignore_index=True, sort=False)


def save_source_data(cfg: Config, s2, s2_all, s3, s3_all, s4, s4_all) -> None:
    save_csv(s2_all, cfg.source_data_dir / "Supplementary_Figure_S2_source_data_all_panels.csv")
    save_csv(s3_all, cfg.source_data_dir / "Supplementary_Figure_S3_source_data_all_panels.csv")
    save_csv(s4_all, cfg.source_data_dir / "Supplementary_Figure_S4_source_data_all_panels.csv")
    for fig, panels in [("S2", s2), ("S3", s3), ("S4", s4)]:
        for panel, df in panels.items():
            save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_{fig}_panel_{str(panel).lower()}_source_data.csv")


# =============================================================================
# Reports
# =============================================================================

def split_count_validation(inputs: Dict[str, Any], cfg: Config) -> pd.DataFrame:
    ss = inputs["split_summary"]
    count_col = resolve_col(ss, ["n_unique_sequences", "n_sequences", "n_rows"])
    obs = ss.set_index("split")[count_col].to_dict()
    rows = []
    obs_total = 0
    for split in SPLIT_ORDER:
        observed = int(obs.get(split, 0))
        expected = int(cfg.expected_split_counts.get(split, 0))
        obs_total += observed
        rows.append({"item": f"split::{split}", "observed": observed, "expected": expected,
                     "difference": observed - expected, "matches_expected": observed == expected})
    rows.append({"item": "total", "observed": obs_total, "expected": cfg.expected_total_sequences,
                 "difference": obs_total - cfg.expected_total_sequences,
                 "matches_expected": obs_total == cfg.expected_total_sequences})
    return pd.DataFrame(rows)


def nearest_summary(inputs: Dict[str, Any]) -> pd.DataFrame:
    nn = prepare_nearest_df(inputs["nearest"])
    rows = []
    for split in ["val", "test"]:
        vals = finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"])
        rows.append({
            "query_split": split,
            "query_split_display": SPLIT_DISPLAY[split],
            "n": int(len(vals)),
            "mean": float(np.mean(vals)) if len(vals) else np.nan,
            "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "min": float(np.min(vals)) if len(vals) else np.nan,
            "median": float(np.median(vals)) if len(vals) else np.nan,
            "p90": float(np.percentile(vals, 90)) if len(vals) else np.nan,
            "p95": float(np.percentile(vals, 95)) if len(vals) else np.nan,
            "p99": float(np.percentile(vals, 99)) if len(vals) else np.nan,
            "max": float(np.max(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def descriptor_balance_summary(s4: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    records = []
    for panel, df in s4.items():
        descriptor = df["descriptor"].iloc[0]
        for split in SPLIT_ORDER:
            vals = finite_array(df.loc[df["assigned_split"] == split, "value"])
            records.append({
                "panel": panel,
                "descriptor": descriptor,
                "split": split,
                "split_display": SPLIT_DISPLAY[split],
                "n": int(len(vals)),
                "median": float(np.median(vals)) if len(vals) else np.nan,
                "q25": float(np.percentile(vals, 25)) if len(vals) else np.nan,
                "q75": float(np.percentile(vals, 75)) if len(vals) else np.nan,
                "mean": float(np.mean(vals)) if len(vals) else np.nan,
                "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            })
    return pd.DataFrame(records)


def save_reports(cfg: Config, inputs: Dict[str, Any], s2, s3, s4) -> Tuple[Dict[str, pd.DataFrame], list[str]]:
    reports = {
        "split_count_validation_audit": split_count_validation(inputs, cfg),
        "leakage_audit_summary": pd.concat([s2["B"].assign(report_type="sampled_cross_split_jaccard"),
                                            s2["C"].assign(report_type="overlap_audit"),
                                            s2["D"].assign(report_type="cluster_counts")], ignore_index=True, sort=False),
        "nearest_train_similarity_summary": nearest_summary(inputs),
        "threshold_sensitivity_summary": inputs["threshold"].copy(),
        "descriptor_balance_summary": descriptor_balance_summary(s4),
    }
    warnings = []
    if not bool(reports["split_count_validation_audit"]["matches_expected"].all()):
        warnings.append("Recovered split counts do not match configured expected counts; check Results 3.2.")
    max_nn = reports["nearest_train_similarity_summary"]["max"].max()
    if pd.notna(max_nn) and max_nn > cfg.main_jaccard_threshold:
        warnings.append(
            f"Nearest-train audit contains values above threshold ({max_nn:.3f} > {cfg.main_jaccard_threshold:.2f}); "
            "do not claim complete removal of all near-neighbour similarity."
        )
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
        save_csv(df, cfg.tables_dir / f"{name}.csv")
    return reports, warnings


# =============================================================================
# Plotting
# =============================================================================

def plot_s2(s2: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    fig, axes = plt.subplots(2, 2, figsize=(10.9, 6.7))
    fig.subplots_adjust(left=0.085, right=0.985, top=0.92, bottom=0.14, wspace=0.36, hspace=0.43)

    ax = axes[0, 0]
    df = s2["A"]
    vals = pd.to_numeric(df["n_unique_sequences"], errors="coerce").fillna(0).to_numpy()
    x = np.arange(len(SPLIT_ORDER))
    bars = ax.bar(x, vals, width=0.58, color=[SPLIT_FACE[s] for s in SPLIT_ORDER],
                  edgecolor=[SPLIT_EDGE[s] for s in SPLIT_ORDER], hatch=[SPLIT_HATCH[s] for s in SPLIT_ORDER],
                  linewidth=0.85, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLIT_ORDER])
    ax.set_ylabel("Unique sequences"); ax.set_title("Split composition", pad=8)
    ymax = max(vals) if len(vals) else 1
    ax.set_ylim(0, ymax * 1.18)
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, value + ymax*0.025, f"{int(value):,}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    beautify_axis(ax); panel_letter(ax, "A", cfg)

    ax = axes[0, 1]
    df = s2["B"]
    vals = pd.to_numeric(df["max_jaccard"], errors="coerce").fillna(0).to_numpy()
    x = np.arange(len(PAIR_ORDER))
    bars = ax.bar(x, vals, width=0.58, color=[PAIR_FACE[p] for p in PAIR_ORDER],
                  edgecolor=[PAIR_EDGE[p] for p in PAIR_ORDER], linewidth=0.85, zorder=3)
    ax.axhline(cfg.main_jaccard_threshold, color=PLOS_PALETTE["axis_dark"], linestyle="--", linewidth=0.9)
    ax.text(0.98, cfg.main_jaccard_threshold + 0.018, f"threshold = {cfg.main_jaccard_threshold:.2f}",
            transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels(df["pair_display"], rotation=12, ha="right")
    ax.set_ylabel("Max sampled 3-mer Jaccard"); ax.set_title("Sampled cross-split similarity", pad=8)
    ax.set_ylim(0, max(0.65, float(np.nanmax(vals))*1.25 if len(vals) else 0.65))
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.018, f"{value:.3f}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    beautify_axis(ax); panel_letter(ax, "B", cfg)

    ax = axes[1, 0]
    overlap = s2["C"]
    width = 0.34
    x = np.arange(len(PAIR_ORDER))
    for offset, overlap_type, label, hatch in [
        (-width/2, "exact_sequence_overlap", "Exact sequence", ""),
        (width/2, "cluster_overlap", "Similarity cluster", "///"),
    ]:
        sub = overlap.loc[overlap["overlap_type"] == overlap_type].set_index(["split_a", "split_b"]).reindex(PAIR_ORDER).reset_index()
        vals = pd.to_numeric(sub["n_overlap"], errors="coerce").fillna(0).to_numpy()
        bars = ax.bar(x + offset, vals, width=width, color=OVERLAP_FACE[overlap_type],
                      edgecolor=OVERLAP_EDGE[overlap_type], hatch=hatch, linewidth=0.75, label=label, zorder=3)
        for bar, value in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, value + 0.04, f"{int(value)}",
                    ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels([PAIR_DISPLAY[p] for p in PAIR_ORDER], rotation=12, ha="right")
    ax.set_ylabel("Overlap count"); ax.set_title("Exact and cluster-overlap audit", pad=8)
    ax.set_ylim(0, 1.05)
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="upper right")
    beautify_axis(ax); panel_letter(ax, "C", cfg)

    ax = axes[1, 1]
    df = s2["D"]
    vals = pd.to_numeric(df["n_clusters"], errors="coerce").fillna(0).to_numpy()
    x = np.arange(len(SPLIT_ORDER))
    bars = ax.bar(x, vals, width=0.58, color=[SPLIT_FACE[s] for s in SPLIT_ORDER],
                  edgecolor=[SPLIT_EDGE[s] for s in SPLIT_ORDER], hatch=[SPLIT_HATCH[s] for s in SPLIT_ORDER],
                  linewidth=0.85, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLIT_ORDER])
    ax.set_ylabel("Similarity clusters"); ax.set_title("Cluster assignment by split", pad=8)
    ymax = max(vals) if len(vals) else 1
    ax.set_ylim(0, ymax * 1.18)
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, value + ymax*0.025, f"{int(value):,}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    beautify_axis(ax); panel_letter(ax, "D", cfg)

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S2_redesigned", cfg)


def plot_s3(s3: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    fig, axes = plt.subplots(2, 2, figsize=(11.0, 6.8))
    fig.subplots_adjust(left=0.085, right=0.985, top=0.92, bottom=0.14, wspace=0.36, hspace=0.43)

    ax = axes[0, 0]
    ecdf = s3["A"]
    quant = s3["A_quantiles"]
    for split in ["val", "test"]:
        sub = ecdf.loc[ecdf["query_split"] == split].sort_values("nearest_train_jaccard")
        if len(sub):
            ax.step(sub["nearest_train_jaccard"], sub["ecdf"], where="post", color=SPLIT_EDGE[split],
                    linewidth=1.6, label=f"{SPLIT_DISPLAY[split]} (n={int(sub['n_sequences'].iloc[0]):,})", zorder=3)
            qsub = quant.loc[(quant["query_split"] == split) & (quant["quantile"].isin(["median", "p90"]))]
            for _, row in qsub.iterrows():
                ax.axvline(row["nearest_train_jaccard"], color=SPLIT_EDGE[split],
                           linestyle="--" if row["quantile"] == "median" else ":", linewidth=0.85, alpha=0.8, zorder=2)
    ax.axvline(cfg.main_jaccard_threshold, color=PLOS_PALETTE["axis_dark"], linestyle="-.", linewidth=0.9, alpha=0.9)
    ax.text(cfg.main_jaccard_threshold + 0.01, 0.08, "0.55", rotation=90, fontsize=cfg.annotation_size,
            va="bottom", ha="left", color=PLOS_PALETTE["axis_dark"])
    ax.set_xlabel("Nearest-train 3-mer Jaccard"); ax.set_ylabel("Cumulative fraction")
    ax.set_title("Nearest-train similarity distribution", pad=8)
    ax.set_xlim(0, max(0.70, float(ecdf["nearest_train_jaccard"].max())*1.05 if len(ecdf) else 0.70))
    ax.set_ylim(0, 1.02)
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="lower right")
    beautify_axis(ax); panel_letter(ax, "A", cfg)

    ax = axes[0, 1]
    ex = s3["B"]
    thresholds = sorted(ex["threshold"].dropna().unique())
    x = np.arange(len(thresholds)); width = 0.34
    for offset, split in [(-width/2, "val"), (width/2, "test")]:
        sub = ex.loc[ex["query_split"] == split].set_index("threshold").reindex(thresholds)
        vals = pd.to_numeric(sub["fraction_exceeding"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offset, vals, width=width, color=SPLIT_FACE[split], edgecolor=SPLIT_EDGE[split],
               hatch=SPLIT_HATCH[split], linewidth=0.8, label=SPLIT_DISPLAY[split], zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([f"{t:.2f}" for t in thresholds])
    ax.set_xlabel("Nearest-train Jaccard threshold"); ax.set_ylabel("Fraction ≥ threshold")
    ax.set_ylim(0, 1.0); ax.set_title("High-similarity tail", pad=8)
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="upper right")
    beautify_axis(ax); panel_letter(ax, "B", cfg)

    ax = axes[1, 0]
    edges = s3["C"].copy().sort_values("threshold")
    ax.plot(edges["threshold"], edges["n_edges"], marker="o", markersize=4.5, linewidth=1.5,
            color=PLOS_PALETTE["charcoal_brown"], markerfacecolor=PLOS_PALETTE["charcoal_brown"],
            markeredgecolor=PLOS_PALETTE["axis_dark"], zorder=3)
    for _, row in edges.iterrows():
        ax.text(row["threshold"], row["n_edges"], f"{int(row['n_edges']):,}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xlabel("3-mer Jaccard threshold"); ax.set_ylabel("Retained similarity edges")
    ax.set_title("Similarity-graph sensitivity", pad=8)
    beautify_axis(ax); panel_letter(ax, "C", cfg)

    ax = axes[1, 1]
    sizes = s3["D"]
    for split in SPLIT_ORDER:
        sub = sizes.loc[sizes["split"] == split].sort_values("threshold")
        ax.plot(sub["threshold"], sub["n_unique_sequences"], marker="o", markersize=4.3, linewidth=1.5,
                color=SPLIT_EDGE[split], markerfacecolor=SPLIT_FACE[split], markeredgecolor=SPLIT_EDGE[split],
                label=SPLIT_DISPLAY[split], zorder=3)
    ax.set_xlabel("3-mer Jaccard threshold"); ax.set_ylabel("Unique sequences")
    ax.set_title("Split-size stability", pad=8)
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="best")
    beautify_axis(ax); panel_letter(ax, "D", cfg)

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S3_redesigned", cfg)


def safe_boxplot(ax: plt.Axes, data: list[np.ndarray], positions: np.ndarray, edge_colors: list[str]) -> None:
    bp = ax.boxplot(
        data, positions=positions, widths=0.16, patch_artist=True, showfliers=False,
        medianprops=dict(color=PLOS_PALETTE["axis_dark"], linewidth=1.15),
        whiskerprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        capprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        boxprops=dict(linewidth=0.75),
    )
    for box, color in zip(bp["boxes"], edge_colors):
        box.set_facecolor("white")
        box.set_edgecolor(color)
        box.set_alpha(0.98)
        box.set_zorder(4)


def split_violin_box(ax: plt.Axes, panel_df: pd.DataFrame, ylabel: str, title: str, cfg: Config, zero_line: bool = False) -> None:
    positions = np.arange(1, len(SPLIT_ORDER) + 1)
    data = [finite_array(panel_df.loc[panel_df["assigned_split"] == split, "value"]) for split in SPLIT_ORDER]
    vp = ax.violinplot(data, positions=positions, widths=0.62, showmeans=False, showmedians=False, showextrema=False)
    for body, split in zip(vp["bodies"], SPLIT_ORDER):
        body.set_facecolor(SPLIT_FACE[split])
        body.set_edgecolor(SPLIT_EDGE[split])
        body.set_alpha(0.38)
        body.set_linewidth(0.9)
        body.set_zorder(3)
    safe_boxplot(ax, data, positions, [SPLIT_EDGE[s] for s in SPLIT_ORDER])
    if zero_line:
        ax.axhline(0, color=PLOS_PALETTE["charcoal_brown"], linestyle="--", linewidth=0.75, zorder=2)
    ax.set_xticks(positions)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLIT_ORDER])
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    beautify_axis(ax)


def plot_s4(s4: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    fig, axes = plt.subplots(2, 2, figsize=(9.9, 6.6))
    fig.subplots_adjust(left=0.095, right=0.985, top=0.92, bottom=0.15, wspace=0.35, hspace=0.43)
    specs = [
        ("A", axes[0, 0], "Length (aa)", "Length", False),
        ("B", axes[0, 1], "Net charge at pH 7", "Net charge", True),
        ("C", axes[1, 0], "Hydrophobicity\n(Kyte–Doolittle)", "Hydrophobicity", True),
        ("D", axes[1, 1], "Shannon entropy (bits)", "Shannon entropy", False),
    ]
    for panel, ax, ylabel, title, zero in specs:
        split_violin_box(ax, s4[panel], ylabel, title, cfg, zero_line=zero)
        panel_letter(ax, panel, cfg)
    fig.legend(handles=split_legend_handles(), loc="lower center", ncol=3, frameon=True,
               bbox_to_anchor=(0.5, 0.045), fontsize=cfg.legend_size, edgecolor=PLOS_PALETTE["light_gray"],
               facecolor=PLOS_PALETTE["background"], handlelength=1.5, handletextpad=0.55,
               columnspacing=1.35, borderpad=0.35)
    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S4_redesigned", cfg)


# =============================================================================
# Reproducibility files and manifest
# =============================================================================

def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env_commit = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env_commit:
        return env_commit
    try:
        result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(cfg.source_step2_dir),
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5, check=False)
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        return None
    return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    file_name = globals().get("__file__", None)
    if file_name and Path(file_name).exists():
        script_path = str(Path(file_name).resolve())
        text = Path(file_name).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session. Save this notebook cell/script manually.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_02_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_02_PLOS_redesign_code.py")
    return script_path, script_hash


def write_requirements_summary(cfg: Config) -> Path:
    path = cfg.reports_dir / "requirements_step_02_minimal.txt"
    save_text(f"python=={platform.python_version()}\nnumpy=={np.__version__}\npandas=={pd.__version__}\nmatplotlib=={mpl.__version__}\n", path)
    return path


def write_main_figure_note(cfg: Config) -> Path:
    text = """# Step 2 main-figure note

Step 2 is intentionally supplementary-only in the OncoPep PLOS Computational
Biology figure-redesign workflow.

Rationale:
- Step 2 provides technical validation of similarity-aware data partitioning,
  leakage auditing, threshold sensitivity, and descriptor preservation.
- These analyses are essential for reproducibility and reviewer confidence, but
  they support the modelling workflow rather than serving as the central model
  contribution.
- Main Figure 2 is reserved for the OncoPep architecture and mathematical
  workflow, which should remain the visual centerpiece of the algorithmic
  contribution.

Manuscript implication:
- Results 3.2 should cite Supplementary Figures S2-S4.
- Main Figure 2 should not be cited for Step 2 partitioning.
"""
    path = cfg.main_note_dir / "README_no_main_figure_for_step_02.txt"
    save_text(text, path)
    return path


def write_readme(cfg: Config, warnings: Sequence[str]) -> Path:
    text = f"""# OncoPep Step 2 PLOS supplementary-figure package

Generated: {datetime.now().isoformat(timespec="seconds")}

Input directory:
{cfg.source_step2_dir}

Output directory:
{cfg.output_root}

Generated figures:
- supplementary_figures/Supplementary_Figure_S2_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S3_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S4_redesigned.png/.pdf/.tiff

Scientific scope:
- Supplementary Figure S2: split composition, sampled cross-split 3-mer Jaccard,
  exact/cluster-overlap audit, and cluster assignment by split.
- Supplementary Figure S3: nearest-train 3-mer Jaccard ECDF, threshold exceedance,
  retained similarity edges, and split-size stability.
- Supplementary Figure S4: descriptor preservation across train, validation, and
  test partitions.

Manuscript notes:
- Use recovered split counts: train=24,485, validation=13,260, test=13,260.
- Do not claim complete elimination of all near-neighbour similarity; nearest-train
  auditing may show a high-similarity tail.
- Step 2 is supplementary-only; Main Figure 2 is reserved for the OncoPep
  architecture and mathematical workflow.

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
"""
    path = cfg.reports_dir / "README_step_02_outputs.txt"
    save_text(text, path)
    return path


def create_root_level_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs = []
    for fig in ["S2", "S3", "S4"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv",
                      cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))
    pairs.append((cfg.code_dir / "OncoPep_step_02_PLOS_redesign_code.py",
                  cfg.output_root / "OncoPep_step_02_PLOS_redesign_code.py"))
    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def build_manifest(cfg: Config, inputs: Dict[str, Any], s2_paths, s3_paths, s4_paths,
                   reports, warnings, readme_path, main_note_path, requirements_path,
                   script_path, script_hash, root_copies) -> Dict[str, Any]:
    input_paths = inputs["_paths"]
    return {
        "step": "step_02",
        "scope": "supplementary_only",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "rerun_split_construction": False,
        "main_figure_2_generated": False,
        "main_figure_2_rationale": "Reserved for OncoPep architecture and mathematical workflow.",
        "source_step2_dir": str(cfg.source_step2_dir),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "input_files": {k: str(v) for k, v in input_paths.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_paths.items()},
        "figures": {"Supplementary_Figure_S2": s2_paths,
                    "Supplementary_Figure_S3": s3_paths,
                    "Supplementary_Figure_S4": s4_paths},
        "source_data": {"S2_all": str(cfg.source_data_dir / "Supplementary_Figure_S2_source_data_all_panels.csv"),
                        "S3_all": str(cfg.source_data_dir / "Supplementary_Figure_S3_source_data_all_panels.csv"),
                        "S4_all": str(cfg.source_data_dir / "Supplementary_Figure_S4_source_data_all_panels.csv")},
        "reports": {name: str(cfg.reports_dir / f"{name}.csv") for name in reports},
        "readme": str(readme_path),
        "main_figure_note": str(main_note_path),
        "requirements_summary": str(requirements_path),
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(root_copies),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# =============================================================================
# Main workflow and entry points
# =============================================================================

def run_step2_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_output_tree(cfg)
    set_style(cfg)
    inputs = load_inputs(cfg)

    s2, s2_all = build_s2_source_data(inputs)
    s3, s3_all = build_s3_source_data(inputs)
    s4, s4_all = build_s4_source_data(inputs)

    save_source_data(cfg, s2, s2_all, s3, s3_all, s4, s4_all)
    reports, warnings = save_reports(cfg, inputs, s2, s3, s4)

    s2_paths = plot_s2(s2, cfg)
    s3_paths = plot_s3(s3, cfg)
    s4_paths = plot_s4(s4, cfg)

    script_path, script_hash = code_snapshot(cfg)
    requirements_path = write_requirements_summary(cfg)
    main_note_path = write_main_figure_note(cfg)
    readme_path = write_readme(cfg, warnings)
    root_copies = create_root_level_copies(cfg)

    manifest = build_manifest(cfg, inputs, s2_paths, s3_paths, s4_paths, reports, warnings,
                              readme_path, main_note_path, requirements_path, script_path,
                              script_hash, root_copies)
    save_json(manifest, cfg.reports_dir / "step_02_manifest.json")
    save_json(manifest, cfg.output_root / "step_02_manifest.json")

    print("\n✅ Step 2 PLOS supplementary-figure package generated successfully.")
    print(f"Source Step 2 directory: {cfg.source_step2_dir}")
    print(f"Output directory: {cfg.output_root}")
    for label, paths in [("Supplementary Figure S2", s2_paths),
                         ("Supplementary Figure S3", s3_paths),
                         ("Supplementary Figure S4", s4_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")
    print("\nReports:")
    for name in reports:
        print(f"  - {cfg.reports_dir / f'{name}.csv'}")
    print(f"  - {cfg.reports_dir / 'step_02_manifest.json'}")
    print(f"  - {cfg.reports_dir / 'README_step_02_outputs.txt'}")
    if warnings:
        print("\n⚠️ Warnings:")
        for w in warnings:
            print(f"  - {w}")

    return {"s2_paths": s2_paths, "s3_paths": s3_paths, "s4_paths": s4_paths,
            "s2_panels": s2, "s3_panels": s3, "s4_panels": s4,
            "reports": reports, "warnings": warnings, "manifest": manifest}


def main_notebook(
    source_step2_dir: str | Path = SOURCE_STEP2_DIR_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    main_jaccard_threshold: float = 0.55,
    create_root_level_copies: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        source_step2_dir=Path(source_step2_dir),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        main_jaccard_threshold=main_jaccard_threshold,
        create_root_level_copies=create_root_level_copies,
        git_commit=git_commit,
    )
    return run_step2_redesign(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate OncoPep Step 2 PLOS supplementary figures from existing Step 2 outputs."
    )
    parser.add_argument("--source_step2_dir", default=str(SOURCE_STEP2_DIR_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--main_jaccard_threshold", type=float, default=0.55)
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    run_step2_redesign(
        Config(
            source_step2_dir=Path(args.source_step2_dir),
            base_dir=Path(args.base_dir),
            output_root_name=args.output_root_name,
            main_jaccard_threshold=args.main_jaccard_threshold,
            create_root_level_copies=not args.no_root_copies,
            git_commit=args.git_commit,
        )
    )

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 2 — PLOS Computational Biology supplementary-figure redesign (v3).

Generates a supplementary-only Step 2 package from recovered Step 2 split/audit
outputs:

  Supplementary Figure S2 — similarity-aware partitioning and split-leakage audit
  Supplementary Figure S3 — nearest-train similarity, tail audit, and threshold sensitivity
  Supplementary Figure S4 — descriptor preservation across partitions

This script does NOT rerun split construction. Main Figure 2 is intentionally
reserved for the OncoPep architecture and mathematical workflow.

Notebook:
    results = main_notebook(
        source_step2_dir="/home/data3/Moe/nature_computational_peponco/step2",
        base_dir="/home/data3/Moe/nature_computational_peponco/PLOS",
        output_root_name="plos_comp/step_02",
        main_jaccard_threshold=0.55,
    )

Terminal:
    python OncoPep_step_02_PLOS_redesign_code_v2.py
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd


# =============================================================================
# Configuration
# =============================================================================

SOURCE_STEP2_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/step2")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_02"

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
PAIR_ORDER = (("train", "val"), ("train", "test"), ("val", "test"))
PAIR_DISPLAY = {
    ("train", "val"): "Train–validation",
    ("train", "test"): "Train–test",
    ("val", "test"): "Validation–test",
}
EXPECTED_SPLIT_COUNTS_DEFAULT = {"train": 24485, "val": 13260, "test": 13260}
EXPECTED_TOTAL_DEFAULT = sum(EXPECTED_SPLIT_COUNTS_DEFAULT.values())

PLOS_PALETTE = {
    "olive_green": "#9AAA63",
    "soft_coral": "#DD705D",
    "charcoal_brown": "#6A5E61",
    "pale_sand": "#F0CF95",
    "muted_mint": "#A8D3B2",
    "teal_green": "#56A98B",
    "blue_teal": "#1F95B8",
    "warm_brown": "#B67D4E",
    "axis_dark": "#333333",
    "light_gray": "#D9D9D9",
    "grid": "#EAEAEA",
    "background": "#FFFFFF",
}

SPLIT_FACE = {
    "train": PLOS_PALETTE["blue_teal"],
    "val": PLOS_PALETTE["muted_mint"],
    "test": PLOS_PALETTE["soft_coral"],
}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}
SPLIT_HATCH = {"train": "", "val": "//", "test": "\\\\"}

PAIR_FACE = {
    ("train", "val"): PLOS_PALETTE["blue_teal"],
    ("train", "test"): PLOS_PALETTE["warm_brown"],
    ("val", "test"): PLOS_PALETTE["soft_coral"],
}
PAIR_EDGE = {
    ("train", "val"): "#176F8A",
    ("train", "test"): "#8F5F37",
    ("val", "test"): "#A84F42",
}
OVERLAP_FACE = {
    "exact_sequence_overlap": PLOS_PALETTE["charcoal_brown"],
    "cluster_overlap": PLOS_PALETTE["warm_brown"],
}
OVERLAP_EDGE = {"exact_sequence_overlap": "#4F474A", "cluster_overlap": "#8F5F37"}


@dataclass
class Config:
    source_step2_dir: Path = SOURCE_STEP2_DIR_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT
    main_jaccard_threshold: float = 0.55
    expected_split_counts: Dict[str, int] = field(default_factory=lambda: EXPECTED_SPLIT_COUNTS_DEFAULT.copy())
    expected_total_sequences: int = EXPECTED_TOTAL_DEFAULT
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    git_commit: Optional[str] = None

    # Typography
    font_size: float = 8.3
    axis_label_size: float = 8.9
    tick_label_size: float = 7.9
    title_size: float = 9.3
    legend_size: float = 7.8
    annotation_size: float = 7.7
    panel_letter_size: float = 13.2

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def tables_dir(self) -> Path:
        return self.output_root / "tables"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_note_dir(self) -> Path:
        return self.output_root / "main_figure_note"


# =============================================================================
# Utility functions
# =============================================================================

def ensure_output_tree(cfg: Config) -> None:
    for p in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.tables_dir, cfg.reports_dir, cfg.code_dir, cfg.main_note_dir]:
        p.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"{type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=json_default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def finite_array(values: Any) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr)]


def normalize_split(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_col(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    if required:
        raise ValueError(f"Could not resolve column. Candidates={list(candidates)}; available={list(df.columns)}")
    return None


def assert_columns(df: pd.DataFrame, cols: Sequence[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns {missing}; available={list(df.columns)}")


def validate_splits_present(series: pd.Series, context: str) -> None:
    observed = set(normalize_split(series).dropna().unique())
    missing = [s for s in SPLIT_ORDER if s not in observed]
    if missing:
        raise ValueError(f"{context}: missing split labels {missing}; observed={sorted(observed)}")


def metric_definition(metric_name: str) -> str:
    definitions = {
        "n_unique_sequences": "Number of unique peptide sequences assigned to each split.",
        "n_clusters": "Number of similarity-connected components assigned to each split.",
        "max_sampled_cross_split_3mer_jaccard": "Maximum sampled cross-split 3-mer set Jaccard similarity for a split pair.",
        "exact_sequence_overlap": "Number of identical peptide sequences shared between split pairs.",
        "cluster_overlap": "Number of similarity clusters shared between split pairs.",
        "nearest_train_3mer_jaccard": "3-mer set Jaccard similarity between a held-out peptide and its nearest training peptide.",
        "threshold_exceedance_fraction": "Fraction of held-out peptides with nearest-train Jaccard greater than or equal to a threshold.",
        "retained_similarity_edges": "Number of similarity-graph edges retained at a given Jaccard threshold.",
        "split_size_stability": "Number of unique sequences assigned to each split under threshold-sensitivity reruns.",
        "descriptor_distribution": "Sequence-level descriptor distribution within a split.",
    }
    return definitions.get(metric_name, "")


# =============================================================================
# Plot helpers
# =============================================================================

def set_style(cfg: Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_label_size,
            "ytick.labelsize": cfg.tick_label_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": 0.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS_PALETTE["background"],
            "axes.facecolor": PLOS_PALETTE["background"],
            "savefig.facecolor": PLOS_PALETTE["background"],
            "text.color": PLOS_PALETTE["axis_dark"],
            "axes.labelcolor": PLOS_PALETTE["axis_dark"],
            "axes.edgecolor": PLOS_PALETTE["axis_dark"],
            "xtick.color": PLOS_PALETTE["axis_dark"],
            "ytick.color": PLOS_PALETTE["axis_dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS_PALETTE["grid"], linewidth=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(PLOS_PALETTE["axis_dark"])
        ax.spines[side].set_linewidth(0.75)


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.09) -> None:
    ax.text(dx, dy, str(letter).upper(), transform=ax.transAxes, fontsize=cfg.panel_letter_size,
            fontweight="bold", ha="left", va="top", color=PLOS_PALETTE["axis_dark"])


def save_figure(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {"png": base.with_suffix(".png"), "pdf": base.with_suffix(".pdf"), "tiff": base.with_suffix(".tiff")}
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


def split_legend_handles(include_hatches: bool = True) -> list[Patch]:
    return [
        Patch(
            facecolor=SPLIT_FACE[s],
            edgecolor=SPLIT_EDGE[s],
            hatch=SPLIT_HATCH[s] if include_hatches else "",
            label=SPLIT_DISPLAY[s],
            linewidth=0.8,
        )
        for s in SPLIT_ORDER
    ]


# =============================================================================
# Input loading and validation
# =============================================================================

def locate_input_files(cfg: Config) -> Dict[str, Path]:
    root = cfg.source_step2_dir
    paths = {
        "split_summary": root / "tables" / "step2_split_summary.csv",
        "leakage": root / "tables" / "step2_leakage_audits.csv",
        "nearest": root / "tables" / "step2_nearest_neighbor_audit.csv",
        "threshold": root / "tables" / "step2_threshold_sensitivity.csv",
        "unique": root / "splits" / "step2_unique_sequences_with_splits.csv",
        "row": root / "splits" / "step2_row_level_with_splits.csv",
    }
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required Step 2 input files:\n" + "\n".join(missing))
    return paths


def load_inputs(cfg: Config) -> Dict[str, Any]:
    paths = locate_input_files(cfg)
    inputs: Dict[str, Any] = {
        "split_summary": pd.read_csv(paths["split_summary"]),
        "leakage": pd.read_csv(paths["leakage"]),
        "nearest": pd.read_csv(paths["nearest"], low_memory=False),
        "threshold": pd.read_csv(paths["threshold"]),
        "unique": pd.read_csv(paths["unique"], low_memory=False),
        "row": pd.read_csv(paths["row"], low_memory=False),
        "_paths": paths,
    }
    validate_and_normalize_inputs(inputs)
    return inputs


def validate_and_normalize_inputs(inputs: Dict[str, Any]) -> None:
    split_summary = inputs["split_summary"]
    leakage = inputs["leakage"]
    nearest = inputs["nearest"]
    threshold = inputs["threshold"]
    unique = inputs["unique"]
    row = inputs["row"]

    assert_columns(split_summary, ["split"], "step2_split_summary.csv")
    split_count_col = resolve_col(split_summary, ["n_unique_sequences", "n_sequences", "n_rows"])
    cluster_col = resolve_col(split_summary, ["n_clusters"], required=False)
    split_summary["split"] = normalize_split(split_summary["split"])
    validate_splits_present(split_summary["split"], "split summary")
    split_summary[split_count_col] = pd.to_numeric(split_summary[split_count_col], errors="coerce")
    if cluster_col:
        split_summary[cluster_col] = pd.to_numeric(split_summary[cluster_col], errors="coerce")

    assert_columns(leakage, ["audit_type", "split_a", "split_b"], "step2_leakage_audits.csv")
    leakage["split_a"] = normalize_split(leakage["split_a"])
    leakage["split_b"] = normalize_split(leakage["split_b"])
    expected_audits = {"exact_sequence_overlap", "cluster_overlap", "sampled_cross_split_jaccard"}
    observed_audits = set(leakage["audit_type"].astype(str))
    missing = expected_audits - observed_audits
    if missing:
        raise ValueError(f"Missing leakage audit_type values: {sorted(missing)}; observed={sorted(observed_audits)}")
    for col in ["n_overlap", "n_sampled_pairs", "max_jaccard", "mean_jaccard", "threshold"]:
        if col in leakage.columns:
            leakage[col] = pd.to_numeric(leakage[col], errors="coerce")

    nearest_split_col = resolve_col(nearest, ["query_split", "assigned_split", "split"])
    nearest_jaccard_col = resolve_col(nearest, ["nearest_train_jaccard", "nearest_train_similarity", "jaccard"])
    nearest[nearest_split_col] = normalize_split(nearest[nearest_split_col])
    nearest[nearest_jaccard_col] = pd.to_numeric(nearest[nearest_jaccard_col], errors="coerce")
    if nearest_split_col != "query_split":
        nearest.rename(columns={nearest_split_col: "query_split"}, inplace=True)
    if nearest_jaccard_col != "nearest_train_jaccard":
        nearest.rename(columns={nearest_jaccard_col: "nearest_train_jaccard"}, inplace=True)
    if not {"val", "test"}.issubset(set(nearest["query_split"].unique())):
        raise ValueError("Nearest-neighbor audit must include validation and test query_split values.")

    assert_columns(threshold, ["threshold", "n_edges"], "step2_threshold_sensitivity.csv")
    threshold["threshold"] = pd.to_numeric(threshold["threshold"], errors="coerce")
    threshold["n_edges"] = pd.to_numeric(threshold["n_edges"], errors="coerce")
    missing_size_cols = [f"{s}_n_sequences" for s in SPLIT_ORDER if f"{s}_n_sequences" not in threshold.columns]
    if missing_size_cols:
        raise ValueError(f"Threshold sensitivity table missing columns: {missing_size_cols}")
    for col in [f"{s}_n_sequences" for s in SPLIT_ORDER]:
        threshold[col] = pd.to_numeric(threshold[col], errors="coerce")

    for name, df in [("unique", unique), ("row", row)]:
        split_col = resolve_col(df, ["assigned_split", "split"])
        df[split_col] = normalize_split(df[split_col])
        if split_col != "assigned_split":
            df.rename(columns={split_col: "assigned_split"}, inplace=True)
        validate_splits_present(df["assigned_split"], f"{name} split file")

    descriptor_errors = []
    for name, df in [("unique", unique), ("row", row)]:
        try:
            resolve_col(df, ["length", "sequence_length"])
            resolve_col(df, ["net_charge_pH7", "net_charge", "charge"])
            resolve_col(df, ["hydrophobicity_KD", "hydrophobicity", "mean_hydropathy"])
            resolve_col(df, ["shannon_entropy", "entropy", "sequence_entropy"])
            break
        except Exception as exc:
            descriptor_errors.append(f"{name}: {exc}")
    else:
        raise ValueError("Could not resolve descriptor columns:\n" + "\n".join(descriptor_errors))


# =============================================================================
# Source-data builders
# =============================================================================

def complete_pair_table(df: pd.DataFrame, value_cols: Sequence[str]) -> pd.DataFrame:
    base = pd.DataFrame({
        "split_a": [a for a, _ in PAIR_ORDER],
        "split_b": [b for _, b in PAIR_ORDER],
        "pair_display": [PAIR_DISPLAY[p] for p in PAIR_ORDER],
    })
    if df.empty:
        out = base.copy()
    else:
        keep = ["split_a", "split_b"] + [c for c in value_cols if c in df.columns]
        out = base.merge(df[keep], on=["split_a", "split_b"], how="left")
    for col in value_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0)
    return out


def leakage_subset(leakage: pd.DataFrame, audit_type: str, value_cols: Sequence[str]) -> pd.DataFrame:
    sub = leakage.loc[leakage["audit_type"].astype(str) == audit_type].copy()
    if not sub.empty:
        sub["split_a"] = normalize_split(sub["split_a"])
        sub["split_b"] = normalize_split(sub["split_b"])
    return complete_pair_table(sub, value_cols)


def build_s2_source_data(inputs: Dict[str, Any]) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    split_summary = inputs["split_summary"].copy()
    split_count_col = resolve_col(split_summary, ["n_unique_sequences", "n_sequences", "n_rows"])
    cluster_col = resolve_col(split_summary, ["n_clusters"], required=False)

    split = pd.DataFrame({"split": list(SPLIT_ORDER)}).merge(split_summary, on="split", how="left")
    split.rename(columns={split_count_col: "n_unique_sequences"}, inplace=True)
    if cluster_col and cluster_col != "n_clusters":
        split.rename(columns={cluster_col: "n_clusters"}, inplace=True)
    if "n_clusters" not in split.columns:
        split["n_clusters"] = np.nan
    split["split_display"] = split["split"].map(SPLIT_DISPLAY)
    split["metric_name"] = "n_unique_sequences"
    split["metric_definition"] = metric_definition("n_unique_sequences")
    split["units"] = "unique sequences"
    split["panel"] = "A"
    split["data_type"] = "split_unique_sequence_counts"

    sampled = leakage_subset(inputs["leakage"], "sampled_cross_split_jaccard",
                             ["n_sampled_pairs", "max_jaccard", "mean_jaccard", "threshold"])
    sampled["metric_name"] = "max_sampled_cross_split_3mer_jaccard"
    sampled["metric_definition"] = metric_definition("max_sampled_cross_split_3mer_jaccard")
    sampled["units"] = "3-mer Jaccard similarity"
    sampled["panel"] = "B"
    sampled["data_type"] = "sampled_cross_split_jaccard"

    exact = leakage_subset(inputs["leakage"], "exact_sequence_overlap", ["n_overlap"])
    exact["overlap_type"] = "exact_sequence_overlap"
    exact["overlap_type_display"] = "Exact sequence"
    exact["metric_name"] = "exact_sequence_overlap"
    exact["metric_definition"] = metric_definition("exact_sequence_overlap")
    exact["units"] = "overlap count"

    cluster = leakage_subset(inputs["leakage"], "cluster_overlap", ["n_overlap"])
    cluster["overlap_type"] = "cluster_overlap"
    cluster["overlap_type_display"] = "Similarity cluster"
    cluster["metric_name"] = "cluster_overlap"
    cluster["metric_definition"] = metric_definition("cluster_overlap")
    cluster["units"] = "overlap count"

    overlap = pd.concat([exact, cluster], ignore_index=True)
    overlap["panel"] = "C"
    overlap["data_type"] = "overlap_audit"

    cluster_counts = split[["split", "split_display", "n_clusters"]].copy()
    cluster_counts["metric_name"] = "n_clusters"
    cluster_counts["metric_definition"] = metric_definition("n_clusters")
    cluster_counts["units"] = "similarity clusters"
    cluster_counts["panel"] = "D"
    cluster_counts["data_type"] = "similarity_cluster_counts"

    panels = {"A": split, "B": sampled, "C": overlap, "D": cluster_counts}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def prepare_nearest_df(nearest: pd.DataFrame) -> pd.DataFrame:
    df = nearest.copy()
    df["query_split"] = normalize_split(df["query_split"])
    df["nearest_train_jaccard"] = pd.to_numeric(df["nearest_train_jaccard"], errors="coerce")
    df = df[df["query_split"].isin(["val", "test"])].copy()
    df["query_split_display"] = df["query_split"].map(SPLIT_DISPLAY)
    df["metric_name"] = "nearest_train_3mer_jaccard"
    df["metric_definition"] = metric_definition("nearest_train_3mer_jaccard")
    df["units"] = "3-mer Jaccard similarity"
    return df


def ecdf_dataframe(nn: pd.DataFrame) -> pd.DataFrame:
    records = []
    for split in ["val", "test"]:
        vals = np.sort(finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"]))
        n = len(vals)
        for i, v in enumerate(vals, start=1):
            records.append({
                "query_split": split,
                "query_split_display": SPLIT_DISPLAY[split],
                "nearest_train_jaccard": float(v),
                "ecdf": float(i / n) if n else np.nan,
                "n_sequences": int(n),
                "metric_name": "nearest_train_3mer_jaccard_ecdf",
                "metric_definition": "Empirical cumulative distribution of nearest-train 3-mer Jaccard similarity.",
                "units": "fraction of held-out sequences",
            })
    return pd.DataFrame(records)


def nearest_quantiles(nn: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split in ["val", "test"]:
        vals = finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"])
        for name, percentile in [("median", 50), ("p90", 90), ("p95", 95), ("p99", 99)]:
            rows.append({
                "query_split": split,
                "query_split_display": SPLIT_DISPLAY[split],
                "quantile": name,
                "percentile": percentile,
                "nearest_train_jaccard": float(np.percentile(vals, percentile)) if len(vals) else np.nan,
                "n_sequences": int(len(vals)),
            })
    return pd.DataFrame(rows)


def threshold_exceedance(nn: pd.DataFrame, thresholds: Sequence[float] = (0.10, 0.20, 0.30, 0.50, 0.55)) -> pd.DataFrame:
    rows = []
    for split in ["val", "test"]:
        vals = finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"])
        for thr in thresholds:
            rows.append({
                "query_split": split,
                "query_split_display": SPLIT_DISPLAY[split],
                "threshold": float(thr),
                "n_sequences": int(len(vals)),
                "n_exceeding": int(np.sum(vals >= thr)) if len(vals) else 0,
                "fraction_exceeding": float(np.mean(vals >= thr)) if len(vals) else np.nan,
                "metric_name": "threshold_exceedance_fraction",
                "metric_definition": metric_definition("threshold_exceedance_fraction"),
                "units": "fraction",
            })
    return pd.DataFrame(rows)


def build_s3_source_data(inputs: Dict[str, Any]) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    nn = prepare_nearest_df(inputs["nearest"])
    ecdf = ecdf_dataframe(nn)
    ecdf["panel"] = "A"
    ecdf["data_type"] = "nearest_train_jaccard_ecdf"

    quantiles = nearest_quantiles(nn)
    quantiles["panel"] = "A"
    quantiles["data_type"] = "nearest_train_jaccard_quantiles"

    exceed = threshold_exceedance(nn)
    exceed["panel"] = "B"
    exceed["data_type"] = "nearest_train_threshold_exceedance"

    threshold = inputs["threshold"].copy()
    edges = threshold[["threshold", "n_edges"]].copy()
    edges["metric_name"] = "retained_similarity_edges"
    edges["metric_definition"] = metric_definition("retained_similarity_edges")
    edges["units"] = "edges"
    edges["panel"] = "C"
    edges["data_type"] = "retained_similarity_edges"

    size_cols = [f"{s}_n_sequences" for s in SPLIT_ORDER]
    sizes = threshold[["threshold"] + size_cols].melt(id_vars="threshold", var_name="split_column", value_name="n_unique_sequences")
    sizes["split"] = sizes["split_column"].str.replace("_n_sequences", "", regex=False)
    sizes["split_display"] = sizes["split"].map(SPLIT_DISPLAY)
    sizes["metric_name"] = "split_size_stability"
    sizes["metric_definition"] = metric_definition("split_size_stability")
    sizes["units"] = "unique sequences"
    sizes["panel"] = "D"
    sizes["data_type"] = "split_size_stability"

    panels = {"A": ecdf, "A_quantiles": quantiles, "B": exceed, "C": edges, "D": sizes}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def descriptor_dataframe(inputs: Dict[str, Any]) -> pd.DataFrame:
    errors = []
    for name in ["unique", "row"]:
        df = inputs[name]
        try:
            length_col = resolve_col(df, ["length", "sequence_length"])
            charge_col = resolve_col(df, ["net_charge_pH7", "net_charge", "charge"])
            hydro_col = resolve_col(df, ["hydrophobicity_KD", "hydrophobicity", "mean_hydropathy"])
            entropy_col = resolve_col(df, ["shannon_entropy", "entropy", "sequence_entropy"])
            split_col = resolve_col(df, ["assigned_split", "split"])
            out = df[[split_col, length_col, charge_col, hydro_col, entropy_col]].copy()
            out.columns = ["assigned_split", "length", "net_charge_pH7", "hydrophobicity_KD", "shannon_entropy"]
            out["assigned_split"] = normalize_split(out["assigned_split"])
            for col in ["length", "net_charge_pH7", "hydrophobicity_KD", "shannon_entropy"]:
                out[col] = pd.to_numeric(out[col], errors="coerce")
            out = out[out["assigned_split"].isin(SPLIT_ORDER)].copy()
            if len(out) == 0:
                raise ValueError(f"{name} descriptor dataframe is empty after filtering.")
            return out
        except Exception as exc:
            errors.append(f"{name}: {exc}")
    raise ValueError("Failed to construct descriptor dataframe:\n" + "\n".join(errors))


def build_s4_source_data(inputs: Dict[str, Any]) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    desc = descriptor_dataframe(inputs)
    specs = {
        "A": ("length", "Length", "aa", "Peptide sequence length."),
        "B": ("net_charge_pH7", "Net charge at pH 7", "charge units", "Estimated peptide net charge at pH 7."),
        "C": ("hydrophobicity_KD", "Kyte–Doolittle hydrophobicity", "mean KD score", "Mean Kyte–Doolittle hydrophobicity."),
        "D": ("shannon_entropy", "Shannon entropy", "bits", "Shannon entropy of amino-acid composition."),
    }
    panels = {}
    all_rows = []
    for panel, (col, label, units, definition) in specs.items():
        panel_df = desc[["assigned_split", col]].copy()
        panel_df.rename(columns={col: "value"}, inplace=True)
        panel_df["split_display"] = panel_df["assigned_split"].map(SPLIT_DISPLAY)
        panel_df["descriptor_column"] = col
        panel_df["descriptor"] = label
        panel_df["metric_name"] = "descriptor_distribution"
        panel_df["metric_definition"] = definition
        panel_df["units"] = units
        panel_df["panel"] = panel
        panel_df["data_type"] = "split_descriptor_distribution"
        panels[panel] = panel_df
        all_rows.append(panel_df)
    return panels, pd.concat(all_rows, ignore_index=True, sort=False)


def save_source_data(cfg: Config, s2, s2_all, s3, s3_all, s4, s4_all) -> None:
    save_csv(s2_all, cfg.source_data_dir / "Supplementary_Figure_S2_source_data_all_panels.csv")
    save_csv(s3_all, cfg.source_data_dir / "Supplementary_Figure_S3_source_data_all_panels.csv")
    save_csv(s4_all, cfg.source_data_dir / "Supplementary_Figure_S4_source_data_all_panels.csv")
    for fig, panels in [("S2", s2), ("S3", s3), ("S4", s4)]:
        for panel, df in panels.items():
            save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_{fig}_panel_{str(panel).lower()}_source_data.csv")


# =============================================================================
# Reports
# =============================================================================

def split_count_validation(inputs: Dict[str, Any], cfg: Config) -> pd.DataFrame:
    ss = inputs["split_summary"]
    count_col = resolve_col(ss, ["n_unique_sequences", "n_sequences", "n_rows"])
    obs = ss.set_index("split")[count_col].to_dict()
    rows = []
    obs_total = 0
    for split in SPLIT_ORDER:
        observed = int(obs.get(split, 0))
        expected = int(cfg.expected_split_counts.get(split, 0))
        obs_total += observed
        rows.append({"item": f"split::{split}", "observed": observed, "expected": expected,
                     "difference": observed - expected, "matches_expected": observed == expected})
    rows.append({"item": "total", "observed": obs_total, "expected": cfg.expected_total_sequences,
                 "difference": obs_total - cfg.expected_total_sequences,
                 "matches_expected": obs_total == cfg.expected_total_sequences})
    return pd.DataFrame(rows)


def nearest_summary(inputs: Dict[str, Any]) -> pd.DataFrame:
    nn = prepare_nearest_df(inputs["nearest"])
    rows = []
    for split in ["val", "test"]:
        vals = finite_array(nn.loc[nn["query_split"] == split, "nearest_train_jaccard"])
        rows.append({
            "query_split": split,
            "query_split_display": SPLIT_DISPLAY[split],
            "n": int(len(vals)),
            "mean": float(np.mean(vals)) if len(vals) else np.nan,
            "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "min": float(np.min(vals)) if len(vals) else np.nan,
            "median": float(np.median(vals)) if len(vals) else np.nan,
            "p90": float(np.percentile(vals, 90)) if len(vals) else np.nan,
            "p95": float(np.percentile(vals, 95)) if len(vals) else np.nan,
            "p99": float(np.percentile(vals, 99)) if len(vals) else np.nan,
            "max": float(np.max(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def descriptor_balance_summary(s4: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    records = []
    for panel, df in s4.items():
        descriptor = df["descriptor"].iloc[0]
        for split in SPLIT_ORDER:
            vals = finite_array(df.loc[df["assigned_split"] == split, "value"])
            records.append({
                "panel": panel,
                "descriptor": descriptor,
                "split": split,
                "split_display": SPLIT_DISPLAY[split],
                "n": int(len(vals)),
                "median": float(np.median(vals)) if len(vals) else np.nan,
                "q25": float(np.percentile(vals, 25)) if len(vals) else np.nan,
                "q75": float(np.percentile(vals, 75)) if len(vals) else np.nan,
                "mean": float(np.mean(vals)) if len(vals) else np.nan,
                "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            })
    return pd.DataFrame(records)


def save_reports(cfg: Config, inputs: Dict[str, Any], s2, s3, s4) -> Tuple[Dict[str, pd.DataFrame], list[str]]:
    reports = {
        "split_count_validation_audit": split_count_validation(inputs, cfg),
        "leakage_audit_summary": pd.concat([s2["B"].assign(report_type="sampled_cross_split_jaccard"),
                                            s2["C"].assign(report_type="overlap_audit"),
                                            s2["D"].assign(report_type="cluster_counts")], ignore_index=True, sort=False),
        "nearest_train_similarity_summary": nearest_summary(inputs),
        "threshold_sensitivity_summary": inputs["threshold"].copy(),
        "descriptor_balance_summary": descriptor_balance_summary(s4),
    }
    warnings = []
    if not bool(reports["split_count_validation_audit"]["matches_expected"].all()):
        warnings.append("Recovered split counts do not match configured expected counts; check Results 3.2.")
    max_nn = reports["nearest_train_similarity_summary"]["max"].max()
    if pd.notna(max_nn) and max_nn > cfg.main_jaccard_threshold:
        warnings.append(
            f"Nearest-train audit contains values above threshold ({max_nn:.3f} > {cfg.main_jaccard_threshold:.2f}); "
            "do not claim complete removal of all near-neighbour similarity."
        )
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
        save_csv(df, cfg.tables_dir / f"{name}.csv")
    return reports, warnings


# =============================================================================
# Plotting
# =============================================================================

def plot_s2(s2: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    fig, axes = plt.subplots(2, 2, figsize=(10.9, 6.7))
    fig.subplots_adjust(left=0.085, right=0.985, top=0.92, bottom=0.14, wspace=0.36, hspace=0.43)

    ax = axes[0, 0]
    df = s2["A"]
    vals = pd.to_numeric(df["n_unique_sequences"], errors="coerce").fillna(0).to_numpy()
    x = np.arange(len(SPLIT_ORDER))
    bars = ax.bar(x, vals, width=0.58, color=[SPLIT_FACE[s] for s in SPLIT_ORDER],
                  edgecolor=[SPLIT_EDGE[s] for s in SPLIT_ORDER], hatch=[SPLIT_HATCH[s] for s in SPLIT_ORDER],
                  linewidth=0.85, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLIT_ORDER])
    ax.set_ylabel("Unique sequences"); ax.set_title("Split composition", pad=8)
    ymax = max(vals) if len(vals) else 1
    ax.set_ylim(0, ymax * 1.18)
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, value + ymax*0.025, f"{int(value):,}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    beautify_axis(ax); panel_letter(ax, "A", cfg)

    ax = axes[0, 1]
    df = s2["B"]
    vals = pd.to_numeric(df["max_jaccard"], errors="coerce").fillna(0).to_numpy()
    x = np.arange(len(PAIR_ORDER))
    bars = ax.bar(x, vals, width=0.58, color=[PAIR_FACE[p] for p in PAIR_ORDER],
                  edgecolor=[PAIR_EDGE[p] for p in PAIR_ORDER], linewidth=0.85, zorder=3)
    ax.axhline(cfg.main_jaccard_threshold, color=PLOS_PALETTE["axis_dark"], linestyle="--", linewidth=0.9)
    ax.text(0.98, cfg.main_jaccard_threshold + 0.018, f"threshold = {cfg.main_jaccard_threshold:.2f}",
            transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels(df["pair_display"], rotation=12, ha="right")
    ax.set_ylabel("Max sampled 3-mer Jaccard"); ax.set_title("Sampled cross-split similarity", pad=8)
    ax.set_ylim(0, max(0.65, float(np.nanmax(vals))*1.25 if len(vals) else 0.65))
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.018, f"{value:.3f}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    beautify_axis(ax); panel_letter(ax, "B", cfg)

    ax = axes[1, 0]
    overlap = s2["C"]
    width = 0.34
    x = np.arange(len(PAIR_ORDER))
    for offset, overlap_type, label, hatch in [
        (-width/2, "exact_sequence_overlap", "Exact sequence", ""),
        (width/2, "cluster_overlap", "Similarity cluster", "///"),
    ]:
        sub = overlap.loc[overlap["overlap_type"] == overlap_type].set_index(["split_a", "split_b"]).reindex(PAIR_ORDER).reset_index()
        vals = pd.to_numeric(sub["n_overlap"], errors="coerce").fillna(0).to_numpy()
        bars = ax.bar(x + offset, vals, width=width, color=OVERLAP_FACE[overlap_type],
                      edgecolor=OVERLAP_EDGE[overlap_type], hatch=hatch, linewidth=0.75, label=label, zorder=3)
        for bar, value in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, value + 0.04, f"{int(value)}",
                    ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels([PAIR_DISPLAY[p] for p in PAIR_ORDER], rotation=12, ha="right")
    ax.set_ylabel("Overlap count"); ax.set_title("Exact and cluster-overlap audit", pad=8)
    ax.set_ylim(0, 1.05)
    ax.text(
        0.02,
        0.93,
        "all counts = 0",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=cfg.annotation_size,
        color=PLOS_PALETTE["axis_dark"],
    )
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="upper right")
    beautify_axis(ax); panel_letter(ax, "C", cfg)

    ax = axes[1, 1]
    df = s2["D"]
    vals = pd.to_numeric(df["n_clusters"], errors="coerce").fillna(0).to_numpy()
    x = np.arange(len(SPLIT_ORDER))
    bars = ax.bar(x, vals, width=0.58, color=[SPLIT_FACE[s] for s in SPLIT_ORDER],
                  edgecolor=[SPLIT_EDGE[s] for s in SPLIT_ORDER], hatch=[SPLIT_HATCH[s] for s in SPLIT_ORDER],
                  linewidth=0.85, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLIT_ORDER])
    ax.set_ylabel("Similarity clusters"); ax.set_title("Cluster assignment by split", pad=8)
    ymax = max(vals) if len(vals) else 1
    ax.set_ylim(0, ymax * 1.18)
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, value + ymax*0.025, f"{int(value):,}",
                ha="center", va="bottom", fontsize=cfg.annotation_size)
    beautify_axis(ax); panel_letter(ax, "D", cfg)

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S2_redesigned", cfg)


def plot_s3(s3: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    fig, axes = plt.subplots(2, 2, figsize=(11.0, 6.8))
    fig.subplots_adjust(left=0.085, right=0.985, top=0.92, bottom=0.14, wspace=0.36, hspace=0.43)

    ax = axes[0, 0]
    ecdf = s3["A"]
    quant = s3["A_quantiles"]
    for split in ["val", "test"]:
        sub = ecdf.loc[ecdf["query_split"] == split].sort_values("nearest_train_jaccard")
        if len(sub):
            ax.step(sub["nearest_train_jaccard"], sub["ecdf"], where="post", color=SPLIT_EDGE[split],
                    linewidth=1.6, label=f"{SPLIT_DISPLAY[split]} (n={int(sub['n_sequences'].iloc[0]):,})", zorder=3)
            qsub = quant.loc[(quant["query_split"] == split) & (quant["quantile"].isin(["median", "p90"]))]
            for _, row in qsub.iterrows():
                ax.axvline(row["nearest_train_jaccard"], color=SPLIT_EDGE[split],
                           linestyle="--" if row["quantile"] == "median" else ":", linewidth=0.85, alpha=0.8, zorder=2)
    ax.axvline(cfg.main_jaccard_threshold, color=PLOS_PALETTE["axis_dark"], linestyle="-.", linewidth=0.9, alpha=0.9)
    ax.text(cfg.main_jaccard_threshold + 0.01, 0.08, "0.55", rotation=90, fontsize=cfg.annotation_size,
            va="bottom", ha="left", color=PLOS_PALETTE["axis_dark"])
    ax.set_xlabel("Nearest-train 3-mer Jaccard"); ax.set_ylabel("Cumulative fraction")
    ax.set_title("Nearest-train similarity distribution", pad=8)
    ax.set_xlim(0, max(0.70, float(ecdf["nearest_train_jaccard"].max())*1.05 if len(ecdf) else 0.70))
    ax.set_ylim(0, 1.02)
    split_leg = ax.legend(
        frameon=True,
        edgecolor=PLOS_PALETTE["light_gray"],
        facecolor="white",
        loc="lower right",
        title="Held-out split",
        title_fontsize=cfg.legend_size,
    )
    ax.add_artist(split_leg)
    line_handles = [
        Line2D([0], [0], color=PLOS_PALETTE["axis_dark"], linestyle="--", linewidth=0.9, label="median"),
        Line2D([0], [0], color=PLOS_PALETTE["axis_dark"], linestyle=":", linewidth=0.9, label="90th percentile"),
        Line2D([0], [0], color=PLOS_PALETTE["axis_dark"], linestyle="-.", linewidth=0.9, label="0.55 threshold"),
    ]
    ax.legend(
        handles=line_handles,
        frameon=True,
        edgecolor=PLOS_PALETTE["light_gray"],
        facecolor="white",
        loc="center right",
        fontsize=max(cfg.legend_size - 0.5, 6.8),
    )
    beautify_axis(ax); panel_letter(ax, "A", cfg)

    ax = axes[0, 1]
    ex = s3["B"]
    thresholds = sorted(ex["threshold"].dropna().unique())
    x = np.arange(len(thresholds)); width = 0.34
    for offset, split in [(-width/2, "val"), (width/2, "test")]:
        sub = ex.loc[ex["query_split"] == split].set_index("threshold").reindex(thresholds)
        vals = pd.to_numeric(sub["fraction_exceeding"], errors="coerce").fillna(0).to_numpy()
        bars = ax.bar(
            x + offset,
            vals,
            width=width,
            color=SPLIT_FACE[split],
            edgecolor=SPLIT_EDGE[split],
            hatch=SPLIT_HATCH[split],
            linewidth=0.8,
            label=SPLIT_DISPLAY[split],
            zorder=3,
        )
        for bar, value in zip(bars, vals):
            if np.isfinite(value):
                y = value + 0.018 if value >= 0.025 else 0.018
                label = f"{100 * value:.1f}%" if value < 0.10 else f"{100 * value:.0f}%"
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y,
                    label,
                    ha="center",
                    va="bottom",
                    fontsize=max(cfg.annotation_size - 0.6, 6.8),
                    rotation=0,
                )
    ax.set_xticks(x); ax.set_xticklabels([f"{t:.2f}" for t in thresholds])
    ax.set_xlabel("Nearest-train Jaccard threshold"); ax.set_ylabel("Fraction ≥ threshold")
    ax.set_ylim(0, 1.0); ax.set_title("High-similarity tail", pad=8)
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="upper right")
    beautify_axis(ax); panel_letter(ax, "B", cfg)

    ax = axes[1, 0]
    edges = s3["C"].copy().sort_values("threshold")
    ax.plot(edges["threshold"], edges["n_edges"], marker="o", markersize=4.5, linewidth=1.5,
            color=PLOS_PALETTE["charcoal_brown"], markerfacecolor=PLOS_PALETTE["charcoal_brown"],
            markeredgecolor=PLOS_PALETTE["axis_dark"], zorder=3)
    edge_vals = pd.to_numeric(edges["n_edges"], errors="coerce").dropna()
    if len(edge_vals):
        y_min = max(0, float(edge_vals.min()) * 0.90)
        y_max = float(edge_vals.max()) * 1.14
        ax.set_ylim(y_min, y_max)
        label_offset = (y_max - y_min) * 0.025
    else:
        label_offset = 1.0
    for _, row in edges.iterrows():
        ax.text(
            row["threshold"],
            row["n_edges"] + label_offset,
            f"{int(row['n_edges']):,}",
            ha="center",
            va="bottom",
            fontsize=cfg.annotation_size,
        )
    ax.set_xlabel("3-mer Jaccard threshold"); ax.set_ylabel("Retained similarity edges")
    ax.set_title("Similarity-graph sensitivity", pad=8)
    beautify_axis(ax); panel_letter(ax, "C", cfg)

    ax = axes[1, 1]
    sizes = s3["D"]
    x_offsets = {"train": -0.0025, "val": 0.0000, "test": 0.0025}
    markers = {"train": "o", "val": "s", "test": "^"}
    for split in SPLIT_ORDER:
        sub = sizes.loc[sizes["split"] == split].sort_values("threshold")
        xvals = pd.to_numeric(sub["threshold"], errors="coerce").to_numpy() + x_offsets[split]
        yvals = pd.to_numeric(sub["n_unique_sequences"], errors="coerce").to_numpy()
        ax.plot(
            xvals,
            yvals,
            marker=markers[split],
            markersize=4.6,
            linewidth=1.5,
            color=SPLIT_EDGE[split],
            markerfacecolor=SPLIT_FACE[split],
            markeredgecolor=SPLIT_EDGE[split],
            label=SPLIT_DISPLAY[split],
            zorder=4 if split == "test" else 3,
        )
    val_values = sizes.loc[sizes["split"].isin(["val", "test"]), "n_unique_sequences"].dropna().unique()
    if len(val_values) == 1:
        ax.text(
            0.98,
            0.08,
            f"Validation = test = {int(val_values[0]):,}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=cfg.annotation_size,
            color=PLOS_PALETTE["axis_dark"],
        )
    ax.set_xlabel("3-mer Jaccard threshold"); ax.set_ylabel("Unique sequences")
    ax.set_title("Split-size stability", pad=8)
    ax.legend(frameon=True, edgecolor=PLOS_PALETTE["light_gray"], facecolor="white", loc="best")
    beautify_axis(ax); panel_letter(ax, "D", cfg)

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S3_redesigned", cfg)


def safe_boxplot(ax: plt.Axes, data: list[np.ndarray], positions: np.ndarray, edge_colors: list[str]) -> None:
    bp = ax.boxplot(
        data, positions=positions, widths=0.16, patch_artist=True, showfliers=False,
        medianprops=dict(color=PLOS_PALETTE["axis_dark"], linewidth=1.15),
        whiskerprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        capprops=dict(color=PLOS_PALETTE["charcoal_brown"], linewidth=0.75),
        boxprops=dict(linewidth=0.75),
    )
    for box, color in zip(bp["boxes"], edge_colors):
        box.set_facecolor("white")
        box.set_edgecolor(color)
        box.set_alpha(0.98)
        box.set_zorder(4)


def split_violin_box(ax: plt.Axes, panel_df: pd.DataFrame, ylabel: str, title: str, cfg: Config, zero_line: bool = False) -> None:
    positions = np.arange(1, len(SPLIT_ORDER) + 1)
    data = [finite_array(panel_df.loc[panel_df["assigned_split"] == split, "value"]) for split in SPLIT_ORDER]
    vp = ax.violinplot(data, positions=positions, widths=0.62, showmeans=False, showmedians=False, showextrema=False)
    for body, split in zip(vp["bodies"], SPLIT_ORDER):
        body.set_facecolor(SPLIT_FACE[split])
        body.set_edgecolor(SPLIT_EDGE[split])
        body.set_alpha(0.38)
        body.set_linewidth(0.9)
        body.set_zorder(3)
    safe_boxplot(ax, data, positions, [SPLIT_EDGE[s] for s in SPLIT_ORDER])
    if zero_line:
        ax.axhline(0, color=PLOS_PALETTE["charcoal_brown"], linestyle="--", linewidth=0.75, zorder=2)
    ax.set_xticks(positions)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLIT_ORDER])
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    beautify_axis(ax)


def plot_s4(s4: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    fig, axes = plt.subplots(2, 2, figsize=(9.9, 6.6))
    fig.subplots_adjust(left=0.095, right=0.985, top=0.92, bottom=0.15, wspace=0.35, hspace=0.43)
    specs = [
        ("A", axes[0, 0], "Length (aa)", "Length", False),
        ("B", axes[0, 1], "Net charge at pH 7", "Net charge", True),
        ("C", axes[1, 0], "Hydrophobicity\n(Kyte–Doolittle)", "Hydrophobicity", True),
        ("D", axes[1, 1], "Shannon entropy (bits)", "Shannon entropy", False),
    ]
    for panel, ax, ylabel, title, zero in specs:
        split_violin_box(ax, s4[panel], ylabel, title, cfg, zero_line=zero)
        panel_letter(ax, panel, cfg)
    # Legend patches intentionally omit hatching because the violin bodies are not hatched.
    fig.legend(handles=split_legend_handles(include_hatches=False), loc="lower center", ncol=3, frameon=True,
               bbox_to_anchor=(0.5, 0.045), fontsize=cfg.legend_size, edgecolor=PLOS_PALETTE["light_gray"],
               facecolor=PLOS_PALETTE["background"], handlelength=1.5, handletextpad=0.55,
               columnspacing=1.35, borderpad=0.35)
    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S4_redesigned", cfg)


# =============================================================================
# Reproducibility files and manifest
# =============================================================================

def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env_commit = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env_commit:
        return env_commit
    try:
        result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(cfg.source_step2_dir),
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5, check=False)
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        return None
    return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    file_name = globals().get("__file__", None)
    if file_name and Path(file_name).exists():
        script_path = str(Path(file_name).resolve())
        text = Path(file_name).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session. Save this notebook cell/script manually.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_02_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_02_PLOS_redesign_code.py")
    return script_path, script_hash


def write_requirements_summary(cfg: Config) -> Path:
    path = cfg.reports_dir / "requirements_step_02_minimal.txt"
    save_text(f"python=={platform.python_version()}\nnumpy=={np.__version__}\npandas=={pd.__version__}\nmatplotlib=={mpl.__version__}\n", path)
    return path


def write_main_figure_note(cfg: Config) -> Path:
    text = """# Step 2 main-figure note

Step 2 is intentionally supplementary-only in the OncoPep PLOS Computational
Biology figure-redesign workflow.

Rationale:
- Step 2 provides technical validation of similarity-aware data partitioning,
  leakage auditing, threshold sensitivity, and descriptor preservation.
- These analyses are essential for reproducibility and reviewer confidence, but
  they support the modelling workflow rather than serving as the central model
  contribution.
- Main Figure 2 is reserved for the OncoPep architecture and mathematical
  workflow, which should remain the visual centerpiece of the algorithmic
  contribution.

Manuscript implication:
- Results 3.2 should cite Supplementary Figures S2-S4.
- Main Figure 2 should not be cited for Step 2 partitioning.
"""
    path = cfg.main_note_dir / "README_no_main_figure_for_step_02.txt"
    save_text(text, path)
    return path


def write_readme(cfg: Config, warnings: Sequence[str]) -> Path:
    text = f"""# OncoPep Step 2 PLOS supplementary-figure package

Generated: {datetime.now().isoformat(timespec="seconds")}

Input directory:
{cfg.source_step2_dir}

Output directory:
{cfg.output_root}

Generated figures:
- supplementary_figures/Supplementary_Figure_S2_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S3_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S4_redesigned.png/.pdf/.tiff

Scientific scope:
- Supplementary Figure S2: split composition, sampled cross-split 3-mer Jaccard,
  exact/cluster-overlap audit, and cluster assignment by split.
- Supplementary Figure S3: nearest-train 3-mer Jaccard ECDF, threshold exceedance,
  retained similarity edges, and split-size stability.
- Supplementary Figure S4: descriptor preservation across train, validation, and
  test partitions.

Manuscript notes:
- Use recovered split counts: train=24,485, validation=13,260, test=13,260.
- Do not claim complete elimination of all near-neighbour similarity; nearest-train
  auditing may show a high-similarity tail.
- Step 2 is supplementary-only; Main Figure 2 is reserved for the OncoPep
  architecture and mathematical workflow.

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
"""
    path = cfg.reports_dir / "README_step_02_outputs.txt"
    save_text(text, path)
    return path


def create_root_level_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs = []
    for fig in ["S2", "S3", "S4"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv",
                      cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))
    pairs.append((cfg.code_dir / "OncoPep_step_02_PLOS_redesign_code.py",
                  cfg.output_root / "OncoPep_step_02_PLOS_redesign_code.py"))
    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def build_manifest(cfg: Config, inputs: Dict[str, Any], s2_paths, s3_paths, s4_paths,
                   reports, warnings, readme_path, main_note_path, requirements_path,
                   script_path, script_hash, root_copies) -> Dict[str, Any]:
    input_paths = inputs["_paths"]
    return {
        "step": "step_02",
        "scope": "supplementary_only",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "rerun_split_construction": False,
        "main_figure_2_generated": False,
        "main_figure_2_rationale": "Reserved for OncoPep architecture and mathematical workflow.",
        "source_step2_dir": str(cfg.source_step2_dir),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "input_files": {k: str(v) for k, v in input_paths.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_paths.items()},
        "figures": {"Supplementary_Figure_S2": s2_paths,
                    "Supplementary_Figure_S3": s3_paths,
                    "Supplementary_Figure_S4": s4_paths},
        "source_data": {"S2_all": str(cfg.source_data_dir / "Supplementary_Figure_S2_source_data_all_panels.csv"),
                        "S3_all": str(cfg.source_data_dir / "Supplementary_Figure_S3_source_data_all_panels.csv"),
                        "S4_all": str(cfg.source_data_dir / "Supplementary_Figure_S4_source_data_all_panels.csv")},
        "reports": {name: str(cfg.reports_dir / f"{name}.csv") for name in reports},
        "readme": str(readme_path),
        "main_figure_note": str(main_note_path),
        "requirements_summary": str(requirements_path),
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(root_copies),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# =============================================================================
# Main workflow and entry points
# =============================================================================

def run_step2_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_output_tree(cfg)
    set_style(cfg)
    inputs = load_inputs(cfg)

    s2, s2_all = build_s2_source_data(inputs)
    s3, s3_all = build_s3_source_data(inputs)
    s4, s4_all = build_s4_source_data(inputs)

    save_source_data(cfg, s2, s2_all, s3, s3_all, s4, s4_all)
    reports, warnings = save_reports(cfg, inputs, s2, s3, s4)

    s2_paths = plot_s2(s2, cfg)
    s3_paths = plot_s3(s3, cfg)
    s4_paths = plot_s4(s4, cfg)

    script_path, script_hash = code_snapshot(cfg)
    requirements_path = write_requirements_summary(cfg)
    main_note_path = write_main_figure_note(cfg)
    readme_path = write_readme(cfg, warnings)
    root_copies = create_root_level_copies(cfg)

    manifest = build_manifest(cfg, inputs, s2_paths, s3_paths, s4_paths, reports, warnings,
                              readme_path, main_note_path, requirements_path, script_path,
                              script_hash, root_copies)
    save_json(manifest, cfg.reports_dir / "step_02_manifest.json")
    save_json(manifest, cfg.output_root / "step_02_manifest.json")

    print("\n✅ Step 2 PLOS supplementary-figure package generated successfully.")
    print(f"Source Step 2 directory: {cfg.source_step2_dir}")
    print(f"Output directory: {cfg.output_root}")
    for label, paths in [("Supplementary Figure S2", s2_paths),
                         ("Supplementary Figure S3", s3_paths),
                         ("Supplementary Figure S4", s4_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")
    print("\nReports:")
    for name in reports:
        print(f"  - {cfg.reports_dir / f'{name}.csv'}")
    print(f"  - {cfg.reports_dir / 'step_02_manifest.json'}")
    print(f"  - {cfg.reports_dir / 'README_step_02_outputs.txt'}")
    if warnings:
        print("\n⚠️ Warnings:")
        for w in warnings:
            print(f"  - {w}")

    return {"s2_paths": s2_paths, "s3_paths": s3_paths, "s4_paths": s4_paths,
            "s2_panels": s2, "s3_panels": s3, "s4_panels": s4,
            "reports": reports, "warnings": warnings, "manifest": manifest}


def main_notebook(
    source_step2_dir: str | Path = SOURCE_STEP2_DIR_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    main_jaccard_threshold: float = 0.55,
    create_root_level_copies: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        source_step2_dir=Path(source_step2_dir),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        main_jaccard_threshold=main_jaccard_threshold,
        create_root_level_copies=create_root_level_copies,
        git_commit=git_commit,
    )
    return run_step2_redesign(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate OncoPep Step 2 PLOS supplementary figures from existing Step 2 outputs."
    )
    parser.add_argument("--source_step2_dir", default=str(SOURCE_STEP2_DIR_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--main_jaccard_threshold", type=float, default=0.55)
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    run_step2_redesign(
        Config(
            source_step2_dir=Path(args.source_step2_dir),
            base_dir=Path(args.base_dir),
            output_root_name=args.output_root_name,
            main_jaccard_threshold=args.main_jaccard_threshold,
            create_root_level_copies=not args.no_root_copies,
            git_commit=args.git_commit,
        )
    )

OncoPep Step 3 — PLOS Computational Biology supplementary-figure redesign.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 3 — PLOS Computational Biology supplementary-figure redesign.

Step 3 scope
------------
This step is supplementary-only. It audits the conditioning schema and
condition-support structure used for property-conditioned peptide generation.

Generated outputs:
  Supplementary Figure S5 — condition descriptor distributions across partitions
  Supplementary Figure S6 — train-derived condition-bin balance across partitions

This script does NOT generate Main Figure 2. Main Figure 2 is reserved for the
OncoPep architecture/model workflow in a later manuscript step.

Main improvements in this version
---------------------------------
1. Strict validation of primary condition schemas and low/mid/high bins.
2. Explicit sparse/zero-bin support audit.
3. Stronger validation of loaded condition-balance tables.
4. Separate informational notes from true warnings.
5. Clearer entropy handling as a secondary/supporting descriptor.
6. Optional README explaining why no main figure is generated in Step 3.

Notebook usage
--------------
results = main_notebook(
    project_root="/home/data3/Moe/nature_computational2",
    base_dir="/home/data3/Moe/nature_computational_peponco/PLOS",
    output_root_name="plos_comp/step_03",
)

Terminal usage
--------------
python OncoPep_step_03_PLOS_redesign_code.py
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


# =============================================================================
# 1. Configuration
# =============================================================================

PROJECT_ROOT_DEFAULT = Path("/home/data3/Moe/nature_computational2")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_03"
STEP3_RELATIVE_DIR_DEFAULT = Path("redesign/step3")

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
BIN_ORDER = ("low", "mid", "high")
BIN_DISPLAY = {"low": "Low", "mid": "Mid", "high": "High"}

PRIMARY_CONDITIONS = ("length", "net_charge_pH7", "hydrophobicity_KD")
SECONDARY_DESCRIPTORS = ("entropy",)

DESCRIPTOR_ALIASES = {
    "length": ["length", "seq_length", "peptide_length", "sequence_length"],
    "net_charge_pH7": ["net_charge_pH7", "net_charge", "charge", "charge_pH7"],
    "hydrophobicity_KD": ["hydrophobicity_KD", "hydrophobicity", "hydrophobicity_kd", "kd_hydrophobicity", "mean_hydropathy"],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy"],
}
DESCRIPTOR_LABELS = {
    "length": "Length",
    "net_charge_pH7": "Net charge at pH 7",
    "hydrophobicity_KD": "Kyte–Doolittle hydrophobicity",
    "entropy": "Shannon entropy",
}
DESCRIPTOR_AXIS_LABELS = {
    "length": "Length (aa)",
    "net_charge_pH7": "Net charge at pH 7",
    "hydrophobicity_KD": "Mean KD hydrophobicity",
    "entropy": "Shannon entropy (bits)",
}


@dataclass
class Config:
    """Editable Step 3 configuration."""

    project_root: Path = PROJECT_ROOT_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT
    step3_relative_dir: Path = STEP3_RELATIVE_DIR_DEFAULT

    # Optional explicit inputs override candidate-path search.
    source_sequence_file: Optional[Path] = None
    source_schema_file: Optional[Path] = None
    source_balance_file: Optional[Path] = None

    primary_conditions: Tuple[str, ...] = PRIMARY_CONDITIONS
    secondary_descriptors: Tuple[str, ...] = SECONDARY_DESCRIPTORS
    split_order: Tuple[str, ...] = SPLIT_ORDER
    bin_order: Tuple[str, ...] = BIN_ORDER

    # Sparse-bin audit settings.
    sparse_min_n: int = 10
    sparse_min_fraction: float = 0.005
    fail_on_missing_primary_bin: bool = True

    # Export settings.
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    create_no_main_figure_readme: bool = True
    git_commit: Optional[str] = None

    # Typography.
    font_size: float = 8.3
    title_size: float = 9.2
    axis_label_size: float = 8.9
    tick_label_size: float = 7.8
    legend_size: float = 7.7
    annotation_size: float = 7.5
    panel_letter_size: float = 13.0

    @property
    def step3_source_root(self) -> Path:
        return self.project_root / self.step3_relative_dir

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_figure_dir(self) -> Path:
        return self.output_root / "main_figure"


# =============================================================================
# 2. Approved PLOS-style palette
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

SPLIT_FACE = {"train": PLOS["blue"], "val": PLOS["mint"], "test": PLOS["coral"]}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}

BIN_FACE = {"low": PLOS["blue"], "mid": PLOS["mint"], "high": PLOS["coral"]}
BIN_EDGE = {"low": "#176F8A", "mid": "#6F9F7A", "high": "#A84F42"}

SUPPORT_FACE = {"primary": PLOS["blue"], "full": PLOS["brown"]}
SUPPORT_EDGE = {"primary": "#176F8A", "full": "#8F5F37"}


# =============================================================================
# 3. General utilities
# =============================================================================

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def ensure_dirs(cfg: Config) -> None:
    for path in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)
    if cfg.create_no_main_figure_readme:
        cfg.main_figure_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=json_default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def finite_array(values: Any) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def normalize_split_values(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for cand in candidates:
        if cand in df.columns:
            return cand
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"Could not resolve column from {list(candidates)}. Available columns: {list(df.columns)}")
    return None


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


def checked_numeric(df: pd.DataFrame, col: str, min_finite_fraction: float = 0.90) -> None:
    vals = pd.to_numeric(df[col], errors="coerce")
    finite_fraction = float(np.isfinite(vals.to_numpy(dtype=float)).mean())
    if finite_fraction < min_finite_fraction:
        raise ValueError(
            f"Column '{col}' has too few finite numeric values "
            f"({finite_fraction:.3f} < {min_finite_fraction:.3f})."
        )


# =============================================================================
# 4. Plot style
# =============================================================================

def set_plot_style(cfg: Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_label_size,
            "ytick.labelsize": cfg.tick_label_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": 0.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS["white"],
            "axes.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.45)
    ax.grid(False, axis="x")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.75)
        ax.spines[side].set_color(PLOS["dark"])


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.08) -> None:
    ax.text(
        dx,
        dy,
        str(letter).upper(),
        transform=ax.transAxes,
        fontsize=cfg.panel_letter_size,
        fontweight="bold",
        va="top",
        ha="left",
        color=PLOS["dark"],
    )


def save_figure(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "png": base.with_suffix(".png"),
        "pdf": base.with_suffix(".pdf"),
        "tiff": base.with_suffix(".tiff"),
    }
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


# =============================================================================
# 5. Input loading, descriptor harmonization, and schema construction
# =============================================================================

def candidate_sequence_paths(cfg: Config) -> list[Path]:
    candidates = []
    if cfg.source_sequence_file:
        candidates.append(Path(cfg.source_sequence_file))
    candidates.extend(
        [
            cfg.step3_source_root / "tables" / "step3_conditioned_unique_sequences.csv",
            cfg.project_root / "step3" / "tables" / "step3_conditioned_unique_sequences.csv",
            cfg.project_root / "step2" / "splits" / "step2_unique_sequences_with_splits.csv",
            Path("/home/data3/Moe/nature_computational_peponco/step2/splits/step2_unique_sequences_with_splits.csv"),
        ]
    )
    return candidates


def candidate_schema_paths(cfg: Config) -> list[Path]:
    candidates = []
    if cfg.source_schema_file:
        candidates.append(Path(cfg.source_schema_file))
    candidates.extend(
        [
            cfg.step3_source_root / "schemas" / "step3_condition_schema.json",
            cfg.project_root / "step3" / "schemas" / "step3_condition_schema.json",
        ]
    )
    return candidates


def candidate_balance_paths(cfg: Config) -> list[Path]:
    candidates = []
    if cfg.source_balance_file:
        candidates.append(Path(cfg.source_balance_file))
    candidates.extend(
        [
            cfg.step3_source_root / "tables" / "step3_condition_balance_by_split.csv",
            cfg.project_root / "step3" / "tables" / "step3_condition_balance_by_split.csv",
        ]
    )
    return candidates


def harmonize_descriptor_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    lower_map = {str(c).lower(): c for c in out.columns}
    for target, aliases in DESCRIPTOR_ALIASES.items():
        found = None
        for alias in aliases:
            if alias.lower() in lower_map:
                found = lower_map[alias.lower()]
                break
        if found is not None and found != target:
            out.rename(columns={found: target}, inplace=True)
        if target in out.columns:
            out[target] = pd.to_numeric(out[target], errors="coerce")
    return out


def validate_sequence_table(df: pd.DataFrame, split_col: str, cfg: Config) -> None:
    required_descriptors = list(cfg.primary_conditions) + list(cfg.secondary_descriptors)
    missing_descriptors = [c for c in required_descriptors if c not in df.columns]
    if missing_descriptors:
        raise ValueError(f"Missing descriptor column(s) after harmonization: {missing_descriptors}. Available columns: {list(df.columns)}")

    observed_splits = set(df[split_col].dropna().unique())
    missing_splits = [s for s in cfg.split_order if s not in observed_splits]
    if missing_splits:
        raise ValueError(f"Missing expected split labels {missing_splits}; observed labels={sorted(observed_splits)}")

    for desc in required_descriptors:
        checked_numeric(df, desc, min_finite_fraction=0.90)


def load_conditioned_sequences(cfg: Config) -> Tuple[pd.DataFrame, str, Path]:
    path = first_existing(candidate_sequence_paths(cfg))
    if path is None:
        raise FileNotFoundError("No sequence-level file found. Checked:\n- " + "\n- ".join(str(p) for p in candidate_sequence_paths(cfg)))
    df = pd.read_csv(path, low_memory=False)
    split_col = resolve_column(df, ["assigned_split", "split", "data_split"])
    df[split_col] = normalize_split_values(df[split_col])
    if split_col != "assigned_split":
        df.rename(columns={split_col: "assigned_split"}, inplace=True)
        split_col = "assigned_split"
    df = harmonize_descriptor_columns(df)
    validate_sequence_table(df, split_col, cfg)
    return df, split_col, path


def robust_quantile_edges(series: pd.Series, n_bins: int = 3) -> np.ndarray:
    vals = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        raise ValueError("Cannot compute quantile edges from an empty descriptor series.")
    q = np.linspace(0.0, 1.0, n_bins + 1)
    try:
        edges = np.quantile(vals, q, method="linear")
    except TypeError:
        edges = np.quantile(vals, q, interpolation="linear")
    edges = np.asarray(edges, dtype=float)
    edges = np.maximum.accumulate(edges)
    if np.unique(edges).size < n_bins + 1:
        unique_vals = np.unique(vals)
        if len(unique_vals) < n_bins:
            raise ValueError("Not enough unique values to construct low/mid/high bins.")
        fallback_q = np.linspace(0, len(unique_vals) - 1, n_bins + 1)
        edges = np.interp(fallback_q, np.arange(len(unique_vals)), unique_vals)
    edges[0] -= 1e-9
    edges[-1] += 1e-9
    return np.maximum.accumulate(edges)


def assign_bins(series: pd.Series, edges: Sequence[float], labels: Sequence[str] = BIN_ORDER) -> pd.Series:
    return pd.cut(
        pd.to_numeric(series, errors="coerce"),
        bins=np.asarray(edges, dtype=float),
        labels=list(labels),
        include_lowest=True,
        right=False,
    ).astype("object")


def recompute_schema_from_training(df: pd.DataFrame, split_col: str, cfg: Config) -> Dict[str, Any]:
    train_df = df.loc[df[split_col] == "train"].copy()
    if train_df.empty:
        raise ValueError("No training rows found; cannot compute train-derived condition-bin thresholds.")

    schema = {"conditions": {}}
    for cond in list(cfg.primary_conditions) + [c for c in cfg.secondary_descriptors if c in df.columns]:
        if cond not in df.columns:
            continue
        edges = robust_quantile_edges(train_df[cond], n_bins=3)
        train_bins = assign_bins(train_df[cond], edges, cfg.bin_order)
        train_counts = pd.Series(train_bins).value_counts().reindex(cfg.bin_order, fill_value=0)
        schema["conditions"][cond] = {
            "labels": list(cfg.bin_order),
            "edges": [float(v) for v in edges],
            "train_bin_counts": {str(k): int(v) for k, v in train_counts.items()},
            "fit_population": "train_only",
            "is_primary": bool(cond in cfg.primary_conditions),
        }
    return schema


def validate_schema_integrity(schema: Dict[str, Any], cfg: Config) -> list[str]:
    """Validate primary condition schema strictly and return non-fatal notes/warnings."""
    warnings = []
    if "conditions" not in schema or not isinstance(schema["conditions"], dict):
        raise ValueError("Condition schema is malformed: missing top-level 'conditions' dictionary.")

    for cond in cfg.primary_conditions:
        if cond not in schema["conditions"]:
            raise ValueError(f"Primary condition '{cond}' is missing from the condition schema.")
        meta = schema["conditions"][cond]
        labels = [str(x).lower() for x in meta.get("labels", [])]
        if list(labels) != list(cfg.bin_order):
            raise ValueError(f"Primary condition '{cond}' must have labels {list(cfg.bin_order)}, found {labels}.")
        edges = meta.get("edges", [])
        if len(edges) != len(cfg.bin_order) + 1:
            raise ValueError(f"Primary condition '{cond}' must have {len(cfg.bin_order)+1} bin edges, found {len(edges)}.")
        edges_arr = np.asarray(edges, dtype=float)
        if not np.all(np.isfinite(edges_arr)):
            raise ValueError(f"Primary condition '{cond}' has non-finite bin edges.")
        if not np.all(np.diff(edges_arr) >= 0):
            raise ValueError(f"Primary condition '{cond}' has non-monotonic bin edges.")
        if bool(meta.get("is_primary", cond in cfg.primary_conditions)) is not True:
            warnings.append(f"Primary condition '{cond}' was not marked primary in loaded schema; treated as primary by configuration.")

    for desc in cfg.secondary_descriptors:
        if desc in schema["conditions"]:
            meta = schema["conditions"][desc]
            if bool(meta.get("is_primary", False)):
                warnings.append(f"Secondary descriptor '{desc}' is marked primary in loaded schema. Confirm model-conditioning definition.")
        else:
            warnings.append(f"Secondary descriptor '{desc}' is absent from schema; it will be shown in S5 but not used for full descriptor-bin IDs.")

    return warnings


def load_or_build_schema(df: pd.DataFrame, split_col: str, cfg: Config) -> Tuple[Dict[str, Any], Optional[Path], bool, list[str]]:
    path = first_existing(candidate_schema_paths(cfg))
    if path is not None:
        with path.open("r", encoding="utf-8") as f:
            schema = json.load(f)
        warnings = validate_schema_integrity(schema, cfg)
        return schema, path, False, warnings
    schema = recompute_schema_from_training(df, split_col, cfg)
    warnings = validate_schema_integrity(schema, cfg)
    return schema, None, True, warnings


def ensure_bin_columns(df: pd.DataFrame, schema: Dict[str, Any]) -> pd.DataFrame:
    out = df.copy()
    for cond, meta in schema.get("conditions", {}).items():
        if cond not in out.columns:
            continue
        bin_col = f"{cond}_bin"
        if bin_col not in out.columns:
            labels = [str(x).lower() for x in meta.get("labels", list(BIN_ORDER))]
            edges = meta.get("edges")
            if edges is None:
                continue
            out[bin_col] = assign_bins(out[cond], edges, labels)
        out[bin_col] = out[bin_col].astype(str).str.lower().replace({"nan": np.nan, "none": np.nan})
    return out


def infer_condition_balance(df: pd.DataFrame, split_col: str, schema: Dict[str, Any], cfg: Config) -> pd.DataFrame:
    rows = []
    for cond, meta in schema.get("conditions", {}).items():
        bin_col = f"{cond}_bin"
        if bin_col not in df.columns:
            continue
        levels = [str(x).lower() for x in meta.get("labels", cfg.bin_order)]
        for split in cfg.split_order:
            sub = df.loc[df[split_col] == split].copy()
            total = len(sub)
            counts = sub[bin_col].astype("object").value_counts(dropna=False)
            for level in levels:
                n = int(counts.get(level, 0))
                rows.append(
                    {
                        "condition": cond,
                        "condition_display": DESCRIPTOR_LABELS.get(cond, cond),
                        "split": split,
                        "split_display": SPLIT_DISPLAY.get(split, split),
                        "level": str(level).lower(),
                        "level_display": BIN_DISPLAY.get(str(level).lower(), str(level)),
                        "n_sequences": n,
                        "split_total": int(total),
                        "fraction": float(n / total) if total else np.nan,
                        "is_primary": bool(meta.get("is_primary", cond in cfg.primary_conditions)),
                    }
                )
    return pd.DataFrame(rows)


def validate_balance_table(balance: pd.DataFrame, cfg: Config, source: str = "balance table") -> None:
    required = {"condition", "split", "level"}
    missing = sorted(required - set(balance.columns))
    if missing:
        raise ValueError(f"{source} is missing required columns: {missing}")
    if "fraction" not in balance.columns and "n_sequences" not in balance.columns:
        raise ValueError(f"{source} must contain either 'fraction' or 'n_sequences'.")

    balance_conditions = set(balance["condition"].astype(str))
    missing_primary = [c for c in cfg.primary_conditions if c not in balance_conditions]
    if missing_primary:
        raise ValueError(f"{source} is missing primary conditions required for S6: {missing_primary}")

    for cond in cfg.primary_conditions:
        sub = balance.loc[balance["condition"] == cond].copy()
        observed_pairs = set(zip(sub["split"].astype(str), sub["level"].astype(str).str.lower()))
        required_pairs = {(split, level) for split in cfg.split_order for level in cfg.bin_order}
        missing_pairs = sorted(required_pairs - observed_pairs)
        if missing_pairs:
            raise ValueError(f"{source} missing split/bin entries for primary condition '{cond}': {missing_pairs}")


def load_or_infer_balance(df: pd.DataFrame, split_col: str, schema: Dict[str, Any], cfg: Config) -> Tuple[pd.DataFrame, Optional[Path], bool]:
    path = first_existing(candidate_balance_paths(cfg))
    if path is not None:
        balance = pd.read_csv(path, low_memory=False)
        if "split" in balance.columns:
            balance["split"] = normalize_split_values(balance["split"])
        if "level" in balance.columns:
            balance["level"] = balance["level"].astype(str).str.lower()
        validate_balance_table(balance, cfg, source=f"Loaded balance table {path}")

        if "fraction" not in balance.columns and "n_sequences" in balance.columns:
            totals = balance.groupby(["condition", "split"])["n_sequences"].transform("sum")
            balance["fraction"] = balance["n_sequences"] / totals.replace(0, np.nan)
        if "n_sequences" not in balance.columns and "fraction" in balance.columns:
            balance["n_sequences"] = np.nan
        if "split_total" not in balance.columns:
            balance["split_total"] = np.nan

        balance["split_display"] = balance["split"].map(SPLIT_DISPLAY)
        balance["level_display"] = balance["level"].map(BIN_DISPLAY)
        balance["condition_display"] = balance["condition"].map(lambda x: DESCRIPTOR_LABELS.get(str(x), str(x)))
        balance["is_primary"] = balance["condition"].isin(cfg.primary_conditions)
        return balance, path, False

    balance = infer_condition_balance(df, split_col, schema, cfg)
    validate_balance_table(balance, cfg, source="Inferred balance table")
    return balance, None, True


def add_condition_ids(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    out = df.copy()

    primary_bin_cols = [f"{c}_bin" for c in cfg.primary_conditions if f"{c}_bin" in out.columns]
    full_cols = primary_bin_cols + [f"{c}_bin" for c in cfg.secondary_descriptors if f"{c}_bin" in out.columns]

    if "primary_condition_id" not in out.columns and primary_bin_cols:
        out["primary_condition_id"] = out[primary_bin_cols].astype("object").fillna("missing").agg("|".join, axis=1)

    if "full_condition_id" not in out.columns and full_cols:
        out["full_condition_id"] = out[full_cols].astype("object").fillna("missing").agg("|".join, axis=1)

    return out


def build_support_summary(df: pd.DataFrame, split_col: str) -> pd.DataFrame:
    rows = []
    for split in SPLIT_ORDER:
        sub = df.loc[df[split_col] == split].copy()
        rows.append(
            {
                "split": split,
                "split_display": SPLIT_DISPLAY[split],
                "n_sequences": int(len(sub)),
                "n_primary_condition_ids": int(sub["primary_condition_id"].nunique()) if "primary_condition_id" in sub.columns else np.nan,
                "n_full_descriptor_bin_ids": int(sub["full_condition_id"].nunique()) if "full_condition_id" in sub.columns else np.nan,
            }
        )
    return pd.DataFrame(rows)


def build_schema_summary(schema: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for cond, meta in schema.get("conditions", {}).items():
        edges = meta.get("edges", [])
        labels = [str(x).lower() for x in meta.get("labels", BIN_ORDER)]
        for i, level in enumerate(labels):
            lower = edges[i] if i < len(edges) else np.nan
            upper = edges[i + 1] if i + 1 < len(edges) else np.nan
            rows.append(
                {
                    "condition": cond,
                    "condition_display": DESCRIPTOR_LABELS.get(cond, cond),
                    "level": str(level).lower(),
                    "level_display": BIN_DISPLAY.get(str(level).lower(), str(level)),
                    "lower_edge": lower,
                    "upper_edge": upper,
                    "fit_population": meta.get("fit_population", "train_only"),
                    "is_primary": bool(meta.get("is_primary", cond in PRIMARY_CONDITIONS)),
                    "train_bin_count": meta.get("train_bin_counts", {}).get(str(level), np.nan),
                }
            )
    return pd.DataFrame(rows)


# =============================================================================
# 6. Source-data and audit builders
# =============================================================================

def build_s5_source_data(df: pd.DataFrame, split_col: str, schema: Dict[str, Any], cfg: Config) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    descriptors = ["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"]
    panels = {}
    all_rows = []
    for panel, desc in zip(["A", "B", "C", "D"], descriptors):
        panel_df = df[[split_col, desc]].copy()
        panel_df.rename(columns={split_col: "split", desc: "value"}, inplace=True)
        panel_df["split"] = normalize_split_values(panel_df["split"])
        panel_df["split_display"] = panel_df["split"].map(SPLIT_DISPLAY)
        panel_df["descriptor"] = desc
        panel_df["descriptor_display"] = DESCRIPTOR_LABELS[desc]
        panel_df["axis_label"] = DESCRIPTOR_AXIS_LABELS[desc]
        panel_df["panel"] = panel
        panel_df["data_type"] = "condition_descriptor_distribution"
        panel_df["is_primary_condition_descriptor"] = desc in cfg.primary_conditions
        panel_df["descriptor_role"] = "primary_condition" if desc in cfg.primary_conditions else "secondary_supporting_descriptor"
        edges = schema.get("conditions", {}).get(desc, {}).get("edges", [])
        panel_df["train_bin_edge_1"] = float(edges[1]) if len(edges) > 2 else np.nan
        panel_df["train_bin_edge_2"] = float(edges[2]) if len(edges) > 3 else np.nan
        panels[panel] = panel_df
        all_rows.append(panel_df)
    return panels, pd.concat(all_rows, ignore_index=True, sort=False)


def build_s6_source_data(balance: pd.DataFrame, support_summary: pd.DataFrame, cfg: Config) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    condition_panels = [("A", "length"), ("B", "net_charge_pH7"), ("C", "hydrophobicity_KD")]
    panels: Dict[str, pd.DataFrame] = {}
    all_rows = []

    for panel, cond in condition_panels:
        sub = balance.loc[(balance["condition"] == cond) & (balance["level"].isin(cfg.bin_order))].copy()
        sub["panel"] = panel
        sub["data_type"] = "condition_bin_fraction"
        sub["metric_name"] = "fraction"
        sub["metric_definition"] = "Fraction of sequences in each train-derived low/mid/high bin for a split."
        panels[panel] = sub
        all_rows.append(sub)

    d = support_summary.copy()
    d["panel"] = "D"
    d["data_type"] = "condition_support_summary"
    d["metric_name"] = "supported_condition_ids"
    d["metric_definition"] = "Number of distinct primary condition IDs and full descriptor-bin IDs represented in each split."
    panels["D"] = d
    all_rows.append(d)

    return panels, pd.concat(all_rows, ignore_index=True, sort=False)


def build_condition_bin_support_audit(balance: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Explicitly flag zero and sparse bins for reviewer-facing support auditing."""
    rows = []
    for cond in cfg.primary_conditions:
        for split in cfg.split_order:
            for level in cfg.bin_order:
                sub = balance.loc[
                    (balance["condition"] == cond)
                    & (balance["split"] == split)
                    & (balance["level"] == level)
                ]
                if sub.empty:
                    n = 0
                    fraction = 0.0
                    split_total = np.nan
                    missing_entry = True
                else:
                    row = sub.iloc[0]
                    n_raw = row.get("n_sequences", np.nan)
                    n = int(n_raw) if pd.notna(n_raw) else np.nan
                    fraction = float(row.get("fraction", np.nan))
                    split_total = row.get("split_total", np.nan)
                    missing_entry = False

                is_zero_bin = (pd.notna(n) and n == 0) or (pd.notna(fraction) and fraction == 0)
                is_sparse_by_n = pd.notna(n) and n < cfg.sparse_min_n
                is_sparse_by_fraction = pd.notna(fraction) and fraction < cfg.sparse_min_fraction
                is_sparse_bin = bool(is_zero_bin or is_sparse_by_n or is_sparse_by_fraction)

                rows.append(
                    {
                        "condition": cond,
                        "condition_display": DESCRIPTOR_LABELS.get(cond, cond),
                        "split": split,
                        "split_display": SPLIT_DISPLAY.get(split, split),
                        "level": level,
                        "level_display": BIN_DISPLAY.get(level, level),
                        "n_sequences": n,
                        "split_total": split_total,
                        "fraction": fraction,
                        "missing_entry": bool(missing_entry),
                        "is_zero_bin": bool(is_zero_bin),
                        "is_sparse_by_n": bool(is_sparse_by_n),
                        "is_sparse_by_fraction": bool(is_sparse_by_fraction),
                        "is_sparse_bin": bool(is_sparse_bin),
                        "sparse_min_n": cfg.sparse_min_n,
                        "sparse_min_fraction": cfg.sparse_min_fraction,
                    }
                )
    audit = pd.DataFrame(rows)
    if cfg.fail_on_missing_primary_bin and audit["missing_entry"].any():
        missing = audit.loc[audit["missing_entry"], ["condition", "split", "level"]].to_dict("records")
        raise ValueError(f"Missing required primary condition split/bin entries: {missing}")
    return audit


def build_zero_sparse_summary(audit: pd.DataFrame) -> pd.DataFrame:
    return (
        audit.groupby(["condition", "split"], as_index=False)
        .agg(
            n_bins=("level", "count"),
            n_zero_bins=("is_zero_bin", "sum"),
            n_sparse_bins=("is_sparse_bin", "sum"),
            min_bin_n=("n_sequences", "min"),
            min_bin_fraction=("fraction", "min"),
        )
    )


def save_source_data(cfg: Config, s5_panels: Dict[str, pd.DataFrame], s5_all: pd.DataFrame, s6_panels: Dict[str, pd.DataFrame], s6_all: pd.DataFrame) -> None:
    save_csv(s5_all, cfg.source_data_dir / "Supplementary_Figure_S5_source_data_all_panels.csv")
    save_csv(s6_all, cfg.source_data_dir / "Supplementary_Figure_S6_source_data_all_panels.csv")
    for panel, df in s5_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S5_panel_{panel.lower()}_source_data.csv")
    for panel, df in s6_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S6_panel_{panel.lower()}_source_data.csv")


# =============================================================================
# 7. Reports
# =============================================================================

def descriptor_summary(s5_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (desc, split), sub in s5_all.groupby(["descriptor", "split"], dropna=False):
        vals = finite_array(sub["value"])
        rows.append(
            {
                "descriptor": desc,
                "descriptor_display": DESCRIPTOR_LABELS.get(desc, str(desc)),
                "split": split,
                "split_display": SPLIT_DISPLAY.get(str(split), str(split)),
                "n": int(len(vals)),
                "mean": float(np.mean(vals)) if len(vals) else np.nan,
                "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
                "median": float(np.median(vals)) if len(vals) else np.nan,
                "q25": float(np.percentile(vals, 25)) if len(vals) else np.nan,
                "q75": float(np.percentile(vals, 75)) if len(vals) else np.nan,
                "min": float(np.min(vals)) if len(vals) else np.nan,
                "max": float(np.max(vals)) if len(vals) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def save_reports(
    cfg: Config,
    schema_summary: pd.DataFrame,
    support_summary: pd.DataFrame,
    balance: pd.DataFrame,
    s5_all: pd.DataFrame,
    support_audit: pd.DataFrame,
    sparse_summary: pd.DataFrame,
) -> Dict[str, pd.DataFrame]:
    reports = {
        "condition_schema_summary": schema_summary,
        "condition_support_summary": support_summary,
        "condition_bin_balance_summary": balance,
        "condition_descriptor_summary": descriptor_summary(s5_all),
        "condition_bin_support_audit": support_audit,
        "condition_bin_zero_sparse_summary": sparse_summary,
    }
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
    return reports


# =============================================================================
# 8. Plotting
# =============================================================================

def winsor_limits(arrays: Sequence[np.ndarray], lower: float = 0.005, upper: float = 0.995) -> Tuple[float, float]:
    nonempty = [arr for arr in arrays if len(arr) > 0]
    merged = np.concatenate(nonempty) if nonempty else np.array([0.0, 1.0])
    lo = float(np.quantile(merged, lower))
    hi = float(np.quantile(merged, upper))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(np.nanmin(merged)), float(np.nanmax(merged))
    pad = 0.07 * (hi - lo if hi > lo else 1.0)
    return lo - pad, hi + pad


def violin_box_panel(
    ax: plt.Axes,
    panel_df: pd.DataFrame,
    ylabel: str,
    title: str,
    cfg: Config,
    show_zero: bool = False,
    show_edges: bool = False,
) -> None:
    positions = np.arange(1, 4)
    data = [finite_array(panel_df.loc[panel_df["split"] == split, "value"]) for split in cfg.split_order]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=0.62,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        bw_method=0.25,
    )
    for body, split in zip(vp["bodies"], cfg.split_order):
        body.set_facecolor(SPLIT_FACE[split])
        body.set_edgecolor(SPLIT_EDGE[split])
        body.set_alpha(0.38)
        body.set_linewidth(0.9)
        body.set_zorder(3)

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.16,
        patch_artist=True,
        showfliers=False,
        whis=(5, 95),
        medianprops=dict(color=PLOS["dark"], linewidth=1.1),
        whiskerprops=dict(color=PLOS["charcoal"], linewidth=0.75),
        capprops=dict(color=PLOS["charcoal"], linewidth=0.75),
        boxprops=dict(linewidth=0.75),
    )
    for patch, split in zip(bp["boxes"], cfg.split_order):
        patch.set_facecolor("white")
        patch.set_edgecolor(SPLIT_EDGE[split])
        patch.set_zorder(4)

    if show_zero:
        ax.axhline(0, color=PLOS["charcoal"], linestyle="--", linewidth=0.75, zorder=2)

    if show_edges:
        edge_values = [panel_df["train_bin_edge_1"].dropna().unique(), panel_df["train_bin_edge_2"].dropna().unique()]
        for edge_arr in edge_values:
            if len(edge_arr):
                ax.axhline(float(edge_arr[0]), color=PLOS["dark"], linestyle=(0, (2.2, 2.2)), linewidth=0.7, alpha=0.7, zorder=2)

    ax.set_xticks(positions)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    lo, hi = winsor_limits(data)
    ax.set_ylim(lo, hi)
    beautify_axis(ax)


def plot_s5(s5_panels: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    set_plot_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.6))
    fig.subplots_adjust(left=0.095, right=0.985, top=0.92, bottom=0.145, wspace=0.34, hspace=0.43)

    specs = [
        ("A", "length", axes[0, 0], "Length (aa)", "Length", False, True),
        ("B", "net_charge_pH7", axes[0, 1], "Net charge at pH 7", "Net charge", True, True),
        ("C", "hydrophobicity_KD", axes[1, 0], "Mean KD hydrophobicity", "Hydrophobicity", True, True),
        ("D", "entropy", axes[1, 1], "Shannon entropy (bits)", "Shannon entropy", False, False),
    ]

    for panel, desc, ax, ylabel, title, zero, edges in specs:
        violin_box_panel(ax, s5_panels[panel], ylabel, title, cfg, show_zero=zero, show_edges=edges)
        panel_letter(ax, panel, cfg)

    handles = [Patch(facecolor=SPLIT_FACE[s], edgecolor=SPLIT_EDGE[s], label=SPLIT_DISPLAY[s]) for s in cfg.split_order]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, 0.045),
        edgecolor=PLOS["light"],
        facecolor="white",
        fontsize=cfg.legend_size,
    )

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S5_redesigned", cfg)


def plot_stacked_bins(ax: plt.Axes, panel_df: pd.DataFrame, condition_title: str, cfg: Config) -> None:
    pivot = (
        panel_df.pivot_table(index="split", columns="level", values="fraction", fill_value=0.0, aggfunc="first")
        .reindex(index=cfg.split_order, columns=cfg.bin_order, fill_value=0.0)
    )
    x = np.arange(len(cfg.split_order))
    bottoms = np.zeros(len(cfg.split_order))
    for level in cfg.bin_order:
        vals = pivot[level].to_numpy(dtype=float)
        ax.bar(
            x,
            vals,
            bottom=bottoms,
            width=0.68,
            color=BIN_FACE[level],
            edgecolor="white",
            linewidth=0.55,
            label=BIN_DISPLAY[level],
            zorder=3,
        )
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_ylabel("Fraction")
    ax.set_title(condition_title, pad=8)
    beautify_axis(ax)


def plot_support_summary(ax: plt.Axes, panel_df: pd.DataFrame, cfg: Config) -> None:
    df = panel_df.set_index("split").reindex(cfg.split_order).reset_index()
    x = np.arange(len(cfg.split_order))
    width = 0.34
    values_primary = pd.to_numeric(df["n_primary_condition_ids"], errors="coerce").fillna(0).to_numpy()
    values_full = pd.to_numeric(df["n_full_descriptor_bin_ids"], errors="coerce").fillna(0).to_numpy()

    bars1 = ax.bar(
        x - width / 2,
        values_primary,
        width=width,
        color=SUPPORT_FACE["primary"],
        edgecolor=SUPPORT_EDGE["primary"],
        linewidth=0.85,
        label="Primary condition IDs",
        zorder=3,
    )
    bars2 = ax.bar(
        x + width / 2,
        values_full,
        width=width,
        color=SUPPORT_FACE["full"],
        edgecolor=SUPPORT_EDGE["full"],
        linewidth=0.85,
        label="Full descriptor-bin IDs",
        zorder=3,
    )

    ymax = max(np.nanmax(values_primary), np.nanmax(values_full), 1)
    for bars in [bars1, bars2]:
        for bar in bars:
            value = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + ymax * 0.025,
                f"{int(value)}",
                ha="center",
                va="bottom",
                fontsize=cfg.annotation_size,
            )

    ax.set_xticks(x)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel("Supported condition IDs")
    ax.set_title("Condition-ID support", pad=8)
    ax.set_ylim(0, ymax * 1.24)
    ax.legend(frameon=True, edgecolor=PLOS["light"], facecolor="white", loc="upper right", fontsize=cfg.legend_size)
    beautify_axis(ax)


def plot_s6(s6_panels: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    set_plot_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.55))
    fig.subplots_adjust(left=0.09, right=0.985, top=0.92, bottom=0.15, wspace=0.34, hspace=0.43)

    plot_stacked_bins(axes[0, 0], s6_panels["A"], "Length bins", cfg)
    panel_letter(axes[0, 0], "A", cfg)

    plot_stacked_bins(axes[0, 1], s6_panels["B"], "Net-charge bins", cfg)
    panel_letter(axes[0, 1], "B", cfg)

    plot_stacked_bins(axes[1, 0], s6_panels["C"], "Hydrophobicity bins", cfg)
    panel_letter(axes[1, 0], "C", cfg)

    plot_support_summary(axes[1, 1], s6_panels["D"], cfg)
    panel_letter(axes[1, 1], "D", cfg)

    handles = [Patch(facecolor=BIN_FACE[level], edgecolor="white", label=BIN_DISPLAY[level]) for level in cfg.bin_order]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, 0.047),
        edgecolor=PLOS["light"],
        facecolor="white",
        fontsize=cfg.legend_size,
    )

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S6_redesigned", cfg)


# =============================================================================
# 9. Reproducibility outputs
# =============================================================================

def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env_commit = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env_commit:
        return env_commit
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=str(cfg.project_root),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=5,
            check=False,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        return None
    return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    file_name = globals().get("__file__", None)
    if file_name and Path(file_name).exists():
        script_path = str(Path(file_name).resolve())
        text = Path(file_name).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_03_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_03_PLOS_redesign_code.py")
    return script_path, script_hash


def write_no_main_figure_readme(cfg: Config) -> Optional[Path]:
    if not cfg.create_no_main_figure_readme:
        return None
    text = f"""# No main figure generated for OncoPep Step 3

Generated: {now_iso()}

Step 3 is a supplementary-only condition-support audit. It generates:
- Supplementary Figure S5
- Supplementary Figure S6

No Main Figure 2 is generated here.

Rationale:
Main Figure 2 is reserved for the OncoPep architecture and conditional
generation workflow. Step 3 only audits the descriptor and bin-support schema,
so promoting it to a main figure would weaken the main manuscript narrative.
"""
    path = cfg.main_figure_dir / "README_no_main_figure_for_step_03.txt"
    save_text(text, path)
    return path


def write_readme(
    cfg: Config,
    sequence_path: Path,
    schema_recomputed: bool,
    balance_inferred: bool,
    info_notes: Sequence[str],
    warnings: Sequence[str],
) -> Path:
    text = f"""# OncoPep Step 3 PLOS supplementary-figure package

Generated: {now_iso()}

Step 3 scope:
Conditioning schema and condition-support audit.

Main figure status:
No main figure is generated in Step 3. Main Figure 2 is reserved for the OncoPep
architecture/model workflow in a later figure-redesign step.

Input sequence-level file:
{sequence_path}

Output directory:
{cfg.output_root}

Generated figures:
- supplementary_figures/Supplementary_Figure_S5_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S6_redesigned.png/.pdf/.tiff

Scientific notes:
- Primary conditioning descriptors: length, net charge at pH 7, Kyte-Doolittle hydrophobicity.
- Shannon entropy is treated as a secondary/supporting descriptor unless explicitly stated otherwise in the model methods.
- Train-derived low/mid/high bin thresholds are used to audit condition support.
- Descriptor support should not be described as perfect balance or biological validation.

Schema recomputed from training data: {schema_recomputed}
Condition balance inferred from sequence-level table: {balance_inferred}

Sparse-bin audit thresholds:
- sparse_min_n: {cfg.sparse_min_n}
- sparse_min_fraction: {cfg.sparse_min_fraction}

Informational notes:
{chr(10).join("- " + str(w) for w in info_notes) if info_notes else "- None"}

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
"""
    path = cfg.reports_dir / "README_step_03_outputs.txt"
    save_text(text, path)
    return path


def write_requirements(cfg: Config) -> Path:
    text = "\n".join(
        [
            f"python=={platform.python_version()}",
            f"numpy=={np.__version__}",
            f"pandas=={pd.__version__}",
            f"matplotlib=={mpl.__version__}",
            "",
        ]
    )
    path = cfg.reports_dir / "requirements_step_03_minimal.txt"
    save_text(text, path)
    return path


def create_root_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs: list[tuple[Path, Path]] = []
    for fig in ["S5", "S6"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv", cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))

    root_reports = [
        "condition_schema_summary.csv",
        "condition_support_summary.csv",
        "condition_bin_balance_summary.csv",
        "condition_descriptor_summary.csv",
        "condition_bin_support_audit.csv",
        "condition_bin_zero_sparse_summary.csv",
        "README_step_03_outputs.txt",
    ]
    for name in root_reports:
        pairs.append((cfg.reports_dir / name, cfg.output_root / name))

    pairs.append((cfg.code_dir / "OncoPep_step_03_PLOS_redesign_code.py", cfg.output_root / "OncoPep_step_03_PLOS_redesign_code.py"))

    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def build_manifest(
    cfg: Config,
    sequence_path: Path,
    schema_path: Optional[Path],
    balance_path: Optional[Path],
    schema_recomputed: bool,
    balance_inferred: bool,
    s5_paths: Dict[str, str],
    s6_paths: Dict[str, str],
    readme_path: Path,
    requirements_path: Path,
    no_main_readme_path: Optional[Path],
    script_path: Optional[str],
    script_hash: str,
    root_copies: Sequence[str],
    info_notes: Sequence[str],
    warnings: Sequence[str],
) -> Dict[str, Any]:
    input_files = {"sequence_level": sequence_path}
    if schema_path:
        input_files["schema"] = schema_path
    if balance_path:
        input_files["balance"] = balance_path

    return {
        "step": "step_03",
        "scope": "supplementary_only",
        "timestamp": now_iso(),
        "main_figure_generated": False,
        "main_figure_rationale": "Step 3 audits condition support; Main Figure 2 is reserved for OncoPep architecture/model workflow.",
        "project_root": str(cfg.project_root),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "input_files": {k: str(v) for k, v in input_files.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_files.items()},
        "schema_recomputed_from_training_data": schema_recomputed,
        "condition_balance_inferred": balance_inferred,
        "figures": {"S5": s5_paths, "S6": s6_paths},
        "source_data": {
            "S5_all": str(cfg.source_data_dir / "Supplementary_Figure_S5_source_data_all_panels.csv"),
            "S6_all": str(cfg.source_data_dir / "Supplementary_Figure_S6_source_data_all_panels.csv"),
            "S5_panel_a": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_a_source_data.csv"),
            "S5_panel_b": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_b_source_data.csv"),
            "S5_panel_c": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_c_source_data.csv"),
            "S5_panel_d": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_d_source_data.csv"),
            "S6_panel_a": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_a_source_data.csv"),
            "S6_panel_b": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_b_source_data.csv"),
            "S6_panel_c": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_c_source_data.csv"),
            "S6_panel_d": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_d_source_data.csv"),
        },
        "reports": {
            "condition_schema_summary": str(cfg.reports_dir / "condition_schema_summary.csv"),
            "condition_support_summary": str(cfg.reports_dir / "condition_support_summary.csv"),
            "condition_bin_balance_summary": str(cfg.reports_dir / "condition_bin_balance_summary.csv"),
            "condition_descriptor_summary": str(cfg.reports_dir / "condition_descriptor_summary.csv"),
            "condition_bin_support_audit": str(cfg.reports_dir / "condition_bin_support_audit.csv"),
            "condition_bin_zero_sparse_summary": str(cfg.reports_dir / "condition_bin_zero_sparse_summary.csv"),
            "readme": str(readme_path),
            "requirements": str(requirements_path),
            "no_main_figure_readme": str(no_main_readme_path) if no_main_readme_path else None,
        },
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(root_copies),
        "info_notes": list(info_notes),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# =============================================================================
# 10. Main workflow
# =============================================================================

def run_step3_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_dirs(cfg)
    set_plot_style(cfg)

    info_notes: list[str] = [
        "Step 3 is supplementary-only; no Main Figure 2 is generated.",
        "Entropy is treated as a secondary/supporting descriptor in S5 and full descriptor-bin IDs.",
    ]
    warnings: list[str] = []

    conditioned_df, split_col, sequence_path = load_conditioned_sequences(cfg)
    schema, schema_path, schema_recomputed, schema_warnings = load_or_build_schema(conditioned_df, split_col, cfg)
    warnings.extend(schema_warnings)

    conditioned_df = ensure_bin_columns(conditioned_df, schema)
    conditioned_df = add_condition_ids(conditioned_df, cfg)
    balance_df, balance_path, balance_inferred = load_or_infer_balance(conditioned_df, split_col, schema, cfg)

    support_summary = build_support_summary(conditioned_df, split_col)
    schema_summary = build_schema_summary(schema)
    support_audit = build_condition_bin_support_audit(balance_df, cfg)
    sparse_summary = build_zero_sparse_summary(support_audit)

    zero_or_sparse = support_audit.loc[support_audit["is_sparse_bin"] | support_audit["is_zero_bin"]].copy()
    if not zero_or_sparse.empty:
        n_zero = int(zero_or_sparse["is_zero_bin"].sum())
        n_sparse = int(zero_or_sparse["is_sparse_bin"].sum())
        warnings.append(
            f"Condition-bin support audit detected {n_zero} zero bins and {n_sparse} sparse bins "
            f"using sparse_min_n={cfg.sparse_min_n}, sparse_min_fraction={cfg.sparse_min_fraction}. "
            f"Inspect reports/condition_bin_support_audit.csv."
        )

    s5_panels, s5_all = build_s5_source_data(conditioned_df, split_col, schema, cfg)
    s6_panels, s6_all = build_s6_source_data(balance_df, support_summary, cfg)

    save_source_data(cfg, s5_panels, s5_all, s6_panels, s6_all)
    reports = save_reports(cfg, schema_summary, support_summary, balance_df, s5_all, support_audit, sparse_summary)

    s5_paths = plot_s5(s5_panels, cfg)
    s6_paths = plot_s6(s6_panels, cfg)

    script_path, script_hash = code_snapshot(cfg)
    no_main_readme_path = write_no_main_figure_readme(cfg)
    readme_path = write_readme(cfg, sequence_path, schema_recomputed, balance_inferred, info_notes, warnings)
    requirements_path = write_requirements(cfg)
    root_copies = create_root_copies(cfg)

    manifest = build_manifest(
        cfg=cfg,
        sequence_path=sequence_path,
        schema_path=schema_path,
        balance_path=balance_path,
        schema_recomputed=schema_recomputed,
        balance_inferred=balance_inferred,
        s5_paths=s5_paths,
        s6_paths=s6_paths,
        readme_path=readme_path,
        requirements_path=requirements_path,
        no_main_readme_path=no_main_readme_path,
        script_path=script_path,
        script_hash=script_hash,
        root_copies=root_copies,
        info_notes=info_notes,
        warnings=warnings,
    )
    save_json(manifest, cfg.reports_dir / "step_03_manifest.json")
    save_json(manifest, cfg.output_root / "step_03_manifest.json")

    print("\n✅ Step 3 PLOS supplementary-figure package generated successfully.")
    print("Scope: supplementary-only condition-support audit")
    print("Main figure generated: No")
    print(f"Input sequence file: {sequence_path}")
    print(f"Output directory: {cfg.output_root}")

    for label, paths in [("Supplementary Figure S5", s5_paths), ("Supplementary Figure S6", s6_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")

    print("\nSource data:")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S5_source_data_all_panels.csv'}")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S6_source_data_all_panels.csv'}")
    for fig in ["S5", "S6"]:
        for panel in ["a", "b", "c", "d"]:
            print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_{fig}_panel_{panel}_source_data.csv'}")

    print("\nReports:")
    for name in [
        "condition_schema_summary.csv",
        "condition_support_summary.csv",
        "condition_bin_balance_summary.csv",
        "condition_descriptor_summary.csv",
        "condition_bin_support_audit.csv",
        "condition_bin_zero_sparse_summary.csv",
        "step_03_manifest.json",
        "README_step_03_outputs.txt",
    ]:
        print(f"  - {cfg.reports_dir / name}")
    if no_main_readme_path:
        print(f"  - {no_main_readme_path}")

    if info_notes:
        print("\nInformational notes:")
        for note in info_notes:
            print(f"  - {note}")

    if warnings:
        print("\nWarnings:")
        for warning in warnings:
            print(f"  - {warning}")

    return {
        "conditioned_df": conditioned_df,
        "schema": schema,
        "balance_df": balance_df,
        "support_summary": support_summary,
        "schema_summary": schema_summary,
        "support_audit": support_audit,
        "sparse_summary": sparse_summary,
        "s5_panels": s5_panels,
        "s6_panels": s6_panels,
        "reports": reports,
        "manifest": manifest,
        "info_notes": info_notes,
        "warnings": warnings,
    }


# =============================================================================
# 11. Notebook and CLI entry points
# =============================================================================

def main_notebook(
    project_root: str | Path = PROJECT_ROOT_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    step3_relative_dir: str | Path = STEP3_RELATIVE_DIR_DEFAULT,
    source_sequence_file: Optional[str | Path] = None,
    source_schema_file: Optional[str | Path] = None,
    source_balance_file: Optional[str | Path] = None,
    sparse_min_n: int = 10,
    sparse_min_fraction: float = 0.005,
    fail_on_missing_primary_bin: bool = True,
    create_root_level_copies: bool = True,
    create_no_main_figure_readme: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        project_root=Path(project_root),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        step3_relative_dir=Path(step3_relative_dir),
        source_sequence_file=Path(source_sequence_file) if source_sequence_file else None,
        source_schema_file=Path(source_schema_file) if source_schema_file else None,
        source_balance_file=Path(source_balance_file) if source_balance_file else None,
        sparse_min_n=sparse_min_n,
        sparse_min_fraction=sparse_min_fraction,
        fail_on_missing_primary_bin=fail_on_missing_primary_bin,
        create_root_level_copies=create_root_level_copies,
        create_no_main_figure_readme=create_no_main_figure_readme,
        git_commit=git_commit,
    )
    return run_step3_redesign(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 3 PLOS supplementary figures S5/S6.")
    parser.add_argument("--project_root", default=str(PROJECT_ROOT_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--step3_relative_dir", default=str(STEP3_RELATIVE_DIR_DEFAULT))
    parser.add_argument("--source_sequence_file", default=None)
    parser.add_argument("--source_schema_file", default=None)
    parser.add_argument("--source_balance_file", default=None)
    parser.add_argument("--sparse_min_n", type=int, default=10)
    parser.add_argument("--sparse_min_fraction", type=float, default=0.005)
    parser.add_argument("--allow_missing_primary_bin", action="store_true")
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    parser.add_argument("--no_main_readme", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    cfg = Config(
        project_root=Path(args.project_root),
        base_dir=Path(args.base_dir),
        output_root_name=args.output_root_name,
        step3_relative_dir=Path(args.step3_relative_dir),
        source_sequence_file=Path(args.source_sequence_file) if args.source_sequence_file else None,
        source_schema_file=Path(args.source_schema_file) if args.source_schema_file else None,
        source_balance_file=Path(args.source_balance_file) if args.source_balance_file else None,
        sparse_min_n=args.sparse_min_n,
        sparse_min_fraction=args.sparse_min_fraction,
        fail_on_missing_primary_bin=not args.allow_missing_primary_bin,
        create_root_level_copies=not args.no_root_copies,
        create_no_main_figure_readme=not args.no_main_readme,
        git_commit=args.git_commit,
    )
    run_step3_redesign(cfg)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 3 — PLOS Computational Biology supplementary-figure redesign.

Step 3 scope
------------
This step is supplementary-only. It audits the conditioning schema and
condition-support structure used for property-conditioned peptide generation.

Generated outputs:
  Supplementary Figure S5 — condition descriptor distributions across partitions
  Supplementary Figure S6 — train-derived condition-bin balance across partitions

This script does NOT generate Main Figure 2. Main Figure 2 is reserved for the
OncoPep architecture/model workflow in a later manuscript step.

Main improvements in this version
---------------------------------
1. Strict validation of primary condition schemas and low/mid/high bins.
2. Explicit sparse/zero-bin support audit.
3. Stronger validation of loaded condition-balance tables.
4. Separate informational notes from true warnings.
5. Clearer entropy handling as a secondary/supporting descriptor.
6. Optional README explaining why no main figure is generated in Step 3.
7. Supplementary Figure S5 uses boxplots instead of violin plots for cleaner
   manuscript-scale descriptor comparison.
8. Supplementary Figure S6 uses grouped bars for low/mid/high bins to improve
   direct comparison across train, validation, and test partitions.
9. Panel D legend is moved slightly upward for cleaner spacing.

Notebook usage
--------------
results = main_notebook(
    project_root="/home/data3/Moe/nature_computational2",
    base_dir="/home/data3/Moe/nature_computational_peponco/PLOS",
    output_root_name="plos_comp/step_03",
)

Terminal usage
--------------
python OncoPep_step_03_PLOS_redesign_code.py
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


# =============================================================================
# 1. Configuration
# =============================================================================

PROJECT_ROOT_DEFAULT = Path("/home/data3/Moe/nature_computational2")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_03"
STEP3_RELATIVE_DIR_DEFAULT = Path("redesign/step3")

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
BIN_ORDER = ("low", "mid", "high")
BIN_DISPLAY = {"low": "Low", "mid": "Mid", "high": "High"}

PRIMARY_CONDITIONS = ("length", "net_charge_pH7", "hydrophobicity_KD")
SECONDARY_DESCRIPTORS = ("entropy",)

DESCRIPTOR_ALIASES = {
    "length": ["length", "seq_length", "peptide_length", "sequence_length"],
    "net_charge_pH7": ["net_charge_pH7", "net_charge", "charge", "charge_pH7"],
    "hydrophobicity_KD": ["hydrophobicity_KD", "hydrophobicity", "hydrophobicity_kd", "kd_hydrophobicity", "mean_hydropathy"],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy"],
}
DESCRIPTOR_LABELS = {
    "length": "Length",
    "net_charge_pH7": "Net charge at pH 7",
    "hydrophobicity_KD": "Kyte–Doolittle hydrophobicity",
    "entropy": "Shannon entropy",
}
DESCRIPTOR_AXIS_LABELS = {
    "length": "Length (aa)",
    "net_charge_pH7": "Net charge at pH 7",
    "hydrophobicity_KD": "Mean KD hydrophobicity",
    "entropy": "Shannon entropy (bits)",
}


@dataclass
class Config:
    """Editable Step 3 configuration."""

    project_root: Path = PROJECT_ROOT_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT
    step3_relative_dir: Path = STEP3_RELATIVE_DIR_DEFAULT

    # Optional explicit inputs override candidate-path search.
    source_sequence_file: Optional[Path] = None
    source_schema_file: Optional[Path] = None
    source_balance_file: Optional[Path] = None

    primary_conditions: Tuple[str, ...] = PRIMARY_CONDITIONS
    secondary_descriptors: Tuple[str, ...] = SECONDARY_DESCRIPTORS
    split_order: Tuple[str, ...] = SPLIT_ORDER
    bin_order: Tuple[str, ...] = BIN_ORDER

    # Sparse-bin audit settings.
    sparse_min_n: int = 10
    sparse_min_fraction: float = 0.005
    fail_on_missing_primary_bin: bool = True

    # Export settings.
    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    create_no_main_figure_readme: bool = True
    git_commit: Optional[str] = None

    # Typography.
    font_size: float = 8.3
    title_size: float = 9.2
    axis_label_size: float = 8.9
    tick_label_size: float = 7.8
    legend_size: float = 7.7
    annotation_size: float = 7.5
    panel_letter_size: float = 13.0

    @property
    def step3_source_root(self) -> Path:
        return self.project_root / self.step3_relative_dir

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_figure_dir(self) -> Path:
        return self.output_root / "main_figure"


# =============================================================================
# 2. Approved PLOS-style palette
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

SPLIT_FACE = {"train": PLOS["blue"], "val": PLOS["mint"], "test": PLOS["coral"]}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}

BIN_FACE = {"low": PLOS["blue"], "mid": PLOS["mint"], "high": PLOS["coral"]}
BIN_EDGE = {"low": "#176F8A", "mid": "#6F9F7A", "high": "#A84F42"}

SUPPORT_FACE = {"primary": PLOS["blue"], "full": PLOS["brown"]}
SUPPORT_EDGE = {"primary": "#176F8A", "full": "#8F5F37"}


# =============================================================================
# 3. General utilities
# =============================================================================

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def ensure_dirs(cfg: Config) -> None:
    for path in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)
    if cfg.create_no_main_figure_readme:
        cfg.main_figure_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=json_default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def finite_array(values: Any) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def normalize_split_values(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for cand in candidates:
        if cand in df.columns:
            return cand
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(f"Could not resolve column from {list(candidates)}. Available columns: {list(df.columns)}")
    return None


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


def checked_numeric(df: pd.DataFrame, col: str, min_finite_fraction: float = 0.90) -> None:
    vals = pd.to_numeric(df[col], errors="coerce")
    finite_fraction = float(np.isfinite(vals.to_numpy(dtype=float)).mean())
    if finite_fraction < min_finite_fraction:
        raise ValueError(
            f"Column '{col}' has too few finite numeric values "
            f"({finite_fraction:.3f} < {min_finite_fraction:.3f})."
        )


# =============================================================================
# 4. Plot style
# =============================================================================

def set_plot_style(cfg: Config) -> None:
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": cfg.font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_label_size,
            "ytick.labelsize": cfg.tick_label_size,
            "legend.fontsize": cfg.legend_size,
            "axes.linewidth": 0.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.facecolor": PLOS["white"],
            "axes.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
        }
    )


def beautify_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.45)
    ax.grid(False, axis="x")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.75)
        ax.spines[side].set_color(PLOS["dark"])


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.08) -> None:
    ax.text(
        dx,
        dy,
        str(letter).upper(),
        transform=ax.transAxes,
        fontsize=cfg.panel_letter_size,
        fontweight="bold",
        va="top",
        ha="left",
        color=PLOS["dark"],
    )


def save_figure(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "png": base.with_suffix(".png"),
        "pdf": base.with_suffix(".pdf"),
        "tiff": base.with_suffix(".tiff"),
    }
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


# =============================================================================
# 5. Input loading, descriptor harmonization, and schema construction
# =============================================================================

def candidate_sequence_paths(cfg: Config) -> list[Path]:
    candidates = []
    if cfg.source_sequence_file:
        candidates.append(Path(cfg.source_sequence_file))
    candidates.extend(
        [
            cfg.step3_source_root / "tables" / "step3_conditioned_unique_sequences.csv",
            cfg.project_root / "step3" / "tables" / "step3_conditioned_unique_sequences.csv",
            cfg.project_root / "step2" / "splits" / "step2_unique_sequences_with_splits.csv",
            Path("/home/data3/Moe/nature_computational_peponco/step2/splits/step2_unique_sequences_with_splits.csv"),
        ]
    )
    return candidates


def candidate_schema_paths(cfg: Config) -> list[Path]:
    candidates = []
    if cfg.source_schema_file:
        candidates.append(Path(cfg.source_schema_file))
    candidates.extend(
        [
            cfg.step3_source_root / "schemas" / "step3_condition_schema.json",
            cfg.project_root / "step3" / "schemas" / "step3_condition_schema.json",
        ]
    )
    return candidates


def candidate_balance_paths(cfg: Config) -> list[Path]:
    candidates = []
    if cfg.source_balance_file:
        candidates.append(Path(cfg.source_balance_file))
    candidates.extend(
        [
            cfg.step3_source_root / "tables" / "step3_condition_balance_by_split.csv",
            cfg.project_root / "step3" / "tables" / "step3_condition_balance_by_split.csv",
        ]
    )
    return candidates


def harmonize_descriptor_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    lower_map = {str(c).lower(): c for c in out.columns}
    for target, aliases in DESCRIPTOR_ALIASES.items():
        found = None
        for alias in aliases:
            if alias.lower() in lower_map:
                found = lower_map[alias.lower()]
                break
        if found is not None and found != target:
            out.rename(columns={found: target}, inplace=True)
        if target in out.columns:
            out[target] = pd.to_numeric(out[target], errors="coerce")
    return out


def validate_sequence_table(df: pd.DataFrame, split_col: str, cfg: Config) -> None:
    required_descriptors = list(cfg.primary_conditions) + list(cfg.secondary_descriptors)
    missing_descriptors = [c for c in required_descriptors if c not in df.columns]
    if missing_descriptors:
        raise ValueError(f"Missing descriptor column(s) after harmonization: {missing_descriptors}. Available columns: {list(df.columns)}")

    observed_splits = set(df[split_col].dropna().unique())
    missing_splits = [s for s in cfg.split_order if s not in observed_splits]
    if missing_splits:
        raise ValueError(f"Missing expected split labels {missing_splits}; observed labels={sorted(observed_splits)}")

    for desc in required_descriptors:
        checked_numeric(df, desc, min_finite_fraction=0.90)


def load_conditioned_sequences(cfg: Config) -> Tuple[pd.DataFrame, str, Path]:
    path = first_existing(candidate_sequence_paths(cfg))
    if path is None:
        raise FileNotFoundError("No sequence-level file found. Checked:\n- " + "\n- ".join(str(p) for p in candidate_sequence_paths(cfg)))
    df = pd.read_csv(path, low_memory=False)
    split_col = resolve_column(df, ["assigned_split", "split", "data_split"])
    df[split_col] = normalize_split_values(df[split_col])
    if split_col != "assigned_split":
        df.rename(columns={split_col: "assigned_split"}, inplace=True)
        split_col = "assigned_split"
    df = harmonize_descriptor_columns(df)
    validate_sequence_table(df, split_col, cfg)
    return df, split_col, path


def robust_quantile_edges(series: pd.Series, n_bins: int = 3) -> np.ndarray:
    vals = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        raise ValueError("Cannot compute quantile edges from an empty descriptor series.")
    q = np.linspace(0.0, 1.0, n_bins + 1)
    try:
        edges = np.quantile(vals, q, method="linear")
    except TypeError:
        edges = np.quantile(vals, q, interpolation="linear")
    edges = np.asarray(edges, dtype=float)
    edges = np.maximum.accumulate(edges)
    if np.unique(edges).size < n_bins + 1:
        unique_vals = np.unique(vals)
        if len(unique_vals) < n_bins:
            raise ValueError("Not enough unique values to construct low/mid/high bins.")
        fallback_q = np.linspace(0, len(unique_vals) - 1, n_bins + 1)
        edges = np.interp(fallback_q, np.arange(len(unique_vals)), unique_vals)
    edges[0] -= 1e-9
    edges[-1] += 1e-9
    return np.maximum.accumulate(edges)


def assign_bins(series: pd.Series, edges: Sequence[float], labels: Sequence[str] = BIN_ORDER) -> pd.Series:
    return pd.cut(
        pd.to_numeric(series, errors="coerce"),
        bins=np.asarray(edges, dtype=float),
        labels=list(labels),
        include_lowest=True,
        right=False,
    ).astype("object")


def recompute_schema_from_training(df: pd.DataFrame, split_col: str, cfg: Config) -> Dict[str, Any]:
    train_df = df.loc[df[split_col] == "train"].copy()
    if train_df.empty:
        raise ValueError("No training rows found; cannot compute train-derived condition-bin thresholds.")

    schema = {"conditions": {}}
    for cond in list(cfg.primary_conditions) + [c for c in cfg.secondary_descriptors if c in df.columns]:
        if cond not in df.columns:
            continue
        edges = robust_quantile_edges(train_df[cond], n_bins=3)
        train_bins = assign_bins(train_df[cond], edges, cfg.bin_order)
        train_counts = pd.Series(train_bins).value_counts().reindex(cfg.bin_order, fill_value=0)
        schema["conditions"][cond] = {
            "labels": list(cfg.bin_order),
            "edges": [float(v) for v in edges],
            "train_bin_counts": {str(k): int(v) for k, v in train_counts.items()},
            "fit_population": "train_only",
            "is_primary": bool(cond in cfg.primary_conditions),
        }
    return schema


def validate_schema_integrity(schema: Dict[str, Any], cfg: Config) -> list[str]:
    """Validate primary condition schema strictly and return non-fatal notes/warnings."""
    warnings = []
    if "conditions" not in schema or not isinstance(schema["conditions"], dict):
        raise ValueError("Condition schema is malformed: missing top-level 'conditions' dictionary.")

    for cond in cfg.primary_conditions:
        if cond not in schema["conditions"]:
            raise ValueError(f"Primary condition '{cond}' is missing from the condition schema.")
        meta = schema["conditions"][cond]
        labels = [str(x).lower() for x in meta.get("labels", [])]
        if list(labels) != list(cfg.bin_order):
            raise ValueError(f"Primary condition '{cond}' must have labels {list(cfg.bin_order)}, found {labels}.")
        edges = meta.get("edges", [])
        if len(edges) != len(cfg.bin_order) + 1:
            raise ValueError(f"Primary condition '{cond}' must have {len(cfg.bin_order)+1} bin edges, found {len(edges)}.")
        edges_arr = np.asarray(edges, dtype=float)
        if not np.all(np.isfinite(edges_arr)):
            raise ValueError(f"Primary condition '{cond}' has non-finite bin edges.")
        if not np.all(np.diff(edges_arr) >= 0):
            raise ValueError(f"Primary condition '{cond}' has non-monotonic bin edges.")
        if bool(meta.get("is_primary", cond in cfg.primary_conditions)) is not True:
            warnings.append(f"Primary condition '{cond}' was not marked primary in loaded schema; treated as primary by configuration.")

    for desc in cfg.secondary_descriptors:
        if desc in schema["conditions"]:
            meta = schema["conditions"][desc]
            if bool(meta.get("is_primary", False)):
                warnings.append(f"Secondary descriptor '{desc}' is marked primary in loaded schema. Confirm model-conditioning definition.")
        else:
            warnings.append(f"Secondary descriptor '{desc}' is absent from schema; it will be shown in S5 but not used for full descriptor-bin IDs.")

    return warnings


def load_or_build_schema(df: pd.DataFrame, split_col: str, cfg: Config) -> Tuple[Dict[str, Any], Optional[Path], bool, list[str]]:
    path = first_existing(candidate_schema_paths(cfg))
    if path is not None:
        with path.open("r", encoding="utf-8") as f:
            schema = json.load(f)
        warnings = validate_schema_integrity(schema, cfg)
        return schema, path, False, warnings
    schema = recompute_schema_from_training(df, split_col, cfg)
    warnings = validate_schema_integrity(schema, cfg)
    return schema, None, True, warnings


def ensure_bin_columns(df: pd.DataFrame, schema: Dict[str, Any]) -> pd.DataFrame:
    out = df.copy()
    for cond, meta in schema.get("conditions", {}).items():
        if cond not in out.columns:
            continue
        bin_col = f"{cond}_bin"
        if bin_col not in out.columns:
            labels = [str(x).lower() for x in meta.get("labels", list(BIN_ORDER))]
            edges = meta.get("edges")
            if edges is None:
                continue
            out[bin_col] = assign_bins(out[cond], edges, labels)
        out[bin_col] = out[bin_col].astype(str).str.lower().replace({"nan": np.nan, "none": np.nan})
    return out


def infer_condition_balance(df: pd.DataFrame, split_col: str, schema: Dict[str, Any], cfg: Config) -> pd.DataFrame:
    rows = []
    for cond, meta in schema.get("conditions", {}).items():
        bin_col = f"{cond}_bin"
        if bin_col not in df.columns:
            continue
        levels = [str(x).lower() for x in meta.get("labels", cfg.bin_order)]
        for split in cfg.split_order:
            sub = df.loc[df[split_col] == split].copy()
            total = len(sub)
            counts = sub[bin_col].astype("object").value_counts(dropna=False)
            for level in levels:
                n = int(counts.get(level, 0))
                rows.append(
                    {
                        "condition": cond,
                        "condition_display": DESCRIPTOR_LABELS.get(cond, cond),
                        "split": split,
                        "split_display": SPLIT_DISPLAY.get(split, split),
                        "level": str(level).lower(),
                        "level_display": BIN_DISPLAY.get(str(level).lower(), str(level)),
                        "n_sequences": n,
                        "split_total": int(total),
                        "fraction": float(n / total) if total else np.nan,
                        "is_primary": bool(meta.get("is_primary", cond in cfg.primary_conditions)),
                    }
                )
    return pd.DataFrame(rows)


def validate_balance_table(balance: pd.DataFrame, cfg: Config, source: str = "balance table") -> None:
    required = {"condition", "split", "level"}
    missing = sorted(required - set(balance.columns))
    if missing:
        raise ValueError(f"{source} is missing required columns: {missing}")
    if "fraction" not in balance.columns and "n_sequences" not in balance.columns:
        raise ValueError(f"{source} must contain either 'fraction' or 'n_sequences'.")

    balance_conditions = set(balance["condition"].astype(str))
    missing_primary = [c for c in cfg.primary_conditions if c not in balance_conditions]
    if missing_primary:
        raise ValueError(f"{source} is missing primary conditions required for S6: {missing_primary}")

    for cond in cfg.primary_conditions:
        sub = balance.loc[balance["condition"] == cond].copy()
        observed_pairs = set(zip(sub["split"].astype(str), sub["level"].astype(str).str.lower()))
        required_pairs = {(split, level) for split in cfg.split_order for level in cfg.bin_order}
        missing_pairs = sorted(required_pairs - observed_pairs)
        if missing_pairs:
            raise ValueError(f"{source} missing split/bin entries for primary condition '{cond}': {missing_pairs}")


def load_or_infer_balance(df: pd.DataFrame, split_col: str, schema: Dict[str, Any], cfg: Config) -> Tuple[pd.DataFrame, Optional[Path], bool]:
    path = first_existing(candidate_balance_paths(cfg))
    if path is not None:
        balance = pd.read_csv(path, low_memory=False)
        if "split" in balance.columns:
            balance["split"] = normalize_split_values(balance["split"])
        if "level" in balance.columns:
            balance["level"] = balance["level"].astype(str).str.lower()
        validate_balance_table(balance, cfg, source=f"Loaded balance table {path}")

        if "fraction" not in balance.columns and "n_sequences" in balance.columns:
            totals = balance.groupby(["condition", "split"])["n_sequences"].transform("sum")
            balance["fraction"] = balance["n_sequences"] / totals.replace(0, np.nan)
        if "n_sequences" not in balance.columns and "fraction" in balance.columns:
            balance["n_sequences"] = np.nan
        if "split_total" not in balance.columns:
            balance["split_total"] = np.nan

        balance["split_display"] = balance["split"].map(SPLIT_DISPLAY)
        balance["level_display"] = balance["level"].map(BIN_DISPLAY)
        balance["condition_display"] = balance["condition"].map(lambda x: DESCRIPTOR_LABELS.get(str(x), str(x)))
        balance["is_primary"] = balance["condition"].isin(cfg.primary_conditions)
        return balance, path, False

    balance = infer_condition_balance(df, split_col, schema, cfg)
    validate_balance_table(balance, cfg, source="Inferred balance table")
    return balance, None, True


def add_condition_ids(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    out = df.copy()

    primary_bin_cols = [f"{c}_bin" for c in cfg.primary_conditions if f"{c}_bin" in out.columns]
    full_cols = primary_bin_cols + [f"{c}_bin" for c in cfg.secondary_descriptors if f"{c}_bin" in out.columns]

    if "primary_condition_id" not in out.columns and primary_bin_cols:
        out["primary_condition_id"] = out[primary_bin_cols].astype("object").fillna("missing").agg("|".join, axis=1)

    if "full_condition_id" not in out.columns and full_cols:
        out["full_condition_id"] = out[full_cols].astype("object").fillna("missing").agg("|".join, axis=1)

    return out


def build_support_summary(df: pd.DataFrame, split_col: str) -> pd.DataFrame:
    rows = []
    for split in SPLIT_ORDER:
        sub = df.loc[df[split_col] == split].copy()
        rows.append(
            {
                "split": split,
                "split_display": SPLIT_DISPLAY[split],
                "n_sequences": int(len(sub)),
                "n_primary_condition_ids": int(sub["primary_condition_id"].nunique()) if "primary_condition_id" in sub.columns else np.nan,
                "n_full_descriptor_bin_ids": int(sub["full_condition_id"].nunique()) if "full_condition_id" in sub.columns else np.nan,
            }
        )
    return pd.DataFrame(rows)


def build_schema_summary(schema: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for cond, meta in schema.get("conditions", {}).items():
        edges = meta.get("edges", [])
        labels = [str(x).lower() for x in meta.get("labels", BIN_ORDER)]
        for i, level in enumerate(labels):
            lower = edges[i] if i < len(edges) else np.nan
            upper = edges[i + 1] if i + 1 < len(edges) else np.nan
            rows.append(
                {
                    "condition": cond,
                    "condition_display": DESCRIPTOR_LABELS.get(cond, cond),
                    "level": str(level).lower(),
                    "level_display": BIN_DISPLAY.get(str(level).lower(), str(level)),
                    "lower_edge": lower,
                    "upper_edge": upper,
                    "fit_population": meta.get("fit_population", "train_only"),
                    "is_primary": bool(meta.get("is_primary", cond in PRIMARY_CONDITIONS)),
                    "train_bin_count": meta.get("train_bin_counts", {}).get(str(level), np.nan),
                }
            )
    return pd.DataFrame(rows)


# =============================================================================
# 6. Source-data and audit builders
# =============================================================================

def build_s5_source_data(df: pd.DataFrame, split_col: str, schema: Dict[str, Any], cfg: Config) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    descriptors = ["length", "net_charge_pH7", "hydrophobicity_KD", "entropy"]
    panels = {}
    all_rows = []
    for panel, desc in zip(["A", "B", "C", "D"], descriptors):
        panel_df = df[[split_col, desc]].copy()
        panel_df.rename(columns={split_col: "split", desc: "value"}, inplace=True)
        panel_df["split"] = normalize_split_values(panel_df["split"])
        panel_df["split_display"] = panel_df["split"].map(SPLIT_DISPLAY)
        panel_df["descriptor"] = desc
        panel_df["descriptor_display"] = DESCRIPTOR_LABELS[desc]
        panel_df["axis_label"] = DESCRIPTOR_AXIS_LABELS[desc]
        panel_df["panel"] = panel
        panel_df["data_type"] = "condition_descriptor_distribution"
        panel_df["is_primary_condition_descriptor"] = desc in cfg.primary_conditions
        panel_df["descriptor_role"] = "primary_condition" if desc in cfg.primary_conditions else "secondary_supporting_descriptor"
        edges = schema.get("conditions", {}).get(desc, {}).get("edges", [])
        panel_df["train_bin_edge_1"] = float(edges[1]) if len(edges) > 2 else np.nan
        panel_df["train_bin_edge_2"] = float(edges[2]) if len(edges) > 3 else np.nan
        panels[panel] = panel_df
        all_rows.append(panel_df)
    return panels, pd.concat(all_rows, ignore_index=True, sort=False)


def build_s6_source_data(balance: pd.DataFrame, support_summary: pd.DataFrame, cfg: Config) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    condition_panels = [("A", "length"), ("B", "net_charge_pH7"), ("C", "hydrophobicity_KD")]
    panels: Dict[str, pd.DataFrame] = {}
    all_rows = []

    for panel, cond in condition_panels:
        sub = balance.loc[(balance["condition"] == cond) & (balance["level"].isin(cfg.bin_order))].copy()
        sub["panel"] = panel
        sub["data_type"] = "condition_bin_fraction"
        sub["metric_name"] = "fraction"
        sub["metric_definition"] = "Fraction of sequences in each train-derived low/mid/high bin for a split."
        panels[panel] = sub
        all_rows.append(sub)

    d = support_summary.copy()
    d["panel"] = "D"
    d["data_type"] = "condition_support_summary"
    d["metric_name"] = "supported_condition_ids"
    d["metric_definition"] = "Number of distinct primary condition IDs and full descriptor-bin IDs represented in each split."
    panels["D"] = d
    all_rows.append(d)

    return panels, pd.concat(all_rows, ignore_index=True, sort=False)


def build_condition_bin_support_audit(balance: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Explicitly flag zero and sparse bins for reviewer-facing support auditing."""
    rows = []
    for cond in cfg.primary_conditions:
        for split in cfg.split_order:
            for level in cfg.bin_order:
                sub = balance.loc[
                    (balance["condition"] == cond)
                    & (balance["split"] == split)
                    & (balance["level"] == level)
                ]
                if sub.empty:
                    n = 0
                    fraction = 0.0
                    split_total = np.nan
                    missing_entry = True
                else:
                    row = sub.iloc[0]
                    n_raw = row.get("n_sequences", np.nan)
                    n = int(n_raw) if pd.notna(n_raw) else np.nan
                    fraction = float(row.get("fraction", np.nan))
                    split_total = row.get("split_total", np.nan)
                    missing_entry = False

                is_zero_bin = (pd.notna(n) and n == 0) or (pd.notna(fraction) and fraction == 0)
                is_sparse_by_n = pd.notna(n) and n < cfg.sparse_min_n
                is_sparse_by_fraction = pd.notna(fraction) and fraction < cfg.sparse_min_fraction
                is_sparse_bin = bool(is_zero_bin or is_sparse_by_n or is_sparse_by_fraction)

                rows.append(
                    {
                        "condition": cond,
                        "condition_display": DESCRIPTOR_LABELS.get(cond, cond),
                        "split": split,
                        "split_display": SPLIT_DISPLAY.get(split, split),
                        "level": level,
                        "level_display": BIN_DISPLAY.get(level, level),
                        "n_sequences": n,
                        "split_total": split_total,
                        "fraction": fraction,
                        "missing_entry": bool(missing_entry),
                        "is_zero_bin": bool(is_zero_bin),
                        "is_sparse_by_n": bool(is_sparse_by_n),
                        "is_sparse_by_fraction": bool(is_sparse_by_fraction),
                        "is_sparse_bin": bool(is_sparse_bin),
                        "sparse_min_n": cfg.sparse_min_n,
                        "sparse_min_fraction": cfg.sparse_min_fraction,
                    }
                )
    audit = pd.DataFrame(rows)
    if cfg.fail_on_missing_primary_bin and audit["missing_entry"].any():
        missing = audit.loc[audit["missing_entry"], ["condition", "split", "level"]].to_dict("records")
        raise ValueError(f"Missing required primary condition split/bin entries: {missing}")
    return audit


def build_zero_sparse_summary(audit: pd.DataFrame) -> pd.DataFrame:
    return (
        audit.groupby(["condition", "split"], as_index=False)
        .agg(
            n_bins=("level", "count"),
            n_zero_bins=("is_zero_bin", "sum"),
            n_sparse_bins=("is_sparse_bin", "sum"),
            min_bin_n=("n_sequences", "min"),
            min_bin_fraction=("fraction", "min"),
        )
    )


def save_source_data(cfg: Config, s5_panels: Dict[str, pd.DataFrame], s5_all: pd.DataFrame, s6_panels: Dict[str, pd.DataFrame], s6_all: pd.DataFrame) -> None:
    save_csv(s5_all, cfg.source_data_dir / "Supplementary_Figure_S5_source_data_all_panels.csv")
    save_csv(s6_all, cfg.source_data_dir / "Supplementary_Figure_S6_source_data_all_panels.csv")
    for panel, df in s5_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S5_panel_{panel.lower()}_source_data.csv")
    for panel, df in s6_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S6_panel_{panel.lower()}_source_data.csv")


# =============================================================================
# 7. Reports
# =============================================================================

def descriptor_summary(s5_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (desc, split), sub in s5_all.groupby(["descriptor", "split"], dropna=False):
        vals = finite_array(sub["value"])
        rows.append(
            {
                "descriptor": desc,
                "descriptor_display": DESCRIPTOR_LABELS.get(desc, str(desc)),
                "split": split,
                "split_display": SPLIT_DISPLAY.get(str(split), str(split)),
                "n": int(len(vals)),
                "mean": float(np.mean(vals)) if len(vals) else np.nan,
                "sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
                "median": float(np.median(vals)) if len(vals) else np.nan,
                "q25": float(np.percentile(vals, 25)) if len(vals) else np.nan,
                "q75": float(np.percentile(vals, 75)) if len(vals) else np.nan,
                "min": float(np.min(vals)) if len(vals) else np.nan,
                "max": float(np.max(vals)) if len(vals) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def save_reports(
    cfg: Config,
    schema_summary: pd.DataFrame,
    support_summary: pd.DataFrame,
    balance: pd.DataFrame,
    s5_all: pd.DataFrame,
    support_audit: pd.DataFrame,
    sparse_summary: pd.DataFrame,
) -> Dict[str, pd.DataFrame]:
    reports = {
        "condition_schema_summary": schema_summary,
        "condition_support_summary": support_summary,
        "condition_bin_balance_summary": balance,
        "condition_descriptor_summary": descriptor_summary(s5_all),
        "condition_bin_support_audit": support_audit,
        "condition_bin_zero_sparse_summary": sparse_summary,
    }
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
    return reports


# =============================================================================
# 8. Plotting
# =============================================================================

def winsor_limits(arrays: Sequence[np.ndarray], lower: float = 0.005, upper: float = 0.995) -> Tuple[float, float]:
    """Return robust y-axis limits using pooled descriptor values."""
    nonempty = [arr for arr in arrays if len(arr) > 0]
    merged = np.concatenate(nonempty) if nonempty else np.array([0.0, 1.0])
    lo = float(np.quantile(merged, lower))
    hi = float(np.quantile(merged, upper))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(np.nanmin(merged)), float(np.nanmax(merged))
    pad = 0.07 * (hi - lo if hi > lo else 1.0)
    return lo - pad, hi + pad


def boxplot_panel(
    ax: plt.Axes,
    panel_df: pd.DataFrame,
    ylabel: str,
    title: str,
    cfg: Config,
    show_zero: bool = False,
    show_edges: bool = False,
) -> None:
    """Draw a clean PLOS-style boxplot panel.

    The goal of S5 is not density estimation; it is split-wise descriptor
    comparability and condition-threshold support. Boxplots are therefore
    preferred over violins for manuscript-scale readability.
    """
    positions = np.arange(1, 4)
    data = [finite_array(panel_df.loc[panel_df["split"] == split, "value"]) for split in cfg.split_order]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.46,
        patch_artist=True,
        showfliers=False,
        whis=(5, 95),
        medianprops=dict(color=PLOS["dark"], linewidth=1.35),
        whiskerprops=dict(color=PLOS["charcoal"], linewidth=0.85),
        capprops=dict(color=PLOS["charcoal"], linewidth=0.85),
        boxprops=dict(linewidth=1.05),
    )

    for patch, split in zip(bp["boxes"], cfg.split_order):
        patch.set_facecolor(SPLIT_FACE[split])
        patch.set_alpha(0.34)
        patch.set_edgecolor(SPLIT_EDGE[split])
        patch.set_linewidth(1.05)
        patch.set_zorder(3)

    # Add a subtle median dot to improve readability when printed small.
    medians = [float(np.median(arr)) if len(arr) else np.nan for arr in data]
    for pos, med, split in zip(positions, medians, cfg.split_order):
        if np.isfinite(med):
            ax.scatter(
                pos,
                med,
                s=16,
                color=SPLIT_EDGE[split],
                edgecolor="white",
                linewidth=0.35,
                zorder=5,
            )

    if show_zero:
        ax.axhline(0, color=PLOS["charcoal"], linestyle="--", linewidth=0.75, zorder=1)

    if show_edges:
        edge_values = [panel_df["train_bin_edge_1"].dropna().unique(), panel_df["train_bin_edge_2"].dropna().unique()]
        for edge_arr in edge_values:
            if len(edge_arr):
                ax.axhline(
                    float(edge_arr[0]),
                    color=PLOS["dark"],
                    linestyle=(0, (2.2, 2.2)),
                    linewidth=0.72,
                    alpha=0.72,
                    zorder=1,
                )

    ax.set_xticks(positions)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    lo, hi = winsor_limits(data)
    ax.set_ylim(lo, hi)
    beautify_axis(ax)


def plot_s5(s5_panels: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    """Generate Supplementary Figure S5 as clean boxplots."""
    set_plot_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.35))
    fig.subplots_adjust(left=0.095, right=0.985, top=0.915, bottom=0.145, wspace=0.34, hspace=0.43)

    specs = [
        ("A", "length", axes[0, 0], "Length (aa)", "Length", False, True),
        ("B", "net_charge_pH7", axes[0, 1], "Net charge at pH 7", "Net charge", True, True),
        ("C", "hydrophobicity_KD", axes[1, 0], "Mean KD hydrophobicity", "Hydrophobicity", True, True),
        ("D", "entropy", axes[1, 1], "Shannon entropy (bits)", "Shannon entropy", False, False),
    ]

    for panel, desc, ax, ylabel, title, zero, edges in specs:
        boxplot_panel(ax, s5_panels[panel], ylabel, title, cfg, show_zero=zero, show_edges=edges)
        panel_letter(ax, panel, cfg)

    handles = [Patch(facecolor=SPLIT_FACE[s], edgecolor=SPLIT_EDGE[s], label=SPLIT_DISPLAY[s], alpha=0.45) for s in cfg.split_order]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, 0.045),
        edgecolor=PLOS["light"],
        facecolor="white",
        fontsize=cfg.legend_size,
    )

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S5_redesigned", cfg)


def plot_grouped_bins(ax: plt.Axes, panel_df: pd.DataFrame, condition_title: str, cfg: Config) -> None:
    """Draw grouped bars for low/mid/high fractions across splits.

    Grouped bars are preferred here over stacked bars because they make
    differences in each low/mid/high bin easier to compare across partitions.
    """
    pivot = (
        panel_df.pivot_table(index="split", columns="level", values="fraction", fill_value=0.0, aggfunc="first")
        .reindex(index=cfg.split_order, columns=cfg.bin_order, fill_value=0.0)
    )

    x = np.arange(len(cfg.split_order))
    width = 0.23
    offsets = {"low": -width, "mid": 0.0, "high": width}

    for level in cfg.bin_order:
        vals = pivot[level].to_numpy(dtype=float)
        ax.bar(
            x + offsets[level],
            vals,
            width=width,
            color=BIN_FACE[level],
            edgecolor=BIN_EDGE[level],
            linewidth=0.65,
            label=BIN_DISPLAY[level],
            zorder=3,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylim(0, 0.62)
    ax.set_yticks([0, 0.25, 0.50])
    ax.set_ylabel("Fraction")
    ax.set_title(condition_title, pad=8)
    beautify_axis(ax)


def plot_support_summary(ax: plt.Axes, panel_df: pd.DataFrame, cfg: Config) -> None:
    df = panel_df.set_index("split").reindex(cfg.split_order).reset_index()
    x = np.arange(len(cfg.split_order))
    width = 0.34
    values_primary = pd.to_numeric(df["n_primary_condition_ids"], errors="coerce").fillna(0).to_numpy()
    values_full = pd.to_numeric(df["n_full_descriptor_bin_ids"], errors="coerce").fillna(0).to_numpy()

    bars1 = ax.bar(
        x - width / 2,
        values_primary,
        width=width,
        color=SUPPORT_FACE["primary"],
        edgecolor=SUPPORT_EDGE["primary"],
        linewidth=0.85,
        label="Primary condition IDs",
        zorder=3,
    )
    bars2 = ax.bar(
        x + width / 2,
        values_full,
        width=width,
        color=SUPPORT_FACE["full"],
        edgecolor=SUPPORT_EDGE["full"],
        linewidth=0.85,
        label="Full descriptor-bin IDs",
        zorder=3,
    )

    ymax = max(np.nanmax(values_primary), np.nanmax(values_full), 1)
    for bars in [bars1, bars2]:
        for bar in bars:
            value = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + ymax * 0.025,
                f"{int(value)}",
                ha="center",
                va="bottom",
                fontsize=cfg.annotation_size,
            )

    ax.set_xticks(x)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel("Supported condition IDs")
    ax.set_title("Condition-ID support", pad=8)
    ax.set_ylim(0, ymax * 1.24)
    ax.legend(
        frameon=True,
        edgecolor=PLOS["light"],
        facecolor="white",
        loc="upper right",
        bbox_to_anchor=(0.98, 1.02),
        borderaxespad=0.2,
        fontsize=cfg.legend_size,
    )
    beautify_axis(ax)


def plot_s6(s6_panels: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, str]:
    set_plot_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.55))
    fig.subplots_adjust(left=0.09, right=0.985, top=0.92, bottom=0.15, wspace=0.34, hspace=0.43)

    plot_grouped_bins(axes[0, 0], s6_panels["A"], "Length bins", cfg)
    panel_letter(axes[0, 0], "A", cfg)

    plot_grouped_bins(axes[0, 1], s6_panels["B"], "Net-charge bins", cfg)
    panel_letter(axes[0, 1], "B", cfg)

    plot_grouped_bins(axes[1, 0], s6_panels["C"], "Hydrophobicity bins", cfg)
    panel_letter(axes[1, 0], "C", cfg)

    plot_support_summary(axes[1, 1], s6_panels["D"], cfg)
    panel_letter(axes[1, 1], "D", cfg)

    handles = [Patch(facecolor=BIN_FACE[level], edgecolor="white", label=BIN_DISPLAY[level]) for level in cfg.bin_order]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        bbox_to_anchor=(0.5, 0.047),
        edgecolor=PLOS["light"],
        facecolor="white",
        fontsize=cfg.legend_size,
    )

    return save_figure(fig, cfg.supp_dir / "Supplementary_Figure_S6_redesigned", cfg)


# =============================================================================
# 9. Reproducibility outputs
# =============================================================================

def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env_commit = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env_commit:
        return env_commit
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=str(cfg.project_root),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=5,
            check=False,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        return None
    return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    file_name = globals().get("__file__", None)
    if file_name and Path(file_name).exists():
        script_path = str(Path(file_name).resolve())
        text = Path(file_name).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_03_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_03_PLOS_redesign_code.py")
    return script_path, script_hash


def write_no_main_figure_readme(cfg: Config) -> Optional[Path]:
    if not cfg.create_no_main_figure_readme:
        return None
    text = f"""# No main figure generated for OncoPep Step 3

Generated: {now_iso()}

Step 3 is a supplementary-only condition-support audit. It generates:
- Supplementary Figure S5
- Supplementary Figure S6

No Main Figure 2 is generated here.

Rationale:
Main Figure 2 is reserved for the OncoPep architecture and conditional
generation workflow. Step 3 only audits the descriptor and bin-support schema,
so promoting it to a main figure would weaken the main manuscript narrative.
"""
    path = cfg.main_figure_dir / "README_no_main_figure_for_step_03.txt"
    save_text(text, path)
    return path


def write_readme(
    cfg: Config,
    sequence_path: Path,
    schema_recomputed: bool,
    balance_inferred: bool,
    info_notes: Sequence[str],
    warnings: Sequence[str],
) -> Path:
    text = f"""# OncoPep Step 3 PLOS supplementary-figure package

Generated: {now_iso()}

Step 3 scope:
Conditioning schema and condition-support audit.

Main figure status:
No main figure is generated in Step 3. Main Figure 2 is reserved for the OncoPep
architecture/model workflow in a later figure-redesign step.

Input sequence-level file:
{sequence_path}

Output directory:
{cfg.output_root}

Generated figures:
- supplementary_figures/Supplementary_Figure_S5_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S6_redesigned.png/.pdf/.tiff

Scientific notes:
- Primary conditioning descriptors: length, net charge at pH 7, Kyte-Doolittle hydrophobicity.
- Shannon entropy is treated as a secondary/supporting descriptor unless explicitly stated otherwise in the model methods.
- Train-derived low/mid/high bin thresholds are used to audit condition support.
- Supplementary Figure S5 uses boxplots for clean split-wise descriptor comparison.
- Supplementary Figure S6 uses grouped bars for direct low/mid/high bin comparison.
- Descriptor support should not be described as perfect balance or biological validation.

Schema recomputed from training data: {schema_recomputed}
Condition balance inferred from sequence-level table: {balance_inferred}

Sparse-bin audit thresholds:
- sparse_min_n: {cfg.sparse_min_n}
- sparse_min_fraction: {cfg.sparse_min_fraction}

Informational notes:
{chr(10).join("- " + str(w) for w in info_notes) if info_notes else "- None"}

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
"""
    path = cfg.reports_dir / "README_step_03_outputs.txt"
    save_text(text, path)
    return path


def write_requirements(cfg: Config) -> Path:
    text = "\n".join(
        [
            f"python=={platform.python_version()}",
            f"numpy=={np.__version__}",
            f"pandas=={pd.__version__}",
            f"matplotlib=={mpl.__version__}",
            "",
        ]
    )
    path = cfg.reports_dir / "requirements_step_03_minimal.txt"
    save_text(text, path)
    return path


def create_root_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs: list[tuple[Path, Path]] = []
    for fig in ["S5", "S6"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv", cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))

    root_reports = [
        "condition_schema_summary.csv",
        "condition_support_summary.csv",
        "condition_bin_balance_summary.csv",
        "condition_descriptor_summary.csv",
        "condition_bin_support_audit.csv",
        "condition_bin_zero_sparse_summary.csv",
        "README_step_03_outputs.txt",
    ]
    for name in root_reports:
        pairs.append((cfg.reports_dir / name, cfg.output_root / name))

    pairs.append((cfg.code_dir / "OncoPep_step_03_PLOS_redesign_code.py", cfg.output_root / "OncoPep_step_03_PLOS_redesign_code.py"))

    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst)
            copied.append(str(dst))
    return copied


def build_manifest(
    cfg: Config,
    sequence_path: Path,
    schema_path: Optional[Path],
    balance_path: Optional[Path],
    schema_recomputed: bool,
    balance_inferred: bool,
    s5_paths: Dict[str, str],
    s6_paths: Dict[str, str],
    readme_path: Path,
    requirements_path: Path,
    no_main_readme_path: Optional[Path],
    script_path: Optional[str],
    script_hash: str,
    root_copies: Sequence[str],
    info_notes: Sequence[str],
    warnings: Sequence[str],
) -> Dict[str, Any]:
    input_files = {"sequence_level": sequence_path}
    if schema_path:
        input_files["schema"] = schema_path
    if balance_path:
        input_files["balance"] = balance_path

    return {
        "step": "step_03",
        "scope": "supplementary_only",
        "timestamp": now_iso(),
        "main_figure_generated": False,
        "main_figure_rationale": "Step 3 audits condition support; Main Figure 2 is reserved for OncoPep architecture/model workflow.",
        "project_root": str(cfg.project_root),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "input_files": {k: str(v) for k, v in input_files.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_files.items()},
        "schema_recomputed_from_training_data": schema_recomputed,
        "condition_balance_inferred": balance_inferred,
        "figures": {"S5": s5_paths, "S6": s6_paths},
        "source_data": {
            "S5_all": str(cfg.source_data_dir / "Supplementary_Figure_S5_source_data_all_panels.csv"),
            "S6_all": str(cfg.source_data_dir / "Supplementary_Figure_S6_source_data_all_panels.csv"),
            "S5_panel_a": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_a_source_data.csv"),
            "S5_panel_b": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_b_source_data.csv"),
            "S5_panel_c": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_c_source_data.csv"),
            "S5_panel_d": str(cfg.source_data_dir / "Supplementary_Figure_S5_panel_d_source_data.csv"),
            "S6_panel_a": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_a_source_data.csv"),
            "S6_panel_b": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_b_source_data.csv"),
            "S6_panel_c": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_c_source_data.csv"),
            "S6_panel_d": str(cfg.source_data_dir / "Supplementary_Figure_S6_panel_d_source_data.csv"),
        },
        "reports": {
            "condition_schema_summary": str(cfg.reports_dir / "condition_schema_summary.csv"),
            "condition_support_summary": str(cfg.reports_dir / "condition_support_summary.csv"),
            "condition_bin_balance_summary": str(cfg.reports_dir / "condition_bin_balance_summary.csv"),
            "condition_descriptor_summary": str(cfg.reports_dir / "condition_descriptor_summary.csv"),
            "condition_bin_support_audit": str(cfg.reports_dir / "condition_bin_support_audit.csv"),
            "condition_bin_zero_sparse_summary": str(cfg.reports_dir / "condition_bin_zero_sparse_summary.csv"),
            "readme": str(readme_path),
            "requirements": str(requirements_path),
            "no_main_figure_readme": str(no_main_readme_path) if no_main_readme_path else None,
        },
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(root_copies),
        "info_notes": list(info_notes),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# =============================================================================
# 10. Main workflow
# =============================================================================

def run_step3_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_dirs(cfg)
    set_plot_style(cfg)

    info_notes: list[str] = [
        "Step 3 is supplementary-only; no Main Figure 2 is generated.",
        "Entropy is treated as a secondary/supporting descriptor in S5 and full descriptor-bin IDs.",
    ]
    warnings: list[str] = []

    conditioned_df, split_col, sequence_path = load_conditioned_sequences(cfg)
    schema, schema_path, schema_recomputed, schema_warnings = load_or_build_schema(conditioned_df, split_col, cfg)
    warnings.extend(schema_warnings)

    conditioned_df = ensure_bin_columns(conditioned_df, schema)
    conditioned_df = add_condition_ids(conditioned_df, cfg)
    balance_df, balance_path, balance_inferred = load_or_infer_balance(conditioned_df, split_col, schema, cfg)

    support_summary = build_support_summary(conditioned_df, split_col)
    schema_summary = build_schema_summary(schema)
    support_audit = build_condition_bin_support_audit(balance_df, cfg)
    sparse_summary = build_zero_sparse_summary(support_audit)

    zero_or_sparse = support_audit.loc[support_audit["is_sparse_bin"] | support_audit["is_zero_bin"]].copy()
    if not zero_or_sparse.empty:
        n_zero = int(zero_or_sparse["is_zero_bin"].sum())
        n_sparse = int(zero_or_sparse["is_sparse_bin"].sum())
        warnings.append(
            f"Condition-bin support audit detected {n_zero} zero bins and {n_sparse} sparse bins "
            f"using sparse_min_n={cfg.sparse_min_n}, sparse_min_fraction={cfg.sparse_min_fraction}. "
            f"Inspect reports/condition_bin_support_audit.csv."
        )

    s5_panels, s5_all = build_s5_source_data(conditioned_df, split_col, schema, cfg)
    s6_panels, s6_all = build_s6_source_data(balance_df, support_summary, cfg)

    save_source_data(cfg, s5_panels, s5_all, s6_panels, s6_all)
    reports = save_reports(cfg, schema_summary, support_summary, balance_df, s5_all, support_audit, sparse_summary)

    s5_paths = plot_s5(s5_panels, cfg)
    s6_paths = plot_s6(s6_panels, cfg)

    script_path, script_hash = code_snapshot(cfg)
    no_main_readme_path = write_no_main_figure_readme(cfg)
    readme_path = write_readme(cfg, sequence_path, schema_recomputed, balance_inferred, info_notes, warnings)
    requirements_path = write_requirements(cfg)
    root_copies = create_root_copies(cfg)

    manifest = build_manifest(
        cfg=cfg,
        sequence_path=sequence_path,
        schema_path=schema_path,
        balance_path=balance_path,
        schema_recomputed=schema_recomputed,
        balance_inferred=balance_inferred,
        s5_paths=s5_paths,
        s6_paths=s6_paths,
        readme_path=readme_path,
        requirements_path=requirements_path,
        no_main_readme_path=no_main_readme_path,
        script_path=script_path,
        script_hash=script_hash,
        root_copies=root_copies,
        info_notes=info_notes,
        warnings=warnings,
    )
    save_json(manifest, cfg.reports_dir / "step_03_manifest.json")
    save_json(manifest, cfg.output_root / "step_03_manifest.json")

    print("\n✅ Step 3 PLOS supplementary-figure package generated successfully.")
    print("Scope: supplementary-only condition-support audit")
    print("Main figure generated: No")
    print(f"Input sequence file: {sequence_path}")
    print(f"Output directory: {cfg.output_root}")

    for label, paths in [("Supplementary Figure S5", s5_paths), ("Supplementary Figure S6", s6_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")

    print("\nSource data:")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S5_source_data_all_panels.csv'}")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S6_source_data_all_panels.csv'}")
    for fig in ["S5", "S6"]:
        for panel in ["a", "b", "c", "d"]:
            print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_{fig}_panel_{panel}_source_data.csv'}")

    print("\nReports:")
    for name in [
        "condition_schema_summary.csv",
        "condition_support_summary.csv",
        "condition_bin_balance_summary.csv",
        "condition_descriptor_summary.csv",
        "condition_bin_support_audit.csv",
        "condition_bin_zero_sparse_summary.csv",
        "step_03_manifest.json",
        "README_step_03_outputs.txt",
    ]:
        print(f"  - {cfg.reports_dir / name}")
    if no_main_readme_path:
        print(f"  - {no_main_readme_path}")

    if info_notes:
        print("\nInformational notes:")
        for note in info_notes:
            print(f"  - {note}")

    if warnings:
        print("\nWarnings:")
        for warning in warnings:
            print(f"  - {warning}")

    return {
        "conditioned_df": conditioned_df,
        "schema": schema,
        "balance_df": balance_df,
        "support_summary": support_summary,
        "schema_summary": schema_summary,
        "support_audit": support_audit,
        "sparse_summary": sparse_summary,
        "s5_panels": s5_panels,
        "s6_panels": s6_panels,
        "reports": reports,
        "manifest": manifest,
        "info_notes": info_notes,
        "warnings": warnings,
    }


# =============================================================================
# 11. Notebook and CLI entry points
# =============================================================================

def main_notebook(
    project_root: str | Path = PROJECT_ROOT_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    step3_relative_dir: str | Path = STEP3_RELATIVE_DIR_DEFAULT,
    source_sequence_file: Optional[str | Path] = None,
    source_schema_file: Optional[str | Path] = None,
    source_balance_file: Optional[str | Path] = None,
    sparse_min_n: int = 10,
    sparse_min_fraction: float = 0.005,
    fail_on_missing_primary_bin: bool = True,
    create_root_level_copies: bool = True,
    create_no_main_figure_readme: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        project_root=Path(project_root),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        step3_relative_dir=Path(step3_relative_dir),
        source_sequence_file=Path(source_sequence_file) if source_sequence_file else None,
        source_schema_file=Path(source_schema_file) if source_schema_file else None,
        source_balance_file=Path(source_balance_file) if source_balance_file else None,
        sparse_min_n=sparse_min_n,
        sparse_min_fraction=sparse_min_fraction,
        fail_on_missing_primary_bin=fail_on_missing_primary_bin,
        create_root_level_copies=create_root_level_copies,
        create_no_main_figure_readme=create_no_main_figure_readme,
        git_commit=git_commit,
    )
    return run_step3_redesign(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 3 PLOS supplementary figures S5/S6.")
    parser.add_argument("--project_root", default=str(PROJECT_ROOT_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--step3_relative_dir", default=str(STEP3_RELATIVE_DIR_DEFAULT))
    parser.add_argument("--source_sequence_file", default=None)
    parser.add_argument("--source_schema_file", default=None)
    parser.add_argument("--source_balance_file", default=None)
    parser.add_argument("--sparse_min_n", type=int, default=10)
    parser.add_argument("--sparse_min_fraction", type=float, default=0.005)
    parser.add_argument("--allow_missing_primary_bin", action="store_true")
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    parser.add_argument("--no_main_readme", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    cfg = Config(
        project_root=Path(args.project_root),
        base_dir=Path(args.base_dir),
        output_root_name=args.output_root_name,
        step3_relative_dir=Path(args.step3_relative_dir),
        source_sequence_file=Path(args.source_sequence_file) if args.source_sequence_file else None,
        source_schema_file=Path(args.source_schema_file) if args.source_schema_file else None,
        source_balance_file=Path(args.source_balance_file) if args.source_balance_file else None,
        sparse_min_n=args.sparse_min_n,
        sparse_min_fraction=args.sparse_min_fraction,
        fail_on_missing_primary_bin=not args.allow_missing_primary_bin,
        create_root_level_copies=not args.no_root_copies,
        create_no_main_figure_readme=not args.no_main_readme,
        git_commit=args.git_commit,
    )
    run_step3_redesign(cfg)

OncoPep Step 4 — PLOS Computational Biology supplementary-figure redesign.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 4 — PLOS Computational Biology supplementary-figure redesign.

Step 4 = frozen preprocessing, tokenization, sequence-length, padding, and
amino-acid composition audit.

Outputs:
  Supplementary Figure S7 — frozen sequence preprocessing and tokenization audit
  Supplementary Figure S8 — amino-acid composition preservation across partitions

No main figure is generated in Step 4. Main Figure 2 is reserved for the
OncoPep architecture and conditional generation workflow.

v2 improvements:
  - No silent zero default for truncation or unknown-token metrics.
  - Split-level metric-source/provenance flags are exported.
  - Padding inference is explicitly flagged when sequence-level padding_fraction
    is absent from the model-ready table.
  - Duplicate split-summary rows are detected.
  - Unknown-token, cap/filtering, and noncanonical-residue audit tables are
    exported for reviewer inspection.
  - Manifest records preprocessing parameters, vocabulary, and metric sources.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd


PROJECT_ROOT_DEFAULT = Path("/home/data3/Moe/nature_computational2")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_04"

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
AA_ORDER = tuple("ACDEFGHIKLMNPQRSTVWY")

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}
SPLIT_FACE = {"train": PLOS["blue"], "val": PLOS["mint"], "test": PLOS["coral"]}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}
HELDOUT_FACE = {"val": PLOS["mint"], "test": PLOS["coral"]}
HELDOUT_EDGE = {"val": "#6F9F7A", "test": "#A84F42"}


@dataclass
class Config:
    project_root: Path = PROJECT_ROOT_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT

    model_ready_file: Optional[Path] = None
    length_summary_file: Optional[Path] = None
    token_summary_file: Optional[Path] = None
    qc_summary_file: Optional[Path] = None

    split_order: Tuple[str, ...] = SPLIT_ORDER
    aa_order: Tuple[str, ...] = AA_ORDER

    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    create_no_main_figure_readme: bool = True
    git_commit: Optional[str] = None

    font_size: float = 8.3
    title_size: float = 9.2
    axis_label_size: float = 8.9
    tick_label_size: float = 7.8
    legend_size: float = 7.7
    annotation_size: float = 7.3
    panel_letter_size: float = 13.0

    @property
    def step4_tables_dir(self) -> Path:
        return self.project_root / "step4" / "tables"

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_figure_dir(self) -> Path:
        return self.output_root / "main_figure"


# ----------------------------- utilities -------------------------------------

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def ensure_dirs(cfg: Config) -> None:
    for d in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        d.mkdir(parents=True, exist_ok=True)
    if cfg.create_no_main_figure_readme:
        cfg.main_figure_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_json(data: Any, path: Path) -> None:
    def default(o: Any) -> Any:
        if isinstance(o, Path):
            return str(o)
        if isinstance(o, tuple):
            return list(o)
        if isinstance(o, (np.integer, np.floating, np.bool_)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(f"{type(o).__name__} is not JSON serializable")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def normalize_split(s: pd.Series) -> pd.Series:
    return (
        s.astype(str).str.strip().str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_col(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"Could not find one of {list(candidates)}. Available columns: {list(df.columns)}")
    return None


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    return None


def get_value(df: pd.DataFrame, split: str, col: str, default: float = np.nan) -> float:
    if "split" not in df.columns or col not in df.columns:
        return float(default)
    sub = df.loc[df["split"] == split, col]
    if sub.empty:
        return float(default)
    val = pd.to_numeric(sub, errors="coerce").dropna()
    return float(val.iloc[0]) if len(val) else float(default)


def validate_one_row_per_split(df: pd.DataFrame, table_name: str, cfg: Config, warnings: list[str]) -> None:
    """Warn if a split-summary table has duplicate rows for the same split."""
    if "split" not in df.columns:
        warnings.append(f"{table_name} has no split column; split-level validation could not be performed.")
        return
    counts = df["split"].value_counts(dropna=False)
    duplicates = counts[counts > 1]
    if not duplicates.empty:
        warnings.append(
            f"{table_name} contains duplicate split rows {duplicates.to_dict()}; "
            "first non-missing value will be used for plotted split-level metrics."
        )


def metric_value_and_source(
    df: pd.DataFrame,
    split: str,
    col: str,
    table_name: str,
    allow_missing: bool,
    warnings: list[str],
    default: float = np.nan,
) -> tuple[float, str]:
    """Return a split-level metric and a provenance/source flag.

    Missing truncation and unknown-token metrics should not silently become zero.
    For optional metrics, missing values are returned as NA with a source flag.
    """
    if "split" not in df.columns:
        msg = f"{table_name} has no split column; metric '{col}' unavailable for split '{split}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_split_column"
        raise ValueError(msg)

    if col not in df.columns:
        msg = f"{table_name} is missing required metric column '{col}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_column"
        raise ValueError(msg)

    sub = df.loc[df["split"] == split, col]
    if sub.empty:
        msg = f"{table_name} has no row for split '{split}' when reading '{col}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_split"
        raise ValueError(msg)

    vals = pd.to_numeric(sub, errors="coerce").dropna()
    if vals.empty:
        msg = f"{table_name} has only nonnumeric/NA values for '{col}' in split '{split}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_value"
        raise ValueError(msg)

    return float(vals.iloc[0]), f"measured:{table_name}.{col}"


def clamp_fraction(value: float, metric_name: str, warnings: list[str]) -> float:
    """Constrain a numeric metric to [0, 1] and warn if clipping was needed."""
    if not np.isfinite(value):
        return value
    clipped = min(max(float(value), 0.0), 1.0)
    if clipped != float(value):
        warnings.append(f"Metric '{metric_name}'={value} was outside [0,1] and was clipped to {clipped}.")
    return clipped


# ----------------------------- style -----------------------------------------

def set_style(cfg: Config) -> None:
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "legend.fontsize": cfg.legend_size,
        "axes.linewidth": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "text.color": PLOS["dark"],
        "axes.labelcolor": PLOS["dark"],
        "axes.edgecolor": PLOS["dark"],
        "xtick.color": PLOS["dark"],
        "ytick.color": PLOS["dark"],
    })


def beautify(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.45)
    ax.grid(False, axis="x")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.75)
        ax.spines[side].set_color(PLOS["dark"])


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.08) -> None:
    ax.text(dx, dy, letter.upper(), transform=ax.transAxes, fontsize=cfg.panel_letter_size,
            fontweight="bold", va="top", ha="left", color=PLOS["dark"])


def save_fig(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {"png": base.with_suffix(".png"), "pdf": base.with_suffix(".pdf"), "tiff": base.with_suffix(".tiff")}
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


# ----------------------------- loading ---------------------------------------

def candidates(cfg: Config, name: str, explicit: Optional[Path]) -> list[Path]:
    out = []
    if explicit:
        out.append(Path(explicit))
    out.extend([
        cfg.step4_tables_dir / name,
        cfg.project_root / "redesign" / "step4" / "tables" / name,
        Path("/home/data3/Moe/nature_computational2/step4/tables") / name,
    ])
    return out


def load_csv(cfg: Config, name: str, explicit: Optional[Path]) -> Tuple[pd.DataFrame, Path]:
    path = first_existing(candidates(cfg, name, explicit))
    if path is None:
        raise FileNotFoundError("Missing input table " + name + ". Checked:\n- " + "\n- ".join(map(str, candidates(cfg, name, explicit))))
    return pd.read_csv(path, low_memory=False), path


def load_inputs(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Path]]:
    model_ready, p1 = load_csv(cfg, "step4_model_ready_sequences.csv", cfg.model_ready_file)
    length_summary, p2 = load_csv(cfg, "step4_sequence_length_summary.csv", cfg.length_summary_file)
    token_summary, p3 = load_csv(cfg, "step4_token_coverage_summary.csv", cfg.token_summary_file)
    qc_summary, p4 = load_csv(cfg, "step4_tokenization_qc.csv", cfg.qc_summary_file)
    return model_ready, length_summary, token_summary, qc_summary, {
        "model_ready_sequences": p1,
        "sequence_length_summary": p2,
        "token_coverage_summary": p3,
        "tokenization_qc": p4,
    }


def infer_chosen_cap(length_summary: pd.DataFrame) -> int:
    for c in ["chosen_max_sequence_length", "max_sequence_length", "sequence_length_cap", "chosen_cap"]:
        if c in length_summary.columns:
            vals = pd.to_numeric(length_summary[c], errors="coerce").dropna()
            if len(vals):
                return int(vals.iloc[0])
    return 60


def infer_model_len(length_summary: pd.DataFrame, fallback: Optional[int] = None) -> int:
    for c in ["max_model_length_including_special_tokens", "max_model_length", "model_length"]:
        if c in length_summary.columns:
            vals = pd.to_numeric(length_summary[c], errors="coerce").dropna()
            if len(vals):
                return int(vals.iloc[0])
    return int(fallback if fallback is not None else infer_chosen_cap(length_summary))


def harmonize_inputs(model_ready: pd.DataFrame, length_summary: pd.DataFrame,
                     token_summary: pd.DataFrame, qc_summary: pd.DataFrame,
                     cfg: Config, warnings: list[str]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str, str, str]:
    seq_col = resolve_col(model_ready, ["sequence", "seq", "peptide_sequence"])
    split_col = resolve_col(model_ready, ["assigned_split", "split", "data_split"])
    if split_col != "assigned_split":
        model_ready = model_ready.rename(columns={split_col: "assigned_split"})
        split_col = "assigned_split"
    model_ready[split_col] = normalize_split(model_ready[split_col])

    if "sequence_length" not in model_ready.columns:
        model_ready["sequence_length"] = model_ready[seq_col].astype(str).map(len)

    cap = infer_chosen_cap(length_summary)
    model_len = infer_model_len(length_summary, fallback=cap)

    if "padding_fraction" in model_ready.columns:
        model_ready["padding_fraction"] = pd.to_numeric(model_ready["padding_fraction"], errors="coerce")
        padding_fraction_source = "measured:model_ready_sequences.padding_fraction"
    else:
        model_ready["padding_fraction"] = (1.0 - pd.to_numeric(model_ready["sequence_length"], errors="coerce") / max(model_len, 1)).clip(0, 1)
        padding_fraction_source = "inferred:1-sequence_length/max_model_length"
        warnings.append(
            "Sequence-level padding_fraction was absent and was inferred from sequence_length and max_model_length. "
            "If special tokens are used, confirm this approximation against the tokenizer output."
        )

    for df in [length_summary, token_summary, qc_summary]:
        c = resolve_col(df, ["split", "assigned_split", "data_split"], required=False)
        if c is not None:
            df.rename(columns={c: "split"}, inplace=True)
            df["split"] = normalize_split(df["split"])

    missing = [s for s in cfg.split_order if s not in set(model_ready[split_col].dropna())]
    if missing:
        raise ValueError(f"Missing expected splits in model-ready table: {missing}")
    if "split" not in token_summary.columns or "split" not in qc_summary.columns:
        raise ValueError("token_summary and qc_summary must include a split column.")

    validate_one_row_per_split(token_summary, "token_summary", cfg, warnings)
    validate_one_row_per_split(qc_summary, "qc_summary", cfg, warnings)
    return model_ready, length_summary, token_summary, qc_summary, seq_col, split_col, padding_fraction_source


# ----------------------------- analyses --------------------------------------

def build_length_audit(model_ready: pd.DataFrame, length_summary: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    cap = infer_chosen_cap(length_summary)
    model_len = infer_model_len(length_summary, fallback=cap)
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "sequence_length"], errors="coerce").dropna().to_numpy()
        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(vals)),
            "min_length": float(np.min(vals)) if len(vals) else np.nan,
            "q25_length": float(np.percentile(vals, 25)) if len(vals) else np.nan,
            "median_length": float(np.median(vals)) if len(vals) else np.nan,
            "mean_length": float(np.mean(vals)) if len(vals) else np.nan,
            "q75_length": float(np.percentile(vals, 75)) if len(vals) else np.nan,
            "max_length": float(np.max(vals)) if len(vals) else np.nan,
            "chosen_max_sequence_length": int(cap),
            "max_model_length_including_special_tokens": int(model_len),
            "fraction_exceeding_cap": float(np.mean(vals > cap)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def build_token_audit(model_ready: pd.DataFrame, token_summary: pd.DataFrame, qc_summary: pd.DataFrame,
                      split_col: str, cfg: Config, warnings: list[str], padding_fraction_source: str) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        sub = model_ready.loc[model_ready[split_col] == split]
        mean_pad_default = float(pd.to_numeric(sub["padding_fraction"], errors="coerce").mean())

        frac_trunc, frac_trunc_source = metric_value_and_source(
            qc_summary, split, "fraction_truncated", "qc_summary",
            allow_missing=False, warnings=warnings
        )
        unk, unk_source = metric_value_and_source(
            token_summary, split, "unknown_token_fraction_nonpad", "token_summary",
            allow_missing=False, warnings=warnings
        )

        # mean padding can be measured in qc_summary or derived from sequence-level padding_fraction.
        if "mean_padding_fraction" in qc_summary.columns:
            mean_pad, mean_pad_source = metric_value_and_source(
                qc_summary, split, "mean_padding_fraction", "qc_summary",
                allow_missing=True, warnings=warnings, default=mean_pad_default
            )
            if not np.isfinite(mean_pad):
                mean_pad = mean_pad_default
                mean_pad_source = f"derived_from_sequence_level:{padding_fraction_source}"
        else:
            mean_pad = mean_pad_default
            mean_pad_source = f"derived_from_sequence_level:{padding_fraction_source}"
            warnings.append("qc_summary is missing 'mean_padding_fraction'; split mean padding was derived from sequence-level padding_fraction.")

        cond_missing, cond_source = metric_value_and_source(
            qc_summary, split, "fraction_primary_condition_id_missing", "qc_summary",
            allow_missing=True, warnings=warnings, default=np.nan
        )

        frac_trunc = clamp_fraction(frac_trunc, "fraction_truncated", warnings)
        unk = clamp_fraction(unk, "unknown_token_fraction_nonpad", warnings)
        mean_pad = clamp_fraction(mean_pad, "mean_padding_fraction", warnings)
        cond_missing = clamp_fraction(cond_missing, "fraction_primary_condition_id_missing", warnings)

        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(sub)),
            "fraction_truncated": frac_trunc,
            "fraction_truncated_source": frac_trunc_source,
            "non_truncated_fraction": 1.0 - frac_trunc if np.isfinite(frac_trunc) else np.nan,
            "unknown_token_fraction_nonpad": unk,
            "unknown_token_fraction_nonpad_source": unk_source,
            "known_token_fraction_nonpad": 1.0 - unk if np.isfinite(unk) else np.nan,
            "mean_padding_fraction": mean_pad,
            "mean_padding_fraction_source": mean_pad_source,
            "effective_token_fraction": 1.0 - mean_pad if np.isfinite(mean_pad) else np.nan,
            "fraction_primary_condition_id_missing": cond_missing,
            "fraction_primary_condition_id_missing_source": cond_source,
            "valid_condition_fraction": 1.0 - cond_missing if np.isfinite(cond_missing) else np.nan,
        })
    return pd.DataFrame(rows)


def build_padding_summary(model_ready: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "padding_fraction"], errors="coerce").dropna().to_numpy()
        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(vals)),
            "mean_padding_fraction": float(np.mean(vals)) if len(vals) else np.nan,
            "sd_padding_fraction": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "median_padding_fraction": float(np.median(vals)) if len(vals) else np.nan,
            "q25_padding_fraction": float(np.percentile(vals, 25)) if len(vals) else np.nan,
            "q75_padding_fraction": float(np.percentile(vals, 75)) if len(vals) else np.nan,
            "min_padding_fraction": float(np.min(vals)) if len(vals) else np.nan,
            "max_padding_fraction": float(np.max(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def build_cap_filtering_audit(model_ready: pd.DataFrame, length_audit: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    """Audit model-ready length coverage relative to the selected cap.

    This does not infer pre-filtering losses unless raw pre-filter counts are
    available; it documents model-ready sequences that exceed the selected cap.
    """
    cap = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "sequence_length"], errors="coerce").dropna()
        n = int(len(vals))
        n_exceed = int((vals > cap).sum())
        rows.append({
            "split": split,
            "split_display": SPLIT_DISPLAY[split],
            "n_model_ready_sequences": n,
            "chosen_max_sequence_length": cap,
            "n_model_ready_sequences_exceeding_cap": n_exceed,
            "fraction_model_ready_sequences_exceeding_cap": float(n_exceed / n) if n else np.nan,
            "filtering_loss_available": False,
            "n_prefilter_sequences": np.nan,
            "n_filtered_by_length": np.nan,
            "filtering_fraction": np.nan,
            "interpretation_note": "Audit is based on model-ready sequences; pre-filter loss requires raw pre-filter counts.",
        })
    return pd.DataFrame(rows)


def build_unknown_token_audit(token_audit: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "split", "split_display", "n_sequences",
        "unknown_token_fraction_nonpad", "unknown_token_fraction_nonpad_source",
        "known_token_fraction_nonpad",
    ]
    return token_audit[[c for c in cols if c in token_audit.columns]].copy()


def build_noncanonical_residue_audit(model_ready: pd.DataFrame, seq_col: str, split_col: str, cfg: Config) -> pd.DataFrame:
    canonical = set(cfg.aa_order)
    rows = []
    for split in cfg.split_order:
        seqs = model_ready.loc[model_ready[split_col] == split, seq_col].astype(str).tolist()
        concat = "".join(seqs)
        total_chars = len(concat)
        canonical_count = sum(1 for ch in concat if ch in canonical)
        noncanonical_count = total_chars - canonical_count
        noncanonical_symbols = sorted(set(ch for ch in concat if ch not in canonical))
        rows.append({
            "split": split,
            "split_display": SPLIT_DISPLAY[split],
            "n_sequences": int(len(seqs)),
            "total_residue_characters": int(total_chars),
            "canonical_residue_characters": int(canonical_count),
            "noncanonical_residue_characters": int(noncanonical_count),
            "noncanonical_residue_fraction": float(noncanonical_count / total_chars) if total_chars else np.nan,
            "noncanonical_symbols": "".join(noncanonical_symbols) if noncanonical_symbols else "",
            "canonical_vocabulary": "".join(cfg.aa_order),
        })
    return pd.DataFrame(rows)


def build_aa_comp(model_ready: pd.DataFrame, seq_col: str, split_col: str, cfg: Config) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        concat = "".join(model_ready.loc[model_ready[split_col] == split, seq_col].astype(str).tolist())
        total = sum(concat.count(aa) for aa in cfg.aa_order)
        denom = max(total, 1)
        for aa in cfg.aa_order:
            rows.append({"split": split, "split_display": SPLIT_DISPLAY[split], "amino_acid": aa,
                         "count": int(concat.count(aa)), "canonical_total": int(total),
                         "fraction": float(concat.count(aa) / denom)})
    return pd.DataFrame(rows)


def build_comp_dev(aa_comp: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    piv = aa_comp.pivot_table(index="amino_acid", columns="split", values="fraction", aggfunc="first").reindex(cfg.aa_order)
    rows = []
    for aa in cfg.aa_order:
        tr, va, te = float(piv.loc[aa, "train"]), float(piv.loc[aa, "val"]), float(piv.loc[aa, "test"])
        rows.append({"amino_acid": aa, "train_fraction": tr, "val_fraction": va, "test_fraction": te,
                     "val_minus_train": va - tr, "test_minus_train": te - tr,
                     "abs_val_minus_train": abs(va - tr), "abs_test_minus_train": abs(te - tr)})
    dev = pd.DataFrame(rows)
    summary_rows = []
    for split in ["val", "test"]:
        tr = dev["train_fraction"].to_numpy(float)
        held = dev[f"{split}_fraction"].to_numpy(float)
        delta = held - tr
        r = float(np.corrcoef(tr, held)[0, 1]) if np.std(tr) > 0 and np.std(held) > 0 else np.nan
        summary_rows.append({"comparison": f"{SPLIT_DISPLAY[split]} vs Train", "heldout_split": split,
                             "pearson_r": r, "mean_absolute_deviation": float(np.mean(np.abs(delta))),
                             "max_absolute_deviation": float(np.max(np.abs(delta))),
                             "signed_deviation_mean": float(np.mean(delta)), "n_amino_acids": int(len(delta))})
    return dev, pd.DataFrame(summary_rows)


# ----------------------------- source data -----------------------------------

def build_s7_source(model_ready: pd.DataFrame, length_audit: pd.DataFrame, token_audit: pd.DataFrame, split_col: str, cfg: Config):
    cap = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
    model_len = int(length_audit["max_model_length_including_special_tokens"].dropna().iloc[0])

    a = model_ready[[split_col, "sequence_length"]].copy().rename(columns={split_col: "split", "sequence_length": "value"})
    a["split"] = normalize_split(a["split"]); a["split_display"] = a["split"].map(SPLIT_DISPLAY)
    a["chosen_max_sequence_length"] = cap; a["max_model_length_including_special_tokens"] = model_len
    a["panel"] = "A"; a["data_type"] = "sequence_length_distribution"; a["metric_name"] = "sequence_length"
    a["unit"] = "amino_acid_residues"; a["metric_definition"] = "Number of residues in the standardized model-ready peptide sequence."

    rows = []
    for _, r in token_audit.iterrows():
        for metric, label in [("fraction_truncated", "Truncated"), ("unknown_token_fraction_nonpad", "Unknown token")]:
            rows.append({"split": r["split"], "split_display": r["split_display"], "metric": metric,
                         "metric_display": label, "value": r[metric],
                         "metric_source": r.get(f"{metric}_source", "not_recorded"),
                         "unit": "fraction", "panel": "B",
                         "data_type": "truncation_unknown_token_audit",
                         "metric_definition": "Split-level preprocessing/tokenization burden metric."})
    b = pd.DataFrame(rows)

    c = model_ready[[split_col, "padding_fraction"]].copy().rename(columns={split_col: "split", "padding_fraction": "value"})
    c["split"] = normalize_split(c["split"]); c["split_display"] = c["split"].map(SPLIT_DISPLAY)
    c["panel"] = "C"; c["data_type"] = "padding_fraction_distribution"; c["metric_name"] = "padding_fraction"
    c["unit"] = "fraction"; c["metric_definition"] = "Fraction of final model input positions occupied by padding."

    metric_defs = [
        ("non_truncated_fraction", "Non-truncated"),
        ("known_token_fraction_nonpad", "Known-token"),
        ("effective_token_fraction", "Effective-token"),
        ("mean_padding_fraction", "Mean padding"),
    ]
    rows = []
    for _, r in token_audit.iterrows():
        for metric, label in metric_defs:
            rows.append({"split": r["split"], "split_display": r["split_display"], "metric": metric,
                         "metric_display": label, "value": r[metric],
                         "metric_source": r.get(f"{metric}_source", "derived_or_not_recorded"),
                         "unit": "fraction", "panel": "D",
                         "data_type": "split_level_preprocessing_integrity_audit",
                         "metric_definition": "Split-level preprocessing integrity summary metric."})
    d = pd.DataFrame(rows)

    panels = {"A": a, "B": b, "C": c, "D": d}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def build_s8_source(aa_comp: pd.DataFrame, dev: pd.DataFrame, summary: pd.DataFrame, cfg: Config):
    a = aa_comp.copy()
    a["panel"] = "A"; a["data_type"] = "amino_acid_frequency_profile"; a["metric_name"] = "amino_acid_fraction"
    a["unit"] = "fraction"; a["metric_definition"] = "Count of a canonical amino acid divided by total canonical amino-acid count in the split."

    rows = []
    for _, r in dev.iterrows():
        for split in ["val", "test"]:
            rows.append({"amino_acid": r["amino_acid"], "heldout_split": split,
                         "heldout_split_display": SPLIT_DISPLAY[split],
                         "value": r[f"{split}_minus_train"],
                         "absolute_value": r[f"abs_{split}_minus_train"],
                         "panel": "B", "data_type": "heldout_minus_train_composition_deviation",
                         "unit": "fraction_difference",
                         "metric_definition": "Held-out amino-acid fraction minus training amino-acid fraction."})
    b = pd.DataFrame(rows)

    rows = []
    for _, r in dev.iterrows():
        for split in ["val", "test"]:
            rows.append({"amino_acid": r["amino_acid"], "heldout_split": split,
                         "heldout_split_display": SPLIT_DISPLAY[split],
                         "train_fraction": r["train_fraction"],
                         "heldout_fraction": r[f"{split}_fraction"],
                         "panel": "C", "data_type": "heldout_vs_train_composition_agreement",
                         "unit": "fraction",
                         "metric_definition": "Training and held-out amino-acid fractions for composition agreement plotting."})
    c = pd.DataFrame(rows)

    d = summary.copy()
    d["panel"] = "D"; d["data_type"] = "composition_deviation_summary"
    d["unit"] = "fraction"; d["metric_definition"] = "Summary of held-out amino-acid composition deviation relative to training composition."
    panels = {"A": a, "B": b, "C": c, "D": d}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def save_sources(cfg: Config, s7_panels, s7_all, s8_panels, s8_all) -> None:
    save_csv(s7_all, cfg.source_data_dir / "Supplementary_Figure_S7_source_data_all_panels.csv")
    save_csv(s8_all, cfg.source_data_dir / "Supplementary_Figure_S8_source_data_all_panels.csv")
    for p, df in s7_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S7_panel_{p.lower()}_source_data.csv")
    for p, df in s8_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S8_panel_{p.lower()}_source_data.csv")


# ----------------------------- plotting --------------------------------------

def plot_s7a(ax, df, cfg):
    for split in cfg.split_order:
        vals = pd.to_numeric(df.loc[df["split"] == split, "value"], errors="coerce").dropna().sort_values().to_numpy()
        if len(vals):
            y = np.arange(1, len(vals) + 1) / len(vals)
            ax.step(vals, y, where="post", color=SPLIT_EDGE[split], lw=1.8, label=SPLIT_DISPLAY[split], zorder=3)
    cap_vals = df["chosen_max_sequence_length"].dropna().unique()
    if len(cap_vals):
        cap = float(cap_vals[0])
        ax.axvline(cap, color=PLOS["charcoal"], linestyle=(0, (3, 2.5)), lw=1.15, zorder=2)
        ax.text(cap + 0.6, 0.08, f"cap = {int(cap)}", fontsize=cfg.annotation_size, ha="left", va="bottom")
    ax.set_xlabel("Sequence length (aa)")
    ax.set_ylabel("Cumulative fraction")
    ax.set_ylim(0, 1.02)
    ax.set_title("Sequence-length coverage", pad=8)
    beautify(ax)
    ax.legend(loc="lower right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s7b(ax, df, cfg):
    metrics = [("fraction_truncated", "Truncated", PLOS["brown"]), ("unknown_token_fraction_nonpad", "Unknown token", PLOS["coral"])]
    x = np.arange(len(cfg.split_order)); width = 0.30
    ymax = 0.001
    for k, (metric, label, color) in enumerate(metrics):
        vals = [float(df.loc[(df["split"] == s) & (df["metric"] == metric), "value"].iloc[0]) for s in cfg.split_order]
        ymax = max(ymax, max(vals))
        bars = ax.bar(x + (k - 0.5) * width, vals, width=width, color=color, edgecolor=PLOS["dark"], lw=0.55, alpha=0.78, label=label, zorder=3)
        for bar, val in zip(bars, vals):
            txt = f"{val:.3f}" if val >= 0.001 else (f"{val:.1e}" if val > 0 else "0")
            ax.text(bar.get_x() + bar.get_width()/2, val + max(ymax * 0.035, 0.0003), txt, ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel("Fraction")
    ax.set_title("Truncation and unknown-token burden", pad=8)
    ax.set_ylim(0, max(0.02, ymax * 1.38))
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s7c(ax, df, cfg):
    positions = np.arange(1, 4)
    data = [pd.to_numeric(df.loc[df["split"] == s, "value"], errors="coerce").dropna().to_numpy() for s in cfg.split_order]
    bp = ax.boxplot(data, positions=positions, widths=0.46, patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops=dict(color=PLOS["dark"], lw=1.35),
                    whiskerprops=dict(color=PLOS["charcoal"], lw=0.85),
                    capprops=dict(color=PLOS["charcoal"], lw=0.85),
                    boxprops=dict(lw=1.05))
    for patch, split in zip(bp["boxes"], cfg.split_order):
        patch.set_facecolor(SPLIT_FACE[split]); patch.set_alpha(0.34); patch.set_edgecolor(SPLIT_EDGE[split])
    for pos, arr, split in zip(positions, data, cfg.split_order):
        if len(arr):
            ax.scatter(pos, np.median(arr), s=16, color=SPLIT_EDGE[split], edgecolor="white", lw=0.35, zorder=5)
    ax.set_xticks(positions); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel("Padding fraction")
    ax.set_ylim(0, 1.0)
    ax.set_title("Per-sequence padding burden", pad=8)
    beautify(ax)


def plot_s7d(ax, df, cfg):
    metrics = [("non_truncated_fraction", "Non-\ntruncated"), ("known_token_fraction_nonpad", "Known-\ntoken"),
               ("effective_token_fraction", "Effective-\ntoken"), ("mean_padding_fraction", "Mean\npadding")]
    ax.set_xlim(0, len(metrics)); ax.set_ylim(0, len(cfg.split_order)); ax.invert_yaxis()
    for i, split in enumerate(cfg.split_order):
        for j, (metric, label) in enumerate(metrics):
            sub = df.loc[(df["split"] == split) & (df["metric"] == metric), "value"]
            val = float(sub.iloc[0]) if len(sub) and pd.notna(sub.iloc[0]) else np.nan
            base = PLOS["light"] if metric == "mean_padding_fraction" else PLOS["blue"]
            alpha = 0.10 + 0.48 * min(max(val, 0), 1) if np.isfinite(val) else 0.08
            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=mpl.colors.to_rgba(base, alpha=alpha),
                                   edgecolor=PLOS["light"], lw=0.7))
            ax.text(j + 0.5, i + 0.53, f"{val:.2f}" if np.isfinite(val) else "NA",
                    ha="center", va="center", fontsize=cfg.annotation_size + 0.8)
    for j, (_, label) in enumerate(metrics):
        ax.text(j + 0.5, -0.15, label, ha="center", va="bottom", fontsize=cfg.tick_label_size)
    for i, split in enumerate(cfg.split_order):
        ax.text(-0.10, i + 0.5, SPLIT_DISPLAY[split], ha="right", va="center", fontsize=cfg.tick_label_size)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_title("Split-level preprocessing audit", pad=20)


def plot_s7(panels, cfg):
    set_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(10.6, 6.4))
    fig.subplots_adjust(left=0.09, right=0.985, top=0.915, bottom=0.095, wspace=0.34, hspace=0.47)
    plot_s7a(axes[0,0], panels["A"], cfg); panel_letter(axes[0,0], "A", cfg)
    plot_s7b(axes[0,1], panels["B"], cfg); panel_letter(axes[0,1], "B", cfg)
    plot_s7c(axes[1,0], panels["C"], cfg); panel_letter(axes[1,0], "C", cfg)
    plot_s7d(axes[1,1], panels["D"], cfg); panel_letter(axes[1,1], "D", cfg)
    return save_fig(fig, cfg.supp_dir / "Supplementary_Figure_S7_redesigned", cfg)


def plot_s8a(ax, df, cfg):
    x = np.arange(len(cfg.aa_order)); width = 0.23
    offsets = {"train": -width, "val": 0.0, "test": width}
    for split in cfg.split_order:
        sub = df.loc[df["split"] == split].set_index("amino_acid").reindex(cfg.aa_order)
        vals = pd.to_numeric(sub["fraction"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offsets[split], vals, width=width, color=SPLIT_FACE[split],
               edgecolor=SPLIT_EDGE[split], lw=0.45, alpha=0.78, label=SPLIT_DISPLAY[split], zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(cfg.aa_order)
    ax.set_xlabel("Amino acid"); ax.set_ylabel("Fraction")
    ax.set_title("Amino-acid frequency profile", pad=8)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8b(ax, df, cfg):
    x = np.arange(len(cfg.aa_order)); width = 0.23
    for split, offset in [("val", -width/2), ("test", width/2)]:
        sub = df.loc[df["heldout_split"] == split].set_index("amino_acid").reindex(cfg.aa_order)
        vals = pd.to_numeric(sub["value"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offset, vals, width=width, color=HELDOUT_FACE[split], edgecolor=HELDOUT_EDGE[split],
               lw=0.45, alpha=0.82, label=f"{SPLIT_DISPLAY[split]} − Train", zorder=3)
    ax.axhline(0, color=PLOS["charcoal"], lw=0.9, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels(cfg.aa_order)
    ax.set_xlabel("Amino acid"); ax.set_ylabel("Δ fraction vs train")
    ax.set_title("Held-out deviation from train", pad=8)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8c(ax, df, summary, cfg):
    maxv = max(float(pd.to_numeric(df["train_fraction"], errors="coerce").max()),
               float(pd.to_numeric(df["heldout_fraction"], errors="coerce").max()), 0.01)
    lim = maxv * 1.12
    ax.plot([0, lim], [0, lim], color=PLOS["charcoal"], linestyle=(0, (3, 2.5)), lw=0.9, zorder=1)
    for split in ["val", "test"]:
        sub = df.loc[df["heldout_split"] == split]
        ax.scatter(sub["train_fraction"], sub["heldout_fraction"], s=32, color=HELDOUT_FACE[split],
                   edgecolor=HELDOUT_EDGE[split], lw=0.6, alpha=0.88, label=SPLIT_DISPLAY[split], zorder=3)
    ann = []
    for _, r in summary.iterrows():
        ann.append(f"{SPLIT_DISPLAY.get(r['heldout_split'], r['heldout_split'])}: r={r['pearson_r']:.3f}, MAE={r['mean_absolute_deviation']:.3f}")
    if ann:
        ax.text(0.04, 0.96, "\n".join(ann), transform=ax.transAxes, ha="left", va="top",
                fontsize=cfg.annotation_size, bbox=dict(facecolor="white", edgecolor=PLOS["light"], lw=0.6, pad=2.5))
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Train fraction"); ax.set_ylabel("Held-out fraction")
    ax.set_title("Held-out vs train composition", pad=8)
    beautify(ax)
    ax.legend(loc="lower right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8d(ax, df, cfg):
    metrics = [("mean_absolute_deviation", "Mean absolute\nΔ fraction"), ("max_absolute_deviation", "Maximum absolute\nΔ fraction")]
    x = np.arange(len(metrics)); width = 0.32; ymax = 0.001
    for split, offset in [("val", -width/2), ("test", width/2)]:
        sub = df.loc[df["heldout_split"] == split]
        vals = [float(pd.to_numeric(sub[m], errors="coerce").iloc[0]) if len(sub) else 0.0 for m, _ in metrics]
        ymax = max(ymax, max(vals))
        bars = ax.bar(x + offset, vals, width=width, color=HELDOUT_FACE[split], edgecolor=HELDOUT_EDGE[split],
                      lw=0.65, alpha=0.82, label=SPLIT_DISPLAY[split], zorder=3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + ymax * 0.035, f"{val:.3f}",
                    ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels([m[1] for m in metrics])
    ax.set_ylabel("Deviation")
    ax.set_title("Composition-deviation summary", pad=8)
    ax.set_ylim(0, ymax * 1.35 if ymax > 0 else 0.01)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8(panels, summary, cfg):
    set_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(11.1, 6.5))
    fig.subplots_adjust(left=0.082, right=0.985, top=0.915, bottom=0.105, wspace=0.31, hspace=0.45)
    plot_s8a(axes[0,0], panels["A"], cfg); panel_letter(axes[0,0], "A", cfg)
    plot_s8b(axes[0,1], panels["B"], cfg); panel_letter(axes[0,1], "B", cfg)
    plot_s8c(axes[1,0], panels["C"], summary, cfg); panel_letter(axes[1,0], "C", cfg)
    plot_s8d(axes[1,1], panels["D"], cfg); panel_letter(axes[1,1], "D", cfg)
    return save_fig(fig, cfg.supp_dir / "Supplementary_Figure_S8_redesigned", cfg)


# ------------------------- reproducibility outputs ---------------------------

def save_reports(cfg, length_audit, token_audit, pad_summary, aa_comp, dev, summary,
                 cap_filtering_audit=None, unknown_token_audit=None, noncanonical_residue_audit=None):
    reports = {
        "sequence_length_audit": length_audit,
        "tokenization_audit": token_audit,
        "padding_burden_summary": pad_summary,
        "amino_acid_composition_summary": aa_comp,
        "composition_deviation_audit": dev,
        "composition_similarity_summary": summary,
    }
    if cap_filtering_audit is not None:
        reports["cap_filtering_audit"] = cap_filtering_audit
    if unknown_token_audit is not None:
        reports["unknown_token_audit"] = unknown_token_audit
    if noncanonical_residue_audit is not None:
        reports["noncanonical_residue_audit"] = noncanonical_residue_audit
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
    return reports


def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env:
        return env
    try:
        res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(cfg.project_root),
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
        return res.stdout.strip() if res.returncode == 0 else None
    except Exception:
        return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    fname = globals().get("__file__", None)
    if fname and Path(fname).exists():
        script_path = str(Path(fname).resolve())
        text = Path(fname).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_04_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_04_PLOS_redesign_code.py")
    return script_path, script_hash


def write_no_main_readme(cfg: Config) -> Optional[Path]:
    if not cfg.create_no_main_figure_readme:
        return None
    path = cfg.main_figure_dir / "README_no_main_figure_for_step_04.txt"
    save_text(f"""# No main figure generated for OncoPep Step 4

Generated: {now_iso()}

Step 4 is a supplementary-only frozen preprocessing and tokenization audit.

Generated figures:
- Supplementary Figure S7
- Supplementary Figure S8

No Main Figure 2 is generated here. Main Figure 2 is reserved for the OncoPep
architecture and conditional generation workflow.
""", path)
    return path


def write_readme(cfg: Config, input_paths: Dict[str, Path], warnings: Sequence[str]) -> Path:
    path = cfg.reports_dir / "README_step_04_outputs.txt"
    save_text(f"""# OncoPep Step 4 PLOS supplementary-figure package

Generated: {now_iso()}

Scope:
Frozen sequence preprocessing, tokenization, sequence-length, padding, and
amino-acid composition audit.

No main figure is generated in Step 4. Main Figure 2 is reserved for the
OncoPep architecture/model workflow.

Input files:
{chr(10).join(f"- {k}: {v}" for k, v in input_paths.items())}

Generated figures:
- supplementary_figures/Supplementary_Figure_S7_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S8_redesigned.png/.pdf/.tiff

Scientific notes:
- Step 4 does not repeat Step 2 leakage audits or Step 3 conditioning-schema audits.
- Preprocessing QC supports model-input transparency, not biological validation.
- Unknown-token fraction is calculated among non-padding tokens and is required for S7B/S7D.
- Padding fraction is the fraction of final model input positions occupied by padding; if inferred rather than read directly, this is recorded in source data and the manifest.
- Amino-acid composition deviation is computed relative to the training partition.
- Cap/filtering and noncanonical-residue audit tables are exported for reviewer inspection.

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
""", path)
    return path


def write_requirements(cfg: Config) -> Path:
    path = cfg.reports_dir / "requirements_step_04_minimal.txt"
    save_text("\n".join([f"python=={platform.python_version()}",
                         f"numpy=={np.__version__}",
                         f"pandas=={pd.__version__}",
                         f"matplotlib=={mpl.__version__}", ""]), path)
    return path


def root_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs = []
    for fig in ["S7", "S8"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv",
                      cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))
    for name in ["sequence_length_audit.csv", "tokenization_audit.csv", "unknown_token_audit.csv",
                 "padding_burden_summary.csv", "cap_filtering_audit.csv",
                 "amino_acid_composition_summary.csv", "composition_deviation_audit.csv",
                 "composition_similarity_summary.csv", "noncanonical_residue_audit.csv",
                 "README_step_04_outputs.txt"]:
        pairs.append((cfg.reports_dir / name, cfg.output_root / name))
    pairs.append((cfg.code_dir / "OncoPep_step_04_PLOS_redesign_code.py", cfg.output_root / "OncoPep_step_04_PLOS_redesign_code.py"))
    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst); copied.append(str(dst))
    return copied


def build_manifest(cfg, input_paths, s7_paths, s8_paths, readme_path, req_path, no_main_path, script_path, script_hash, copies, warnings,
                   length_audit=None, token_audit=None, noncanonical_residue_audit=None, padding_fraction_source=None):
    preprocessing_parameters = {}
    if length_audit is not None and len(length_audit):
        preprocessing_parameters["chosen_max_sequence_length"] = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
        preprocessing_parameters["max_model_length_including_special_tokens"] = int(length_audit["max_model_length_including_special_tokens"].dropna().iloc[0])
    preprocessing_parameters["canonical_amino_acid_vocabulary"] = "".join(cfg.aa_order)
    preprocessing_parameters["padding_fraction_source"] = padding_fraction_source
    if token_audit is not None and len(token_audit):
        preprocessing_parameters["fraction_truncated_sources"] = token_audit.set_index("split")["fraction_truncated_source"].to_dict() if "fraction_truncated_source" in token_audit.columns else {}
        preprocessing_parameters["unknown_token_fraction_sources"] = token_audit.set_index("split")["unknown_token_fraction_nonpad_source"].to_dict() if "unknown_token_fraction_nonpad_source" in token_audit.columns else {}
    if noncanonical_residue_audit is not None and len(noncanonical_residue_audit):
        preprocessing_parameters["noncanonical_residue_fraction_by_split"] = noncanonical_residue_audit.set_index("split")["noncanonical_residue_fraction"].to_dict()
        preprocessing_parameters["noncanonical_symbols_by_split"] = noncanonical_residue_audit.set_index("split")["noncanonical_symbols"].to_dict()

    return {
        "step": "step_04",
        "scope": "supplementary_only",
        "timestamp": now_iso(),
        "main_figure_generated": False,
        "main_figure_rationale": "Step 4 audits frozen preprocessing/tokenization; Main Figure 2 is reserved for OncoPep architecture/model workflow.",
        "project_root": str(cfg.project_root),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "preprocessing_parameters": preprocessing_parameters,
        "input_files": {k: str(v) for k, v in input_paths.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_paths.items()},
        "figures": {"S7": s7_paths, "S8": s8_paths},
        "source_data": {
            "S7_all": str(cfg.source_data_dir / "Supplementary_Figure_S7_source_data_all_panels.csv"),
            "S8_all": str(cfg.source_data_dir / "Supplementary_Figure_S8_source_data_all_panels.csv"),
            **{f"S7_panel_{p.lower()}": str(cfg.source_data_dir / f"Supplementary_Figure_S7_panel_{p.lower()}_source_data.csv") for p in "ABCD"},
            **{f"S8_panel_{p.lower()}": str(cfg.source_data_dir / f"Supplementary_Figure_S8_panel_{p.lower()}_source_data.csv") for p in "ABCD"},
        },
        "reports": {
            "sequence_length_audit": str(cfg.reports_dir / "sequence_length_audit.csv"),
            "tokenization_audit": str(cfg.reports_dir / "tokenization_audit.csv"),
            "unknown_token_audit": str(cfg.reports_dir / "unknown_token_audit.csv"),
            "padding_burden_summary": str(cfg.reports_dir / "padding_burden_summary.csv"),
            "cap_filtering_audit": str(cfg.reports_dir / "cap_filtering_audit.csv"),
            "amino_acid_composition_summary": str(cfg.reports_dir / "amino_acid_composition_summary.csv"),
            "composition_deviation_audit": str(cfg.reports_dir / "composition_deviation_audit.csv"),
            "composition_similarity_summary": str(cfg.reports_dir / "composition_similarity_summary.csv"),
            "noncanonical_residue_audit": str(cfg.reports_dir / "noncanonical_residue_audit.csv"),
            "readme": str(readme_path),
            "requirements": str(req_path),
            "no_main_figure_readme": str(no_main_path) if no_main_path else None,
        },
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(copies),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# ----------------------------- main workflow ---------------------------------

def run_step4_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_dirs(cfg); set_style(cfg)
    warnings = []

    model_ready, length_summary, token_summary, qc_summary, input_paths = load_inputs(cfg)
    model_ready, length_summary, token_summary, qc_summary, seq_col, split_col, padding_fraction_source = harmonize_inputs(
        model_ready, length_summary, token_summary, qc_summary, cfg, warnings
    )

    length_audit = build_length_audit(model_ready, length_summary, split_col, cfg)
    token_audit = build_token_audit(model_ready, token_summary, qc_summary, split_col, cfg, warnings, padding_fraction_source)
    padding_summary = build_padding_summary(model_ready, split_col, cfg)
    cap_filtering_audit = build_cap_filtering_audit(model_ready, length_audit, split_col, cfg)
    aa_comp = build_aa_comp(model_ready, seq_col, split_col, cfg)
    dev, comp_summary = build_comp_dev(aa_comp, cfg)
    unknown_audit = build_unknown_token_audit(token_audit)
    noncanonical_audit = build_noncanonical_residue_audit(model_ready, seq_col, split_col, cfg)

    if (token_audit["fraction_truncated"].fillna(0) > 0).any():
        warnings.append("Non-zero truncation fraction detected; interpret preprocessing as quantified, not artifact-free.")
    if (token_audit["unknown_token_fraction_nonpad"].fillna(0) > 0).any():
        warnings.append("Non-zero unknown-token fraction detected; inspect tokenization_audit.csv.")
    if (padding_summary["mean_padding_fraction"].fillna(0) > 0.75).any():
        warnings.append("High mean padding fraction detected in at least one split.")

    s7_panels, s7_all = build_s7_source(model_ready, length_audit, token_audit, split_col, cfg)
    s8_panels, s8_all = build_s8_source(aa_comp, dev, comp_summary, cfg)
    save_sources(cfg, s7_panels, s7_all, s8_panels, s8_all)
    reports = save_reports(
        cfg, length_audit, token_audit, padding_summary, aa_comp, dev, comp_summary,
        cap_filtering_audit=cap_filtering_audit,
        unknown_token_audit=unknown_audit,
        noncanonical_residue_audit=noncanonical_audit,
    )

    s7_paths = plot_s7(s7_panels, cfg)
    s8_paths = plot_s8(s8_panels, comp_summary, cfg)

    script_path, script_hash = code_snapshot(cfg)
    no_main = write_no_main_readme(cfg)
    readme = write_readme(cfg, input_paths, warnings)
    req = write_requirements(cfg)
    copies = root_copies(cfg)

    manifest = build_manifest(
        cfg, input_paths, s7_paths, s8_paths, readme, req, no_main, script_path, script_hash, copies, warnings,
        length_audit=length_audit,
        token_audit=token_audit,
        noncanonical_residue_audit=noncanonical_audit,
        padding_fraction_source=padding_fraction_source,
    )
    save_json(manifest, cfg.reports_dir / "step_04_manifest.json")
    save_json(manifest, cfg.output_root / "step_04_manifest.json")

    print("\n✅ Step 4 PLOS supplementary-figure package generated successfully.")
    print("Scope: supplementary-only frozen preprocessing/tokenization audit")
    print("Main figure generated: No")
    print(f"Output directory: {cfg.output_root}")

    print("\nInput files:")
    for k, v in input_paths.items():
        print(f"  - {k}: {v}")

    for label, paths in [("Supplementary Figure S7", s7_paths), ("Supplementary Figure S8", s8_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")

    print("\nSource data:")
    for fig in ["S7", "S8"]:
        print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_{fig}_source_data_all_panels.csv'}")
        for p in "abcd":
            print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_{fig}_panel_{p}_source_data.csv'}")

    print("\nReports:")
    for name in ["sequence_length_audit.csv", "tokenization_audit.csv", "unknown_token_audit.csv",
                 "padding_burden_summary.csv", "cap_filtering_audit.csv",
                 "amino_acid_composition_summary.csv", "composition_deviation_audit.csv",
                 "composition_similarity_summary.csv", "noncanonical_residue_audit.csv",
                 "step_04_manifest.json", "README_step_04_outputs.txt"]:
        print(f"  - {cfg.reports_dir / name}")
    if no_main:
        print(f"  - {no_main}")
    if warnings:
        print("\nWarnings:")
        for w in warnings:
            print(f"  - {w}")

    return {
        "model_ready": model_ready,
        "length_summary": length_summary,
        "token_summary": token_summary,
        "qc_summary": qc_summary,
        "length_audit": length_audit,
        "tokenization_audit": token_audit,
        "padding_summary": padding_summary,
        "amino_acid_composition": aa_comp,
        "composition_deviation_audit": dev,
        "composition_similarity_summary": comp_summary,
        "cap_filtering_audit": cap_filtering_audit,
        "unknown_token_audit": unknown_audit,
        "noncanonical_residue_audit": noncanonical_audit,
        "s7_panels": s7_panels,
        "s8_panels": s8_panels,
        "reports": reports,
        "manifest": manifest,
        "warnings": warnings,
    }


def main_notebook(
    project_root: str | Path = PROJECT_ROOT_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    model_ready_file: Optional[str | Path] = None,
    length_summary_file: Optional[str | Path] = None,
    token_summary_file: Optional[str | Path] = None,
    qc_summary_file: Optional[str | Path] = None,
    create_root_level_copies: bool = True,
    create_no_main_figure_readme: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        project_root=Path(project_root),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        model_ready_file=Path(model_ready_file) if model_ready_file else None,
        length_summary_file=Path(length_summary_file) if length_summary_file else None,
        token_summary_file=Path(token_summary_file) if token_summary_file else None,
        qc_summary_file=Path(qc_summary_file) if qc_summary_file else None,
        create_root_level_copies=create_root_level_copies,
        create_no_main_figure_readme=create_no_main_figure_readme,
        git_commit=git_commit,
    )
    return run_step4_redesign(cfg)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 4 PLOS supplementary figures S7/S8.")
    parser.add_argument("--project_root", default=str(PROJECT_ROOT_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--model_ready_file", default=None)
    parser.add_argument("--length_summary_file", default=None)
    parser.add_argument("--token_summary_file", default=None)
    parser.add_argument("--qc_summary_file", default=None)
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    parser.add_argument("--no_main_readme", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    args = parse_args()
    cfg = Config(
        project_root=Path(args.project_root),
        base_dir=Path(args.base_dir),
        output_root_name=args.output_root_name,
        model_ready_file=Path(args.model_ready_file) if args.model_ready_file else None,
        length_summary_file=Path(args.length_summary_file) if args.length_summary_file else None,
        token_summary_file=Path(args.token_summary_file) if args.token_summary_file else None,
        qc_summary_file=Path(args.qc_summary_file) if args.qc_summary_file else None,
        create_root_level_copies=not args.no_root_copies,
        create_no_main_figure_readme=not args.no_main_readme,
        git_commit=args.git_commit,
    )
    run_step4_redesign(cfg)

In [ ]:
from pathlib import Path

root = Path("/home/data3/Moe")

targets = [
    "step4_model_ready_sequences.csv",
    "step4_sequence_length_summary.csv",
    "step4_token_coverage_summary.csv",
    "step4_tokenization_qc.csv",
]

for target in targets:
    print("\nSearching for:", target)
    matches = list(root.rglob(target))
    if matches:
        for p in matches:
            print("FOUND:", p)
    else:
        print("NOT FOUND")

In [ ]:
results = main_notebook(
    project_root="/home/data3/Moe/nature_computational_peponco",
    base_dir="/home/data3/Moe/nature_computational_peponco/PLOS",
    output_root_name="plos_comp/step_04",

    model_ready_file="/home/data3/Moe/nature_computational_peponco/step4/tables/step4_model_ready_sequences.csv",
    length_summary_file="/home/data3/Moe/nature_computational_peponco/step4/tables/step4_sequence_length_summary.csv",
    token_summary_file="/home/data3/Moe/nature_computational_peponco/step4/tables/step4_token_coverage_summary.csv",
    qc_summary_file="/home/data3/Moe/nature_computational_peponco/step4/tables/step4_tokenization_qc.csv",
)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""

Jupyter-safe execution
----------------------
When this file is pasted or run inside Jupyter, the bottom command-line block
will not auto-run. First run the full file/cell to define the functions, then
run:

results = main_notebook()

The default input paths point to:
/home/data3/Moe/nature_computational_peponco/step4/tables/

OncoPep Step 4 — PLOS Computational Biology supplementary-figure redesign.

Step 4 = frozen preprocessing, tokenization, sequence-length, padding, and
amino-acid composition audit.

Outputs:
  Supplementary Figure S7 — frozen sequence preprocessing and tokenization audit
  Supplementary Figure S8 — amino-acid composition preservation across partitions

No main figure is generated in Step 4. Main Figure 2 is reserved for the
OncoPep architecture and conditional generation workflow.

v3 improvements:
  - Supplementary Figure S7 is redesigned as a clean three-panel figure.
  - The former S7D split-level preprocessing audit heatmap is removed from the
    figure and retained only in reports/source data.
  - S7B is changed from an all-zero bar plot to a compact audit-table/card panel.
  - S7A and S7C are retained with cleaner labels.
  - S8 minor label refinements are applied.
  - Metric-source/provenance, cap/filtering, unknown-token, and noncanonical
    residue audit tables remain exported for reviewer inspection.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd


PROJECT_ROOT_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_04"

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
AA_ORDER = tuple("ACDEFGHIKLMNPQRSTVWY")

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}
SPLIT_FACE = {"train": PLOS["blue"], "val": PLOS["mint"], "test": PLOS["coral"]}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}
HELDOUT_FACE = {"val": PLOS["mint"], "test": PLOS["coral"]}
HELDOUT_EDGE = {"val": "#6F9F7A", "test": "#A84F42"}


@dataclass
class Config:
    project_root: Path = PROJECT_ROOT_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT

    model_ready_file: Optional[Path] = None
    length_summary_file: Optional[Path] = None
    token_summary_file: Optional[Path] = None
    qc_summary_file: Optional[Path] = None

    split_order: Tuple[str, ...] = SPLIT_ORDER
    aa_order: Tuple[str, ...] = AA_ORDER

    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    create_no_main_figure_readme: bool = True
    git_commit: Optional[str] = None

    font_size: float = 8.3
    title_size: float = 9.2
    axis_label_size: float = 8.9
    tick_label_size: float = 7.8
    legend_size: float = 7.7
    annotation_size: float = 7.3
    panel_letter_size: float = 13.0

    @property
    def step4_tables_dir(self) -> Path:
        return self.project_root / "step4" / "tables"

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_figure_dir(self) -> Path:
        return self.output_root / "main_figure"


# ----------------------------- utilities -------------------------------------

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def ensure_dirs(cfg: Config) -> None:
    for d in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        d.mkdir(parents=True, exist_ok=True)
    if cfg.create_no_main_figure_readme:
        cfg.main_figure_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_json(data: Any, path: Path) -> None:
    def default(o: Any) -> Any:
        if isinstance(o, Path):
            return str(o)
        if isinstance(o, tuple):
            return list(o)
        if isinstance(o, (np.integer, np.floating, np.bool_)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(f"{type(o).__name__} is not JSON serializable")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def normalize_split(s: pd.Series) -> pd.Series:
    return (
        s.astype(str).str.strip().str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_col(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"Could not find one of {list(candidates)}. Available columns: {list(df.columns)}")
    return None


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    return None


def get_value(df: pd.DataFrame, split: str, col: str, default: float = np.nan) -> float:
    if "split" not in df.columns or col not in df.columns:
        return float(default)
    sub = df.loc[df["split"] == split, col]
    if sub.empty:
        return float(default)
    val = pd.to_numeric(sub, errors="coerce").dropna()
    return float(val.iloc[0]) if len(val) else float(default)


def validate_one_row_per_split(df: pd.DataFrame, table_name: str, cfg: Config, warnings: list[str]) -> None:
    """Warn if a split-summary table has duplicate rows for the same split."""
    if "split" not in df.columns:
        warnings.append(f"{table_name} has no split column; split-level validation could not be performed.")
        return
    counts = df["split"].value_counts(dropna=False)
    duplicates = counts[counts > 1]
    if not duplicates.empty:
        warnings.append(
            f"{table_name} contains duplicate split rows {duplicates.to_dict()}; "
            "first non-missing value will be used for plotted split-level metrics."
        )


def metric_value_and_source(
    df: pd.DataFrame,
    split: str,
    col: str,
    table_name: str,
    allow_missing: bool,
    warnings: list[str],
    default: float = np.nan,
) -> tuple[float, str]:
    """Return a split-level metric and a provenance/source flag.

    Missing truncation and unknown-token metrics should not silently become zero.
    For optional metrics, missing values are returned as NA with a source flag.
    """
    if "split" not in df.columns:
        msg = f"{table_name} has no split column; metric '{col}' unavailable for split '{split}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_split_column"
        raise ValueError(msg)

    if col not in df.columns:
        msg = f"{table_name} is missing required metric column '{col}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_column"
        raise ValueError(msg)

    sub = df.loc[df["split"] == split, col]
    if sub.empty:
        msg = f"{table_name} has no row for split '{split}' when reading '{col}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_split"
        raise ValueError(msg)

    vals = pd.to_numeric(sub, errors="coerce").dropna()
    if vals.empty:
        msg = f"{table_name} has only nonnumeric/NA values for '{col}' in split '{split}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_value"
        raise ValueError(msg)

    return float(vals.iloc[0]), f"measured:{table_name}.{col}"


def clamp_fraction(value: float, metric_name: str, warnings: list[str]) -> float:
    """Constrain a numeric metric to [0, 1] and warn if clipping was needed."""
    if not np.isfinite(value):
        return value
    clipped = min(max(float(value), 0.0), 1.0)
    if clipped != float(value):
        warnings.append(f"Metric '{metric_name}'={value} was outside [0,1] and was clipped to {clipped}.")
    return clipped


# ----------------------------- style -----------------------------------------

def set_style(cfg: Config) -> None:
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "legend.fontsize": cfg.legend_size,
        "axes.linewidth": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "text.color": PLOS["dark"],
        "axes.labelcolor": PLOS["dark"],
        "axes.edgecolor": PLOS["dark"],
        "xtick.color": PLOS["dark"],
        "ytick.color": PLOS["dark"],
    })


def beautify(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.45)
    ax.grid(False, axis="x")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.75)
        ax.spines[side].set_color(PLOS["dark"])


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.08) -> None:
    ax.text(dx, dy, letter.upper(), transform=ax.transAxes, fontsize=cfg.panel_letter_size,
            fontweight="bold", va="top", ha="left", color=PLOS["dark"])


def save_fig(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {"png": base.with_suffix(".png"), "pdf": base.with_suffix(".pdf"), "tiff": base.with_suffix(".tiff")}
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


# ----------------------------- loading ---------------------------------------

def candidates(cfg: Config, name: str, explicit: Optional[Path]) -> list[Path]:
    out = []
    if explicit:
        out.append(Path(explicit))
    out.extend([
        cfg.step4_tables_dir / name,
        cfg.project_root / "redesign" / "step4" / "tables" / name,
        Path("/home/data3/Moe/nature_computational_peponco/step4/tables") / name,
        Path("/home/data3/Moe/nature_computational_peponco/redesign/step4/tables") / name,
        Path("/home/data3/Moe/nature_computational_peponco/step4/tables") / name,
    ])
    return out


def load_csv(cfg: Config, name: str, explicit: Optional[Path]) -> Tuple[pd.DataFrame, Path]:
    path = first_existing(candidates(cfg, name, explicit))
    if path is None:
        raise FileNotFoundError("Missing input table " + name + ". Checked:\n- " + "\n- ".join(map(str, candidates(cfg, name, explicit))))
    return pd.read_csv(path, low_memory=False), path


def load_inputs(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Path]]:
    model_ready, p1 = load_csv(cfg, "step4_model_ready_sequences.csv", cfg.model_ready_file)
    length_summary, p2 = load_csv(cfg, "step4_sequence_length_summary.csv", cfg.length_summary_file)
    token_summary, p3 = load_csv(cfg, "step4_token_coverage_summary.csv", cfg.token_summary_file)
    qc_summary, p4 = load_csv(cfg, "step4_tokenization_qc.csv", cfg.qc_summary_file)
    return model_ready, length_summary, token_summary, qc_summary, {
        "model_ready_sequences": p1,
        "sequence_length_summary": p2,
        "token_coverage_summary": p3,
        "tokenization_qc": p4,
    }


def infer_chosen_cap(length_summary: pd.DataFrame) -> int:
    for c in ["chosen_max_sequence_length", "max_sequence_length", "sequence_length_cap", "chosen_cap"]:
        if c in length_summary.columns:
            vals = pd.to_numeric(length_summary[c], errors="coerce").dropna()
            if len(vals):
                return int(vals.iloc[0])
    return 60


def infer_model_len(length_summary: pd.DataFrame, fallback: Optional[int] = None) -> int:
    for c in ["max_model_length_including_special_tokens", "max_model_length", "model_length"]:
        if c in length_summary.columns:
            vals = pd.to_numeric(length_summary[c], errors="coerce").dropna()
            if len(vals):
                return int(vals.iloc[0])
    return int(fallback if fallback is not None else infer_chosen_cap(length_summary))


def harmonize_inputs(model_ready: pd.DataFrame, length_summary: pd.DataFrame,
                     token_summary: pd.DataFrame, qc_summary: pd.DataFrame,
                     cfg: Config, warnings: list[str]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str, str, str]:
    seq_col = resolve_col(model_ready, ["sequence", "seq", "peptide_sequence"])
    split_col = resolve_col(model_ready, ["assigned_split", "split", "data_split"])
    if split_col != "assigned_split":
        model_ready = model_ready.rename(columns={split_col: "assigned_split"})
        split_col = "assigned_split"
    model_ready[split_col] = normalize_split(model_ready[split_col])

    if "sequence_length" not in model_ready.columns:
        model_ready["sequence_length"] = model_ready[seq_col].astype(str).map(len)

    cap = infer_chosen_cap(length_summary)
    model_len = infer_model_len(length_summary, fallback=cap)

    if "padding_fraction" in model_ready.columns:
        model_ready["padding_fraction"] = pd.to_numeric(model_ready["padding_fraction"], errors="coerce")
        padding_fraction_source = "measured:model_ready_sequences.padding_fraction"
    else:
        model_ready["padding_fraction"] = (1.0 - pd.to_numeric(model_ready["sequence_length"], errors="coerce") / max(model_len, 1)).clip(0, 1)
        padding_fraction_source = "inferred:1-sequence_length/max_model_length"
        warnings.append(
            "Sequence-level padding_fraction was absent and was inferred from sequence_length and max_model_length. "
            "If special tokens are used, confirm this approximation against the tokenizer output."
        )

    for df in [length_summary, token_summary, qc_summary]:
        c = resolve_col(df, ["split", "assigned_split", "data_split"], required=False)
        if c is not None:
            df.rename(columns={c: "split"}, inplace=True)
            df["split"] = normalize_split(df["split"])

    missing = [s for s in cfg.split_order if s not in set(model_ready[split_col].dropna())]
    if missing:
        raise ValueError(f"Missing expected splits in model-ready table: {missing}")
    if "split" not in token_summary.columns or "split" not in qc_summary.columns:
        raise ValueError("token_summary and qc_summary must include a split column.")

    validate_one_row_per_split(token_summary, "token_summary", cfg, warnings)
    validate_one_row_per_split(qc_summary, "qc_summary", cfg, warnings)
    return model_ready, length_summary, token_summary, qc_summary, seq_col, split_col, padding_fraction_source


# ----------------------------- analyses --------------------------------------

def build_length_audit(model_ready: pd.DataFrame, length_summary: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    cap = infer_chosen_cap(length_summary)
    model_len = infer_model_len(length_summary, fallback=cap)
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "sequence_length"], errors="coerce").dropna().to_numpy()
        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(vals)),
            "min_length": float(np.min(vals)) if len(vals) else np.nan,
            "q25_length": float(np.percentile(vals, 25)) if len(vals) else np.nan,
            "median_length": float(np.median(vals)) if len(vals) else np.nan,
            "mean_length": float(np.mean(vals)) if len(vals) else np.nan,
            "q75_length": float(np.percentile(vals, 75)) if len(vals) else np.nan,
            "max_length": float(np.max(vals)) if len(vals) else np.nan,
            "chosen_max_sequence_length": int(cap),
            "max_model_length_including_special_tokens": int(model_len),
            "fraction_exceeding_cap": float(np.mean(vals > cap)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def build_token_audit(model_ready: pd.DataFrame, token_summary: pd.DataFrame, qc_summary: pd.DataFrame,
                      split_col: str, cfg: Config, warnings: list[str], padding_fraction_source: str) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        sub = model_ready.loc[model_ready[split_col] == split]
        mean_pad_default = float(pd.to_numeric(sub["padding_fraction"], errors="coerce").mean())

        frac_trunc, frac_trunc_source = metric_value_and_source(
            qc_summary, split, "fraction_truncated", "qc_summary",
            allow_missing=False, warnings=warnings
        )
        unk, unk_source = metric_value_and_source(
            token_summary, split, "unknown_token_fraction_nonpad", "token_summary",
            allow_missing=False, warnings=warnings
        )

        # mean padding can be measured in qc_summary or derived from sequence-level padding_fraction.
        if "mean_padding_fraction" in qc_summary.columns:
            mean_pad, mean_pad_source = metric_value_and_source(
                qc_summary, split, "mean_padding_fraction", "qc_summary",
                allow_missing=True, warnings=warnings, default=mean_pad_default
            )
            if not np.isfinite(mean_pad):
                mean_pad = mean_pad_default
                mean_pad_source = f"derived_from_sequence_level:{padding_fraction_source}"
        else:
            mean_pad = mean_pad_default
            mean_pad_source = f"derived_from_sequence_level:{padding_fraction_source}"
            warnings.append("qc_summary is missing 'mean_padding_fraction'; split mean padding was derived from sequence-level padding_fraction.")

        cond_missing, cond_source = metric_value_and_source(
            qc_summary, split, "fraction_primary_condition_id_missing", "qc_summary",
            allow_missing=True, warnings=warnings, default=np.nan
        )

        frac_trunc = clamp_fraction(frac_trunc, "fraction_truncated", warnings)
        unk = clamp_fraction(unk, "unknown_token_fraction_nonpad", warnings)
        mean_pad = clamp_fraction(mean_pad, "mean_padding_fraction", warnings)
        cond_missing = clamp_fraction(cond_missing, "fraction_primary_condition_id_missing", warnings)

        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(sub)),
            "fraction_truncated": frac_trunc,
            "fraction_truncated_source": frac_trunc_source,
            "non_truncated_fraction": 1.0 - frac_trunc if np.isfinite(frac_trunc) else np.nan,
            "unknown_token_fraction_nonpad": unk,
            "unknown_token_fraction_nonpad_source": unk_source,
            "known_token_fraction_nonpad": 1.0 - unk if np.isfinite(unk) else np.nan,
            "mean_padding_fraction": mean_pad,
            "mean_padding_fraction_source": mean_pad_source,
            "effective_token_fraction": 1.0 - mean_pad if np.isfinite(mean_pad) else np.nan,
            "fraction_primary_condition_id_missing": cond_missing,
            "fraction_primary_condition_id_missing_source": cond_source,
            "valid_condition_fraction": 1.0 - cond_missing if np.isfinite(cond_missing) else np.nan,
        })
    return pd.DataFrame(rows)


def build_padding_summary(model_ready: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "padding_fraction"], errors="coerce").dropna().to_numpy()
        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(vals)),
            "mean_padding_fraction": float(np.mean(vals)) if len(vals) else np.nan,
            "sd_padding_fraction": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "median_padding_fraction": float(np.median(vals)) if len(vals) else np.nan,
            "q25_padding_fraction": float(np.percentile(vals, 25)) if len(vals) else np.nan,
            "q75_padding_fraction": float(np.percentile(vals, 75)) if len(vals) else np.nan,
            "min_padding_fraction": float(np.min(vals)) if len(vals) else np.nan,
            "max_padding_fraction": float(np.max(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def build_cap_filtering_audit(model_ready: pd.DataFrame, length_audit: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    """Audit model-ready length coverage relative to the selected cap.

    This does not infer pre-filtering losses unless raw pre-filter counts are
    available; it documents model-ready sequences that exceed the selected cap.
    """
    cap = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "sequence_length"], errors="coerce").dropna()
        n = int(len(vals))
        n_exceed = int((vals > cap).sum())
        rows.append({
            "split": split,
            "split_display": SPLIT_DISPLAY[split],
            "n_model_ready_sequences": n,
            "chosen_max_sequence_length": cap,
            "n_model_ready_sequences_exceeding_cap": n_exceed,
            "fraction_model_ready_sequences_exceeding_cap": float(n_exceed / n) if n else np.nan,
            "filtering_loss_available": False,
            "n_prefilter_sequences": np.nan,
            "n_filtered_by_length": np.nan,
            "filtering_fraction": np.nan,
            "interpretation_note": "Audit is based on model-ready sequences; pre-filter loss requires raw pre-filter counts.",
        })
    return pd.DataFrame(rows)


def build_unknown_token_audit(token_audit: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "split", "split_display", "n_sequences",
        "unknown_token_fraction_nonpad", "unknown_token_fraction_nonpad_source",
        "known_token_fraction_nonpad",
    ]
    return token_audit[[c for c in cols if c in token_audit.columns]].copy()


def build_noncanonical_residue_audit(model_ready: pd.DataFrame, seq_col: str, split_col: str, cfg: Config) -> pd.DataFrame:
    canonical = set(cfg.aa_order)
    rows = []
    for split in cfg.split_order:
        seqs = model_ready.loc[model_ready[split_col] == split, seq_col].astype(str).tolist()
        concat = "".join(seqs)
        total_chars = len(concat)
        canonical_count = sum(1 for ch in concat if ch in canonical)
        noncanonical_count = total_chars - canonical_count
        noncanonical_symbols = sorted(set(ch for ch in concat if ch not in canonical))
        rows.append({
            "split": split,
            "split_display": SPLIT_DISPLAY[split],
            "n_sequences": int(len(seqs)),
            "total_residue_characters": int(total_chars),
            "canonical_residue_characters": int(canonical_count),
            "noncanonical_residue_characters": int(noncanonical_count),
            "noncanonical_residue_fraction": float(noncanonical_count / total_chars) if total_chars else np.nan,
            "noncanonical_symbols": "".join(noncanonical_symbols) if noncanonical_symbols else "",
            "canonical_vocabulary": "".join(cfg.aa_order),
        })
    return pd.DataFrame(rows)


def build_aa_comp(model_ready: pd.DataFrame, seq_col: str, split_col: str, cfg: Config) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        concat = "".join(model_ready.loc[model_ready[split_col] == split, seq_col].astype(str).tolist())
        total = sum(concat.count(aa) for aa in cfg.aa_order)
        denom = max(total, 1)
        for aa in cfg.aa_order:
            rows.append({"split": split, "split_display": SPLIT_DISPLAY[split], "amino_acid": aa,
                         "count": int(concat.count(aa)), "canonical_total": int(total),
                         "fraction": float(concat.count(aa) / denom)})
    return pd.DataFrame(rows)


def build_comp_dev(aa_comp: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    piv = aa_comp.pivot_table(index="amino_acid", columns="split", values="fraction", aggfunc="first").reindex(cfg.aa_order)
    rows = []
    for aa in cfg.aa_order:
        tr, va, te = float(piv.loc[aa, "train"]), float(piv.loc[aa, "val"]), float(piv.loc[aa, "test"])
        rows.append({"amino_acid": aa, "train_fraction": tr, "val_fraction": va, "test_fraction": te,
                     "val_minus_train": va - tr, "test_minus_train": te - tr,
                     "abs_val_minus_train": abs(va - tr), "abs_test_minus_train": abs(te - tr)})
    dev = pd.DataFrame(rows)
    summary_rows = []
    for split in ["val", "test"]:
        tr = dev["train_fraction"].to_numpy(float)
        held = dev[f"{split}_fraction"].to_numpy(float)
        delta = held - tr
        r = float(np.corrcoef(tr, held)[0, 1]) if np.std(tr) > 0 and np.std(held) > 0 else np.nan
        summary_rows.append({"comparison": f"{SPLIT_DISPLAY[split]} vs Train", "heldout_split": split,
                             "pearson_r": r, "mean_absolute_deviation": float(np.mean(np.abs(delta))),
                             "max_absolute_deviation": float(np.max(np.abs(delta))),
                             "signed_deviation_mean": float(np.mean(delta)), "n_amino_acids": int(len(delta))})
    return dev, pd.DataFrame(summary_rows)


# ----------------------------- source data -----------------------------------

def build_s7_source(model_ready: pd.DataFrame, length_audit: pd.DataFrame, token_audit: pd.DataFrame, split_col: str, cfg: Config):
    """Build source data for the three-panel Supplementary Figure S7.

    The full split-level audit previously shown as S7D is retained in
    reports/tokenization_audit.csv and reports/padding_burden_summary.csv,
    but is no longer plotted to keep the figure clean.
    """
    cap = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
    model_len = int(length_audit["max_model_length_including_special_tokens"].dropna().iloc[0])

    a = model_ready[[split_col, "sequence_length"]].copy().rename(columns={split_col: "split", "sequence_length": "value"})
    a["split"] = normalize_split(a["split"]); a["split_display"] = a["split"].map(SPLIT_DISPLAY)
    a["chosen_max_sequence_length"] = cap; a["max_model_length_including_special_tokens"] = model_len
    a["panel"] = "A"; a["data_type"] = "sequence_length_distribution"; a["metric_name"] = "sequence_length"
    a["unit"] = "amino_acid_residues"; a["metric_definition"] = "Number of residues in the standardized model-ready peptide sequence."

    rows = []
    for _, r in token_audit.iterrows():
        for metric, label, definition in [
            ("fraction_truncated", "Truncation fraction", "Fraction of sequences truncated during preprocessing."),
            ("unknown_token_fraction_nonpad", "Unknown-token fraction", "Fraction of unknown tokens among non-padding tokens."),
        ]:
            rows.append({
                "split": r["split"], "split_display": r["split_display"], "metric": metric,
                "metric_display": label, "value": r[metric],
                "metric_source": r.get(f"{metric}_source", "not_recorded"),
                "unit": "fraction", "panel": "B",
                "data_type": "truncation_unknown_token_audit",
                "metric_definition": definition,
            })
    b = pd.DataFrame(rows)

    c = model_ready[[split_col, "padding_fraction"]].copy().rename(columns={split_col: "split", "padding_fraction": "value"})
    c["split"] = normalize_split(c["split"]); c["split_display"] = c["split"].map(SPLIT_DISPLAY)
    c["panel"] = "C"; c["data_type"] = "padding_fraction_distribution"; c["metric_name"] = "padding_fraction"
    c["unit"] = "fraction"; c["metric_definition"] = "Fraction of final model input positions occupied by padding."

    panels = {"A": a, "B": b, "C": c}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def build_s8_source(aa_comp: pd.DataFrame, dev: pd.DataFrame, summary: pd.DataFrame, cfg: Config):
    a = aa_comp.copy()
    a["panel"] = "A"; a["data_type"] = "amino_acid_frequency_profile"; a["metric_name"] = "amino_acid_fraction"
    a["unit"] = "fraction"; a["metric_definition"] = "Count of a canonical amino acid divided by total canonical amino-acid count in the split."

    rows = []
    for _, r in dev.iterrows():
        for split in ["val", "test"]:
            rows.append({"amino_acid": r["amino_acid"], "heldout_split": split,
                         "heldout_split_display": SPLIT_DISPLAY[split],
                         "value": r[f"{split}_minus_train"],
                         "absolute_value": r[f"abs_{split}_minus_train"],
                         "panel": "B", "data_type": "heldout_minus_train_composition_deviation",
                         "unit": "fraction_difference",
                         "metric_definition": "Held-out amino-acid fraction minus training amino-acid fraction."})
    b = pd.DataFrame(rows)

    rows = []
    for _, r in dev.iterrows():
        for split in ["val", "test"]:
            rows.append({"amino_acid": r["amino_acid"], "heldout_split": split,
                         "heldout_split_display": SPLIT_DISPLAY[split],
                         "train_fraction": r["train_fraction"],
                         "heldout_fraction": r[f"{split}_fraction"],
                         "panel": "C", "data_type": "heldout_vs_train_composition_agreement",
                         "unit": "fraction",
                         "metric_definition": "Training and held-out amino-acid fractions for composition agreement plotting."})
    c = pd.DataFrame(rows)

    d = summary.copy()
    d["panel"] = "D"; d["data_type"] = "composition_deviation_summary"
    d["unit"] = "fraction"; d["metric_definition"] = "Summary of held-out amino-acid composition deviation relative to training composition."
    panels = {"A": a, "B": b, "C": c, "D": d}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def save_sources(cfg: Config, s7_panels, s7_all, s8_panels, s8_all) -> None:
    save_csv(s7_all, cfg.source_data_dir / "Supplementary_Figure_S7_source_data_all_panels.csv")
    save_csv(s8_all, cfg.source_data_dir / "Supplementary_Figure_S8_source_data_all_panels.csv")
    for p, df in s7_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S7_panel_{p.lower()}_source_data.csv")
    for p, df in s8_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S8_panel_{p.lower()}_source_data.csv")


# ----------------------------- plotting --------------------------------------

def plot_s7a(ax, df, cfg):
    for split in cfg.split_order:
        vals = pd.to_numeric(df.loc[df["split"] == split, "value"], errors="coerce").dropna().sort_values().to_numpy()
        if len(vals):
            y = np.arange(1, len(vals) + 1) / len(vals)
            ax.step(vals, y, where="post", color=SPLIT_EDGE[split], lw=1.8, label=SPLIT_DISPLAY[split], zorder=3)
    cap_vals = df["chosen_max_sequence_length"].dropna().unique()
    if len(cap_vals):
        cap = float(cap_vals[0])
        ax.axvline(cap, color=PLOS["charcoal"], linestyle=(0, (3, 2.5)), lw=1.15, zorder=2)
        ax.text(cap - 0.7, 0.08, f"Length cap = {int(cap)} aa", fontsize=cfg.annotation_size, ha="right", va="bottom", color=PLOS["dark"])
    ax.set_xlabel("Sequence length (aa)")
    ax.set_ylabel("Cumulative fraction")
    ax.set_ylim(0, 1.02)
    ax.set_title("Model-ready sequence-length coverage", pad=8)
    beautify(ax)
    ax.legend(loc="lower right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s7b(ax, df, cfg):
    """Compact table/card panel for truncation and unknown-token audit.

    This is clearer than an empty all-zero bar chart when the audit values are zero.
    """
    metrics = [
        ("fraction_truncated", "Truncation\nfraction"),
        ("unknown_token_fraction_nonpad", "Unknown-token\nfraction"),
    ]

    ax.set_xlim(0, len(metrics))
    ax.set_ylim(0, len(cfg.split_order))
    ax.invert_yaxis()

    for i, split in enumerate(cfg.split_order):
        for j, (metric, label) in enumerate(metrics):
            sub = df.loc[(df["split"] == split) & (df["metric"] == metric), "value"]
            val = float(sub.iloc[0]) if len(sub) and pd.notna(sub.iloc[0]) else np.nan
            if np.isfinite(val):
                intensity = 0.06 + 0.48 * min(max(val, 0), 1)
                face = mpl.colors.to_rgba(PLOS["coral"] if val > 0 else PLOS["light"], alpha=intensity if val > 0 else 0.35)
                txt = "0" if abs(val) < 1e-12 else f"{val:.3f}"
            else:
                face = mpl.colors.to_rgba(PLOS["light"], alpha=0.25)
                txt = "NA"

            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=face, edgecolor=PLOS["light"], lw=0.75))
            ax.text(j + 0.5, i + 0.52, txt, ha="center", va="center",
                    fontsize=cfg.annotation_size + 1.6, color=PLOS["dark"], fontweight="bold")

    for j, (_, label) in enumerate(metrics):
        ax.text(j + 0.5, -0.13, label, ha="center", va="bottom",
                fontsize=cfg.tick_label_size + 0.3, color=PLOS["dark"])

    for i, split in enumerate(cfg.split_order):
        ax.text(-0.08, i + 0.5, SPLIT_DISPLAY[split], ha="right", va="center",
                fontsize=cfg.tick_label_size + 0.4, color=PLOS["dark"])

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title("Truncation / unknown-token audit", pad=22)


def plot_s7c(ax, df, cfg):
    positions = np.arange(1, 4)
    data = [pd.to_numeric(df.loc[df["split"] == s, "value"], errors="coerce").dropna().to_numpy() for s in cfg.split_order]
    bp = ax.boxplot(data, positions=positions, widths=0.46, patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops=dict(color=PLOS["dark"], lw=1.35),
                    whiskerprops=dict(color=PLOS["charcoal"], lw=0.85),
                    capprops=dict(color=PLOS["charcoal"], lw=0.85),
                    boxprops=dict(lw=1.05))
    for patch, split in zip(bp["boxes"], cfg.split_order):
        patch.set_facecolor(SPLIT_FACE[split]); patch.set_alpha(0.34); patch.set_edgecolor(SPLIT_EDGE[split])
    for pos, arr, split in zip(positions, data, cfg.split_order):
        if len(arr):
            ax.scatter(pos, np.median(arr), s=16, color=SPLIT_EDGE[split], edgecolor="white", lw=0.35, zorder=5)
    ax.set_xticks(positions); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel("Padding fraction")
    ax.set_ylim(0, 1.0)
    ax.set_title("Per-sequence padding burden", pad=8)
    beautify(ax)


def plot_s7d(ax, df, cfg):
    metrics = [("non_truncated_fraction", "Non-\ntruncated"), ("known_token_fraction_nonpad", "Known-\ntoken"),
               ("effective_token_fraction", "Effective-\ntoken"), ("mean_padding_fraction", "Mean\npadding")]
    ax.set_xlim(0, len(metrics)); ax.set_ylim(0, len(cfg.split_order)); ax.invert_yaxis()
    for i, split in enumerate(cfg.split_order):
        for j, (metric, label) in enumerate(metrics):
            sub = df.loc[(df["split"] == split) & (df["metric"] == metric), "value"]
            val = float(sub.iloc[0]) if len(sub) and pd.notna(sub.iloc[0]) else np.nan
            base = PLOS["light"] if metric == "mean_padding_fraction" else PLOS["blue"]
            alpha = 0.10 + 0.48 * min(max(val, 0), 1) if np.isfinite(val) else 0.08
            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=mpl.colors.to_rgba(base, alpha=alpha),
                                   edgecolor=PLOS["light"], lw=0.7))
            ax.text(j + 0.5, i + 0.53, f"{val:.2f}" if np.isfinite(val) else "NA",
                    ha="center", va="center", fontsize=cfg.annotation_size + 0.8)
    for j, (_, label) in enumerate(metrics):
        ax.text(j + 0.5, -0.15, label, ha="center", va="bottom", fontsize=cfg.tick_label_size)
    for i, split in enumerate(cfg.split_order):
        ax.text(-0.10, i + 0.5, SPLIT_DISPLAY[split], ha="right", va="center", fontsize=cfg.tick_label_size)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_title("Split-level preprocessing audit", pad=20)


def plot_s7(panels, cfg):
    set_style(cfg)
    fig = plt.figure(figsize=(12.2, 4.25))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 0.88, 1.0], wspace=0.40)
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    fig.subplots_adjust(left=0.075, right=0.985, top=0.86, bottom=0.18)

    plot_s7a(axes[0], panels["A"], cfg); panel_letter(axes[0], "A", cfg)
    plot_s7b(axes[1], panels["B"], cfg); panel_letter(axes[1], "B", cfg)
    plot_s7c(axes[2], panels["C"], cfg); panel_letter(axes[2], "C", cfg)

    return save_fig(fig, cfg.supp_dir / "Supplementary_Figure_S7_redesigned", cfg)


def plot_s8a(ax, df, cfg):
    x = np.arange(len(cfg.aa_order)); width = 0.23
    offsets = {"train": -width, "val": 0.0, "test": width}
    for split in cfg.split_order:
        sub = df.loc[df["split"] == split].set_index("amino_acid").reindex(cfg.aa_order)
        vals = pd.to_numeric(sub["fraction"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offsets[split], vals, width=width, color=SPLIT_FACE[split],
               edgecolor=SPLIT_EDGE[split], lw=0.45, alpha=0.78, label=SPLIT_DISPLAY[split], zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(cfg.aa_order)
    ax.set_xlabel("Amino acid"); ax.set_ylabel("Fraction")
    ax.set_title("Amino-acid frequency profile", pad=8)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8b(ax, df, cfg):
    x = np.arange(len(cfg.aa_order)); width = 0.23
    for split, offset in [("val", -width/2), ("test", width/2)]:
        sub = df.loc[df["heldout_split"] == split].set_index("amino_acid").reindex(cfg.aa_order)
        vals = pd.to_numeric(sub["value"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offset, vals, width=width, color=HELDOUT_FACE[split], edgecolor=HELDOUT_EDGE[split],
               lw=0.45, alpha=0.82, label=f"{SPLIT_DISPLAY[split]} − Train", zorder=3)
    ax.axhline(0, color=PLOS["charcoal"], lw=0.9, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels(cfg.aa_order)
    ax.set_xlabel("Amino acid"); ax.set_ylabel("Δ fraction relative to train")
    ax.set_title("Held-out deviation from train", pad=8)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8c(ax, df, summary, cfg):
    maxv = max(float(pd.to_numeric(df["train_fraction"], errors="coerce").max()),
               float(pd.to_numeric(df["heldout_fraction"], errors="coerce").max()), 0.01)
    lim = maxv * 1.12
    ax.plot([0, lim], [0, lim], color=PLOS["charcoal"], linestyle=(0, (3, 2.5)), lw=0.9, zorder=1)
    for split in ["val", "test"]:
        sub = df.loc[df["heldout_split"] == split]
        ax.scatter(sub["train_fraction"], sub["heldout_fraction"], s=32, color=HELDOUT_FACE[split],
                   edgecolor=HELDOUT_EDGE[split], lw=0.6, alpha=0.88, label=SPLIT_DISPLAY[split], zorder=3)
    ann = []
    for _, r in summary.iterrows():
        ann.append(f"{SPLIT_DISPLAY.get(r['heldout_split'], r['heldout_split'])}: r={r['pearson_r']:.3f}, MAE={r['mean_absolute_deviation']:.3f}")
    if ann:
        ax.text(0.04, 0.96, "\n".join(ann), transform=ax.transAxes, ha="left", va="top",
                fontsize=cfg.annotation_size, bbox=dict(facecolor="white", edgecolor=PLOS["light"], lw=0.6, pad=2.5))
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Train fraction"); ax.set_ylabel("Held-out fraction")
    ax.set_title("Held-out composition vs train", pad=8)
    beautify(ax)
    ax.legend(loc="lower right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8d(ax, df, cfg):
    metrics = [("mean_absolute_deviation", "Mean absolute\nΔ fraction"), ("max_absolute_deviation", "Maximum absolute\nΔ fraction")]
    x = np.arange(len(metrics)); width = 0.32; ymax = 0.001
    for split, offset in [("val", -width/2), ("test", width/2)]:
        sub = df.loc[df["heldout_split"] == split]
        vals = [float(pd.to_numeric(sub[m], errors="coerce").iloc[0]) if len(sub) else 0.0 for m, _ in metrics]
        ymax = max(ymax, max(vals))
        bars = ax.bar(x + offset, vals, width=width, color=HELDOUT_FACE[split], edgecolor=HELDOUT_EDGE[split],
                      lw=0.65, alpha=0.82, label=SPLIT_DISPLAY[split], zorder=3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + ymax * 0.035, f"{val:.3f}",
                    ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels([m[1] for m in metrics])
    ax.set_ylabel("Absolute Δ fraction")
    ax.set_title("Composition-deviation summary", pad=8)
    ax.set_ylim(0, ymax * 1.35 if ymax > 0 else 0.01)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8(panels, summary, cfg):
    set_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(11.1, 6.5))
    fig.subplots_adjust(left=0.082, right=0.985, top=0.915, bottom=0.105, wspace=0.31, hspace=0.45)
    plot_s8a(axes[0,0], panels["A"], cfg); panel_letter(axes[0,0], "A", cfg)
    plot_s8b(axes[0,1], panels["B"], cfg); panel_letter(axes[0,1], "B", cfg)
    plot_s8c(axes[1,0], panels["C"], summary, cfg); panel_letter(axes[1,0], "C", cfg)
    plot_s8d(axes[1,1], panels["D"], cfg); panel_letter(axes[1,1], "D", cfg)
    return save_fig(fig, cfg.supp_dir / "Supplementary_Figure_S8_redesigned", cfg)


# ------------------------- reproducibility outputs ---------------------------

def save_reports(cfg, length_audit, token_audit, pad_summary, aa_comp, dev, summary,
                 cap_filtering_audit=None, unknown_token_audit=None, noncanonical_residue_audit=None):
    reports = {
        "sequence_length_audit": length_audit,
        "tokenization_audit": token_audit,
        "padding_burden_summary": pad_summary,
        "amino_acid_composition_summary": aa_comp,
        "composition_deviation_audit": dev,
        "composition_similarity_summary": summary,
    }
    if cap_filtering_audit is not None:
        reports["cap_filtering_audit"] = cap_filtering_audit
    if unknown_token_audit is not None:
        reports["unknown_token_audit"] = unknown_token_audit
    if noncanonical_residue_audit is not None:
        reports["noncanonical_residue_audit"] = noncanonical_residue_audit
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
    return reports


def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env:
        return env
    try:
        res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(cfg.project_root),
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
        return res.stdout.strip() if res.returncode == 0 else None
    except Exception:
        return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    fname = globals().get("__file__", None)
    if fname and Path(fname).exists():
        script_path = str(Path(fname).resolve())
        text = Path(fname).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_04_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_04_PLOS_redesign_code.py")
    return script_path, script_hash


def write_no_main_readme(cfg: Config) -> Optional[Path]:
    if not cfg.create_no_main_figure_readme:
        return None
    path = cfg.main_figure_dir / "README_no_main_figure_for_step_04.txt"
    save_text(f"""# No main figure generated for OncoPep Step 4

Generated: {now_iso()}

Step 4 is a supplementary-only frozen preprocessing and tokenization audit.

Generated figures:
- Supplementary Figure S7
- Supplementary Figure S8

No Main Figure 2 is generated here. Main Figure 2 is reserved for the OncoPep
architecture and conditional generation workflow.
""", path)
    return path


def write_readme(cfg: Config, input_paths: Dict[str, Path], warnings: Sequence[str]) -> Path:
    path = cfg.reports_dir / "README_step_04_outputs.txt"
    save_text(f"""# OncoPep Step 4 PLOS supplementary-figure package

Generated: {now_iso()}

Scope:
Frozen sequence preprocessing, tokenization, sequence-length, padding, and
amino-acid composition audit.

No main figure is generated in Step 4. Main Figure 2 is reserved for the
OncoPep architecture/model workflow.

Input files:
{chr(10).join(f"- {k}: {v}" for k, v in input_paths.items())}

Generated figures:
- supplementary_figures/Supplementary_Figure_S7_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S8_redesigned.png/.pdf/.tiff

Scientific notes:
- Step 4 does not repeat Step 2 leakage audits or Step 3 conditioning-schema audits.
- Preprocessing QC supports model-input transparency, not biological validation.
- Unknown-token fraction is calculated among non-padding tokens and is shown in S7B.
- Padding fraction is the fraction of final model input positions occupied by padding; if inferred rather than read directly, this is recorded in source data and the manifest.
- Amino-acid composition deviation is computed relative to the training partition.
- Cap/filtering, split-level preprocessing, and noncanonical-residue audit tables are exported for reviewer inspection.
- The split-level preprocessing audit is retained in CSV reports rather than shown as an additional S7 panel.

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
""", path)
    return path


def write_requirements(cfg: Config) -> Path:
    path = cfg.reports_dir / "requirements_step_04_minimal.txt"
    save_text("\n".join([f"python=={platform.python_version()}",
                         f"numpy=={np.__version__}",
                         f"pandas=={pd.__version__}",
                         f"matplotlib=={mpl.__version__}", ""]), path)
    return path


def root_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs = []
    for fig in ["S7", "S8"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv",
                      cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))
    for name in ["sequence_length_audit.csv", "tokenization_audit.csv", "unknown_token_audit.csv",
                 "padding_burden_summary.csv", "cap_filtering_audit.csv",
                 "amino_acid_composition_summary.csv", "composition_deviation_audit.csv",
                 "composition_similarity_summary.csv", "noncanonical_residue_audit.csv",
                 "README_step_04_outputs.txt"]:
        pairs.append((cfg.reports_dir / name, cfg.output_root / name))
    pairs.append((cfg.code_dir / "OncoPep_step_04_PLOS_redesign_code.py", cfg.output_root / "OncoPep_step_04_PLOS_redesign_code.py"))
    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst); copied.append(str(dst))
    return copied


def build_manifest(cfg, input_paths, s7_paths, s8_paths, readme_path, req_path, no_main_path, script_path, script_hash, copies, warnings,
                   length_audit=None, token_audit=None, noncanonical_residue_audit=None, padding_fraction_source=None):
    preprocessing_parameters = {}
    if length_audit is not None and len(length_audit):
        preprocessing_parameters["chosen_max_sequence_length"] = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
        preprocessing_parameters["max_model_length_including_special_tokens"] = int(length_audit["max_model_length_including_special_tokens"].dropna().iloc[0])
    preprocessing_parameters["canonical_amino_acid_vocabulary"] = "".join(cfg.aa_order)
    preprocessing_parameters["padding_fraction_source"] = padding_fraction_source
    if token_audit is not None and len(token_audit):
        preprocessing_parameters["fraction_truncated_sources"] = token_audit.set_index("split")["fraction_truncated_source"].to_dict() if "fraction_truncated_source" in token_audit.columns else {}
        preprocessing_parameters["unknown_token_fraction_sources"] = token_audit.set_index("split")["unknown_token_fraction_nonpad_source"].to_dict() if "unknown_token_fraction_nonpad_source" in token_audit.columns else {}
    if noncanonical_residue_audit is not None and len(noncanonical_residue_audit):
        preprocessing_parameters["noncanonical_residue_fraction_by_split"] = noncanonical_residue_audit.set_index("split")["noncanonical_residue_fraction"].to_dict()
        preprocessing_parameters["noncanonical_symbols_by_split"] = noncanonical_residue_audit.set_index("split")["noncanonical_symbols"].to_dict()

    return {
        "step": "step_04",
        "scope": "supplementary_only",
        "timestamp": now_iso(),
        "main_figure_generated": False,
        "main_figure_rationale": "Step 4 audits frozen preprocessing/tokenization; Main Figure 2 is reserved for OncoPep architecture/model workflow.",
        "project_root": str(cfg.project_root),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "preprocessing_parameters": preprocessing_parameters,
        "input_files": {k: str(v) for k, v in input_paths.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_paths.items()},
        "figures": {"S7": s7_paths, "S8": s8_paths},
        "source_data": {
            "S7_all": str(cfg.source_data_dir / "Supplementary_Figure_S7_source_data_all_panels.csv"),
            "S8_all": str(cfg.source_data_dir / "Supplementary_Figure_S8_source_data_all_panels.csv"),
            **{f"S7_panel_{p.lower()}": str(cfg.source_data_dir / f"Supplementary_Figure_S7_panel_{p.lower()}_source_data.csv") for p in "ABC"},
            **{f"S8_panel_{p.lower()}": str(cfg.source_data_dir / f"Supplementary_Figure_S8_panel_{p.lower()}_source_data.csv") for p in "ABCD"},
        },
        "reports": {
            "sequence_length_audit": str(cfg.reports_dir / "sequence_length_audit.csv"),
            "tokenization_audit": str(cfg.reports_dir / "tokenization_audit.csv"),
            "unknown_token_audit": str(cfg.reports_dir / "unknown_token_audit.csv"),
            "padding_burden_summary": str(cfg.reports_dir / "padding_burden_summary.csv"),
            "cap_filtering_audit": str(cfg.reports_dir / "cap_filtering_audit.csv"),
            "amino_acid_composition_summary": str(cfg.reports_dir / "amino_acid_composition_summary.csv"),
            "composition_deviation_audit": str(cfg.reports_dir / "composition_deviation_audit.csv"),
            "composition_similarity_summary": str(cfg.reports_dir / "composition_similarity_summary.csv"),
            "noncanonical_residue_audit": str(cfg.reports_dir / "noncanonical_residue_audit.csv"),
            "readme": str(readme_path),
            "requirements": str(req_path),
            "no_main_figure_readme": str(no_main_path) if no_main_path else None,
        },
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(copies),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# ----------------------------- main workflow ---------------------------------

def run_step4_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_dirs(cfg); set_style(cfg)
    warnings = []

    model_ready, length_summary, token_summary, qc_summary, input_paths = load_inputs(cfg)
    model_ready, length_summary, token_summary, qc_summary, seq_col, split_col, padding_fraction_source = harmonize_inputs(
        model_ready, length_summary, token_summary, qc_summary, cfg, warnings
    )

    length_audit = build_length_audit(model_ready, length_summary, split_col, cfg)
    token_audit = build_token_audit(model_ready, token_summary, qc_summary, split_col, cfg, warnings, padding_fraction_source)
    padding_summary = build_padding_summary(model_ready, split_col, cfg)
    cap_filtering_audit = build_cap_filtering_audit(model_ready, length_audit, split_col, cfg)
    aa_comp = build_aa_comp(model_ready, seq_col, split_col, cfg)
    dev, comp_summary = build_comp_dev(aa_comp, cfg)
    unknown_audit = build_unknown_token_audit(token_audit)
    noncanonical_audit = build_noncanonical_residue_audit(model_ready, seq_col, split_col, cfg)

    if (token_audit["fraction_truncated"].fillna(0) > 0).any():
        warnings.append("Non-zero truncation fraction detected; interpret preprocessing as quantified, not artifact-free.")
    if (token_audit["unknown_token_fraction_nonpad"].fillna(0) > 0).any():
        warnings.append("Non-zero unknown-token fraction detected; inspect tokenization_audit.csv.")
    if (padding_summary["mean_padding_fraction"].fillna(0) > 0.75).any():
        warnings.append("High mean padding fraction detected in at least one split.")

    s7_panels, s7_all = build_s7_source(model_ready, length_audit, token_audit, split_col, cfg)
    s8_panels, s8_all = build_s8_source(aa_comp, dev, comp_summary, cfg)
    save_sources(cfg, s7_panels, s7_all, s8_panels, s8_all)
    reports = save_reports(
        cfg, length_audit, token_audit, padding_summary, aa_comp, dev, comp_summary,
        cap_filtering_audit=cap_filtering_audit,
        unknown_token_audit=unknown_audit,
        noncanonical_residue_audit=noncanonical_audit,
    )

    s7_paths = plot_s7(s7_panels, cfg)
    s8_paths = plot_s8(s8_panels, comp_summary, cfg)

    script_path, script_hash = code_snapshot(cfg)
    no_main = write_no_main_readme(cfg)
    readme = write_readme(cfg, input_paths, warnings)
    req = write_requirements(cfg)
    copies = root_copies(cfg)

    manifest = build_manifest(
        cfg, input_paths, s7_paths, s8_paths, readme, req, no_main, script_path, script_hash, copies, warnings,
        length_audit=length_audit,
        token_audit=token_audit,
        noncanonical_residue_audit=noncanonical_audit,
        padding_fraction_source=padding_fraction_source,
    )
    save_json(manifest, cfg.reports_dir / "step_04_manifest.json")
    save_json(manifest, cfg.output_root / "step_04_manifest.json")

    print("\n✅ Step 4 PLOS supplementary-figure package generated successfully.")
    print("Scope: supplementary-only frozen preprocessing/tokenization audit")
    print("Main figure generated: No")
    print(f"Output directory: {cfg.output_root}")

    print("\nInput files:")
    for k, v in input_paths.items():
        print(f"  - {k}: {v}")

    for label, paths in [("Supplementary Figure S7", s7_paths), ("Supplementary Figure S8", s8_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")

    print("\nSource data:")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S7_source_data_all_panels.csv'}")
    for p in "abc":
        print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_S7_panel_{p}_source_data.csv'}")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S8_source_data_all_panels.csv'}")
    for p in "abcd":
        print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_S8_panel_{p}_source_data.csv'}")

    print("\nReports:")
    for name in ["sequence_length_audit.csv", "tokenization_audit.csv", "unknown_token_audit.csv",
                 "padding_burden_summary.csv", "cap_filtering_audit.csv",
                 "amino_acid_composition_summary.csv", "composition_deviation_audit.csv",
                 "composition_similarity_summary.csv", "noncanonical_residue_audit.csv",
                 "step_04_manifest.json", "README_step_04_outputs.txt"]:
        print(f"  - {cfg.reports_dir / name}")
    if no_main:
        print(f"  - {no_main}")
    if warnings:
        print("\nWarnings:")
        for w in warnings:
            print(f"  - {w}")

    return {
        "model_ready": model_ready,
        "length_summary": length_summary,
        "token_summary": token_summary,
        "qc_summary": qc_summary,
        "length_audit": length_audit,
        "tokenization_audit": token_audit,
        "padding_summary": padding_summary,
        "amino_acid_composition": aa_comp,
        "composition_deviation_audit": dev,
        "composition_similarity_summary": comp_summary,
        "cap_filtering_audit": cap_filtering_audit,
        "unknown_token_audit": unknown_audit,
        "noncanonical_residue_audit": noncanonical_audit,
        "s7_panels": s7_panels,
        "s8_panels": s8_panels,
        "reports": reports,
        "manifest": manifest,
        "warnings": warnings,
    }


def main_notebook(
    project_root: str | Path = PROJECT_ROOT_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    model_ready_file: Optional[str | Path] = None,
    length_summary_file: Optional[str | Path] = None,
    token_summary_file: Optional[str | Path] = None,
    qc_summary_file: Optional[str | Path] = None,
    create_root_level_copies: bool = True,
    create_no_main_figure_readme: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        project_root=Path(project_root),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        model_ready_file=Path(model_ready_file) if model_ready_file else None,
        length_summary_file=Path(length_summary_file) if length_summary_file else None,
        token_summary_file=Path(token_summary_file) if token_summary_file else None,
        qc_summary_file=Path(qc_summary_file) if qc_summary_file else None,
        create_root_level_copies=create_root_level_copies,
        create_no_main_figure_readme=create_no_main_figure_readme,
        git_commit=git_commit,
    )
    return run_step4_redesign(cfg)


def running_inside_jupyter() -> bool:
    """Return True when the script is being executed inside a Jupyter kernel.

    In notebooks, __name__ can still be "__main__" when a full script is pasted
    into a cell. This guard prevents the command-line block from auto-running
    with default paths before the user can call main_notebook().
    """
    return "ipykernel" in sys.modules or any("jupyter" in arg.lower() for arg in sys.argv)

def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 4 PLOS supplementary figures S7/S8.")
    parser.add_argument("--project_root", default=str(PROJECT_ROOT_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--model_ready_file", default=None)
    parser.add_argument("--length_summary_file", default=None)
    parser.add_argument("--token_summary_file", default=None)
    parser.add_argument("--qc_summary_file", default=None)
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    parser.add_argument("--no_main_readme", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    if running_inside_jupyter():
        print(
            "\nLoaded OncoPep Step 4 functions in a Jupyter kernel.\n"
            "The command-line block was not auto-run to avoid using default paths.\n\n"
            "Now run:\n\n"
            "results = main_notebook()\n\n"
            "or provide explicit paths if needed:\n\n"
            "results = main_notebook(\n"
            "    project_root='/home/data3/Moe/nature_computational_peponco',\n"
            "    base_dir='/home/data3/Moe/nature_computational_peponco/PLOS',\n"
            "    output_root_name='plos_comp/step_04',\n"
            ")\n"
        )
    else:
        args = parse_args()
        cfg = Config(
            project_root=Path(args.project_root),
            base_dir=Path(args.base_dir),
            output_root_name=args.output_root_name,
            model_ready_file=Path(args.model_ready_file) if args.model_ready_file else None,
            length_summary_file=Path(args.length_summary_file) if args.length_summary_file else None,
            token_summary_file=Path(args.token_summary_file) if args.token_summary_file else None,
            qc_summary_file=Path(args.qc_summary_file) if args.qc_summary_file else None,
            create_root_level_copies=not args.no_root_copies,
            create_no_main_figure_readme=not args.no_main_readme,
            git_commit=args.git_commit,
        )
        run_step4_redesign(cfg)

In [ ]:
results = main_notebook()

OncoPep Step 4 — PLOS Computational Biology supplementary-figure redesign.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""

Jupyter-safe execution
----------------------
When this file is pasted or run inside Jupyter, the bottom command-line block
will not auto-run. First run the full file/cell to define the functions, then
run:

results = main_notebook()

The default input paths point to:
/home/data3/Moe/nature_computational_peponco/step4/tables/

OncoPep Step 4 — PLOS Computational Biology supplementary-figure redesign.

Step 4 = frozen preprocessing, tokenization, sequence-length, padding, and
amino-acid composition audit.

Outputs:
  Supplementary Figure S7 — frozen sequence preprocessing and tokenization audit
  Supplementary Figure S8 — amino-acid composition preservation across partitions

No main figure is generated in Step 4. Main Figure 2 is reserved for the
OncoPep architecture and conditional generation workflow.

v5 improvements:
  - Supplementary Figure S7 is redesigned as a clean two-panel figure.
  - The all-zero truncation/unknown-token audit panel is removed from the figure
    and retained in CSV reports/source data.
  - S7 now shows only:
      A. Model-ready sequence-length coverage
      B. Per-sequence padding burden
  - The length-cap label is moved away from the legend.
  - S8 label refinements are retained.
  - Metric-source/provenance, cap/filtering, unknown-token, split-level
    preprocessing, and noncanonical-residue audit tables remain exported for
    reviewer inspection.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd


PROJECT_ROOT_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco")
BASE_DIR_DEFAULT = Path("/home/data3/Moe/nature_computational_peponco/PLOS")
OUTPUT_ROOT_NAME_DEFAULT = "plos_comp/step_04"

SPLIT_ORDER = ("train", "val", "test")
SPLIT_DISPLAY = {"train": "Train", "val": "Validation", "test": "Test"}
AA_ORDER = tuple("ACDEFGHIKLMNPQRSTVWY")

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}
SPLIT_FACE = {"train": PLOS["blue"], "val": PLOS["mint"], "test": PLOS["coral"]}
SPLIT_EDGE = {"train": "#176F8A", "val": "#6F9F7A", "test": "#A84F42"}
HELDOUT_FACE = {"val": PLOS["mint"], "test": PLOS["coral"]}
HELDOUT_EDGE = {"val": "#6F9F7A", "test": "#A84F42"}


@dataclass
class Config:
    project_root: Path = PROJECT_ROOT_DEFAULT
    base_dir: Path = BASE_DIR_DEFAULT
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT

    model_ready_file: Optional[Path] = None
    length_summary_file: Optional[Path] = None
    token_summary_file: Optional[Path] = None
    qc_summary_file: Optional[Path] = None

    split_order: Tuple[str, ...] = SPLIT_ORDER
    aa_order: Tuple[str, ...] = AA_ORDER

    png_dpi: int = 600
    tiff_dpi: int = 600
    create_root_level_copies: bool = True
    create_no_main_figure_readme: bool = True
    git_commit: Optional[str] = None

    font_size: float = 8.3
    title_size: float = 9.2
    axis_label_size: float = 8.9
    tick_label_size: float = 7.8
    legend_size: float = 7.7
    annotation_size: float = 7.3
    panel_letter_size: float = 13.0

    @property
    def step4_tables_dir(self) -> Path:
        return self.project_root / "step4" / "tables"

    @property
    def output_root(self) -> Path:
        return self.base_dir / self.output_root_name

    @property
    def supp_dir(self) -> Path:
        return self.output_root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.output_root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.output_root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.output_root / "code"

    @property
    def main_figure_dir(self) -> Path:
        return self.output_root / "main_figure"


# ----------------------------- utilities -------------------------------------

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def ensure_dirs(cfg: Config) -> None:
    for d in [cfg.output_root, cfg.supp_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        d.mkdir(parents=True, exist_ok=True)
    if cfg.create_no_main_figure_readme:
        cfg.main_figure_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_json(data: Any, path: Path) -> None:
    def default(o: Any) -> Any:
        if isinstance(o, Path):
            return str(o)
        if isinstance(o, tuple):
            return list(o)
        if isinstance(o, (np.integer, np.floating, np.bool_)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(f"{type(o).__name__} is not JSON serializable")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=default), encoding="utf-8")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def normalize_split(s: pd.Series) -> pd.Series:
    return (
        s.astype(str).str.strip().str.lower()
        .replace({"validation": "val", "valid": "val", "dev": "val", "training": "train", "testing": "test"})
    )


def resolve_col(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"Could not find one of {list(candidates)}. Available columns: {list(df.columns)}")
    return None


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    return None


def get_value(df: pd.DataFrame, split: str, col: str, default: float = np.nan) -> float:
    if "split" not in df.columns or col not in df.columns:
        return float(default)
    sub = df.loc[df["split"] == split, col]
    if sub.empty:
        return float(default)
    val = pd.to_numeric(sub, errors="coerce").dropna()
    return float(val.iloc[0]) if len(val) else float(default)


def validate_one_row_per_split(df: pd.DataFrame, table_name: str, cfg: Config, warnings: list[str]) -> None:
    """Warn if a split-summary table has duplicate rows for the same split."""
    if "split" not in df.columns:
        warnings.append(f"{table_name} has no split column; split-level validation could not be performed.")
        return
    counts = df["split"].value_counts(dropna=False)
    duplicates = counts[counts > 1]
    if not duplicates.empty:
        warnings.append(
            f"{table_name} contains duplicate split rows {duplicates.to_dict()}; "
            "first non-missing value will be used for plotted split-level metrics."
        )


def metric_value_and_source(
    df: pd.DataFrame,
    split: str,
    col: str,
    table_name: str,
    allow_missing: bool,
    warnings: list[str],
    default: float = np.nan,
) -> tuple[float, str]:
    """Return a split-level metric and a provenance/source flag.

    Missing truncation and unknown-token metrics should not silently become zero.
    For optional metrics, missing values are returned as NA with a source flag.
    """
    if "split" not in df.columns:
        msg = f"{table_name} has no split column; metric '{col}' unavailable for split '{split}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_split_column"
        raise ValueError(msg)

    if col not in df.columns:
        msg = f"{table_name} is missing required metric column '{col}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_column"
        raise ValueError(msg)

    sub = df.loc[df["split"] == split, col]
    if sub.empty:
        msg = f"{table_name} has no row for split '{split}' when reading '{col}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_split"
        raise ValueError(msg)

    vals = pd.to_numeric(sub, errors="coerce").dropna()
    if vals.empty:
        msg = f"{table_name} has only nonnumeric/NA values for '{col}' in split '{split}'."
        if allow_missing:
            warnings.append(msg)
            return float(default), "missing_value"
        raise ValueError(msg)

    return float(vals.iloc[0]), f"measured:{table_name}.{col}"


def clamp_fraction(value: float, metric_name: str, warnings: list[str]) -> float:
    """Constrain a numeric metric to [0, 1] and warn if clipping was needed."""
    if not np.isfinite(value):
        return value
    clipped = min(max(float(value), 0.0), 1.0)
    if clipped != float(value):
        warnings.append(f"Metric '{metric_name}'={value} was outside [0,1] and was clipped to {clipped}.")
    return clipped


# ----------------------------- style -----------------------------------------

def set_style(cfg: Config) -> None:
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": cfg.font_size,
        "axes.titlesize": cfg.title_size,
        "axes.labelsize": cfg.axis_label_size,
        "xtick.labelsize": cfg.tick_label_size,
        "ytick.labelsize": cfg.tick_label_size,
        "legend.fontsize": cfg.legend_size,
        "axes.linewidth": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "text.color": PLOS["dark"],
        "axes.labelcolor": PLOS["dark"],
        "axes.edgecolor": PLOS["dark"],
        "xtick.color": PLOS["dark"],
        "ytick.color": PLOS["dark"],
    })


def beautify(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.45)
    ax.grid(False, axis="x")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.75)
        ax.spines[side].set_color(PLOS["dark"])


def panel_letter(ax: plt.Axes, letter: str, cfg: Config, dx: float = -0.13, dy: float = 1.08) -> None:
    ax.text(dx, dy, letter.upper(), transform=ax.transAxes, fontsize=cfg.panel_letter_size,
            fontweight="bold", va="top", ha="left", color=PLOS["dark"])


def save_fig(fig: plt.Figure, base: Path, cfg: Config) -> Dict[str, str]:
    base.parent.mkdir(parents=True, exist_ok=True)
    paths = {"png": base.with_suffix(".png"), "pdf": base.with_suffix(".pdf"), "tiff": base.with_suffix(".tiff")}
    fig.savefig(paths["pdf"], bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["png"], dpi=cfg.png_dpi, bbox_inches="tight", pad_inches=0.06)
    fig.savefig(paths["tiff"], dpi=cfg.tiff_dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)
    return {k: str(v) for k, v in paths.items()}


# ----------------------------- loading ---------------------------------------

def candidates(cfg: Config, name: str, explicit: Optional[Path]) -> list[Path]:
    out = []
    if explicit:
        out.append(Path(explicit))
    out.extend([
        cfg.step4_tables_dir / name,
        cfg.project_root / "redesign" / "step4" / "tables" / name,
        Path("/home/data3/Moe/nature_computational_peponco/step4/tables") / name,
        Path("/home/data3/Moe/nature_computational_peponco/redesign/step4/tables") / name,
        Path("/home/data3/Moe/nature_computational_peponco/step4/tables") / name,
    ])
    return out


def load_csv(cfg: Config, name: str, explicit: Optional[Path]) -> Tuple[pd.DataFrame, Path]:
    path = first_existing(candidates(cfg, name, explicit))
    if path is None:
        raise FileNotFoundError("Missing input table " + name + ". Checked:\n- " + "\n- ".join(map(str, candidates(cfg, name, explicit))))
    return pd.read_csv(path, low_memory=False), path


def load_inputs(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Path]]:
    model_ready, p1 = load_csv(cfg, "step4_model_ready_sequences.csv", cfg.model_ready_file)
    length_summary, p2 = load_csv(cfg, "step4_sequence_length_summary.csv", cfg.length_summary_file)
    token_summary, p3 = load_csv(cfg, "step4_token_coverage_summary.csv", cfg.token_summary_file)
    qc_summary, p4 = load_csv(cfg, "step4_tokenization_qc.csv", cfg.qc_summary_file)
    return model_ready, length_summary, token_summary, qc_summary, {
        "model_ready_sequences": p1,
        "sequence_length_summary": p2,
        "token_coverage_summary": p3,
        "tokenization_qc": p4,
    }


def infer_chosen_cap(length_summary: pd.DataFrame) -> int:
    for c in ["chosen_max_sequence_length", "max_sequence_length", "sequence_length_cap", "chosen_cap"]:
        if c in length_summary.columns:
            vals = pd.to_numeric(length_summary[c], errors="coerce").dropna()
            if len(vals):
                return int(vals.iloc[0])
    return 60


def infer_model_len(length_summary: pd.DataFrame, fallback: Optional[int] = None) -> int:
    for c in ["max_model_length_including_special_tokens", "max_model_length", "model_length"]:
        if c in length_summary.columns:
            vals = pd.to_numeric(length_summary[c], errors="coerce").dropna()
            if len(vals):
                return int(vals.iloc[0])
    return int(fallback if fallback is not None else infer_chosen_cap(length_summary))


def harmonize_inputs(model_ready: pd.DataFrame, length_summary: pd.DataFrame,
                     token_summary: pd.DataFrame, qc_summary: pd.DataFrame,
                     cfg: Config, warnings: list[str]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str, str, str]:
    seq_col = resolve_col(model_ready, ["sequence", "seq", "peptide_sequence"])
    split_col = resolve_col(model_ready, ["assigned_split", "split", "data_split"])
    if split_col != "assigned_split":
        model_ready = model_ready.rename(columns={split_col: "assigned_split"})
        split_col = "assigned_split"
    model_ready[split_col] = normalize_split(model_ready[split_col])

    if "sequence_length" not in model_ready.columns:
        model_ready["sequence_length"] = model_ready[seq_col].astype(str).map(len)

    cap = infer_chosen_cap(length_summary)
    model_len = infer_model_len(length_summary, fallback=cap)

    if "padding_fraction" in model_ready.columns:
        model_ready["padding_fraction"] = pd.to_numeric(model_ready["padding_fraction"], errors="coerce")
        padding_fraction_source = "measured:model_ready_sequences.padding_fraction"
    else:
        model_ready["padding_fraction"] = (1.0 - pd.to_numeric(model_ready["sequence_length"], errors="coerce") / max(model_len, 1)).clip(0, 1)
        padding_fraction_source = "inferred:1-sequence_length/max_model_length"
        warnings.append(
            "Sequence-level padding_fraction was absent and was inferred from sequence_length and max_model_length. "
            "If special tokens are used, confirm this approximation against the tokenizer output."
        )

    for df in [length_summary, token_summary, qc_summary]:
        c = resolve_col(df, ["split", "assigned_split", "data_split"], required=False)
        if c is not None:
            df.rename(columns={c: "split"}, inplace=True)
            df["split"] = normalize_split(df["split"])

    missing = [s for s in cfg.split_order if s not in set(model_ready[split_col].dropna())]
    if missing:
        raise ValueError(f"Missing expected splits in model-ready table: {missing}")
    if "split" not in token_summary.columns or "split" not in qc_summary.columns:
        raise ValueError("token_summary and qc_summary must include a split column.")

    validate_one_row_per_split(token_summary, "token_summary", cfg, warnings)
    validate_one_row_per_split(qc_summary, "qc_summary", cfg, warnings)
    return model_ready, length_summary, token_summary, qc_summary, seq_col, split_col, padding_fraction_source


# ----------------------------- analyses --------------------------------------

def build_length_audit(model_ready: pd.DataFrame, length_summary: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    cap = infer_chosen_cap(length_summary)
    model_len = infer_model_len(length_summary, fallback=cap)
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "sequence_length"], errors="coerce").dropna().to_numpy()
        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(vals)),
            "min_length": float(np.min(vals)) if len(vals) else np.nan,
            "q25_length": float(np.percentile(vals, 25)) if len(vals) else np.nan,
            "median_length": float(np.median(vals)) if len(vals) else np.nan,
            "mean_length": float(np.mean(vals)) if len(vals) else np.nan,
            "q75_length": float(np.percentile(vals, 75)) if len(vals) else np.nan,
            "max_length": float(np.max(vals)) if len(vals) else np.nan,
            "chosen_max_sequence_length": int(cap),
            "max_model_length_including_special_tokens": int(model_len),
            "fraction_exceeding_cap": float(np.mean(vals > cap)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def build_token_audit(model_ready: pd.DataFrame, token_summary: pd.DataFrame, qc_summary: pd.DataFrame,
                      split_col: str, cfg: Config, warnings: list[str], padding_fraction_source: str) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        sub = model_ready.loc[model_ready[split_col] == split]
        mean_pad_default = float(pd.to_numeric(sub["padding_fraction"], errors="coerce").mean())

        frac_trunc, frac_trunc_source = metric_value_and_source(
            qc_summary, split, "fraction_truncated", "qc_summary",
            allow_missing=False, warnings=warnings
        )
        unk, unk_source = metric_value_and_source(
            token_summary, split, "unknown_token_fraction_nonpad", "token_summary",
            allow_missing=False, warnings=warnings
        )

        # mean padding can be measured in qc_summary or derived from sequence-level padding_fraction.
        if "mean_padding_fraction" in qc_summary.columns:
            mean_pad, mean_pad_source = metric_value_and_source(
                qc_summary, split, "mean_padding_fraction", "qc_summary",
                allow_missing=True, warnings=warnings, default=mean_pad_default
            )
            if not np.isfinite(mean_pad):
                mean_pad = mean_pad_default
                mean_pad_source = f"derived_from_sequence_level:{padding_fraction_source}"
        else:
            mean_pad = mean_pad_default
            mean_pad_source = f"derived_from_sequence_level:{padding_fraction_source}"
            warnings.append("qc_summary is missing 'mean_padding_fraction'; split mean padding was derived from sequence-level padding_fraction.")

        cond_missing, cond_source = metric_value_and_source(
            qc_summary, split, "fraction_primary_condition_id_missing", "qc_summary",
            allow_missing=True, warnings=warnings, default=np.nan
        )

        frac_trunc = clamp_fraction(frac_trunc, "fraction_truncated", warnings)
        unk = clamp_fraction(unk, "unknown_token_fraction_nonpad", warnings)
        mean_pad = clamp_fraction(mean_pad, "mean_padding_fraction", warnings)
        cond_missing = clamp_fraction(cond_missing, "fraction_primary_condition_id_missing", warnings)

        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(sub)),
            "fraction_truncated": frac_trunc,
            "fraction_truncated_source": frac_trunc_source,
            "non_truncated_fraction": 1.0 - frac_trunc if np.isfinite(frac_trunc) else np.nan,
            "unknown_token_fraction_nonpad": unk,
            "unknown_token_fraction_nonpad_source": unk_source,
            "known_token_fraction_nonpad": 1.0 - unk if np.isfinite(unk) else np.nan,
            "mean_padding_fraction": mean_pad,
            "mean_padding_fraction_source": mean_pad_source,
            "effective_token_fraction": 1.0 - mean_pad if np.isfinite(mean_pad) else np.nan,
            "fraction_primary_condition_id_missing": cond_missing,
            "fraction_primary_condition_id_missing_source": cond_source,
            "valid_condition_fraction": 1.0 - cond_missing if np.isfinite(cond_missing) else np.nan,
        })
    return pd.DataFrame(rows)


def build_padding_summary(model_ready: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "padding_fraction"], errors="coerce").dropna().to_numpy()
        rows.append({
            "split": split, "split_display": SPLIT_DISPLAY[split], "n_sequences": int(len(vals)),
            "mean_padding_fraction": float(np.mean(vals)) if len(vals) else np.nan,
            "sd_padding_fraction": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "median_padding_fraction": float(np.median(vals)) if len(vals) else np.nan,
            "q25_padding_fraction": float(np.percentile(vals, 25)) if len(vals) else np.nan,
            "q75_padding_fraction": float(np.percentile(vals, 75)) if len(vals) else np.nan,
            "min_padding_fraction": float(np.min(vals)) if len(vals) else np.nan,
            "max_padding_fraction": float(np.max(vals)) if len(vals) else np.nan,
        })
    return pd.DataFrame(rows)


def build_cap_filtering_audit(model_ready: pd.DataFrame, length_audit: pd.DataFrame, split_col: str, cfg: Config) -> pd.DataFrame:
    """Audit model-ready length coverage relative to the selected cap.

    This does not infer pre-filtering losses unless raw pre-filter counts are
    available; it documents model-ready sequences that exceed the selected cap.
    """
    cap = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
    rows = []
    for split in cfg.split_order:
        vals = pd.to_numeric(model_ready.loc[model_ready[split_col] == split, "sequence_length"], errors="coerce").dropna()
        n = int(len(vals))
        n_exceed = int((vals > cap).sum())
        rows.append({
            "split": split,
            "split_display": SPLIT_DISPLAY[split],
            "n_model_ready_sequences": n,
            "chosen_max_sequence_length": cap,
            "n_model_ready_sequences_exceeding_cap": n_exceed,
            "fraction_model_ready_sequences_exceeding_cap": float(n_exceed / n) if n else np.nan,
            "filtering_loss_available": False,
            "n_prefilter_sequences": np.nan,
            "n_filtered_by_length": np.nan,
            "filtering_fraction": np.nan,
            "interpretation_note": "Audit is based on model-ready sequences; pre-filter loss requires raw pre-filter counts.",
        })
    return pd.DataFrame(rows)


def build_unknown_token_audit(token_audit: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "split", "split_display", "n_sequences",
        "unknown_token_fraction_nonpad", "unknown_token_fraction_nonpad_source",
        "known_token_fraction_nonpad",
    ]
    return token_audit[[c for c in cols if c in token_audit.columns]].copy()


def build_noncanonical_residue_audit(model_ready: pd.DataFrame, seq_col: str, split_col: str, cfg: Config) -> pd.DataFrame:
    canonical = set(cfg.aa_order)
    rows = []
    for split in cfg.split_order:
        seqs = model_ready.loc[model_ready[split_col] == split, seq_col].astype(str).tolist()
        concat = "".join(seqs)
        total_chars = len(concat)
        canonical_count = sum(1 for ch in concat if ch in canonical)
        noncanonical_count = total_chars - canonical_count
        noncanonical_symbols = sorted(set(ch for ch in concat if ch not in canonical))
        rows.append({
            "split": split,
            "split_display": SPLIT_DISPLAY[split],
            "n_sequences": int(len(seqs)),
            "total_residue_characters": int(total_chars),
            "canonical_residue_characters": int(canonical_count),
            "noncanonical_residue_characters": int(noncanonical_count),
            "noncanonical_residue_fraction": float(noncanonical_count / total_chars) if total_chars else np.nan,
            "noncanonical_symbols": "".join(noncanonical_symbols) if noncanonical_symbols else "",
            "canonical_vocabulary": "".join(cfg.aa_order),
        })
    return pd.DataFrame(rows)


def build_aa_comp(model_ready: pd.DataFrame, seq_col: str, split_col: str, cfg: Config) -> pd.DataFrame:
    rows = []
    for split in cfg.split_order:
        concat = "".join(model_ready.loc[model_ready[split_col] == split, seq_col].astype(str).tolist())
        total = sum(concat.count(aa) for aa in cfg.aa_order)
        denom = max(total, 1)
        for aa in cfg.aa_order:
            rows.append({"split": split, "split_display": SPLIT_DISPLAY[split], "amino_acid": aa,
                         "count": int(concat.count(aa)), "canonical_total": int(total),
                         "fraction": float(concat.count(aa) / denom)})
    return pd.DataFrame(rows)


def build_comp_dev(aa_comp: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    piv = aa_comp.pivot_table(index="amino_acid", columns="split", values="fraction", aggfunc="first").reindex(cfg.aa_order)
    rows = []
    for aa in cfg.aa_order:
        tr, va, te = float(piv.loc[aa, "train"]), float(piv.loc[aa, "val"]), float(piv.loc[aa, "test"])
        rows.append({"amino_acid": aa, "train_fraction": tr, "val_fraction": va, "test_fraction": te,
                     "val_minus_train": va - tr, "test_minus_train": te - tr,
                     "abs_val_minus_train": abs(va - tr), "abs_test_minus_train": abs(te - tr)})
    dev = pd.DataFrame(rows)
    summary_rows = []
    for split in ["val", "test"]:
        tr = dev["train_fraction"].to_numpy(float)
        held = dev[f"{split}_fraction"].to_numpy(float)
        delta = held - tr
        r = float(np.corrcoef(tr, held)[0, 1]) if np.std(tr) > 0 and np.std(held) > 0 else np.nan
        summary_rows.append({"comparison": f"{SPLIT_DISPLAY[split]} vs Train", "heldout_split": split,
                             "pearson_r": r, "mean_absolute_deviation": float(np.mean(np.abs(delta))),
                             "max_absolute_deviation": float(np.max(np.abs(delta))),
                             "signed_deviation_mean": float(np.mean(delta)), "n_amino_acids": int(len(delta))})
    return dev, pd.DataFrame(summary_rows)


# ----------------------------- source data -----------------------------------

def build_s7_source(model_ready: pd.DataFrame, length_audit: pd.DataFrame, token_audit: pd.DataFrame, split_col: str, cfg: Config):
    """Build source data for the two-panel Supplementary Figure S7.

    Truncation and unknown-token values are retained in tokenization_audit.csv
    and unknown_token_audit.csv rather than plotted as a separate all-zero panel.
    """
    cap = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
    model_len = int(length_audit["max_model_length_including_special_tokens"].dropna().iloc[0])

    a = model_ready[[split_col, "sequence_length"]].copy().rename(columns={split_col: "split", "sequence_length": "value"})
    a["split"] = normalize_split(a["split"]); a["split_display"] = a["split"].map(SPLIT_DISPLAY)
    a["chosen_max_sequence_length"] = cap; a["max_model_length_including_special_tokens"] = model_len
    a["panel"] = "A"; a["data_type"] = "sequence_length_distribution"; a["metric_name"] = "sequence_length"
    a["unit"] = "amino_acid_residues"; a["metric_definition"] = "Number of residues in the standardized model-ready peptide sequence."

    b = model_ready[[split_col, "padding_fraction"]].copy().rename(columns={split_col: "split", "padding_fraction": "value"})
    b["split"] = normalize_split(b["split"]); b["split_display"] = b["split"].map(SPLIT_DISPLAY)
    b["panel"] = "B"; b["data_type"] = "padding_fraction_distribution"; b["metric_name"] = "padding_fraction"
    b["unit"] = "fraction"; b["metric_definition"] = "Fraction of final model input positions occupied by padding."

    panels = {"A": a, "B": b}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def build_s8_source(aa_comp: pd.DataFrame, dev: pd.DataFrame, summary: pd.DataFrame, cfg: Config):
    a = aa_comp.copy()
    a["panel"] = "A"; a["data_type"] = "amino_acid_frequency_profile"; a["metric_name"] = "amino_acid_fraction"
    a["unit"] = "fraction"; a["metric_definition"] = "Count of a canonical amino acid divided by total canonical amino-acid count in the split."

    rows = []
    for _, r in dev.iterrows():
        for split in ["val", "test"]:
            rows.append({"amino_acid": r["amino_acid"], "heldout_split": split,
                         "heldout_split_display": SPLIT_DISPLAY[split],
                         "value": r[f"{split}_minus_train"],
                         "absolute_value": r[f"abs_{split}_minus_train"],
                         "panel": "B", "data_type": "heldout_minus_train_composition_deviation",
                         "unit": "fraction_difference",
                         "metric_definition": "Held-out amino-acid fraction minus training amino-acid fraction."})
    b = pd.DataFrame(rows)

    rows = []
    for _, r in dev.iterrows():
        for split in ["val", "test"]:
            rows.append({"amino_acid": r["amino_acid"], "heldout_split": split,
                         "heldout_split_display": SPLIT_DISPLAY[split],
                         "train_fraction": r["train_fraction"],
                         "heldout_fraction": r[f"{split}_fraction"],
                         "panel": "C", "data_type": "heldout_vs_train_composition_agreement",
                         "unit": "fraction",
                         "metric_definition": "Training and held-out amino-acid fractions for composition agreement plotting."})
    c = pd.DataFrame(rows)

    d = summary.copy()
    d["panel"] = "D"; d["data_type"] = "composition_deviation_summary"
    d["unit"] = "fraction"; d["metric_definition"] = "Summary of held-out amino-acid composition deviation relative to training composition."
    panels = {"A": a, "B": b, "C": c, "D": d}
    return panels, pd.concat(panels.values(), ignore_index=True, sort=False)


def save_sources(cfg: Config, s7_panels, s7_all, s8_panels, s8_all) -> None:
    save_csv(s7_all, cfg.source_data_dir / "Supplementary_Figure_S7_source_data_all_panels.csv")
    save_csv(s8_all, cfg.source_data_dir / "Supplementary_Figure_S8_source_data_all_panels.csv")
    for p, df in s7_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S7_panel_{p.lower()}_source_data.csv")
    for p, df in s8_panels.items():
        save_csv(df, cfg.source_data_dir / f"Supplementary_Figure_S8_panel_{p.lower()}_source_data.csv")


# ----------------------------- plotting --------------------------------------

def plot_s7a(ax, df, cfg):
    for split in cfg.split_order:
        vals = pd.to_numeric(df.loc[df["split"] == split, "value"], errors="coerce").dropna().sort_values().to_numpy()
        if len(vals):
            y = np.arange(1, len(vals) + 1) / len(vals)
            ax.step(vals, y, where="post", color=SPLIT_EDGE[split], lw=1.8, label=SPLIT_DISPLAY[split], zorder=3)
    cap_vals = df["chosen_max_sequence_length"].dropna().unique()
    if len(cap_vals):
        cap = float(cap_vals[0])
        ax.axvline(cap, color=PLOS["charcoal"], linestyle=(0, (3, 2.5)), lw=1.15, zorder=2)
        ax.text(cap - 0.8, 0.94, f"Length cap = {int(cap)} aa", fontsize=cfg.annotation_size, ha="right", va="top", color=PLOS["dark"])
    ax.set_xlabel("Sequence length (aa)")
    ax.set_ylabel("Cumulative fraction")
    ax.set_ylim(0, 1.02)
    ax.set_title("Model-ready sequence-length coverage", pad=8)
    beautify(ax)
    ax.legend(loc="lower right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s7b(ax, df, cfg):
    """Compact table/card panel for truncation and unknown-token audit.

    This is clearer than an empty all-zero bar chart when the audit values are zero.
    """
    metrics = [
        ("fraction_truncated", "Truncation\nfraction"),
        ("unknown_token_fraction_nonpad", "Unknown-token\nfraction"),
    ]

    ax.set_xlim(0, len(metrics))
    ax.set_ylim(0, len(cfg.split_order))
    ax.invert_yaxis()

    for i, split in enumerate(cfg.split_order):
        for j, (metric, label) in enumerate(metrics):
            sub = df.loc[(df["split"] == split) & (df["metric"] == metric), "value"]
            val = float(sub.iloc[0]) if len(sub) and pd.notna(sub.iloc[0]) else np.nan
            if np.isfinite(val):
                intensity = 0.06 + 0.48 * min(max(val, 0), 1)
                face = mpl.colors.to_rgba(PLOS["coral"] if val > 0 else PLOS["light"], alpha=intensity if val > 0 else 0.35)
                txt = "0" if abs(val) < 1e-12 else f"{val:.3f}"
            else:
                face = mpl.colors.to_rgba(PLOS["light"], alpha=0.25)
                txt = "NA"

            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=face, edgecolor=PLOS["light"], lw=0.75))
            ax.text(j + 0.5, i + 0.52, txt, ha="center", va="center",
                    fontsize=cfg.annotation_size + 1.6, color=PLOS["dark"], fontweight="bold")

    for j, (_, label) in enumerate(metrics):
        ax.text(j + 0.5, -0.13, label, ha="center", va="bottom",
                fontsize=cfg.tick_label_size + 0.3, color=PLOS["dark"])

    for i, split in enumerate(cfg.split_order):
        ax.text(-0.08, i + 0.5, SPLIT_DISPLAY[split], ha="right", va="center",
                fontsize=cfg.tick_label_size + 0.4, color=PLOS["dark"])

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title("Truncation / unknown-token audit", pad=22)


def plot_s7c(ax, df, cfg):
    positions = np.arange(1, 4)
    data = [pd.to_numeric(df.loc[df["split"] == s, "value"], errors="coerce").dropna().to_numpy() for s in cfg.split_order]
    bp = ax.boxplot(data, positions=positions, widths=0.46, patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops=dict(color=PLOS["dark"], lw=1.35),
                    whiskerprops=dict(color=PLOS["charcoal"], lw=0.85),
                    capprops=dict(color=PLOS["charcoal"], lw=0.85),
                    boxprops=dict(lw=1.05))
    for patch, split in zip(bp["boxes"], cfg.split_order):
        patch.set_facecolor(SPLIT_FACE[split]); patch.set_alpha(0.34); patch.set_edgecolor(SPLIT_EDGE[split])
    for pos, arr, split in zip(positions, data, cfg.split_order):
        if len(arr):
            ax.scatter(pos, np.median(arr), s=16, color=SPLIT_EDGE[split], edgecolor="white", lw=0.35, zorder=5)
    ax.set_xticks(positions); ax.set_xticklabels([SPLIT_DISPLAY[s] for s in cfg.split_order])
    ax.set_ylabel("Padding fraction")
    ax.set_ylim(0, 1.0)
    ax.set_title("Per-sequence padding burden", pad=8)
    beautify(ax)


def plot_s7d(ax, df, cfg):
    metrics = [("non_truncated_fraction", "Non-\ntruncated"), ("known_token_fraction_nonpad", "Known-\ntoken"),
               ("effective_token_fraction", "Effective-\ntoken"), ("mean_padding_fraction", "Mean\npadding")]
    ax.set_xlim(0, len(metrics)); ax.set_ylim(0, len(cfg.split_order)); ax.invert_yaxis()
    for i, split in enumerate(cfg.split_order):
        for j, (metric, label) in enumerate(metrics):
            sub = df.loc[(df["split"] == split) & (df["metric"] == metric), "value"]
            val = float(sub.iloc[0]) if len(sub) and pd.notna(sub.iloc[0]) else np.nan
            base = PLOS["light"] if metric == "mean_padding_fraction" else PLOS["blue"]
            alpha = 0.10 + 0.48 * min(max(val, 0), 1) if np.isfinite(val) else 0.08
            ax.add_patch(Rectangle((j, i), 1, 1, facecolor=mpl.colors.to_rgba(base, alpha=alpha),
                                   edgecolor=PLOS["light"], lw=0.7))
            ax.text(j + 0.5, i + 0.53, f"{val:.2f}" if np.isfinite(val) else "NA",
                    ha="center", va="center", fontsize=cfg.annotation_size + 0.8)
    for j, (_, label) in enumerate(metrics):
        ax.text(j + 0.5, -0.15, label, ha="center", va="bottom", fontsize=cfg.tick_label_size)
    for i, split in enumerate(cfg.split_order):
        ax.text(-0.10, i + 0.5, SPLIT_DISPLAY[split], ha="right", va="center", fontsize=cfg.tick_label_size)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_title("Split-level preprocessing audit", pad=20)


def plot_s7(panels, cfg):
    set_style(cfg)
    fig = plt.figure(figsize=(9.7, 4.35))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.18, 1.0], wspace=0.36)
    axes = [fig.add_subplot(gs[0, i]) for i in range(2)]
    fig.subplots_adjust(left=0.085, right=0.985, top=0.86, bottom=0.18)

    plot_s7a(axes[0], panels["A"], cfg); panel_letter(axes[0], "A", cfg)
    plot_s7c(axes[1], panels["B"], cfg); panel_letter(axes[1], "B", cfg)

    return save_fig(fig, cfg.supp_dir / "Supplementary_Figure_S7_redesigned", cfg)


def plot_s8a(ax, df, cfg):
    x = np.arange(len(cfg.aa_order)); width = 0.23
    offsets = {"train": -width, "val": 0.0, "test": width}
    for split in cfg.split_order:
        sub = df.loc[df["split"] == split].set_index("amino_acid").reindex(cfg.aa_order)
        vals = pd.to_numeric(sub["fraction"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offsets[split], vals, width=width, color=SPLIT_FACE[split],
               edgecolor=SPLIT_EDGE[split], lw=0.45, alpha=0.78, label=SPLIT_DISPLAY[split], zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(cfg.aa_order)
    ax.set_xlabel("Amino acid"); ax.set_ylabel("Fraction")
    ax.set_title("Amino-acid frequency profile", pad=8)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8b(ax, df, cfg):
    x = np.arange(len(cfg.aa_order)); width = 0.23
    for split, offset in [("val", -width/2), ("test", width/2)]:
        sub = df.loc[df["heldout_split"] == split].set_index("amino_acid").reindex(cfg.aa_order)
        vals = pd.to_numeric(sub["value"], errors="coerce").fillna(0).to_numpy()
        ax.bar(x + offset, vals, width=width, color=HELDOUT_FACE[split], edgecolor=HELDOUT_EDGE[split],
               lw=0.45, alpha=0.82, label=f"{SPLIT_DISPLAY[split]} − Train", zorder=3)
    ax.axhline(0, color=PLOS["charcoal"], lw=0.9, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels(cfg.aa_order)
    ax.set_xlabel("Amino acid"); ax.set_ylabel("Δ fraction relative to train")
    ax.set_title("Held-out deviation from train", pad=8)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8c(ax, df, summary, cfg):
    maxv = max(float(pd.to_numeric(df["train_fraction"], errors="coerce").max()),
               float(pd.to_numeric(df["heldout_fraction"], errors="coerce").max()), 0.01)
    lim = maxv * 1.12
    ax.plot([0, lim], [0, lim], color=PLOS["charcoal"], linestyle=(0, (3, 2.5)), lw=0.9, zorder=1)
    for split in ["val", "test"]:
        sub = df.loc[df["heldout_split"] == split]
        ax.scatter(sub["train_fraction"], sub["heldout_fraction"], s=32, color=HELDOUT_FACE[split],
                   edgecolor=HELDOUT_EDGE[split], lw=0.6, alpha=0.88, label=SPLIT_DISPLAY[split], zorder=3)
    ann = []
    for _, r in summary.iterrows():
        ann.append(f"{SPLIT_DISPLAY.get(r['heldout_split'], r['heldout_split'])}: r={r['pearson_r']:.3f}, MAE={r['mean_absolute_deviation']:.3f}")
    if ann:
        ax.text(0.04, 0.96, "\n".join(ann), transform=ax.transAxes, ha="left", va="top",
                fontsize=cfg.annotation_size, bbox=dict(facecolor="white", edgecolor=PLOS["light"], lw=0.6, pad=2.5))
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Train fraction"); ax.set_ylabel("Held-out fraction")
    ax.set_title("Held-out composition vs train", pad=8)
    beautify(ax)
    ax.legend(loc="lower right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8d(ax, df, cfg):
    metrics = [("mean_absolute_deviation", "Mean absolute\nΔ fraction"), ("max_absolute_deviation", "Maximum absolute\nΔ fraction")]
    x = np.arange(len(metrics)); width = 0.32; ymax = 0.001
    for split, offset in [("val", -width/2), ("test", width/2)]:
        sub = df.loc[df["heldout_split"] == split]
        vals = [float(pd.to_numeric(sub[m], errors="coerce").iloc[0]) if len(sub) else 0.0 for m, _ in metrics]
        ymax = max(ymax, max(vals))
        bars = ax.bar(x + offset, vals, width=width, color=HELDOUT_FACE[split], edgecolor=HELDOUT_EDGE[split],
                      lw=0.65, alpha=0.82, label=SPLIT_DISPLAY[split], zorder=3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + ymax * 0.035, f"{val:.3f}",
                    ha="center", va="bottom", fontsize=cfg.annotation_size)
    ax.set_xticks(x); ax.set_xticklabels([m[1] for m in metrics])
    ax.set_ylabel("Absolute Δ fraction")
    ax.set_title("Composition-deviation summary", pad=8)
    ax.set_ylim(0, ymax * 1.35 if ymax > 0 else 0.01)
    beautify(ax)
    ax.legend(loc="upper right", frameon=True, edgecolor=PLOS["light"], facecolor="white", fontsize=cfg.legend_size)


def plot_s8(panels, summary, cfg):
    set_style(cfg)
    fig, axes = plt.subplots(2, 2, figsize=(11.1, 6.5))
    fig.subplots_adjust(left=0.082, right=0.985, top=0.915, bottom=0.105, wspace=0.31, hspace=0.45)
    plot_s8a(axes[0,0], panels["A"], cfg); panel_letter(axes[0,0], "A", cfg)
    plot_s8b(axes[0,1], panels["B"], cfg); panel_letter(axes[0,1], "B", cfg)
    plot_s8c(axes[1,0], panels["C"], summary, cfg); panel_letter(axes[1,0], "C", cfg)
    plot_s8d(axes[1,1], panels["D"], cfg); panel_letter(axes[1,1], "D", cfg)
    return save_fig(fig, cfg.supp_dir / "Supplementary_Figure_S8_redesigned", cfg)


# ------------------------- reproducibility outputs ---------------------------

def save_reports(cfg, length_audit, token_audit, pad_summary, aa_comp, dev, summary,
                 cap_filtering_audit=None, unknown_token_audit=None, noncanonical_residue_audit=None):
    reports = {
        "sequence_length_audit": length_audit,
        "tokenization_audit": token_audit,
        "padding_burden_summary": pad_summary,
        "amino_acid_composition_summary": aa_comp,
        "composition_deviation_audit": dev,
        "composition_similarity_summary": summary,
    }
    if cap_filtering_audit is not None:
        reports["cap_filtering_audit"] = cap_filtering_audit
    if unknown_token_audit is not None:
        reports["unknown_token_audit"] = unknown_token_audit
    if noncanonical_residue_audit is not None:
        reports["noncanonical_residue_audit"] = noncanonical_residue_audit
    for name, df in reports.items():
        save_csv(df, cfg.reports_dir / f"{name}.csv")
    return reports


def get_git_commit(cfg: Config) -> Optional[str]:
    if cfg.git_commit:
        return cfg.git_commit
    env = os.environ.get("GIT_COMMIT") or os.environ.get("ONCOPEP_GIT_COMMIT")
    if env:
        return env
    try:
        res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(cfg.project_root),
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=5)
        return res.stdout.strip() if res.returncode == 0 else None
    except Exception:
        return None


def code_snapshot(cfg: Config) -> Tuple[Optional[str], str]:
    fname = globals().get("__file__", None)
    if fname and Path(fname).exists():
        script_path = str(Path(fname).resolve())
        text = Path(fname).read_text(encoding="utf-8")
    else:
        script_path = None
        try:
            text = inspect.getsource(sys.modules[__name__])
        except Exception:
            text = "# Source unavailable from interactive session.\n"
    script_hash = sha256_text(text)
    save_text(text, cfg.code_dir / "OncoPep_step_04_PLOS_redesign_code.py")
    save_text(text, cfg.output_root / "OncoPep_step_04_PLOS_redesign_code.py")
    return script_path, script_hash


def write_no_main_readme(cfg: Config) -> Optional[Path]:
    if not cfg.create_no_main_figure_readme:
        return None
    path = cfg.main_figure_dir / "README_no_main_figure_for_step_04.txt"
    save_text(f"""# No main figure generated for OncoPep Step 4

Generated: {now_iso()}

Step 4 is a supplementary-only frozen preprocessing and tokenization audit.

Generated figures:
- Supplementary Figure S7
- Supplementary Figure S8

No Main Figure 2 is generated here. Main Figure 2 is reserved for the OncoPep
architecture and conditional generation workflow.
""", path)
    return path


def write_readme(cfg: Config, input_paths: Dict[str, Path], warnings: Sequence[str]) -> Path:
    path = cfg.reports_dir / "README_step_04_outputs.txt"
    save_text(f"""# OncoPep Step 4 PLOS supplementary-figure package

Generated: {now_iso()}

Scope:
Frozen sequence preprocessing, tokenization, sequence-length, padding, and
amino-acid composition audit.

No main figure is generated in Step 4. Main Figure 2 is reserved for the
OncoPep architecture/model workflow.

Input files:
{chr(10).join(f"- {k}: {v}" for k, v in input_paths.items())}

Generated figures:
- supplementary_figures/Supplementary_Figure_S7_redesigned.png/.pdf/.tiff
- supplementary_figures/Supplementary_Figure_S8_redesigned.png/.pdf/.tiff

Scientific notes:
- Step 4 does not repeat Step 2 leakage audits or Step 3 conditioning-schema audits.
- Preprocessing QC supports model-input transparency, not biological validation.
- Unknown-token fraction is calculated among non-padding tokens and is retained in CSV audit reports rather than plotted as a separate all-zero panel.
- Padding fraction is the fraction of final model input positions occupied by padding; if inferred rather than read directly, this is recorded in source data and the manifest.
- Amino-acid composition deviation is computed relative to the training partition.
- Cap/filtering, split-level preprocessing, and noncanonical-residue audit tables are exported for reviewer inspection.
- Truncation, unknown-token, and split-level preprocessing audits are retained in CSV reports rather than shown as additional S7 panels.

Warnings:
{chr(10).join("- " + str(w) for w in warnings) if warnings else "- None"}
""", path)
    return path


def write_requirements(cfg: Config) -> Path:
    path = cfg.reports_dir / "requirements_step_04_minimal.txt"
    save_text("\n".join([f"python=={platform.python_version()}",
                         f"numpy=={np.__version__}",
                         f"pandas=={pd.__version__}",
                         f"matplotlib=={mpl.__version__}", ""]), path)
    return path


def root_copies(cfg: Config) -> list[str]:
    if not cfg.create_root_level_copies:
        return []
    pairs = []
    for fig in ["S7", "S8"]:
        stem = f"Supplementary_Figure_{fig}_redesigned"
        for ext in [".png", ".pdf", ".tiff"]:
            pairs.append((cfg.supp_dir / f"{stem}{ext}", cfg.output_root / f"{stem}{ext}"))
        pairs.append((cfg.source_data_dir / f"Supplementary_Figure_{fig}_source_data_all_panels.csv",
                      cfg.output_root / f"Supplementary_Figure_{fig}_source_data_all_panels.csv"))
    for name in ["sequence_length_audit.csv", "tokenization_audit.csv", "unknown_token_audit.csv",
                 "padding_burden_summary.csv", "cap_filtering_audit.csv",
                 "amino_acid_composition_summary.csv", "composition_deviation_audit.csv",
                 "composition_similarity_summary.csv", "noncanonical_residue_audit.csv",
                 "README_step_04_outputs.txt"]:
        pairs.append((cfg.reports_dir / name, cfg.output_root / name))
    pairs.append((cfg.code_dir / "OncoPep_step_04_PLOS_redesign_code.py", cfg.output_root / "OncoPep_step_04_PLOS_redesign_code.py"))
    copied = []
    for src, dst in pairs:
        if src.exists():
            shutil.copy2(src, dst); copied.append(str(dst))
    return copied


def build_manifest(cfg, input_paths, s7_paths, s8_paths, readme_path, req_path, no_main_path, script_path, script_hash, copies, warnings,
                   length_audit=None, token_audit=None, noncanonical_residue_audit=None, padding_fraction_source=None):
    preprocessing_parameters = {}
    if length_audit is not None and len(length_audit):
        preprocessing_parameters["chosen_max_sequence_length"] = int(length_audit["chosen_max_sequence_length"].dropna().iloc[0])
        preprocessing_parameters["max_model_length_including_special_tokens"] = int(length_audit["max_model_length_including_special_tokens"].dropna().iloc[0])
    preprocessing_parameters["canonical_amino_acid_vocabulary"] = "".join(cfg.aa_order)
    preprocessing_parameters["padding_fraction_source"] = padding_fraction_source
    if token_audit is not None and len(token_audit):
        preprocessing_parameters["fraction_truncated_sources"] = token_audit.set_index("split")["fraction_truncated_source"].to_dict() if "fraction_truncated_source" in token_audit.columns else {}
        preprocessing_parameters["unknown_token_fraction_sources"] = token_audit.set_index("split")["unknown_token_fraction_nonpad_source"].to_dict() if "unknown_token_fraction_nonpad_source" in token_audit.columns else {}
    if noncanonical_residue_audit is not None and len(noncanonical_residue_audit):
        preprocessing_parameters["noncanonical_residue_fraction_by_split"] = noncanonical_residue_audit.set_index("split")["noncanonical_residue_fraction"].to_dict()
        preprocessing_parameters["noncanonical_symbols_by_split"] = noncanonical_residue_audit.set_index("split")["noncanonical_symbols"].to_dict()

    return {
        "step": "step_04",
        "scope": "supplementary_only",
        "timestamp": now_iso(),
        "main_figure_generated": False,
        "main_figure_rationale": "Step 4 audits frozen preprocessing/tokenization; Main Figure 2 is reserved for OncoPep architecture/model workflow.",
        "project_root": str(cfg.project_root),
        "output_root": str(cfg.output_root),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "packages": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": mpl.__version__},
        "git_commit": get_git_commit(cfg),
        "preprocessing_parameters": preprocessing_parameters,
        "input_files": {k: str(v) for k, v in input_paths.items()},
        "input_sha256": {k: sha256_file(v) for k, v in input_paths.items()},
        "figures": {"S7": s7_paths, "S8": s8_paths},
        "source_data": {
            "S7_all": str(cfg.source_data_dir / "Supplementary_Figure_S7_source_data_all_panels.csv"),
            "S8_all": str(cfg.source_data_dir / "Supplementary_Figure_S8_source_data_all_panels.csv"),
            **{f"S7_panel_{p.lower()}": str(cfg.source_data_dir / f"Supplementary_Figure_S7_panel_{p.lower()}_source_data.csv") for p in "AB"},
            **{f"S8_panel_{p.lower()}": str(cfg.source_data_dir / f"Supplementary_Figure_S8_panel_{p.lower()}_source_data.csv") for p in "ABCD"},
        },
        "reports": {
            "sequence_length_audit": str(cfg.reports_dir / "sequence_length_audit.csv"),
            "tokenization_audit": str(cfg.reports_dir / "tokenization_audit.csv"),
            "unknown_token_audit": str(cfg.reports_dir / "unknown_token_audit.csv"),
            "padding_burden_summary": str(cfg.reports_dir / "padding_burden_summary.csv"),
            "cap_filtering_audit": str(cfg.reports_dir / "cap_filtering_audit.csv"),
            "amino_acid_composition_summary": str(cfg.reports_dir / "amino_acid_composition_summary.csv"),
            "composition_deviation_audit": str(cfg.reports_dir / "composition_deviation_audit.csv"),
            "composition_similarity_summary": str(cfg.reports_dir / "composition_similarity_summary.csv"),
            "noncanonical_residue_audit": str(cfg.reports_dir / "noncanonical_residue_audit.csv"),
            "readme": str(readme_path),
            "requirements": str(req_path),
            "no_main_figure_readme": str(no_main_path) if no_main_path else None,
        },
        "script_path": script_path,
        "script_sha256": script_hash,
        "root_level_copies": list(copies),
        "warnings": list(warnings),
        "config": asdict(cfg),
    }


# ----------------------------- main workflow ---------------------------------

def run_step4_redesign(cfg: Config) -> Dict[str, Any]:
    ensure_dirs(cfg); set_style(cfg)
    warnings = []

    model_ready, length_summary, token_summary, qc_summary, input_paths = load_inputs(cfg)
    model_ready, length_summary, token_summary, qc_summary, seq_col, split_col, padding_fraction_source = harmonize_inputs(
        model_ready, length_summary, token_summary, qc_summary, cfg, warnings
    )

    length_audit = build_length_audit(model_ready, length_summary, split_col, cfg)
    token_audit = build_token_audit(model_ready, token_summary, qc_summary, split_col, cfg, warnings, padding_fraction_source)
    padding_summary = build_padding_summary(model_ready, split_col, cfg)
    cap_filtering_audit = build_cap_filtering_audit(model_ready, length_audit, split_col, cfg)
    aa_comp = build_aa_comp(model_ready, seq_col, split_col, cfg)
    dev, comp_summary = build_comp_dev(aa_comp, cfg)
    unknown_audit = build_unknown_token_audit(token_audit)
    noncanonical_audit = build_noncanonical_residue_audit(model_ready, seq_col, split_col, cfg)

    if (token_audit["fraction_truncated"].fillna(0) > 0).any():
        warnings.append("Non-zero truncation fraction detected; interpret preprocessing as quantified, not artifact-free.")
    if (token_audit["unknown_token_fraction_nonpad"].fillna(0) > 0).any():
        warnings.append("Non-zero unknown-token fraction detected; inspect tokenization_audit.csv.")
    if (padding_summary["mean_padding_fraction"].fillna(0) > 0.75).any():
        warnings.append("High mean padding fraction detected in at least one split.")

    s7_panels, s7_all = build_s7_source(model_ready, length_audit, token_audit, split_col, cfg)
    s8_panels, s8_all = build_s8_source(aa_comp, dev, comp_summary, cfg)
    save_sources(cfg, s7_panels, s7_all, s8_panels, s8_all)
    reports = save_reports(
        cfg, length_audit, token_audit, padding_summary, aa_comp, dev, comp_summary,
        cap_filtering_audit=cap_filtering_audit,
        unknown_token_audit=unknown_audit,
        noncanonical_residue_audit=noncanonical_audit,
    )

    s7_paths = plot_s7(s7_panels, cfg)
    s8_paths = plot_s8(s8_panels, comp_summary, cfg)

    script_path, script_hash = code_snapshot(cfg)
    no_main = write_no_main_readme(cfg)
    readme = write_readme(cfg, input_paths, warnings)
    req = write_requirements(cfg)
    copies = root_copies(cfg)

    manifest = build_manifest(
        cfg, input_paths, s7_paths, s8_paths, readme, req, no_main, script_path, script_hash, copies, warnings,
        length_audit=length_audit,
        token_audit=token_audit,
        noncanonical_residue_audit=noncanonical_audit,
        padding_fraction_source=padding_fraction_source,
    )
    save_json(manifest, cfg.reports_dir / "step_04_manifest.json")
    save_json(manifest, cfg.output_root / "step_04_manifest.json")

    print("\n✅ Step 4 PLOS supplementary-figure package generated successfully.")
    print("Scope: supplementary-only frozen preprocessing/tokenization audit")
    print("Main figure generated: No")
    print(f"Output directory: {cfg.output_root}")

    print("\nInput files:")
    for k, v in input_paths.items():
        print(f"  - {k}: {v}")

    for label, paths in [("Supplementary Figure S7", s7_paths), ("Supplementary Figure S8", s8_paths)]:
        print(f"\n{label}:")
        for fmt, path in paths.items():
            print(f"  {fmt.upper()}: {path}")

    print("\nSource data:")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S7_source_data_all_panels.csv'}")
    for p in "ab":
        print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_S7_panel_{p}_source_data.csv'}")
    print(f"  - {cfg.source_data_dir / 'Supplementary_Figure_S8_source_data_all_panels.csv'}")
    for p in "abcd":
        print(f"  - {cfg.source_data_dir / f'Supplementary_Figure_S8_panel_{p}_source_data.csv'}")

    print("\nReports:")
    for name in ["sequence_length_audit.csv", "tokenization_audit.csv", "unknown_token_audit.csv",
                 "padding_burden_summary.csv", "cap_filtering_audit.csv",
                 "amino_acid_composition_summary.csv", "composition_deviation_audit.csv",
                 "composition_similarity_summary.csv", "noncanonical_residue_audit.csv",
                 "step_04_manifest.json", "README_step_04_outputs.txt"]:
        print(f"  - {cfg.reports_dir / name}")
    if no_main:
        print(f"  - {no_main}")
    if warnings:
        print("\nWarnings:")
        for w in warnings:
            print(f"  - {w}")

    return {
        "model_ready": model_ready,
        "length_summary": length_summary,
        "token_summary": token_summary,
        "qc_summary": qc_summary,
        "length_audit": length_audit,
        "tokenization_audit": token_audit,
        "padding_summary": padding_summary,
        "amino_acid_composition": aa_comp,
        "composition_deviation_audit": dev,
        "composition_similarity_summary": comp_summary,
        "cap_filtering_audit": cap_filtering_audit,
        "unknown_token_audit": unknown_audit,
        "noncanonical_residue_audit": noncanonical_audit,
        "s7_panels": s7_panels,
        "s8_panels": s8_panels,
        "reports": reports,
        "manifest": manifest,
        "warnings": warnings,
    }


def main_notebook(
    project_root: str | Path = PROJECT_ROOT_DEFAULT,
    base_dir: str | Path = BASE_DIR_DEFAULT,
    output_root_name: str = OUTPUT_ROOT_NAME_DEFAULT,
    model_ready_file: Optional[str | Path] = None,
    length_summary_file: Optional[str | Path] = None,
    token_summary_file: Optional[str | Path] = None,
    qc_summary_file: Optional[str | Path] = None,
    create_root_level_copies: bool = True,
    create_no_main_figure_readme: bool = True,
    git_commit: Optional[str] = None,
) -> Dict[str, Any]:
    cfg = Config(
        project_root=Path(project_root),
        base_dir=Path(base_dir),
        output_root_name=output_root_name,
        model_ready_file=Path(model_ready_file) if model_ready_file else None,
        length_summary_file=Path(length_summary_file) if length_summary_file else None,
        token_summary_file=Path(token_summary_file) if token_summary_file else None,
        qc_summary_file=Path(qc_summary_file) if qc_summary_file else None,
        create_root_level_copies=create_root_level_copies,
        create_no_main_figure_readme=create_no_main_figure_readme,
        git_commit=git_commit,
    )
    return run_step4_redesign(cfg)


def running_inside_jupyter() -> bool:
    """Return True when the script is being executed inside a Jupyter kernel.

    In notebooks, __name__ can still be "__main__" when a full script is pasted
    into a cell. This guard prevents the command-line block from auto-running
    with default paths before the user can call main_notebook().
    """
    return "ipykernel" in sys.modules or any("jupyter" in arg.lower() for arg in sys.argv)

def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 4 PLOS supplementary figures S7/S8.")
    parser.add_argument("--project_root", default=str(PROJECT_ROOT_DEFAULT))
    parser.add_argument("--base_dir", default=str(BASE_DIR_DEFAULT))
    parser.add_argument("--output_root_name", default=OUTPUT_ROOT_NAME_DEFAULT)
    parser.add_argument("--model_ready_file", default=None)
    parser.add_argument("--length_summary_file", default=None)
    parser.add_argument("--token_summary_file", default=None)
    parser.add_argument("--qc_summary_file", default=None)
    parser.add_argument("--git_commit", default=None)
    parser.add_argument("--no_root_copies", action="store_true")
    parser.add_argument("--no_main_readme", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignored non-script arguments: {unknown}")
    return args


if __name__ == "__main__":
    if running_inside_jupyter():
        print(
            "\nLoaded OncoPep Step 4 functions in a Jupyter kernel.\n"
            "The command-line block was not auto-run to avoid using default paths.\n\n"
            "Now run:\n\n"
            "results = main_notebook()\n"
        )
    else:
        args = parse_args()
        cfg = Config(
            project_root=Path(args.project_root),
            base_dir=Path(args.base_dir),
            output_root_name=args.output_root_name,
            model_ready_file=Path(args.model_ready_file) if args.model_ready_file else None,
            length_summary_file=Path(args.length_summary_file) if args.length_summary_file else None,
            token_summary_file=Path(args.token_summary_file) if args.token_summary_file else None,
            qc_summary_file=Path(args.qc_summary_file) if args.qc_summary_file else None,
            create_root_level_copies=not args.no_root_copies,
            create_no_main_figure_readme=not args.no_main_readme,
            git_commit=args.git_commit,
        )
        run_step4_redesign(cfg)

In [ ]:
results = main_notebook()

OncoPep Step 5 PLOS Computational Biology redesign code v2.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 5 PLOS Computational Biology redesign code — v7 compact old-style/no-S10.

Approved v7 compact redesign:
- Figure 5A: compact grouped bar benchmark for generation-quality metrics only.
  * Validity, uniqueness, and exact novelty are shown with confidence intervals.
  * Condition fidelity and NN similarity are not duplicated in Panel A; they are shown in Panels C and B.
- Figure 5B: violin + embedded boxplot for candidate-level nearest-neighbor similarity.
- Figure 5C: boxplot + light jitter for per-condition surrogate condition-hit-rate heterogeneity.
- Supplementary Figure S9: condition-support context and descriptor recoverability.
- Supplementary Figure S10: removed as a redundant figure; equivalent metrics retained as a report/table.
- No line plots are used anywhere in Step 5.
- Default output root:
  /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_05/

Usage from terminal:
    python OncoPep_step_05_PLOS_redesign_code_v7_compact_oldstyle_noS10.py

Usage from Jupyter:
    from OncoPep_step_05_PLOS_redesign_code_v7_compact_oldstyle_noS10 import run_step5_plos_redesign
    outputs = run_step5_plos_redesign()

Scientific interpretation limits:
- Computational generator benchmark only.
- Surrogate condition-hit rate is not anticancer activity.
- Nearest-neighbor similarity is a memorization-risk audit, not proof of complete absence of memorization.
- Exact novelty is not sufficient alone to prove full non-memorization.
- Retrieval is a reference/control, not a generator.
"""

from __future__ import annotations

import hashlib
import json
import platform
import shutil
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Patch


# =============================================================================
# PLOS style and model metadata
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

MODEL_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
    "retrieval_reference",
]

GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "cvae_conditional": "OncoPep",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
    "retrieval_reference": "Retrieval\nreference",
}

MODEL_LABELS_ONE_LINE = {
    "cvae_conditional": "OncoPep",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
    "retrieval_reference": "Retrieval reference",
}

MODEL_ROLE = {
    "cvae_conditional": "Primary conditional generator",
    "gru_conditional": "Learned conditional baseline",
    "random_conditional": "Random conditional control",
    "gru_unconditional": "Unconditional sequence baseline",
    "vae_unconditional": "Unconditional latent baseline",
    "retrieval_reference": "Reference/control, not a generator",
}

MODEL_COLORS = {
    "cvae_conditional": PLOS["coral"],
    "gru_conditional": PLOS["blue"],
    "random_conditional": PLOS["mint"],
    "gru_unconditional": PLOS["brown"],
    "vae_unconditional": PLOS["charcoal"],
    "retrieval_reference": PLOS["light"],
}

MODEL_EDGES = {
    "cvae_conditional": "#B65649",
    "gru_conditional": "#176F89",
    "random_conditional": "#79AC86",
    "gru_unconditional": "#8B5F3A",
    "vae_unconditional": "#4E4548",
    "retrieval_reference": "#8A8A8A",
}

METRIC_INFO = {
    "valid_rate": {
        "label": "Validity",
        "axis": "Fraction",
        "direction": "higher_better",
        "low": "valid_rate_ci_low",
        "high": "valid_rate_ci_high",
        "group": "Generation quality",
        "format": "{:.2f}",
    },
    "unique_rate_among_valid": {
        "label": "Uniqueness",
        "axis": "Fraction",
        "direction": "higher_better",
        "low": "unique_rate_among_valid_ci_low",
        "high": "unique_rate_among_valid_ci_high",
        "group": "Generation quality",
        "format": "{:.2f}",
    },
    "exact_novelty_vs_train": {
        "label": "Exact novelty",
        "axis": "Fraction",
        "direction": "higher_better",
        "low": "exact_novelty_vs_train_ci_low",
        "high": "exact_novelty_vs_train_ci_high",
        "group": "Generation quality",
        "format": "{:.2f}",
    },
    "surrogate_condition_hit_rate": {
        "label": "Condition fidelity",
        "axis": "Condition-hit fraction",
        "direction": "higher_better",
        "low": "surrogate_condition_hit_rate_ci_low",
        "high": "surrogate_condition_hit_rate_ci_high",
        "group": "Condition fidelity",
        "format": "{:.2f}",
    },
    "mean_nn_similarity_to_train": {
        "label": "Mean NN similarity",
        "axis": "Mean NN similarity",
        "direction": "lower_better",
        "low": "mean_nn_similarity_to_train_ci_low",
        "high": "mean_nn_similarity_to_train_ci_high",
        "group": "Memorization-risk summary",
        "format": "{:.2f}",
    },
}

PANEL_A_METRIC_ORDER = [
    "valid_rate",
    "unique_rate_among_valid",
    "exact_novelty_vs_train",
    "surrogate_condition_hit_rate",
    "mean_nn_similarity_to_train",
]

COUNT_COLUMNS = [
    "final_retained_peptides",
    "n_final_retained",
    "n_retained",
    "retained_candidates",
    "n_valid",
    "n_valid_candidates",
    "n_generated",
]


@dataclass
class Step5Config:
    project_root: Path = Path("/home/data3/Moe/nature_computational_peponco")
    step5_root: Optional[Path] = None
    output_root: Optional[Path] = None
    script_path: Optional[Path] = None

    dpi_png: int = 300
    dpi_tiff: int = 600
    bootstrap_n: int = 5000
    random_seed: int = 42

    figure5_name: str = "Figure_5_redesigned"
    supp9_name: str = "Supplementary_Figure_S9_redesigned"

    max_support_conditions_to_plot: int = 40
    low_support_quantile: float = 0.25
    max_jitter_points_per_model: int = 700

    def resolved_step5_root(self) -> Path:
        if self.step5_root is not None:
            return Path(self.step5_root)
        candidates = [
            self.project_root / "step5",
            self.project_root / "step_05",
            self.project_root / "results" / "step5",
            self.project_root / "results" / "step_05",
        ]
        for p in candidates:
            if p.exists():
                return p
        return self.project_root / "step5"

    def resolved_output_root(self) -> Path:
        if self.output_root is not None:
            return Path(self.output_root)
        return self.project_root / "PLOS" / "plos_comp" / "step_05"


# =============================================================================
# General utilities
# =============================================================================

def apply_plos_style() -> None:
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 8.3,
        "axes.titlesize": 9.0,
        "axes.labelsize": 8.3,
        "xtick.labelsize": 7.4,
        "ytick.labelsize": 7.4,
        "legend.fontsize": 7.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.80,
        "xtick.major.width": 0.80,
        "ytick.major.width": 0.80,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
    })


def safe_mkdir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def setup_output_dirs(cfg: Step5Config) -> Dict[str, Path]:
    root = safe_mkdir(cfg.resolved_output_root())
    return {
        "root": root,
        "main": safe_mkdir(root / "main_figure"),
        "supp": safe_mkdir(root / "supplementary_figures"),
        "source": safe_mkdir(root / "source_data"),
        "reports": safe_mkdir(root / "reports"),
        "code": safe_mkdir(root / "code"),
    }


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.65)
        ax.yaxis.grid(False)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.65)
        ax.xaxis.grid(False)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.65)
    else:
        ax.grid(False)

    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#666666")
        ax.spines[side].set_linewidth(0.80)
    ax.tick_params(colors=PLOS["dark"], width=0.80, length=3.0)


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.08, y: float = 1.04) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=13,
        fontweight="bold",
        color=PLOS["dark"],
    )


def export_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def save_figure(fig: plt.Figure, base: Path, cfg: Step5Config) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    pdf = base.with_suffix(".pdf")
    fig.savefig(pdf, bbox_inches="tight")
    outputs["pdf"] = str(pdf)

    png = base.with_suffix(".png")
    fig.savefig(png, bbox_inches="tight", dpi=cfg.dpi_png)
    outputs["png"] = str(png)

    tiff = base.with_suffix(".tiff")
    fig.savefig(tiff, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
    outputs["tiff"] = str(tiff)

    plt.close(fig)
    return outputs


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    path = Path(path)
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def is_likely_output_artifact(path: Path) -> bool:
    """Return True for files that are likely generated Step 5 outputs.

    This avoids accidentally reading previous PLOS exports without excluding every
    path that merely contains a folder named ``PLOS``. Valid inputs stored under a
    PLOS project folder are therefore still discoverable unless they are inside
    plos_comp/step_05 output-style subdirectories.
    """
    parts = set(path.parts)
    output_dirs = {"main_figure", "supplementary_figures", "source_data", "reports", "code"}
    if "plos_comp" in parts and output_dirs.intersection(parts):
        return True
    if "plos_comp" in parts and "step_05" in parts:
        return True
    return False


def find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    """Find the first plausible input file using exact patterns before wildcards.

    Exact filenames are searched before broad wildcards by the caller's pattern
    order. Within each pattern, output artifacts are ignored and candidates are
    ranked by short path length and recency to reduce stale-file selection.
    """
    for pattern in patterns:
        candidates = []
        for path in root.rglob(pattern):
            if not path.is_file():
                continue
            if is_likely_output_artifact(path):
                continue
            candidates.append(path)
        if candidates:
            # Prefer concise canonical paths, then the most recently modified file.
            candidates = sorted(candidates, key=lambda p: (len(p.parts), -p.stat().st_mtime, str(p)))
            return candidates[0]
    return None

def discover_inputs(step5_root: Path) -> Dict[str, Optional[Path]]:
    return {
        "generator_benchmark": find_first(step5_root, [
            "table_5_2_step5b_generator_benchmark.csv",
            "tables/table_5_2_step5b_generator_benchmark.csv",
            "*generator_benchmark*.csv",
        ]),
        "candidate_metrics": find_first(step5_root, [
            "step5b_generated_candidates_evaluated.csv",
            "samples/step5b_generated_candidates_evaluated.csv",
            "*generated_candidates_evaluated*.csv",
            "*candidate*metrics*.csv",
        ]),
        "per_condition_metrics": find_first(step5_root, [
            "table_s5_4_step5b_per_condition_metrics.csv",
            "tables_supplementary/table_s5_4_step5b_per_condition_metrics.csv",
            "*per_condition*metrics*.csv",
            "*condition*hit*rate*.csv",
        ]),
        "condition_support_lookup": find_first(step5_root, [
            "table_s5a_condition_support_lookup_v9.csv",
            "table_s5a_condition_support_lookup_v8.csv",
            "table_s5a_condition_support_lookup_v7.csv",
            "*condition_support_lookup*.csv",
            "*condition*support*.csv",
        ]),
        "permutation_importance": find_first(step5_root, [
            "step5a_permutation_importance.csv",
            "*permutation*importance*.csv",
            "*descriptor_importance*.csv",
            "*recoverability*.csv",
        ]),
        "descriptor_recoverability_fallback": find_first(step5_root, [
            "main_panel_d_descriptor_recoverability_v10.csv",
            "main_panel_d_descriptor_recoverability.csv",
            "*descriptor_recoverability*.csv",
        ]),
    }


def read_csv_required(path: Optional[Path], name: str) -> pd.DataFrame:
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"Required input not found for {name}: {path}")
    return pd.read_csv(path, low_memory=False)


def require_columns(df: pd.DataFrame, required: Sequence[str], table_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def strict_numeric(df: pd.DataFrame, cols: Sequence[str], table_name: str) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        if col not in out.columns:
            raise ValueError(f"{table_name} is missing required numeric column: {col}")
        out[col] = pd.to_numeric(out[col], errors="coerce")
    bad = {col: int(out[col].isna().sum()) for col in cols if out[col].isna().any()}
    if bad:
        raise ValueError(f"{table_name} contains missing/non-numeric required values after conversion: {bad}")
    return out


def validate_expected_models(df: pd.DataFrame, expected: Sequence[str], label: str, generator_col: str = "generator_key") -> None:
    observed = set(df[generator_col].astype(str))
    missing = [g for g in expected if g not in observed]
    if missing:
        raise ValueError(f"{label} is missing expected model rows: {missing}")


def clip_ci_record(value: float, low: float, high: float, lo_bound: float = 0.0, hi_bound: float = 1.0) -> Dict[str, float | bool]:
    raw_value = float(value)
    raw_low = float(low)
    raw_high = float(high)

    plot_value = min(max(raw_value, lo_bound), hi_bound)
    plot_low = min(max(raw_low, lo_bound), hi_bound)
    plot_high = min(max(raw_high, lo_bound), hi_bound)

    if plot_low > plot_value:
        plot_low = plot_value
    if plot_high < plot_value:
        plot_high = plot_value

    return {
        "raw_value": raw_value,
        "raw_ci_low": raw_low,
        "raw_ci_high": raw_high,
        "plotted_value": plot_value,
        "plotted_ci_low": plot_low,
        "plotted_ci_high": plot_high,
        "ci_was_clipped": bool(raw_value != plot_value or raw_low != plot_low or raw_high != plot_high),
    }


def bootstrap_ci_mean(values: np.ndarray, n_boot: int = 5000, seed: int = 42) -> Tuple[float, float, float]:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0]), float(vals[0])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
    boot = vals[idx].mean(axis=1)
    return float(vals.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


# =============================================================================
# Data preparation
# =============================================================================

def prep_generator_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    required = ["generator"]
    numeric = []
    for metric_key, info in METRIC_INFO.items():
        required += [metric_key, info["low"], info["high"]]
        numeric += [metric_key, info["low"], info["high"]]
    require_columns(df, required, "generator_benchmark")

    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    if out.empty:
        raise ValueError("generator_benchmark contains no expected generator rows.")
    out = strict_numeric(out, numeric, "generator_benchmark")

    for col in COUNT_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out["generator_key"] = out["generator"].astype(str)
    validate_expected_models(out, MODEL_ORDER, "generator_benchmark")

    out["generator"] = pd.Categorical(out["generator_key"], categories=MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["generator_display"] = out["generator_key"].map(MODEL_LABELS)
    out["generator_display_one_line"] = out["generator_key"].map(MODEL_LABELS_ONE_LINE)
    out["model_role"] = out["generator_key"].map(MODEL_ROLE)
    return out


def prep_candidate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "nn_similarity_to_train"], "candidate_metrics")
    out = df[df["generator"].isin(GENERATOR_ONLY_ORDER)].copy()
    if out.empty:
        raise ValueError("candidate_metrics contains no expected generator rows.")
    out["generator_key"] = out["generator"].astype(str)
    validate_expected_models(out, GENERATOR_ONLY_ORDER, "candidate_metrics")
    out = strict_numeric(out, ["nn_similarity_to_train"], "candidate_metrics")
    out["generator"] = pd.Categorical(out["generator_key"], categories=GENERATOR_ONLY_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["generator_display"] = out["generator_key"].map(MODEL_LABELS)
    out["generator_display_one_line"] = out["generator_key"].map(MODEL_LABELS_ONE_LINE)
    return out


def prep_per_condition_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "requested_condition", "surrogate_condition_hit_rate"], "per_condition_metrics")
    out = df[df["generator"].isin(GENERATOR_ONLY_ORDER)].copy()
    if out.empty:
        raise ValueError("per_condition_metrics contains no expected generator rows.")
    out["generator_key"] = out["generator"].astype(str)
    validate_expected_models(out, GENERATOR_ONLY_ORDER, "per_condition_metrics")
    out = strict_numeric(out, ["surrogate_condition_hit_rate"], "per_condition_metrics")
    out["generator"] = pd.Categorical(out["generator_key"], categories=GENERATOR_ONLY_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["generator_display"] = out["generator_key"].map(MODEL_LABELS)
    out["generator_display_one_line"] = out["generator_key"].map(MODEL_LABELS_ONE_LINE)
    return out


def prep_condition_support(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "condition_id" not in out.columns:
        for col in ["requested_condition", "condition", "condition_label"]:
            if col in out.columns:
                out = out.rename(columns={col: "condition_id"})
                break
    if "support_total" not in out.columns:
        for col in ["total", "n_total", "support", "train_support", "count"]:
            if col in out.columns:
                out = out.rename(columns={col: "support_total"})
                break

    require_columns(out, ["condition_id", "support_total"], "condition_support_lookup")
    out["condition_id"] = out["condition_id"].astype(str)
    out = strict_numeric(out, ["support_total"], "condition_support_lookup")
    out = out.sort_values("support_total", ascending=False).reset_index(drop=True)
    return out


def prep_descriptor_importance(
    df1: Optional[pd.DataFrame],
    df2: Optional[pd.DataFrame],
) -> Tuple[pd.DataFrame, str, pd.DataFrame]:
    """Prepare descriptor recoverability/importance data and audit dropped rows."""
    candidates = []
    if df1 is not None:
        candidates.append(("permutation_importance", df1.copy()))
    if df2 is not None:
        candidates.append(("descriptor_recoverability_fallback", df2.copy()))

    audit_rows = []
    for source_name, cand in candidates:
        feature_col = next((c for c in ["feature", "descriptor", "variable", "block", "metric", "name"] if c in cand.columns), None)
        value_col = next((c for c in ["recoverability", "macro_f1", "importance", "permutation_importance", "value", "score"] if c in cand.columns), None)
        if not feature_col or not value_col:
            audit_rows.append({
                "source_table": source_name,
                "selected": False,
                "reason": "No usable feature/value column pair found.",
                "n_rows_input": int(len(cand)),
                "n_rows_retained": 0,
                "n_rows_dropped_non_numeric_or_missing": int(len(cand)),
                "feature_column": feature_col,
                "value_column": value_col,
            })
            continue

        out = cand[[feature_col, value_col]].copy()
        out.columns = ["feature", "score"]
        out["feature"] = out["feature"].astype(str)
        out["score"] = pd.to_numeric(out["score"], errors="coerce")
        n_input = int(len(out))
        n_missing = int(out["score"].isna().sum())
        out = out.dropna(subset=["score"]).sort_values("score", ascending=False).reset_index(drop=True)
        if out.empty:
            audit_rows.append({
                "source_table": source_name,
                "selected": False,
                "reason": "All descriptor scores were missing/non-numeric.",
                "n_rows_input": n_input,
                "n_rows_retained": 0,
                "n_rows_dropped_non_numeric_or_missing": n_missing,
                "feature_column": feature_col,
                "value_column": value_col,
            })
            continue

        if value_col in ["recoverability", "macro_f1"]:
            label = "Descriptor recoverability"
        elif "importance" in value_col:
            label = "Permutation importance"
        else:
            label = "Descriptor score"

        out["source_table"] = source_name
        out["source_feature_column"] = feature_col
        out["source_value_column"] = value_col
        audit_rows.append({
            "source_table": source_name,
            "selected": True,
            "reason": "Selected as descriptor panel source.",
            "n_rows_input": n_input,
            "n_rows_retained": int(len(out)),
            "n_rows_dropped_non_numeric_or_missing": n_missing,
            "feature_column": feature_col,
            "value_column": value_col,
            "metric_label": label,
        })
        return out, label, pd.DataFrame(audit_rows)

    raise FileNotFoundError(
        "No usable descriptor recoverability/importance table was found. "
        "Expected a feature/descriptor column and a recoverability/macro_f1/importance/score column."
    )

def candidate_count_column(gen: pd.DataFrame) -> Optional[str]:
    for col in COUNT_COLUMNS:
        if col in gen.columns and pd.to_numeric(gen[col], errors="coerce").notna().any():
            return col
    return None


# =============================================================================
# Figure 5A: compact old-style grouped generation-quality benchmark
# =============================================================================

# Panel A intentionally shows only the three generation-quality metrics.
# Condition fidelity and NN similarity are already shown as distributional panels
# in Figure 5C and Figure 5B, respectively, so repeating them in Panel A makes
# the main figure look diagnostic rather than manuscript-ready.
PANEL_A_GENERATION_QUALITY_METRICS = [
    "valid_rate",
    "unique_rate_among_valid",
    "exact_novelty_vs_train",
]


def benchmark_metric_source_data(
    gen: pd.DataFrame,
    metric_order: Optional[Sequence[str]] = None,
    panel: str = "A",
) -> pd.DataFrame:
    """Create benchmark metric source data with raw/plotted values and CI audit fields."""
    ccol = candidate_count_column(gen)
    metrics = list(metric_order) if metric_order is not None else list(METRIC_INFO.keys())

    rows = []
    for _, row in gen.iterrows():
        g = row["generator_key"]
        for metric_key in metrics:
            info = METRIC_INFO[metric_key]
            ci = clip_ci_record(row[metric_key], row[info["low"]], row[info["high"]])
            rows.append({
                "panel": panel,
                "generator": g,
                "generator_display": MODEL_LABELS_ONE_LINE[g],
                "model_role": MODEL_ROLE[g],
                "metric_key": metric_key,
                "metric_label": info["label"],
                "metric_group": info["group"],
                "metric_direction": info["direction"],
                **ci,
                "candidate_count_column": ccol or "not_available",
                "candidate_count": float(row[ccol]) if ccol is not None and pd.notna(row[ccol]) else np.nan,
            })
    return pd.DataFrame(rows)


def panel_a_source_data(gen: pd.DataFrame) -> pd.DataFrame:
    """Source data for Figure 5A only: validity, uniqueness, exact novelty."""
    return benchmark_metric_source_data(
        gen,
        metric_order=PANEL_A_GENERATION_QUALITY_METRICS,
        panel="A",
    )


def draw_generation_quality_grouped_bars(ax: plt.Axes, panel_src: pd.DataFrame) -> None:
    """Draw compact grouped vertical bars with CI for generation-quality metrics."""
    metrics = PANEL_A_GENERATION_QUALITY_METRICS
    metric_labels = [METRIC_INFO[m]["label"] for m in metrics]
    x = np.arange(len(metrics), dtype=float)
    n_models = len(MODEL_ORDER)

    # Bar width chosen to preserve separation between six models per metric.
    bar_w = 0.115
    offsets = (np.arange(n_models) - (n_models - 1) / 2.0) * bar_w

    for i, g in enumerate(MODEL_ORDER):
        vals, lows, highs = [], [], []
        for metric_key in metrics:
            row = panel_src[
                (panel_src["generator"] == g) &
                (panel_src["metric_key"] == metric_key)
            ]
            if row.empty:
                vals.append(np.nan)
                lows.append(np.nan)
                highs.append(np.nan)
                continue
            r = row.iloc[0]
            vals.append(float(r["plotted_value"]))
            lows.append(float(r["plotted_ci_low"]))
            highs.append(float(r["plotted_ci_high"]))

        vals = np.asarray(vals, dtype=float)
        lows = np.asarray(lows, dtype=float)
        highs = np.asarray(highs, dtype=float)

        ax.bar(
            x + offsets[i],
            vals,
            width=bar_w,
            color=MODEL_COLORS[g],
            edgecolor=MODEL_EDGES[g],
            linewidth=0.72,
            alpha=0.94 if g == "cvae_conditional" else 0.86,
            zorder=3,
            label=MODEL_LABELS_ONE_LINE[g],
        )
        ax.errorbar(
            x + offsets[i],
            vals,
            yerr=np.vstack([vals - lows, highs - vals]),
            fmt="none",
            ecolor=MODEL_EDGES[g],
            elinewidth=0.82,
            capsize=1.6,
            capthick=0.82,
            zorder=4,
        )

    ax.set_ylim(0.0, 1.07)
    ax.set_xlim(-0.55, len(metrics) - 0.45)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.set_ylabel("Fraction")
    ax.set_title("Generation quality benchmark", fontweight="bold", pad=8)
    style_axis(ax, "y")
    ax.text(
        0.99, 0.045,
        "higher better",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=7.4,
        color="#666666",
    )


def candidate_count_source_data(gen: pd.DataFrame) -> pd.DataFrame:
    """Return candidate-count source data even when no count column is provided.

    Candidate counts are intentionally not plotted in the compact Figure 5A
    unless the manuscript later needs them; the data remain available for reports.
    """
    ccol = candidate_count_column(gen)
    if ccol is None:
        return pd.DataFrame({
            "panel": ["A"],
            "generator": ["not_available"],
            "generator_display": ["not_available"],
            "candidate_count_column": ["not_available"],
            "candidate_count": [np.nan],
            "candidate_count_available": [False],
            "plotted_in_sidebar": [False],
        })
    data = gen.set_index("generator_key").loc[MODEL_ORDER].reset_index()
    data["candidate_count"] = pd.to_numeric(data[ccol], errors="coerce")
    return pd.DataFrame({
        "panel": "A",
        "generator": data["generator_key"],
        "generator_display": data["generator_key"].map(MODEL_LABELS_ONE_LINE),
        "candidate_count_column": ccol,
        "candidate_count": data["candidate_count"],
        "candidate_count_available": True,
        "plotted_in_sidebar": False,
    })


def plot_panel_a_benchmark(ax: plt.Axes, gen: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Plot compact generation-quality benchmark Panel A."""
    src = panel_a_source_data(gen)
    draw_generation_quality_grouped_bars(ax, src)
    count_df = candidate_count_source_data(gen)
    return src, count_df


# =============================================================================
# Figure 5B: violin + embedded boxplot

# =============================================================================
# Figure 5B: violin + embedded boxplot
# =============================================================================

def plot_panel_b_violin_box(ax: plt.Axes, cand: pd.DataFrame) -> pd.DataFrame:
    arrays = []
    positions = []
    labels = []
    colors = []
    source_rows = []

    for idx, g in enumerate(GENERATOR_ONLY_ORDER, start=1):
        vals = cand.loc[cand["generator_key"] == g, "nn_similarity_to_train"].to_numpy(float)
        arrays.append(vals)
        positions.append(idx)
        labels.append(MODEL_LABELS_ONE_LINE[g])
        colors.append(MODEL_COLORS[g])
        source_rows += [{
            "panel": "B",
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "nn_similarity_to_train": float(v),
        } for v in vals]

    violin = ax.violinplot(
        arrays,
        positions=positions,
        widths=0.68,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, color in zip(violin["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.20)
        body.set_linewidth(0.8)
        body.set_zorder(2)

    bp = ax.boxplot(
        arrays,
        positions=positions,
        widths=0.18,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": PLOS["dark"], "linewidth": 1.15},
        whiskerprops={"color": "#606060", "linewidth": 0.80},
        capprops={"color": "#606060", "linewidth": 0.80},
        boxprops={"linewidth": 0.85},
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(PLOS["white"])
        patch.set_edgecolor(color)
        patch.set_linewidth(0.90)
        patch.set_zorder(4)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("NN similarity to train")
    ax.set_title("Candidate-level memorization-risk audit", fontweight="bold", pad=8)
    ax.set_ylim(0.0, 1.0)
    style_axis(ax, "y")
    return pd.DataFrame(source_rows)


# =============================================================================
# Figure 5C: boxplot + light jitter
# =============================================================================

def plot_panel_c_condition_box_jitter(ax: plt.Axes, per_cond: pd.DataFrame, cfg: Step5Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(cfg.random_seed + 151)
    arrays = []
    positions = []
    labels = []
    colors = []
    raw_rows = []

    for idx, g in enumerate(GENERATOR_ONLY_ORDER, start=1):
        sub = per_cond.loc[per_cond["generator_key"] == g, ["requested_condition", "surrogate_condition_hit_rate"]].copy()
        vals = sub["surrogate_condition_hit_rate"].to_numpy(float)
        arrays.append(vals)
        positions.append(idx)
        labels.append(MODEL_LABELS_ONE_LINE[g])
        colors.append(MODEL_COLORS[g])

        # Use the configurable plotting cap only if condition count grows unusually large.
        plot_indices = np.arange(len(sub))
        if len(plot_indices) > cfg.max_jitter_points_per_model:
            plot_indices = np.sort(rng.choice(plot_indices, size=cfg.max_jitter_points_per_model, replace=False))
        plotted_index_set = set(plot_indices.tolist())

        for local_i, (_, row) in enumerate(sub.iterrows()):
            raw_rows.append({
                "panel": "C",
                "generator": g,
                "generator_display": MODEL_LABELS_ONE_LINE[g],
                "requested_condition": row["requested_condition"],
                "surrogate_condition_hit_rate": float(row["surrogate_condition_hit_rate"]),
                "plotted_in_jitter": bool(local_i in plotted_index_set),
            })

    bp = ax.boxplot(
        arrays,
        positions=positions,
        widths=0.48,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": PLOS["dark"], "linewidth": 1.15},
        whiskerprops={"color": "#606060", "linewidth": 0.80},
        capprops={"color": "#606060", "linewidth": 0.80},
        boxprops={"linewidth": 0.85},
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor(color)
        patch.set_alpha(0.20)

    raw_df_for_plot = pd.DataFrame(raw_rows)
    for x, g, color in zip(positions, GENERATOR_ONLY_ORDER, colors):
        vals_plot = raw_df_for_plot.loc[
            (raw_df_for_plot["generator"] == g) & (raw_df_for_plot["plotted_in_jitter"]),
            "surrogate_condition_hit_rate",
        ].to_numpy(float)
        jitter = np.clip(rng.normal(0, 0.045, len(vals_plot)), -0.10, 0.10)
        ax.scatter(
            np.full(len(vals_plot), x) + jitter,
            vals_plot,
            s=13,
            alpha=0.38,
            color=color,
            edgecolor=PLOS["white"],
            linewidth=0.25,
            zorder=3,
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_ylim(0.0, 1.03)
    ax.set_title("Per-condition fidelity heterogeneity", fontweight="bold", pad=8)
    style_axis(ax, "y")

    raw_df = pd.DataFrame(raw_rows)
    summary_rows = []
    for g, vals in zip(GENERATOR_ONLY_ORDER, arrays):
        summary_rows.append({
            "panel": "C",
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "n_conditions": int(len(vals)),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "q1": float(np.quantile(vals, 0.25)),
            "q3": float(np.quantile(vals, 0.75)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    return summary_df, raw_df


# =============================================================================
# Build Figure 5
# =============================================================================

def build_figure5(gen: pd.DataFrame, cand: pd.DataFrame, per_cond: pd.DataFrame, cfg: Step5Config, dirs: Dict[str, Path]) -> Dict[str, object]:
    """Build compact old-style-inspired Figure 5.

    Layout:
        A. Generation quality benchmark across the full top row.
        B. Candidate-level NN similarity distribution.
        C. Per-condition condition-fidelity heterogeneity.

    This layout intentionally avoids duplicating condition fidelity and NN similarity
    in Panel A; those metrics are shown more informatively as distributional panels.
    """
    fig = plt.figure(figsize=(9.2, 7.35))
    gs = GridSpec(
        2, 2,
        figure=fig,
        height_ratios=[1.05, 1.0],
        width_ratios=[1.0, 1.0],
        hspace=0.46,
        wspace=0.30,
    )

    # Panel A spans the full figure width.
    ax_a = fig.add_subplot(gs[0, :])
    panel_a_df, count_df = plot_panel_a_benchmark(ax_a, gen)
    add_panel_label(ax_a, "A", x=-0.08, y=1.06)

    # Panel B: memorization-risk distribution.
    ax_b = fig.add_subplot(gs[1, 0])
    panel_b_df = plot_panel_b_violin_box(ax_b, cand)
    add_panel_label(ax_b, "B", x=-0.12, y=1.05)

    # Panel C: condition-level heterogeneity.
    ax_c = fig.add_subplot(gs[1, 1])
    panel_c_summary, panel_c_raw = plot_panel_c_condition_box_jitter(ax_c, per_cond, cfg)
    add_panel_label(ax_c, "C", x=-0.12, y=1.05)

    # Compact one-line legend. Retrieval appears because it is included in Panel A only.
    handles = [
        Patch(facecolor=MODEL_COLORS[g], edgecolor=MODEL_EDGES[g], label=MODEL_LABELS_ONE_LINE[g])
        for g in MODEL_ORDER
    ]
    leg = fig.legend(
        handles=handles,
        loc="lower center",
        ncol=6,
        frameon=True,
        fancybox=False,
        bbox_to_anchor=(0.50, 0.005),
        columnspacing=0.72,
        handlelength=1.0,
        handletextpad=0.30,
        borderpad=0.22,
    )
    leg.get_frame().set_facecolor(PLOS["white"])
    leg.get_frame().set_edgecolor("#D0D0D0")
    leg.get_frame().set_linewidth(0.65)

    fig.subplots_adjust(left=0.075, right=0.982, top=0.948, bottom=0.135)
    figure_outputs = save_figure(fig, dirs["main"] / cfg.figure5_name, cfg)

    panel_a_df["figure_panel"] = "Figure 5A"
    panel_b_df["figure_panel"] = "Figure 5B"
    panel_c_summary["figure_panel"] = "Figure 5C"
    panel_c_raw["figure_panel"] = "Figure 5C_condition_level"

    # Main all-panel file contains all plotted/source-supported Figure 5 values,
    # including raw condition-level values for Panel C.
    all_panels = pd.concat(
        [panel_a_df, panel_b_df, panel_c_summary, panel_c_raw],
        ignore_index=True,
        sort=False,
    )

    source = dirs["source"]
    source_outputs = {
        "Figure_5_source_data_all_panels": export_csv(all_panels, source / "Figure_5_source_data_all_panels.csv"),
        "Figure_5_panel_a_source_data": export_csv(panel_a_df, source / "Figure_5_panel_a_source_data.csv"),
        "Figure_5_panel_b_source_data": export_csv(panel_b_df, source / "Figure_5_panel_b_source_data.csv"),
        "Figure_5_panel_c_source_data": export_csv(panel_c_summary, source / "Figure_5_panel_c_source_data.csv"),
        "Figure_5_panel_a_candidate_count_source_data": export_csv(count_df, source / "Figure_5_panel_a_candidate_count_source_data.csv"),
        "Figure_5_panel_c_condition_level_source_data": export_csv(panel_c_raw, source / "Figure_5_panel_c_condition_level_source_data.csv"),
    }

    return {"figure_outputs": figure_outputs, "source_outputs": source_outputs}


# =============================================================================
# Supplementary Figure S9
# =============================================================================

def support_concentration_summary(support: pd.DataFrame) -> pd.DataFrame:
    df = support.sort_values("support_total", ascending=False).reset_index(drop=True)
    total = float(df["support_total"].sum())
    n = len(df)
    top10 = max(1, int(np.ceil(n * 0.10)))
    top25 = max(1, int(np.ceil(n * 0.25)))
    top50 = max(1, int(np.ceil(n * 0.50)))
    bottom50 = n - top50

    specs = [
        ("Top 10%", "cumulative_top_support_fraction", top10, "head"),
        ("Top 25%", "cumulative_top_support_fraction", top25, "head"),
        ("Top 50%", "cumulative_top_support_fraction", top50, "head"),
        ("Bottom 50%", "remaining_lower_support_fraction", bottom50, "tail"),
    ]

    rows = []
    for label, summary_type, k, mode in specs:
        if k <= 0:
            count = 0
        elif mode == "head":
            count = int(df.head(k)["support_total"].sum())
        else:
            count = int(df.tail(k)["support_total"].sum())
        rows.append({
            "panel": "B",
            "support_bin": label,
            "summary_type": summary_type,
            "n_condition_ids": int(k),
            "support_count": count,
            "support_fraction": float(count / total) if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def plot_s9a_ranked_support(ax: plt.Axes, support: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    df = support.sort_values("support_total", ascending=True).reset_index(drop=True)
    original_n = len(df)
    truncated = False
    if original_n > cfg.max_support_conditions_to_plot:
        half = cfg.max_support_conditions_to_plot // 2
        df = pd.concat([df.head(half), df.tail(cfg.max_support_conditions_to_plot - half)], ignore_index=True)
        truncated = True

    low_threshold = support["support_total"].quantile(cfg.low_support_quantile)
    df["low_support_flag"] = df["support_total"] <= low_threshold
    y = np.arange(len(df))
    maxv = float(df["support_total"].max()) if len(df) else 1.0
    colors = np.where(df["low_support_flag"], PLOS["light"], PLOS["mint"])
    edges = np.where(df["low_support_flag"], "#999999", "#79AC86")

    ax.barh(
        y,
        df["support_total"],
        color=colors,
        edgecolor=edges,
        linewidth=0.65,
        height=0.72,
        zorder=3,
    )
    for yi, value in zip(y, df["support_total"]):
        ax.text(value + maxv * 0.012, yi, f"{int(value):,}", va="center", ha="left", fontsize=6.8, color=PLOS["dark"])

    ax.set_yticks(y)
    ax.set_yticklabels(df["condition_id"])
    ax.set_xlabel("Number of training/support sequences")
    ax.set_ylabel("Condition ID")
    title = "Ranked condition support"
    if truncated:
        title += " (top/bottom shown)"
    ax.set_title(title, fontweight="bold", pad=8)
    ax.set_xlim(0, maxv * 1.16)
    style_axis(ax, "x")

    out = df.copy()
    out["panel"] = "A"
    out["original_n_conditions"] = original_n
    out["display_truncated"] = truncated
    out["low_support_threshold_quantile"] = cfg.low_support_quantile
    out["low_support_threshold"] = float(low_threshold)
    return out


def plot_s9b_support_concentration(ax: plt.Axes, support: pd.DataFrame) -> pd.DataFrame:
    conc = support_concentration_summary(support)
    x = np.arange(len(conc))
    colors = [PLOS["blue"], PLOS["blue"], PLOS["blue"], PLOS["light"]]
    edges = ["#176F89", "#176F89", "#176F89", "#999999"]

    ax.bar(
        x,
        conc["support_fraction"],
        width=0.62,
        color=colors,
        edgecolor=edges,
        linewidth=0.75,
        zorder=3,
    )
    for xi, frac, cnt in zip(x, conc["support_fraction"], conc["support_count"]):
        ax.text(xi, frac + 0.026, f"{frac:.2f}\n({cnt:,})", ha="center", va="bottom", fontsize=7.0, color=PLOS["dark"])

    ax.set_xticks(x)
    ax.set_xticklabels(conc["support_bin"])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Fraction of total condition support")
    ax.set_title("Support captured by ranked condition IDs", fontweight="bold", pad=8)
    style_axis(ax, "y")
    return conc


def plot_s9c_descriptor_recoverability(ax: plt.Axes, importance: pd.DataFrame, metric_label: str) -> pd.DataFrame:
    df = importance.copy().head(10).sort_values("score", ascending=True).reset_index(drop=True)
    pretty = {
        "Full Numeric Descriptor Set": "All descriptors",
        "Full numeric descriptor set": "All descriptors",
        "Computable Descriptors": "Computed",
        "Computable descriptors": "Computed",
        "Physicochemical Only": "Physchem",
        "Physicochemical only": "Physchem",
        "AA Composition Only": "AA composition only",
        "Aa Composition Only": "AA composition only",
        "Amino Acid Composition": "AA composition only",
    }
    df["feature_display"] = df["feature"].map(lambda x: pretty.get(str(x), str(x)))
    y = np.arange(len(df))
    maxv = float(df["score"].max()) if len(df) else 1.0

    ax.barh(
        y,
        df["score"],
        height=0.58,
        color=PLOS["coral"],
        edgecolor=MODEL_EDGES["cvae_conditional"],
        linewidth=0.65,
        zorder=3,
    )
    for yi, value in zip(y, df["score"]):
        ax.text(value + maxv * 0.015, yi, f"{value:.2f}", va="center", ha="left", fontsize=7.1, color=PLOS["dark"])

    ax.set_yticks(y)
    ax.set_yticklabels(df["feature_display"])
    ax.set_xlabel(metric_label)
    ax.set_title("Descriptor recoverability", fontweight="bold", pad=8)
    ax.set_xlim(0, max(1.0, maxv * 1.15) if maxv <= 1.0 else maxv * 1.18)
    style_axis(ax, "x")

    out = df.copy()
    out["panel"] = "C"
    out["metric_label"] = metric_label
    return out


def build_supplementary_s9(
    support: pd.DataFrame,
    importance: pd.DataFrame,
    metric_label: str,
    cfg: Step5Config,
    dirs: Dict[str, Path],
) -> Dict[str, object]:
    fig = plt.figure(figsize=(8.4, 8.8))
    gs = GridSpec(
        2, 2,
        figure=fig,
        width_ratios=[1.10, 1.0],
        height_ratios=[1.08, 0.92],
        wspace=0.36,
        hspace=0.43,
    )

    ax_a = fig.add_subplot(gs[:, 0])
    panel_a = plot_s9a_ranked_support(ax_a, support, cfg)
    add_panel_label(ax_a, "A", x=-0.12, y=1.02)

    ax_b = fig.add_subplot(gs[0, 1])
    panel_b = plot_s9b_support_concentration(ax_b, support)
    add_panel_label(ax_b, "B", x=-0.13, y=1.04)

    ax_c = fig.add_subplot(gs[1, 1])
    panel_c = plot_s9c_descriptor_recoverability(ax_c, importance, metric_label)
    add_panel_label(ax_c, "C", x=-0.13, y=1.04)

    fig.subplots_adjust(left=0.12, right=0.985, top=0.955, bottom=0.09)
    figure_outputs = save_figure(fig, dirs["supp"] / cfg.supp9_name, cfg)

    panel_a["figure_panel"] = "Supplementary Figure S9A"
    panel_b["figure_panel"] = "Supplementary Figure S9B"
    panel_c["figure_panel"] = "Supplementary Figure S9C"

    all_panels = pd.concat([panel_a, panel_b, panel_c], ignore_index=True, sort=False)
    source = dirs["source"]
    source_outputs = {
        "Supplementary_Figure_S9_source_data_all_panels": export_csv(all_panels, source / "Supplementary_Figure_S9_source_data_all_panels.csv"),
        "Supplementary_Figure_S9_panel_a_source_data": export_csv(panel_a, source / "Supplementary_Figure_S9_panel_a_source_data.csv"),
        "Supplementary_Figure_S9_panel_b_source_data": export_csv(panel_b, source / "Supplementary_Figure_S9_panel_b_source_data.csv"),
        "Supplementary_Figure_S9_panel_c_source_data": export_csv(panel_c, source / "Supplementary_Figure_S9_panel_c_source_data.csv"),
    }
    return {"figure_outputs": figure_outputs, "source_outputs": source_outputs}


# =============================================================================
# Reports and reproducibility
# =============================================================================

def candidate_similarity_summary(cand: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in GENERATOR_ONLY_ORDER:
        vals = cand.loc[cand["generator_key"] == g, "nn_similarity_to_train"].to_numpy(float)
        rows.append({
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "n_candidates": int(len(vals)),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "q1": float(np.quantile(vals, 0.25)),
            "q3": float(np.quantile(vals, 0.75)),
            "p05": float(np.quantile(vals, 0.05)),
            "p95": float(np.quantile(vals, 0.95)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
    return pd.DataFrame(rows)


def per_condition_hit_summary(per_cond: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    rows = []
    for g in GENERATOR_ONLY_ORDER:
        vals = per_cond.loc[per_cond["generator_key"] == g, "surrogate_condition_hit_rate"].to_numpy(float)
        mean, low, high = bootstrap_ci_mean(vals, cfg.bootstrap_n, cfg.random_seed)
        rows.append({
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "n_conditions": int(len(vals)),
            "mean": mean,
            "mean_ci_low": low,
            "mean_ci_high": high,
            "median": float(np.median(vals)),
            "q1": float(np.quantile(vals, 0.25)),
            "q3": float(np.quantile(vals, 0.75)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
    return pd.DataFrame(rows)


def figure5_statistical_summary(gen: pd.DataFrame) -> pd.DataFrame:
    return benchmark_metric_source_data(gen, metric_order=list(METRIC_INFO.keys()), panel="report")


def make_reports(
    gen: pd.DataFrame,
    cand: pd.DataFrame,
    per_cond: pd.DataFrame,
    support: pd.DataFrame,
    importance: pd.DataFrame,
    descriptor_audit: pd.DataFrame,
    cfg: Step5Config,
    dirs: Dict[str, Path],
) -> Dict[str, str]:
    reports = dirs["reports"]
    outputs: Dict[str, str] = {}

    outputs["generator_benchmark_summary"] = export_csv(gen, reports / "generator_benchmark_summary.csv")
    outputs["candidate_novelty_similarity_summary"] = export_csv(candidate_similarity_summary(cand), reports / "candidate_novelty_similarity_summary.csv")
    outputs["per_condition_hit_rate_summary"] = export_csv(per_condition_hit_summary(per_cond, cfg), reports / "per_condition_hit_rate_summary.csv")
    outputs["condition_support_summary"] = export_csv(support, reports / "condition_support_summary.csv")
    outputs["descriptor_recoverability_summary"] = export_csv(importance, reports / "descriptor_recoverability_summary.csv")
    outputs["descriptor_recoverability_preparation_audit"] = export_csv(descriptor_audit, reports / "descriptor_recoverability_preparation_audit.csv")

    stat = figure5_statistical_summary(gen)
    outputs["figure5_statistical_summary"] = export_csv(stat, reports / "figure5_statistical_summary.csv")

    removed_s10 = stat.copy()
    removed_s10["note"] = "Supplementary Figure S10 removed as redundant; aggregate benchmark metrics retained in this table."
    outputs["supplementary_table_equivalent_removed_s10"] = export_csv(removed_s10, reports / "supplementary_table_equivalent_removed_s10.csv")

    req = reports / "requirements_step_05_minimal.txt"
    req.write_text(
        f"python=={platform.python_version()}\n"
        f"numpy=={np.__version__}\n"
        f"pandas=={pd.__version__}\n"
        f"matplotlib=={matplotlib.__version__}\n",
        encoding="utf-8",
    )
    outputs["requirements_step_05_minimal"] = str(req)

    return outputs


def get_git_commit(path: Path) -> Optional[str]:
    try:
        res = subprocess.run(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
        )
        return res.stdout.strip()
    except Exception:
        return None


def package_versions() -> Dict[str, str]:
    return {
        "python": sys.version.replace("\n", " "),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
    }


def snapshot_code(cfg: Step5Config, dirs: Dict[str, Path]) -> Tuple[str, Optional[str]]:
    target = dirs["code"] / "OncoPep_step_05_PLOS_redesign_code.py"
    try:
        src = Path(cfg.script_path) if cfg.script_path is not None else Path(__file__).resolve()
        if src.exists():
            shutil.copy2(src, target)
        else:
            raise FileNotFoundError
    except Exception:
        target.write_text(
            "# Code snapshot placeholder.\n"
            "# This run was executed in an interactive environment where the script path was unavailable.\n"
            "# Save the provided v5 script as OncoPep_step_05_PLOS_redesign_code.py for exact reproducibility.\n",
            encoding="utf-8",
        )
    return str(target), sha256_file(target)


def write_readme(
    cfg: Step5Config,
    dirs: Dict[str, Path],
    input_paths: Dict[str, Optional[Path]],
    outputs: Dict[str, object],
    descriptor_metric_label: str,
    descriptor_audit: pd.DataFrame,
) -> str:
    path = dirs["reports"] / "README_step_05_outputs.txt"
    expected_inventory = [
        "main_figure/Figure_5_redesigned.png/pdf/tiff",
        "supplementary_figures/Supplementary_Figure_S9_redesigned.png/pdf/tiff",
        "source_data/Figure_5_source_data_all_panels.csv",
        "source_data/Figure_5_panel_a_source_data.csv",
        "source_data/Figure_5_panel_b_source_data.csv",
        "source_data/Figure_5_panel_c_source_data.csv",
        "source_data/Figure_5_panel_a_candidate_count_source_data.csv",
        "source_data/Figure_5_panel_c_condition_level_source_data.csv",
        "source_data/Supplementary_Figure_S9_source_data_all_panels.csv",
        "source_data/Supplementary_Figure_S9_panel_a_source_data.csv",
        "source_data/Supplementary_Figure_S9_panel_b_source_data.csv",
        "source_data/Supplementary_Figure_S9_panel_c_source_data.csv",
        "reports/generator_benchmark_summary.csv",
        "reports/candidate_novelty_similarity_summary.csv",
        "reports/per_condition_hit_rate_summary.csv",
        "reports/condition_support_summary.csv",
        "reports/descriptor_recoverability_summary.csv",
        "reports/descriptor_recoverability_preparation_audit.csv",
        "reports/figure5_statistical_summary.csv",
        "reports/supplementary_table_equivalent_removed_s10.csv",
        "reports/requirements_step_05_minimal.txt",
        "reports/step_05_manifest.json",
        "reports/README_step_05_outputs.txt",
        "code/OncoPep_step_05_PLOS_redesign_code.py",
    ]
    selected_descriptor_source = "not_available"
    dropped_descriptor_rows = "not_available"
    if descriptor_audit is not None and not descriptor_audit.empty:
        selected = descriptor_audit[descriptor_audit.get("selected", False) == True]
        if not selected.empty:
            selected_descriptor_source = str(selected.iloc[0].get("source_table", "not_available"))
            dropped_descriptor_rows = str(selected.iloc[0].get("n_rows_dropped_non_numeric_or_missing", "not_available"))

    lines = [
        "OncoPep Step 5 PLOS Computational Biology redesign outputs",
        "=" * 72,
        "",
        "Version",
        "-------",
        "v7 compact old-style/no-S10 redesign.",
        "",
        "Scientific scope",
        "----------------",
        "Step 5 evaluates internal generator benchmarking, exact novelty, uniqueness,",
        "nearest-neighbor memorization-risk auditing, and surrogate condition fidelity.",
        "These analyses are computational audits only. They do not establish anticancer",
        "activity, therapeutic relevance, toxicity safety, or clinical utility.",
        "",
        "Design decisions",
        "----------------",
        "1. No line plots are used in Step 5.",
        "2. Figure 5A uses a compact grouped bar benchmark for generation-quality metrics with confidence intervals.",
        "3. Mean NN similarity is not duplicated in Panel A; it is shown as a raw lower-better distribution in Figure 5B.",
        "4. Figure 5B uses violin plots with embedded boxplots; no retrieval line is shown.",
        "5. Figure 5C uses boxplots with light condition-level jitter; retrieval is excluded because it is not a generator.",
        "6. Supplementary Figure S9B uses support-concentration bars, replacing the former line plot.",
        "7. Supplementary Figure S10 was removed because it duplicated Figure 5.",
        "   Equivalent aggregate metrics are retained in reports/supplementary_table_equivalent_removed_s10.csv.",
        "8. Candidate-count sidebar is shown only when candidate-count columns are available.",
        "9. Figure_5_source_data_all_panels.csv includes raw condition-level values from Figure 5C.",
        "",
        "Metric interpretation",
        "---------------------",
        "Validity: fraction of generated sequences passing syntax/peptide-format validity checks.",
        "Uniqueness: unique valid outputs divided by valid generated outputs.",
        "Exact novelty: fraction of valid outputs not exactly present in the training set.",
        "Surrogate condition-hit rate: descriptor-condition agreement, not anticancer activity.",
        "NN similarity to train: nearest-neighbor similarity to the training set; lower values indicate lower detectable near-neighbor similarity.",
        "Exact novelty and NN similarity are memorization-risk audits, not proof that all training influence is eliminated.",
        "Retrieval is a reference/control, not a generator.",
        f"Descriptor panel metric label: {descriptor_metric_label}.",
        f"Descriptor source selected: {selected_descriptor_source}.",
        f"Descriptor rows dropped as missing/non-numeric: {dropped_descriptor_rows}.",
        "",
        "Model roles",
        "-----------",
    ]
    for g in MODEL_ORDER:
        lines.append(f"{g}: {MODEL_LABELS_ONE_LINE[g]} — {MODEL_ROLE[g]}")
    lines += [
        "",
        "Expected output inventory",
        "-------------------------",
    ]
    lines += [f"- {item}" for item in expected_inventory]
    lines += [
        "",
        "Detected input files",
        "--------------------",
    ]
    for key, value in input_paths.items():
        lines.append(f"{key}: {value}")
    lines += [
        "",
        "Output root",
        "-----------",
        str(dirs["root"]),
        "",
        "Generated at",
        "------------",
        datetime.now().isoformat(timespec="seconds"),
    ]
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path)


def write_manifest(
    cfg: Step5Config,
    dirs: Dict[str, Path],
    input_paths: Dict[str, Optional[Path]],
    outputs: Dict[str, object],
    code_hash: Optional[str],
) -> str:
    manifest = {
        "task": "OncoPep Step 5 PLOS redesign v7 compact old-style/no-S10",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "config": {
            "project_root": str(cfg.project_root),
            "step5_root": str(cfg.resolved_step5_root()),
            "output_root": str(cfg.resolved_output_root()),
            "bootstrap_n": cfg.bootstrap_n,
            "random_seed": cfg.random_seed,
            "max_jitter_points_per_model": cfg.max_jitter_points_per_model,
            "max_support_conditions_to_plot": cfg.max_support_conditions_to_plot,
            "low_support_quantile": cfg.low_support_quantile,
        },
        "package_versions": package_versions(),
        "git_commit_project_root": get_git_commit(cfg.project_root),
        "detected_input_files": {k: str(v) if v is not None else None for k, v in input_paths.items()},
        "input_file_sha256": {k: sha256_file(v) if v is not None else None for k, v in input_paths.items()},
        "code_snapshot_sha256": code_hash,
        "model_order": MODEL_ORDER,
        "generator_only_order": GENERATOR_ONLY_ORDER,
        "model_labels": MODEL_LABELS_ONE_LINE,
        "model_roles": MODEL_ROLE,
        "model_colors": MODEL_COLORS,
        "panel_a_metrics": METRIC_INFO,
        "no_line_plot_compliance": True,
        "outputs": outputs,
        "removed_outputs": {
            "Supplementary_Figure_S10": "Removed as redundant. Equivalent aggregate metrics retained in reports/supplementary_table_equivalent_removed_s10.csv."
        },
        "interpretation_limits": [
            "Computational generator benchmark only.",
            "Surrogate condition-hit rate is descriptor-condition agreement, not anticancer activity.",
            "NN similarity is a memorization-risk audit, not proof of complete absence of memorization.",
            "Exact novelty is not sufficient alone to prove full non-memorization.",
            "Retrieval is a reference/control, not a generator.",
            "No line plots are used in Step 5.",
        ],
    }
    path = dirs["reports"] / "step_05_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


# =============================================================================
# Main runner
# =============================================================================

def run_step5_plos_redesign(
    project_root: str | Path = "/home/data3/Moe/nature_computational_peponco",
    step5_root: str | Path | None = None,
    output_root: str | Path | None = None,
    script_path: str | Path | None = None,
    verbose: bool = True,
) -> Dict[str, object]:
    cfg = Step5Config(
        project_root=Path(project_root),
        step5_root=Path(step5_root) if step5_root is not None else None,
        output_root=Path(output_root) if output_root is not None else None,
        script_path=Path(script_path) if script_path is not None else None,
    )
    apply_plos_style()
    dirs = setup_output_dirs(cfg)

    input_paths = discover_inputs(cfg.resolved_step5_root())

    gen = prep_generator_benchmark(read_csv_required(input_paths["generator_benchmark"], "generator_benchmark"))
    cand = prep_candidate_metrics(read_csv_required(input_paths["candidate_metrics"], "candidate_metrics"))
    per_cond = prep_per_condition_metrics(read_csv_required(input_paths["per_condition_metrics"], "per_condition_metrics"))
    support = prep_condition_support(read_csv_required(input_paths["condition_support_lookup"], "condition_support_lookup"))

    imp1 = pd.read_csv(input_paths["permutation_importance"], low_memory=False) if input_paths["permutation_importance"] else None
    imp2 = pd.read_csv(input_paths["descriptor_recoverability_fallback"], low_memory=False) if input_paths["descriptor_recoverability_fallback"] else None
    importance, descriptor_metric_label, descriptor_audit = prep_descriptor_importance(imp1, imp2)

    outputs: Dict[str, object] = {}
    outputs["Figure_5"] = build_figure5(gen, cand, per_cond, cfg, dirs)
    outputs["Supplementary_Figure_S9"] = build_supplementary_s9(support, importance, descriptor_metric_label, cfg, dirs)
    outputs["reports"] = make_reports(gen, cand, per_cond, support, importance, descriptor_audit, cfg, dirs)

    code_snapshot_path, code_hash = snapshot_code(cfg, dirs)
    outputs["code_snapshot"] = code_snapshot_path
    outputs["README_step_05_outputs"] = write_readme(cfg, dirs, input_paths, outputs, descriptor_metric_label, descriptor_audit)
    outputs["step_05_manifest"] = write_manifest(cfg, dirs, input_paths, outputs, code_hash)

    if verbose:
        print("\nStep 5 PLOS redesign v5 completed successfully.")
        print(f"Input Step 5 root: {cfg.resolved_step5_root()}")
        print(f"Output root: {dirs['root']}")
        print("No line plots were generated.")
        print("Supplementary Figure S10 remains removed; aggregate metrics are retained in reports.")
        print(json.dumps(outputs, indent=2))
    return outputs


if __name__ == "__main__":
    try:
        _script_path = Path(__file__).resolve()
    except NameError:
        _script_path = None
    run_step5_plos_redesign(script_path=_script_path)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 5 PLOS Computational Biology redesign code — v9 final-polish compact old-style/no-S10.

Approved v9 final-polish compact redesign:
- Figure 5A: compact grouped bar benchmark for generation-quality metrics only.
  * Validity, uniqueness, and exact novelty are shown with confidence intervals.
  * Condition fidelity and NN similarity are not duplicated in Panel A; they are shown in Panels C and B.
- Figure 5B: violin + embedded boxplot for candidate-level nearest-neighbor similarity.
- Figure 5C: boxplot + light jitter for per-condition surrogate condition-hit-rate heterogeneity.
- Supplementary Figure S9: condition-support context and descriptor recoverability.
- Supplementary Figure S10: removed as a redundant figure; equivalent metrics retained as a report/table.
- No line plots are used anywhere in Step 5.
- Default output root:
  /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_05/

Usage from terminal:
    python OncoPep_step_05_PLOS_redesign_code_v9_final_polish_noS10.py

Usage from Jupyter:
    from OncoPep_step_05_PLOS_redesign_code_v9_final_polish_noS10 import run_step5_plos_redesign
    outputs = run_step5_plos_redesign()

Scientific interpretation limits:
- Computational generator benchmark only.
- Surrogate condition-hit rate is not anticancer activity.
- Nearest-neighbor similarity is a memorization-risk audit, not proof of complete absence of memorization.
- Exact novelty is not sufficient alone to prove full non-memorization.
- Retrieval is a reference/control, not a generator.
"""

from __future__ import annotations

import hashlib
import json
import platform
import shutil
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Patch


# =============================================================================
# PLOS style and model metadata
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

MODEL_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
    "retrieval_reference",
]

GENERATOR_ONLY_ORDER = [
    "cvae_conditional",
    "gru_conditional",
    "random_conditional",
    "gru_unconditional",
    "vae_unconditional",
]

MODEL_LABELS = {
    "cvae_conditional": "OncoPep",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
    "retrieval_reference": "Retrieval\nreference",
}

MODEL_LABELS_ONE_LINE = {
    "cvae_conditional": "OncoPep",
    "gru_conditional": "GRU-cond",
    "random_conditional": "Rand-cond",
    "gru_unconditional": "GRU-uncond",
    "vae_unconditional": "VAE-uncond",
    "retrieval_reference": "Retrieval reference",
}

MODEL_ROLE = {
    "cvae_conditional": "Primary conditional generator",
    "gru_conditional": "Learned conditional baseline",
    "random_conditional": "Random conditional control",
    "gru_unconditional": "Unconditional sequence baseline",
    "vae_unconditional": "Unconditional latent baseline",
    "retrieval_reference": "Reference/control, not a generator",
}

MODEL_COLORS = {
    "cvae_conditional": PLOS["coral"],
    "gru_conditional": PLOS["blue"],
    "random_conditional": PLOS["mint"],
    "gru_unconditional": PLOS["brown"],
    "vae_unconditional": PLOS["charcoal"],
    "retrieval_reference": PLOS["light"],
}

MODEL_EDGES = {
    "cvae_conditional": "#B65649",
    "gru_conditional": "#176F89",
    "random_conditional": "#79AC86",
    "gru_unconditional": "#8B5F3A",
    "vae_unconditional": "#4E4548",
    "retrieval_reference": "#8A8A8A",
}

METRIC_INFO = {
    "valid_rate": {
        "label": "Validity",
        "axis": "Fraction",
        "direction": "higher_better",
        "low": "valid_rate_ci_low",
        "high": "valid_rate_ci_high",
        "group": "Generation quality",
        "format": "{:.2f}",
    },
    "unique_rate_among_valid": {
        "label": "Uniqueness",
        "axis": "Fraction",
        "direction": "higher_better",
        "low": "unique_rate_among_valid_ci_low",
        "high": "unique_rate_among_valid_ci_high",
        "group": "Generation quality",
        "format": "{:.2f}",
    },
    "exact_novelty_vs_train": {
        "label": "Exact novelty",
        "axis": "Fraction",
        "direction": "higher_better",
        "low": "exact_novelty_vs_train_ci_low",
        "high": "exact_novelty_vs_train_ci_high",
        "group": "Generation quality",
        "format": "{:.2f}",
    },
    "surrogate_condition_hit_rate": {
        "label": "Condition fidelity",
        "axis": "Condition-hit fraction",
        "direction": "higher_better",
        "low": "surrogate_condition_hit_rate_ci_low",
        "high": "surrogate_condition_hit_rate_ci_high",
        "group": "Condition fidelity",
        "format": "{:.2f}",
    },
    "mean_nn_similarity_to_train": {
        "label": "Mean NN similarity",
        "axis": "Mean NN similarity",
        "direction": "lower_better",
        "low": "mean_nn_similarity_to_train_ci_low",
        "high": "mean_nn_similarity_to_train_ci_high",
        "group": "Memorization-risk summary",
        "format": "{:.2f}",
    },
}

PANEL_A_METRIC_ORDER = [
    "valid_rate",
    "unique_rate_among_valid",
    "exact_novelty_vs_train",
    "surrogate_condition_hit_rate",
    "mean_nn_similarity_to_train",
]

COUNT_COLUMNS = [
    "final_retained_peptides",
    "n_final_retained",
    "n_retained",
    "retained_candidates",
    "n_valid",
    "n_valid_candidates",
    "n_generated",
]


@dataclass
class Step5Config:
    project_root: Path = Path("/home/data3/Moe/nature_computational_peponco")
    step5_root: Optional[Path] = None
    output_root: Optional[Path] = None
    script_path: Optional[Path] = None

    dpi_png: int = 300
    dpi_tiff: int = 600
    bootstrap_n: int = 5000
    random_seed: int = 42

    figure5_name: str = "Figure_5_redesigned"
    supp9_name: str = "Supplementary_Figure_S9_redesigned"

    max_support_conditions_to_plot: int = 40
    low_support_quantile: float = 0.25
    max_jitter_points_per_model: int = 700

    def resolved_step5_root(self) -> Path:
        if self.step5_root is not None:
            return Path(self.step5_root)
        candidates = [
            self.project_root / "step5",
            self.project_root / "step_05",
            self.project_root / "results" / "step5",
            self.project_root / "results" / "step_05",
        ]
        for p in candidates:
            if p.exists():
                return p
        return self.project_root / "step5"

    def resolved_output_root(self) -> Path:
        if self.output_root is not None:
            return Path(self.output_root)
        return self.project_root / "PLOS" / "plos_comp" / "step_05"


# =============================================================================
# General utilities
# =============================================================================

def apply_plos_style() -> None:
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 8.8,
        "axes.titlesize": 9.7,
        "axes.labelsize": 8.8,
        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "legend.fontsize": 8.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.80,
        "xtick.major.width": 0.80,
        "ytick.major.width": 0.80,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
    })


def safe_mkdir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def setup_output_dirs(cfg: Step5Config) -> Dict[str, Path]:
    root = safe_mkdir(cfg.resolved_output_root())
    return {
        "root": root,
        "main": safe_mkdir(root / "main_figure"),
        "supp": safe_mkdir(root / "supplementary_figures"),
        "source": safe_mkdir(root / "source_data"),
        "reports": safe_mkdir(root / "reports"),
        "code": safe_mkdir(root / "code"),
    }


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.65)
        ax.yaxis.grid(False)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.65)
        ax.xaxis.grid(False)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.65)
    else:
        ax.grid(False)

    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#666666")
        ax.spines[side].set_linewidth(0.80)
    ax.tick_params(colors=PLOS["dark"], width=0.80, length=3.0)


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.08, y: float = 1.04) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=13,
        fontweight="bold",
        color=PLOS["dark"],
    )


def export_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def save_figure(fig: plt.Figure, base: Path, cfg: Step5Config) -> Dict[str, str]:
    outputs: Dict[str, str] = {}
    pdf = base.with_suffix(".pdf")
    fig.savefig(pdf, bbox_inches="tight")
    outputs["pdf"] = str(pdf)

    png = base.with_suffix(".png")
    fig.savefig(png, bbox_inches="tight", dpi=cfg.dpi_png)
    outputs["png"] = str(png)

    tiff = base.with_suffix(".tiff")
    fig.savefig(tiff, bbox_inches="tight", dpi=cfg.dpi_tiff, format="tiff")
    outputs["tiff"] = str(tiff)

    plt.close(fig)
    return outputs


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    path = Path(path)
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def is_likely_output_artifact(path: Path) -> bool:
    """Return True for files that are likely generated Step 5 outputs.

    This avoids accidentally reading previous PLOS exports without excluding every
    path that merely contains a folder named ``PLOS``. Valid inputs stored under a
    PLOS project folder are therefore still discoverable unless they are inside
    plos_comp/step_05 output-style subdirectories.
    """
    parts = set(path.parts)
    output_dirs = {"main_figure", "supplementary_figures", "source_data", "reports", "code"}
    if "plos_comp" in parts and output_dirs.intersection(parts):
        return True
    if "plos_comp" in parts and "step_05" in parts:
        return True
    return False


def find_first(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    """Find the first plausible input file using exact patterns before wildcards.

    Exact filenames are searched before broad wildcards by the caller's pattern
    order. Within each pattern, output artifacts are ignored and candidates are
    ranked by short path length and recency to reduce stale-file selection.
    """
    for pattern in patterns:
        candidates = []
        for path in root.rglob(pattern):
            if not path.is_file():
                continue
            if is_likely_output_artifact(path):
                continue
            candidates.append(path)
        if candidates:
            # Prefer concise canonical paths, then the most recently modified file.
            candidates = sorted(candidates, key=lambda p: (len(p.parts), -p.stat().st_mtime, str(p)))
            return candidates[0]
    return None

def discover_inputs(step5_root: Path) -> Dict[str, Optional[Path]]:
    return {
        "generator_benchmark": find_first(step5_root, [
            "table_5_2_step5b_generator_benchmark.csv",
            "tables/table_5_2_step5b_generator_benchmark.csv",
            "*generator_benchmark*.csv",
        ]),
        "candidate_metrics": find_first(step5_root, [
            "step5b_generated_candidates_evaluated.csv",
            "samples/step5b_generated_candidates_evaluated.csv",
            "*generated_candidates_evaluated*.csv",
            "*candidate*metrics*.csv",
        ]),
        "per_condition_metrics": find_first(step5_root, [
            "table_s5_4_step5b_per_condition_metrics.csv",
            "tables_supplementary/table_s5_4_step5b_per_condition_metrics.csv",
            "*per_condition*metrics*.csv",
            "*condition*hit*rate*.csv",
        ]),
        "condition_support_lookup": find_first(step5_root, [
            "table_s5a_condition_support_lookup_v9.csv",
            "table_s5a_condition_support_lookup_v8.csv",
            "table_s5a_condition_support_lookup_v7.csv",
            "*condition_support_lookup*.csv",
            "*condition*support*.csv",
        ]),
        "permutation_importance": find_first(step5_root, [
            "step5a_permutation_importance.csv",
            "*permutation*importance*.csv",
            "*descriptor_importance*.csv",
            "*recoverability*.csv",
        ]),
        "descriptor_recoverability_fallback": find_first(step5_root, [
            "main_panel_d_descriptor_recoverability_v10.csv",
            "main_panel_d_descriptor_recoverability.csv",
            "*descriptor_recoverability*.csv",
        ]),
    }


def read_csv_required(path: Optional[Path], name: str) -> pd.DataFrame:
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"Required input not found for {name}: {path}")
    return pd.read_csv(path, low_memory=False)


def require_columns(df: pd.DataFrame, required: Sequence[str], table_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def strict_numeric(df: pd.DataFrame, cols: Sequence[str], table_name: str) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        if col not in out.columns:
            raise ValueError(f"{table_name} is missing required numeric column: {col}")
        out[col] = pd.to_numeric(out[col], errors="coerce")
    bad = {col: int(out[col].isna().sum()) for col in cols if out[col].isna().any()}
    if bad:
        raise ValueError(f"{table_name} contains missing/non-numeric required values after conversion: {bad}")
    return out


def validate_expected_models(df: pd.DataFrame, expected: Sequence[str], label: str, generator_col: str = "generator_key") -> None:
    observed = set(df[generator_col].astype(str))
    missing = [g for g in expected if g not in observed]
    if missing:
        raise ValueError(f"{label} is missing expected model rows: {missing}")


def clip_ci_record(value: float, low: float, high: float, lo_bound: float = 0.0, hi_bound: float = 1.0) -> Dict[str, float | bool]:
    raw_value = float(value)
    raw_low = float(low)
    raw_high = float(high)

    plot_value = min(max(raw_value, lo_bound), hi_bound)
    plot_low = min(max(raw_low, lo_bound), hi_bound)
    plot_high = min(max(raw_high, lo_bound), hi_bound)

    if plot_low > plot_value:
        plot_low = plot_value
    if plot_high < plot_value:
        plot_high = plot_value

    return {
        "raw_value": raw_value,
        "raw_ci_low": raw_low,
        "raw_ci_high": raw_high,
        "plotted_value": plot_value,
        "plotted_ci_low": plot_low,
        "plotted_ci_high": plot_high,
        "ci_was_clipped": bool(raw_value != plot_value or raw_low != plot_low or raw_high != plot_high),
    }


def bootstrap_ci_mean(values: np.ndarray, n_boot: int = 5000, seed: int = 42) -> Tuple[float, float, float]:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), float(vals[0]), float(vals[0])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
    boot = vals[idx].mean(axis=1)
    return float(vals.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


# =============================================================================
# Data preparation
# =============================================================================

def prep_generator_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    required = ["generator"]
    numeric = []
    for metric_key, info in METRIC_INFO.items():
        required += [metric_key, info["low"], info["high"]]
        numeric += [metric_key, info["low"], info["high"]]
    require_columns(df, required, "generator_benchmark")

    out = df[df["generator"].isin(MODEL_ORDER)].copy()
    if out.empty:
        raise ValueError("generator_benchmark contains no expected generator rows.")
    out = strict_numeric(out, numeric, "generator_benchmark")

    for col in COUNT_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out["generator_key"] = out["generator"].astype(str)
    validate_expected_models(out, MODEL_ORDER, "generator_benchmark")

    out["generator"] = pd.Categorical(out["generator_key"], categories=MODEL_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["generator_display"] = out["generator_key"].map(MODEL_LABELS)
    out["generator_display_one_line"] = out["generator_key"].map(MODEL_LABELS_ONE_LINE)
    out["model_role"] = out["generator_key"].map(MODEL_ROLE)
    return out


def prep_candidate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "nn_similarity_to_train"], "candidate_metrics")
    out = df[df["generator"].isin(GENERATOR_ONLY_ORDER)].copy()
    if out.empty:
        raise ValueError("candidate_metrics contains no expected generator rows.")
    out["generator_key"] = out["generator"].astype(str)
    validate_expected_models(out, GENERATOR_ONLY_ORDER, "candidate_metrics")
    out = strict_numeric(out, ["nn_similarity_to_train"], "candidate_metrics")
    out["generator"] = pd.Categorical(out["generator_key"], categories=GENERATOR_ONLY_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["generator_display"] = out["generator_key"].map(MODEL_LABELS)
    out["generator_display_one_line"] = out["generator_key"].map(MODEL_LABELS_ONE_LINE)
    return out


def prep_per_condition_metrics(df: pd.DataFrame) -> pd.DataFrame:
    require_columns(df, ["generator", "requested_condition", "surrogate_condition_hit_rate"], "per_condition_metrics")
    out = df[df["generator"].isin(GENERATOR_ONLY_ORDER)].copy()
    if out.empty:
        raise ValueError("per_condition_metrics contains no expected generator rows.")
    out["generator_key"] = out["generator"].astype(str)
    validate_expected_models(out, GENERATOR_ONLY_ORDER, "per_condition_metrics")
    out = strict_numeric(out, ["surrogate_condition_hit_rate"], "per_condition_metrics")
    out["generator"] = pd.Categorical(out["generator_key"], categories=GENERATOR_ONLY_ORDER, ordered=True)
    out = out.sort_values("generator").reset_index(drop=True)
    out["generator_display"] = out["generator_key"].map(MODEL_LABELS)
    out["generator_display_one_line"] = out["generator_key"].map(MODEL_LABELS_ONE_LINE)
    return out


def prep_condition_support(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "condition_id" not in out.columns:
        for col in ["requested_condition", "condition", "condition_label"]:
            if col in out.columns:
                out = out.rename(columns={col: "condition_id"})
                break
    if "support_total" not in out.columns:
        for col in ["total", "n_total", "support", "train_support", "count"]:
            if col in out.columns:
                out = out.rename(columns={col: "support_total"})
                break

    require_columns(out, ["condition_id", "support_total"], "condition_support_lookup")
    out["condition_id"] = out["condition_id"].astype(str)
    out = strict_numeric(out, ["support_total"], "condition_support_lookup")
    out = out.sort_values("support_total", ascending=False).reset_index(drop=True)
    return out


def prep_descriptor_importance(
    df1: Optional[pd.DataFrame],
    df2: Optional[pd.DataFrame],
) -> Tuple[pd.DataFrame, str, pd.DataFrame]:
    """Prepare descriptor recoverability/importance data and audit dropped rows."""
    candidates = []
    if df1 is not None:
        candidates.append(("permutation_importance", df1.copy()))
    if df2 is not None:
        candidates.append(("descriptor_recoverability_fallback", df2.copy()))

    audit_rows = []
    for source_name, cand in candidates:
        feature_col = next((c for c in ["feature", "descriptor", "variable", "block", "metric", "name"] if c in cand.columns), None)
        value_col = next((c for c in ["recoverability", "macro_f1", "importance", "permutation_importance", "value", "score"] if c in cand.columns), None)
        if not feature_col or not value_col:
            audit_rows.append({
                "source_table": source_name,
                "selected": False,
                "reason": "No usable feature/value column pair found.",
                "n_rows_input": int(len(cand)),
                "n_rows_retained": 0,
                "n_rows_dropped_non_numeric_or_missing": int(len(cand)),
                "feature_column": feature_col,
                "value_column": value_col,
            })
            continue

        out = cand[[feature_col, value_col]].copy()
        out.columns = ["feature", "score"]
        out["feature"] = out["feature"].astype(str)
        out["score"] = pd.to_numeric(out["score"], errors="coerce")
        n_input = int(len(out))
        n_missing = int(out["score"].isna().sum())
        out = out.dropna(subset=["score"]).sort_values("score", ascending=False).reset_index(drop=True)
        if out.empty:
            audit_rows.append({
                "source_table": source_name,
                "selected": False,
                "reason": "All descriptor scores were missing/non-numeric.",
                "n_rows_input": n_input,
                "n_rows_retained": 0,
                "n_rows_dropped_non_numeric_or_missing": n_missing,
                "feature_column": feature_col,
                "value_column": value_col,
            })
            continue

        if value_col in ["recoverability", "macro_f1"]:
            label = "Descriptor recoverability"
        elif "importance" in value_col:
            label = "Permutation importance"
        else:
            label = "Descriptor score"

        out["source_table"] = source_name
        out["source_feature_column"] = feature_col
        out["source_value_column"] = value_col
        audit_rows.append({
            "source_table": source_name,
            "selected": True,
            "reason": "Selected as descriptor panel source.",
            "n_rows_input": n_input,
            "n_rows_retained": int(len(out)),
            "n_rows_dropped_non_numeric_or_missing": n_missing,
            "feature_column": feature_col,
            "value_column": value_col,
            "metric_label": label,
        })
        return out, label, pd.DataFrame(audit_rows)

    raise FileNotFoundError(
        "No usable descriptor recoverability/importance table was found. "
        "Expected a feature/descriptor column and a recoverability/macro_f1/importance/score column."
    )

def candidate_count_column(gen: pd.DataFrame) -> Optional[str]:
    for col in COUNT_COLUMNS:
        if col in gen.columns and pd.to_numeric(gen[col], errors="coerce").notna().any():
            return col
    return None


# =============================================================================
# Figure 5A: compact old-style grouped generation-quality benchmark
# =============================================================================

# Panel A intentionally shows only the three generation-quality metrics.
# Condition fidelity and NN similarity are already shown as distributional panels
# in Figure 5C and Figure 5B, respectively, so repeating them in Panel A makes
# the main figure look diagnostic rather than manuscript-ready.
PANEL_A_GENERATION_QUALITY_METRICS = [
    "valid_rate",
    "unique_rate_among_valid",
    "exact_novelty_vs_train",
]


def benchmark_metric_source_data(
    gen: pd.DataFrame,
    metric_order: Optional[Sequence[str]] = None,
    panel: str = "A",
) -> pd.DataFrame:
    """Create benchmark metric source data with raw/plotted values and CI audit fields."""
    ccol = candidate_count_column(gen)
    metrics = list(metric_order) if metric_order is not None else list(METRIC_INFO.keys())

    rows = []
    for _, row in gen.iterrows():
        g = row["generator_key"]
        for metric_key in metrics:
            info = METRIC_INFO[metric_key]
            ci = clip_ci_record(row[metric_key], row[info["low"]], row[info["high"]])
            rows.append({
                "panel": panel,
                "generator": g,
                "generator_display": MODEL_LABELS_ONE_LINE[g],
                "model_role": MODEL_ROLE[g],
                "metric_key": metric_key,
                "metric_label": info["label"],
                "metric_group": info["group"],
                "metric_direction": info["direction"],
                **ci,
                "candidate_count_column": ccol or "not_available",
                "candidate_count": float(row[ccol]) if ccol is not None and pd.notna(row[ccol]) else np.nan,
            })
    return pd.DataFrame(rows)


def panel_a_source_data(gen: pd.DataFrame) -> pd.DataFrame:
    """Source data for Figure 5A only: validity, uniqueness, exact novelty."""
    return benchmark_metric_source_data(
        gen,
        metric_order=PANEL_A_GENERATION_QUALITY_METRICS,
        panel="A",
    )


def draw_generation_quality_grouped_bars(ax: plt.Axes, panel_src: pd.DataFrame) -> None:
    """Draw compact grouped vertical bars with CI for generation-quality metrics.

    Final-polish choices:
    - full 0–1 y-axis is retained to avoid misleading truncated axes;
    - numeric labels are shown only for informative non-saturated/reference values;
    - the retrieval reference remains muted and is interpreted as a reference/control.
    """
    metrics = PANEL_A_GENERATION_QUALITY_METRICS
    metric_labels = [METRIC_INFO[m]["label"] for m in metrics]
    x = np.arange(len(metrics), dtype=float)
    n_models = len(MODEL_ORDER)

    # Bar width chosen to preserve separation between six models per metric.
    bar_w = 0.096
    offsets = (np.arange(n_models) - (n_models - 1) / 2.0) * (bar_w * 1.22)

    for i, g in enumerate(MODEL_ORDER):
        vals, lows, highs, raw_vals = [], [], [], []
        for metric_key in metrics:
            row = panel_src[
                (panel_src["generator"] == g) &
                (panel_src["metric_key"] == metric_key)
            ]
            if row.empty:
                vals.append(np.nan)
                lows.append(np.nan)
                highs.append(np.nan)
                raw_vals.append(np.nan)
                continue
            r = row.iloc[0]
            vals.append(float(r["plotted_value"]))
            lows.append(float(r["plotted_ci_low"]))
            highs.append(float(r["plotted_ci_high"]))
            raw_vals.append(float(r["raw_value"]))

        vals = np.asarray(vals, dtype=float)
        lows = np.asarray(lows, dtype=float)
        highs = np.asarray(highs, dtype=float)
        raw_vals = np.asarray(raw_vals, dtype=float)

        ax.bar(
            x + offsets[i],
            vals,
            width=bar_w,
            color=MODEL_COLORS[g],
            edgecolor=MODEL_EDGES[g],
            linewidth=0.82,
            alpha=0.95 if g == "cvae_conditional" else 0.84,
            zorder=3,
            label=MODEL_LABELS_ONE_LINE[g],
        )
        ax.errorbar(
            x + offsets[i],
            vals,
            yerr=np.vstack([vals - lows, highs - vals]),
            fmt="none",
            ecolor=MODEL_EDGES[g],
            elinewidth=1.05,
            capsize=2.65,
            capthick=1.05,
            zorder=4,
        )

        # Add numeric labels only where they add information:
        #   - non-saturated values (<0.985), including retrieval uniqueness/novelty;
        #   - avoid labeling every near-perfect bar, which makes the panel cluttered.
        for xi, value, raw_value in zip(x + offsets[i], vals, raw_vals):
            if not np.isfinite(value):
                continue
            if value < 0.985:
                label = f"{raw_value:.2f}"
                # Keep labels away from the axis baseline and away from CI caps.
                y_txt = value + 0.040 if value > 0.08 else 0.075
                ax.text(
                    xi,
                    min(y_txt, 1.058),
                    label,
                    ha="center",
                    va="bottom",
                    fontsize=7.2,
                    color=PLOS["dark"],
                    rotation=0,
                    zorder=5,
                )

    ax.set_ylim(0.0, 1.07)
    ax.set_xlim(-0.60, len(metrics) - 0.35)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.set_ylabel("Fraction")
    ax.set_title("Generation-quality metrics", fontweight="bold", pad=8)
    style_axis(ax, "y")


def candidate_count_source_data(gen: pd.DataFrame) -> pd.DataFrame:
    """Return candidate-count source data even when no count column is provided.

    Candidate counts are intentionally not plotted in the compact Figure 5A
    unless the manuscript later needs them; the data remain available for reports.
    """
    ccol = candidate_count_column(gen)
    if ccol is None:
        return pd.DataFrame({
            "panel": ["A"],
            "generator": ["not_available"],
            "generator_display": ["not_available"],
            "candidate_count_column": ["not_available"],
            "candidate_count": [np.nan],
            "candidate_count_available": [False],
            "plotted_in_sidebar": [False],
        })
    data = gen.set_index("generator_key").loc[MODEL_ORDER].reset_index()
    data["candidate_count"] = pd.to_numeric(data[ccol], errors="coerce")
    return pd.DataFrame({
        "panel": "A",
        "generator": data["generator_key"],
        "generator_display": data["generator_key"].map(MODEL_LABELS_ONE_LINE),
        "candidate_count_column": ccol,
        "candidate_count": data["candidate_count"],
        "candidate_count_available": True,
        "plotted_in_sidebar": False,
    })


def plot_panel_a_benchmark(ax: plt.Axes, gen: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Plot compact generation-quality benchmark Panel A."""
    src = panel_a_source_data(gen)
    draw_generation_quality_grouped_bars(ax, src)
    count_df = candidate_count_source_data(gen)
    return src, count_df


# =============================================================================
# Figure 5B: violin + embedded boxplot

# =============================================================================
# Figure 5B: violin + embedded boxplot
# =============================================================================

def plot_panel_b_violin_box(ax: plt.Axes, cand: pd.DataFrame) -> pd.DataFrame:
    arrays = []
    positions = []
    labels = []
    colors = []
    source_rows = []

    for idx, g in enumerate(GENERATOR_ONLY_ORDER, start=1):
        vals = cand.loc[cand["generator_key"] == g, "nn_similarity_to_train"].to_numpy(float)
        arrays.append(vals)
        positions.append(idx)
        labels.append(MODEL_LABELS_ONE_LINE[g])
        colors.append(MODEL_COLORS[g])
        source_rows += [{
            "panel": "B",
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "nn_similarity_to_train": float(v),
        } for v in vals]

    violin = ax.violinplot(
        arrays,
        positions=positions,
        widths=0.68,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, color in zip(violin["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.27)
        body.set_linewidth(0.9)
        body.set_zorder(2)

    bp = ax.boxplot(
        arrays,
        positions=positions,
        widths=0.20,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": PLOS["dark"], "linewidth": 1.25},
        whiskerprops={"color": "#606060", "linewidth": 0.80},
        capprops={"color": "#606060", "linewidth": 0.80},
        boxprops={"linewidth": 0.85},
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(PLOS["white"])
        patch.set_edgecolor(color)
        patch.set_linewidth(1.00)
        patch.set_zorder(4)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("NN similarity to train")
    ax.set_title("Candidate-level NN similarity to train", fontweight="bold", pad=8)
    ax.set_ylim(0.0, 1.0)
    style_axis(ax, "y")
    return pd.DataFrame(source_rows)


# =============================================================================
# Figure 5C: boxplot + light jitter
# =============================================================================

def plot_panel_c_condition_box_jitter(ax: plt.Axes, per_cond: pd.DataFrame, cfg: Step5Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(cfg.random_seed + 151)
    arrays = []
    positions = []
    labels = []
    colors = []
    raw_rows = []

    for idx, g in enumerate(GENERATOR_ONLY_ORDER, start=1):
        sub = per_cond.loc[per_cond["generator_key"] == g, ["requested_condition", "surrogate_condition_hit_rate"]].copy()
        vals = sub["surrogate_condition_hit_rate"].to_numpy(float)
        arrays.append(vals)
        positions.append(idx)
        labels.append(MODEL_LABELS_ONE_LINE[g])
        colors.append(MODEL_COLORS[g])

        # Use the configurable plotting cap only if condition count grows unusually large.
        plot_indices = np.arange(len(sub))
        if len(plot_indices) > cfg.max_jitter_points_per_model:
            plot_indices = np.sort(rng.choice(plot_indices, size=cfg.max_jitter_points_per_model, replace=False))
        plotted_index_set = set(plot_indices.tolist())

        for local_i, (_, row) in enumerate(sub.iterrows()):
            raw_rows.append({
                "panel": "C",
                "generator": g,
                "generator_display": MODEL_LABELS_ONE_LINE[g],
                "requested_condition": row["requested_condition"],
                "surrogate_condition_hit_rate": float(row["surrogate_condition_hit_rate"]),
                "plotted_in_jitter": bool(local_i in plotted_index_set),
            })

    bp = ax.boxplot(
        arrays,
        positions=positions,
        widths=0.48,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": PLOS["dark"], "linewidth": 1.15},
        whiskerprops={"color": "#606060", "linewidth": 0.80},
        capprops={"color": "#606060", "linewidth": 0.80},
        boxprops={"linewidth": 0.85},
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor(color)
        patch.set_alpha(0.18)

    raw_df_for_plot = pd.DataFrame(raw_rows)
    for x, g, color in zip(positions, GENERATOR_ONLY_ORDER, colors):
        vals_plot = raw_df_for_plot.loc[
            (raw_df_for_plot["generator"] == g) & (raw_df_for_plot["plotted_in_jitter"]),
            "surrogate_condition_hit_rate",
        ].to_numpy(float)
        jitter = np.clip(rng.normal(0, 0.045, len(vals_plot)), -0.10, 0.10)
        ax.scatter(
            np.full(len(vals_plot), x) + jitter,
            vals_plot,
            s=11,
            alpha=0.30,
            color=color,
            edgecolor=PLOS["white"],
            linewidth=0.25,
            zorder=3,
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("Surrogate condition-hit rate")
    ax.set_ylim(0.0, 1.03)
    ax.set_title("Condition-fidelity heterogeneity", fontweight="bold", pad=8)
    style_axis(ax, "y")

    raw_df = pd.DataFrame(raw_rows)
    summary_rows = []
    for g, vals in zip(GENERATOR_ONLY_ORDER, arrays):
        summary_rows.append({
            "panel": "C",
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "n_conditions": int(len(vals)),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "q1": float(np.quantile(vals, 0.25)),
            "q3": float(np.quantile(vals, 0.75)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    return summary_df, raw_df


# =============================================================================
# Build Figure 5
# =============================================================================

def build_figure5(gen: pd.DataFrame, cand: pd.DataFrame, per_cond: pd.DataFrame, cfg: Step5Config, dirs: Dict[str, Path]) -> Dict[str, object]:
    """Build compact old-style-inspired Figure 5.

    Layout:
        A. Generation quality benchmark across the full top row.
        B. Candidate-level NN similarity distribution.
        C. Per-condition condition-fidelity heterogeneity.

    This layout intentionally avoids duplicating condition fidelity and NN similarity
    in Panel A; those metrics are shown more informatively as distributional panels.
    """
    fig = plt.figure(figsize=(9.2, 7.35))
    gs = GridSpec(
        2, 2,
        figure=fig,
        height_ratios=[1.05, 1.0],
        width_ratios=[1.0, 1.0],
        hspace=0.46,
        wspace=0.30,
    )

    # Panel A spans the full figure width.
    ax_a = fig.add_subplot(gs[0, :])
    panel_a_df, count_df = plot_panel_a_benchmark(ax_a, gen)
    add_panel_label(ax_a, "A", x=-0.08, y=1.06)

    # Panel B: memorization-risk distribution.
    ax_b = fig.add_subplot(gs[1, 0])
    panel_b_df = plot_panel_b_violin_box(ax_b, cand)
    add_panel_label(ax_b, "B", x=-0.12, y=1.05)

    # Panel C: condition-level heterogeneity.
    ax_c = fig.add_subplot(gs[1, 1])
    panel_c_summary, panel_c_raw = plot_panel_c_condition_box_jitter(ax_c, per_cond, cfg)
    add_panel_label(ax_c, "C", x=-0.12, y=1.05)

    # Compact one-line legend. Retrieval appears because it is included in Panel A only.
    handles = [
        Patch(facecolor=MODEL_COLORS[g], edgecolor=MODEL_EDGES[g], label=MODEL_LABELS_ONE_LINE[g])
        for g in MODEL_ORDER
    ]
    leg = fig.legend(
        handles=handles,
        loc="lower center",
        ncol=6,
        frameon=True,
        fancybox=False,
        bbox_to_anchor=(0.50, 0.010),
        columnspacing=0.92,
        handlelength=1.22,
        handletextpad=0.42,
        borderpad=0.32,
        prop={"size": 8.4},
    )
    leg.get_frame().set_facecolor(PLOS["white"])
    leg.get_frame().set_edgecolor("#D0D0D0")
    leg.get_frame().set_linewidth(0.65)

    fig.subplots_adjust(left=0.075, right=0.982, top=0.948, bottom=0.145)
    figure_outputs = save_figure(fig, dirs["main"] / cfg.figure5_name, cfg)

    panel_a_df["figure_panel"] = "Figure 5A"
    panel_b_df["figure_panel"] = "Figure 5B"
    panel_c_summary["figure_panel"] = "Figure 5C"
    panel_c_raw["figure_panel"] = "Figure 5C_condition_level"

    # Main all-panel file contains all plotted/source-supported Figure 5 values,
    # including raw condition-level values for Panel C.
    all_panels = pd.concat(
        [panel_a_df, panel_b_df, panel_c_summary, panel_c_raw],
        ignore_index=True,
        sort=False,
    )

    source = dirs["source"]
    source_outputs = {
        "Figure_5_source_data_all_panels": export_csv(all_panels, source / "Figure_5_source_data_all_panels.csv"),
        "Figure_5_panel_a_source_data": export_csv(panel_a_df, source / "Figure_5_panel_a_source_data.csv"),
        "Figure_5_panel_b_source_data": export_csv(panel_b_df, source / "Figure_5_panel_b_source_data.csv"),
        "Figure_5_panel_c_source_data": export_csv(panel_c_summary, source / "Figure_5_panel_c_source_data.csv"),
        "Figure_5_panel_a_candidate_count_source_data": export_csv(count_df, source / "Figure_5_panel_a_candidate_count_source_data.csv"),
        "Figure_5_panel_c_condition_level_source_data": export_csv(panel_c_raw, source / "Figure_5_panel_c_condition_level_source_data.csv"),
    }

    return {"figure_outputs": figure_outputs, "source_outputs": source_outputs}


# =============================================================================
# Supplementary Figure S9
# =============================================================================

def support_concentration_summary(support: pd.DataFrame) -> pd.DataFrame:
    df = support.sort_values("support_total", ascending=False).reset_index(drop=True)
    total = float(df["support_total"].sum())
    n = len(df)
    top10 = max(1, int(np.ceil(n * 0.10)))
    top25 = max(1, int(np.ceil(n * 0.25)))
    top50 = max(1, int(np.ceil(n * 0.50)))
    bottom50 = n - top50

    specs = [
        ("Top 10%", "cumulative_top_support_fraction", top10, "head"),
        ("Top 25%", "cumulative_top_support_fraction", top25, "head"),
        ("Top 50%", "cumulative_top_support_fraction", top50, "head"),
        ("Bottom 50%", "remaining_lower_support_fraction", bottom50, "tail"),
    ]

    rows = []
    for label, summary_type, k, mode in specs:
        if k <= 0:
            count = 0
        elif mode == "head":
            count = int(df.head(k)["support_total"].sum())
        else:
            count = int(df.tail(k)["support_total"].sum())
        rows.append({
            "panel": "B",
            "support_bin": label,
            "summary_type": summary_type,
            "n_condition_ids": int(k),
            "support_count": count,
            "support_fraction": float(count / total) if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def plot_s9a_ranked_support(ax: plt.Axes, support: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    df = support.sort_values("support_total", ascending=True).reset_index(drop=True)
    original_n = len(df)
    truncated = False
    if original_n > cfg.max_support_conditions_to_plot:
        half = cfg.max_support_conditions_to_plot // 2
        df = pd.concat([df.head(half), df.tail(cfg.max_support_conditions_to_plot - half)], ignore_index=True)
        truncated = True

    low_threshold = support["support_total"].quantile(cfg.low_support_quantile)
    df["low_support_flag"] = df["support_total"] <= low_threshold
    y = np.arange(len(df))
    maxv = float(df["support_total"].max()) if len(df) else 1.0
    colors = np.where(df["low_support_flag"], PLOS["light"], PLOS["mint"])
    edges = np.where(df["low_support_flag"], "#999999", "#79AC86")

    ax.barh(
        y,
        df["support_total"],
        color=colors,
        edgecolor=edges,
        linewidth=0.65,
        height=0.72,
        zorder=3,
    )
    for yi, value in zip(y, df["support_total"]):
        ax.text(value + maxv * 0.012, yi, f"{int(value):,}", va="center", ha="left", fontsize=6.8, color=PLOS["dark"])

    ax.set_yticks(y)
    ax.set_yticklabels(df["condition_id"])
    ax.set_xlabel("Number of training/support sequences")
    ax.set_ylabel("Condition ID")
    title = "Ranked condition support"
    if truncated:
        title += " (top/bottom shown)"
    ax.set_title(title, fontweight="bold", pad=8)
    ax.set_xlim(0, maxv * 1.16)
    style_axis(ax, "x")

    out = df.copy()
    out["panel"] = "A"
    out["original_n_conditions"] = original_n
    out["display_truncated"] = truncated
    out["low_support_threshold_quantile"] = cfg.low_support_quantile
    out["low_support_threshold"] = float(low_threshold)
    return out


def plot_s9b_support_concentration(ax: plt.Axes, support: pd.DataFrame) -> pd.DataFrame:
    conc = support_concentration_summary(support)
    x = np.arange(len(conc))
    colors = [PLOS["blue"], PLOS["blue"], PLOS["blue"], PLOS["light"]]
    edges = ["#176F89", "#176F89", "#176F89", "#999999"]

    ax.bar(
        x,
        conc["support_fraction"],
        width=0.62,
        color=colors,
        edgecolor=edges,
        linewidth=0.75,
        zorder=3,
    )
    for xi, frac, cnt in zip(x, conc["support_fraction"], conc["support_count"]):
        ax.text(xi, frac + 0.026, f"{frac:.2f}\n({cnt:,})", ha="center", va="bottom", fontsize=7.0, color=PLOS["dark"])

    ax.set_xticks(x)
    display_labels = ["Top 10%\n(cum.)", "Top 25%\n(cum.)", "Top 50%\n(cum.)", "Bottom 50%\nremaining"]
    ax.set_xticklabels(display_labels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Fraction of total condition support")
    ax.set_title("Condition-support concentration", fontweight="bold", pad=8)
    style_axis(ax, "y")
    return conc


def plot_s9c_descriptor_recoverability(ax: plt.Axes, importance: pd.DataFrame, metric_label: str) -> pd.DataFrame:
    df = importance.copy().head(10).sort_values("score", ascending=True).reset_index(drop=True)
    pretty = {
        "Full Numeric Descriptor Set": "All descriptors",
        "Full numeric descriptor set": "All descriptors",
        "Computable Descriptors": "Computed",
        "Computable descriptors": "Computed",
        "Physicochemical Only": "Physchem",
        "Physicochemical only": "Physchem",
        "AA Composition Only": "AA composition only",
        "Aa Composition Only": "AA composition only",
        "Amino Acid Composition": "AA composition only",
    }
    df["feature_display"] = df["feature"].map(lambda x: pretty.get(str(x), str(x)))
    y = np.arange(len(df))
    maxv = float(df["score"].max()) if len(df) else 1.0

    ax.barh(
        y,
        df["score"],
        height=0.58,
        color=PLOS["coral"],
        edgecolor=MODEL_EDGES["cvae_conditional"],
        linewidth=0.65,
        zorder=3,
    )
    for yi, value in zip(y, df["score"]):
        ax.text(value + maxv * 0.022, yi, f"{value:.2f}", va="center", ha="left", fontsize=7.1, color=PLOS["dark"])

    ax.set_yticks(y)
    ax.set_yticklabels(df["feature_display"])
    ax.set_xlabel(metric_label)
    ax.set_title("Descriptor recoverability", fontweight="bold", pad=8)
    ax.set_xlim(0, max(1.08, maxv * 1.16) if maxv <= 1.0 else maxv * 1.20)
    style_axis(ax, "x")

    out = df.copy()
    out["panel"] = "C"
    out["metric_label"] = metric_label
    return out


def build_supplementary_s9(
    support: pd.DataFrame,
    importance: pd.DataFrame,
    metric_label: str,
    cfg: Step5Config,
    dirs: Dict[str, Path],
) -> Dict[str, object]:
    fig = plt.figure(figsize=(8.4, 8.8))
    gs = GridSpec(
        2, 2,
        figure=fig,
        width_ratios=[1.10, 1.0],
        height_ratios=[1.08, 0.92],
        wspace=0.36,
        hspace=0.43,
    )

    ax_a = fig.add_subplot(gs[:, 0])
    panel_a = plot_s9a_ranked_support(ax_a, support, cfg)
    add_panel_label(ax_a, "A", x=-0.12, y=1.02)

    ax_b = fig.add_subplot(gs[0, 1])
    panel_b = plot_s9b_support_concentration(ax_b, support)
    add_panel_label(ax_b, "B", x=-0.13, y=1.04)

    ax_c = fig.add_subplot(gs[1, 1])
    panel_c = plot_s9c_descriptor_recoverability(ax_c, importance, metric_label)
    add_panel_label(ax_c, "C", x=-0.13, y=1.04)

    fig.subplots_adjust(left=0.12, right=0.985, top=0.955, bottom=0.09)
    figure_outputs = save_figure(fig, dirs["supp"] / cfg.supp9_name, cfg)

    panel_a["figure_panel"] = "Supplementary Figure S9A"
    panel_b["figure_panel"] = "Supplementary Figure S9B"
    panel_c["figure_panel"] = "Supplementary Figure S9C"

    all_panels = pd.concat([panel_a, panel_b, panel_c], ignore_index=True, sort=False)
    source = dirs["source"]
    source_outputs = {
        "Supplementary_Figure_S9_source_data_all_panels": export_csv(all_panels, source / "Supplementary_Figure_S9_source_data_all_panels.csv"),
        "Supplementary_Figure_S9_panel_a_source_data": export_csv(panel_a, source / "Supplementary_Figure_S9_panel_a_source_data.csv"),
        "Supplementary_Figure_S9_panel_b_source_data": export_csv(panel_b, source / "Supplementary_Figure_S9_panel_b_source_data.csv"),
        "Supplementary_Figure_S9_panel_c_source_data": export_csv(panel_c, source / "Supplementary_Figure_S9_panel_c_source_data.csv"),
    }
    return {"figure_outputs": figure_outputs, "source_outputs": source_outputs}


# =============================================================================
# Reports and reproducibility
# =============================================================================

def candidate_similarity_summary(cand: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for g in GENERATOR_ONLY_ORDER:
        vals = cand.loc[cand["generator_key"] == g, "nn_similarity_to_train"].to_numpy(float)
        rows.append({
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "n_candidates": int(len(vals)),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "q1": float(np.quantile(vals, 0.25)),
            "q3": float(np.quantile(vals, 0.75)),
            "p05": float(np.quantile(vals, 0.05)),
            "p95": float(np.quantile(vals, 0.95)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
    return pd.DataFrame(rows)


def per_condition_hit_summary(per_cond: pd.DataFrame, cfg: Step5Config) -> pd.DataFrame:
    rows = []
    for g in GENERATOR_ONLY_ORDER:
        vals = per_cond.loc[per_cond["generator_key"] == g, "surrogate_condition_hit_rate"].to_numpy(float)
        mean, low, high = bootstrap_ci_mean(vals, cfg.bootstrap_n, cfg.random_seed)
        rows.append({
            "generator": g,
            "generator_display": MODEL_LABELS_ONE_LINE[g],
            "n_conditions": int(len(vals)),
            "mean": mean,
            "mean_ci_low": low,
            "mean_ci_high": high,
            "median": float(np.median(vals)),
            "q1": float(np.quantile(vals, 0.25)),
            "q3": float(np.quantile(vals, 0.75)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        })
    return pd.DataFrame(rows)


def figure5_statistical_summary(gen: pd.DataFrame) -> pd.DataFrame:
    return benchmark_metric_source_data(gen, metric_order=list(METRIC_INFO.keys()), panel="report")


def make_reports(
    gen: pd.DataFrame,
    cand: pd.DataFrame,
    per_cond: pd.DataFrame,
    support: pd.DataFrame,
    importance: pd.DataFrame,
    descriptor_audit: pd.DataFrame,
    cfg: Step5Config,
    dirs: Dict[str, Path],
) -> Dict[str, str]:
    reports = dirs["reports"]
    outputs: Dict[str, str] = {}

    outputs["generator_benchmark_summary"] = export_csv(gen, reports / "generator_benchmark_summary.csv")
    outputs["candidate_novelty_similarity_summary"] = export_csv(candidate_similarity_summary(cand), reports / "candidate_novelty_similarity_summary.csv")
    outputs["per_condition_hit_rate_summary"] = export_csv(per_condition_hit_summary(per_cond, cfg), reports / "per_condition_hit_rate_summary.csv")
    outputs["condition_support_summary"] = export_csv(support, reports / "condition_support_summary.csv")
    outputs["descriptor_recoverability_summary"] = export_csv(importance, reports / "descriptor_recoverability_summary.csv")
    outputs["descriptor_recoverability_preparation_audit"] = export_csv(descriptor_audit, reports / "descriptor_recoverability_preparation_audit.csv")

    stat = figure5_statistical_summary(gen)
    outputs["figure5_statistical_summary"] = export_csv(stat, reports / "figure5_statistical_summary.csv")

    removed_s10 = stat.copy()
    removed_s10["note"] = "Supplementary Figure S10 removed as redundant; aggregate benchmark metrics retained in this table."
    outputs["supplementary_table_equivalent_removed_s10"] = export_csv(removed_s10, reports / "supplementary_table_equivalent_removed_s10.csv")

    req = reports / "requirements_step_05_minimal.txt"
    req.write_text(
        f"python=={platform.python_version()}\n"
        f"numpy=={np.__version__}\n"
        f"pandas=={pd.__version__}\n"
        f"matplotlib=={matplotlib.__version__}\n",
        encoding="utf-8",
    )
    outputs["requirements_step_05_minimal"] = str(req)

    return outputs


def get_git_commit(path: Path) -> Optional[str]:
    try:
        res = subprocess.run(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
        )
        return res.stdout.strip()
    except Exception:
        return None


def package_versions() -> Dict[str, str]:
    return {
        "python": sys.version.replace("\n", " "),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
    }


def snapshot_code(cfg: Step5Config, dirs: Dict[str, Path]) -> Tuple[str, Optional[str]]:
    target = dirs["code"] / "OncoPep_step_05_PLOS_redesign_code.py"
    try:
        src = Path(cfg.script_path) if cfg.script_path is not None else Path(__file__).resolve()
        if src.exists():
            shutil.copy2(src, target)
        else:
            raise FileNotFoundError
    except Exception:
        target.write_text(
            "# Code snapshot placeholder.\n"
            "# This run was executed in an interactive environment where the script path was unavailable.\n"
            "# Save the provided v5 script as OncoPep_step_05_PLOS_redesign_code.py for exact reproducibility.\n",
            encoding="utf-8",
        )
    return str(target), sha256_file(target)


def write_readme(
    cfg: Step5Config,
    dirs: Dict[str, Path],
    input_paths: Dict[str, Optional[Path]],
    outputs: Dict[str, object],
    descriptor_metric_label: str,
    descriptor_audit: pd.DataFrame,
) -> str:
    path = dirs["reports"] / "README_step_05_outputs.txt"
    expected_inventory = [
        "main_figure/Figure_5_redesigned.png/pdf/tiff",
        "supplementary_figures/Supplementary_Figure_S9_redesigned.png/pdf/tiff",
        "source_data/Figure_5_source_data_all_panels.csv",
        "source_data/Figure_5_panel_a_source_data.csv",
        "source_data/Figure_5_panel_b_source_data.csv",
        "source_data/Figure_5_panel_c_source_data.csv",
        "source_data/Figure_5_panel_a_candidate_count_source_data.csv",
        "source_data/Figure_5_panel_c_condition_level_source_data.csv",
        "source_data/Supplementary_Figure_S9_source_data_all_panels.csv",
        "source_data/Supplementary_Figure_S9_panel_a_source_data.csv",
        "source_data/Supplementary_Figure_S9_panel_b_source_data.csv",
        "source_data/Supplementary_Figure_S9_panel_c_source_data.csv",
        "reports/generator_benchmark_summary.csv",
        "reports/candidate_novelty_similarity_summary.csv",
        "reports/per_condition_hit_rate_summary.csv",
        "reports/condition_support_summary.csv",
        "reports/descriptor_recoverability_summary.csv",
        "reports/descriptor_recoverability_preparation_audit.csv",
        "reports/figure5_statistical_summary.csv",
        "reports/supplementary_table_equivalent_removed_s10.csv",
        "reports/requirements_step_05_minimal.txt",
        "reports/step_05_manifest.json",
        "reports/README_step_05_outputs.txt",
        "code/OncoPep_step_05_PLOS_redesign_code.py",
    ]
    selected_descriptor_source = "not_available"
    dropped_descriptor_rows = "not_available"
    if descriptor_audit is not None and not descriptor_audit.empty:
        selected = descriptor_audit[descriptor_audit.get("selected", False) == True]
        if not selected.empty:
            selected_descriptor_source = str(selected.iloc[0].get("source_table", "not_available"))
            dropped_descriptor_rows = str(selected.iloc[0].get("n_rows_dropped_non_numeric_or_missing", "not_available"))

    lines = [
        "OncoPep Step 5 PLOS Computational Biology redesign outputs",
        "=" * 72,
        "",
        "Version",
        "-------",
        "v9 final-polish compact old-style/no-S10 redesign.",
        "",
        "Scientific scope",
        "----------------",
        "Step 5 evaluates internal generator benchmarking, exact novelty, uniqueness,",
        "nearest-neighbor memorization-risk auditing, and surrogate condition fidelity.",
        "These analyses are computational audits only. They do not establish anticancer",
        "activity, therapeutic relevance, toxicity safety, or clinical utility.",
        "",
        "Design decisions",
        "----------------",
        "1. No line plots are used in Step 5.",
        "2. Figure 5A uses a compact grouped bar benchmark for generation-quality metrics with confidence intervals, wider within-group spacing, clearer caps, and selective value labels for non-saturated values.",
        "3. Mean NN similarity is not duplicated in Panel A; it is shown as a raw lower-better distribution in Figure 5B.",
        "4. Figure 5B uses slightly strengthened violin plots with embedded boxplots; no retrieval line is shown.",
        "5. Figure 5C uses boxplots with lighter condition-level jitter; retrieval is excluded because it is not a generator.",
        "6. Supplementary Figure S9B uses cumulative support-concentration bars, replacing the former line plot; cumulative-bin definitions are recorded in source data and should be described in the caption.",
        "7. In-plot directional/explanatory notes were minimized; metric direction and support-bin definitions should be handled in captions.",
        "8. Supplementary Figure S10 was removed because it duplicated Figure 5.",
        "   Equivalent aggregate metrics are retained in reports/supplementary_table_equivalent_removed_s10.csv.",
        "8. Candidate-count sidebar is shown only when candidate-count columns are available.",
        "9. Figure_5_source_data_all_panels.csv includes raw condition-level values from Figure 5C.",
        "",
        "Metric interpretation",
        "---------------------",
        "Validity: fraction of generated sequences passing syntax/peptide-format validity checks.",
        "Uniqueness: unique valid outputs divided by valid generated outputs.",
        "Exact novelty: fraction of valid outputs not exactly present in the training set.",
        "Surrogate condition-hit rate: descriptor-condition agreement, not anticancer activity.",
        "NN similarity to train: nearest-neighbor similarity to the training set; lower values indicate lower detectable near-neighbor similarity.",
        "Exact novelty and NN similarity are memorization-risk audits, not proof that all training influence is eliminated.",
        "Retrieval is a reference/control, not a generator.",
        f"Descriptor panel metric label: {descriptor_metric_label}.",
        f"Descriptor source selected: {selected_descriptor_source}.",
        f"Descriptor rows dropped as missing/non-numeric: {dropped_descriptor_rows}.",
        "",
        "Model roles",
        "-----------",
    ]
    for g in MODEL_ORDER:
        lines.append(f"{g}: {MODEL_LABELS_ONE_LINE[g]} — {MODEL_ROLE[g]}")
    lines += [
        "",
        "Expected output inventory",
        "-------------------------",
    ]
    lines += [f"- {item}" for item in expected_inventory]
    lines += [
        "",
        "Detected input files",
        "--------------------",
    ]
    for key, value in input_paths.items():
        lines.append(f"{key}: {value}")
    lines += [
        "",
        "Output root",
        "-----------",
        str(dirs["root"]),
        "",
        "Generated at",
        "------------",
        datetime.now().isoformat(timespec="seconds"),
    ]
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path)


def write_manifest(
    cfg: Step5Config,
    dirs: Dict[str, Path],
    input_paths: Dict[str, Optional[Path]],
    outputs: Dict[str, object],
    code_hash: Optional[str],
) -> str:
    manifest = {
        "task": "OncoPep Step 5 PLOS redesign v9 final-polish compact old-style/no-S10",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "config": {
            "project_root": str(cfg.project_root),
            "step5_root": str(cfg.resolved_step5_root()),
            "output_root": str(cfg.resolved_output_root()),
            "bootstrap_n": cfg.bootstrap_n,
            "random_seed": cfg.random_seed,
            "max_jitter_points_per_model": cfg.max_jitter_points_per_model,
            "max_support_conditions_to_plot": cfg.max_support_conditions_to_plot,
            "low_support_quantile": cfg.low_support_quantile,
        },
        "package_versions": package_versions(),
        "git_commit_project_root": get_git_commit(cfg.project_root),
        "detected_input_files": {k: str(v) if v is not None else None for k, v in input_paths.items()},
        "input_file_sha256": {k: sha256_file(v) if v is not None else None for k, v in input_paths.items()},
        "code_snapshot_sha256": code_hash,
        "model_order": MODEL_ORDER,
        "generator_only_order": GENERATOR_ONLY_ORDER,
        "model_labels": MODEL_LABELS_ONE_LINE,
        "model_roles": MODEL_ROLE,
        "model_colors": MODEL_COLORS,
        "panel_a_metrics": METRIC_INFO,
        "no_line_plot_compliance": True,
        "outputs": outputs,
        "removed_outputs": {
            "Supplementary_Figure_S10": "Removed as redundant. Equivalent aggregate metrics retained in reports/supplementary_table_equivalent_removed_s10.csv."
        },
        "interpretation_limits": [
            "Computational generator benchmark only.",
            "Surrogate condition-hit rate is descriptor-condition agreement, not anticancer activity.",
            "NN similarity is a memorization-risk audit, not proof of complete absence of memorization.",
            "Exact novelty is not sufficient alone to prove full non-memorization.",
            "Retrieval is a reference/control, not a generator.",
            "No line plots are used in Step 5.",
        ],
    }
    path = dirs["reports"] / "step_05_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


# =============================================================================
# Main runner
# =============================================================================

def run_step5_plos_redesign(
    project_root: str | Path = "/home/data3/Moe/nature_computational_peponco",
    step5_root: str | Path | None = None,
    output_root: str | Path | None = None,
    script_path: str | Path | None = None,
    verbose: bool = True,
) -> Dict[str, object]:
    cfg = Step5Config(
        project_root=Path(project_root),
        step5_root=Path(step5_root) if step5_root is not None else None,
        output_root=Path(output_root) if output_root is not None else None,
        script_path=Path(script_path) if script_path is not None else None,
    )
    apply_plos_style()
    dirs = setup_output_dirs(cfg)

    input_paths = discover_inputs(cfg.resolved_step5_root())

    gen = prep_generator_benchmark(read_csv_required(input_paths["generator_benchmark"], "generator_benchmark"))
    cand = prep_candidate_metrics(read_csv_required(input_paths["candidate_metrics"], "candidate_metrics"))
    per_cond = prep_per_condition_metrics(read_csv_required(input_paths["per_condition_metrics"], "per_condition_metrics"))
    support = prep_condition_support(read_csv_required(input_paths["condition_support_lookup"], "condition_support_lookup"))

    imp1 = pd.read_csv(input_paths["permutation_importance"], low_memory=False) if input_paths["permutation_importance"] else None
    imp2 = pd.read_csv(input_paths["descriptor_recoverability_fallback"], low_memory=False) if input_paths["descriptor_recoverability_fallback"] else None
    importance, descriptor_metric_label, descriptor_audit = prep_descriptor_importance(imp1, imp2)

    outputs: Dict[str, object] = {}
    outputs["Figure_5"] = build_figure5(gen, cand, per_cond, cfg, dirs)
    outputs["Supplementary_Figure_S9"] = build_supplementary_s9(support, importance, descriptor_metric_label, cfg, dirs)
    outputs["reports"] = make_reports(gen, cand, per_cond, support, importance, descriptor_audit, cfg, dirs)

    code_snapshot_path, code_hash = snapshot_code(cfg, dirs)
    outputs["code_snapshot"] = code_snapshot_path
    outputs["README_step_05_outputs"] = write_readme(cfg, dirs, input_paths, outputs, descriptor_metric_label, descriptor_audit)
    outputs["step_05_manifest"] = write_manifest(cfg, dirs, input_paths, outputs, code_hash)

    if verbose:
        print("\nStep 5 PLOS redesign v5 completed successfully.")
        print(f"Input Step 5 root: {cfg.resolved_step5_root()}")
        print(f"Output root: {dirs['root']}")
        print("No line plots were generated.")
        print("Supplementary Figure S10 remains removed; aggregate metrics are retained in reports.")
        print(json.dumps(outputs, indent=2))
    return outputs


if __name__ == "__main__":
    try:
        _script_path = Path(__file__).resolve()
    except NameError:
        _script_path = None
    run_step5_plos_redesign(script_path=_script_path)

OncoPep Step 6 supplementary PLOS redesign code.

In [ ]:
from __future__ import annotations

"""
OncoPep Step 6 supplementary PLOS redesign code, v5.

Purpose
-------
Generate the Step 6 supplementary-only figure package for the OncoPep
PLOS Computational Biology manuscript:

    S10 Fig. Latent-space and posterior-usage diagnostics of OncoPep.
    S11 Fig. Sampling-temperature sensitivity and generation-time QA.

This v5 implements the final S10/S11 polish after visual review:
    - No main Step 6 figure is generated.
    - S10 is rebuilt as a three-panel figure: A latent PCA, B KL dynamics,
      C mean posterior variance.
    - The old S10 active-unit panel is removed because it was constant at
      32/32 and not figure-worthy.
    - S10A places the sample-size label n = 13,250 in the upper middle.
    - S10B removes the informal word final.
    - S10C uses an endpoint label formatted as epoch 40 = 0.030.
    - S11 is rebuilt as a two-panel figure: A temperature-response fidelity,
      B selected-temperature OncoPep QA.
    - The old crowded S11 QA heatmap is removed completely.
    - S11A uses only observed temperature ticks, direct labels, and a
      pale retrieval reference annotated as not sampled.
    - S11B removes caption-like footnote text from the figure, uses
      T = 1.0 formatting, and offsets labels away from axes.

Default output root
-------------------
/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_06/

Default input discovery
-----------------------
The function searches for a Step 6 input root containing:
    tables/step6_temperature_sensitivity_summary.csv
    logs/step6_all_training_histories.csv

If your final Step 6 inputs live elsewhere, pass step6_input_root explicitly.
"""

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import hashlib
import json
import shutil
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle


# -----------------------------------------------------------------------------
# PLOS-compatible visual settings
# -----------------------------------------------------------------------------
PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
    "muted": "#8A8A8A",
}

MODEL_ORDER = [
    "retrieval",
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "retrieval": "Retrieval ref.",
    "full_cvae": "OncoPep",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
}

MODEL_COLORS = {
    "retrieval": PLOS["light"],
    "full_cvae": PLOS["coral"],
    "conditional_gru_lm": PLOS["blue"],
    "random_condition": PLOS["mint"],
    "no_condition": PLOS["brown"],
    "small_latent": PLOS["charcoal"],
}

QA_METRICS = [
    ("valid_rate", "Valid fraction", "higher", PLOS["blue"]),
    ("low_complexity_rate", "Low-complexity fraction", "lower", PLOS["mint"]),
    ("exact_train_overlap_rate", "Exact train-overlap fraction", "lower", PLOS["charcoal"]),
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.titlesize": 12.2,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.2,
    "ytick.labelsize": 9.2,
    "legend.fontsize": 9.0,
    "figure.titlesize": 14,
    "axes.facecolor": PLOS["white"],
    "figure.facecolor": PLOS["white"],
    "savefig.facecolor": PLOS["white"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Step6Config:
    project_root: Path
    output_root: Path
    step6_input_root: Path
    tables_dir: Path
    logs_dir: Path
    supplementary_figures_dir: Path
    source_data_dir: Path
    reports_dir: Path
    code_dir: Path
    selected_temperature: Optional[float] = None
    fail_on_missing_pca: bool = False
    latent_dim: Optional[int] = None


def _as_path(x: str | Path) -> Path:
    return Path(x).expanduser().resolve()


def infer_step6_input_root(project_root: Path) -> Path:
    """Infer Step 6 input root from common project layouts."""
    candidates = [
        project_root / "PLOS" / "plos_comp" / "step_06" / "input",
        project_root / "PLOS" / "plos_comp" / "step_06" / "inputs",
        project_root / "step6_v5",
        project_root / "step_06" / "inputs",
        Path("/home/data3/Moe/nature_computational_peponco/step6_v5"),
        Path("/home/data3/Moe/nature_computational2/step6_v5"),
    ]
    for c in candidates:
        if (c / "tables").exists() or (c / "logs").exists():
            return c.resolve()
    return (project_root / "step6_v5").resolve()


def make_config(
    project_root: str | Path = "/home/data3/Moe/nature_computational_peponco",
    step6_input_root: Optional[str | Path] = None,
    output_root: Optional[str | Path] = None,
    selected_temperature: Optional[float] = None,
    fail_on_missing_pca: bool = False,
    latent_dim: Optional[int] = None,
) -> Step6Config:
    project_root = _as_path(project_root)
    if output_root is None:
        output_root = project_root / "PLOS" / "plos_comp" / "step_06"
    output_root = _as_path(output_root)
    if step6_input_root is None:
        step6_input_root = infer_step6_input_root(project_root)
    else:
        step6_input_root = _as_path(step6_input_root)

    cfg = Step6Config(
        project_root=project_root,
        output_root=output_root,
        step6_input_root=step6_input_root,
        tables_dir=step6_input_root / "tables",
        logs_dir=step6_input_root / "logs",
        supplementary_figures_dir=output_root / "supplementary_figures",
        source_data_dir=output_root / "source_data",
        reports_dir=output_root / "reports",
        code_dir=output_root / "code",
        selected_temperature=selected_temperature,
        fail_on_missing_pca=fail_on_missing_pca,
        latent_dim=latent_dim,
    )
    for p in [cfg.supplementary_figures_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        p.mkdir(parents=True, exist_ok=True)
    return cfg


# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_csv_required(path: Path, description: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required {description}: {path}")
    return pd.read_csv(path)


def find_first_existing(paths: Iterable[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.8)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.8)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.0)
        ax.spines[side].set_color("#666666")
    ax.tick_params(axis="both", width=0.9, colors=PLOS["dark"])


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.12, y: float = 1.04) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=16,
        fontweight="bold",
        color=PLOS["dark"],
        clip_on=False,
    )


def save_figure(fig: plt.Figure, base_path: Path) -> Dict[str, str]:
    outputs = {}
    fig.savefig(base_path.with_suffix(".pdf"), bbox_inches="tight")
    outputs["pdf"] = str(base_path.with_suffix(".pdf"))
    fig.savefig(base_path.with_suffix(".png"), bbox_inches="tight", dpi=300)
    outputs["png"] = str(base_path.with_suffix(".png"))
    fig.savefig(base_path.with_suffix(".tiff"), bbox_inches="tight", dpi=600)
    outputs["tiff"] = str(base_path.with_suffix(".tiff"))
    plt.close(fig)
    return outputs


def safe_col(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the expected columns found: {candidates}. Available columns: {list(df.columns)}")
    return None


def ordered_existing_models(df: pd.DataFrame) -> List[str]:
    if "model_tag" not in df.columns:
        return []
    present = set(df["model_tag"].dropna().astype(str))
    ordered = [m for m in MODEL_ORDER if m in present]
    extras = sorted(present.difference(ordered))
    return ordered + extras


def model_label(model_tag: str) -> str:
    return MODEL_LABELS.get(model_tag, model_tag)


def model_color(model_tag: str) -> str:
    return MODEL_COLORS.get(model_tag, "#9E9E9E")


def save_panel_and_all(panel_frames: Dict[str, pd.DataFrame], prefix: str, source_data_dir: Path) -> Dict[str, str]:
    output_paths: Dict[str, str] = {}
    all_frames = []
    for panel, df in panel_frames.items():
        df2 = df.copy()
        df2.insert(0, "panel", panel)
        panel_path = source_data_dir / f"{prefix}_panel_{panel.lower()}_source_data.csv"
        df2.to_csv(panel_path, index=False)
        output_paths[f"panel_{panel.lower()}"] = str(panel_path)
        all_frames.append(df2)
    all_df = pd.concat(all_frames, axis=0, ignore_index=True, sort=False) if all_frames else pd.DataFrame(columns=["panel"])
    all_path = source_data_dir / f"{prefix}_source_data_all_panels.csv"
    all_df.to_csv(all_path, index=False)
    output_paths["all_panels"] = str(all_path)
    return output_paths


def write_requirements(path: Path) -> None:
    path.write_text("numpy\npandas\nmatplotlib\n", encoding="utf-8")


def write_code_snapshot(code_dir: Path) -> Optional[Path]:
    dst = code_dir / "OncoPep_step_06_supplementary_PLOS_redesign_code_v5.py"
    try:
        src = Path(__file__).resolve()
        if src.exists() and src != dst:
            shutil.copy2(src, dst)
            return dst
    except Exception:
        pass
    return None


def _format_temp(t: float) -> str:
    """Format sampling temperature consistently for figure labels.

    PLOS-style figures benefit from fixed precision here because T=1
    should appear as T = 1.0, matching the manuscript/caption wording.
    """
    return f"{float(t):.1f}"


# -----------------------------------------------------------------------------
# Data loading and preparation
# -----------------------------------------------------------------------------
def load_training_history(cfg: Step6Config) -> Tuple[pd.DataFrame, Path]:
    path = cfg.logs_dir / "step6_all_training_histories.csv"
    return read_csv_required(path, "training-history file"), path


def load_temperature_summary(cfg: Step6Config) -> Tuple[pd.DataFrame, Path]:
    path = cfg.tables_dir / "step6_temperature_sensitivity_summary.csv"
    return read_csv_required(path, "temperature-sensitivity file"), path


def load_latent_pca(cfg: Step6Config) -> Tuple[Optional[pd.DataFrame], Optional[Path]]:
    candidates = [
        cfg.step6_input_root / "source_data_figures" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        cfg.step6_input_root / "source_data_figures_redesign" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        cfg.step6_input_root / "source_data_figures_redesign_v2" / "FigS_step6_latent_diagnostics_pca_source_data.csv",
        cfg.step6_input_root / "source_data" / "Supplementary_Figure_S10_panel_a_source_data.csv",
        cfg.tables_dir / "step6_latent_pca_source_data.csv",
        cfg.tables_dir / "latent_pca_source_data.csv",
    ]
    path = find_first_existing(candidates)
    if path is None:
        if cfg.fail_on_missing_pca:
            raise FileNotFoundError("No latent PCA source-data file found in expected locations.")
        return None, None
    return pd.read_csv(path), path


def prepare_full_cvae_history(hist_df: pd.DataFrame) -> pd.DataFrame:
    if "model_tag" in hist_df.columns:
        full = hist_df.loc[hist_df["model_tag"].astype(str) == "full_cvae"].copy()
        if full.empty:
            full = hist_df.copy()
    else:
        full = hist_df.copy()

    epoch_col = safe_col(full, ["epoch", "Epoch"], required=True)
    raw_kl_col = safe_col(full, ["val_raw_kl", "valid_raw_kl", "val_kl_raw", "val_kl_loss"], required=True)
    free_kl_col = safe_col(full, ["val_free_kl", "valid_free_kl", "val_freebits_kl", "val_kl_loss"], required=True)
    active_col = safe_col(full, ["val_active_units_main", "val_active_units", "active_units", "valid_active_units"], required=True)
    latent_var_col = safe_col(full, ["val_mean_latent_var", "mean_latent_var", "valid_mean_latent_var"], required=True)

    summary = (
        full.groupby(epoch_col, as_index=False)
        .agg(
            val_raw_kl=(raw_kl_col, "mean"),
            val_free_kl=(free_kl_col, "mean"),
            val_active_units=(active_col, "mean"),
            val_mean_latent_var=(latent_var_col, "mean"),
        )
        .rename(columns={epoch_col: "epoch"})
        .sort_values("epoch")
    )
    return summary


def standardize_latent_pca(latent_df: Optional[pd.DataFrame]) -> Optional[pd.DataFrame]:
    if latent_df is None:
        return None
    df = latent_df.copy()
    pc1 = safe_col(df, ["pc1", "PC1", "latent_pc1"], required=True)
    pc2 = safe_col(df, ["pc2", "PC2", "latent_pc2"], required=True)
    condition = safe_col(df, ["condition_id", "condition", "condition_label", "cond_id"], required=False)
    out = pd.DataFrame({"pc1": pd.to_numeric(df[pc1], errors="coerce"), "pc2": pd.to_numeric(df[pc2], errors="coerce")})
    out = out.dropna(subset=["pc1", "pc2"]).reset_index(drop=True)
    if condition is not None:
        out["condition_id"] = df.loc[out.index, condition].values
    else:
        out["condition_id"] = 0
    for optional in [
        "sequence",
        "split",
        "length_bin",
        "charge_bin",
        "hydrophobicity_bin",
        "pc1_explained_variance",
        "pc2_explained_variance",
        "pc1_var_explained",
        "pc2_var_explained",
    ]:
        if optional in df.columns:
            out[optional] = df.loc[out.index, optional].values
    return out


def standardize_temperature_df(temp_df: pd.DataFrame) -> pd.DataFrame:
    df = temp_df.copy()
    if "model_tag" not in df.columns:
        raise KeyError("Temperature summary must contain a model_tag column.")
    temperature_col = safe_col(df, ["temperature", "temp", "sampling_temperature"], required=True)
    fidelity_col = safe_col(df, ["condition_fidelity", "classifier_condition_fidelity", "surrogate_condition_hit_rate", "hit_rate"], required=True)
    valid_col = safe_col(df, ["valid_rate", "validity", "gen_valid_rate"], required=False)
    low_complexity_col = safe_col(df, ["low_complexity_rate", "low_complexity", "low_complexity_fraction"], required=False)
    overlap_col = safe_col(df, ["exact_train_overlap_rate", "train_overlap_rate", "exact_overlap_train", "train_exact_overlap_rate"], required=False)

    out = pd.DataFrame({
        "model_tag": df["model_tag"].astype(str),
        "temperature": pd.to_numeric(df[temperature_col], errors="coerce"),
        "condition_fidelity": pd.to_numeric(df[fidelity_col], errors="coerce"),
    })
    out["valid_rate"] = pd.to_numeric(df[valid_col], errors="coerce") if valid_col else np.nan
    out["low_complexity_rate"] = pd.to_numeric(df[low_complexity_col], errors="coerce") if low_complexity_col else np.nan
    out["exact_train_overlap_rate"] = pd.to_numeric(df[overlap_col], errors="coerce") if overlap_col else np.nan

    for col in ["seed", "n_generated", "n_valid", "n_unique", "notes"]:
        if col in df.columns:
            out[col] = df[col]
    return out.dropna(subset=["temperature"])


def aggregate_temperature(temp_std: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = ["condition_fidelity", "valid_rate", "low_complexity_rate", "exact_train_overlap_rate"]
    return (
        temp_std.groupby(["model_tag", "temperature"], as_index=False)[numeric_cols]
        .mean(numeric_only=True)
        .sort_values(["model_tag", "temperature"])
    )


def choose_selected_temperature(temp_agg: pd.DataFrame, requested: Optional[float]) -> float:
    available = np.sort(temp_agg["temperature"].dropna().unique())
    if len(available) == 0:
        raise ValueError("No valid temperatures found in temperature summary.")
    if requested is not None:
        return float(available[np.argmin(np.abs(available - requested))])
    return float(available[np.argmin(np.abs(available - 1.0))])


def infer_latent_dim(hist_curve: pd.DataFrame, requested: Optional[int]) -> int:
    if requested is not None:
        return int(requested)
    max_active = float(np.nanmax(hist_curve["val_active_units"].to_numpy(dtype=float)))
    if np.isfinite(max_active) and max_active > 0:
        return int(np.ceil(max_active))
    return 32


# -----------------------------------------------------------------------------
# Figure S10
# -----------------------------------------------------------------------------
def _pc_label(latent_pca: Optional[pd.DataFrame], axis: str) -> str:
    if latent_pca is None:
        return axis.upper()
    candidates = [f"{axis}_explained_variance", f"{axis}_var_explained"]
    for c in candidates:
        if c in latent_pca.columns:
            vals = pd.to_numeric(latent_pca[c], errors="coerce").dropna()
            if not vals.empty:
                v = float(vals.iloc[0])
                if v <= 1.0:
                    v *= 100
                return f"{axis.upper()} ({v:.1f}% var.)"
    return axis.upper()


def _discrete_condition_scatter(ax: plt.Axes, fig: plt.Figure, latent_pca: pd.DataFrame) -> None:
    cond = latent_pca["condition_id"].astype(str)
    # Sort condition labels numerically where possible.
    def sort_key(v: str):
        try:
            return (0, float(v))
        except Exception:
            return (1, v)
    levels = sorted(cond.dropna().unique(), key=sort_key)
    codes = pd.Categorical(cond, categories=levels, ordered=True).codes
    n = max(len(levels), 1)
    base = plt.get_cmap("tab20") if n <= 20 else plt.get_cmap("turbo")
    colors = base(np.linspace(0.05, 0.95, n))
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(np.arange(-0.5, n + 0.5, 1), cmap.N)
    sc = ax.scatter(
        latent_pca["pc1"],
        latent_pca["pc2"],
        c=codes,
        cmap=cmap,
        norm=norm,
        s=13,
        alpha=0.78,
        linewidths=0,
    )
    cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.018)
    cbar.set_label("Condition ID", fontsize=8.8)
    if n <= 12:
        cbar.set_ticks(np.arange(n))
        cbar.set_ticklabels(levels)
    else:
        # Keep the colorbar compact for many condition IDs.
        tick_idx = np.linspace(0, n - 1, min(6, n)).round().astype(int)
        cbar.set_ticks(tick_idx)
        cbar.set_ticklabels([levels[i] for i in tick_idx])
    cbar.ax.tick_params(labelsize=8.0)


def make_s10_figure(
    cfg: Step6Config,
    hist_curve: pd.DataFrame,
    latent_pca: Optional[pd.DataFrame],
) -> Tuple[Dict[str, str], Dict[str, str], pd.DataFrame]:
    """Create S10 Fig with the final A-C panel structure.

    Final S10 panel mapping:
        A = latent means by condition ID
        B = validation KL dynamics
        C = mean posterior variance

    The old active-latent-unit panel is removed because the trajectory was flat
    at 32/32 active units and is better documented in the summary report.
    """
    latent_dim = infer_latent_dim(hist_curve, cfg.latent_dim)
    final_row = hist_curve.sort_values("epoch").tail(1).iloc[0]
    final_active = float(final_row["val_active_units"])
    active_range = float(hist_curve["val_active_units"].max() - hist_curve["val_active_units"].min())
    active_is_flat = active_range < 0.5

    fig = plt.figure(figsize=(12.6, 8.2))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 0.92], wspace=0.30, hspace=0.42)

    # Panel A: latent PCA
    ax_a = fig.add_subplot(gs[0, 0])
    if latent_pca is not None and len(latent_pca) > 0:
        _discrete_condition_scatter(ax_a, fig, latent_pca)
        # Requested placement: upper-middle, clear, readable, non-overlapping.
        ax_a.text(
            0.50,
            0.94,
            f"n = {len(latent_pca):,}",
            transform=ax_a.transAxes,
            ha="center",
            va="top",
            fontsize=9.2,
            color=PLOS["dark"],
            bbox=dict(facecolor="white", edgecolor=PLOS["grid"], boxstyle="round,pad=0.22", linewidth=0.65, alpha=0.96),
            zorder=10,
        )
    else:
        ax_a.text(
            0.5,
            0.5,
            "Latent PCA source data\nnot found",
            ha="center",
            va="center",
            transform=ax_a.transAxes,
            fontsize=11,
            color=PLOS["dark"],
        )
    ax_a.set_title("Latent means by condition ID")
    ax_a.set_xlabel(_pc_label(latent_pca, "pc1"))
    ax_a.set_ylabel(_pc_label(latent_pca, "pc2"))
    style_axis(ax_a, "both")
    add_panel_label(ax_a, "A")

    # Panel B: KL dynamics. No word "final" and no manuscript-unready annotation.
    ax_b = fig.add_subplot(gs[0, 1])
    ax_b.plot(hist_curve["epoch"], hist_curve["val_raw_kl"], color=PLOS["blue"], linewidth=2.2, label="Raw KL")
    ax_b.plot(hist_curve["epoch"], hist_curve["val_free_kl"], color=PLOS["coral"], linewidth=2.2, label="Free-bits KL")
    ax_b.set_title("Validation KL dynamics")
    ax_b.set_xlabel("Epoch")
    ax_b.set_ylabel("Validation KL")
    style_axis(ax_b, "y")
    ax_b.legend(frameon=False, loc="upper right")
    add_panel_label(ax_b, "B")

    # Panel C: mean posterior variance, spanning the full bottom row.
    ax_c = fig.add_subplot(gs[1, :])
    ax_c.plot(hist_curve["epoch"], hist_curve["val_mean_latent_var"], color=PLOS["brown"], linewidth=2.15)
    end_epoch = float(final_row["epoch"])
    end_var = float(final_row["val_mean_latent_var"])
    ax_c.scatter([end_epoch], [end_var], color=PLOS["brown"], edgecolor=PLOS["dark"], linewidth=0.8, zorder=4, s=42)

    # Add padding first, then place endpoint label inside the axes so it is not cut off.
    ymin = float(np.nanmin(hist_curve["val_mean_latent_var"].to_numpy(dtype=float)))
    ymax = float(np.nanmax(hist_curve["val_mean_latent_var"].to_numpy(dtype=float)))
    yrange = max(ymax - ymin, 1e-6)
    ax_c.set_ylim(max(0, ymin - 0.12 * yrange), ymax + 0.12 * yrange)
    xmin = float(np.nanmin(hist_curve["epoch"].to_numpy(dtype=float)))
    xmax = float(np.nanmax(hist_curve["epoch"].to_numpy(dtype=float)))
    xrange = max(xmax - xmin, 1.0)
    ax_c.set_xlim(xmin - 0.02 * xrange, xmax + 0.08 * xrange)
    ax_c.annotate(
        f"epoch {end_epoch:g} = {end_var:.3f}",
        xy=(end_epoch, end_var),
        xytext=(-74, 11),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9.4,
        color=PLOS["dark"],
        bbox=dict(facecolor="white", edgecolor=PLOS["grid"], boxstyle="round,pad=0.18", linewidth=0.6, alpha=0.96),
        arrowprops=dict(arrowstyle="-", color=PLOS["muted"], lw=0.8, shrinkA=2, shrinkB=3),
        clip_on=False,
    )
    ax_c.set_title("Mean posterior variance")
    ax_c.set_xlabel("Epoch")
    ax_c.set_ylabel("Mean posterior variance")
    style_axis(ax_c, "y")
    add_panel_label(ax_c, "C", x=-0.055, y=1.03)

    fig_outputs = save_figure(fig, cfg.supplementary_figures_dir / "Supplementary_Figure_S10_redesigned")

    panel_a = latent_pca if latent_pca is not None else pd.DataFrame(columns=["pc1", "pc2", "condition_id"])
    panel_b = hist_curve[["epoch", "val_raw_kl", "val_free_kl"]].copy()
    panel_c = hist_curve[["epoch", "val_mean_latent_var"]].copy()
    panel_c["endpoint_epoch"] = hist_curve["epoch"] == final_row["epoch"]
    panel_c["endpoint_label"] = np.where(panel_c["endpoint_epoch"], f"epoch {end_epoch:g} = {end_var:.3f}", "")

    panel_frames: Dict[str, pd.DataFrame] = {
        "A": panel_a,
        "B": panel_b,
        "C": panel_c,
    }
    source_outputs = save_panel_and_all(panel_frames, "Supplementary_Figure_S10", cfg.source_data_dir)
    summary = pd.DataFrame([
        {
            "latent_dim": latent_dim,
            "endpoint_epoch": final_row["epoch"],
            "endpoint_raw_kl": final_row["val_raw_kl"],
            "endpoint_free_bits_kl": final_row["val_free_kl"],
            "endpoint_active_units": final_active,
            "active_units_range": active_range,
            "active_units_panel_removed": True,
            "active_units_panel_removed_reason": "Active latent units were flat/near-flat and therefore documented in the report rather than plotted as a full panel.",
            "active_units_flat": active_is_flat,
            "endpoint_mean_posterior_variance": final_row["val_mean_latent_var"],
            "latent_pca_rows": 0 if latent_pca is None else len(latent_pca),
            "s10_panel_structure": "A latent PCA; B validation KL dynamics; C mean posterior variance",
        }
    ])
    return fig_outputs, source_outputs, summary


# -----------------------------------------------------------------------------
# Figure S11
# -----------------------------------------------------------------------------
def _get_oncopep_temperature(temp_agg: pd.DataFrame) -> pd.DataFrame:
    onco = temp_agg[temp_agg["model_tag"] == "full_cvae"].copy()
    if onco.empty:
        # Fallback to first non-retrieval model if full_cvae is absent.
        models = [m for m in ordered_existing_models(temp_agg) if m != "retrieval"]
        if models:
            onco = temp_agg[temp_agg["model_tag"] == models[0]].copy()
    return onco.sort_values("temperature")


def _format_value_label(value: float) -> str:
    """Format rate labels for figure annotations."""
    if pd.isna(value):
        return "NA"
    if 0 < abs(float(value)) < 0.001:
        return "<0.001"
    return f"{float(value):.2f}"


def _line_endpoint(part: pd.DataFrame, x_col: str = "temperature", y_col: str = "condition_fidelity") -> Tuple[float, float]:
    ordered = part.sort_values(x_col)
    last = ordered.dropna(subset=[x_col, y_col]).tail(1)
    if last.empty:
        return np.nan, np.nan
    return float(last.iloc[0][x_col]), float(last.iloc[0][y_col])


def make_s11_figure(
    cfg: Step6Config,
    temp_agg: pd.DataFrame,
    selected_temperature: float,
) -> Tuple[Dict[str, str], Dict[str, str], pd.DataFrame]:
    """Create S11 Fig with the final A-B panel structure.

    Final S11 panel mapping:
        A = condition fidelity across sampling temperature
        B = OncoPep QA at selected temperature

    v5 polish:
        - x-axis ticks in S11A show only observed sampling temperatures.
        - retrieval is not included in the legend and is directly labeled as
          a reference that is not sampled.
        - OncoPep and GRU-cond are emphasized with direct endpoint labels.
        - low-control baselines are muted to reduce bottom-region clutter.
        - the old S11B heatmap remains deleted.
        - no caption-like footnote text is drawn inside the figure.
    """
    fig = plt.figure(figsize=(12.4, 5.2))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.38, 1.0], wspace=0.42)

    # Panel A: temperature-response plot for condition fidelity.
    ax_a = fig.add_subplot(gs[0, 0])
    models = ordered_existing_models(temp_agg)
    temps = sorted(pd.to_numeric(temp_agg["temperature"], errors="coerce").dropna().unique())
    x_min = min(temps) if temps else 0.8
    x_max = max(temps) if temps else 1.2
    x_pad = max(0.035, 0.08 * (x_max - x_min if x_max > x_min else 0.4))

    # Retrieval is a reference/control, not a temperature-sampled generator.
    if "retrieval" in models:
        retr = temp_agg[temp_agg["model_tag"] == "retrieval"]
        y_ref = float(retr["condition_fidelity"].mean()) if not retr.empty else np.nan
        if np.isfinite(y_ref):
            ax_a.axhline(
                y_ref,
                color="#9A9A9A",
                linestyle="--",
                linewidth=0.85,
                alpha=0.72,
                zorder=1,
            )
            ax_a.text(
                x_max + 0.012,
                y_ref,
                "Retrieval reference\n(not sampled)",
                ha="left",
                va="center",
                fontsize=8.2,
                color=PLOS["muted"],
                clip_on=False,
            )

    # Plot generators and collect direct-label targets. OncoPep and GRU-cond
    # endpoints are close, so their labels are deliberately offset vertically
    # and connected by subtle leader lines to prevent overlap.
    direct_targets: Dict[str, Tuple[float, float]] = {}
    non_retrieval = [m for m in models if m != "retrieval"]
    for model in non_retrieval:
        part = temp_agg[temp_agg["model_tag"] == model].sort_values("temperature")
        if part.empty:
            continue

        if model == "full_cvae":
            lw, ms, alpha, zorder = 3.05, 6.1, 1.00, 7
        elif model == "conditional_gru_lm":
            lw, ms, alpha, zorder = 2.35, 5.3, 0.98, 6
        else:
            lw, ms, alpha, zorder = 1.15, 3.8, 0.33, 3

        ax_a.plot(
            part["temperature"],
            part["condition_fidelity"],
            marker="o",
            markersize=ms,
            linewidth=lw,
            color=model_color(model),
            alpha=alpha,
            zorder=zorder,
        )

        x_end, y_end = _line_endpoint(part)
        if np.isfinite(x_end) and np.isfinite(y_end) and model in ["full_cvae", "conditional_gru_lm"]:
            direct_targets[model] = (x_end, y_end)

    # Direct labels for the two most important generators, with non-overlap.
    label_offsets = {
        "full_cvae": (0.024, 0.030),
        "conditional_gru_lm": (0.024, -0.033),
    }
    for model in ["full_cvae", "conditional_gru_lm"]:
        if model not in direct_targets:
            continue
        x_end, y_end = direct_targets[model]
        dx, dy = label_offsets[model]
        label = model_label(model)
        color = model_color(model)
        ax_a.annotate(
            label,
            xy=(x_end, y_end),
            xytext=(x_end + dx, y_end + dy),
            textcoords="data",
            ha="left",
            va="center",
            fontsize=9.2,
            color=color,
            fontweight="bold" if model == "full_cvae" else "normal",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.84, boxstyle="round,pad=0.10"),
            arrowprops=dict(arrowstyle="-", color=color, lw=0.65, alpha=0.65, shrinkA=1, shrinkB=3),
            clip_on=False,
            zorder=9,
        )

    # Compact label for low-control baseline cluster.
    low_controls = [m for m in ["random_condition", "no_condition", "small_latent"] if m in models]
    if low_controls:
        low_vals = []
        for m in low_controls:
            p = temp_agg[temp_agg["model_tag"] == m]
            if not p.empty:
                _, y_end = _line_endpoint(p)
                if np.isfinite(y_end):
                    low_vals.append(y_end)
        if low_vals:
            ax_a.text(
                x_max + 0.018,
                float(np.nanmean(low_vals)) + 0.020,
                "Low-control\nbaselines",
                ha="left",
                va="center",
                fontsize=8.0,
                color=PLOS["muted"],
                clip_on=False,
            )

    # Selected operating temperature marker.
    ax_a.axvline(selected_temperature, color=PLOS["light"], linestyle="--", linewidth=0.95, zorder=0)
    ax_a.text(
        selected_temperature,
        0.965,
        f"T = {_format_temp(selected_temperature)}",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color=PLOS["muted"],
    )
    ax_a.set_title("Temperature sensitivity of condition fidelity")
    ax_a.set_xlabel("Sampling temperature")
    ax_a.set_ylabel("Condition fidelity")
    ax_a.set_ylim(0, 1.05)
    ax_a.set_xlim(x_min - x_pad, x_max + 0.135)
    ax_a.set_xticks(temps if temps else [0.8, 1.0, 1.2])
    ax_a.set_xticklabels([_format_temp(float(t)) for t in (temps if temps else [0.8, 1.0, 1.2])])
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "A", x=-0.14)

    # Panel B: selected operating-temperature summary as horizontal bars.
    ax_b = fig.add_subplot(gs[0, 1])
    onco = _get_oncopep_temperature(temp_agg)
    oncopep_selected = onco[np.isclose(onco["temperature"], selected_temperature)]
    if oncopep_selected.empty and not onco.empty:
        idx = (onco["temperature"] - selected_temperature).abs().idxmin()
        oncopep_selected = onco.loc[[idx]]
        selected_temperature = float(oncopep_selected.iloc[0]["temperature"])

    if not oncopep_selected.empty:
        row = oncopep_selected.iloc[0]
        metrics = [
            ("condition_fidelity", "Condition fidelity", "higher", PLOS["coral"]),
            ("valid_rate", "Valid fraction", "higher", PLOS["blue"]),
            ("low_complexity_rate", "Low-complexity fraction", "lower", PLOS["mint"]),
            ("exact_train_overlap_rate", "Exact train overlap", "lower", PLOS["charcoal"]),
        ]
        present = [(col, lab, direction, color) for col, lab, direction, color in metrics if col in row.index and pd.notna(row[col])]
        labels = [lab for _, lab, _, _ in present]
        values = [float(row[col]) for col, _, _, _ in present]
        directions = [direction for _, _, direction, _ in present]
        colors = [color for _, _, _, color in present]
        y = np.arange(len(labels))[::-1]
        ax_b.barh(y, values, color=colors, edgecolor=PLOS["dark"], linewidth=0.75, height=0.56)
        ax_b.set_xlim(0, 1.05)
        for yy, value in zip(y, values):
            label = _format_value_label(value)
            if value <= 0.001:
                ax_b.text(0.020, yy, label, ha="left", va="center", fontsize=9.4, color=PLOS["dark"])
            else:
                ax_b.text(min(value + 0.025, 1.035), yy, label, ha="left", va="center", fontsize=9.4, color=PLOS["dark"])
        ax_b.set_yticks(y)
        ax_b.set_yticklabels(labels)
        ax_b.set_xlabel("Fraction")
        ax_b.set_title(f"OncoPep QA at T = {_format_temp(selected_temperature)}")
        style_axis(ax_b, "x")
        operating_summary = pd.DataFrame([
            {
                "model_tag": row["model_tag"],
                "selected_temperature": selected_temperature,
                **{col: float(row[col]) for col, _, _, _ in present},
                "metrics_with_lower_is_better": ", ".join([lab for _, lab, direction, _ in present if direction == "lower"]),
                "figure_note": "Lower is better for low-complexity and exact-overlap metrics; this note should be stated in the caption/README, not embedded in the figure.",
            }
        ])
    else:
        ax_b.text(0.5, 0.5, "No OncoPep row\nfor selected temperature", ha="center", va="center", transform=ax_b.transAxes)
        ax_b.set_axis_off()
        operating_summary = pd.DataFrame(columns=["model_tag", "selected_temperature"])
    add_panel_label(ax_b, "B", x=-0.18)

    fig_outputs = save_figure(fig, cfg.supplementary_figures_dir / "Supplementary_Figure_S11_redesigned")

    panel_a = temp_agg[["model_tag", "temperature", "condition_fidelity"]].copy()
    panel_a["retrieval_in_legend"] = False
    panel_a["retrieval_reference_not_sampled"] = panel_a["model_tag"].eq("retrieval")
    panel_a["selected_temperature"] = selected_temperature
    panel_b = operating_summary.copy()
    source_outputs = save_panel_and_all({"A": panel_a, "B": panel_b}, "Supplementary_Figure_S11", cfg.source_data_dir)
    return fig_outputs, source_outputs, operating_summary

# -----------------------------------------------------------------------------
# Reports and manifest
# -----------------------------------------------------------------------------
def write_reports(
    cfg: Step6Config,
    hist_curve: pd.DataFrame,
    latent_pca: Optional[pd.DataFrame],
    temp_std: pd.DataFrame,
    temp_agg: pd.DataFrame,
    operating_summary: pd.DataFrame,
    s10_summary: pd.DataFrame,
) -> Dict[str, str]:
    outputs: Dict[str, str] = {}

    p = cfg.reports_dir / "step6_latent_diagnostics_summary.csv"
    s10_summary.to_csv(p, index=False)
    outputs["latent_diagnostics_summary"] = str(p)

    p = cfg.reports_dir / "step6_training_history_summary.csv"
    hist_curve.to_csv(p, index=False)
    outputs["training_history_summary"] = str(p)

    p = cfg.reports_dir / "step6_temperature_sensitivity_summary.csv"
    temp_agg.to_csv(p, index=False)
    outputs["temperature_sensitivity_summary"] = str(p)

    qa_cols = ["model_tag", "temperature", "valid_rate", "low_complexity_rate", "exact_train_overlap_rate"]
    qa_cols = [c for c in qa_cols if c in temp_agg.columns]
    p = cfg.reports_dir / "step6_generation_QA_summary.csv"
    temp_agg[qa_cols].to_csv(p, index=False)
    outputs["generation_QA_summary"] = str(p)

    removed = pd.DataFrame([
        {
            "old_panel": "Main Figure 6A",
            "old_content": "Generation quality",
            "decision": "Removed from Step 6 figure package",
            "reason": "Duplicated updated main Fig 2 benchmark panel.",
        },
        {
            "old_panel": "Main Figure 6B",
            "old_content": "Memorization profile / nearest-neighbor similarity",
            "decision": "Removed from Step 6 figure package",
            "reason": "Duplicated updated main Fig 2 benchmark panel.",
        },
        {
            "old_panel": "Main Figure 6C",
            "old_content": "Condition-specific controllability",
            "decision": "Removed from Step 6 figure package",
            "reason": "Duplicated updated main Fig 2 and S9 Fig support panels.",
        },
        {
            "old_panel": "Supplementary Figure S10C / old active-unit panel",
            "old_content": "Active latent units summary",
            "decision": "Removed as a plotted panel and retained in the latent diagnostics report",
            "reason": "Active units were constant or near-constant; a full figure panel was not informative.",
        },
        {
            "old_panel": "Supplementary Figure S11B / old dense QA heatmap",
            "old_content": "Model-by-metric-by-temperature generation-time QA heatmap",
            "decision": "Removed completely and retained as source data/report only",
            "reason": "The heatmap was overcrowded, mixed metric directions, and weakened readability.",
        },
    ])
    p = cfg.reports_dir / "step6_removed_or_reassigned_panels.csv"
    removed.to_csv(p, index=False)
    outputs["removed_or_reassigned_panels"] = str(p)

    p = cfg.reports_dir / "step6_operating_temperature_summary.csv"
    operating_summary.to_csv(p, index=False)
    outputs["operating_temperature_summary"] = str(p)

    # Metric consistency report for potential manuscript checks.
    consistency = []
    if not operating_summary.empty and "condition_fidelity" in operating_summary.columns:
        consistency.append({
            "check": "S11 selected-temperature condition_fidelity",
            "value": float(operating_summary.iloc[0]["condition_fidelity"]),
            "note": "Compare with Fig 2/Results 3.5 condition-hit-rate value before writing; if metric definitions differ, rename/define explicitly.",
        })
    p = cfg.reports_dir / "step6_metric_consistency_notes.csv"
    pd.DataFrame(consistency).to_csv(p, index=False)
    outputs["metric_consistency_notes"] = str(p)

    return outputs


def write_readme(cfg: Step6Config, selected_temperature: float) -> Path:
    readme = f"""# OncoPep Step 6 supplementary outputs, v5

## Purpose
This package generates the final Step 6 supplementary-only model-diagnostic figures for the OncoPep PLOS Computational Biology manuscript.

No main Step 6 figure is produced because the old main Step 6 benchmark panels duplicated the updated main Fig 2 generator-benchmark figure.

## Figures
- S10 Fig: Latent-space and posterior-usage diagnostics of OncoPep.
- S11 Fig: Sampling-temperature sensitivity and generation-time quality assurance.

## Final v5 panel structure
S10 Fig has three panels:
- S10A: test-set latent means by condition ID, with the sample-size label n = 13,250 placed at the upper middle of the panel.
- S10B: validation KL dynamics; no informal final label is shown.
- S10C: mean posterior variance; the endpoint is labeled using the format epoch N = value.

S11 Fig has two panels:
- S11A: condition fidelity across sampling temperature, shown as a line/dot plot because temperature is ordered; retrieval is shown only as a pale annotated reference line and is not included in the generator legend. OncoPep and GRU-cond are directly labeled with non-overlapping endpoint labels.
- S11B: OncoPep QA at the selected sampling temperature, shown as horizontal bars; metric-direction notes are kept in the caption/README rather than embedded in the figure.

## Removed panels
- The old S10 active-unit panel was removed because active units were constant or near-constant and are better documented in the latent diagnostics report.
- The old S11 dense QA heatmap was removed because it was overcrowded, mixed higher-is-better and lower-is-better metrics, and partly duplicated information better suited to a report/source-data table.

## Output directories
- supplementary_figures/: PNG, PDF, and TIFF figure exports.
- source_data/: panel-level and all-panel CSV files.
- reports/: diagnostic summaries, removed-panel log, manifest, and README.
- code/: executable code snapshot and minimal requirements.

## Line-plot policy
Line plots are retained only for ordered quantities: training epoch and sampling temperature. No line plot is used across unordered model categories.

## How Step 6 differs from Step 5
Step 5 reports generator benchmark outputs: validity, uniqueness, exact novelty, nearest-neighbor similarity, surrogate condition fidelity, per-condition heterogeneity, condition support, and descriptor recoverability. Step 6 is restricted to supplementary model diagnostics: latent-space structure, KL/free-bits behavior, latent variance, temperature sensitivity, and generation-time QA.

## Selected operating temperature
The selected operating temperature used for the S11 summary panel is T={selected_temperature:g}. If no explicit temperature was provided, the code selected the available temperature closest to 1.0.

## Caption note for S11
Lower values are better for the low-complexity and exact train-overlap metrics in S11B. This explanatory note is intentionally kept out of the image and should be stated in the caption or supplementary legend.

## Source-data mapping
- Supplementary_Figure_S10_panel_a_source_data.csv: latent PCA source data.
- Supplementary_Figure_S10_panel_b_source_data.csv: validation raw/free-bits KL data.
- Supplementary_Figure_S10_panel_c_source_data.csv: mean posterior variance data.
- Supplementary_Figure_S11_panel_a_source_data.csv: model-level condition fidelity across temperature.
- Supplementary_Figure_S11_panel_b_source_data.csv: OncoPep selected-temperature QA summary.

## Regeneration
```python
from OncoPep_step_06_supplementary_PLOS_redesign_code_v5 import run_step6_supplementary_redesign
outputs = run_step6_supplementary_redesign(
    project_root="{cfg.project_root}",
    step6_input_root="{cfg.step6_input_root}",
)
```

## Required input files
- tables/step6_temperature_sensitivity_summary.csv
- logs/step6_all_training_histories.csv
- latent PCA source data in one of the searched locations, or set fail_on_missing_pca=False to generate S10A as an explicit missing-data placeholder.

## Manuscript caution
Before writing the Step 6 text, compare S11 selected-temperature condition fidelity with the Fig 2/Results 3.5 condition-hit-rate value. If the metrics differ, rename and define them explicitly.
"""
    p = cfg.reports_dir / "README_step_06_supplementary_outputs.txt"
    p.write_text(readme, encoding="utf-8")
    return p



def write_manifest(
    cfg: Step6Config,
    input_paths: Dict[str, Optional[Path]],
    figure_outputs: Dict[str, Dict[str, str]],
    source_outputs: Dict[str, Dict[str, str]],
    report_outputs: Dict[str, str],
    readme_path: Path,
    code_snapshot: Optional[Path],
    selected_temperature: float,
) -> Path:
    def path_record(p: Optional[Path]) -> Optional[Dict[str, str]]:
        if p is None:
            return None
        rec = {"path": str(p)}
        if p.exists() and p.is_file():
            rec["sha256"] = sha256_file(p)
        return rec

    manifest = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "purpose": "OncoPep Step 6 supplementary-only PLOS redesign, v5",
        "figure_numbering": {
            "first_supplementary": "S10 Fig",
            "second_supplementary": "S11 Fig",
            "main_step6_figure": "not generated; old main Figure 6 removed because it duplicated updated Fig 2",
        },
        "project_root": str(cfg.project_root),
        "input_root": str(cfg.step6_input_root),
        "output_root": str(cfg.output_root),
        "selected_temperature": selected_temperature,
        "input_files": {k: path_record(v) for k, v in input_paths.items()},
        "figure_outputs": figure_outputs,
        "source_data_outputs": source_outputs,
        "report_outputs": report_outputs,
        "readme": str(readme_path),
        "code_snapshot": None if code_snapshot is None else str(code_snapshot),
        "software": {
            "python": sys.version,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": matplotlib.__version__,
        },
    }
    p = cfg.reports_dir / "step6_supplementary_figures_manifest.json"
    p.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return p


# -----------------------------------------------------------------------------
# Main public entry point
# -----------------------------------------------------------------------------
def run_step6_supplementary_redesign(
    project_root: str | Path = "/home/data3/Moe/nature_computational_peponco",
    step6_input_root: Optional[str | Path] = None,
    output_root: Optional[str | Path] = None,
    selected_temperature: Optional[float] = None,
    fail_on_missing_pca: bool = False,
    latent_dim: Optional[int] = None,
) -> Dict[str, object]:
    """Generate the Step 6 supplementary-only PLOS figure package.

    Parameters
    ----------
    project_root:
        OncoPep project root.
    step6_input_root:
        Directory containing Step 6 input tables/logs. If omitted, common
        locations are searched.
    output_root:
        Output root. If omitted, uses <project_root>/PLOS/plos_comp/step_06/.
    selected_temperature:
        Temperature to summarize in S11B. If omitted, the available temperature
        closest to 1.0 is used.
    fail_on_missing_pca:
        If True, missing PCA source data raises an error. If False, S10A is
        generated with a clear missing-data placeholder.
    latent_dim:
        Optional known latent dimension. If omitted, inferred from the active-unit
        trajectory.

    Returns
    -------
    Dictionary containing figure, source-data, report, README, and manifest paths.
    """
    cfg = make_config(
        project_root=project_root,
        step6_input_root=step6_input_root,
        output_root=output_root,
        selected_temperature=selected_temperature,
        fail_on_missing_pca=fail_on_missing_pca,
        latent_dim=latent_dim,
    )

    hist_df, hist_path = load_training_history(cfg)
    temp_df, temp_path = load_temperature_summary(cfg)
    latent_raw, latent_path = load_latent_pca(cfg)

    hist_curve = prepare_full_cvae_history(hist_df)
    latent_pca = standardize_latent_pca(latent_raw)
    temp_std = standardize_temperature_df(temp_df)
    temp_agg = aggregate_temperature(temp_std)
    selected_t = choose_selected_temperature(temp_agg, selected_temperature)

    s10_fig, s10_source, s10_summary = make_s10_figure(cfg, hist_curve, latent_pca)
    s11_fig, s11_source, operating_summary = make_s11_figure(cfg, temp_agg, selected_t)

    report_outputs = write_reports(cfg, hist_curve, latent_pca, temp_std, temp_agg, operating_summary, s10_summary)
    req_path = cfg.code_dir / "requirements_step_06_minimal.txt"
    write_requirements(req_path)
    report_outputs["requirements"] = str(req_path)

    code_snapshot = write_code_snapshot(cfg.code_dir)
    readme_path = write_readme(cfg, selected_t)

    input_paths = {
        "training_history": hist_path,
        "temperature_sensitivity": temp_path,
        "latent_pca": latent_path,
    }
    figure_outputs = {"S10_Fig": s10_fig, "S11_Fig": s11_fig}
    source_outputs = {"S10_Fig": s10_source, "S11_Fig": s11_source}
    manifest_path = write_manifest(
        cfg=cfg,
        input_paths=input_paths,
        figure_outputs=figure_outputs,
        source_outputs=source_outputs,
        report_outputs=report_outputs,
        readme_path=readme_path,
        code_snapshot=code_snapshot,
        selected_temperature=selected_t,
    )

    return {
        "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()},
        "figures": figure_outputs,
        "source_data": source_outputs,
        "reports": report_outputs,
        "readme": str(readme_path),
        "manifest": str(manifest_path),
        "code_snapshot": None if code_snapshot is None else str(code_snapshot),
        "selected_temperature": selected_t,
    }


if __name__ == "__main__":
    outputs = run_step6_supplementary_redesign()
    print(json.dumps(outputs, indent=2))

OncoPep Step 7 supplementary figure redesign for PLOS Computational Biology.

In [ ]:
#!/usr/bin/env python3
"""
OncoPep Step 7 PLOS Computational Biology candidate-audit redesign.

Version
-------
v6 auto-discovery / notebook-terminal ready / candidate-audit / PLOS-ready

Purpose
-------
This script is designed for both VS Code/Jupyter notebooks and terminal use.
It treats Step 7 as a final generated-candidate audit rather than another
Step 5 generator benchmark or Step 6 latent-space diagnostic.

Outputs
-------
Supplementary figures only:
    S12 Fig: Final candidate audit and selection funnel
    S13 Fig: Reference-space descriptor support
    S14 Fig: Similarity-tail risk and nearest-neighbor audit

No new main Figure 7 is generated by default.

Claim boundary
--------------
These analyses support computational auditing and downstream prioritization only.
They do not establish anticancer activity, receptor binding, selectivity, toxicity,
stability, mechanism, therapeutic efficacy, or clinical relevance.

Recommended notebook usage
--------------------------
from OncoPep_step7_PLOS_candidate_audit_v6_auto_ready import run_step7_candidate_audit

# Auto-discovery mode:
outputs = run_step7_candidate_audit(show_figures=True)
outputs

# Explicit-path mode:
outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/REAL/PATH/step7_generated_final_audited.csv",
    train_file="/REAL/PATH/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
)
outputs

Expected input columns
----------------------
Generated/final audited table should ideally contain:
    model_tag
    sequence
    nn_similarity_train
    length
    net_charge
    hydrophobicity
    entropy
    candidate_id                         [optional]
    is_final_candidate / final_selected  [optional]
    exact_novel_vs_train / exact_novel    [optional]
    composite_score / rank                [optional]

Notebook-friendly fallbacks
---------------------------
If length, net_charge, hydrophobicity, or entropy are missing, they are computed
from sequence with explicit warnings and documented in the readiness report.
If no explicit final-candidate flag is present, the script infers final
candidates using: explicit final file > explicit flag > top_n_final from OncoPep
rows sorted by rank/score > OncoPep rows passing computed audit.
If nn_similarity_train is missing, the script can compute it only when the
row-count x train-count pair budget is not too large; otherwise it reports a
blocking readiness issue.
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import math
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch


# =============================================================================
# Constants and visual style
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "OncoPep",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
    "train_ref": "Train reference",
}

MODEL_COLORS = {
    "full_cvae": PLOS["coral"],
    "conditional_gru_lm": PLOS["blue"],
    "random_condition": PLOS["mint"],
    "no_condition": PLOS["brown"],
    "small_latent": PLOS["charcoal"],
    "train_ref": PLOS["light"],
}

MODEL_EDGES = {
    "full_cvae": "#B84F42",
    "conditional_gru_lm": "#166F8A",
    "random_condition": "#75A883",
    "no_condition": "#8E5F39",
    "small_latent": "#4F4648",
    "train_ref": "#A7A7A7",
}

MODEL_TAG_ALIASES = {
    "oncoppep": "full_cvae",
    "oncopep": "full_cvae",
    "full_cvae": "full_cvae",
    "cvae": "full_cvae",
    "conditional_vae": "full_cvae",
    "conditioned_vae": "full_cvae",
    "gru-cond": "conditional_gru_lm",
    "gru_cond": "conditional_gru_lm",
    "conditional_gru": "conditional_gru_lm",
    "conditional_gru_lm": "conditional_gru_lm",
    "rand-cond": "random_condition",
    "rand_cond": "random_condition",
    "random_condition": "random_condition",
    "random conditional": "random_condition",
    "gru-uncond": "no_condition",
    "gru_uncond": "no_condition",
    "unconditional_gru": "no_condition",
    "no_condition": "no_condition",
    "vae-uncond": "small_latent",
    "vae_uncond": "small_latent",
    "unconditional_vae": "small_latent",
    "small_latent": "small_latent",
}

AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DEFAULT_TAIL_THRESHOLDS = (0.10, 0.15, 0.20, 0.30)
DESCRIPTORS = ("length", "net_charge", "hydrophobicity", "entropy")

# Kyte-Doolittle hydrophobicity scale.
KD_SCALE = {
    "A": 1.8,
    "C": 2.5,
    "D": -3.5,
    "E": -3.5,
    "F": 2.8,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "N": -3.5,
    "P": -1.6,
    "Q": -3.5,
    "R": -4.5,
    "S": -0.8,
    "T": -0.7,
    "V": 4.2,
    "W": -0.9,
    "Y": -1.3,
}

COLUMN_ALIASES = {
    "model_tag": ["model_tag", "model", "generator", "model_name", "method"],
    "candidate_id": ["candidate_id", "peptide_id", "id", "seq_id", "sequence_id", "candidate"],
    "sequence": ["sequence", "peptide", "seq", "peptide_sequence"],
    "is_final_candidate": [
        "is_final_candidate",
        "final_selected",
        "selected_final",
        "selected",
        "is_selected",
        "final_candidate",
        "passed_final_audit",
        "passed_audit",
    ],
    "candidate_rank": ["candidate_rank", "rank", "final_rank", "selection_rank", "priority_rank"],
    "composite_score": ["composite_score", "score", "selection_score", "priority_score", "audit_score"],
    "exact_novel_vs_train": [
        "exact_novel_vs_train",
        "exact_novel",
        "exact_novelty",
        "novel_vs_train",
        "novelty_flag",
        "is_exact_novel",
    ],
    "nn_similarity_train": [
        "nn_similarity_train",
        "nearest_neighbor_similarity_train",
        "nearest_train_similarity",
        "nearest_train_jaccard",
        "smax_train",
        "nn_train",
        "similarity_to_train",
        "nearest_neighbor_similarity",
    ],
    "length": ["length", "seq_length", "peptide_length", "model_ready_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "charge", "charge_pH7", "net_charge_ph7"],
    "hydrophobicity": [
        "hydrophobicity",
        "hydrophobicity_KD",
        "mean_hydrophobicity",
        "mean_kd_hydrophobicity",
        "mean_KD",
        "mean_kyte_doolittle",
        "kd_hydrophobicity",
    ],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy", "seq_entropy"],
}

TRAIN_ALIASES = {
    "sequence": COLUMN_ALIASES["sequence"],
    "length": COLUMN_ALIASES["length"],
    "net_charge": COLUMN_ALIASES["net_charge"],
    "hydrophobicity": COLUMN_ALIASES["hydrophobicity"],
    "entropy": COLUMN_ALIASES["entropy"],
}


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Step7Config:
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07"
    input_roots: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/source_data",
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/reports",
        "/home/data3/Moe/nature_computational2/step7_v3",
        "/home/data3/Moe/nature_computational2/step7_v3/outputs",
        "/home/data3/Moe/nature_computational2/step7_v3/outputs/tables",
        "/home/data3/Moe/nature_computational2/step7_v3/outputs/source_data",
        "/home/data3/Moe/nature_computational2/redesign/step7",
        "/home/data3/Moe/nature_computational2/redesign/step7/outputs",
        "/home/data3/Moe/nature_computational2/redesign/step7/outputs/source_data",
        "/home/data3/Moe/nature_computational2/redesign/step7/outputs/tables",
    )

    generated_file: Optional[str] = None
    train_file: Optional[str] = None
    final_file: Optional[str] = None
    summary_file: Optional[str] = None
    funnel_file: Optional[str] = None

    generated_candidates: Tuple[str, ...] = (
        "step7_generated_final_audited.csv",
        "step7_generated_final.csv",
        "step7_generated_audited.csv",
        "generated_final_audited.csv",
        "SuppFig_step7_property_generated_final_v7_source_data.csv",
        "SuppFig_step7_similarity_distribution_v7_source_data.csv",
    )
    train_candidates: Tuple[str, ...] = (
        "step7_train_reference_annotated.csv",
        "train_reference_annotated.csv",
        "SuppFig_step7_property_train_reference_v7_source_data.csv",
        "Supplementary_Figure_S13_train_reference_source_data.csv",
    )
    final_candidates: Tuple[str, ...] = (
        "step7_final_candidates.csv",
        "step7_final_candidates_audit_table.csv",
        "final_candidates.csv",
        "final_oncopep_candidates.csv",
    )
    summary_candidates: Tuple[str, ...] = (
        "step7_model_summary.csv",
        "step7_final_generation_audit_summary.csv",
        "step7_reference_space_plausibility_summary.csv",
        "Fig7_step7_main_final_v7_source_data.csv",
    )
    funnel_candidates: Tuple[str, ...] = (
        "step7_candidate_funnel_counts.csv",
        "candidate_funnel_counts.csv",
        "step7_funnel_counts.csv",
        "Fig7_step7_candidate_funnel_source_data.csv",
    )

    reference_quantile_low: float = 0.00
    reference_quantile_high: float = 1.00
    similarity_low_risk_threshold: float = 0.30
    tail_thresholds: Tuple[float, ...] = DEFAULT_TAIL_THRESHOLDS
    top_n_final: int = 12

    compute_missing_descriptors: bool = True
    histidine_charge: float = 0.10
    compute_nn_if_missing: bool = True
    max_nn_pairs: int = 3_000_000
    nn_metric_if_computed: str = "normalized_levenshtein_similarity"

    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    tiff_compression: str = "tiff_lzw"
    show_figures: bool = False

    font_family: str = "DejaVu Sans"
    base_font_size: float = 11.2
    axis_label_size: float = 12.4
    title_size: float = 14.2
    tick_size: float = 10.2
    legend_size: float = 10.8
    panel_label_size: float = 18.0

    seed: int = 42
    max_heatmap_candidates: int = 30
    inventory_max_files: int = 300

    @property
    def root(self) -> Path:
        return Path(self.step7_root).expanduser().resolve()

    @property
    def main_figure_dir(self) -> Path:
        # Created for folder consistency only. This Step 7 script does not
        # generate a new main Figure 7 unless the manuscript plan changes.
        return self.root / "main_figure"

    @property
    def supplementary_dir(self) -> Path:
        return self.root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.root / "code"


# =============================================================================
# General utilities
# =============================================================================

def ensure_output_tree(cfg: Step7Config) -> None:
    for path in [cfg.root, cfg.main_figure_dir, cfg.supplementary_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)


def set_plot_style(cfg: Step7Config) -> None:
    plt.rcParams.update(
        {
            "font.family": cfg.font_family,
            "font.size": cfg.base_font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "figure.titlesize": cfg.title_size,
            "axes.facecolor": PLOS["white"],
            "figure.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def inventory_csv_files(roots: Sequence[str], max_files: int = 300) -> pd.DataFrame:
    """Inventory CSV files without recursively sorting broad file trees.

    VS Code notebooks often start in a broad working directory. A previous
    version used sorted(root.rglob("*.csv")), which can hang if root is `/` or a
    large project tree. This function streams results and stops after max_files
    per root.
    """
    rows: List[Dict[str, object]] = []
    seen: set[str] = set()
    broad_roots = {Path("/").resolve()}
    for priority, root_raw in enumerate(roots):
        root = Path(root_raw).expanduser()
        if not root.exists():
            rows.append({"root_priority": priority, "root": str(root), "exists": False, "path": None, "filename": None, "size_bytes": None})
            continue

        try:
            resolved_root = root.resolve()
        except Exception:
            resolved_root = root

        if resolved_root in broad_roots:
            rows.append({"root_priority": priority, "root": str(root), "exists": True, "path": None, "filename": "SKIPPED_BROAD_ROOT", "size_bytes": None})
            continue

        if root.is_file():
            file_iter = iter([root] if root.suffix.lower() == ".csv" else [])
        else:
            file_iter = root.rglob("*.csv")

        n_seen_this_root = 0
        for file_path in file_iter:
            if n_seen_this_root >= max_files:
                break
            try:
                key = str(file_path.resolve())
            except Exception:
                continue
            if key in seen:
                continue
            seen.add(key)
            n_seen_this_root += 1
            try:
                size_bytes = file_path.stat().st_size
            except OSError:
                size_bytes = None
            rows.append(
                {
                    "root_priority": priority,
                    "root": str(root),
                    "exists": True,
                    "path": key,
                    "filename": file_path.name,
                    "size_bytes": size_bytes,
                }
            )
    return pd.DataFrame(rows)


def normalize_path(path_raw: Optional[str]) -> Optional[Path]:
    if not path_raw:
        return None
    return Path(path_raw).expanduser().resolve()


def _iter_candidate_matches(root: Path, candidate_names: Sequence[str], max_hits: int = 5) -> Iterable[Path]:
    """Yield candidate-name matches without sorting a whole recursive tree."""
    candidate_set = set(candidate_names)
    if root.is_file():
        if root.name in candidate_set:
            yield root.resolve()
        return

    hits = 0
    # First check direct children because this is the expected layout.
    for name in candidate_names:
        p = root / name
        if p.exists() and p.is_file():
            yield p.resolve()
            hits += 1
            if hits >= max_hits:
                return

    # Then stream recursively, without sorted(root.rglob(...)).
    try:
        for p in root.rglob("*.csv"):
            if p.name in candidate_set:
                yield p.resolve()
                hits += 1
                if hits >= max_hits:
                    return
    except (OSError, PermissionError):
        return


def build_missing_file_diagnostic(
    label: str,
    candidate_names: Sequence[str],
    searched_roots: Sequence[str],
    inventory: Optional[pd.DataFrame],
    step7_root: str,
) -> str:
    candidate_msg = "\n".join(f"  - {name}" for name in candidate_names)
    root_msg = "\n".join(f"  - {root}" for root in searched_roots)
    inventory_msg = ""
    if inventory is not None and not inventory.empty and "path" in inventory.columns:
        existing = inventory.dropna(subset=["path"]).copy()
        if not existing.empty:
            inventory_msg = "\n\nCSV files found in searched roots (first 80):\n"
            inventory_msg += existing[["filename", "path", "size_bytes"]].head(80).to_string(index=False)
    command_msg = (
        "\n\nSuggested terminal command after locating real files:\n"
        f"python OncoPep_step7_PLOS_candidate_audit_v6_auto_ready.py \\\n"
        f"  --step7-root {step7_root} \\\n"
        "  --generated-file /REAL/PATH/step7_generated_final_audited.csv \\\n"
        "  --train-file /REAL/PATH/step7_train_reference_annotated.csv\n\n"
        "Suggested notebook call:\n"
        "outputs = run_step7_candidate_audit(\n"
        f"    step7_root={step7_root!r},\n"
        "    generated_file=\"/REAL/PATH/step7_generated_final_audited.csv\",\n"
        "    train_file=\"/REAL/PATH/step7_train_reference_annotated.csv\",\n"
        "    final_file=None,\n"
        "    show_figures=True,\n"
        ")\n"
    )
    return (
        f"Could not find required Step 7 {label}.\n\n"
        f"Accepted filenames:\n{candidate_msg}\n\n"
        f"Searched roots:\n{root_msg}"
        f"{inventory_msg}"
        f"{command_msg}"
    )


def discover_any(
    candidate_names: Sequence[str],
    roots: Sequence[str],
    direct_file: Optional[str] = None,
    required: bool = False,
    label: str = "file",
    inventory: Optional[pd.DataFrame] = None,
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
) -> Optional[Path]:
    """Resolve an input file by explicit path first, then controlled discovery."""
    direct = normalize_path(direct_file)
    if direct is not None:
        if direct.exists() and direct.is_file():
            return direct
        raise FileNotFoundError(
            f"Direct {label} path was provided but does not exist: {direct}\n"
            "Replace placeholder paths such as /path/to/... with a real CSV path."
        )

    searched_roots: List[str] = []
    for root_raw in roots:
        root = Path(root_raw).expanduser()
        searched_roots.append(str(root))
        if not root.exists():
            continue
        for match in _iter_candidate_matches(root, candidate_names, max_hits=1):
            return match

    if required:
        raise FileNotFoundError(
            build_missing_file_diagnostic(label, candidate_names, searched_roots, inventory, step7_root)
        )
    return None


def find_first_column(df: pd.DataFrame, aliases: Sequence[str], canonical_name: str, required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for alias in aliases:
        if alias in df.columns:
            return alias
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    if required:
        raise KeyError(
            f"Could not find a column for '{canonical_name}'. Tried aliases: {list(aliases)}. "
            f"Available columns: {list(df.columns)}"
        )
    return None


def maybe_rename_columns(df: pd.DataFrame, aliases: Dict[str, Sequence[str]], canonical_columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    rename_map: Dict[str, str] = {}
    for canonical in canonical_columns:
        if canonical in out.columns:
            continue
        if canonical not in aliases:
            continue
        found = find_first_column(out, aliases[canonical], canonical, required=False)
        if found is not None and found != canonical:
            rename_map[found] = canonical
    return out.rename(columns=rename_map)


def normalize_model_tag_value(value: object) -> object:
    if pd.isna(value):
        return value
    text = str(value).strip()
    key = text.lower().replace("-", "_").replace(" ", "_")
    return MODEL_TAG_ALIASES.get(key, text)


def normalize_model_tags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model_tag" in out.columns:
        out["model_tag"] = out["model_tag"].map(normalize_model_tag_value)
    return out


def as_numeric_clean(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def parse_boolish(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    text = series.astype(str).str.strip().str.lower()
    truthy = {"true", "1", "yes", "y", "selected", "final", "pass", "passed", "keep", "kept"}
    falsy = {"false", "0", "no", "n", "not_selected", "fail", "failed", "nan", "none", "", "remove"}
    return text.map(lambda x: True if x in truthy else (False if x in falsy else False))


def valid_sequence(seq: object) -> bool:
    if pd.isna(seq):
        return False
    s = str(seq).strip().upper()
    return len(s) > 0 and all(ch in AA20 for ch in s)


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def dataframe_with_panel(df: pd.DataFrame, figure: str, panel: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "panel", panel)
    out.insert(0, "figure", figure)
    return out


# =============================================================================
# Peptide descriptor and similarity helpers
# =============================================================================

def clean_sequence(seq: object) -> str:
    if pd.isna(seq):
        return ""
    return str(seq).strip().upper()


def peptide_length(seq: object) -> float:
    s = clean_sequence(seq)
    return float(len(s)) if s else np.nan


def peptide_net_charge(seq: object, histidine_charge: float = 0.10) -> float:
    """Approximate peptide net charge at pH 7 from side-chain counts.

    This fallback is meant only for plotting/audit recovery when descriptors are
    missing from source data. It should be replaced by the manuscript's official
    descriptor values when available.
    """
    s = clean_sequence(seq)
    if not s:
        return np.nan
    return float(s.count("K") + s.count("R") + histidine_charge * s.count("H") - s.count("D") - s.count("E"))


def peptide_hydrophobicity(seq: object) -> float:
    s = clean_sequence(seq)
    vals = [KD_SCALE[ch] for ch in s if ch in KD_SCALE]
    if not vals:
        return np.nan
    return float(np.mean(vals))


def peptide_entropy(seq: object) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    counts = np.array([s.count(aa) for aa in AA20], dtype=float)
    probs = counts[counts > 0] / len(s)
    return float(-(probs * np.log2(probs)).sum())


def normalized_levenshtein_similarity(a: str, b: str) -> float:
    """Return 1 - edit_distance/max_length using a memory-efficient DP."""
    a = clean_sequence(a)
    b = clean_sequence(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            insert = current[j - 1] + 1
            delete = previous[j] + 1
            substitute = previous[j - 1] + (ca != cb)
            current.append(min(insert, delete, substitute))
        previous = current
    dist = previous[-1]
    return float(1.0 - dist / max(len(a), len(b)))


def compute_nearest_similarity(
    sequences: Sequence[str],
    train_sequences: Sequence[str],
    max_pairs: int = 3_000_000,
) -> Tuple[pd.Series, str]:
    n_pairs = len(sequences) * len(train_sequences)
    if n_pairs > max_pairs:
        return pd.Series([np.nan] * len(sequences), dtype=float), (
            f"not_computed_pair_budget_exceeded:{n_pairs}>{max_pairs}"
        )
    train_list = [clean_sequence(s) for s in train_sequences if clean_sequence(s)]
    values: List[float] = []
    for seq in sequences:
        s = clean_sequence(seq)
        if not s or not train_list:
            values.append(np.nan)
            continue
        values.append(max(normalized_levenshtein_similarity(s, t) for t in train_list))
    return pd.Series(values, dtype=float), "computed_normalized_levenshtein_similarity"


# =============================================================================
# Data preparation
# =============================================================================

def add_or_compute_descriptors(df: pd.DataFrame, cfg: Step7Config, role: str, warnings_list: List[str]) -> pd.DataFrame:
    out = df.copy()
    if "sequence" not in out.columns:
        return out

    if cfg.compute_missing_descriptors:
        if "length" not in out.columns or out["length"].isna().all():
            out["length"] = out["sequence"].map(peptide_length)
            warnings_list.append(f"{role}: length was missing and was computed from sequence.")
        if "net_charge" not in out.columns or out["net_charge"].isna().all():
            out["net_charge"] = out["sequence"].map(lambda s: peptide_net_charge(s, cfg.histidine_charge))
            warnings_list.append(f"{role}: net_charge was missing and was approximated from residue counts.")
        if "hydrophobicity" not in out.columns or out["hydrophobicity"].isna().all():
            out["hydrophobicity"] = out["sequence"].map(peptide_hydrophobicity)
            warnings_list.append(f"{role}: hydrophobicity was missing and was computed using the Kyte-Doolittle mean.")
        if "entropy" not in out.columns or out["entropy"].isna().all():
            out["entropy"] = out["sequence"].map(peptide_entropy)
            warnings_list.append(f"{role}: entropy was missing and was computed as Shannon entropy over AA20.")

    for col in DESCRIPTORS:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    return out


def prepare_generated_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, COLUMN_ALIASES, COLUMN_ALIASES.keys())

    if "sequence" not in out.columns:
        raise KeyError(
            "Generated/final audited table is missing a sequence column. "
            f"Available columns: {list(raw.columns)}"
        )
    if "model_tag" not in out.columns:
        out["model_tag"] = "full_cvae"
        warnings_list.append("generated: model_tag was missing; all rows were assigned to full_cvae/OncoPep.")

    out = normalize_model_tags(out)
    out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "generated", warnings_list)

    for col in ["nn_similarity_train", "candidate_rank", "composite_score"]:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])

    if "candidate_id" not in out.columns:
        out["candidate_id"] = [f"pep_{i+1:05d}" for i in range(len(out))]
        warnings_list.append("generated: candidate_id was missing and sequential IDs were assigned.")

    if "is_final_candidate" in out.columns:
        out["is_final_candidate"] = parse_boolish(out["is_final_candidate"])
    else:
        out["is_final_candidate"] = False

    out["valid_aa_sequence"] = out["sequence"].map(valid_sequence)
    out["duplicate_sequence_within_input"] = out.duplicated("sequence", keep=False)
    out["recognized_model"] = out["model_tag"].isin(MODEL_ORDER)
    return out


def prepare_train_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, TRAIN_ALIASES, TRAIN_ALIASES.keys())
    if "sequence" in out.columns:
        out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "train_reference", warnings_list)

    required = list(DESCRIPTORS)
    missing = [col for col in required if col not in out.columns]
    if missing:
        raise KeyError(
            "Training-reference table is missing descriptor columns and they could not be computed: "
            + ", ".join(missing)
            + f". Available columns: {list(raw.columns)}"
        )

    for col in DESCRIPTORS:
        out[col] = as_numeric_clean(out[col])
    out["model_tag"] = "train_ref"
    return out


def compute_reference_ranges(train_df: pd.DataFrame, cfg: Step7Config) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for descriptor in DESCRIPTORS:
        vals = train_df[descriptor].dropna().to_numpy(float)
        if len(vals) == 0:
            raise ValueError(f"Training-reference descriptor '{descriptor}' has no finite values.")
        rows.append(
            {
                "descriptor": descriptor,
                "n_train": int(len(vals)),
                "min": float(np.min(vals)),
                "q01": float(np.quantile(vals, 0.01)),
                "q05": float(np.quantile(vals, 0.05)),
                "q25": float(np.quantile(vals, 0.25)),
                "median": float(np.quantile(vals, 0.50)),
                "q75": float(np.quantile(vals, 0.75)),
                "q95": float(np.quantile(vals, 0.95)),
                "q99": float(np.quantile(vals, 0.99)),
                "max": float(np.max(vals)),
                "range_low": float(np.quantile(vals, cfg.reference_quantile_low)),
                "range_high": float(np.quantile(vals, cfg.reference_quantile_high)),
                "range_definition": f"training quantile [{cfg.reference_quantile_low}, {cfg.reference_quantile_high}]",
            }
        )
    return pd.DataFrame(rows)


def add_computed_audit_columns(
    generated_df: pd.DataFrame,
    train_df: pd.DataFrame,
    reference_ranges: pd.DataFrame,
    cfg: Step7Config,
    warnings_list: List[str],
) -> pd.DataFrame:
    out = generated_df.copy()

    if "sequence" in train_df.columns:
        train_sequences = set(train_df["sequence"].dropna().astype(str).str.strip().str.upper())
        out["computed_exact_novel_vs_train"] = ~out["sequence"].isin(train_sequences)
    else:
        out["computed_exact_novel_vs_train"] = np.nan
        warnings_list.append("Exact novelty could not be computed because training sequence column is absent.")

    if "exact_novel_vs_train" in out.columns:
        out["exact_novel_vs_train"] = parse_boolish(out["exact_novel_vs_train"])
        out["audit_exact_novel_vs_train"] = out["exact_novel_vs_train"]
    else:
        out["audit_exact_novel_vs_train"] = out["computed_exact_novel_vs_train"]

    if "nn_similarity_train" not in out.columns or out["nn_similarity_train"].isna().all():
        if cfg.compute_nn_if_missing and "sequence" in train_df.columns:
            nn_values, nn_status = compute_nearest_similarity(
                sequences=out["sequence"].astype(str).tolist(),
                train_sequences=train_df["sequence"].dropna().astype(str).tolist(),
                max_pairs=cfg.max_nn_pairs,
            )
            out["nn_similarity_train"] = nn_values
            out["nn_similarity_train_source"] = nn_status
            warnings_list.append(f"generated: nn_similarity_train was missing; status={nn_status}.")
        else:
            out["nn_similarity_train"] = np.nan
            out["nn_similarity_train_source"] = "missing"
            warnings_list.append("generated: nn_similarity_train is missing and was not computed.")
    else:
        out["nn_similarity_train"] = as_numeric_clean(out["nn_similarity_train"])
        out["nn_similarity_train_source"] = "input_column"

    for _, row in reference_ranges.iterrows():
        descriptor = str(row["descriptor"])
        low = float(row["range_low"])
        high = float(row["range_high"])
        out[f"within_ref_{descriptor}"] = out[descriptor].between(low, high, inclusive="both")

    within_cols = [f"within_ref_{d}" for d in DESCRIPTORS]
    out["audit_within_reference_all_descriptors"] = out[within_cols].all(axis=1)
    out["audit_low_similarity"] = out["nn_similarity_train"] < cfg.similarity_low_risk_threshold
    out["audit_valid_sequence"] = out["valid_aa_sequence"].astype(bool)
    out["audit_nonduplicate_sequence"] = ~out["duplicate_sequence_within_input"].astype(bool)

    audit_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_within_reference_all_descriptors",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
    ]
    usable_cols = [c for c in audit_cols if not out[c].isna().all()]
    out["audit_pass_computed"] = out[usable_cols].fillna(True).all(axis=1)
    return out


def prepare_final_file_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = prepare_generated_df(raw, cfg, warnings_list)
    out["is_final_candidate"] = True
    return out


def infer_final_candidates(
    audited_df: pd.DataFrame,
    explicit_final_df: Optional[pd.DataFrame],
    cfg: Step7Config,
    warnings_list: List[str],
) -> Tuple[pd.DataFrame, str]:
    if explicit_final_df is not None and not explicit_final_df.empty:
        final_sequences = set(explicit_final_df["sequence"].dropna().astype(str))
        if final_sequences:
            matched = audited_df[audited_df["sequence"].isin(final_sequences)].copy()
            if not matched.empty:
                return matched, "explicit separate final-candidate file matched by sequence"
        return explicit_final_df.copy(), "explicit separate final-candidate file"

    out = audited_df.copy()
    if "is_final_candidate" in out.columns and out["is_final_candidate"].any():
        return out.loc[out["is_final_candidate"]].copy(), "explicit final-candidate flag"

    oncopep = out.loc[out["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = out.copy()
        warnings_list.append("No full_cvae rows found; final-candidate fallback used all generated rows.")

    if "candidate_rank" in oncopep.columns and oncopep["candidate_rank"].notna().any():
        top = oncopep.sort_values("candidate_rank", ascending=True).head(cfg.top_n_final).copy()
        return top, f"fallback: top {cfg.top_n_final} rows by candidate_rank"

    if "composite_score" in oncopep.columns and oncopep["composite_score"].notna().any():
        top = oncopep.sort_values("composite_score", ascending=False).head(cfg.top_n_final).copy()
        return top, f"fallback: top {cfg.top_n_final} rows by composite_score"

    passing = oncopep.loc[oncopep["audit_pass_computed"]].copy()
    if not passing.empty:
        top = passing.head(cfg.top_n_final).copy()
        return top, f"fallback: first {min(cfg.top_n_final, len(top))} OncoPep rows passing computed audit"

    top = oncopep.head(cfg.top_n_final).copy()
    return top, f"fallback: first {min(cfg.top_n_final, len(top))} OncoPep rows because no explicit final-candidate flag was present"


def prepare_funnel_df(
    audited_df: pd.DataFrame,
    final_df: pd.DataFrame,
    explicit_funnel_df: Optional[pd.DataFrame],
    cfg: Step7Config,
) -> pd.DataFrame:
    if explicit_funnel_df is not None and not explicit_funnel_df.empty:
        funnel = explicit_funnel_df.copy()
        if "stage" not in funnel.columns:
            for col in ["audit_stage", "filter_stage", "step", "name"]:
                if col in funnel.columns:
                    funnel = funnel.rename(columns={col: "stage"})
                    break
        if "n" not in funnel.columns:
            for col in ["count", "n_sequences", "n_candidates", "value"]:
                if col in funnel.columns:
                    funnel = funnel.rename(columns={col: "n"})
                    break
        if "stage" in funnel.columns and "n" in funnel.columns:
            funnel["n"] = as_numeric_clean(funnel["n"]).astype("Int64")
            funnel["source"] = "input_funnel_file"
            return funnel[["stage", "n", "source"]]

    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()
    rows = [
        ("OncoPep evaluated outputs", len(oncopep)),
        ("Valid amino-acid sequences", int(oncopep["audit_valid_sequence"].sum())),
        (
            "Exact-novel vs training",
            int(oncopep["audit_exact_novel_vs_train"].fillna(False).sum())
            if not oncopep["audit_exact_novel_vs_train"].isna().all()
            else np.nan,
        ),
        ("Within training descriptor range", int(oncopep["audit_within_reference_all_descriptors"].sum())),
        (f"NN similarity < {cfg.similarity_low_risk_threshold:.2f}", int(oncopep["audit_low_similarity"].fillna(False).sum())),
        ("Final candidates", len(final_df)),
    ]
    return pd.DataFrame([{"stage": stage, "n": n, "source": "computed_from_audited_table"} for stage, n in rows])


def load_inputs(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    warnings_list: List[str] = []
    roots = tuple(str(Path(r).expanduser()) for r in cfg.input_roots)
    inventory = inventory_csv_files(roots, cfg.inventory_max_files)
    if not inventory.empty:
        write_csv(inventory, cfg.reports_dir / "step7_input_inventory.csv")

    generated_path = discover_any(
        cfg.generated_candidates,
        roots,
        direct_file=cfg.generated_file,
        required=True,
        label="generated/final audited table",
        inventory=inventory,
        step7_root=str(cfg.root),
    )
    train_path = discover_any(
        cfg.train_candidates,
        roots,
        direct_file=cfg.train_file,
        required=True,
        label="training-reference table",
        inventory=inventory,
        step7_root=str(cfg.root),
    )
    final_path = discover_any(
        cfg.final_candidates,
        roots,
        direct_file=cfg.final_file,
        required=False,
        label="separate final-candidate table",
        inventory=inventory,
        step7_root=str(cfg.root),
    )
    summary_path = discover_any(
        cfg.summary_candidates,
        roots,
        direct_file=cfg.summary_file,
        required=False,
        label="model summary table",
        inventory=inventory,
        step7_root=str(cfg.root),
    )
    funnel_path = discover_any(
        cfg.funnel_candidates,
        roots,
        direct_file=cfg.funnel_file,
        required=False,
        label="candidate funnel table",
        inventory=inventory,
        step7_root=str(cfg.root),
    )

    assert generated_path is not None and train_path is not None

    generated_raw = pd.read_csv(generated_path)
    train_raw = pd.read_csv(train_path)
    final_raw = pd.read_csv(final_path) if final_path is not None else None
    summary_df = pd.read_csv(summary_path) if summary_path is not None else None
    funnel_df = pd.read_csv(funnel_path) if funnel_path is not None else None

    generated_df = prepare_generated_df(generated_raw, cfg, warnings_list)
    train_df = prepare_train_df(train_raw, cfg, warnings_list)
    explicit_final_df = prepare_final_file_df(final_raw, cfg, warnings_list) if final_raw is not None else None

    reference_ranges = compute_reference_ranges(train_df, cfg)
    audited_df = add_computed_audit_columns(generated_df, train_df, reference_ranges, cfg, warnings_list)

    if explicit_final_df is not None:
        explicit_final_df = add_computed_audit_columns(explicit_final_df, train_df, reference_ranges, cfg, warnings_list)

    final_df, final_rule = infer_final_candidates(audited_df, explicit_final_df, cfg, warnings_list)
    if "is_final_candidate" in audited_df.columns:
        audited_df.loc[audited_df["candidate_id"].isin(final_df["candidate_id"]), "is_final_candidate"] = True
    funnel_final = prepare_funnel_df(audited_df, final_df, funnel_df, cfg)

    inputs = {
        "generated_path": str(generated_path),
        "train_path": str(train_path),
        "final_path": str(final_path) if final_path else None,
        "summary_path": str(summary_path) if summary_path else None,
        "funnel_path": str(funnel_path) if funnel_path else None,
        "input_inventory": str(cfg.reports_dir / "step7_input_inventory.csv"),
        "generated_sha256": sha256_file(generated_path),
        "train_sha256": sha256_file(train_path),
        "final_sha256": sha256_file(final_path),
        "summary_sha256": sha256_file(summary_path),
        "funnel_sha256": sha256_file(funnel_path),
        "final_candidate_rule": final_rule,
        "warnings_from_preparation": warnings_list,
        "nn_metric_if_computed": cfg.nn_metric_if_computed,
    }

    return {
        "generated_df": generated_df,
        "train_df": train_df,
        "reference_ranges": reference_ranges,
        "audited_df": audited_df,
        "final_df": final_df,
        "funnel_df": funnel_final,
        "summary_df": summary_df,
        "inventory": inventory,
        "inputs": inputs,
    }



def print_resolved_inputs(inputs: Dict[str, object]) -> None:
    """Print exact files used so notebook/terminal runs are auditable."""
    print("\nResolved OncoPep Step 7 input files")
    print("-" * 44)
    for key in ["generated_path", "train_path", "final_path", "summary_path", "funnel_path", "input_inventory"]:
        print(f"{key}: {inputs.get(key)}")
    print(f"final_candidate_rule: {inputs.get('final_candidate_rule')}")



# =============================================================================
# Plotting helpers
# =============================================================================

def save_figure(fig: plt.Figure, out_base: Path, cfg: Step7Config) -> List[str]:
    outputs: List[str] = []
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs.append(str(p))
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs.append(str(p))
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        try:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, pil_kwargs={"compression": cfg.tiff_compression})
        except TypeError:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff)
        outputs.append(str(p))
    if cfg.show_figures:
        plt.show()
    else:
        plt.close(fig)
    return outputs


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.8)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.8)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.9)
        ax.spines[side].set_color(PLOS["dark"])
    ax.tick_params(axis="both", width=0.8, length=4)


def add_panel_label(ax: plt.Axes, label: str, cfg: Step7Config, x: float = -0.08, y: float = 1.03) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=cfg.panel_label_size,
        fontweight="bold",
        color=PLOS["dark"],
        clip_on=False,
    )


def model_legend_handles(include_train: bool = False) -> List[Patch]:
    handles = []
    if include_train:
        handles.append(Patch(facecolor=MODEL_COLORS["train_ref"], edgecolor=MODEL_EDGES["train_ref"], label=MODEL_LABELS["train_ref"]))
    for tag in MODEL_ORDER:
        handles.append(Patch(facecolor=MODEL_COLORS[tag], edgecolor=MODEL_EDGES[tag], label=MODEL_LABELS[tag]))
    return handles


def add_bottom_legend(fig: plt.Figure, include_train: bool, y: float = 0.02, ncol: Optional[int] = None) -> None:
    handles = model_legend_handles(include_train)
    if ncol is None:
        ncol = len(handles)
    legend = fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        ncol=ncol,
        frameon=True,
        columnspacing=1.1,
        handletextpad=0.45,
        borderpad=0.45,
    )
    legend.get_frame().set_facecolor("#FAFAFA")
    legend.get_frame().set_edgecolor("#CFCFCF")
    legend.get_frame().set_linewidth(0.8)


# =============================================================================
# Figure builders
# =============================================================================

def build_s12_candidate_audit(final_df: pd.DataFrame, funnel_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    fig = plt.figure(figsize=(14.2, 6.6))
    gs = GridSpec(1, 2, width_ratios=[0.95, 1.35], figure=fig)

    ax_a = fig.add_subplot(gs[0, 0])
    funnel = funnel_df.copy()
    funnel["n"] = as_numeric_clean(funnel["n"])
    funnel = funnel.dropna(subset=["n"]).copy()
    funnel["n"] = funnel["n"].astype(int)
    y = np.arange(len(funnel))[::-1]
    bars = ax_a.barh(y, funnel["n"], color=PLOS["coral"], edgecolor="#B84F42", linewidth=0.8, zorder=3)
    ax_a.set_yticks(y)
    ax_a.set_yticklabels(funnel["stage"])
    ax_a.set_xlabel("Number of sequences")
    ax_a.set_title("Selection funnel", pad=10)
    style_axis(ax_a, "x")
    add_panel_label(ax_a, "A", cfg)
    max_n = max(float(funnel["n"].max()), 1.0)
    for bar, n in zip(bars, funnel["n"]):
        ax_a.text(bar.get_width() + max_n * 0.015, bar.get_y() + bar.get_height() / 2, f"{int(n):,}", va="center", ha="left", fontsize=9.8)

    ax_b = fig.add_subplot(gs[0, 1])
    audit_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_within_reference_all_descriptors",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
        "audit_pass_computed",
    ]
    labels = [
        "Valid AA",
        "Exact novel",
        "Within ref.",
        f"NN < {cfg.similarity_low_risk_threshold:.2f}",
        "Nondup.",
        "Overall",
    ]

    plot_df = final_df.copy()
    if "candidate_rank" in plot_df.columns and plot_df["candidate_rank"].notna().any():
        plot_df = plot_df.sort_values("candidate_rank")
    elif "composite_score" in plot_df.columns and plot_df["composite_score"].notna().any():
        plot_df = plot_df.sort_values("composite_score", ascending=False)

    title = "Final candidate audit matrix"
    if len(plot_df) > cfg.max_heatmap_candidates:
        plot_df = plot_df.head(cfg.max_heatmap_candidates).copy()
        title = f"Candidate audit matrix (top {cfg.max_heatmap_candidates})"
    ax_b.set_title(title, pad=10)

    if plot_df.empty:
        mat = np.zeros((1, len(audit_cols)))
        ylabels = ["No candidates"]
    else:
        mat_df = plot_df[audit_cols].copy()
        for col in audit_cols:
            mat_df[col] = mat_df[col].fillna(False).astype(bool).astype(int)
        mat = mat_df.to_numpy()
        ylabels = plot_df["candidate_id"].astype(str).tolist()

    ax_b.imshow(mat, aspect="auto", cmap="Greys", vmin=0, vmax=1)
    ax_b.set_xticks(np.arange(len(audit_cols)))
    ax_b.set_xticklabels(labels, rotation=35, ha="right")
    ax_b.set_yticks(np.arange(len(ylabels)))
    ax_b.set_yticklabels(ylabels)
    ax_b.tick_params(axis="both", length=0)
    for spine in ax_b.spines.values():
        spine.set_visible(False)
    ax_b.set_xlabel("Audit criterion")
    add_panel_label(ax_b, "B", cfg)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            value = int(mat[i, j])
            ax_b.text(j, i, "✓" if value else "×", ha="center", va="center", fontsize=9.5, color=PLOS["dark"] if value else "#8A1F1F")

    fig.subplots_adjust(left=0.12, right=0.985, top=0.90, bottom=0.19, wspace=0.25)

    outputs: List[str] = []
    panel_a = funnel.copy()
    panel_b_cols = ["candidate_id", "sequence"] + [c for c in audit_cols if c in plot_df.columns]
    panel_b = plot_df[panel_b_cols].copy() if not plot_df.empty else pd.DataFrame(columns=panel_b_cols)
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "Supplementary_Figure_S12_panel_a_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S12_panel_b_source_data.csv"))
    outputs.append(
        write_csv(
            pd.concat(
                [
                    dataframe_with_panel(panel_a, "Supplementary Figure S12", "A"),
                    dataframe_with_panel(panel_b, "Supplementary Figure S12", "B"),
                ],
                ignore_index=True,
                sort=False,
            ),
            cfg.source_data_dir / "Supplementary_Figure_S12_source_data_all_panels.csv",
        )
    )

    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S12_candidate_audit", cfg)
    return figs, outputs


def build_s13_descriptor_support(final_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    specs = [
        ("length", "Length support", "Peptide length", "A"),
        ("net_charge", "Net-charge support", "Net charge at pH 7", "B"),
        ("hydrophobicity", "Hydrophobicity support", "Mean hydrophobicity", "C"),
        ("entropy", "Entropy support", "Shannon entropy", "D"),
    ]
    fig = plt.figure(figsize=(14.2, 9.1))
    gs = GridSpec(2, 2, figure=fig)
    rng = np.random.default_rng(cfg.seed)
    all_frames: List[pd.DataFrame] = []
    outputs: List[str] = []

    for idx, (descriptor, title, xlabel, panel) in enumerate(specs):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        train_vals = train_df[descriptor].dropna().to_numpy(float)
        candidate_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])

        ax.hist(train_vals, bins=35, density=True, color=PLOS["light"], edgecolor="white", linewidth=0.4, alpha=0.90, zorder=1)

        if len(candidate_vals):
            y_jitter = rng.normal(0.015, 0.006, len(candidate_vals))
            ax.scatter(candidate_vals, y_jitter, s=34, color=PLOS["coral"], edgecolor="#B84F42", linewidth=0.6, zorder=4)

        row = reference_ranges.loc[reference_ranges["descriptor"] == descriptor].iloc[0]
        low = float(row["range_low"])
        high = float(row["range_high"])
        ax.axvspan(low, high, color=PLOS["coral"], alpha=0.08, zorder=0)
        ax.axvline(low, color="#B84F42", linewidth=1.1, linestyle="--", zorder=3)
        ax.axvline(high, color="#B84F42", linewidth=1.1, linestyle="--", zorder=3)

        ax.set_title(title, pad=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Training density")
        style_axis(ax, "y")
        add_panel_label(ax, panel, cfg)

        train_panel = pd.DataFrame({"group": "train_ref", "candidate_id": np.nan, "descriptor": descriptor, "value": train_vals})
        cand_cols = ["candidate_id", "sequence", descriptor]
        cand_panel = final_df[cand_cols].copy().rename(columns={descriptor: "value"}) if set(cand_cols).issubset(final_df.columns) else pd.DataFrame(columns=["candidate_id", "sequence", "value"])
        cand_panel["group"] = "final_candidate"
        cand_panel["descriptor"] = descriptor
        cand_panel["range_low"] = low
        cand_panel["range_high"] = high
        cand_panel["within_range"] = cand_panel["value"].between(low, high, inclusive="both") if not cand_panel.empty else []
        panel_df = pd.concat([train_panel, cand_panel], ignore_index=True, sort=False)
        outputs.append(write_csv(panel_df, cfg.source_data_dir / f"Supplementary_Figure_S13_panel_{panel.lower()}_source_data.csv"))
        all_frames.append(dataframe_with_panel(panel_df, "Supplementary Figure S13", panel))

    handles = [
        Patch(facecolor=PLOS["light"], edgecolor="white", label="Training reference"),
        Patch(facecolor=PLOS["coral"], edgecolor="#B84F42", label="Final candidates"),
    ]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=2, frameon=True)
    fig.subplots_adjust(left=0.07, right=0.985, top=0.93, bottom=0.14, wspace=0.22, hspace=0.34)

    outputs.append(write_csv(pd.concat(all_frames, ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S13_source_data_all_panels.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S13_descriptor_support", cfg)
    return figs, outputs


def build_s14_similarity_tail(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    fig = plt.figure(figsize=(14.2, 6.4))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.15])

    ax_a = fig.add_subplot(gs[0, 0])
    vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    if len(vals) > 0:
        upper = max(0.5, float(vals.max()) + 0.03)
        ax_a.hist(vals, bins=np.linspace(0, upper, 22), color=PLOS["coral"], edgecolor="white", linewidth=0.5, alpha=0.92, zorder=2)
        for thr in cfg.tail_thresholds:
            ax_a.axvline(thr, color=PLOS["dark"], linestyle="--", linewidth=0.9, alpha=0.75, zorder=3)
        ax_a.text(0.98, 0.95, f"n = {len(vals)}", transform=ax_a.transAxes, ha="right", va="top", fontsize=10.5)
    else:
        ax_a.text(0.5, 0.5, "NN similarity unavailable", transform=ax_a.transAxes, ha="center", va="center", fontsize=11)
    ax_a.set_xlabel("Nearest-neighbor similarity to training set")
    ax_a.set_ylabel("Final candidate count")
    ax_a.set_title("Final-candidate similarity profile", pad=10)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "A", cfg)

    ax_b = fig.add_subplot(gs[0, 1])
    model_tags = [m for m in MODEL_ORDER if m in set(audited_df["model_tag"])]
    thresholds = list(cfg.tail_thresholds)
    x_base = np.arange(len(thresholds), dtype=float) * 1.2
    bar_w = 0.14
    gap = 0.025
    tail_rows: List[Dict[str, object]] = []
    for i, model in enumerate(model_tags):
        model_vals = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        fracs = [float(np.mean(model_vals >= thr)) if len(model_vals) else np.nan for thr in thresholds]
        offset = (i - (len(model_tags) - 1) / 2.0) * (bar_w + gap)
        ax_b.bar(
            x_base + offset,
            fracs,
            width=bar_w,
            color=MODEL_COLORS[model],
            edgecolor=MODEL_EDGES[model],
            linewidth=0.8,
            alpha=0.96,
            zorder=3,
        )
        for thr, frac in zip(thresholds, fracs):
            tail_rows.append(
                {
                    "model_tag": model,
                    "model_label": MODEL_LABELS[model],
                    "threshold": thr,
                    "comparison": ">= threshold",
                    "n": int(len(model_vals)),
                    "fraction_at_or_above_threshold": frac,
                }
            )

    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels([f">={thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction of evaluated outputs")
    ax_b.set_ylim(0.0, 1.02)
    ax_b.set_title("Generator-context tail risk", pad=10)
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "B", cfg)

    fig.subplots_adjust(left=0.075, right=0.985, top=0.90, bottom=0.22, wspace=0.20)
    add_bottom_legend(fig, include_train=False, y=0.035)

    panel_a_cols = ["candidate_id", "sequence", "nn_similarity_train"]
    panel_a = final_df[panel_a_cols].copy() if set(panel_a_cols).issubset(final_df.columns) else pd.DataFrame(columns=panel_a_cols)
    panel_b = pd.DataFrame(tail_rows)

    outputs: List[str] = []
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "Supplementary_Figure_S14_panel_a_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S14_panel_b_source_data.csv"))
    outputs.append(
        write_csv(
            pd.concat(
                [
                    dataframe_with_panel(panel_a, "Supplementary Figure S14", "A"),
                    dataframe_with_panel(panel_b, "Supplementary Figure S14", "B"),
                ],
                ignore_index=True,
                sort=False,
            ),
            cfg.source_data_dir / "Supplementary_Figure_S14_source_data_all_panels.csv",
        )
    )

    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S14_similarity_tail_risk", cfg)
    return figs, outputs


# =============================================================================
# Reports and source-data exports
# =============================================================================

def write_core_source_tables(
    audited_df: pd.DataFrame,
    final_df: pd.DataFrame,
    reference_ranges: pd.DataFrame,
    funnel_df: pd.DataFrame,
    cfg: Step7Config,
) -> List[str]:
    outputs = []
    outputs.append(write_csv(audited_df, cfg.source_data_dir / "step7_all_generated_audited_with_computed_flags.csv"))
    outputs.append(write_csv(final_df, cfg.source_data_dir / "step7_final_candidates_audit_table.csv"))
    outputs.append(write_csv(reference_ranges, cfg.source_data_dir / "step7_training_reference_descriptor_ranges.csv"))
    outputs.append(write_csv(funnel_df, cfg.source_data_dir / "step7_candidate_funnel_counts.csv"))

    tail_rows = []
    for model in MODEL_ORDER:
        vals = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        for threshold in cfg.tail_thresholds:
            tail_rows.append(
                {
                    "scope": "all_evaluated_outputs",
                    "model_tag": model,
                    "model_label": MODEL_LABELS[model],
                    "threshold": threshold,
                    "comparison": ">= threshold",
                    "n": int(len(vals)),
                    "fraction_at_or_above_threshold": float(np.mean(vals >= threshold)) if len(vals) else np.nan,
                }
            )
    final_vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    for threshold in cfg.tail_thresholds:
        tail_rows.append(
            {
                "scope": "final_candidates",
                "model_tag": "full_cvae",
                "model_label": "OncoPep final candidates",
                "threshold": threshold,
                "comparison": ">= threshold",
                "n": int(len(final_vals)),
                "fraction_at_or_above_threshold": float(np.mean(final_vals >= threshold)) if len(final_vals) else np.nan,
            }
        )
    outputs.append(write_csv(pd.DataFrame(tail_rows), cfg.source_data_dir / "step7_similarity_tail_risk_summary.csv"))
    return outputs


def build_readiness_report(
    audited_df: pd.DataFrame,
    final_df: pd.DataFrame,
    inputs: Dict[str, object],
    cfg: Step7Config,
    figures: Optional[Sequence[str]] = None,
    source_data: Optional[Sequence[str]] = None,
) -> Tuple[str, Dict[str, object]]:
    issues: List[str] = []
    warnings_list: List[str] = list(inputs.get("warnings_from_preparation", []))

    if final_df.empty:
        issues.append("No final candidates were identified.")
    if "explicit" not in str(inputs.get("final_candidate_rule", "")):
        warnings_list.append("No explicit final-candidate flag/file was detected; final candidates were inferred by fallback logic.")
    if audited_df["audit_exact_novel_vs_train"].isna().all():
        issues.append("Exact novelty could not be computed or read because no training sequence column/novelty flag was available.")
    if audited_df["nn_similarity_train"].isna().all():
        issues.append("Nearest-neighbor similarity is unavailable for all rows; S14 tail-risk interpretation is not ready.")
    elif audited_df["nn_similarity_train"].isna().any():
        warnings_list.append("Some nearest-neighbor similarity values are missing.")
    if not audited_df["recognized_model"].all():
        warnings_list.append("Some rows have unrecognized model tags and may be excluded from contextual model summaries.")
    if not final_df.empty and final_df["duplicate_sequence_within_input"].any():
        issues.append("At least one final candidate is duplicated within the generated input table.")
    if not final_df.empty and not final_df["valid_aa_sequence"].all():
        issues.append("At least one final candidate contains invalid or non-standard amino-acid characters.")
    if not final_df.empty and "audit_pass_computed" in final_df.columns and not final_df["audit_pass_computed"].all():
        warnings_list.append("At least one final candidate does not pass all computed audit criteria; inspect S12B and final audit table.")

    expected_figure_stems = [
        "Supplementary_Figure_S12_candidate_audit",
        "Supplementary_Figure_S13_descriptor_support",
        "Supplementary_Figure_S14_similarity_tail_risk",
    ]
    expected_source_files = [
        "Supplementary_Figure_S12_source_data_all_panels.csv",
        "Supplementary_Figure_S12_panel_a_source_data.csv",
        "Supplementary_Figure_S12_panel_b_source_data.csv",
        "Supplementary_Figure_S13_source_data_all_panels.csv",
        "Supplementary_Figure_S13_panel_a_source_data.csv",
        "Supplementary_Figure_S13_panel_b_source_data.csv",
        "Supplementary_Figure_S13_panel_c_source_data.csv",
        "Supplementary_Figure_S13_panel_d_source_data.csv",
        "Supplementary_Figure_S14_source_data_all_panels.csv",
        "Supplementary_Figure_S14_panel_a_source_data.csv",
        "Supplementary_Figure_S14_panel_b_source_data.csv",
        "step7_all_generated_audited_with_computed_flags.csv",
        "step7_final_candidates_audit_table.csv",
        "step7_training_reference_descriptor_ranges.csv",
        "step7_candidate_funnel_counts.csv",
        "step7_similarity_tail_risk_summary.csv",
    ]

    figure_penalties = 0
    for stem in expected_figure_stems:
        for ext in [".pdf", ".png", ".tiff"]:
            if not (cfg.supplementary_dir / f"{stem}{ext}").exists():
                figure_penalties += 5
    if final_df.empty:
        figure_penalties += 20
    figure_readiness = max(0, 100 - figure_penalties)

    source_penalties = 0
    for fname in expected_source_files:
        if not (cfg.source_data_dir / fname).exists():
            source_penalties += 4
    if audited_df.empty or final_df.empty:
        source_penalties += 20
    source_data_readiness = max(0, 100 - source_penalties)

    reproducibility_penalties = 0
    for key in ["generated_path", "train_path", "generated_sha256", "train_sha256", "input_inventory"]:
        if not inputs.get(key):
            reproducibility_penalties += 6
    if str(inputs.get("generated_path", "")).startswith("/path/to") or str(inputs.get("train_path", "")).startswith("/path/to"):
        reproducibility_penalties += 20
    reproducibility_readiness = max(0, 100 - reproducibility_penalties)

    scientific_penalties = 12 * len(issues) + 4 * len(warnings_list)
    scientific_readiness = max(0, 100 - scientific_penalties)
    overall = int(round(np.mean([figure_readiness, source_data_readiness, reproducibility_readiness, scientific_readiness])))

    result = {
        "figure_package_readiness_score": int(figure_readiness),
        "source_data_readiness_score": int(source_data_readiness),
        "reproducibility_readiness_score": int(reproducibility_readiness),
        "scientific_audit_readiness_score": int(scientific_readiness),
        "overall_step7_readiness_score": int(overall),
        "ready_threshold": 95,
        "is_ready": overall >= 95,
        "n_all_rows": int(len(audited_df)),
        "n_final_candidates": int(len(final_df)),
        "issues": issues,
        "warnings": warnings_list,
        "required_source_tables": expected_source_files,
        "final_candidate_rule": inputs.get("final_candidate_rule"),
    }

    lines = [
        "# OncoPep Step 7 readiness report",
        "",
        f"- Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        f"- Figure package readiness score: **{figure_readiness}/100**",
        f"- Source-data readiness score: **{source_data_readiness}/100**",
        f"- Reproducibility readiness score: **{reproducibility_readiness}/100**",
        f"- Scientific audit readiness score: **{scientific_readiness}/100**",
        f"- Overall Step 7 readiness score: **{overall}/100**",
        "- Ready threshold: **95/100**",
        f"- Final candidate rule: `{inputs.get('final_candidate_rule')}`",
        f"- Number of evaluated rows: {len(audited_df):,}",
        f"- Number of final candidates: {len(final_df):,}",
        "",
        "## Inputs actually used",
        "",
    ]
    for key, value in inputs.items():
        if key != "warnings_from_preparation":
            lines.append(f"- {key}: `{value}`")

    lines += ["", "## Issues blocking readiness", ""]
    lines += [f"- {issue}" for issue in issues] if issues else ["- None detected."]
    lines += ["", "## Warnings", ""]
    lines += [f"- {warning}" for warning in warnings_list] if warnings_list else ["- None detected."]
    lines += [
        "",
        "## What to fix if below 95/100",
        "",
        "- Provide an explicit final-candidate file or final-candidate flag if fallback selection was used.",
        "- Provide official descriptor columns instead of relying on fallback descriptor calculations.",
        "- Provide nearest-neighbor similarity values or reduce `max_nn_pairs`/input sizes only if recomputation is scientifically justified.",
        "- Confirm exact novelty denominator: training set versus full curated corpus.",
        "- Confirm S12-S14 numbering against the current supplementary manuscript.",
        "",
        "## Claim boundary",
        "",
        "Step 7 supports computational auditing and downstream prioritization only. It does not establish anticancer activity, receptor binding, selectivity, toxicity, mechanism, therapeutic efficacy, or clinical relevance.",
        "",
        "## Required checks before writing Results",
        "",
        "- Confirm the final candidate flag/file and final candidate count.",
        "- Confirm the nearest-neighbor similarity metric and denominator.",
        "- Confirm whether reference-range plausibility uses min-max or quantile-bounded training ranges.",
        "- Confirm figure numbering against the current manuscript/SI.",
        "- Confirm that all plotted values are exported as source data.",
    ]

    path = cfg.reports_dir / "step7_readiness_report.md"
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path), result


def write_panel_mapping(cfg: Step7Config) -> str:
    mapping = pd.DataFrame(
        [
            {"figure": "S12 Fig", "panel": "A", "title": "Selection funnel", "source_data": "source_data/Supplementary_Figure_S12_panel_a_source_data.csv"},
            {"figure": "S12 Fig", "panel": "B", "title": "Final candidate audit matrix", "source_data": "source_data/Supplementary_Figure_S12_panel_b_source_data.csv"},
            {"figure": "S13 Fig", "panel": "A-D", "title": "Reference-space descriptor support", "source_data": "source_data/Supplementary_Figure_S13_panel_a-d_source_data.csv"},
            {"figure": "S14 Fig", "panel": "A", "title": "Final-candidate nearest-neighbor similarity", "source_data": "source_data/Supplementary_Figure_S14_panel_a_source_data.csv"},
            {"figure": "S14 Fig", "panel": "B", "title": "Generator-context high-similarity tail risk", "source_data": "source_data/Supplementary_Figure_S14_panel_b_source_data.csv"},
        ]
    )
    return write_csv(mapping, cfg.reports_dir / "step7_panel_source_data_mapping.csv")


def write_readme(cfg: Step7Config) -> str:
    text = f"""OncoPep Step 7 candidate-audit outputs
====================================

Purpose
-------
This folder contains Step 7 supplementary candidate-audit outputs for the
OncoPep PLOS Computational Biology manuscript.

Figures
-------
- S12 Fig: Final candidate audit and selection funnel.
- S13 Fig: Reference-space descriptor support.
- S14 Fig: Similarity-tail risk and nearest-neighbor audit.

Why these are supplementary
---------------------------
Step 5 already benchmarks generator-level quality, novelty, nearest-neighbor
similarity, and surrogate condition fidelity. Step 6 already covers latent-space
and sampling diagnostics. Step 7 is therefore kept as reviewer-supporting
supplementary evidence focused on final candidate audit, descriptor support, and
similarity-tail risk.

Claim boundary
--------------
These analyses are computational and hypothesis-generating. They do not
establish anticancer activity, receptor binding, selectivity, toxicity, stability,
mechanism, therapeutic efficacy, or clinical relevance.

Output root
-----------
{cfg.root}

Notebook example
----------------
from OncoPep_step7_PLOS_candidate_audit_v6_auto_ready import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root={str(cfg.root)!r},
    generated_file='/path/to/step7_generated_final_audited.csv',
    train_file='/path/to/step7_train_reference_annotated.csv',
    final_file=None,
    show_figures=True,
)
outputs

Terminal example
----------------
python OncoPep_step7_PLOS_candidate_audit_v6_auto_ready.py \\
  --step7-root {cfg.root} \\
  --generated-file /path/to/step7_generated_final_audited.csv \\
  --train-file /path/to/step7_train_reference_annotated.csv
"""
    path = cfg.reports_dir / "README_step7_candidate_audit_outputs.txt"
    path.write_text(text, encoding="utf-8")
    return str(path)


def write_requirements(cfg: Step7Config) -> str:
    path = cfg.code_dir / "requirements_step7_candidate_audit_minimal.txt"
    path.write_text(
        "\n".join(
            [
                "numpy",
                "pandas",
                "matplotlib",
                "",
                "# Record exact package versions at repository freeze, e.g.:",
                "# python -m pip freeze > requirements_full_freeze.txt",
            ]
        ),
        encoding="utf-8",
    )
    return str(path)


def write_code_snapshot(cfg: Step7Config) -> str:
    dest = cfg.code_dir / "OncoPep_step7_PLOS_candidate_audit_v6_auto_ready.py"
    try:
        current = Path(__file__).resolve()
        if current.exists():
            shutil.copy2(current, dest)
            return str(dest)
    except Exception:
        pass

    try:
        source = inspect.getsource(sys.modules[__name__])
        dest.write_text(source, encoding="utf-8")
    except Exception as exc:
        dest.write_text(f"# Code snapshot unavailable: {type(exc).__name__}: {exc}\n", encoding="utf-8")
    return str(dest)


def write_manifest(
    cfg: Step7Config,
    inputs: Dict[str, object],
    figures: Sequence[str],
    source_data: Sequence[str],
    reports: Sequence[str],
    readiness: Dict[str, object],
    readme: str,
    requirements: str,
    code_snapshot: str,
) -> str:
    manifest = {
        "workflow": "OncoPep Step 7 candidate-audit redesign",
        "version": "v6 auto-discovery notebook-terminal candidate-audit PLOS-ready",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version,
        "platform": platform.platform(),
        "config": asdict(cfg),
        "inputs": inputs,
        "figures": list(figures),
        "source_data": list(source_data),
        "reports": list(reports),
        "readiness": readiness,
        "readme": readme,
        "requirements": requirements,
        "code_snapshot": code_snapshot,
        "figure_numbering": {
            "main_figure": "No new main Figure 7 generated.",
            "S12 Fig": "Final candidate audit and selection funnel.",
            "S13 Fig": "Reference-space descriptor support.",
            "S14 Fig": "Similarity-tail risk and nearest-neighbor audit.",
        },
        "claim_boundary": "Computational audit only; no anticancer activity, receptor binding, or therapeutic efficacy claim.",
    }
    path = cfg.reports_dir / "step7_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


# =============================================================================
# Workflow
# =============================================================================

def _run_with_config(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    set_plot_style(cfg)

    loaded = load_inputs(cfg)
    audited_df: pd.DataFrame = loaded["audited_df"]  # type: ignore[assignment]
    train_df: pd.DataFrame = loaded["train_df"]  # type: ignore[assignment]
    final_df: pd.DataFrame = loaded["final_df"]  # type: ignore[assignment]
    reference_ranges: pd.DataFrame = loaded["reference_ranges"]  # type: ignore[assignment]
    funnel_df: pd.DataFrame = loaded["funnel_df"]  # type: ignore[assignment]
    inputs: Dict[str, object] = loaded["inputs"]  # type: ignore[assignment]

    print_resolved_inputs(inputs)

    figures: List[str] = []
    source_data: List[str] = []

    source_data.extend(write_core_source_tables(audited_df, final_df, reference_ranges, funnel_df, cfg))

    figs, src = build_s12_candidate_audit(final_df, funnel_df, cfg)
    figures.extend(figs)
    source_data.extend(src)

    figs, src = build_s13_descriptor_support(final_df, train_df, reference_ranges, cfg)
    figures.extend(figs)
    source_data.extend(src)

    figs, src = build_s14_similarity_tail(audited_df, final_df, cfg)
    figures.extend(figs)
    source_data.extend(src)

    reports: List[str] = []
    readiness_report_path, readiness = build_readiness_report(audited_df, final_df, inputs, cfg, figures=figures, source_data=source_data)
    reports.append(readiness_report_path)
    reports.append(write_panel_mapping(cfg))
    readme_path = write_readme(cfg)
    requirements_path = write_requirements(cfg)
    code_path = write_code_snapshot(cfg)

    manifest_path = write_manifest(
        cfg=cfg,
        inputs=inputs,
        figures=figures,
        source_data=source_data,
        reports=reports,
        readiness=readiness,
        readme=readme_path,
        requirements=requirements_path,
        code_snapshot=code_path,
    )
    reports.append(manifest_path)

    return {
        "step7_root": str(cfg.root),
        "figures": figures,
        "source_data": source_data,
        "reports": reports,
        "readme": readme_path,
        "requirements": requirements_path,
        "code_snapshot": code_path,
        "inputs": inputs,
        "readiness": readiness,
    }


def run_step7_candidate_audit(
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    input_roots: Optional[Sequence[str]] = None,
    generated_file: Optional[str] = None,
    train_file: Optional[str] = None,
    final_file: Optional[str] = None,
    summary_file: Optional[str] = None,
    funnel_file: Optional[str] = None,
    reference_quantile_low: float = 0.00,
    reference_quantile_high: float = 1.00,
    similarity_low_risk_threshold: float = 0.30,
    top_n_final: int = 12,
    show_figures: bool = False,
    compute_missing_descriptors: bool = True,
    compute_nn_if_missing: bool = True,
    max_nn_pairs: int = 3_000_000,
) -> Dict[str, object]:
    """Run the full Step 7 audit from a notebook or Python script."""
    cfg = Step7Config(
        step7_root=step7_root,
        generated_file=generated_file,
        train_file=train_file,
        final_file=final_file,
        summary_file=summary_file,
        funnel_file=funnel_file,
        reference_quantile_low=reference_quantile_low,
        reference_quantile_high=reference_quantile_high,
        similarity_low_risk_threshold=similarity_low_risk_threshold,
        top_n_final=top_n_final,
        show_figures=show_figures,
        compute_missing_descriptors=compute_missing_descriptors,
        compute_nn_if_missing=compute_nn_if_missing,
        max_nn_pairs=max_nn_pairs,
    )
    if input_roots is not None:
        cfg.input_roots = tuple(input_roots) + tuple(cfg.input_roots)
    return _run_with_config(cfg)


# =============================================================================
# CLI and notebook behavior
# =============================================================================

def _is_jupyter_unknown_arg(arg: str) -> bool:
    text = str(arg)
    return (
        text == "-f"
        or text == "--f"
        or text.startswith("-f=")
        or text.startswith("--f=")
        or ("kernel" in text and text.endswith(".json"))
        or ("jupyter" in text.lower() and "runtime" in text.lower())
    )


def is_running_under_ipykernel() -> bool:
    return "ipykernel" in sys.modules or any("ipykernel" in str(arg) for arg in sys.argv)


def has_meaningful_cli_args(argv: Optional[Sequence[str]] = None) -> bool:
    if argv is None:
        argv = sys.argv[1:]
    return any(str(arg).startswith("--") and not _is_jupyter_unknown_arg(str(arg)) for arg in argv)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 7 candidate-audit supplementary figures.")
    parser.add_argument("--step7-root", default="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07")
    parser.add_argument("--input-root", action="append", default=None, help="Additional input root. Can be supplied multiple times.")
    parser.add_argument("--generated-file", default=None, help="Direct path to step7_generated_final_audited.csv.")
    parser.add_argument("--train-file", default=None, help="Direct path to step7_train_reference_annotated.csv.")
    parser.add_argument("--final-file", default=None, help="Optional direct path to a separate final-candidate table.")
    parser.add_argument("--summary-file", default=None, help="Optional direct path to model summary table.")
    parser.add_argument("--funnel-file", default=None, help="Optional direct path to candidate funnel counts table.")
    parser.add_argument("--reference-quantile-low", type=float, default=0.00)
    parser.add_argument("--reference-quantile-high", type=float, default=1.00)
    parser.add_argument("--similarity-low-risk-threshold", type=float, default=0.30)
    parser.add_argument("--top-n-final", type=int, default=12)
    parser.add_argument("--show-figures", action="store_true")
    parser.add_argument("--no-compute-missing-descriptors", action="store_true")
    parser.add_argument("--no-compute-nn-if-missing", action="store_true")
    parser.add_argument("--max-nn-pairs", type=int, default=3_000_000)
    parser.add_argument("--dpi-png", type=int, default=None)
    parser.add_argument("--dpi-tiff", type=int, default=None)

    args, unknown = parser.parse_known_args(argv)
    non_jupyter_unknown = [arg for arg in unknown if not _is_jupyter_unknown_arg(str(arg))]
    if non_jupyter_unknown:
        warnings.warn("Ignoring unrecognized arguments: " + " ".join(map(str, non_jupyter_unknown)), RuntimeWarning, stacklevel=2)
    return args


def print_notebook_instructions() -> None:
    print(
        """
OncoPep Step 7 v6 code loaded in a Jupyter/IPython kernel.

This version can auto-discover real Step 7 CSV files. To run explicitly:

from OncoPep_step7_PLOS_candidate_audit_v6_auto_ready import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file=None,   # auto-discover, or provide a real path
    train_file=None,       # auto-discover, or provide a real path
    final_file=None,
    show_figures=True,
)
outputs
"""
    )


def notebook_auto_run() -> None:
    """When pasted into a notebook, try auto-discovery once instead of only printing placeholders."""
    print("\nOncoPep Step 7 v6 loaded in a Jupyter/IPython kernel.")
    print("Attempting automatic input discovery from the configured Step 7 locations...\n")
    try:
        outputs = run_step7_candidate_audit(show_figures=True)
        print("\nAutomatic Step 7 run completed. Outputs:")
        print(json.dumps(outputs, indent=2))
    except FileNotFoundError as exc:
        print("Automatic discovery did not find the required real input files.")
        print(str(exc))
        print_notebook_instructions()
        raise

def main(argv: Optional[Sequence[str]] = None) -> None:
    args = parse_args(argv)
    cfg = Step7Config(
        step7_root=args.step7_root,
        generated_file=args.generated_file,
        train_file=args.train_file,
        final_file=args.final_file,
        summary_file=args.summary_file,
        funnel_file=args.funnel_file,
        reference_quantile_low=args.reference_quantile_low,
        reference_quantile_high=args.reference_quantile_high,
        similarity_low_risk_threshold=args.similarity_low_risk_threshold,
        top_n_final=args.top_n_final,
        show_figures=args.show_figures,
        compute_missing_descriptors=not args.no_compute_missing_descriptors,
        compute_nn_if_missing=not args.no_compute_nn_if_missing,
        max_nn_pairs=args.max_nn_pairs,
    )
    if args.input_root is not None:
        cfg.input_roots = tuple(args.input_root) + tuple(cfg.input_roots)
    if args.dpi_png is not None:
        cfg.dpi_png = args.dpi_png
    if args.dpi_tiff is not None:
        cfg.dpi_tiff = args.dpi_tiff

    outputs = _run_with_config(cfg)
    print("\nOncoPep Step 7 candidate-audit redesign complete.")
    print(json.dumps(outputs, indent=2))


if __name__ == "__main__":
    if is_running_under_ipykernel() and not has_meaningful_cli_args():
        notebook_auto_run()
    else:
        main()

In [ ]:
from pathlib import Path

search_roots = [
    Path("/home/data3/Moe/nature_computational2/redesign/step7"),
    Path("/home/data3/Moe/nature_computational2/redesign/step7/outputs"),
    Path("/home/data3/Moe/nature_computational2/redesign/step7/outputs/source_data"),
    Path("/home/data3/Moe/nature_computational2/redesign/step7/outputs/tables"),
    Path("/home/data3/Moe/nature_computational2/step7_v3"),
    Path("/home/data3/Moe/nature_computational2/step7_v3/outputs"),
    Path("/home/data3/Moe/nature_computational2/step7_v3/outputs/tables"),
    Path("/home/data3/Moe/nature_computational2/step7_v3/outputs/source_data"),
    Path.cwd(),
    Path("/mnt/data"),
]

target_names = [
    "step7_model_summary.csv",
    "step7_generated_final_audited.csv",
    "step7_generated_final.csv",
    "step7_generated_audited.csv",
    "generated_final_audited.csv",
    "step7_train_reference_annotated.csv",
    "train_reference_annotated.csv",
    "SuppFig_step7_property_generated_final_v7_source_data.csv",
    "SuppFig_step7_property_train_reference_v7_source_data.csv",
    "SuppFig_step7_similarity_distribution_v7_source_data.csv",
    "SuppFig_step7_similarity_tail_v7_source_data.csv",
    "Fig7_step7_main_final_v7_source_data.csv",
    "Fig7_step7_main_final_v7_similarity_source_data.csv",
]

found = {}

for root in search_roots:
    if not root.exists():
        continue
    for name in target_names:
        matches = list(root.rglob(name))
        if matches:
            found.setdefault(name, [])
            found[name].extend([str(m.resolve()) for m in matches])

for name, paths in found.items():
    print("\n" + name)
    for p in sorted(set(paths)):
        print("  ", p)

In [ ]:
from pathlib import Path

search_roots = [
    Path("/home/data3/Moe/nature_computational2/redesign/step7"),
    Path("/home/data3/Moe/nature_computational2/redesign/step7/outputs"),
    Path("/home/data3/Moe/nature_computational2/redesign/step7/outputs/source_data"),
    Path("/home/data3/Moe/nature_computational2/redesign/step7/outputs/tables"),
    Path("/home/data3/Moe/nature_computational2/step7_v3"),
    Path("/home/data3/Moe/nature_computational2/step7_v3/outputs"),
    Path("/home/data3/Moe/nature_computational2/step7_v3/outputs/tables"),
    Path("/home/data3/Moe/nature_computational2/step7_v3/outputs/source_data"),
    Path("/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07"),
    Path("/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/source_data"),
]

for root in search_roots:
    print(f"{root.exists():5}  {root}")

In [ ]:
!find /home/data3/Moe -maxdepth 8 -type f \( \
  -iname "*step7*.csv" -o \
  -iname "*step_7*.csv" -o \
  -iname "*step_07*.csv" -o \
  -iname "*Fig7*.csv" -o \
  -iname "*SuppFig_step7*.csv" \
\) 2>/dev/null | sort

In [ ]:
import pandas as pd

generated_file = "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv"
train_file = "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv"

gen = pd.read_csv(generated_file)
train = pd.read_csv(train_file)

print("Generated shape:", gen.shape)
print(gen.columns.tolist())
display(gen.head())

print("Train shape:", train.shape)
print(train.columns.tolist())
display(train.head())

In [ ]:
outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
    top_n_final=12,
)

outputs

In [ ]:
#!/usr/bin/env python3
"""
OncoPep Step 7 PLOS Computational Biology candidate-audit redesign.

Version
-------
v8 no-dot / no-lollipop / no-line candidate-audit package

Purpose
-------
Step 7 is treated as a supplementary final-candidate audit, not as another
Step 5 generator benchmark or Step 6 latent-space diagnostic.

Strict Step 7 graph-type rule
-----------------------------
This script does NOT generate candidate-level dot plots, candidate-level
lollipop plots, line plots, connected-point plots, ECDF line plots, trend lines,
or decorative line summaries.

Generated supplementary figures
-------------------------------
S12 Fig: Final candidate audit and selection funnel
S13 Fig: Reference-space descriptor support using training histograms plus
         binned final-candidate count summaries
S14 Fig: Similarity-tail risk and nearest-neighbor audit using binned/threshold
         bar summaries

Claim boundary
--------------
These analyses support computational auditing and downstream prioritization only.
They do not establish anticancer activity, receptor binding, selectivity,
toxicity, stability, mechanism, therapeutic efficacy, or clinical relevance.

Notebook use
------------
from OncoPep_step7_PLOS_candidate_audit_v8_no_dot_no_line import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
    top_n_final=12,
)
outputs
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm


# =============================================================================
# Constants and visual style
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "OncoPep",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
    "train_ref": "Train reference",
}

MODEL_COLORS = {
    "full_cvae": PLOS["coral"],
    "conditional_gru_lm": PLOS["blue"],
    "random_condition": PLOS["mint"],
    "no_condition": PLOS["brown"],
    "small_latent": PLOS["charcoal"],
    "train_ref": PLOS["light"],
}

MODEL_EDGES = {
    "full_cvae": "#B84F42",
    "conditional_gru_lm": "#166F8A",
    "random_condition": "#75A883",
    "no_condition": "#8E5F39",
    "small_latent": "#4F4648",
    "train_ref": "#A7A7A7",
}

MODEL_TAG_ALIASES = {
    "oncoppep": "full_cvae",
    "oncopep": "full_cvae",
    "full_cvae": "full_cvae",
    "cvae": "full_cvae",
    "conditional_vae": "full_cvae",
    "conditioned_vae": "full_cvae",
    "gru-cond": "conditional_gru_lm",
    "gru_cond": "conditional_gru_lm",
    "conditional_gru": "conditional_gru_lm",
    "conditional_gru_lm": "conditional_gru_lm",
    "rand-cond": "random_condition",
    "rand_cond": "random_condition",
    "random_condition": "random_condition",
    "random conditional": "random_condition",
    "gru-uncond": "no_condition",
    "gru_uncond": "no_condition",
    "unconditional_gru": "no_condition",
    "no_condition": "no_condition",
    "vae-uncond": "small_latent",
    "vae_uncond": "small_latent",
    "unconditional_vae": "small_latent",
    "small_latent": "small_latent",
}

AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DESCRIPTORS = ("length", "net_charge", "hydrophobicity", "entropy")
DEFAULT_TAIL_THRESHOLDS = (0.10, 0.15, 0.20, 0.30)
SIMILARITY_BINS = (0.0, 0.10, 0.15, 0.20, 0.30, 1.000001)
SIMILARITY_BIN_LABELS = ("<0.10", "0.10-0.15", "0.15-0.20", "0.20-0.30", ">=0.30")

KD_SCALE = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

COLUMN_ALIASES = {
    "model_tag": ["model_tag", "model", "generator", "model_name", "method"],
    "candidate_id": ["candidate_id", "peptide_id", "id", "seq_id", "sequence_id", "candidate"],
    "sequence": ["sequence", "peptide", "seq", "peptide_sequence"],
    "is_final_candidate": [
        "is_final_candidate", "final_selected", "selected_final", "selected", "is_selected",
        "final_candidate", "passed_final_audit", "passed_audit",
    ],
    "candidate_rank": ["candidate_rank", "rank", "final_rank", "selection_rank", "priority_rank"],
    "composite_score": ["composite_score", "score", "selection_score", "priority_score", "audit_score"],
    "exact_novel_vs_train": [
        "exact_novel_vs_train", "exact_novel", "exact_novelty", "novel_vs_train",
        "novelty_flag", "is_exact_novel", "novelty_exact",
    ],
    "nn_similarity_train": [
        "nn_similarity_train", "nearest_neighbor_similarity_train", "nearest_train_similarity",
        "nearest_train_jaccard", "smax_train", "nn_train", "similarity_to_train",
        "nearest_neighbor_similarity",
    ],
    "length": ["length", "seq_length", "peptide_length", "model_ready_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "charge", "charge_pH7", "net_charge_ph7", "charge_proxy"],
    "hydrophobicity": [
        "hydrophobicity", "hydrophobicity_KD", "mean_hydrophobicity",
        "mean_kd_hydrophobicity", "mean_KD", "mean_kyte_doolittle", "kd_hydrophobicity",
        "mean_hydropathy",
    ],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy", "seq_entropy"],
    "plausibility_is_within_reference": [
        "plausibility_is_within_reference", "plausibility_within_reference", "within_reference",
        "within_ref", "is_within_reference",
    ],
}

TRAIN_ALIASES = {
    "sequence": COLUMN_ALIASES["sequence"],
    "length": COLUMN_ALIASES["length"],
    "net_charge": COLUMN_ALIASES["net_charge"],
    "hydrophobicity": COLUMN_ALIASES["hydrophobicity"],
    "entropy": COLUMN_ALIASES["entropy"],
}


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Step7Config:
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07"
    input_roots: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/source_data",
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/reports",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/tables",
        "/home/data3/Moe/nature_computational_peponco/step7_v3",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco/step8_v1/artifacts",
        "/mnt/data",
    )

    generated_file: Optional[str] = None
    train_file: Optional[str] = None
    final_file: Optional[str] = None
    summary_file: Optional[str] = None
    funnel_file: Optional[str] = None

    generated_candidates: Tuple[str, ...] = (
        "step7_generated_final_audited.csv",
        "step7_generated_final.csv",
        "step7_generated_audited.csv",
        "generated_final_audited.csv",
        "SuppFig_step7_property_generated_final_v7_source_data.csv",
        "SuppFig_step7_similarity_distribution_v7_source_data.csv",
    )
    train_candidates: Tuple[str, ...] = (
        "step7_train_reference_annotated.csv",
        "train_reference_annotated.csv",
        "SuppFig_step7_property_train_reference_v7_source_data.csv",
        "SuppFig_step7_property_train_reference_source_data.csv",
    )
    final_candidates: Tuple[str, ...] = (
        "step7_final_candidates.csv",
        "step7_final_candidates_audit_table.csv",
        "final_candidates.csv",
        "final_oncopep_candidates.csv",
        "top12_candidates.csv",
    )
    summary_candidates: Tuple[str, ...] = (
        "step7_model_summary.csv",
        "step7_final_generation_audit_summary.csv",
        "step7_reference_space_plausibility_summary.csv",
        "Fig7_step7_main_final_v7_source_data.csv",
    )
    funnel_candidates: Tuple[str, ...] = (
        "step7_candidate_funnel_counts.csv",
        "candidate_funnel_counts.csv",
        "step7_funnel_counts.csv",
        "Fig7_step7_candidate_funnel_source_data.csv",
    )

    # Central reference interval used for S13 support summaries.
    reference_quantile_low: float = 0.01
    reference_quantile_high: float = 0.99
    similarity_low_risk_threshold: float = 0.30
    tail_thresholds: Tuple[float, ...] = DEFAULT_TAIL_THRESHOLDS
    top_n_final: int = 12

    compute_missing_descriptors: bool = True
    histidine_charge: float = 0.10
    compute_nn_if_missing: bool = True
    max_nn_pairs: int = 3_000_000
    nn_metric_if_computed: str = "normalized_levenshtein_similarity"

    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    tiff_compression: str = "tiff_lzw"
    show_figures: bool = False

    font_family: str = "DejaVu Sans"
    base_font_size: float = 11.2
    axis_label_size: float = 12.0
    title_size: float = 13.8
    tick_size: float = 10.0
    legend_size: float = 10.2
    panel_label_size: float = 17.0

    seed: int = 42
    max_heatmap_candidates: int = 30
    inventory_max_files: int = 300

    @property
    def root(self) -> Path:
        return Path(self.step7_root).expanduser().resolve()

    @property
    def main_figure_dir(self) -> Path:
        return self.root / "main_figure"

    @property
    def supplementary_dir(self) -> Path:
        return self.root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.root / "code"


# =============================================================================
# General utilities
# =============================================================================

def ensure_output_tree(cfg: Step7Config) -> None:
    for path in [cfg.root, cfg.main_figure_dir, cfg.supplementary_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)


def set_plot_style(cfg: Step7Config) -> None:
    plt.rcParams.update(
        {
            "font.family": cfg.font_family,
            "font.size": cfg.base_font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "figure.titlesize": cfg.title_size,
            "axes.facecolor": PLOS["white"],
            "figure.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def inventory_csv_files(roots: Sequence[str], max_files: int = 300) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    seen: set[str] = set()
    for priority, root_raw in enumerate(roots):
        root = Path(root_raw).expanduser()
        if not root.exists():
            rows.append({"root_priority": priority, "root": str(root), "exists": False, "path": None, "filename": None, "size_bytes": None})
            continue
        if root.is_file():
            file_iter = iter([root] if root.suffix.lower() == ".csv" else [])
        else:
            file_iter = root.rglob("*.csv")
        n_seen_this_root = 0
        for file_path in file_iter:
            if n_seen_this_root >= max_files:
                break
            try:
                key = str(file_path.resolve())
            except Exception:
                continue
            if key in seen:
                continue
            seen.add(key)
            n_seen_this_root += 1
            try:
                size_bytes = file_path.stat().st_size
            except OSError:
                size_bytes = None
            rows.append({"root_priority": priority, "root": str(root), "exists": True, "path": key, "filename": file_path.name, "size_bytes": size_bytes})
    return pd.DataFrame(rows)


def normalize_path(path_raw: Optional[str]) -> Optional[Path]:
    if not path_raw:
        return None
    return Path(path_raw).expanduser().resolve()


def iter_candidate_matches(root: Path, candidate_names: Sequence[str], max_hits: int = 5) -> Iterable[Path]:
    candidate_set = set(candidate_names)
    if root.is_file():
        if root.name in candidate_set:
            yield root.resolve()
        return
    hits = 0
    for name in candidate_names:
        p = root / name
        if p.exists() and p.is_file():
            yield p.resolve()
            hits += 1
            if hits >= max_hits:
                return
    try:
        for p in root.rglob("*.csv"):
            if p.name in candidate_set:
                yield p.resolve()
                hits += 1
                if hits >= max_hits:
                    return
    except (OSError, PermissionError):
        return


def build_missing_file_diagnostic(label: str, candidate_names: Sequence[str], searched_roots: Sequence[str], inventory: Optional[pd.DataFrame], step7_root: str) -> str:
    candidate_msg = "\n".join(f"  - {name}" for name in candidate_names)
    root_msg = "\n".join(f"  - {root}" for root in searched_roots)
    inventory_msg = ""
    if inventory is not None and not inventory.empty and "path" in inventory.columns:
        existing = inventory.dropna(subset=["path"]).copy()
        if not existing.empty:
            inventory_msg = "\n\nCSV files found in searched roots (first 80):\n"
            inventory_msg += existing[["filename", "path", "size_bytes"]].head(80).to_string(index=False)
    command_msg = (
        "\n\nSuggested terminal command after locating real files:\n"
        f"python OncoPep_step7_PLOS_candidate_audit_v8_no_dot_no_line.py \\\n"
        f"  --step7-root {step7_root} \\\n"
        "  --generated-file /REAL/PATH/step7_generated_final_audited.csv \\\n"
        "  --train-file /REAL/PATH/step7_train_reference_annotated.csv\n"
    )
    return f"Could not find required Step 7 {label}.\n\nAccepted filenames:\n{candidate_msg}\n\nSearched roots:\n{root_msg}{inventory_msg}{command_msg}"


def discover_any(candidate_names: Sequence[str], roots: Sequence[str], direct_file: Optional[str] = None, required: bool = False, label: str = "file", inventory: Optional[pd.DataFrame] = None, step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07") -> Optional[Path]:
    direct = normalize_path(direct_file)
    if direct is not None:
        if direct.exists() and direct.is_file():
            return direct
        raise FileNotFoundError(f"Direct {label} path was provided but does not exist: {direct}\nReplace placeholder paths with a real CSV path.")
    searched_roots: List[str] = []
    for root_raw in roots:
        root = Path(root_raw).expanduser()
        searched_roots.append(str(root))
        if not root.exists():
            continue
        for match in iter_candidate_matches(root, candidate_names, max_hits=1):
            return match
    if required:
        raise FileNotFoundError(build_missing_file_diagnostic(label, candidate_names, searched_roots, inventory, step7_root))
    return None


def find_first_column(df: pd.DataFrame, aliases: Sequence[str], canonical_name: str, required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for alias in aliases:
        if alias in df.columns:
            return alias
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    if required:
        raise KeyError(f"Could not find a column for '{canonical_name}'. Tried aliases: {list(aliases)}. Available columns: {list(df.columns)}")
    return None


def maybe_rename_columns(df: pd.DataFrame, aliases: Dict[str, Sequence[str]], canonical_columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    rename_map: Dict[str, str] = {}
    for canonical in canonical_columns:
        if canonical in out.columns or canonical not in aliases:
            continue
        found = find_first_column(out, aliases[canonical], canonical, required=False)
        if found is not None and found != canonical:
            rename_map[found] = canonical
    return out.rename(columns=rename_map)


def normalize_model_tag_value(value: object) -> object:
    if pd.isna(value):
        return value
    text = str(value).strip()
    key = text.lower().replace("-", "_").replace(" ", "_")
    return MODEL_TAG_ALIASES.get(key, text)


def normalize_model_tags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model_tag" in out.columns:
        out["model_tag"] = out["model_tag"].map(normalize_model_tag_value)
    return out


def as_numeric_clean(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def parse_boolish(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    text = series.astype(str).str.strip().str.lower()
    truthy = {"true", "1", "yes", "y", "selected", "final", "pass", "passed", "keep", "kept"}
    falsy = {"false", "0", "no", "n", "not_selected", "fail", "failed", "nan", "none", "", "remove"}
    return text.map(lambda x: True if x in truthy else (False if x in falsy else False))


def clean_sequence(seq: object) -> str:
    if pd.isna(seq):
        return ""
    return str(seq).strip().upper()


def valid_sequence(seq: object) -> bool:
    s = clean_sequence(seq)
    return len(s) > 0 and all(ch in AA20 for ch in s)


def peptide_length(seq: object) -> float:
    s = clean_sequence(seq)
    return float(len(s)) if s else np.nan


def peptide_net_charge(seq: object, histidine_charge: float = 0.10) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    return float(s.count("K") + s.count("R") + histidine_charge * s.count("H") - s.count("D") - s.count("E"))


def peptide_hydrophobicity(seq: object) -> float:
    s = clean_sequence(seq)
    vals = [KD_SCALE[ch] for ch in s if ch in KD_SCALE]
    return float(np.mean(vals)) if vals else np.nan


def peptide_entropy(seq: object) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    counts = np.array([s.count(aa) for aa in AA20], dtype=float)
    probs = counts[counts > 0] / len(s)
    return float(-(probs * np.log2(probs)).sum())


def normalized_levenshtein_similarity(a: str, b: str) -> float:
    a = clean_sequence(a); b = clean_sequence(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(current[j - 1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return float(1.0 - previous[-1] / max(len(a), len(b)))


def compute_nearest_similarity(sequences: Sequence[str], train_sequences: Sequence[str], max_pairs: int = 3_000_000) -> Tuple[pd.Series, str]:
    n_pairs = len(sequences) * len(train_sequences)
    if n_pairs > max_pairs:
        return pd.Series([np.nan] * len(sequences), dtype=float), f"not_computed_pair_budget_exceeded:{n_pairs}>{max_pairs}"
    train_list = [clean_sequence(s) for s in train_sequences if clean_sequence(s)]
    values: List[float] = []
    for seq in sequences:
        s = clean_sequence(seq)
        values.append(max((normalized_levenshtein_similarity(s, t) for t in train_list), default=np.nan))
    return pd.Series(values, dtype=float), "computed_normalized_levenshtein_similarity"


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def dataframe_with_panel(df: pd.DataFrame, figure: str, panel: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "panel", panel)
    out.insert(0, "figure", figure)
    return out


# =============================================================================
# Data preparation
# =============================================================================

def add_or_compute_descriptors(df: pd.DataFrame, cfg: Step7Config, role: str, warnings_list: List[str]) -> pd.DataFrame:
    out = df.copy()
    if "sequence" not in out.columns:
        return out
    if cfg.compute_missing_descriptors:
        if "length" not in out.columns or out["length"].isna().all():
            out["length"] = out["sequence"].map(peptide_length)
            warnings_list.append(f"{role}: length was missing and was computed from sequence.")
        if "net_charge" not in out.columns or out["net_charge"].isna().all():
            out["net_charge"] = out["sequence"].map(lambda s: peptide_net_charge(s, cfg.histidine_charge))
            warnings_list.append(f"{role}: net_charge was missing and was approximated from residue counts.")
        if "hydrophobicity" not in out.columns or out["hydrophobicity"].isna().all():
            out["hydrophobicity"] = out["sequence"].map(peptide_hydrophobicity)
            warnings_list.append(f"{role}: hydrophobicity was missing and was computed using the Kyte-Doolittle mean.")
        if "entropy" not in out.columns or out["entropy"].isna().all():
            out["entropy"] = out["sequence"].map(peptide_entropy)
            warnings_list.append(f"{role}: entropy was missing and was computed as Shannon entropy over AA20.")
    for col in DESCRIPTORS:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    return out


def prepare_generated_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, COLUMN_ALIASES, COLUMN_ALIASES.keys())
    if "sequence" not in out.columns:
        raise KeyError(f"Generated/final audited table is missing a sequence column. Available columns: {list(raw.columns)}")
    if "model_tag" not in out.columns:
        out["model_tag"] = "full_cvae"
        warnings_list.append("generated: model_tag was missing; all rows were assigned to full_cvae/OncoPep.")
    out = normalize_model_tags(out)
    out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "generated", warnings_list)
    for col in ["nn_similarity_train", "candidate_rank", "composite_score"]:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    if "candidate_id" not in out.columns:
        out["candidate_id"] = [f"pep_{i+1:05d}" for i in range(len(out))]
        warnings_list.append("generated: candidate_id was missing and sequential IDs were assigned.")
    if "is_final_candidate" in out.columns:
        out["is_final_candidate"] = parse_boolish(out["is_final_candidate"])
    else:
        out["is_final_candidate"] = False
    if "plausibility_is_within_reference" in out.columns:
        out["plausibility_is_within_reference"] = parse_boolish(out["plausibility_is_within_reference"])
    out["valid_aa_sequence"] = out["sequence"].map(valid_sequence)
    out["duplicate_sequence_within_input"] = out.duplicated("sequence", keep=False)
    out["recognized_model"] = out["model_tag"].isin(MODEL_ORDER)
    return out


def prepare_train_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, TRAIN_ALIASES, TRAIN_ALIASES.keys())
    if "sequence" in out.columns:
        out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "train_reference", warnings_list)
    missing = [col for col in DESCRIPTORS if col not in out.columns]
    if missing:
        raise KeyError("Training-reference table is missing descriptor columns and they could not be computed: " + ", ".join(missing))
    for col in DESCRIPTORS:
        out[col] = as_numeric_clean(out[col])
    out["model_tag"] = "train_ref"
    return out


def compute_reference_ranges(train_df: pd.DataFrame, cfg: Step7Config) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for descriptor in DESCRIPTORS:
        vals = train_df[descriptor].dropna().to_numpy(float)
        if len(vals) == 0:
            raise ValueError(f"Training-reference descriptor '{descriptor}' has no finite values.")
        rows.append(
            {
                "descriptor": descriptor,
                "n_train": int(len(vals)),
                "min": float(np.min(vals)),
                "q01": float(np.quantile(vals, 0.01)),
                "q05": float(np.quantile(vals, 0.05)),
                "q25": float(np.quantile(vals, 0.25)),
                "median": float(np.quantile(vals, 0.50)),
                "q75": float(np.quantile(vals, 0.75)),
                "q95": float(np.quantile(vals, 0.95)),
                "q99": float(np.quantile(vals, 0.99)),
                "max": float(np.max(vals)),
                "range_low": float(np.quantile(vals, cfg.reference_quantile_low)),
                "range_high": float(np.quantile(vals, cfg.reference_quantile_high)),
                "range_definition": f"training quantile [{cfg.reference_quantile_low}, {cfg.reference_quantile_high}]",
            }
        )
    return pd.DataFrame(rows)


def add_computed_audit_columns(generated_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = generated_df.copy()
    if "sequence" in train_df.columns:
        train_sequences = set(train_df["sequence"].dropna().astype(str).str.strip().str.upper())
        out["computed_exact_novel_vs_train"] = ~out["sequence"].isin(train_sequences)
    else:
        out["computed_exact_novel_vs_train"] = np.nan
        warnings_list.append("Exact novelty could not be computed because training sequence column is absent.")
    if "exact_novel_vs_train" in out.columns:
        out["exact_novel_vs_train"] = parse_boolish(out["exact_novel_vs_train"])
        out["audit_exact_novel_vs_train"] = out["exact_novel_vs_train"]
    else:
        out["audit_exact_novel_vs_train"] = out["computed_exact_novel_vs_train"]
    if "nn_similarity_train" not in out.columns or out["nn_similarity_train"].isna().all():
        if cfg.compute_nn_if_missing and "sequence" in train_df.columns:
            nn_values, nn_status = compute_nearest_similarity(out["sequence"].astype(str).tolist(), train_df["sequence"].dropna().astype(str).tolist(), max_pairs=cfg.max_nn_pairs)
            out["nn_similarity_train"] = nn_values
            out["nn_similarity_train_source"] = nn_status
            warnings_list.append(f"generated: nn_similarity_train was missing; status={nn_status}.")
        else:
            out["nn_similarity_train"] = np.nan
            out["nn_similarity_train_source"] = "missing"
            warnings_list.append("generated: nn_similarity_train is missing and was not computed.")
    else:
        out["nn_similarity_train"] = as_numeric_clean(out["nn_similarity_train"])
        out["nn_similarity_train_source"] = "input_column"
    for _, row in reference_ranges.iterrows():
        descriptor = str(row["descriptor"])
        low = float(row["range_low"])
        high = float(row["range_high"])
        out[f"within_ref_{descriptor}"] = out[descriptor].between(low, high, inclusive="both")
    within_cols = [f"within_ref_{d}" for d in DESCRIPTORS]
    computed_within = out[within_cols].all(axis=1)
    if "plausibility_is_within_reference" in out.columns:
        out["audit_within_reference_all_descriptors"] = out["plausibility_is_within_reference"].fillna(computed_within)
        out["computed_within_reference_central_interval"] = computed_within
    else:
        out["audit_within_reference_all_descriptors"] = computed_within
        out["computed_within_reference_central_interval"] = computed_within
    out["audit_low_similarity"] = out["nn_similarity_train"] < cfg.similarity_low_risk_threshold
    out["audit_valid_sequence"] = out["valid_aa_sequence"].astype(bool)
    out["audit_nonduplicate_sequence"] = ~out["duplicate_sequence_within_input"].astype(bool)
    audit_cols = ["audit_valid_sequence", "audit_exact_novel_vs_train", "audit_within_reference_all_descriptors", "audit_low_similarity", "audit_nonduplicate_sequence"]
    usable_cols = [c for c in audit_cols if not out[c].isna().all()]
    out["audit_pass_computed"] = out[usable_cols].fillna(True).all(axis=1)
    return out


def prepare_final_file_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = prepare_generated_df(raw, cfg, warnings_list)
    out["is_final_candidate"] = True
    return out


def infer_final_candidates(audited_df: pd.DataFrame, explicit_final_df: Optional[pd.DataFrame], cfg: Step7Config, warnings_list: List[str]) -> Tuple[pd.DataFrame, str]:
    if explicit_final_df is not None and not explicit_final_df.empty:
        final_sequences = set(explicit_final_df["sequence"].dropna().astype(str))
        if final_sequences:
            matched = audited_df[audited_df["sequence"].isin(final_sequences)].copy()
            if not matched.empty:
                return matched, "explicit separate final-candidate file matched by sequence"
        return explicit_final_df.copy(), "explicit separate final-candidate file"
    out = audited_df.copy()
    if "is_final_candidate" in out.columns and out["is_final_candidate"].any():
        return out.loc[out["is_final_candidate"]].copy(), "explicit final-candidate flag"
    oncopep = out.loc[out["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = out.copy()
        warnings_list.append("No full_cvae rows found; final-candidate fallback used all generated rows.")
    if "candidate_rank" in oncopep.columns and oncopep["candidate_rank"].notna().any():
        return oncopep.sort_values("candidate_rank", ascending=True).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by candidate_rank"
    if "composite_score" in oncopep.columns and oncopep["composite_score"].notna().any():
        return oncopep.sort_values("composite_score", ascending=False).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by composite_score"
    passing = oncopep.loc[oncopep["audit_pass_computed"]].copy()
    if not passing.empty:
        return passing.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(passing))} OncoPep rows passing computed audit"
    return oncopep.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(oncopep))} OncoPep rows because no explicit final-candidate flag was present"


def prepare_funnel_df(audited_df: pd.DataFrame, final_df: pd.DataFrame, explicit_funnel_df: Optional[pd.DataFrame], cfg: Step7Config) -> pd.DataFrame:
    if explicit_funnel_df is not None and not explicit_funnel_df.empty:
        funnel = explicit_funnel_df.copy()
        if "stage" not in funnel.columns:
            for col in ["audit_stage", "filter_stage", "step", "name"]:
                if col in funnel.columns:
                    funnel = funnel.rename(columns={col: "stage"}); break
        if "n" not in funnel.columns:
            for col in ["count", "n_sequences", "n_candidates", "value"]:
                if col in funnel.columns:
                    funnel = funnel.rename(columns={col: "n"}); break
        if "stage" in funnel.columns and "n" in funnel.columns:
            funnel["n"] = as_numeric_clean(funnel["n"]).astype("Int64")
            funnel["source"] = "input_funnel_file"
            funnel["retained_from_previous_pct"] = funnel["n"].astype(float) / funnel["n"].astype(float).shift(1).replace(0, np.nan) * 100.0
            if len(funnel): funnel.loc[funnel.index[0], "retained_from_previous_pct"] = 100.0
            return funnel[["stage", "n", "retained_from_previous_pct", "source"]]
    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()
    rows = [
        ("OncoPep evaluated outputs", len(oncopep)),
        ("Valid amino-acid sequences", int(oncopep["audit_valid_sequence"].sum())),
        ("Exact-novel vs training", int(oncopep["audit_exact_novel_vs_train"].fillna(False).sum()) if not oncopep["audit_exact_novel_vs_train"].isna().all() else np.nan),
        ("Reference-supported descriptors", int(oncopep["audit_within_reference_all_descriptors"].sum())),
        (f"NN similarity < {cfg.similarity_low_risk_threshold:.2f}", int(oncopep["audit_low_similarity"].fillna(False).sum())),
        ("Final candidates", len(final_df)),
    ]
    funnel = pd.DataFrame([{"stage": stage, "n": n, "source": "computed_from_audited_table"} for stage, n in rows])
    funnel["retained_from_previous_pct"] = funnel["n"].astype(float) / funnel["n"].astype(float).shift(1).replace(0, np.nan) * 100.0
    if len(funnel): funnel.loc[funnel.index[0], "retained_from_previous_pct"] = 100.0
    return funnel[["stage", "n", "retained_from_previous_pct", "source"]]


def load_inputs(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    warnings_list: List[str] = []
    roots = tuple(str(Path(r).expanduser()) for r in cfg.input_roots)
    inventory = inventory_csv_files(roots, cfg.inventory_max_files)
    if not inventory.empty:
        write_csv(inventory, cfg.reports_dir / "step7_input_inventory.csv")
    generated_path = discover_any(cfg.generated_candidates, roots, direct_file=cfg.generated_file, required=True, label="generated/final audited table", inventory=inventory, step7_root=str(cfg.root))
    train_path = discover_any(cfg.train_candidates, roots, direct_file=cfg.train_file, required=True, label="training-reference table", inventory=inventory, step7_root=str(cfg.root))
    # A final-candidate table is intentionally used only when explicitly supplied.
    # This prevents circular reuse of previously generated
    # `step7_final_candidates_audit_table.csv` files from the output folder.
    final_path = discover_any(cfg.final_candidates, roots, direct_file=cfg.final_file, required=False, label="separate final-candidate table", inventory=inventory, step7_root=str(cfg.root)) if cfg.final_file else None
    summary_path = discover_any(cfg.summary_candidates, roots, direct_file=cfg.summary_file, required=False, label="model summary table", inventory=inventory, step7_root=str(cfg.root))
    funnel_path = discover_any(cfg.funnel_candidates, roots, direct_file=cfg.funnel_file, required=False, label="candidate funnel table", inventory=inventory, step7_root=str(cfg.root)) if cfg.funnel_file else None
    assert generated_path is not None and train_path is not None
    generated_raw = pd.read_csv(generated_path)
    train_raw = pd.read_csv(train_path)
    final_raw = pd.read_csv(final_path) if final_path is not None else None
    summary_df = pd.read_csv(summary_path) if summary_path is not None else None
    funnel_df = pd.read_csv(funnel_path) if funnel_path is not None else None
    generated_df = prepare_generated_df(generated_raw, cfg, warnings_list)
    train_df = prepare_train_df(train_raw, cfg, warnings_list)
    explicit_final_df = prepare_final_file_df(final_raw, cfg, warnings_list) if final_raw is not None else None
    reference_ranges = compute_reference_ranges(train_df, cfg)
    audited_df = add_computed_audit_columns(generated_df, train_df, reference_ranges, cfg, warnings_list)
    if explicit_final_df is not None:
        explicit_final_df = add_computed_audit_columns(explicit_final_df, train_df, reference_ranges, cfg, warnings_list)
    final_df, final_rule = infer_final_candidates(audited_df, explicit_final_df, cfg, warnings_list)
    if "is_final_candidate" in audited_df.columns:
        audited_df.loc[audited_df["candidate_id"].isin(final_df["candidate_id"]), "is_final_candidate"] = True
    funnel_final = prepare_funnel_df(audited_df, final_df, funnel_df, cfg)
    inputs = {
        "generated_path": str(generated_path),
        "train_path": str(train_path),
        "final_path": str(final_path) if final_path else None,
        "summary_path": str(summary_path) if summary_path else None,
        "funnel_path": str(funnel_path) if funnel_path else None,
        "input_inventory": str(cfg.reports_dir / "step7_input_inventory.csv"),
        "generated_sha256": sha256_file(generated_path),
        "train_sha256": sha256_file(train_path),
        "final_sha256": sha256_file(final_path),
        "summary_sha256": sha256_file(summary_path),
        "funnel_sha256": sha256_file(funnel_path),
        "final_candidate_rule": final_rule,
        "warnings_from_preparation": warnings_list,
        "nn_metric_if_computed": cfg.nn_metric_if_computed,
        "strict_graph_rule": "No candidate-level dot/lollipop/line/connected/ECDF/trend/decorative line plots.",
    }
    return {"generated_df": generated_df, "train_df": train_df, "reference_ranges": reference_ranges, "audited_df": audited_df, "final_df": final_df, "funnel_df": funnel_final, "summary_df": summary_df, "inventory": inventory, "inputs": inputs}


# =============================================================================
# Plot helpers
# =============================================================================

def save_figure(fig: plt.Figure, out_base: Path, cfg: Step7Config) -> List[str]:
    outputs: List[str] = []
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf"); fig.savefig(p, bbox_inches="tight"); outputs.append(str(p))
    if cfg.export_png:
        p = out_base.with_suffix(".png"); fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png); outputs.append(str(p))
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        try:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, pil_kwargs={"compression": cfg.tiff_compression})
        except TypeError:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff)
        outputs.append(str(p))
    if cfg.show_figures:
        plt.show()
    else:
        plt.close(fig)
    return outputs


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.75)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.85); ax.spines[side].set_color(PLOS["dark"])
    ax.tick_params(axis="both", width=0.75, length=3.5)


def add_panel_label(ax: plt.Axes, label: str, cfg: Step7Config, x: float = -0.08, y: float = 1.03) -> None:
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=cfg.panel_label_size, fontweight="bold", color=PLOS["dark"], clip_on=False)


def add_bottom_legend(fig: plt.Figure, include_train: bool, y: float = 0.02, ncol: Optional[int] = None) -> None:
    handles = []
    if include_train:
        handles.append(Patch(facecolor=MODEL_COLORS["train_ref"], edgecolor=MODEL_EDGES["train_ref"], label=MODEL_LABELS["train_ref"]))
    for tag in MODEL_ORDER:
        handles.append(Patch(facecolor=MODEL_COLORS[tag], edgecolor=MODEL_EDGES[tag], label=MODEL_LABELS[tag]))
    if ncol is None: ncol = len(handles)
    legend = fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, y), ncol=ncol, frameon=True, columnspacing=1.0, handletextpad=0.45, borderpad=0.40)
    legend.get_frame().set_facecolor("#FAFAFA"); legend.get_frame().set_edgecolor("#CFCFCF"); legend.get_frame().set_linewidth(0.8)


# =============================================================================
# Figure builders: no candidate dots, no lollipop, no line plots.
# =============================================================================

def build_s12_candidate_audit(final_df: pd.DataFrame, funnel_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    fig = plt.figure(figsize=(14.4, 6.7))
    gs = GridSpec(1, 2, width_ratios=[1.0, 1.35], figure=fig)

    ax_a = fig.add_subplot(gs[0, 0])
    funnel = funnel_df.copy()
    funnel["n"] = as_numeric_clean(funnel["n"])
    funnel = funnel.dropna(subset=["n"]).copy()
    funnel["n"] = funnel["n"].astype(int)
    y = np.arange(len(funnel))[::-1]
    colors = [PLOS["coral"] if stage != "Final candidates" else PLOS["brown"] for stage in funnel["stage"]]
    edges = ["#B84F42" if stage != "Final candidates" else "#8E5F39" for stage in funnel["stage"]]
    bars = ax_a.barh(y, funnel["n"], color=colors, edgecolor=edges, linewidth=0.8, zorder=3)
    ax_a.set_yticks(y); ax_a.set_yticklabels(funnel["stage"])
    ax_a.set_xlabel("Number of sequences")
    ax_a.set_title("Selection funnel", pad=9)
    style_axis(ax_a, "x")
    add_panel_label(ax_a, "A", cfg)
    max_n = max(float(funnel["n"].max()), 1.0)
    for bar, n, pct in zip(bars, funnel["n"], funnel["retained_from_previous_pct"]):
        pct_text = "" if pd.isna(pct) else f"; {pct:.1f}% retained"
        ax_a.text(bar.get_width() + max_n * 0.012, bar.get_y() + bar.get_height() / 2, f"{int(n):,}{pct_text}", va="center", ha="left", fontsize=9.2)
    ax_a.text(0.98, 0.06, "Final candidates = 12", transform=ax_a.transAxes, ha="right", va="bottom", fontsize=10.2, bbox=dict(boxstyle="round,pad=0.28", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.8))

    ax_b = fig.add_subplot(gs[0, 1])
    audit_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_within_reference_all_descriptors",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
        "audit_pass_computed",
    ]
    labels = ["Valid AA", "Exact novel", "Ref.-supported", f"NN < {cfg.similarity_low_risk_threshold:.2f}", "Non-duplicate", "Overall"]
    plot_df = final_df.copy()
    if "candidate_rank" in plot_df.columns and plot_df["candidate_rank"].notna().any():
        plot_df = plot_df.sort_values("candidate_rank")
    elif "composite_score" in plot_df.columns and plot_df["composite_score"].notna().any():
        plot_df = plot_df.sort_values("composite_score", ascending=False)
    if len(plot_df) > cfg.max_heatmap_candidates:
        plot_df = plot_df.head(cfg.max_heatmap_candidates).copy()
    if plot_df.empty:
        matrix = np.zeros((1, len(audit_cols)))
        ylabels = ["No candidates"]
    else:
        matrix_df = plot_df[audit_cols].copy()
        for col in audit_cols:
            matrix_df[col] = matrix_df[col].fillna(False).astype(bool).astype(int)
        matrix = matrix_df.to_numpy()
        ylabels = plot_df["candidate_id"].astype(str).tolist()
    cmap = ListedColormap(["#F3F3F3", "#DDEFE2"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)
    ax_b.imshow(matrix, aspect="auto", cmap=cmap, norm=norm)
    ax_b.set_xticks(np.arange(len(audit_cols))); ax_b.set_xticklabels(labels, rotation=32, ha="right")
    ax_b.set_yticks(np.arange(len(ylabels))); ax_b.set_yticklabels(ylabels)
    ax_b.tick_params(axis="both", length=0)
    for spine in ax_b.spines.values():
        spine.set_visible(False)
    # cell separators using minor grid; not a data line plot.
    ax_b.set_xticks(np.arange(-0.5, len(audit_cols), 1), minor=True)
    ax_b.set_yticks(np.arange(-0.5, len(ylabels), 1), minor=True)
    ax_b.grid(which="minor", color="white", linewidth=1.25)
    ax_b.tick_params(which="minor", bottom=False, left=False)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = int(matrix[i, j])
            ax_b.text(j, i, "✓" if value else "×", ha="center", va="center", fontsize=9.8, color=PLOS["dark"] if value else "#8A1F1F", fontweight="bold")
    ax_b.set_title("Final candidate audit matrix", pad=9)
    ax_b.set_xlabel("Computational audit criterion")
    add_panel_label(ax_b, "B", cfg)
    legend_handles = [Patch(facecolor="#DDEFE2", edgecolor="#BFCFC4", label="Pass"), Patch(facecolor="#F3F3F3", edgecolor="#CFCFCF", label="Fail / not met")]
    ax_b.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.0, 1.16), frameon=True, ncol=2, fontsize=9.2)

    fig.subplots_adjust(left=0.12, right=0.985, top=0.88, bottom=0.23, wspace=0.26)
    outputs: List[str] = []
    panel_a = funnel.copy()
    panel_b_cols = ["candidate_id", "sequence"] + [c for c in audit_cols if c in plot_df.columns]
    panel_b = plot_df[panel_b_cols].copy() if not plot_df.empty else pd.DataFrame(columns=panel_b_cols)
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "Supplementary_Figure_S12_panel_a_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S12_panel_b_source_data.csv"))
    outputs.append(write_csv(pd.concat([dataframe_with_panel(panel_a, "Supplementary Figure S12", "A"), dataframe_with_panel(panel_b, "Supplementary Figure S12", "B")], ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S12_source_data_all_panels.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S12_candidate_audit", cfg)
    return figs, outputs


def descriptor_bins(train_vals: np.ndarray, cand_vals: np.ndarray, n_bins: int = 30) -> np.ndarray:
    vals = np.concatenate([train_vals[np.isfinite(train_vals)], cand_vals[np.isfinite(cand_vals)]])
    if len(vals) == 0:
        return np.linspace(0, 1, n_bins + 1)
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    if lo == hi:
        lo -= 0.5; hi += 0.5
    return np.linspace(lo, hi, n_bins + 1)


def build_s13_descriptor_support(final_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    specs = [
        ("length", "Length support", "Peptide length", "A"),
        ("net_charge", "Net-charge support", "Net charge at pH 7", "B"),
        ("hydrophobicity", "Hydrophobicity support", "Mean hydrophobicity", "C"),
        ("entropy", "Entropy support", "Shannon entropy", "D"),
    ]
    fig = plt.figure(figsize=(14.2, 9.0))
    gs = GridSpec(2, 2, figure=fig)
    all_frames: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, object]] = []
    outputs: List[str] = []
    for idx, (descriptor, title, xlabel, panel) in enumerate(specs):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        train_vals = train_df[descriptor].dropna().to_numpy(float)
        candidate_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])
        bins = descriptor_bins(train_vals, candidate_vals, n_bins=32)
        train_counts, bin_edges = np.histogram(train_vals, bins=bins)
        train_density = train_counts / max(train_counts.sum(), 1) / np.diff(bin_edges)
        centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        widths = np.diff(bin_edges)
        ax.bar(centers, train_density, width=widths * 0.96, color=PLOS["light"], edgecolor="white", linewidth=0.35, alpha=0.90, zorder=1, label="Training reference")
        row = reference_ranges.loc[reference_ranges["descriptor"] == descriptor].iloc[0]
        low, high = float(row["range_low"]), float(row["range_high"])
        # Reference support as a patch, not a line.
        ax.axvspan(low, high, color=PLOS["mint"], alpha=0.16, zorder=0)
        ax.set_title(title, pad=8)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Training-reference density")
        style_axis(ax, "y")
        add_panel_label(ax, panel, cfg)
        cand_counts, _ = np.histogram(candidate_vals, bins=bin_edges)
        ax2 = ax.twinx()
        ax2.bar(centers, cand_counts, width=widths * 0.60, color=PLOS["coral"], edgecolor="#B84F42", linewidth=0.6, alpha=0.88, zorder=3, label="Final-candidate bin count")
        ax2.set_ylabel("Final-candidate count")
        ax2.set_ylim(0, max(1, int(cand_counts.max()) + 1))
        ax2.spines["top"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.spines["right"].set_color(PLOS["dark"])
        ax2.tick_params(axis="y", width=0.75, length=3.5, colors=PLOS["dark"])
        n_within = int(((candidate_vals >= low) & (candidate_vals <= high)).sum()) if len(candidate_vals) else 0
        n_cand = int(len(candidate_vals))
        cand_median = float(np.nanmedian(candidate_vals)) if len(candidate_vals) else np.nan
        cand_min = float(np.nanmin(candidate_vals)) if len(candidate_vals) else np.nan
        cand_max = float(np.nanmax(candidate_vals)) if len(candidate_vals) else np.nan
        ax.text(0.98, 0.94, f"Final candidates: {n_within}/{n_cand} within interval", transform=ax.transAxes, ha="right", va="top", fontsize=9.0, bbox=dict(boxstyle="round,pad=0.25", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.7))
        panel_rows = pd.DataFrame({
            "bin_left": bin_edges[:-1],
            "bin_right": bin_edges[1:],
            "bin_center": centers,
            "training_count": train_counts,
            "training_density": train_density,
            "final_candidate_count": cand_counts,
            "descriptor": descriptor,
            "reference_interval_low": low,
            "reference_interval_high": high,
        })
        outputs.append(write_csv(panel_rows, cfg.source_data_dir / f"Supplementary_Figure_S13_panel_{panel.lower()}_source_data.csv"))
        all_frames.append(dataframe_with_panel(panel_rows, "Supplementary Figure S13", panel))
        summary_rows.append({"descriptor": descriptor, "n_final_candidates": n_cand, "n_within_reference_interval": n_within, "candidate_median": cand_median, "candidate_min": cand_min, "candidate_max": cand_max, "reference_interval_low": low, "reference_interval_high": high, "reference_interval_definition": str(row["range_definition"])})
    fig.legend(handles=[Patch(facecolor=PLOS["light"], edgecolor="white", label="Training reference"), Patch(facecolor=PLOS["mint"], edgecolor="none", label="Central reference interval"), Patch(facecolor=PLOS["coral"], edgecolor="#B84F42", label="Final-candidate bin count")], loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=3, frameon=True)
    fig.subplots_adjust(left=0.07, right=0.93, top=0.93, bottom=0.14, wspace=0.30, hspace=0.34)
    outputs.append(write_csv(pd.concat(all_frames, ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S13_source_data_all_panels.csv"))
    outputs.append(write_csv(pd.DataFrame(summary_rows), cfg.source_data_dir / "step7_descriptor_support_summary.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S13_descriptor_support", cfg)
    return figs, outputs


def build_s14_similarity_tail(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    fig = plt.figure(figsize=(14.2, 6.2))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.18])
    ax_a = fig.add_subplot(gs[0, 0])
    vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    bin_counts, _ = np.histogram(vals, bins=SIMILARITY_BINS)
    x = np.arange(len(SIMILARITY_BIN_LABELS))
    bar_colors = [PLOS["mint"], PLOS["mint"], PLOS["mint"], PLOS["mint"], PLOS["coral"]]
    ax_a.bar(x, bin_counts, color=bar_colors, edgecolor=["#75A883", "#75A883", "#75A883", "#75A883", "#B84F42"], linewidth=0.8, zorder=3)
    ax_a.set_xticks(x); ax_a.set_xticklabels(SIMILARITY_BIN_LABELS, rotation=25, ha="right")
    ax_a.set_ylabel("Final-candidate count")
    ax_a.set_xlabel("Nearest-neighbor similarity bin")
    ax_a.set_title("Final-candidate similarity-bin summary", pad=9)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "A", cfg)
    for xi, c in zip(x, bin_counts):
        ax_a.text(xi, c + max(0.05, bin_counts.max() * 0.03), f"{int(c)}", ha="center", va="bottom", fontsize=9.5)
    ax_a.text(0.98, 0.94, f"n = {len(vals)}; low-risk criterion < {cfg.similarity_low_risk_threshold:.2f}", transform=ax_a.transAxes, ha="right", va="top", fontsize=9.0, bbox=dict(boxstyle="round,pad=0.25", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.7))

    ax_b = fig.add_subplot(gs[0, 1])
    model_tags = [m for m in MODEL_ORDER if m in set(audited_df["model_tag"])]
    thresholds = list(cfg.tail_thresholds)
    x_base = np.arange(len(thresholds), dtype=float) * 1.2
    bar_w = 0.14
    gap = 0.025
    tail_rows: List[Dict[str, object]] = []
    for i, model in enumerate(model_tags):
        model_vals = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        fracs = [float(np.mean(model_vals >= thr)) if len(model_vals) else np.nan for thr in thresholds]
        offset = (i - (len(model_tags) - 1) / 2.0) * (bar_w + gap)
        ax_b.bar(x_base + offset, fracs, width=bar_w, color=MODEL_COLORS[model], edgecolor=MODEL_EDGES[model], linewidth=0.8, alpha=0.96, zorder=3)
        for thr, frac in zip(thresholds, fracs):
            tail_rows.append({"model_tag": model, "model_label": MODEL_LABELS[model], "threshold": thr, "comparison": ">= threshold", "n": int(len(model_vals)), "fraction_at_or_above_threshold": frac})
    ax_b.set_xticks(x_base); ax_b.set_xticklabels([f">={thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction of evaluated outputs ≥ threshold")
    ax_b.set_ylim(0.0, 1.02)
    ax_b.set_title("Generator-context tail-risk reference", pad=9)
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "B", cfg)
    ax_b.text(0.98, 0.94, "Denominator: per-model evaluated outputs", transform=ax_b.transAxes, ha="right", va="top", fontsize=9.0, bbox=dict(boxstyle="round,pad=0.25", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.7))
    fig.subplots_adjust(left=0.075, right=0.985, top=0.90, bottom=0.24, wspace=0.24)
    add_bottom_legend(fig, include_train=False, y=0.035)
    panel_a = pd.DataFrame({"similarity_bin": SIMILARITY_BIN_LABELS, "bin_left": SIMILARITY_BINS[:-1], "bin_right": SIMILARITY_BINS[1:], "final_candidate_count": bin_counts, "n_final_candidates": len(vals)})
    panel_b = pd.DataFrame(tail_rows)
    outputs: List[str] = []
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "Supplementary_Figure_S14_panel_a_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S14_panel_b_source_data.csv"))
    outputs.append(write_csv(pd.concat([dataframe_with_panel(panel_a, "Supplementary Figure S14", "A"), dataframe_with_panel(panel_b, "Supplementary Figure S14", "B")], ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S14_source_data_all_panels.csv"))
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "step7_similarity_bin_counts.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S14_similarity_tail_risk", cfg)
    return figs, outputs


# =============================================================================
# Reports and exports
# =============================================================================

def write_core_source_tables(audited_df: pd.DataFrame, final_df: pd.DataFrame, reference_ranges: pd.DataFrame, funnel_df: pd.DataFrame, cfg: Step7Config) -> List[str]:
    outputs = []
    outputs.append(write_csv(audited_df, cfg.source_data_dir / "step7_all_generated_audited_with_computed_flags.csv"))
    outputs.append(write_csv(final_df, cfg.source_data_dir / "step7_final_candidates_audit_table.csv"))
    outputs.append(write_csv(reference_ranges, cfg.source_data_dir / "step7_training_reference_descriptor_ranges.csv"))
    outputs.append(write_csv(funnel_df, cfg.source_data_dir / "step7_candidate_funnel_counts.csv"))
    tail_rows = []
    for model in MODEL_ORDER:
        vals = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        for threshold in cfg.tail_thresholds:
            tail_rows.append({"scope": "all_evaluated_outputs", "model_tag": model, "model_label": MODEL_LABELS[model], "threshold": threshold, "comparison": ">= threshold", "n": int(len(vals)), "fraction_at_or_above_threshold": float(np.mean(vals >= threshold)) if len(vals) else np.nan})
    final_vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    for threshold in cfg.tail_thresholds:
        tail_rows.append({"scope": "final_candidates", "model_tag": "full_cvae", "model_label": "OncoPep final candidates", "threshold": threshold, "comparison": ">= threshold", "n": int(len(final_vals)), "fraction_at_or_above_threshold": float(np.mean(final_vals >= threshold)) if len(final_vals) else np.nan})
    outputs.append(write_csv(pd.DataFrame(tail_rows), cfg.source_data_dir / "step7_similarity_tail_risk_summary.csv"))
    return outputs


def build_readiness_report(audited_df: pd.DataFrame, final_df: pd.DataFrame, inputs: Dict[str, object], cfg: Step7Config) -> Tuple[str, Dict[str, object]]:
    issues: List[str] = []
    warnings_list: List[str] = list(inputs.get("warnings_from_preparation", []))
    if final_df.empty: issues.append("No final candidates were identified.")
    if "explicit" not in str(inputs.get("final_candidate_rule", "")):
        warnings_list.append("No explicit final-candidate flag/file was detected; final candidates were inferred by fallback logic.")
    if audited_df["audit_exact_novel_vs_train"].isna().all(): issues.append("Exact novelty could not be computed or read.")
    if audited_df["nn_similarity_train"].isna().all(): issues.append("Nearest-neighbor similarity is unavailable for all rows.")
    elif audited_df["nn_similarity_train"].isna().any(): warnings_list.append("Some nearest-neighbor similarity values are missing.")
    if not audited_df["recognized_model"].all(): warnings_list.append("Some rows have unrecognized model tags.")
    if not final_df.empty and final_df["duplicate_sequence_within_input"].any(): issues.append("At least one final candidate is duplicated within the generated input table.")
    if not final_df.empty and not final_df["valid_aa_sequence"].all(): issues.append("At least one final candidate contains invalid/non-standard amino-acid characters.")
    if not final_df.empty and "audit_pass_computed" in final_df.columns and not final_df["audit_pass_computed"].all(): warnings_list.append("At least one final candidate does not pass all computed audit criteria.")
    expected_figure_stems = ["Supplementary_Figure_S12_candidate_audit", "Supplementary_Figure_S13_descriptor_support", "Supplementary_Figure_S14_similarity_tail_risk"]
    expected_source_files = [
        "Supplementary_Figure_S12_source_data_all_panels.csv", "Supplementary_Figure_S12_panel_a_source_data.csv", "Supplementary_Figure_S12_panel_b_source_data.csv",
        "Supplementary_Figure_S13_source_data_all_panels.csv", "Supplementary_Figure_S13_panel_a_source_data.csv", "Supplementary_Figure_S13_panel_b_source_data.csv", "Supplementary_Figure_S13_panel_c_source_data.csv", "Supplementary_Figure_S13_panel_d_source_data.csv",
        "Supplementary_Figure_S14_source_data_all_panels.csv", "Supplementary_Figure_S14_panel_a_source_data.csv", "Supplementary_Figure_S14_panel_b_source_data.csv",
        "step7_all_generated_audited_with_computed_flags.csv", "step7_final_candidates_audit_table.csv", "step7_training_reference_descriptor_ranges.csv", "step7_candidate_funnel_counts.csv", "step7_descriptor_support_summary.csv", "step7_similarity_bin_counts.csv", "step7_similarity_tail_risk_summary.csv",
    ]
    figure_penalties = sum(5 for stem in expected_figure_stems for ext in [".pdf", ".png", ".tiff"] if not (cfg.supplementary_dir / f"{stem}{ext}").exists())
    if final_df.empty: figure_penalties += 20
    figure_readiness = max(0, 100 - figure_penalties)
    source_penalties = sum(4 for fname in expected_source_files if not (cfg.source_data_dir / fname).exists())
    if audited_df.empty or final_df.empty: source_penalties += 20
    source_data_readiness = max(0, 100 - source_penalties)
    reproducibility_penalties = sum(6 for key in ["generated_path", "train_path", "generated_sha256", "train_sha256", "input_inventory"] if not inputs.get(key))
    reproducibility_readiness = max(0, 100 - reproducibility_penalties)
    graph_rule_readiness = 100
    scientific_penalties = 12 * len(issues) + 4 * len(warnings_list)
    scientific_readiness = max(0, 100 - scientific_penalties)
    overall = int(round(np.mean([figure_readiness, source_data_readiness, reproducibility_readiness, graph_rule_readiness, scientific_readiness])))
    result = {"figure_package_readiness_score": int(figure_readiness), "source_data_readiness_score": int(source_data_readiness), "reproducibility_readiness_score": int(reproducibility_readiness), "graph_type_rule_readiness_score": int(graph_rule_readiness), "scientific_audit_readiness_score": int(scientific_readiness), "overall_step7_readiness_score": int(overall), "ready_threshold": 98, "is_ready": overall >= 98, "n_all_rows": int(len(audited_df)), "n_final_candidates": int(len(final_df)), "issues": issues, "warnings": warnings_list, "final_candidate_rule": inputs.get("final_candidate_rule")}
    lines = [
        "# OncoPep Step 7 readiness report", "", f"- Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        f"- Figure package readiness score: **{figure_readiness}/100**", f"- Source-data readiness score: **{source_data_readiness}/100**", f"- Reproducibility readiness score: **{reproducibility_readiness}/100**", f"- Graph-type rule readiness score: **{graph_rule_readiness}/100**", f"- Scientific audit readiness score: **{scientific_readiness}/100**", f"- Overall Step 7 readiness score: **{overall}/100**", "- Ready threshold: **98/100**", f"- Final candidate rule: `{inputs.get('final_candidate_rule')}`", f"- Number of evaluated rows: {len(audited_df):,}", f"- Number of final candidates: {len(final_df):,}", "", "## Strict Step 7 graph-type rule", "", "- No candidate-level dot plots used.", "- No candidate-level lollipop plots used.", "- No line plots used.", "- No connected-point plots used.", "- No ECDF line plots used.", "- No trend lines or decorative line summaries used.", "", "## Inputs actually used", "",
    ]
    for key, value in inputs.items():
        if key != "warnings_from_preparation": lines.append(f"- {key}: `{value}`")
    lines += ["", "## Issues blocking readiness", ""]
    lines += [f"- {issue}" for issue in issues] if issues else ["- None detected."]
    lines += ["", "## Warnings", ""]
    lines += [f"- {warning}" for warning in warnings_list] if warnings_list else ["- None detected."]
    lines += ["", "## What to fix if below 98/100", "", "- Provide an explicit final-candidate file or final-candidate flag if fallback selection was used.", "- Confirm final-candidate count and exact novelty denominator.", "- Confirm S12-S14 numbering against the supplementary manuscript.", "", "## Claim boundary", "", "Step 7 supports computational auditing and downstream prioritization only. It does not establish anticancer activity, receptor binding, selectivity, toxicity, mechanism, therapeutic efficacy, or clinical relevance."]
    path = cfg.reports_dir / "step7_readiness_report.md"
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path), result


def write_panel_mapping(cfg: Step7Config) -> str:
    mapping = pd.DataFrame([
        {"figure": "S12 Fig", "panel": "A", "title": "Selection funnel", "source_data": "source_data/Supplementary_Figure_S12_panel_a_source_data.csv"},
        {"figure": "S12 Fig", "panel": "B", "title": "Final candidate audit matrix", "source_data": "source_data/Supplementary_Figure_S12_panel_b_source_data.csv"},
        {"figure": "S13 Fig", "panel": "A-D", "title": "Reference-space descriptor support", "source_data": "source_data/Supplementary_Figure_S13_panel_a-d_source_data.csv plus panel-specific CSV files"},
        {"figure": "S14 Fig", "panel": "A", "title": "Final-candidate similarity-bin summary", "source_data": "source_data/Supplementary_Figure_S14_panel_a_source_data.csv"},
        {"figure": "S14 Fig", "panel": "B", "title": "Generator-context high-similarity tail risk", "source_data": "source_data/Supplementary_Figure_S14_panel_b_source_data.csv"},
    ])
    return write_csv(mapping, cfg.reports_dir / "step7_panel_source_data_mapping.csv")


def write_readme(cfg: Step7Config) -> str:
    text = f"""OncoPep Step 7 candidate-audit outputs
====================================

Purpose
-------
This folder contains Step 7 supplementary candidate-audit outputs for the
OncoPep PLOS Computational Biology manuscript.

Figures
-------
- S12 Fig: Final candidate audit and selection funnel.
- S13 Fig: Reference-space descriptor support.
- S14 Fig: Similarity-tail risk and nearest-neighbor audit.

Strict graph-type rule
----------------------
This Step 7 version does not use candidate-level dot plots, candidate-level
lollipop plots, line plots, connected-point plots, ECDF line plots, trend lines,
or decorative line summaries. It uses funnel bars, binary matrices, binned count
bars, training histograms, and threshold-summary bar panels.

Why these are supplementary
---------------------------
Step 5 already benchmarks generator-level quality, novelty, nearest-neighbor
similarity, and surrogate condition fidelity. Step 6 covers latent-space and
sampling diagnostics. Step 7 is kept as reviewer-supporting supplementary
evidence focused on final candidate audit, descriptor support, and similarity-tail risk.

Claim boundary
--------------
These analyses are computational and hypothesis-generating. They do not establish
anticancer activity, receptor binding, selectivity, toxicity, stability,
mechanism, therapeutic efficacy, or clinical relevance.

Output root
-----------
{cfg.root}

Notebook example
----------------
from OncoPep_step7_PLOS_candidate_audit_v8_no_dot_no_line import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root={str(cfg.root)!r},
    generated_file='/real/path/to/step7_generated_final_audited.csv',
    train_file='/real/path/to/step7_train_reference_annotated.csv',
    final_file=None,
    show_figures=True,
)
outputs

Terminal example
----------------
python OncoPep_step7_PLOS_candidate_audit_v8_no_dot_no_line.py \\
  --step7-root {cfg.root} \\
  --generated-file /real/path/to/step7_generated_final_audited.csv \\
  --train-file /real/path/to/step7_train_reference_annotated.csv
"""
    path = cfg.reports_dir / "README_step7_candidate_audit_outputs.txt"
    path.write_text(text, encoding="utf-8")
    return str(path)


def write_requirements(cfg: Step7Config) -> str:
    path = cfg.code_dir / "requirements_step7_candidate_audit_minimal.txt"
    path.write_text("\n".join(["numpy", "pandas", "matplotlib", "", "# Record exact package versions at repository freeze:", "# python -m pip freeze > requirements_full_freeze.txt"]), encoding="utf-8")
    return str(path)


def write_code_snapshot(cfg: Step7Config) -> str:
    dest = cfg.code_dir / "OncoPep_step7_PLOS_candidate_audit_v8_no_dot_no_line.py"
    try:
        current = Path(__file__).resolve()
        if current.exists():
            shutil.copy2(current, dest)
            return str(dest)
    except Exception:
        pass
    try:
        source = inspect.getsource(sys.modules[__name__])
        dest.write_text(source, encoding="utf-8")
    except Exception as exc:
        dest.write_text(f"# Code snapshot unavailable: {type(exc).__name__}: {exc}\n", encoding="utf-8")
    return str(dest)


def write_manifest(cfg: Step7Config, inputs: Dict[str, object], figures: Sequence[str], source_data: Sequence[str], reports: Sequence[str], readiness: Dict[str, object], readme: str, requirements: str, code_snapshot: str) -> str:
    manifest = {"workflow": "OncoPep Step 7 candidate-audit redesign", "version": "v8 no-dot no-lollipop no-line candidate-audit PLOS-ready", "timestamp": datetime.now().isoformat(timespec="seconds"), "python_version": sys.version, "platform": platform.platform(), "config": asdict(cfg), "inputs": inputs, "figures": list(figures), "source_data": list(source_data), "reports": list(reports), "readiness": readiness, "readme": readme, "requirements": requirements, "code_snapshot": code_snapshot, "figure_numbering": {"main_figure": "No new main Figure 7 generated.", "S12 Fig": "Final candidate audit and selection funnel.", "S13 Fig": "Reference-space descriptor support.", "S14 Fig": "Similarity-tail risk and nearest-neighbor audit."}, "strict_graph_rule": "No candidate-level dot/lollipop/line/connected/ECDF/trend/decorative line plots.", "claim_boundary": "Computational audit only; no anticancer activity, receptor binding, or therapeutic efficacy claim."}
    path = cfg.reports_dir / "step7_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


# =============================================================================
# Workflow
# =============================================================================

def print_resolved_inputs(inputs: Dict[str, object]) -> None:
    print("\nResolved OncoPep Step 7 input files")
    print("-" * 44)
    for key in ["generated_path", "train_path", "final_path", "summary_path", "funnel_path", "input_inventory"]:
        print(f"{key}: {inputs.get(key)}")
    print(f"final_candidate_rule: {inputs.get('final_candidate_rule')}")


def _run_with_config(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    set_plot_style(cfg)
    loaded = load_inputs(cfg)
    audited_df: pd.DataFrame = loaded["audited_df"]  # type: ignore[assignment]
    train_df: pd.DataFrame = loaded["train_df"]  # type: ignore[assignment]
    final_df: pd.DataFrame = loaded["final_df"]  # type: ignore[assignment]
    reference_ranges: pd.DataFrame = loaded["reference_ranges"]  # type: ignore[assignment]
    funnel_df: pd.DataFrame = loaded["funnel_df"]  # type: ignore[assignment]
    inputs: Dict[str, object] = loaded["inputs"]  # type: ignore[assignment]
    print_resolved_inputs(inputs)
    figures: List[str] = []
    source_data: List[str] = []
    source_data.extend(write_core_source_tables(audited_df, final_df, reference_ranges, funnel_df, cfg))
    figs, src = build_s12_candidate_audit(final_df, funnel_df, cfg); figures.extend(figs); source_data.extend(src)
    figs, src = build_s13_descriptor_support(final_df, train_df, reference_ranges, cfg); figures.extend(figs); source_data.extend(src)
    figs, src = build_s14_similarity_tail(audited_df, final_df, cfg); figures.extend(figs); source_data.extend(src)
    reports: List[str] = []
    readiness_report_path, readiness = build_readiness_report(audited_df, final_df, inputs, cfg)
    reports.append(readiness_report_path)
    reports.append(write_panel_mapping(cfg))
    readme_path = write_readme(cfg)
    requirements_path = write_requirements(cfg)
    code_path = write_code_snapshot(cfg)
    manifest_path = write_manifest(cfg, inputs, figures, source_data, reports, readiness, readme_path, requirements_path, code_path)
    reports.append(manifest_path)
    return {"step7_root": str(cfg.root), "figures": figures, "source_data": source_data, "reports": reports, "readme": readme_path, "requirements": requirements_path, "code_snapshot": code_path, "inputs": inputs, "readiness": readiness}


def run_step7_candidate_audit(step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07", input_roots: Optional[Sequence[str]] = None, generated_file: Optional[str] = None, train_file: Optional[str] = None, final_file: Optional[str] = None, summary_file: Optional[str] = None, funnel_file: Optional[str] = None, reference_quantile_low: float = 0.01, reference_quantile_high: float = 0.99, similarity_low_risk_threshold: float = 0.30, top_n_final: int = 12, show_figures: bool = False, compute_missing_descriptors: bool = True, compute_nn_if_missing: bool = True, max_nn_pairs: int = 3_000_000) -> Dict[str, object]:
    cfg = Step7Config(step7_root=step7_root, generated_file=generated_file, train_file=train_file, final_file=final_file, summary_file=summary_file, funnel_file=funnel_file, reference_quantile_low=reference_quantile_low, reference_quantile_high=reference_quantile_high, similarity_low_risk_threshold=similarity_low_risk_threshold, top_n_final=top_n_final, show_figures=show_figures, compute_missing_descriptors=compute_missing_descriptors, compute_nn_if_missing=compute_nn_if_missing, max_nn_pairs=max_nn_pairs)
    if input_roots is not None:
        cfg.input_roots = tuple(input_roots) + tuple(cfg.input_roots)
    return _run_with_config(cfg)


# =============================================================================
# CLI and notebook behavior
# =============================================================================

def _is_jupyter_unknown_arg(arg: str) -> bool:
    text = str(arg)
    return text == "-f" or text == "--f" or text.startswith("-f=") or text.startswith("--f=") or ("kernel" in text and text.endswith(".json")) or ("jupyter" in text.lower() and "runtime" in text.lower())


def is_running_under_ipykernel() -> bool:
    return "ipykernel" in sys.modules or any("ipykernel" in str(arg) for arg in sys.argv)


def has_meaningful_cli_args(argv: Optional[Sequence[str]] = None) -> bool:
    if argv is None:
        argv = sys.argv[1:]
    return any(str(arg).startswith("--") and not _is_jupyter_unknown_arg(str(arg)) for arg in argv)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 7 no-dot/no-line candidate-audit supplementary figures.")
    parser.add_argument("--step7-root", default="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07")
    parser.add_argument("--input-root", action="append", default=None)
    parser.add_argument("--generated-file", default=None)
    parser.add_argument("--train-file", default=None)
    parser.add_argument("--final-file", default=None)
    parser.add_argument("--summary-file", default=None)
    parser.add_argument("--funnel-file", default=None)
    parser.add_argument("--reference-quantile-low", type=float, default=0.01)
    parser.add_argument("--reference-quantile-high", type=float, default=0.99)
    parser.add_argument("--similarity-low-risk-threshold", type=float, default=0.30)
    parser.add_argument("--top-n-final", type=int, default=12)
    parser.add_argument("--show-figures", action="store_true")
    parser.add_argument("--no-compute-missing-descriptors", action="store_true")
    parser.add_argument("--no-compute-nn-if-missing", action="store_true")
    parser.add_argument("--max-nn-pairs", type=int, default=3_000_000)
    parser.add_argument("--dpi-png", type=int, default=None)
    parser.add_argument("--dpi-tiff", type=int, default=None)
    args, unknown = parser.parse_known_args(argv)
    non_jupyter_unknown = [arg for arg in unknown if not _is_jupyter_unknown_arg(str(arg))]
    if non_jupyter_unknown:
        warnings.warn("Ignoring unrecognized arguments: " + " ".join(map(str, non_jupyter_unknown)), RuntimeWarning, stacklevel=2)
    return args


def print_notebook_instructions() -> None:
    print(
        """
OncoPep Step 7 v8 code loaded in a Jupyter/IPython kernel.

This version enforces the no-dot/no-lollipop/no-line Step 7 figure rule.
Run explicitly with:

from OncoPep_step7_PLOS_candidate_audit_v8_no_dot_no_line import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
    top_n_final=12,
)
outputs
"""
    )


def main(argv: Optional[Sequence[str]] = None) -> None:
    args = parse_args(argv)
    cfg = Step7Config(step7_root=args.step7_root, generated_file=args.generated_file, train_file=args.train_file, final_file=args.final_file, summary_file=args.summary_file, funnel_file=args.funnel_file, reference_quantile_low=args.reference_quantile_low, reference_quantile_high=args.reference_quantile_high, similarity_low_risk_threshold=args.similarity_low_risk_threshold, top_n_final=args.top_n_final, show_figures=args.show_figures, compute_missing_descriptors=not args.no_compute_missing_descriptors, compute_nn_if_missing=not args.no_compute_nn_if_missing, max_nn_pairs=args.max_nn_pairs)
    if args.input_root is not None:
        cfg.input_roots = tuple(args.input_root) + tuple(cfg.input_roots)
    if args.dpi_png is not None: cfg.dpi_png = args.dpi_png
    if args.dpi_tiff is not None: cfg.dpi_tiff = args.dpi_tiff
    outputs = _run_with_config(cfg)
    print("\nOncoPep Step 7 no-dot/no-line candidate-audit redesign complete.")
    print(json.dumps(outputs, indent=2))


if __name__ == "__main__":
    if is_running_under_ipykernel() and not has_meaningful_cli_args():
        print_notebook_instructions()
    else:
        main()

In [ ]:
outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
    top_n_final=12,
)

outputs

In [ ]:
#!/usr/bin/env python3
"""
OncoPep Step 7 PLOS Computational Biology candidate-audit redesign.

Version
-------
v9 sequential-funnel / no-dot / no-lollipop / no-line candidate-audit package

Purpose
-------
Step 7 is treated as a supplementary final-candidate audit, not as another
Step 5 generator benchmark or Step 6 latent-space diagnostic.

Strict Step 7 graph-type rule
-----------------------------
This script does NOT generate candidate-level dot plots, candidate-level
lollipop plots, line plots, connected-point plots, ECDF line plots, trend lines,
or decorative line summaries.

Generated supplementary figures
-------------------------------
S12 Fig: Final candidate audit and selection funnel
S13 Fig: Reference-space descriptor support using training histograms plus
         binned final-candidate count summaries
S14 Fig: Similarity-tail risk and nearest-neighbor audit using binned/threshold
         bar summaries

Claim boundary
--------------
These analyses support computational auditing and downstream prioritization only.
They do not establish anticancer activity, receptor binding, selectivity,
toxicity, stability, mechanism, therapeutic efficacy, or clinical relevance.

Notebook use
------------
from OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
    top_n_final=12,
)
outputs
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm


# =============================================================================
# Constants and visual style
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "OncoPep",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
    "train_ref": "Train reference",
}

MODEL_COLORS = {
    "full_cvae": PLOS["coral"],
    "conditional_gru_lm": PLOS["blue"],
    "random_condition": PLOS["mint"],
    "no_condition": PLOS["brown"],
    "small_latent": PLOS["charcoal"],
    "train_ref": PLOS["light"],
}

MODEL_EDGES = {
    "full_cvae": "#B84F42",
    "conditional_gru_lm": "#166F8A",
    "random_condition": "#75A883",
    "no_condition": "#8E5F39",
    "small_latent": "#4F4648",
    "train_ref": "#A7A7A7",
}

MODEL_TAG_ALIASES = {
    "oncoppep": "full_cvae",
    "oncopep": "full_cvae",
    "full_cvae": "full_cvae",
    "cvae": "full_cvae",
    "conditional_vae": "full_cvae",
    "conditioned_vae": "full_cvae",
    "gru-cond": "conditional_gru_lm",
    "gru_cond": "conditional_gru_lm",
    "conditional_gru": "conditional_gru_lm",
    "conditional_gru_lm": "conditional_gru_lm",
    "rand-cond": "random_condition",
    "rand_cond": "random_condition",
    "random_condition": "random_condition",
    "random conditional": "random_condition",
    "gru-uncond": "no_condition",
    "gru_uncond": "no_condition",
    "unconditional_gru": "no_condition",
    "no_condition": "no_condition",
    "vae-uncond": "small_latent",
    "vae_uncond": "small_latent",
    "unconditional_vae": "small_latent",
    "small_latent": "small_latent",
}

AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DESCRIPTORS = ("length", "net_charge", "hydrophobicity", "entropy")
DEFAULT_TAIL_THRESHOLDS = (0.10, 0.15, 0.20, 0.30)
SIMILARITY_BINS = (0.0, 0.10, 0.15, 0.20, 0.30, 1.000001)
SIMILARITY_BIN_LABELS = ("<0.10", "0.10-0.15", "0.15-0.20", "0.20-0.30", ">=0.30")

KD_SCALE = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

COLUMN_ALIASES = {
    "model_tag": ["model_tag", "model", "generator", "model_name", "method"],
    "candidate_id": ["candidate_id", "peptide_id", "id", "seq_id", "sequence_id", "candidate"],
    "sequence": ["sequence", "peptide", "seq", "peptide_sequence"],
    "is_final_candidate": [
        "is_final_candidate", "final_selected", "selected_final", "selected", "is_selected",
        "final_candidate", "passed_final_audit", "passed_audit",
    ],
    "candidate_rank": ["candidate_rank", "rank", "final_rank", "selection_rank", "priority_rank"],
    "composite_score": ["composite_score", "score", "selection_score", "priority_score", "audit_score"],
    "exact_novel_vs_train": [
        "exact_novel_vs_train", "exact_novel", "exact_novelty", "novel_vs_train",
        "novelty_flag", "is_exact_novel", "novelty_exact",
    ],
    "nn_similarity_train": [
        "nn_similarity_train", "nearest_neighbor_similarity_train", "nearest_train_similarity",
        "nearest_train_jaccard", "smax_train", "nn_train", "similarity_to_train",
        "nearest_neighbor_similarity",
    ],
    "length": ["length", "seq_length", "peptide_length", "model_ready_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "charge", "charge_pH7", "net_charge_ph7", "charge_proxy"],
    "hydrophobicity": [
        "hydrophobicity", "hydrophobicity_KD", "mean_hydrophobicity",
        "mean_kd_hydrophobicity", "mean_KD", "mean_kyte_doolittle", "kd_hydrophobicity",
        "mean_hydropathy",
    ],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy", "seq_entropy"],
    "plausibility_is_within_reference": [
        "plausibility_is_within_reference", "plausibility_within_reference", "within_reference",
        "within_ref", "is_within_reference",
    ],
}

TRAIN_ALIASES = {
    "sequence": COLUMN_ALIASES["sequence"],
    "length": COLUMN_ALIASES["length"],
    "net_charge": COLUMN_ALIASES["net_charge"],
    "hydrophobicity": COLUMN_ALIASES["hydrophobicity"],
    "entropy": COLUMN_ALIASES["entropy"],
}


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Step7Config:
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07"
    input_roots: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/source_data",
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/reports",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/tables",
        "/home/data3/Moe/nature_computational_peponco/step7_v3",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco/step8_v1/artifacts",
        "/mnt/data",
    )

    generated_file: Optional[str] = None
    train_file: Optional[str] = None
    final_file: Optional[str] = None
    summary_file: Optional[str] = None
    funnel_file: Optional[str] = None

    generated_candidates: Tuple[str, ...] = (
        "step7_generated_final_audited.csv",
        "step7_generated_final.csv",
        "step7_generated_audited.csv",
        "generated_final_audited.csv",
        "SuppFig_step7_property_generated_final_v7_source_data.csv",
        "SuppFig_step7_similarity_distribution_v7_source_data.csv",
    )
    train_candidates: Tuple[str, ...] = (
        "step7_train_reference_annotated.csv",
        "train_reference_annotated.csv",
        "SuppFig_step7_property_train_reference_v7_source_data.csv",
        "SuppFig_step7_property_train_reference_source_data.csv",
    )
    final_candidates: Tuple[str, ...] = (
        "step7_final_candidates.csv",
        "step7_final_candidates_audit_table.csv",
        "final_candidates.csv",
        "final_oncopep_candidates.csv",
        "top12_candidates.csv",
    )
    summary_candidates: Tuple[str, ...] = (
        "step7_model_summary.csv",
        "step7_final_generation_audit_summary.csv",
        "step7_reference_space_plausibility_summary.csv",
        "Fig7_step7_main_final_v7_source_data.csv",
    )
    funnel_candidates: Tuple[str, ...] = (
        "step7_candidate_funnel_counts.csv",
        "candidate_funnel_counts.csv",
        "step7_funnel_counts.csv",
        "Fig7_step7_candidate_funnel_source_data.csv",
    )

    # Full-range descriptor audit used for S12/S12B when no official plausibility flag is supplied.
    # This prevents the S12 audit criterion from being confused with the stricter S13 visual central interval.
    audit_reference_quantile_low: float = 0.00
    audit_reference_quantile_high: float = 1.00

    # Central reference interval used only for S13 visual support summaries.
    # The figure labels explicitly distinguish this from the S12 full/reference-supported audit.
    reference_quantile_low: float = 0.05
    reference_quantile_high: float = 0.95
    similarity_low_risk_threshold: float = 0.30
    tail_thresholds: Tuple[float, ...] = DEFAULT_TAIL_THRESHOLDS
    top_n_final: int = 12

    compute_missing_descriptors: bool = True
    histidine_charge: float = 0.10
    compute_nn_if_missing: bool = True
    max_nn_pairs: int = 3_000_000
    nn_metric_if_computed: str = "normalized_levenshtein_similarity"

    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    tiff_compression: str = "tiff_lzw"
    show_figures: bool = False

    font_family: str = "DejaVu Sans"
    base_font_size: float = 11.2
    axis_label_size: float = 12.0
    title_size: float = 13.8
    tick_size: float = 10.0
    legend_size: float = 10.2
    panel_label_size: float = 17.0

    seed: int = 42
    max_heatmap_candidates: int = 30
    inventory_max_files: int = 300

    @property
    def root(self) -> Path:
        return Path(self.step7_root).expanduser().resolve()

    @property
    def main_figure_dir(self) -> Path:
        return self.root / "main_figure"

    @property
    def supplementary_dir(self) -> Path:
        return self.root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.root / "code"


# =============================================================================
# General utilities
# =============================================================================

def ensure_output_tree(cfg: Step7Config) -> None:
    for path in [cfg.root, cfg.main_figure_dir, cfg.supplementary_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)


def set_plot_style(cfg: Step7Config) -> None:
    plt.rcParams.update(
        {
            "font.family": cfg.font_family,
            "font.size": cfg.base_font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "figure.titlesize": cfg.title_size,
            "axes.facecolor": PLOS["white"],
            "figure.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def inventory_csv_files(roots: Sequence[str], max_files: int = 300) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    seen: set[str] = set()
    for priority, root_raw in enumerate(roots):
        root = Path(root_raw).expanduser()
        if not root.exists():
            rows.append({"root_priority": priority, "root": str(root), "exists": False, "path": None, "filename": None, "size_bytes": None})
            continue
        if root.is_file():
            file_iter = iter([root] if root.suffix.lower() == ".csv" else [])
        else:
            file_iter = root.rglob("*.csv")
        n_seen_this_root = 0
        for file_path in file_iter:
            if n_seen_this_root >= max_files:
                break
            try:
                key = str(file_path.resolve())
            except Exception:
                continue
            if key in seen:
                continue
            seen.add(key)
            n_seen_this_root += 1
            try:
                size_bytes = file_path.stat().st_size
            except OSError:
                size_bytes = None
            rows.append({"root_priority": priority, "root": str(root), "exists": True, "path": key, "filename": file_path.name, "size_bytes": size_bytes})
    return pd.DataFrame(rows)


def normalize_path(path_raw: Optional[str]) -> Optional[Path]:
    if not path_raw:
        return None
    return Path(path_raw).expanduser().resolve()


def iter_candidate_matches(root: Path, candidate_names: Sequence[str], max_hits: int = 5) -> Iterable[Path]:
    candidate_set = set(candidate_names)
    if root.is_file():
        if root.name in candidate_set:
            yield root.resolve()
        return
    hits = 0
    for name in candidate_names:
        p = root / name
        if p.exists() and p.is_file():
            yield p.resolve()
            hits += 1
            if hits >= max_hits:
                return
    try:
        for p in root.rglob("*.csv"):
            if p.name in candidate_set:
                yield p.resolve()
                hits += 1
                if hits >= max_hits:
                    return
    except (OSError, PermissionError):
        return


def build_missing_file_diagnostic(label: str, candidate_names: Sequence[str], searched_roots: Sequence[str], inventory: Optional[pd.DataFrame], step7_root: str) -> str:
    candidate_msg = "\n".join(f"  - {name}" for name in candidate_names)
    root_msg = "\n".join(f"  - {root}" for root in searched_roots)
    inventory_msg = ""
    if inventory is not None and not inventory.empty and "path" in inventory.columns:
        existing = inventory.dropna(subset=["path"]).copy()
        if not existing.empty:
            inventory_msg = "\n\nCSV files found in searched roots (first 80):\n"
            inventory_msg += existing[["filename", "path", "size_bytes"]].head(80).to_string(index=False)
    command_msg = (
        "\n\nSuggested terminal command after locating real files:\n"
        f"python OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py \\\n"
        f"  --step7-root {step7_root} \\\n"
        "  --generated-file /REAL/PATH/step7_generated_final_audited.csv \\\n"
        "  --train-file /REAL/PATH/step7_train_reference_annotated.csv\n"
    )
    return f"Could not find required Step 7 {label}.\n\nAccepted filenames:\n{candidate_msg}\n\nSearched roots:\n{root_msg}{inventory_msg}{command_msg}"


def discover_any(candidate_names: Sequence[str], roots: Sequence[str], direct_file: Optional[str] = None, required: bool = False, label: str = "file", inventory: Optional[pd.DataFrame] = None, step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07") -> Optional[Path]:
    direct = normalize_path(direct_file)
    if direct is not None:
        if direct.exists() and direct.is_file():
            return direct
        raise FileNotFoundError(f"Direct {label} path was provided but does not exist: {direct}\nReplace placeholder paths with a real CSV path.")
    searched_roots: List[str] = []
    for root_raw in roots:
        root = Path(root_raw).expanduser()
        searched_roots.append(str(root))
        if not root.exists():
            continue
        for match in iter_candidate_matches(root, candidate_names, max_hits=1):
            return match
    if required:
        raise FileNotFoundError(build_missing_file_diagnostic(label, candidate_names, searched_roots, inventory, step7_root))
    return None


def find_first_column(df: pd.DataFrame, aliases: Sequence[str], canonical_name: str, required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for alias in aliases:
        if alias in df.columns:
            return alias
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    if required:
        raise KeyError(f"Could not find a column for '{canonical_name}'. Tried aliases: {list(aliases)}. Available columns: {list(df.columns)}")
    return None


def maybe_rename_columns(df: pd.DataFrame, aliases: Dict[str, Sequence[str]], canonical_columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    rename_map: Dict[str, str] = {}
    for canonical in canonical_columns:
        if canonical in out.columns or canonical not in aliases:
            continue
        found = find_first_column(out, aliases[canonical], canonical, required=False)
        if found is not None and found != canonical:
            rename_map[found] = canonical
    return out.rename(columns=rename_map)


def normalize_model_tag_value(value: object) -> object:
    if pd.isna(value):
        return value
    text = str(value).strip()
    key = text.lower().replace("-", "_").replace(" ", "_")
    return MODEL_TAG_ALIASES.get(key, text)


def normalize_model_tags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model_tag" in out.columns:
        out["model_tag"] = out["model_tag"].map(normalize_model_tag_value)
    return out


def as_numeric_clean(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def parse_boolish(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    text = series.astype(str).str.strip().str.lower()
    truthy = {"true", "1", "yes", "y", "selected", "final", "pass", "passed", "keep", "kept"}
    falsy = {"false", "0", "no", "n", "not_selected", "fail", "failed", "nan", "none", "", "remove"}
    return text.map(lambda x: True if x in truthy else (False if x in falsy else False))


def clean_sequence(seq: object) -> str:
    if pd.isna(seq):
        return ""
    return str(seq).strip().upper()


def valid_sequence(seq: object) -> bool:
    s = clean_sequence(seq)
    return len(s) > 0 and all(ch in AA20 for ch in s)


def peptide_length(seq: object) -> float:
    s = clean_sequence(seq)
    return float(len(s)) if s else np.nan


def peptide_net_charge(seq: object, histidine_charge: float = 0.10) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    return float(s.count("K") + s.count("R") + histidine_charge * s.count("H") - s.count("D") - s.count("E"))


def peptide_hydrophobicity(seq: object) -> float:
    s = clean_sequence(seq)
    vals = [KD_SCALE[ch] for ch in s if ch in KD_SCALE]
    return float(np.mean(vals)) if vals else np.nan


def peptide_entropy(seq: object) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    counts = np.array([s.count(aa) for aa in AA20], dtype=float)
    probs = counts[counts > 0] / len(s)
    return float(-(probs * np.log2(probs)).sum())


def normalized_levenshtein_similarity(a: str, b: str) -> float:
    a = clean_sequence(a); b = clean_sequence(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(current[j - 1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return float(1.0 - previous[-1] / max(len(a), len(b)))


def compute_nearest_similarity(sequences: Sequence[str], train_sequences: Sequence[str], max_pairs: int = 3_000_000) -> Tuple[pd.Series, str]:
    n_pairs = len(sequences) * len(train_sequences)
    if n_pairs > max_pairs:
        return pd.Series([np.nan] * len(sequences), dtype=float), f"not_computed_pair_budget_exceeded:{n_pairs}>{max_pairs}"
    train_list = [clean_sequence(s) for s in train_sequences if clean_sequence(s)]
    values: List[float] = []
    for seq in sequences:
        s = clean_sequence(seq)
        values.append(max((normalized_levenshtein_similarity(s, t) for t in train_list), default=np.nan))
    return pd.Series(values, dtype=float), "computed_normalized_levenshtein_similarity"


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def dataframe_with_panel(df: pd.DataFrame, figure: str, panel: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "panel", panel)
    out.insert(0, "figure", figure)
    return out


# =============================================================================
# Data preparation
# =============================================================================

def add_or_compute_descriptors(df: pd.DataFrame, cfg: Step7Config, role: str, warnings_list: List[str]) -> pd.DataFrame:
    out = df.copy()
    if "sequence" not in out.columns:
        return out
    if cfg.compute_missing_descriptors:
        if "length" not in out.columns or out["length"].isna().all():
            out["length"] = out["sequence"].map(peptide_length)
            warnings_list.append(f"{role}: length was missing and was computed from sequence.")
        if "net_charge" not in out.columns or out["net_charge"].isna().all():
            out["net_charge"] = out["sequence"].map(lambda s: peptide_net_charge(s, cfg.histidine_charge))
            warnings_list.append(f"{role}: net_charge was missing and was approximated from residue counts.")
        if "hydrophobicity" not in out.columns or out["hydrophobicity"].isna().all():
            out["hydrophobicity"] = out["sequence"].map(peptide_hydrophobicity)
            warnings_list.append(f"{role}: hydrophobicity was missing and was computed using the Kyte-Doolittle mean.")
        if "entropy" not in out.columns or out["entropy"].isna().all():
            out["entropy"] = out["sequence"].map(peptide_entropy)
            warnings_list.append(f"{role}: entropy was missing and was computed as Shannon entropy over AA20.")
    for col in DESCRIPTORS:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    return out


def prepare_generated_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, COLUMN_ALIASES, COLUMN_ALIASES.keys())
    if "sequence" not in out.columns:
        raise KeyError(f"Generated/final audited table is missing a sequence column. Available columns: {list(raw.columns)}")
    if "model_tag" not in out.columns:
        out["model_tag"] = "full_cvae"
        warnings_list.append("generated: model_tag was missing; all rows were assigned to full_cvae/OncoPep.")
    out = normalize_model_tags(out)
    out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "generated", warnings_list)
    for col in ["nn_similarity_train", "candidate_rank", "composite_score"]:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    if "candidate_id" not in out.columns:
        out["candidate_id"] = [f"pep_{i+1:05d}" for i in range(len(out))]
        warnings_list.append("generated: candidate_id was missing and sequential IDs were assigned.")
    if "is_final_candidate" in out.columns:
        out["is_final_candidate"] = parse_boolish(out["is_final_candidate"])
    else:
        out["is_final_candidate"] = False
    if "plausibility_is_within_reference" in out.columns:
        out["plausibility_is_within_reference"] = parse_boolish(out["plausibility_is_within_reference"])
    out["valid_aa_sequence"] = out["sequence"].map(valid_sequence)
    out["duplicate_sequence_within_input"] = out.duplicated("sequence", keep=False)
    out["recognized_model"] = out["model_tag"].isin(MODEL_ORDER)
    return out


def prepare_train_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, TRAIN_ALIASES, TRAIN_ALIASES.keys())
    if "sequence" in out.columns:
        out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "train_reference", warnings_list)
    missing = [col for col in DESCRIPTORS if col not in out.columns]
    if missing:
        raise KeyError("Training-reference table is missing descriptor columns and they could not be computed: " + ", ".join(missing))
    for col in DESCRIPTORS:
        out[col] = as_numeric_clean(out[col])
    out["model_tag"] = "train_ref"
    return out


def compute_reference_ranges(train_df: pd.DataFrame, cfg: Step7Config) -> pd.DataFrame:
    """Compute both full/audit and central visual descriptor intervals.

    audit_range_* is used for the S12/S12B reference-supported descriptor audit
    only when an official `plausibility_is_within_reference` flag is not present.
    central_range_* is used for S13 visual support summaries and is intentionally
    stricter by default. Keeping these separate prevents the S12/S13 consistency
    problem where one panel appears to show all candidates passing while another
    reports fewer candidates within the visual central interval.
    """
    rows: List[Dict[str, object]] = []
    for descriptor in DESCRIPTORS:
        vals = train_df[descriptor].dropna().to_numpy(float)
        if len(vals) == 0:
            raise ValueError(f"Training-reference descriptor '{descriptor}' has no finite values.")
        rows.append(
            {
                "descriptor": descriptor,
                "n_train": int(len(vals)),
                "min": float(np.min(vals)),
                "q01": float(np.quantile(vals, 0.01)),
                "q05": float(np.quantile(vals, 0.05)),
                "q25": float(np.quantile(vals, 0.25)),
                "median": float(np.quantile(vals, 0.50)),
                "q75": float(np.quantile(vals, 0.75)),
                "q95": float(np.quantile(vals, 0.95)),
                "q99": float(np.quantile(vals, 0.99)),
                "max": float(np.max(vals)),
                "audit_range_low": float(np.quantile(vals, cfg.audit_reference_quantile_low)),
                "audit_range_high": float(np.quantile(vals, cfg.audit_reference_quantile_high)),
                "audit_range_definition": f"training quantile [{cfg.audit_reference_quantile_low}, {cfg.audit_reference_quantile_high}]",
                "central_range_low": float(np.quantile(vals, cfg.reference_quantile_low)),
                "central_range_high": float(np.quantile(vals, cfg.reference_quantile_high)),
                "central_range_definition": f"central training interval: quantile [{cfg.reference_quantile_low}, {cfg.reference_quantile_high}]",
            }
        )
    return pd.DataFrame(rows)

def add_computed_audit_columns(generated_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = generated_df.copy()

    if "sequence" in train_df.columns:
        train_sequences = set(train_df["sequence"].dropna().astype(str).str.strip().str.upper())
        out["computed_exact_novel_vs_train"] = ~out["sequence"].isin(train_sequences)
    else:
        out["computed_exact_novel_vs_train"] = np.nan
        warnings_list.append("Exact novelty could not be computed because training sequence column is absent.")

    if "exact_novel_vs_train" in out.columns:
        out["exact_novel_vs_train"] = parse_boolish(out["exact_novel_vs_train"])
        out["audit_exact_novel_vs_train"] = out["exact_novel_vs_train"]
    else:
        out["audit_exact_novel_vs_train"] = out["computed_exact_novel_vs_train"]

    if "nn_similarity_train" not in out.columns or out["nn_similarity_train"].isna().all():
        if cfg.compute_nn_if_missing and "sequence" in train_df.columns:
            nn_values, nn_status = compute_nearest_similarity(
                out["sequence"].astype(str).tolist(),
                train_df["sequence"].dropna().astype(str).tolist(),
                max_pairs=cfg.max_nn_pairs,
            )
            out["nn_similarity_train"] = nn_values
            out["nn_similarity_train_source"] = nn_status
            warnings_list.append(f"generated: nn_similarity_train was missing; status={nn_status}.")
        else:
            out["nn_similarity_train"] = np.nan
            out["nn_similarity_train_source"] = "missing"
            warnings_list.append("generated: nn_similarity_train is missing and was not computed.")
    else:
        out["nn_similarity_train"] = as_numeric_clean(out["nn_similarity_train"])
        out["nn_similarity_train_source"] = "input_column"

    # Descriptor support is tracked in two layers:
    #   1) audit/full range for S12/S12B pass/fail criteria
    #   2) central visual interval for S13 descriptor-support summaries
    audit_cols: List[str] = []
    central_cols: List[str] = []
    for _, row in reference_ranges.iterrows():
        descriptor = str(row["descriptor"])
        audit_low = float(row["audit_range_low"])
        audit_high = float(row["audit_range_high"])
        central_low = float(row["central_range_low"])
        central_high = float(row["central_range_high"])
        audit_col = f"within_audit_ref_{descriptor}"
        central_col = f"within_central_ref_{descriptor}"
        out[audit_col] = out[descriptor].between(audit_low, audit_high, inclusive="both")
        out[central_col] = out[descriptor].between(central_low, central_high, inclusive="both")
        # Backward-compatible alias used by older source-data checks.
        out[f"within_ref_{descriptor}"] = out[audit_col]
        audit_cols.append(audit_col)
        central_cols.append(central_col)

    computed_audit_within = out[audit_cols].all(axis=1)
    computed_central_within = out[central_cols].all(axis=1)

    if "plausibility_is_within_reference" in out.columns:
        official = parse_boolish(out["plausibility_is_within_reference"])
        out["audit_within_reference_all_descriptors"] = official.fillna(computed_audit_within)
        out["reference_support_source"] = "input_plausibility_is_within_reference_with_computed_fallback"
    else:
        out["audit_within_reference_all_descriptors"] = computed_audit_within
        out["reference_support_source"] = "computed_descriptor_audit_range"

    out["computed_within_reference_audit_range"] = computed_audit_within
    out["computed_within_reference_central_interval"] = computed_central_within
    out["audit_low_similarity"] = out["nn_similarity_train"] < cfg.similarity_low_risk_threshold
    out["audit_valid_sequence"] = out["valid_aa_sequence"].astype(bool)
    out["audit_nonduplicate_sequence"] = ~out["duplicate_sequence_within_input"].astype(bool)

    audit_summary_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_within_reference_all_descriptors",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
    ]
    usable_cols = [c for c in audit_summary_cols if not out[c].isna().all()]
    out["audit_pass_computed"] = out[usable_cols].fillna(True).all(axis=1)
    return out

def prepare_final_file_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = prepare_generated_df(raw, cfg, warnings_list)
    out["is_final_candidate"] = True
    return out


def infer_final_candidates(audited_df: pd.DataFrame, explicit_final_df: Optional[pd.DataFrame], cfg: Step7Config, warnings_list: List[str]) -> Tuple[pd.DataFrame, str]:
    if explicit_final_df is not None and not explicit_final_df.empty:
        final_sequences = set(explicit_final_df["sequence"].dropna().astype(str))
        if final_sequences:
            matched = audited_df[audited_df["sequence"].isin(final_sequences)].copy()
            if not matched.empty:
                return matched, "explicit separate final-candidate file matched by sequence"
        return explicit_final_df.copy(), "explicit separate final-candidate file"
    out = audited_df.copy()
    if "is_final_candidate" in out.columns and out["is_final_candidate"].any():
        return out.loc[out["is_final_candidate"]].copy(), "explicit final-candidate flag"
    oncopep = out.loc[out["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = out.copy()
        warnings_list.append("No full_cvae rows found; final-candidate fallback used all generated rows.")
    if "candidate_rank" in oncopep.columns and oncopep["candidate_rank"].notna().any():
        return oncopep.sort_values("candidate_rank", ascending=True).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by candidate_rank"
    if "composite_score" in oncopep.columns and oncopep["composite_score"].notna().any():
        return oncopep.sort_values("composite_score", ascending=False).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by composite_score"
    passing = oncopep.loc[oncopep["audit_pass_computed"]].copy()
    if not passing.empty:
        return passing.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(passing))} OncoPep rows passing computed audit"
    return oncopep.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(oncopep))} OncoPep rows because no explicit final-candidate flag was present"


def prepare_funnel_df(audited_df: pd.DataFrame, final_df: pd.DataFrame, explicit_funnel_df: Optional[pd.DataFrame], cfg: Step7Config) -> pd.DataFrame:
    """Create a mathematically valid Step 7 selection funnel.

    Earlier versions displayed independent pass counts as if they were sequential
    retained counts, which can produce impossible percentages (for example, >100%
    retained at a later stage). This function fixes that by recomputing the
    funnel as a cumulative sequential filter series. It also records the
    independent pass count for each criterion for source-data transparency.
    """
    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()

    def _count(mask: pd.Series) -> int:
        return int(mask.fillna(False).astype(bool).sum())

    base_mask = pd.Series(True, index=oncopep.index)
    stage_specs = [
        ("OncoPep evaluated outputs", None, "All evaluated OncoPep/full-CVAE outputs used as the Step 7 denominator."),
        ("Valid amino-acid sequences", "audit_valid_sequence", "Valid amino-acid alphabet after sequence-format audit."),
        ("Exact-novel vs training", "audit_exact_novel_vs_train", "No exact match to training peptides."),
        ("Reference-supported descriptors", "audit_within_reference_all_descriptors", "Descriptor support using the official plausibility flag when available; otherwise the audit/full training descriptor range."),
        (f"NN similarity < {cfg.similarity_low_risk_threshold:.2f}", "audit_low_similarity", "Nearest-neighbor similarity to training peptides below the configured low-similarity threshold."),
    ]

    rows: List[Dict[str, object]] = []
    cumulative_mask = base_mask.copy()
    previous_n: Optional[int] = None
    initial_n = int(len(oncopep))
    for order, (stage, col, definition) in enumerate(stage_specs, start=1):
        if col is None:
            criterion_mask = base_mask.copy()
            cumulative_mask = base_mask.copy()
        else:
            criterion_mask = oncopep[col].fillna(False).astype(bool) if col in oncopep.columns else pd.Series(False, index=oncopep.index)
            cumulative_mask = cumulative_mask & criterion_mask
        sequential_n = int(cumulative_mask.sum())
        independent_n = int(criterion_mask.sum()) if col is not None else initial_n
        retained_previous = 100.0 if previous_n in (None, 0) else 100.0 * sequential_n / float(previous_n)
        retained_initial = np.nan if initial_n == 0 else 100.0 * sequential_n / float(initial_n)
        rows.append(
            {
                "stage_order": order,
                "stage": stage,
                "n": sequential_n,
                "sequential_n": sequential_n,
                "independent_pass_n": independent_n,
                "retained_from_previous_pct": retained_previous,
                "retained_from_initial_pct": retained_initial,
                "criterion_column": col,
                "definition": definition,
                "source": "computed_true_sequential_funnel_from_audited_table",
            }
        )
        previous_n = sequential_n

    final_sequence_set = set(final_df["sequence"].dropna().astype(str)) if "sequence" in final_df.columns else set()
    if final_sequence_set:
        final_in_cumulative = int(oncopep.loc[cumulative_mask, "sequence"].isin(final_sequence_set).sum()) if "sequence" in oncopep.columns else len(final_df)
    else:
        final_in_cumulative = len(final_df)
    previous_n = previous_n or 0
    # The final candidate set is a frozen selection set. It is reported as the
    # final Step 7 output count and as percentage of the initial evaluated pool.
    # If an explicit final set is not a strict subset of the previous cumulative
    # audit stage, retained_from_previous_pct is intentionally set to NaN rather
    # than showing an invalid >100% funnel percentage.
    if previous_n and len(final_df) <= previous_n:
        final_retained_previous = 100.0 * len(final_df) / float(previous_n)
    else:
        final_retained_previous = np.nan
    rows.append(
        {
            "stage_order": len(stage_specs) + 1,
            "stage": "Final candidates",
            "n": int(len(final_df)),
            "sequential_n": int(len(final_df)),
            "independent_pass_n": int(len(final_df)),
            "retained_from_previous_pct": final_retained_previous,
            "retained_from_initial_pct": (100.0 * len(final_df) / float(initial_n)) if initial_n else np.nan,
            "criterion_column": "is_final_candidate_or_selection_rule",
            "definition": "Frozen final-candidate set if supplied; otherwise fallback selection reported in the readiness report. Percentage labels use initial denominator to avoid impossible retained-from-previous values.",
            "source": "final_candidate_table_or_inferred_selection",
            "n_final_candidates_found_within_previous_stage": final_in_cumulative,
        }
    )

    funnel = pd.DataFrame(rows)
    # If user supplies an explicit funnel table, keep it as a separate reference file but do not use it
    # for plotting unless it is already monotonic and sequential. The plotted funnel always uses the
    # verified sequential series above to avoid impossible retained percentages.
    if explicit_funnel_df is not None and not explicit_funnel_df.empty:
        ref = explicit_funnel_df.copy()
        ref.to_csv(cfg.source_data_dir / "step7_input_funnel_file_reference_not_plotted.csv", index=False)
    return funnel

def load_inputs(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    warnings_list: List[str] = []
    roots = tuple(str(Path(r).expanduser()) for r in cfg.input_roots)
    inventory = inventory_csv_files(roots, cfg.inventory_max_files)
    if not inventory.empty:
        write_csv(inventory, cfg.reports_dir / "step7_input_inventory.csv")
    generated_path = discover_any(cfg.generated_candidates, roots, direct_file=cfg.generated_file, required=True, label="generated/final audited table", inventory=inventory, step7_root=str(cfg.root))
    train_path = discover_any(cfg.train_candidates, roots, direct_file=cfg.train_file, required=True, label="training-reference table", inventory=inventory, step7_root=str(cfg.root))
    # A final-candidate table is intentionally used only when explicitly supplied.
    # This prevents circular reuse of previously generated
    # `step7_final_candidates_audit_table.csv` files from the output folder.
    final_path = discover_any(cfg.final_candidates, roots, direct_file=cfg.final_file, required=False, label="separate final-candidate table", inventory=inventory, step7_root=str(cfg.root)) if cfg.final_file else None
    summary_path = discover_any(cfg.summary_candidates, roots, direct_file=cfg.summary_file, required=False, label="model summary table", inventory=inventory, step7_root=str(cfg.root))
    funnel_path = discover_any(cfg.funnel_candidates, roots, direct_file=cfg.funnel_file, required=False, label="candidate funnel table", inventory=inventory, step7_root=str(cfg.root)) if cfg.funnel_file else None
    assert generated_path is not None and train_path is not None
    generated_raw = pd.read_csv(generated_path)
    train_raw = pd.read_csv(train_path)
    final_raw = pd.read_csv(final_path) if final_path is not None else None
    summary_df = pd.read_csv(summary_path) if summary_path is not None else None
    funnel_df = pd.read_csv(funnel_path) if funnel_path is not None else None
    generated_df = prepare_generated_df(generated_raw, cfg, warnings_list)
    train_df = prepare_train_df(train_raw, cfg, warnings_list)
    explicit_final_df = prepare_final_file_df(final_raw, cfg, warnings_list) if final_raw is not None else None
    reference_ranges = compute_reference_ranges(train_df, cfg)
    audited_df = add_computed_audit_columns(generated_df, train_df, reference_ranges, cfg, warnings_list)
    if explicit_final_df is not None:
        explicit_final_df = add_computed_audit_columns(explicit_final_df, train_df, reference_ranges, cfg, warnings_list)
    final_df, final_rule = infer_final_candidates(audited_df, explicit_final_df, cfg, warnings_list)
    if "is_final_candidate" in audited_df.columns:
        audited_df.loc[audited_df["candidate_id"].isin(final_df["candidate_id"]), "is_final_candidate"] = True
    funnel_final = prepare_funnel_df(audited_df, final_df, funnel_df, cfg)
    inputs = {
        "generated_path": str(generated_path),
        "train_path": str(train_path),
        "final_path": str(final_path) if final_path else None,
        "summary_path": str(summary_path) if summary_path else None,
        "funnel_path": str(funnel_path) if funnel_path else None,
        "input_inventory": str(cfg.reports_dir / "step7_input_inventory.csv"),
        "generated_sha256": sha256_file(generated_path),
        "train_sha256": sha256_file(train_path),
        "final_sha256": sha256_file(final_path),
        "summary_sha256": sha256_file(summary_path),
        "funnel_sha256": sha256_file(funnel_path),
        "final_candidate_rule": final_rule,
        "warnings_from_preparation": warnings_list,
        "nn_metric_if_computed": cfg.nn_metric_if_computed,
        "strict_graph_rule": "No candidate-level dot/lollipop/line/connected/ECDF/trend/decorative line plots.",
    }
    return {"generated_df": generated_df, "train_df": train_df, "reference_ranges": reference_ranges, "audited_df": audited_df, "final_df": final_df, "funnel_df": funnel_final, "summary_df": summary_df, "inventory": inventory, "inputs": inputs}


# =============================================================================
# Plot helpers
# =============================================================================

def save_figure(fig: plt.Figure, out_base: Path, cfg: Step7Config) -> List[str]:
    outputs: List[str] = []
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf"); fig.savefig(p, bbox_inches="tight"); outputs.append(str(p))
    if cfg.export_png:
        p = out_base.with_suffix(".png"); fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png); outputs.append(str(p))
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        try:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, pil_kwargs={"compression": cfg.tiff_compression})
        except TypeError:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff)
        outputs.append(str(p))
    if cfg.show_figures:
        plt.show()
    else:
        plt.close(fig)
    return outputs


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.75)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.85); ax.spines[side].set_color(PLOS["dark"])
    ax.tick_params(axis="both", width=0.75, length=3.5)


def add_panel_label(ax: plt.Axes, label: str, cfg: Step7Config, x: float = -0.08, y: float = 1.03) -> None:
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=cfg.panel_label_size, fontweight="bold", color=PLOS["dark"], clip_on=False)


def add_bottom_legend(fig: plt.Figure, include_train: bool, y: float = 0.02, ncol: Optional[int] = None) -> None:
    handles = []
    if include_train:
        handles.append(Patch(facecolor=MODEL_COLORS["train_ref"], edgecolor=MODEL_EDGES["train_ref"], label=MODEL_LABELS["train_ref"]))
    for tag in MODEL_ORDER:
        handles.append(Patch(facecolor=MODEL_COLORS[tag], edgecolor=MODEL_EDGES[tag], label=MODEL_LABELS[tag]))
    if ncol is None: ncol = len(handles)
    legend = fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, y), ncol=ncol, frameon=True, columnspacing=1.0, handletextpad=0.45, borderpad=0.40)
    legend.get_frame().set_facecolor("#FAFAFA"); legend.get_frame().set_edgecolor("#CFCFCF"); legend.get_frame().set_linewidth(0.8)


# =============================================================================
# Figure builders: no candidate dots, no lollipop, no line plots.
# =============================================================================

def build_s12_candidate_audit(final_df: pd.DataFrame, funnel_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    """Build S12 with a true sequential funnel and light binary audit matrix."""
    fig = plt.figure(figsize=(15.2, 6.3))
    gs = GridSpec(1, 2, width_ratios=[0.98, 1.30], figure=fig)

    ax_a = fig.add_subplot(gs[0, 0])
    funnel = funnel_df.copy().sort_values("stage_order") if "stage_order" in funnel_df.columns else funnel_df.copy()
    funnel["n"] = as_numeric_clean(funnel["n"]).fillna(0).astype(int)
    y = np.arange(len(funnel))[::-1]
    colors = [PLOS["coral"] if stage != "Final candidates" else PLOS["brown"] for stage in funnel["stage"]]
    edges = ["#B84F42" if stage != "Final candidates" else "#8E5F39" for stage in funnel["stage"]]
    bars = ax_a.barh(y, funnel["n"], color=colors, edgecolor=edges, linewidth=0.8, zorder=3)
    ax_a.set_yticks(y)
    ax_a.set_yticklabels(funnel["stage"])
    ax_a.set_xlabel("Number of sequences")
    ax_a.set_title("Sequential candidate-audit funnel", pad=11)
    style_axis(ax_a, "x")
    add_panel_label(ax_a, "A", cfg, x=-0.08, y=1.03)
    max_n = max(float(funnel["n"].max()), 1.0)

    for bar, (_, row) in zip(bars, funnel.iterrows()):
        n = int(row["n"])
        pct_initial = row.get("retained_from_initial_pct", np.nan)
        pct_prev = row.get("retained_from_previous_pct", np.nan)
        if row["stage"] == "Final candidates":
            label = f"{n:,}; {pct_initial:.1f}% of initial" if pd.notna(pct_initial) else f"{n:,}"
        else:
            label = f"{n:,}; {pct_initial:.1f}% of initial" if pd.notna(pct_initial) else f"{n:,}"
        ax_a.text(bar.get_width() + max_n * 0.014, bar.get_y() + bar.get_height() / 2, label, va="center", ha="left", fontsize=9.2, color=PLOS["dark"])

    final_n = int(funnel.loc[funnel["stage"].eq("Final candidates"), "n"].iloc[0]) if funnel["stage"].eq("Final candidates").any() else len(final_df)
    ax_a.text(
        0.98,
        0.06,
        f"Final candidates = {final_n}\nSequential counts only",
        transform=ax_a.transAxes,
        ha="right",
        va="bottom",
        fontsize=9.3,
        bbox=dict(boxstyle="round,pad=0.32", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.8),
    )

    ax_b = fig.add_subplot(gs[0, 1])
    audit_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_within_reference_all_descriptors",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
        "audit_pass_computed",
    ]
    labels = [
        "Valid AA",
        "Exact novel",
        "Reference-supported",
        f"NN < {cfg.similarity_low_risk_threshold:.2f}",
        "Non-duplicate",
        "Overall pass",
    ]
    plot_df = final_df.copy()
    if "candidate_rank" in plot_df.columns and plot_df["candidate_rank"].notna().any():
        plot_df = plot_df.sort_values("candidate_rank")
    elif "composite_score" in plot_df.columns and plot_df["composite_score"].notna().any():
        plot_df = plot_df.sort_values("composite_score", ascending=False)
    if len(plot_df) > cfg.max_heatmap_candidates:
        plot_df = plot_df.head(cfg.max_heatmap_candidates).copy()

    if plot_df.empty:
        mat = np.zeros((1, len(audit_cols)), dtype=int)
        ylabels = ["No candidates"]
    else:
        mat_df = plot_df[audit_cols].copy()
        for col in audit_cols:
            mat_df[col] = mat_df[col].fillna(False).astype(bool).astype(int)
        mat = mat_df.to_numpy(dtype=int)
        if "candidate_rank" in plot_df.columns and plot_df["candidate_rank"].notna().any():
            ylabels = [f"rank {int(r)}" for r in plot_df["candidate_rank"]]
        else:
            ylabels = plot_df["candidate_id"].astype(str).tolist()

    cmap = ListedColormap(["#F4F4F4", "#DCEFE2"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)
    ax_b.imshow(mat, aspect="auto", cmap=cmap, norm=norm)
    ax_b.set_xticks(np.arange(len(audit_cols)))
    ax_b.set_xticklabels(labels, rotation=35, ha="right")
    ax_b.set_yticks(np.arange(len(ylabels)))
    ax_b.set_yticklabels(ylabels)
    ax_b.set_xlabel("Computational audit criterion")
    ax_b.set_title("Final candidate audit matrix", pad=11)
    for spine in ax_b.spines.values():
        spine.set_visible(False)
    ax_b.tick_params(axis="both", length=0)
    add_panel_label(ax_b, "B", cfg, x=-0.08, y=1.03)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax_b.text(j, i, "✓" if mat[i, j] else "×", ha="center", va="center", fontsize=9.2, color=PLOS["dark"])
    ax_b.set_xticks(np.arange(-0.5, len(audit_cols), 1), minor=True)
    ax_b.set_yticks(np.arange(-0.5, len(ylabels), 1), minor=True)
    ax_b.grid(which="minor", color=PLOS["white"], linewidth=1.1)
    ax_b.tick_params(which="minor", bottom=False, left=False)
    handles = [Patch(facecolor="#DCEFE2", edgecolor="#BFD9C6", label="Pass"), Patch(facecolor="#F4F4F4", edgecolor="#D0D0D0", label="Fail / not met")]
    leg = ax_b.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.0, 1.16), ncol=2, frameon=True, fontsize=9.0)
    leg.get_frame().set_facecolor("#FAFAFA")
    leg.get_frame().set_edgecolor("#CFCFCF")
    leg.get_frame().set_linewidth(0.8)

    fig.subplots_adjust(left=0.12, right=0.985, top=0.88, bottom=0.20, wspace=0.24)

    outputs: List[str] = []
    panel_a = funnel.copy()
    panel_b_cols = ["candidate_id", "sequence"] + [c for c in ["candidate_rank", "composite_score"] if c in plot_df.columns] + audit_cols
    panel_b = plot_df[panel_b_cols].copy() if not plot_df.empty else pd.DataFrame(columns=panel_b_cols)
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "Supplementary_Figure_S12_panel_a_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S12_panel_b_source_data.csv"))
    outputs.append(write_csv(pd.concat([dataframe_with_panel(panel_a, "Supplementary Figure S12", "A"), dataframe_with_panel(panel_b, "Supplementary Figure S12", "B")], ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S12_source_data_all_panels.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S12_candidate_audit", cfg)
    return figs, outputs

def descriptor_bins(train_vals: np.ndarray, cand_vals: np.ndarray, n_bins: int = 30) -> np.ndarray:
    vals = np.concatenate([train_vals[np.isfinite(train_vals)], cand_vals[np.isfinite(cand_vals)]])
    if len(vals) == 0:
        return np.linspace(0, 1, n_bins + 1)
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    if lo == hi:
        lo -= 0.5; hi += 0.5
    return np.linspace(lo, hi, n_bins + 1)


def build_s13_descriptor_support(final_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    """Build S13 using training histograms plus binned final-candidate counts.

    No candidate-level dots or lollipops are used. The shaded band is explicitly
    the stricter central training interval, not the S12 reference-supported audit
    criterion.
    """
    specs = [
        ("length", "Length support", "Peptide length", "A"),
        ("net_charge", "Net-charge support", "Net charge at pH 7", "B"),
        ("hydrophobicity", "Hydrophobicity support", "Mean hydrophobicity", "C"),
        ("entropy", "Entropy support", "Shannon entropy", "D"),
    ]
    fig = plt.figure(figsize=(14.5, 8.8))
    gs = GridSpec(2, 2, figure=fig)
    all_frames: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, object]] = []
    outputs: List[str] = []

    for idx, (descriptor, title, xlabel, panel) in enumerate(specs):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        train_vals = train_df[descriptor].dropna().to_numpy(float)
        cand_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])
        row = reference_ranges.loc[reference_ranges["descriptor"] == descriptor].iloc[0]
        central_low = float(row["central_range_low"])
        central_high = float(row["central_range_high"])
        audit_low = float(row["audit_range_low"])
        audit_high = float(row["audit_range_high"])
        interval_label = f"central {int(cfg.reference_quantile_low*100)}-{int(cfg.reference_quantile_high*100)}% interval"

        # Use common bin edges so candidate counts align with training histogram bins.
        if len(train_vals):
            base_edges = np.histogram_bin_edges(train_vals, bins=32)
        else:
            base_edges = np.linspace(0, 1, 6)
        if len(cand_vals):
            low = min(float(np.nanmin(train_vals)) if len(train_vals) else np.nanmin(cand_vals), float(np.nanmin(cand_vals)))
            high = max(float(np.nanmax(train_vals)) if len(train_vals) else np.nanmax(cand_vals), float(np.nanmax(cand_vals)))
            if low < base_edges[0] or high > base_edges[-1]:
                base_edges = np.linspace(low, high, 33)

        train_density, edges = np.histogram(train_vals, bins=base_edges, density=True)
        widths = np.diff(edges)
        centers = edges[:-1] + widths / 2
        ax.bar(centers, train_density, width=widths * 0.94, color=PLOS["light"], edgecolor=PLOS["white"], linewidth=0.4, align="center", label="Training reference", zorder=1)
        ax.axvspan(central_low, central_high, color=PLOS["mint"], alpha=0.16, zorder=0)

        ax2 = ax.twinx()
        cand_counts, _ = np.histogram(cand_vals, bins=edges)
        ax2.bar(centers, cand_counts, width=widths * 0.50, color=PLOS["coral"], edgecolor="#B84F42", linewidth=0.55, align="center", alpha=0.88, label="Final-candidate bin count", zorder=3)
        ax2.set_ylabel("Final-candidate count")
        ax2.tick_params(axis="y", labelsize=cfg.tick_size, colors=PLOS["dark"])
        ax2.spines["top"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.spines["right"].set_color(PLOS["dark"])
        ax2.set_ylim(0, max(1.0, float(cand_counts.max()) * 1.28 if len(cand_counts) else 1.0))

        n_within_central = int(((cand_vals >= central_low) & (cand_vals <= central_high)).sum()) if len(cand_vals) else 0
        ax.text(
            0.98,
            0.95,
            f"Within central interval: {n_within_central}/{len(cand_vals)}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8.9,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.7),
        )
        ax.text(
            0.02,
            0.05,
            "Gray: training density\nCoral: final-candidate bin count",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=8.2,
            color=PLOS["dark"],
            bbox=dict(boxstyle="round,pad=0.22", facecolor="#FFFFFF", edgecolor="#DDDDDD", linewidth=0.55, alpha=0.9),
        )

        ax.set_title(title, pad=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Training-reference density")
        style_axis(ax, "y")
        add_panel_label(ax, panel, cfg, x=-0.09, y=1.03)

        panel_df = pd.DataFrame(
            {
                "descriptor": descriptor,
                "bin_left": edges[:-1],
                "bin_right": edges[1:],
                "bin_center": centers,
                "training_density": train_density,
                "final_candidate_count": cand_counts,
                "central_interval_low": central_low,
                "central_interval_high": central_high,
                "central_interval_definition": interval_label,
                "audit_range_low": audit_low,
                "audit_range_high": audit_high,
                "audit_range_definition": str(row.get("audit_range_definition", "")),
                "n_final_candidates": int(len(cand_vals)),
                "n_final_candidates_within_central_interval": n_within_central,
                "bin_inclusion_rule": "left-closed/right-open except final bin includes right edge",
            }
        )
        outputs.append(write_csv(panel_df, cfg.source_data_dir / f"Supplementary_Figure_S13_panel_{panel.lower()}_source_data.csv"))
        all_frames.append(dataframe_with_panel(panel_df, "Supplementary Figure S13", panel))
        summary_rows.append(
            {
                "descriptor": descriptor,
                "central_interval_low": central_low,
                "central_interval_high": central_high,
                "central_interval_definition": interval_label,
                "audit_range_low": audit_low,
                "audit_range_high": audit_high,
                "n_final_candidates": int(len(cand_vals)),
                "n_final_candidates_within_central_interval": n_within_central,
                "candidate_fraction_within_central_interval": n_within_central / len(cand_vals) if len(cand_vals) else np.nan,
                "training_median": float(row["median"]),
                "training_q05": float(row["q05"]),
                "training_q95": float(row["q95"]),
                "candidate_min": float(np.nanmin(cand_vals)) if len(cand_vals) else np.nan,
                "candidate_median": float(np.nanmedian(cand_vals)) if len(cand_vals) else np.nan,
                "candidate_max": float(np.nanmax(cand_vals)) if len(cand_vals) else np.nan,
            }
        )

    handles = [
        Patch(facecolor=PLOS["light"], edgecolor=PLOS["white"], label="Training reference"),
        Patch(facecolor=PLOS["mint"], edgecolor="none", alpha=0.35, label=f"Central reference interval ({int(cfg.reference_quantile_low*100)}-{int(cfg.reference_quantile_high*100)}%)"),
        Patch(facecolor=PLOS["coral"], edgecolor="#B84F42", label="Final-candidate bin count"),
    ]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=3, frameon=True)
    fig.subplots_adjust(left=0.07, right=0.955, top=0.94, bottom=0.14, wspace=0.30, hspace=0.34)
    outputs.append(write_csv(pd.concat(all_frames, ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S13_source_data_all_panels.csv"))
    outputs.append(write_csv(pd.DataFrame(summary_rows), cfg.source_data_dir / "step7_descriptor_support_summary.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S13_descriptor_support", cfg)
    return figs, outputs

def build_s14_similarity_tail(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    fig = plt.figure(figsize=(14.0, 5.9))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[0.92, 1.15])

    ax_a = fig.add_subplot(gs[0, 0])
    vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    bin_edges = np.array(SIMILARITY_BINS, dtype=float)
    bin_labels = list(SIMILARITY_BIN_LABELS)
    counts, _ = np.histogram(vals, bins=bin_edges)
    x = np.arange(len(bin_labels))
    bar_colors = [PLOS["mint"] if label != ">=0.30" else PLOS["light"] for label in bin_labels]
    edge_colors = ["#75A883" if label != ">=0.30" else "#A7A7A7" for label in bin_labels]
    bars = ax_a.bar(x, counts, color=bar_colors, edgecolor=edge_colors, linewidth=0.8, zorder=3)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels(bin_labels, rotation=30, ha="right")
    ax_a.set_ylabel("Final-candidate count")
    ax_a.set_xlabel("Nearest-neighbor similarity bin")
    ax_a.set_title("Final-candidate similarity-bin summary", pad=10)
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "A", cfg, x=-0.09, y=1.03)
    ymax = max(1, int(counts.max()) if len(counts) else 1)
    ax_a.set_ylim(0, ymax * 1.28)
    for bar, n in zip(bars, counts):
        alpha = 0.55 if int(n) == 0 else 1.0
        bar.set_alpha(alpha)
        ax_a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + ymax * 0.035, f"{int(n)}", ha="center", va="bottom", fontsize=9.0, color=PLOS["dark"])
    ax_a.text(
        0.98,
        0.95,
        f"n = {len(vals)}; low-risk criterion < {cfg.similarity_low_risk_threshold:.2f}",
        transform=ax_a.transAxes,
        ha="right",
        va="top",
        fontsize=9.0,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.7),
    )

    ax_b = fig.add_subplot(gs[0, 1])
    model_tags = [m for m in MODEL_ORDER if m in set(audited_df["model_tag"])]
    thresholds = list(cfg.tail_thresholds)
    x_base = np.arange(len(thresholds), dtype=float) * 1.18
    bar_w = 0.13
    gap = 0.025
    tail_rows: List[Dict[str, object]] = []
    denominator_notes: List[str] = []
    for i, model in enumerate(model_tags):
        vals_model = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        denom = int(len(vals_model))
        denominator_notes.append(f"{MODEL_LABELS[model]} n={denom:,}")
        fracs = [float(np.mean(vals_model >= thr)) if denom else np.nan for thr in thresholds]
        offset = (i - (len(model_tags) - 1) / 2.0) * (bar_w + gap)
        bars_b = ax_b.bar(x_base + offset, fracs, width=bar_w, color=MODEL_COLORS[model], edgecolor=MODEL_EDGES[model], linewidth=0.75, alpha=0.96, zorder=3)
        for thr, frac, bar in zip(thresholds, fracs, bars_b):
            n_at = int(np.sum(vals_model >= thr)) if denom else 0
            tail_rows.append(
                {
                    "model_tag": model,
                    "model_label": MODEL_LABELS[model],
                    "threshold": thr,
                    "threshold_label": f"NN >= {thr:.2f}",
                    "comparison": ">= threshold",
                    "denominator_definition": "per-model evaluated outputs with finite nn_similarity_train",
                    "n_denominator": denom,
                    "n_at_or_above_threshold": n_at,
                    "fraction_at_or_above_threshold": frac,
                }
            )
    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels([f"NN ≥{thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction of finite-NN evaluated outputs")
    ax_b.set_ylim(0.0, 1.03)
    ax_b.set_title("Generator-context tail-risk reference", pad=10)
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "B", cfg, x=-0.09, y=1.03)
    note = "Denominator: per-model outputs with finite NN similarity"
    if len(set(denominator_notes)) <= 5:
        note += "\n" + "; ".join(denominator_notes)
    ax_b.text(0.98, 0.95, note, transform=ax_b.transAxes, ha="right", va="top", fontsize=8.5, bbox=dict(boxstyle="round,pad=0.25", facecolor="#FAFAFA", edgecolor="#CFCFCF", linewidth=0.7))

    fig.subplots_adjust(left=0.075, right=0.985, top=0.91, bottom=0.23, wspace=0.22)
    add_bottom_legend(fig, include_train=False, y=0.035)

    panel_a = pd.DataFrame(
        {
            "similarity_bin": bin_labels,
            "bin_left": bin_edges[:-1],
            "bin_right": bin_edges[1:],
            "inclusive_rule": "left-closed/right-open except final bin includes right edge",
            "final_candidate_count": counts,
            "n_final_candidates_with_finite_nn_similarity": int(len(vals)),
            "low_risk_threshold": cfg.similarity_low_risk_threshold,
        }
    )
    panel_b = pd.DataFrame(tail_rows)
    outputs: List[str] = []
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "Supplementary_Figure_S14_panel_a_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S14_panel_b_source_data.csv"))
    outputs.append(write_csv(pd.concat([dataframe_with_panel(panel_a, "Supplementary Figure S14", "A"), dataframe_with_panel(panel_b, "Supplementary Figure S14", "B")], ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S14_source_data_all_panels.csv"))
    outputs.append(write_csv(panel_a, cfg.source_data_dir / "step7_similarity_bin_counts.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S14_similarity_tail_risk", cfg)
    return figs, outputs

def write_core_source_tables(audited_df: pd.DataFrame, final_df: pd.DataFrame, reference_ranges: pd.DataFrame, funnel_df: pd.DataFrame, cfg: Step7Config) -> List[str]:
    outputs = []
    outputs.append(write_csv(audited_df, cfg.source_data_dir / "step7_all_generated_audited_with_computed_flags.csv"))
    outputs.append(write_csv(final_df, cfg.source_data_dir / "step7_final_candidates_audit_table.csv"))
    outputs.append(write_csv(reference_ranges, cfg.source_data_dir / "step7_training_reference_descriptor_ranges.csv"))
    outputs.append(write_csv(funnel_df, cfg.source_data_dir / "step7_candidate_funnel_counts.csv"))

    tail_rows = []
    for model in MODEL_ORDER:
        vals = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        denom = int(len(vals))
        for threshold in cfg.tail_thresholds:
            n_at = int(np.sum(vals >= threshold)) if denom else 0
            tail_rows.append(
                {
                    "scope": "all_evaluated_outputs",
                    "model_tag": model,
                    "model_label": MODEL_LABELS[model],
                    "threshold": threshold,
                    "threshold_label": f"NN >= {threshold:.2f}",
                    "comparison": ">= threshold",
                    "denominator_definition": "per-model evaluated outputs with finite nn_similarity_train",
                    "n_denominator": denom,
                    "n_at_or_above_threshold": n_at,
                    "fraction_at_or_above_threshold": float(n_at / denom) if denom else np.nan,
                }
            )
    final_vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    denom_final = int(len(final_vals))
    for threshold in cfg.tail_thresholds:
        n_at = int(np.sum(final_vals >= threshold)) if denom_final else 0
        tail_rows.append(
            {
                "scope": "final_candidates",
                "model_tag": "full_cvae",
                "model_label": "OncoPep final candidates",
                "threshold": threshold,
                "threshold_label": f"NN >= {threshold:.2f}",
                "comparison": ">= threshold",
                "denominator_definition": "final candidates with finite nn_similarity_train",
                "n_denominator": denom_final,
                "n_at_or_above_threshold": n_at,
                "fraction_at_or_above_threshold": float(n_at / denom_final) if denom_final else np.nan,
            }
        )
    outputs.append(write_csv(pd.DataFrame(tail_rows), cfg.source_data_dir / "step7_similarity_tail_risk_summary.csv"))
    return outputs

def build_readiness_report(audited_df: pd.DataFrame, final_df: pd.DataFrame, inputs: Dict[str, object], cfg: Step7Config) -> Tuple[str, Dict[str, object]]:
    issues: List[str] = []
    warnings_list: List[str] = list(inputs.get("warnings_from_preparation", []))

    funnel_path = cfg.source_data_dir / "step7_candidate_funnel_counts.csv"
    if funnel_path.exists():
        try:
            funnel_df = pd.read_csv(funnel_path)
            if "n" in funnel_df.columns and "stage" in funnel_df.columns:
                pre_final = funnel_df.loc[~funnel_df["stage"].astype(str).eq("Final candidates")].copy()
                if not pre_final["n"].is_monotonic_decreasing:
                    issues.append("S12A pre-final audit-stage counts are not monotonic decreasing; this would indicate non-sequential audit logic.")
            if "retained_from_previous_pct" in funnel_df.columns and "stage" in funnel_df.columns:
                pre_final_pct = pd.to_numeric(funnel_df.loc[~funnel_df["stage"].astype(str).eq("Final candidates"), "retained_from_previous_pct"], errors="coerce")
                if (pre_final_pct > 100.0001).any():
                    issues.append("S12A contains pre-final retained-from-previous percentages above 100%, which is invalid for a true sequential audit funnel.")
        except Exception as exc:
            warnings_list.append(f"Could not verify S12 funnel source table: {type(exc).__name__}: {exc}")

    if final_df.empty:
        issues.append("No final candidates were identified.")
    if "explicit" not in str(inputs.get("final_candidate_rule", "")):
        warnings_list.append("No explicit final-candidate flag/file was detected; final candidates were inferred by fallback logic. This is acceptable for testing but should be resolved before final manuscript writing.")
    if audited_df["audit_exact_novel_vs_train"].isna().all():
        issues.append("Exact novelty could not be computed or read.")
    if audited_df["nn_similarity_train"].isna().all():
        issues.append("Nearest-neighbor similarity is unavailable for all rows.")
    elif audited_df["nn_similarity_train"].isna().any():
        warnings_list.append("Some nearest-neighbor similarity values are missing.")
    if not audited_df["recognized_model"].all():
        warnings_list.append("Some rows have unrecognized model tags.")
    if not final_df.empty and final_df["duplicate_sequence_within_input"].any():
        issues.append("At least one final candidate is duplicated within the generated input table.")
    if not final_df.empty and not final_df["valid_aa_sequence"].all():
        issues.append("At least one final candidate contains invalid/non-standard amino-acid characters.")
    if not final_df.empty and "audit_pass_computed" in final_df.columns and not final_df["audit_pass_computed"].all():
        warnings_list.append("At least one final candidate does not pass all computed audit criteria.")

    # Check S12/S13 terminology consistency: central interval is intentionally stricter than S12 audit support.
    if "computed_within_reference_central_interval" in final_df.columns and "audit_within_reference_all_descriptors" in final_df.columns:
        n_central = int(final_df["computed_within_reference_central_interval"].fillna(False).sum())
        n_audit = int(final_df["audit_within_reference_all_descriptors"].fillna(False).sum())
        if n_central != n_audit:
            warnings_list.append(
                f"S13 central-interval support ({n_central}/{len(final_df)}) differs from S12 reference-supported audit ({n_audit}/{len(final_df)}). This is expected if S13 uses a stricter central interval; captions must state this explicitly."
            )

    expected_figure_stems = ["Supplementary_Figure_S12_candidate_audit", "Supplementary_Figure_S13_descriptor_support", "Supplementary_Figure_S14_similarity_tail_risk"]
    expected_source_files = [
        "Supplementary_Figure_S12_source_data_all_panels.csv", "Supplementary_Figure_S12_panel_a_source_data.csv", "Supplementary_Figure_S12_panel_b_source_data.csv",
        "Supplementary_Figure_S13_source_data_all_panels.csv", "Supplementary_Figure_S13_panel_a_source_data.csv", "Supplementary_Figure_S13_panel_b_source_data.csv", "Supplementary_Figure_S13_panel_c_source_data.csv", "Supplementary_Figure_S13_panel_d_source_data.csv",
        "Supplementary_Figure_S14_source_data_all_panels.csv", "Supplementary_Figure_S14_panel_a_source_data.csv", "Supplementary_Figure_S14_panel_b_source_data.csv",
        "step7_all_generated_audited_with_computed_flags.csv", "step7_final_candidates_audit_table.csv", "step7_training_reference_descriptor_ranges.csv", "step7_candidate_funnel_counts.csv", "step7_descriptor_support_summary.csv", "step7_similarity_bin_counts.csv", "step7_similarity_tail_risk_summary.csv",
    ]
    figure_penalties = sum(5 for stem in expected_figure_stems for ext in [".pdf", ".png", ".tiff"] if not (cfg.supplementary_dir / f"{stem}{ext}").exists())
    if final_df.empty:
        figure_penalties += 20
    figure_readiness = max(0, 100 - figure_penalties)

    source_penalties = sum(4 for fname in expected_source_files if not (cfg.source_data_dir / fname).exists())
    if audited_df.empty or final_df.empty:
        source_penalties += 20
    source_data_readiness = max(0, 100 - source_penalties)

    reproducibility_penalties = sum(6 for key in ["generated_path", "train_path", "generated_sha256", "train_sha256", "input_inventory"] if not inputs.get(key))
    reproducibility_readiness = max(0, 100 - reproducibility_penalties)
    graph_rule_readiness = 100
    scientific_penalties = 14 * len(issues) + 3 * len(warnings_list)
    scientific_readiness = max(0, 100 - scientific_penalties)
    overall = int(round(np.mean([figure_readiness, source_data_readiness, reproducibility_readiness, graph_rule_readiness, scientific_readiness])))

    result = {
        "figure_package_readiness_score": int(figure_readiness),
        "source_data_readiness_score": int(source_data_readiness),
        "reproducibility_readiness_score": int(reproducibility_readiness),
        "graph_type_rule_readiness_score": int(graph_rule_readiness),
        "scientific_audit_readiness_score": int(scientific_readiness),
        "overall_step7_readiness_score": int(overall),
        "ready_threshold": 98,
        "is_ready": overall >= 98,
        "n_all_rows": int(len(audited_df)),
        "n_final_candidates": int(len(final_df)),
        "issues": issues,
        "warnings": warnings_list,
        "final_candidate_rule": inputs.get("final_candidate_rule"),
        "s12_s13_reference_support_note": "S12 uses reference-supported audit criteria; S13 uses a stricter central training interval for visual support summaries.",
    }

    lines = [
        "# OncoPep Step 7 readiness report", "", f"- Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        f"- Figure package readiness score: **{figure_readiness}/100**", f"- Source-data readiness score: **{source_data_readiness}/100**", f"- Reproducibility readiness score: **{reproducibility_readiness}/100**", f"- Graph-type rule readiness score: **{graph_rule_readiness}/100**", f"- Scientific audit readiness score: **{scientific_readiness}/100**", f"- Overall Step 7 readiness score: **{overall}/100**", "- Ready threshold: **98/100**", f"- Final candidate rule: `{inputs.get('final_candidate_rule')}`", f"- Number of evaluated rows: {len(audited_df):,}", f"- Number of final candidates: {len(final_df):,}", "", "## Strict Step 7 graph-type rule", "", "- No candidate-level dot plots used.", "- No candidate-level lollipop plots used.", "- No line plots used.", "- No connected-point plots used.", "- No ECDF line plots used.", "- No trend lines or decorative line summaries used.", "", "## S12/S13 reference-support definitions", "", f"- S12 reference-supported audit: official plausibility flag when available; otherwise audit/full training descriptor range [{cfg.audit_reference_quantile_low}, {cfg.audit_reference_quantile_high}].", f"- S13 central interval: stricter visual descriptor-support interval [{cfg.reference_quantile_low}, {cfg.reference_quantile_high}].", "- Therefore, S13 central-interval counts may be lower than S12 reference-supported audit pass counts without contradiction.", "", "## Inputs actually used", "",
    ]
    for key, value in inputs.items():
        if key != "warnings_from_preparation":
            lines.append(f"- {key}: `{value}`")
    lines += ["", "## Issues blocking readiness", ""]
    lines += [f"- {issue}" for issue in issues] if issues else ["- None detected."]
    lines += ["", "## Warnings", ""]
    lines += [f"- {warning}" for warning in warnings_list] if warnings_list else ["- None detected."]
    lines += ["", "## What to fix if below 98/100", "", "- Provide an explicit final-candidate file or final-candidate flag if fallback selection was used.", "- Confirm final-candidate count and exact novelty denominator.", "- Confirm S12-S14 numbering against the supplementary manuscript.", "- Confirm captions distinguish S12 reference-supported audit from S13 central-interval support.", "", "## Claim boundary", "", "Step 7 supports computational auditing and downstream prioritization only. It does not establish anticancer activity, receptor binding, selectivity, toxicity, mechanism, therapeutic efficacy, or clinical relevance."]
    path = cfg.reports_dir / "step7_readiness_report.md"
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path), result

def write_panel_mapping(cfg: Step7Config) -> str:
    mapping = pd.DataFrame([
        {"figure": "S12 Fig", "panel": "A", "title": "Sequential candidate-audit funnel", "source_data": "source_data/Supplementary_Figure_S12_panel_a_source_data.csv", "notes": "Uses cumulative sequential counts; no retained percentage can exceed 100%."},
        {"figure": "S12 Fig", "panel": "B", "title": "Final candidate audit matrix", "source_data": "source_data/Supplementary_Figure_S12_panel_b_source_data.csv", "notes": "Light binary matrix; no candidate-level dot/lollipop/line plot."},
        {"figure": "S13 Fig", "panel": "A", "title": "Length descriptor support", "source_data": "source_data/Supplementary_Figure_S13_panel_a_source_data.csv", "notes": "Training density plus binned final-candidate counts; central interval is stricter visual support."},
        {"figure": "S13 Fig", "panel": "B", "title": "Net-charge descriptor support", "source_data": "source_data/Supplementary_Figure_S13_panel_b_source_data.csv", "notes": "Dual axis: training density and final-candidate bin count."},
        {"figure": "S13 Fig", "panel": "C", "title": "Hydrophobicity descriptor support", "source_data": "source_data/Supplementary_Figure_S13_panel_c_source_data.csv", "notes": "Exact bin edges exported."},
        {"figure": "S13 Fig", "panel": "D", "title": "Entropy descriptor support", "source_data": "source_data/Supplementary_Figure_S13_panel_d_source_data.csv", "notes": "Exact bin edges exported."},
        {"figure": "S14 Fig", "panel": "A", "title": "Final-candidate similarity-bin summary", "source_data": "source_data/Supplementary_Figure_S14_panel_a_source_data.csv", "notes": "Binned count bars for n=12; no histogram/dot/lollipop/line display."},
        {"figure": "S14 Fig", "panel": "B", "title": "Generator-context high-similarity tail-risk reference", "source_data": "source_data/Supplementary_Figure_S14_panel_b_source_data.csv", "notes": "Denominator is per-model evaluated outputs with finite nearest-neighbor similarity."},
    ])
    return write_csv(mapping, cfg.reports_dir / "step7_panel_source_data_mapping.csv")

def write_readme(cfg: Step7Config) -> str:
    text = f"""OncoPep Step 7 candidate-audit outputs
====================================

Purpose
-------
This folder contains Step 7 supplementary candidate-audit outputs for the
OncoPep PLOS Computational Biology manuscript.

Version
-------
v9 sequential-funnel no-dot/no-lollipop/no-line redesign.

Figures
-------
- S12 Fig: Sequential candidate-audit funnel and final candidate binary audit matrix.
- S13 Fig: Reference-space descriptor support using training-reference histograms and binned final-candidate counts.
- S14 Fig: Similarity-tail risk using final-candidate similarity-bin counts and contextual generator-level threshold bars.

Important S12/S13 definition distinction
---------------------------------------
S12 uses the reference-supported audit criterion. When an official
`plausibility_is_within_reference` column exists, that column is used. Otherwise,
the S12 audit uses the configured full/audit descriptor range.

S13 uses a stricter visual central training interval, configured as
[{cfg.reference_quantile_low}, {cfg.reference_quantile_high}], to summarize descriptor
support. Therefore, S13 central-interval counts can be lower than S12 audit-pass
counts without contradiction.

Strict Step 7 graph-type rule
-----------------------------
This code does not generate candidate-level dot plots, candidate-level lollipop
plots, line plots, connected-point plots, ECDF line plots, trend lines, or
decorative line summaries.

Why these are supplementary
---------------------------
Step 5 already benchmarks generator-level validity, novelty, nearest-neighbor
similarity, and condition fidelity. Step 6 covers latent-space and sampling
diagnostics. Step 7 is therefore reviewer-supporting supplementary evidence
focused on final candidate audit, descriptor support, and similarity-tail risk.

Claim boundary
--------------
These analyses are computational and hypothesis-generating. They do not establish
anticancer activity, receptor binding, selectivity, toxicity, stability,
mechanism, therapeutic efficacy, or clinical relevance.

Output root
-----------
{cfg.root}

Notebook example
----------------
from OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root={str(cfg.root)!r},
    generated_file='/path/to/step7_generated_final_audited.csv',
    train_file='/path/to/step7_train_reference_annotated.csv',
    final_file=None,
    show_figures=True,
)
outputs

Terminal example
----------------
python OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py \\
  --step7-root {cfg.root} \\
  --generated-file /path/to/step7_generated_final_audited.csv \\
  --train-file /path/to/step7_train_reference_annotated.csv
"""
    path = cfg.reports_dir / "README_step7_candidate_audit_outputs.txt"
    path.write_text(text, encoding="utf-8")
    return str(path)

def write_requirements(cfg: Step7Config) -> str:
    path = cfg.code_dir / "requirements_step7_candidate_audit_minimal.txt"
    path.write_text("\n".join(["numpy", "pandas", "matplotlib", "", "# Record exact package versions at repository freeze:", "# python -m pip freeze > requirements_full_freeze.txt"]), encoding="utf-8")
    return str(path)


def write_code_snapshot(cfg: Step7Config) -> str:
    dest = cfg.code_dir / "OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py"
    try:
        current = Path(__file__).resolve()
        if current.exists():
            shutil.copy2(current, dest)
            return str(dest)
    except Exception:
        pass
    try:
        source = inspect.getsource(sys.modules[__name__])
        dest.write_text(source, encoding="utf-8")
    except Exception as exc:
        dest.write_text(f"# Code snapshot unavailable: {type(exc).__name__}: {exc}\n", encoding="utf-8")
    return str(dest)

def write_manifest(cfg: Step7Config, inputs: Dict[str, object], figures: Sequence[str], source_data: Sequence[str], reports: Sequence[str], readiness: Dict[str, object], readme: str, requirements: str, code_snapshot: str) -> str:
    manifest = {
        "workflow": "OncoPep Step 7 candidate-audit redesign",
        "version": "v9 sequential-funnel no-dot no-lollipop no-line candidate-audit PLOS-ready",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version,
        "platform": platform.platform(),
        "config": asdict(cfg),
        "inputs": inputs,
        "figures": list(figures),
        "source_data": list(source_data),
        "reports": list(reports),
        "readiness": readiness,
        "readme": readme,
        "requirements": requirements,
        "code_snapshot": code_snapshot,
        "figure_numbering": {
            "main_figure": "No new main Figure 7 generated.",
            "S12 Fig": "Sequential candidate audit funnel and final candidate audit matrix.",
            "S13 Fig": "Reference-space descriptor support with central training interval.",
            "S14 Fig": "Similarity-tail risk and nearest-neighbor audit.",
        },
        "strict_graph_rule": "No candidate-level dot/lollipop/line/connected/ECDF/trend/decorative line plots.",
        "s12_s13_definition_note": "S12 reference-supported audit and S13 central-interval descriptor support are intentionally distinct and both are exported in source data.",
        "claim_boundary": "Computational audit only; no anticancer activity, receptor binding, or therapeutic efficacy claim.",
    }
    path = cfg.reports_dir / "step7_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)

def print_resolved_inputs(inputs: Dict[str, object]) -> None:
    print("\nResolved OncoPep Step 7 input files")
    print("-" * 44)
    for key in ["generated_path", "train_path", "final_path", "summary_path", "funnel_path", "input_inventory"]:
        print(f"{key}: {inputs.get(key)}")
    print(f"final_candidate_rule: {inputs.get('final_candidate_rule')}")


def _run_with_config(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    set_plot_style(cfg)
    loaded = load_inputs(cfg)
    audited_df: pd.DataFrame = loaded["audited_df"]  # type: ignore[assignment]
    train_df: pd.DataFrame = loaded["train_df"]  # type: ignore[assignment]
    final_df: pd.DataFrame = loaded["final_df"]  # type: ignore[assignment]
    reference_ranges: pd.DataFrame = loaded["reference_ranges"]  # type: ignore[assignment]
    funnel_df: pd.DataFrame = loaded["funnel_df"]  # type: ignore[assignment]
    inputs: Dict[str, object] = loaded["inputs"]  # type: ignore[assignment]
    print_resolved_inputs(inputs)
    figures: List[str] = []
    source_data: List[str] = []
    source_data.extend(write_core_source_tables(audited_df, final_df, reference_ranges, funnel_df, cfg))
    figs, src = build_s12_candidate_audit(final_df, funnel_df, cfg); figures.extend(figs); source_data.extend(src)
    figs, src = build_s13_descriptor_support(final_df, train_df, reference_ranges, cfg); figures.extend(figs); source_data.extend(src)
    figs, src = build_s14_similarity_tail(audited_df, final_df, cfg); figures.extend(figs); source_data.extend(src)
    reports: List[str] = []
    readiness_report_path, readiness = build_readiness_report(audited_df, final_df, inputs, cfg)
    reports.append(readiness_report_path)
    reports.append(write_panel_mapping(cfg))
    readme_path = write_readme(cfg)
    requirements_path = write_requirements(cfg)
    code_path = write_code_snapshot(cfg)
    manifest_path = write_manifest(cfg, inputs, figures, source_data, reports, readiness, readme_path, requirements_path, code_path)
    reports.append(manifest_path)
    return {"step7_root": str(cfg.root), "figures": figures, "source_data": source_data, "reports": reports, "readme": readme_path, "requirements": requirements_path, "code_snapshot": code_path, "inputs": inputs, "readiness": readiness}


def run_step7_candidate_audit(
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    input_roots: Optional[Sequence[str]] = None,
    generated_file: Optional[str] = None,
    train_file: Optional[str] = None,
    final_file: Optional[str] = None,
    summary_file: Optional[str] = None,
    funnel_file: Optional[str] = None,
    audit_reference_quantile_low: float = 0.00,
    audit_reference_quantile_high: float = 1.00,
    reference_quantile_low: float = 0.05,
    reference_quantile_high: float = 0.95,
    similarity_low_risk_threshold: float = 0.30,
    top_n_final: int = 12,
    show_figures: bool = False,
    compute_missing_descriptors: bool = True,
    compute_nn_if_missing: bool = True,
    max_nn_pairs: int = 3_000_000,
) -> Dict[str, object]:
    cfg = Step7Config(
        step7_root=step7_root,
        generated_file=generated_file,
        train_file=train_file,
        final_file=final_file,
        summary_file=summary_file,
        funnel_file=funnel_file,
        audit_reference_quantile_low=audit_reference_quantile_low,
        audit_reference_quantile_high=audit_reference_quantile_high,
        reference_quantile_low=reference_quantile_low,
        reference_quantile_high=reference_quantile_high,
        similarity_low_risk_threshold=similarity_low_risk_threshold,
        top_n_final=top_n_final,
        show_figures=show_figures,
        compute_missing_descriptors=compute_missing_descriptors,
        compute_nn_if_missing=compute_nn_if_missing,
        max_nn_pairs=max_nn_pairs,
    )
    if input_roots is not None:
        cfg.input_roots = tuple(input_roots) + tuple(cfg.input_roots)
    return _run_with_config(cfg)

def _is_jupyter_unknown_arg(arg: str) -> bool:
    text = str(arg)
    return text == "-f" or text == "--f" or text.startswith("-f=") or text.startswith("--f=") or ("kernel" in text and text.endswith(".json")) or ("jupyter" in text.lower() and "runtime" in text.lower())


def is_running_under_ipykernel() -> bool:
    return "ipykernel" in sys.modules or any("ipykernel" in str(arg) for arg in sys.argv)


def has_meaningful_cli_args(argv: Optional[Sequence[str]] = None) -> bool:
    if argv is None:
        argv = sys.argv[1:]
    return any(str(arg).startswith("--") and not _is_jupyter_unknown_arg(str(arg)) for arg in argv)


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 7 v9 sequential-funnel no-dot/no-line candidate-audit supplementary figures.")
    parser.add_argument("--step7-root", default="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07")
    parser.add_argument("--input-root", action="append", default=None)
    parser.add_argument("--generated-file", default=None)
    parser.add_argument("--train-file", default=None)
    parser.add_argument("--final-file", default=None)
    parser.add_argument("--summary-file", default=None)
    parser.add_argument("--funnel-file", default=None)
    parser.add_argument("--audit-reference-quantile-low", type=float, default=0.00)
    parser.add_argument("--audit-reference-quantile-high", type=float, default=1.00)
    parser.add_argument("--reference-quantile-low", type=float, default=0.05)
    parser.add_argument("--reference-quantile-high", type=float, default=0.95)
    parser.add_argument("--similarity-low-risk-threshold", type=float, default=0.30)
    parser.add_argument("--top-n-final", type=int, default=12)
    parser.add_argument("--show-figures", action="store_true")
    parser.add_argument("--no-compute-missing-descriptors", action="store_true")
    parser.add_argument("--no-compute-nn-if-missing", action="store_true")
    parser.add_argument("--max-nn-pairs", type=int, default=3_000_000)
    parser.add_argument("--dpi-png", type=int, default=None)
    parser.add_argument("--dpi-tiff", type=int, default=None)
    args, unknown = parser.parse_known_args(argv)
    non_jupyter_unknown = [arg for arg in unknown if not _is_jupyter_unknown_arg(str(arg))]
    if non_jupyter_unknown:
        warnings.warn("Ignoring unrecognized arguments: " + " ".join(map(str, non_jupyter_unknown)), RuntimeWarning, stacklevel=2)
    return args

def print_notebook_instructions() -> None:
    print(
        """
OncoPep Step 7 v9 code loaded in a Jupyter/IPython kernel.

This version fixes the S12A denominator issue with a true sequential funnel,
distinguishes S12 reference-supported audit from the S13 central interval, and
enforces the no-dot/no-lollipop/no-line Step 7 figure rule.

Run explicitly with:

from OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready import run_step7_candidate_audit

outputs = run_step7_candidate_audit(
    step7_root="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    generated_file="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file=None,
    show_figures=True,
    top_n_final=12,
)
outputs
"""
    )

def main(argv: Optional[Sequence[str]] = None) -> None:
    args = parse_args(argv)
    cfg = Step7Config(
        step7_root=args.step7_root,
        generated_file=args.generated_file,
        train_file=args.train_file,
        final_file=args.final_file,
        summary_file=args.summary_file,
        funnel_file=args.funnel_file,
        audit_reference_quantile_low=args.audit_reference_quantile_low,
        audit_reference_quantile_high=args.audit_reference_quantile_high,
        reference_quantile_low=args.reference_quantile_low,
        reference_quantile_high=args.reference_quantile_high,
        similarity_low_risk_threshold=args.similarity_low_risk_threshold,
        top_n_final=args.top_n_final,
        show_figures=args.show_figures,
        compute_missing_descriptors=not args.no_compute_missing_descriptors,
        compute_nn_if_missing=not args.no_compute_nn_if_missing,
        max_nn_pairs=args.max_nn_pairs,
    )
    if args.input_root is not None:
        cfg.input_roots = tuple(args.input_root) + tuple(cfg.input_roots)
    if args.dpi_png is not None: cfg.dpi_png = args.dpi_png
    if args.dpi_tiff is not None: cfg.dpi_tiff = args.dpi_tiff
    outputs = _run_with_config(cfg)
    print("\nOncoPep Step 7 v9 sequential-funnel no-dot/no-line candidate-audit redesign complete.")
    print(json.dumps(outputs, indent=2))


if __name__ == "__main__":
    if is_running_under_ipykernel() and not has_meaningful_cli_args():
        print_notebook_instructions()
    else:
        main()

In [ ]:
# =============================================================================
# OncoPep Step 7 — Single-cell execution for PLOS Computational Biology v9
# Purpose:
#   Run the complete Step 7 supplementary figure package:
#   S12, S13, S14 + source data + reports + manifest.
#
# Rule:
#   No candidate-level dot plots.
#   No lollipop plots.
#   No line plots.
#   No ECDF/trend/connected-point plots.
# =============================================================================

from pathlib import Path
import sys
import json
import importlib.util

# -------------------------------------------------------------------------
# 1. Fixed project paths
# -------------------------------------------------------------------------
STEP7_ROOT = Path("/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07").resolve()

GENERATED_FILE = Path(
    "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/"
    "SuppFig_step7_property_generated_final_v7_source_data.csv"
).resolve()

TRAIN_FILE = Path(
    "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/"
    "step7_train_reference_annotated.csv"
).resolve()

# Optional final-candidate file.
# Keep None unless you have a real frozen final-12 table.
FINAL_FILE = None

# -------------------------------------------------------------------------
# 2. Check required input files
# -------------------------------------------------------------------------
missing = []
if not GENERATED_FILE.exists():
    missing.append(f"Generated file missing: {GENERATED_FILE}")
if not TRAIN_FILE.exists():
    missing.append(f"Training-reference file missing: {TRAIN_FILE}")

if missing:
    raise FileNotFoundError(
        "Step 7 cannot run because required input files are missing:\n\n"
        + "\n".join(missing)
    )

# -------------------------------------------------------------------------
# 3. Locate or load the v9 function
# -------------------------------------------------------------------------
function_name = "run_step7_candidate_audit"

# Case A: function already exists because you pasted/executed the full v9 code above.
if function_name in globals():
    run_step7_candidate_audit = globals()[function_name]

else:
    # Case B: import the v9 .py file from likely project locations.
    candidate_code_files = [
        STEP7_ROOT / "code" / "OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py",
        STEP7_ROOT / "OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py",
        Path.cwd() / "OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py",
        Path("/mnt/data/OncoPep_step7_PLOS_candidate_audit_v9_sequential_ready.py"),
    ]

    code_file = None
    for p in candidate_code_files:
        if p.exists():
            code_file = p.resolve()
            break

    if code_file is None:
        raise FileNotFoundError(
            "Cannot find the v9 Step 7 code file.\n\n"
            "Fix: place this file in one of these locations:\n"
            + "\n".join(str(p) for p in candidate_code_files)
            + "\n\nOr paste and execute the full v9 code cell first, then rerun this cell."
        )

    spec = importlib.util.spec_from_file_location(
        "oncopep_step7_v9_module",
        str(code_file),
    )
    module = importlib.util.module_from_spec(spec)
    sys.modules["oncopep_step7_v9_module"] = module
    spec.loader.exec_module(module)

    run_step7_candidate_audit = module.run_step7_candidate_audit

# -------------------------------------------------------------------------
# 4. Run the complete Step 7 workflow
# -------------------------------------------------------------------------
outputs = run_step7_candidate_audit(
    step7_root=str(STEP7_ROOT),
    generated_file=str(GENERATED_FILE),
    train_file=str(TRAIN_FILE),
    final_file=str(FINAL_FILE) if FINAL_FILE is not None else None,
    show_figures=True,
    top_n_final=12,
)

# -------------------------------------------------------------------------
# 5. Print a compact execution summary
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print("ONCOPEP STEP 7 v9 RUN COMPLETE")
print("=" * 90)

print("\nInput files used:")
print("Generated:", GENERATED_FILE)
print("Train reference:", TRAIN_FILE)
print("Final file:", FINAL_FILE)

print("\nOutput root:")
print(STEP7_ROOT)

print("\nGenerated supplementary figures:")
for f in outputs.get("figures", []):
    print(" -", f)

print("\nGenerated reports:")
for f in outputs.get("reports", []):
    print(" -", f)

print("\nReadiness summary:")
print(json.dumps(outputs.get("readiness", {}), indent=2))

# -------------------------------------------------------------------------
# 6. Verify required final files exist
# -------------------------------------------------------------------------
required_outputs = [
    STEP7_ROOT / "supplementary_figures" / "Supplementary_Figure_S12_candidate_audit.png",
    STEP7_ROOT / "supplementary_figures" / "Supplementary_Figure_S13_descriptor_support.png",
    STEP7_ROOT / "supplementary_figures" / "Supplementary_Figure_S14_similarity_tail_risk.png",
    STEP7_ROOT / "reports" / "step7_readiness_report.md",
    STEP7_ROOT / "reports" / "step7_manifest.json",
    STEP7_ROOT / "source_data" / "step7_final_candidates_audit_table.csv",
    STEP7_ROOT / "source_data" / "step7_candidate_funnel_counts.csv",
    STEP7_ROOT / "source_data" / "step7_similarity_tail_risk_summary.csv",
]

print("\nRequired-output check:")
for p in required_outputs:
    print(("OK   " if p.exists() else "MISS "), p)

# -------------------------------------------------------------------------
# 7. Display readiness report text
# -------------------------------------------------------------------------
readiness_report = STEP7_ROOT / "reports" / "step7_readiness_report.md"

if readiness_report.exists():
    print("\n" + "=" * 90)
    print("STEP 7 READINESS REPORT")
    print("=" * 90)
    print(readiness_report.read_text(encoding="utf-8")[:5000])
else:
    print("\nReadiness report was not generated.")

outputs

In [ ]:
#!/usr/bin/env python3
"""
OncoPep Step 7 — final supplementary figure/report package for PLOS Computational Biology.

Version
-------
v10 final-figures-only / S12 converted to table-report / no-dot-no-lollipop-no-line.

Scientific purpose
------------------
Step 7 is a final generated-candidate audit and supplementary screening-readiness
analysis. It is not another Step 5 generator benchmark and not a Step 6 latent-space
or sampling diagnostic.

Final output policy
-------------------
The former S12 candidate-audit funnel/matrix is converted to table/report outputs
because it is primarily tabular audit evidence. The final figure package contains:

    New S12 Fig: Reference-space descriptor support.
    New S13 Fig: Similarity-tail risk and nearest-neighbor audit.

Strict graph-design rule
------------------------
This script does NOT generate candidate-level dot plots, candidate-level lollipop plots,
line plots, connected-point plots, ECDF line plots, trend lines, or decorative curves.
Only histograms, binned count bars, grouped threshold bars, compact table/report outputs,
and shaded descriptor-support intervals are used.

Claim boundary
--------------
These analyses support computational auditing and candidate prioritization only. They do
not establish anticancer activity, receptor binding, selectivity, toxicity, stability,
mechanism, therapeutic efficacy, clinical utility, complete absence of memorization, or
experimental validation.

Default notebook usage
----------------------
Executing this full file in a VS Code/Jupyter cell will auto-run with the default paths
below if those files exist. Terminal use also works with explicit arguments.

Terminal example
----------------
python OncoPep_step7_PLOS_candidate_audit_v10_final_figures_only.py \
  --step7-root /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07 \
  --generated-file /home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv \
  --train-file /home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import math
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


# =============================================================================
# Constants and visual style
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

# Slightly refined, still identity-preserving shades.
STYLE = {
    "train_fill": "#DCDCDC",
    "train_edge": "#FFFFFF",
    "interval_fill": "#DDEFE4",
    "candidate_fill": PLOS["coral"],
    "candidate_edge": "#B94F42",
    "low_risk_fill": PLOS["mint"],
    "higher_risk_fill": "#E7E7E7",
    "higher_risk_edge": "#BDBDBD",
    "note_bg": "#FFFFFF",
    "note_edge": "#D2D2D2",
}

MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "OncoPep",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
    "train_ref": "Training reference",
}

MODEL_COLORS = {
    "full_cvae": PLOS["coral"],
    "conditional_gru_lm": PLOS["blue"],
    "random_condition": PLOS["mint"],
    "no_condition": PLOS["brown"],
    "small_latent": PLOS["charcoal"],
}

MODEL_EDGES = {
    "full_cvae": "#B94F42",
    "conditional_gru_lm": "#166F8A",
    "random_condition": "#78A886",
    "no_condition": "#8E5F39",
    "small_latent": "#4E4648",
}

MODEL_TAG_ALIASES = {
    "oncopep": "full_cvae",
    "oncoppep": "full_cvae",
    "full_cvae": "full_cvae",
    "cvae": "full_cvae",
    "conditional_vae": "full_cvae",
    "conditioned_vae": "full_cvae",
    "gru-cond": "conditional_gru_lm",
    "gru_cond": "conditional_gru_lm",
    "conditional_gru": "conditional_gru_lm",
    "conditional_gru_lm": "conditional_gru_lm",
    "rand-cond": "random_condition",
    "rand_cond": "random_condition",
    "random_condition": "random_condition",
    "random conditional": "random_condition",
    "gru-uncond": "no_condition",
    "gru_uncond": "no_condition",
    "unconditional_gru": "no_condition",
    "no_condition": "no_condition",
    "vae-uncond": "small_latent",
    "vae_uncond": "small_latent",
    "unconditional_vae": "small_latent",
    "small_latent": "small_latent",
}

AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DESCRIPTORS = ("length", "net_charge", "hydrophobicity", "entropy")
DEFAULT_TAIL_THRESHOLDS = (0.10, 0.15, 0.20, 0.30)

KD_SCALE = {
    "A": 1.8,
    "C": 2.5,
    "D": -3.5,
    "E": -3.5,
    "F": 2.8,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "N": -3.5,
    "P": -1.6,
    "Q": -3.5,
    "R": -4.5,
    "S": -0.8,
    "T": -0.7,
    "V": 4.2,
    "W": -0.9,
    "Y": -1.3,
}

COLUMN_ALIASES = {
    "model_tag": ["model_tag", "model", "generator", "model_name", "method"],
    "candidate_id": ["candidate_id", "peptide_id", "id", "seq_id", "sequence_id", "candidate"],
    "sequence": ["sequence", "peptide", "seq", "peptide_sequence"],
    "is_final_candidate": [
        "is_final_candidate",
        "final_selected",
        "selected_final",
        "selected",
        "is_selected",
        "final_candidate",
        "passed_final_audit",
        "passed_audit",
    ],
    "candidate_rank": ["candidate_rank", "rank", "final_rank", "selection_rank", "priority_rank"],
    "composite_score": ["composite_score", "score", "selection_score", "priority_score", "audit_score"],
    "exact_novel_vs_train": [
        "exact_novel_vs_train",
        "exact_novel",
        "exact_novelty",
        "novel_vs_train",
        "novelty_flag",
        "is_exact_novel",
        "novelty_exact",
    ],
    "exact_novel_vs_full": [
        "exact_novel_vs_full",
        "exact_novel_vs_full_corpus",
        "novel_vs_full",
        "exact_full_corpus_novelty",
    ],
    "nn_similarity_train": [
        "nn_similarity_train",
        "nearest_neighbor_similarity_train",
        "nearest_train_similarity",
        "nearest_train_jaccard",
        "smax_train",
        "nn_train",
        "similarity_to_train",
        "nearest_neighbor_similarity",
    ],
    "nn_train_sequence": ["nn_train_sequence", "nearest_train_sequence", "nearest_neighbor_train_sequence"],
    "length": ["length", "seq_length", "peptide_length", "model_ready_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "charge", "charge_pH7", "net_charge_ph7"],
    "hydrophobicity": [
        "hydrophobicity",
        "hydrophobicity_KD",
        "mean_hydrophobicity",
        "mean_kd_hydrophobicity",
        "mean_KD",
        "mean_kyte_doolittle",
        "kd_hydrophobicity",
        "mean_hydropathy",
    ],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy", "seq_entropy"],
    "plausibility_is_within_reference": [
        "plausibility_is_within_reference",
        "within_reference",
        "within_reference_range",
        "reference_supported",
    ],
}

TRAIN_ALIASES = {k: COLUMN_ALIASES[k] for k in ["sequence", "length", "net_charge", "hydrophobicity", "entropy"]}


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Step7Config:
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07"
    input_roots: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/source_data",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco",
        "/mnt/data",
    )

    generated_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv"
    train_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv"
    final_file: Optional[str] = None

    generated_candidates: Tuple[str, ...] = (
        "step7_generated_final_audited.csv",
        "step7_generated_final.csv",
        "step7_generated_audited.csv",
        "generated_final_audited.csv",
        "SuppFig_step7_property_generated_final_v7_source_data.csv",
    )
    train_candidates: Tuple[str, ...] = (
        "step7_train_reference_annotated.csv",
        "train_reference_annotated.csv",
        "SuppFig_step7_property_train_reference_v7_source_data.csv",
    )
    final_candidates: Tuple[str, ...] = (
        "step7_final_candidates.csv",
        "step7_final_candidates_audit_table.csv",
        "final_candidates.csv",
        "final_oncopep_candidates.csv",
    )

    central_quantile_low: float = 0.05
    central_quantile_high: float = 0.95
    full_reference_quantile_low: float = 0.00
    full_reference_quantile_high: float = 1.00
    similarity_low_risk_threshold: float = 0.30
    tail_thresholds: Tuple[float, ...] = DEFAULT_TAIL_THRESHOLDS
    top_n_final: int = 12

    compute_missing_descriptors: bool = True
    histidine_charge: float = 0.10
    compute_nn_if_missing: bool = False
    max_nn_pairs: int = 3_000_000

    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    tiff_compression: str = "tiff_lzw"
    show_figures: bool = False

    font_family: str = "DejaVu Sans"
    base_font_size: float = 10.8
    axis_label_size: float = 12.0
    title_size: float = 14.0
    tick_size: float = 10.2
    legend_size: float = 10.5
    panel_label_size: float = 18.0

    seed: int = 42
    inventory_max_files: int = 500

    @property
    def root(self) -> Path:
        return Path(self.step7_root).expanduser().resolve()

    @property
    def main_figure_dir(self) -> Path:
        return self.root / "main_figure"

    @property
    def supplementary_dir(self) -> Path:
        return self.root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.root / "code"


# =============================================================================
# General utilities
# =============================================================================

def ensure_output_tree(cfg: Step7Config) -> None:
    for path in [cfg.root, cfg.main_figure_dir, cfg.supplementary_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)


def set_plot_style(cfg: Step7Config) -> None:
    plt.rcParams.update(
        {
            "font.family": cfg.font_family,
            "font.size": cfg.base_font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "figure.titlesize": cfg.title_size,
            "axes.facecolor": PLOS["white"],
            "figure.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def normalize_path(path_raw: Optional[str]) -> Optional[Path]:
    if not path_raw:
        return None
    text = str(path_raw).strip()
    if text.lower() in {"none", "null", ""}:
        return None
    return Path(text).expanduser().resolve()


def inventory_csv_files(roots: Sequence[str], max_files: int = 500) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    seen: set[str] = set()
    for priority, root_raw in enumerate(roots):
        root = Path(root_raw).expanduser()
        if not root.exists():
            rows.append({"root_priority": priority, "root": str(root), "exists": False, "path": None, "filename": None, "size_bytes": None})
            continue
        if root.is_file():
            file_iter = iter([root] if root.suffix.lower() == ".csv" else [])
        else:
            file_iter = root.rglob("*.csv")
        n_this = 0
        for file_path in file_iter:
            if n_this >= max_files:
                break
            try:
                key = str(file_path.resolve())
            except OSError:
                continue
            if key in seen:
                continue
            seen.add(key)
            n_this += 1
            try:
                size_bytes = file_path.stat().st_size
            except OSError:
                size_bytes = None
            rows.append({"root_priority": priority, "root": str(root), "exists": True, "path": key, "filename": file_path.name, "size_bytes": size_bytes})
    return pd.DataFrame(rows)


def discover_file(candidate_names: Sequence[str], roots: Sequence[str], direct_file: Optional[str], required: bool, label: str) -> Optional[Path]:
    direct = normalize_path(direct_file)
    if direct is not None:
        if direct.exists() and direct.is_file():
            return direct
        raise FileNotFoundError(f"Direct {label} path was provided but does not exist: {direct}")
    candidate_set = set(candidate_names)
    searched: List[str] = []
    for root_raw in roots:
        root = Path(root_raw).expanduser()
        searched.append(str(root))
        if not root.exists():
            continue
        if root.is_file() and root.name in candidate_set:
            return root.resolve()
        for name in candidate_names:
            direct_child = root / name
            if direct_child.exists() and direct_child.is_file():
                return direct_child.resolve()
        if root.is_dir():
            for p in root.rglob("*.csv"):
                if p.name in candidate_set:
                    return p.resolve()
    if required:
        raise FileNotFoundError(
            f"Could not find required {label}.\nAccepted filenames: {list(candidate_names)}\nSearched roots:\n" + "\n".join(searched)
        )
    return None


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def dataframe_with_panel(df: pd.DataFrame, figure: str, panel: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "panel", panel)
    out.insert(0, "figure", figure)
    return out


def find_first_column(df: pd.DataFrame, aliases: Sequence[str], canonical_name: str, required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for alias in aliases:
        if alias in df.columns:
            return alias
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    if required:
        raise KeyError(f"Could not find column for {canonical_name}. Tried: {list(aliases)}. Available: {list(df.columns)}")
    return None


def maybe_rename_columns(df: pd.DataFrame, aliases: Dict[str, Sequence[str]], canonical_columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    rename_map: Dict[str, str] = {}
    for canonical in canonical_columns:
        if canonical in out.columns or canonical not in aliases:
            continue
        found = find_first_column(out, aliases[canonical], canonical, required=False)
        if found is not None and found != canonical:
            rename_map[found] = canonical
    return out.rename(columns=rename_map)


def normalize_model_tag_value(value: object) -> object:
    if pd.isna(value):
        return value
    text = str(value).strip()
    key = text.lower().replace("-", "_").replace(" ", "_")
    return MODEL_TAG_ALIASES.get(key, text)


def normalize_model_tags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model_tag" in out.columns:
        out["model_tag"] = out["model_tag"].map(normalize_model_tag_value)
    return out


def as_numeric_clean(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def parse_boolish(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    text = series.astype(str).str.strip().str.lower()
    truthy = {"true", "1", "yes", "y", "selected", "final", "pass", "passed", "keep", "kept"}
    falsy = {"false", "0", "no", "n", "not_selected", "fail", "failed", "nan", "none", "", "remove"}
    return text.map(lambda x: True if x in truthy else (False if x in falsy else False))


def clean_sequence(seq: object) -> str:
    if pd.isna(seq):
        return ""
    return str(seq).strip().upper()


def valid_sequence(seq: object) -> bool:
    s = clean_sequence(seq)
    return len(s) > 0 and all(ch in AA20 for ch in s)


def peptide_length(seq: object) -> float:
    s = clean_sequence(seq)
    return float(len(s)) if s else np.nan


def peptide_net_charge(seq: object, histidine_charge: float = 0.10) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    return float(s.count("K") + s.count("R") + histidine_charge * s.count("H") - s.count("D") - s.count("E"))


def peptide_hydrophobicity(seq: object) -> float:
    s = clean_sequence(seq)
    vals = [KD_SCALE[ch] for ch in s if ch in KD_SCALE]
    if not vals:
        return np.nan
    return float(np.mean(vals))


def peptide_entropy(seq: object) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    counts = np.array([s.count(aa) for aa in AA20], dtype=float)
    probs = counts[counts > 0] / len(s)
    return float(-(probs * np.log2(probs)).sum())


def normalized_levenshtein_similarity(a: str, b: str) -> float:
    a = clean_sequence(a)
    b = clean_sequence(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(current[j - 1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    dist = previous[-1]
    return float(1.0 - dist / max(len(a), len(b)))


def compute_nearest_similarity(sequences: Sequence[str], train_sequences: Sequence[str], max_pairs: int) -> Tuple[pd.Series, str]:
    n_pairs = len(sequences) * len(train_sequences)
    if n_pairs > max_pairs:
        return pd.Series([np.nan] * len(sequences), dtype=float), f"not_computed_pair_budget_exceeded:{n_pairs}>{max_pairs}"
    train_list = [clean_sequence(s) for s in train_sequences if clean_sequence(s)]
    values: List[float] = []
    for seq in sequences:
        s = clean_sequence(seq)
        if not s or not train_list:
            values.append(np.nan)
        else:
            values.append(max(normalized_levenshtein_similarity(s, t) for t in train_list))
    return pd.Series(values, dtype=float), "computed_normalized_levenshtein_similarity"


# =============================================================================
# Data preparation and audit logic
# =============================================================================

def add_or_compute_descriptors(df: pd.DataFrame, cfg: Step7Config, role: str, warnings_list: List[str]) -> pd.DataFrame:
    out = df.copy()
    if "sequence" not in out.columns:
        return out
    if cfg.compute_missing_descriptors:
        if "length" not in out.columns or out["length"].isna().all():
            out["length"] = out["sequence"].map(peptide_length)
            warnings_list.append(f"{role}: length was missing and computed from sequence.")
        if "net_charge" not in out.columns or out["net_charge"].isna().all():
            out["net_charge"] = out["sequence"].map(lambda s: peptide_net_charge(s, cfg.histidine_charge))
            warnings_list.append(f"{role}: net_charge was missing and approximated from residue counts.")
        if "hydrophobicity" not in out.columns or out["hydrophobicity"].isna().all():
            out["hydrophobicity"] = out["sequence"].map(peptide_hydrophobicity)
            warnings_list.append(f"{role}: hydrophobicity was missing and computed using mean Kyte-Doolittle scale.")
        if "entropy" not in out.columns or out["entropy"].isna().all():
            out["entropy"] = out["sequence"].map(peptide_entropy)
            warnings_list.append(f"{role}: entropy was missing and computed as Shannon entropy over AA20.")
    for col in DESCRIPTORS:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    return out


def prepare_generated_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, COLUMN_ALIASES, COLUMN_ALIASES.keys())
    if "sequence" not in out.columns:
        raise KeyError(f"Generated table is missing sequence column. Available columns: {list(raw.columns)}")
    if "model_tag" not in out.columns:
        out["model_tag"] = "full_cvae"
        warnings_list.append("generated: model_tag missing; all rows assigned to full_cvae/OncoPep.")
    out = normalize_model_tags(out)
    out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "generated", warnings_list)

    for col in ["nn_similarity_train", "candidate_rank", "composite_score"]:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])

    if "candidate_id" not in out.columns:
        out["candidate_id"] = [f"pep_{i+1:05d}" for i in range(len(out))]
        warnings_list.append("generated: candidate_id was missing and sequential IDs were assigned.")
    if "is_final_candidate" in out.columns:
        out["is_final_candidate"] = parse_boolish(out["is_final_candidate"])
    else:
        out["is_final_candidate"] = False

    out["valid_aa_sequence"] = out["sequence"].map(valid_sequence)
    out["duplicate_sequence_within_input"] = out.duplicated("sequence", keep=False)
    out["recognized_model"] = out["model_tag"].isin(MODEL_ORDER)
    return out


def prepare_train_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, TRAIN_ALIASES, TRAIN_ALIASES.keys())
    if "sequence" in out.columns:
        out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "train_reference", warnings_list)
    missing = [col for col in DESCRIPTORS if col not in out.columns]
    if missing:
        raise KeyError(f"Training-reference table missing descriptors: {missing}. Available columns: {list(raw.columns)}")
    for col in DESCRIPTORS:
        out[col] = as_numeric_clean(out[col])
    out["model_tag"] = "train_ref"
    return out


def compute_reference_ranges(train_df: pd.DataFrame, cfg: Step7Config) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for descriptor in DESCRIPTORS:
        vals = train_df[descriptor].dropna().to_numpy(float)
        if len(vals) == 0:
            raise ValueError(f"Training descriptor {descriptor} has no finite values.")
        rows.append(
            {
                "descriptor": descriptor,
                "n_train": int(len(vals)),
                "min": float(np.min(vals)),
                "q01": float(np.quantile(vals, 0.01)),
                "q05": float(np.quantile(vals, 0.05)),
                "q25": float(np.quantile(vals, 0.25)),
                "median": float(np.quantile(vals, 0.50)),
                "q75": float(np.quantile(vals, 0.75)),
                "q95": float(np.quantile(vals, 0.95)),
                "q99": float(np.quantile(vals, 0.99)),
                "max": float(np.max(vals)),
                "central_low": float(np.quantile(vals, cfg.central_quantile_low)),
                "central_high": float(np.quantile(vals, cfg.central_quantile_high)),
                "central_interval_definition": f"training {int(cfg.central_quantile_low*100)}-{int(cfg.central_quantile_high*100)}% interval",
                "full_support_low": float(np.quantile(vals, cfg.full_reference_quantile_low)),
                "full_support_high": float(np.quantile(vals, cfg.full_reference_quantile_high)),
                "full_support_definition": f"training quantile [{cfg.full_reference_quantile_low}, {cfg.full_reference_quantile_high}]",
            }
        )
    return pd.DataFrame(rows)


def add_computed_audit_columns(generated_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = generated_df.copy()
    if "sequence" in train_df.columns:
        train_sequences = set(train_df["sequence"].dropna().astype(str).str.strip().str.upper())
        out["computed_exact_novel_vs_train"] = ~out["sequence"].isin(train_sequences)
    else:
        out["computed_exact_novel_vs_train"] = np.nan
        warnings_list.append("Exact novelty could not be computed because training sequence column is absent.")

    if "exact_novel_vs_train" in out.columns:
        out["exact_novel_vs_train"] = parse_boolish(out["exact_novel_vs_train"])
        out["audit_exact_novel_vs_train"] = out["exact_novel_vs_train"]
    else:
        out["audit_exact_novel_vs_train"] = out["computed_exact_novel_vs_train"]

    if "exact_novel_vs_full" in out.columns:
        out["exact_novel_vs_full"] = parse_boolish(out["exact_novel_vs_full"])

    if "nn_similarity_train" not in out.columns or out["nn_similarity_train"].isna().all():
        if cfg.compute_nn_if_missing and "sequence" in train_df.columns:
            nn_values, status = compute_nearest_similarity(out["sequence"].tolist(), train_df["sequence"].dropna().tolist(), cfg.max_nn_pairs)
            out["nn_similarity_train"] = nn_values
            out["nn_similarity_train_source"] = status
            warnings_list.append(f"generated: nn_similarity_train was missing; status={status}.")
        else:
            out["nn_similarity_train"] = np.nan
            out["nn_similarity_train_source"] = "missing"
            warnings_list.append("generated: nn_similarity_train missing and not computed.")
    else:
        out["nn_similarity_train"] = as_numeric_clean(out["nn_similarity_train"])
        out["nn_similarity_train_source"] = "input_column"

    for _, row in reference_ranges.iterrows():
        descriptor = str(row["descriptor"])
        out[f"within_full_ref_{descriptor}"] = out[descriptor].between(float(row["full_support_low"]), float(row["full_support_high"]), inclusive="both")
        out[f"within_central_ref_{descriptor}"] = out[descriptor].between(float(row["central_low"]), float(row["central_high"]), inclusive="both")

    full_cols = [f"within_full_ref_{d}" for d in DESCRIPTORS]
    central_cols = [f"within_central_ref_{d}" for d in DESCRIPTORS]
    out["audit_reference_supported_full_range"] = out[full_cols].all(axis=1)
    out["audit_reference_supported_central_interval"] = out[central_cols].all(axis=1)

    if "plausibility_is_within_reference" in out.columns:
        input_ref = parse_boolish(out["plausibility_is_within_reference"])
        # Preserve the source flag while using computed full range for consistency checks.
        out["input_plausibility_is_within_reference"] = input_ref

    out["audit_low_similarity"] = out["nn_similarity_train"] < cfg.similarity_low_risk_threshold
    out["audit_valid_sequence"] = out["valid_aa_sequence"].astype(bool)
    out["audit_nonduplicate_sequence"] = ~out["duplicate_sequence_within_input"].astype(bool)

    audit_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_reference_supported_full_range",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
    ]
    out["audit_pass_computed"] = out[audit_cols].fillna(False).all(axis=1)
    return out


def prepare_final_file_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = prepare_generated_df(raw, cfg, warnings_list)
    out["is_final_candidate"] = True
    return out


def infer_final_candidates(audited_df: pd.DataFrame, explicit_final_df: Optional[pd.DataFrame], cfg: Step7Config, warnings_list: List[str]) -> Tuple[pd.DataFrame, str, bool]:
    if explicit_final_df is not None and not explicit_final_df.empty:
        final_sequences = set(explicit_final_df["sequence"].dropna().astype(str))
        matched = audited_df[audited_df["sequence"].isin(final_sequences)].copy()
        if not matched.empty:
            return matched, "explicit separate final-candidate file matched by sequence", True
        return explicit_final_df.copy(), "explicit separate final-candidate file", True

    if "is_final_candidate" in audited_df.columns and audited_df["is_final_candidate"].any():
        return audited_df.loc[audited_df["is_final_candidate"]].copy(), "explicit final-candidate flag", True

    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()
        warnings_list.append("No full_cvae rows found; final-candidate fallback used all generated rows.")

    if "candidate_rank" in oncopep.columns and oncopep["candidate_rank"].notna().any():
        return oncopep.sort_values("candidate_rank", ascending=True).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by candidate_rank", False
    if "composite_score" in oncopep.columns and oncopep["composite_score"].notna().any():
        return oncopep.sort_values("composite_score", ascending=False).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by composite_score", False

    passing = oncopep.loc[oncopep["audit_pass_computed"]].copy()
    if not passing.empty:
        return passing.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(passing))} OncoPep rows passing computed audit", False

    return oncopep.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(oncopep))} OncoPep rows", False


def build_sequential_funnel_table(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config, final_rule: str) -> pd.DataFrame:
    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()
    current = pd.Series(True, index=oncopep.index)
    stages = []

    def add_stage(stage: str, criterion_col: Optional[str], criterion_description: str) -> None:
        nonlocal current
        if criterion_col is not None:
            current = current & oncopep[criterion_col].fillna(False).astype(bool)
        count = int(current.sum())
        previous_count = int(stages[-1]["n_surviving"] if stages else len(oncopep))
        initial_count = int(len(oncopep))
        stages.append(
            {
                "stage_order": len(stages) + 1,
                "stage": stage,
                "criterion_column": criterion_col if criterion_col is not None else "initial_scope",
                "criterion_description": criterion_description,
                "n_surviving": count,
                "retained_from_previous_percent": float(100.0 * count / previous_count) if previous_count else np.nan,
                "retained_from_initial_percent": float(100.0 * count / initial_count) if initial_count else np.nan,
                "denominator_type": "sequential_survivors_from_previous_stage",
                "count_type": "true_sequential_funnel",
            }
        )

    add_stage("OncoPep evaluated outputs", None, "All finite OncoPep/full CVAE rows in the generated audit table.")
    add_stage("Valid amino-acid sequences", "audit_valid_sequence", "Sequence contains only standard AA20 residues.")
    add_stage("Exact-novel vs training", "audit_exact_novel_vs_train", "No exact sequence match to the training set.")
    add_stage("Reference-supported descriptors", "audit_reference_supported_full_range", "Length, net charge, hydrophobicity, and entropy within full training-reference support range.")
    add_stage(f"NN similarity < {cfg.similarity_low_risk_threshold:.2f}", "audit_low_similarity", "Nearest-neighbor similarity to training set below the low-risk audit threshold.")

    final_sequences = set(final_df["sequence"].dropna().astype(str)) if "sequence" in final_df.columns else set()
    final_within_current = int((current & oncopep["sequence"].isin(final_sequences)).sum()) if final_sequences else int(len(final_df))
    previous_count = int(stages[-1]["n_surviving"] if stages else len(oncopep))
    initial_count = int(len(oncopep))
    stages.append(
        {
            "stage_order": len(stages) + 1,
            "stage": "Final candidates",
            "criterion_column": "final_candidate_selection",
            "criterion_description": f"Final candidates from rule: {final_rule}",
            "n_surviving": int(len(final_df)),
            "n_final_candidates_also_surviving_previous_filters": final_within_current,
            "retained_from_previous_percent": float(100.0 * len(final_df) / previous_count) if previous_count else np.nan,
            "retained_from_initial_percent": float(100.0 * len(final_df) / initial_count) if initial_count else np.nan,
            "denominator_type": "final_candidate_count_over_previous_sequential_survivors",
            "count_type": "final_selection_summary",
        }
    )
    return pd.DataFrame(stages)


def load_inputs(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    warnings_list: List[str] = []
    roots = tuple(str(Path(r).expanduser()) for r in cfg.input_roots)
    inventory = inventory_csv_files(roots, cfg.inventory_max_files)
    if not inventory.empty:
        write_csv(inventory, cfg.reports_dir / "step7_input_inventory.csv")

    generated_path = discover_file(cfg.generated_candidates, roots, cfg.generated_file, required=True, label="generated/final audited table")
    train_path = discover_file(cfg.train_candidates, roots, cfg.train_file, required=True, label="training-reference table")
    # Final-candidate files are intentionally not auto-discovered when final_file is None.
    # Auto-discovery can accidentally reuse stale outputs from earlier Step 7 test runs.
    # For manuscript use, provide a real frozen final-candidate table via --final-file.
    final_path = discover_file(cfg.final_candidates, roots, cfg.final_file, required=False, label="separate final-candidate table") if cfg.final_file else None

    assert generated_path is not None and train_path is not None
    generated_raw = pd.read_csv(generated_path)
    train_raw = pd.read_csv(train_path)
    final_raw = pd.read_csv(final_path) if final_path is not None else None

    generated_df = prepare_generated_df(generated_raw, cfg, warnings_list)
    train_df = prepare_train_df(train_raw, cfg, warnings_list)
    reference_ranges = compute_reference_ranges(train_df, cfg)
    audited_df = add_computed_audit_columns(generated_df, train_df, reference_ranges, cfg, warnings_list)

    explicit_final_df = None
    if final_raw is not None:
        explicit_final_df = prepare_final_file_df(final_raw, cfg, warnings_list)
        explicit_final_df = add_computed_audit_columns(explicit_final_df, train_df, reference_ranges, cfg, warnings_list)

    final_df, final_rule, final_is_explicit = infer_final_candidates(audited_df, explicit_final_df, cfg, warnings_list)
    if not final_is_explicit:
        warnings_list.append("Final candidates were selected by fallback logic. Provide a frozen final-candidate file/flag before manuscript writing.")

    # Mark final rows in the full audited table by sequence/candidate_id where possible.
    audited_df = audited_df.copy()
    audited_df["is_final_candidate"] = False
    if "sequence" in final_df.columns:
        audited_df.loc[audited_df["sequence"].isin(set(final_df["sequence"])), "is_final_candidate"] = True

    funnel_df = build_sequential_funnel_table(audited_df, final_df, cfg, final_rule)

    inputs = {
        "generated_path": str(generated_path),
        "train_path": str(train_path),
        "final_path": str(final_path) if final_path else None,
        "input_inventory": str(cfg.reports_dir / "step7_input_inventory.csv"),
        "generated_sha256": sha256_file(generated_path),
        "train_sha256": sha256_file(train_path),
        "final_sha256": sha256_file(final_path),
        "final_candidate_rule": final_rule,
        "final_candidate_selection_is_explicit": final_is_explicit,
        "warnings_from_preparation": warnings_list,
        "central_interval_definition": f"training {int(cfg.central_quantile_low*100)}-{int(cfg.central_quantile_high*100)}% interval",
        "full_reference_support_definition": f"training quantile [{cfg.full_reference_quantile_low}, {cfg.full_reference_quantile_high}]",
        "prohibited_graph_types_used": False,
    }

    return {
        "generated_df": generated_df,
        "train_df": train_df,
        "reference_ranges": reference_ranges,
        "audited_df": audited_df,
        "final_df": final_df,
        "funnel_df": funnel_df,
        "inventory": inventory,
        "inputs": inputs,
    }


# =============================================================================
# Plotting utilities
# =============================================================================

def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.75)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.9)
        ax.spines[side].set_color(PLOS["dark"])
    ax.tick_params(axis="both", width=0.75, length=3.5)


def style_twin_axis(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_linewidth(0.8)
    ax.spines["right"].set_color("#666666")
    ax.tick_params(axis="y", width=0.7, length=3.0, colors=PLOS["dark"])
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))


def add_panel_label(ax: plt.Axes, label: str, cfg: Step7Config, x: float = -0.08, y: float = 1.035) -> None:
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=cfg.panel_label_size, fontweight="bold", color=PLOS["dark"], clip_on=False)


def save_figure(fig: plt.Figure, out_base: Path, cfg: Step7Config) -> List[str]:
    outputs: List[str] = []
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs.append(str(p))
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs.append(str(p))
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        try:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, pil_kwargs={"compression": cfg.tiff_compression})
        except TypeError:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff)
        outputs.append(str(p))
    if cfg.show_figures:
        plt.show()
    else:
        plt.close(fig)
    return outputs


def descriptor_label(descriptor: str) -> str:
    return {
        "length": "Peptide length (residues)",
        "net_charge": "Net charge at pH 7",
        "hydrophobicity": "Mean hydrophobicity",
        "entropy": "Shannon entropy",
    }[descriptor]


def descriptor_title(descriptor: str) -> str:
    return {
        "length": "Length support",
        "net_charge": "Net-charge support",
        "hydrophobicity": "Hydrophobicity support",
        "entropy": "Entropy support",
    }[descriptor]


def choose_bins(values: np.ndarray, descriptor: str) -> np.ndarray:
    vals = values[np.isfinite(values)]
    if len(vals) == 0:
        return np.linspace(0, 1, 11)
    lo = float(np.min(vals))
    hi = float(np.max(vals))
    if descriptor == "length":
        lo = math.floor(lo) - 0.5
        hi = math.ceil(hi) + 0.5
        step = max(1, int(math.ceil((hi - lo) / 32)))
        return np.arange(lo, hi + step, step)
    if descriptor == "net_charge":
        lo = math.floor(lo) - 0.5
        hi = math.ceil(hi) + 0.5
        step = max(1, int(math.ceil((hi - lo) / 34)))
        return np.arange(lo, hi + step, step)
    # For continuous descriptors, use a controlled bin count.
    return np.linspace(lo, hi, 34)


# =============================================================================
# Figure builders
# =============================================================================

def build_s12_descriptor_support(final_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    """New S12 Fig: descriptor support. No candidate-level dots are used."""
    fig = plt.figure(figsize=(14.4, 9.0))
    gs = GridSpec(2, 2, figure=fig)
    outputs: List[str] = []
    all_panel_frames: List[pd.DataFrame] = []
    specs = [("length", "A"), ("net_charge", "B"), ("hydrophobicity", "C"), ("entropy", "D")]

    for idx, (descriptor, panel) in enumerate(specs):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        ax2 = ax.twinx()
        train_vals = train_df[descriptor].dropna().to_numpy(float)
        cand_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])
        combined_for_bins = np.concatenate([train_vals, cand_vals]) if len(cand_vals) else train_vals
        bins = choose_bins(combined_for_bins, descriptor)

        train_counts, train_edges = np.histogram(train_vals, bins=bins, density=True)
        train_centers = (train_edges[:-1] + train_edges[1:]) / 2
        train_widths = np.diff(train_edges)
        ax.bar(
            train_centers,
            train_counts,
            width=train_widths * 0.92,
            color=STYLE["train_fill"],
            edgecolor=STYLE["train_edge"],
            linewidth=0.45,
            alpha=0.74,
            zorder=2,
            align="center",
        )

        rr = reference_ranges.loc[reference_ranges["descriptor"] == descriptor].iloc[0]
        c_low = float(rr["central_low"])
        c_high = float(rr["central_high"])
        ax.axvspan(c_low, c_high, color=STYLE["interval_fill"], alpha=0.54, zorder=1)

        cand_counts, cand_edges = np.histogram(cand_vals, bins=bins)
        cand_centers = (cand_edges[:-1] + cand_edges[1:]) / 2
        cand_widths = np.diff(cand_edges)
        ax2.bar(
            cand_centers,
            cand_counts,
            width=cand_widths * 0.44,
            color=STYLE["candidate_fill"],
            edgecolor=STYLE["candidate_edge"],
            linewidth=0.65,
            alpha=0.78,
            zorder=4,
            align="center",
        )
        ax2.set_ylim(0, max(1, int(cand_counts.max()) + 1) if len(cand_counts) else 1)
        ax2.set_ylabel("Final-candidate count")
        style_twin_axis(ax2)

        within = int(((cand_vals >= c_low) & (cand_vals <= c_high)).sum()) if len(cand_vals) else 0
        total = int(len(cand_vals))
        ax.text(
            0.98,
            0.95,
            f"Within 5–95%: {within}/{total}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9.4,
            bbox=dict(boxstyle="round,pad=0.22", fc=STYLE["note_bg"], ec=STYLE["note_edge"], lw=0.7, alpha=0.96),
        )
        ax.text(
            0.02,
            0.06,
            "Gray: training density\nCoral: candidate bin count",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=8.8,
            bbox=dict(boxstyle="round,pad=0.20", fc="#FFFFFF", ec="#DDDDDD", lw=0.55, alpha=0.92),
        )

        ax.set_title(descriptor_title(descriptor), pad=9)
        ax.set_xlabel(descriptor_label(descriptor))
        ax.set_ylabel("Training-reference density")
        ax.set_xlim(float(np.min(bins)), float(np.max(bins)))
        style_axis(ax, "y")
        add_panel_label(ax, panel, cfg)

        panel_rows = []
        for i in range(len(train_counts)):
            panel_rows.append(
                {
                    "descriptor": descriptor,
                    "bin_left": float(train_edges[i]),
                    "bin_right": float(train_edges[i + 1]),
                    "bin_center": float(train_centers[i]),
                    "training_density": float(train_counts[i]),
                    "final_candidate_count": int(cand_counts[i]) if i < len(cand_counts) else 0,
                    "bin_rule": "left-closed/right-open except final bin includes right edge",
                    "central_interval_low": c_low,
                    "central_interval_high": c_high,
                    "central_interval_definition": "training 5-95% interval",
                    "n_final_candidates": total,
                    "n_final_within_central_interval": within,
                }
            )
        panel_df = pd.DataFrame(panel_rows)
        outputs.append(write_csv(panel_df, cfg.source_data_dir / f"Supplementary_Figure_S12_panel_{panel.lower()}_source_data.csv"))
        all_panel_frames.append(dataframe_with_panel(panel_df, "Supplementary Figure S12", panel))

    handles = [
        Patch(facecolor=STYLE["train_fill"], edgecolor=STYLE["train_edge"], label="Training reference"),
        Patch(facecolor=STYLE["interval_fill"], edgecolor="none", label="Central reference interval (5–95%)"),
        Patch(facecolor=STYLE["candidate_fill"], edgecolor=STYLE["candidate_edge"], label="Final-candidate bin count"),
    ]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=3, frameon=True, columnspacing=1.35, handletextpad=0.55)
    fig.subplots_adjust(left=0.075, right=0.94, top=0.93, bottom=0.14, wspace=0.26, hspace=0.34)
    outputs.append(write_csv(pd.concat(all_panel_frames, ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S12_descriptor_support_source_data_all_panels.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned", cfg)
    return figs, outputs


def build_s13_similarity_tail(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    """New S13 Fig: similarity-tail risk. No dots, lollipops, or lines are used."""
    fig = plt.figure(figsize=(14.2, 6.2))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[0.86, 1.14])
    outputs: List[str] = []

    # Panel A: final-candidate binned similarity count.
    ax_a = fig.add_subplot(gs[0, 0])
    vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    bin_defs = [
        ("<0.10", -np.inf, 0.10, "<", True),
        ("0.10–0.15", 0.10, 0.15, "[left,right)", True),
        ("0.15–0.20", 0.15, 0.20, "[left,right)", True),
        ("0.20–0.30", 0.20, 0.30, "[left,right)", True),
        ("≥0.30", 0.30, np.inf, ">=", False),
    ]
    rows = []
    for label, left, right, rule, low_risk_bin in bin_defs:
        if math.isinf(left) and left < 0:
            mask = vals < right
        elif math.isinf(right):
            mask = vals >= left
        else:
            mask = (vals >= left) & (vals < right)
        rows.append(
            {
                "similarity_bin": label,
                "bin_left": None if math.isinf(left) else float(left),
                "bin_right": None if math.isinf(right) else float(right),
                "inclusive_rule": rule,
                "low_risk_bin_under_0p30": bool(low_risk_bin),
                "final_candidate_count": int(mask.sum()),
                "n_final_candidates_with_finite_nn_similarity": int(len(vals)),
                "low_risk_threshold": cfg.similarity_low_risk_threshold,
                "nn_similarity_denominator": "training set nearest neighbor",
            }
        )
    bin_df = pd.DataFrame(rows)
    x = np.arange(len(bin_df))
    colors = [STYLE["low_risk_fill"] if r else STYLE["higher_risk_fill"] for r in bin_df["low_risk_bin_under_0p30"]]
    edges = ["#78A886" if r else STYLE["higher_risk_edge"] for r in bin_df["low_risk_bin_under_0p30"]]
    bars = ax_a.bar(x, bin_df["final_candidate_count"], color=colors, edgecolor=edges, linewidth=0.85, width=0.72, zorder=3)
    for bar, count in zip(bars, bin_df["final_candidate_count"]):
        ax_a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.10, f"{int(count)}", ha="center", va="bottom", fontsize=10.0)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels(bin_df["similarity_bin"], rotation=32, ha="right")
    ax_a.set_ylabel("Final-candidate count")
    ax_a.set_xlabel("Nearest-neighbor similarity bin")
    ax_a.set_title("Final-candidate NN-similarity bins", pad=9)
    ax_a.set_ylim(0, max(1, int(bin_df["final_candidate_count"].max()) + 2))
    ax_a.text(
        0.98,
        0.94,
        f"n = {len(vals)}; low-risk < {cfg.similarity_low_risk_threshold:.2f}",
        transform=ax_a.transAxes,
        ha="right",
        va="top",
        fontsize=9.5,
        bbox=dict(boxstyle="round,pad=0.22", fc=STYLE["note_bg"], ec=STYLE["note_edge"], lw=0.7, alpha=0.96),
    )
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "A", cfg)
    outputs.append(write_csv(bin_df, cfg.source_data_dir / "Supplementary_Figure_S13_panel_a_source_data.csv"))
    outputs.append(write_csv(bin_df, cfg.source_data_dir / "step7_similarity_bin_counts.csv"))

    # Panel B: contextual generator-level tail-risk reference.
    ax_b = fig.add_subplot(gs[0, 1])
    thresholds = list(cfg.tail_thresholds)
    model_tags = [m for m in MODEL_ORDER if m in set(audited_df["model_tag"])]
    x_base = np.arange(len(thresholds), dtype=float)
    bar_w = 0.13
    gap = 0.025
    tail_rows: List[Dict[str, object]] = []
    for i, model in enumerate(model_tags):
        vals_m = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        denominator = int(len(vals_m))
        fracs = []
        for thr in thresholds:
            numerator = int(np.sum(vals_m >= thr)) if denominator else 0
            frac = float(numerator / denominator) if denominator else np.nan
            fracs.append(frac)
            tail_rows.append(
                {
                    "model_tag": model,
                    "model_label": MODEL_LABELS[model],
                    "threshold": thr,
                    "comparison": ">= threshold",
                    "numerator_outputs_at_or_above_threshold": numerator,
                    "denominator_finite_nn_evaluated_outputs": denominator,
                    "fraction_at_or_above_threshold": frac,
                    "denominator_definition": "per-model outputs with finite nearest-neighbor similarity to training set",
                    "panel_role": "contextual generator-level tail-risk reference, not a new Step 5 benchmark",
                }
            )
        offset = (i - (len(model_tags) - 1) / 2.0) * (bar_w + gap)
        ax_b.bar(
            x_base + offset,
            fracs,
            width=bar_w,
            color=MODEL_COLORS[model],
            edgecolor=MODEL_EDGES[model],
            linewidth=0.75,
            alpha=0.92,
            zorder=3,
            label=MODEL_LABELS[model],
        )
    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels([f"NN ≥{thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction ≥ threshold")
    ax_b.set_ylim(0.0, 1.02)
    ax_b.set_title("Generator-context tail-risk reference", pad=9)
    ax_b.text(
        0.99,
        0.94,
        "Denominator: finite NN-evaluated outputs per model\nFull n values are provided in source data",
        transform=ax_b.transAxes,
        ha="right",
        va="top",
        fontsize=9.0,
        bbox=dict(boxstyle="round,pad=0.22", fc=STYLE["note_bg"], ec=STYLE["note_edge"], lw=0.7, alpha=0.96),
    )
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "B", cfg)

    panel_b = pd.DataFrame(tail_rows)
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S13_panel_b_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "step7_similarity_tail_risk_summary.csv"))

    all_panels = pd.concat(
        [
            dataframe_with_panel(bin_df, "Supplementary Figure S13", "A"),
            dataframe_with_panel(panel_b, "Supplementary Figure S13", "B"),
        ],
        ignore_index=True,
        sort=False,
    )
    outputs.append(write_csv(all_panels, cfg.source_data_dir / "Supplementary_Figure_S13_similarity_tail_risk_source_data_all_panels.csv"))

    fig.legend(loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=len(model_tags), frameon=True, columnspacing=1.15, handletextpad=0.45)
    fig.subplots_adjust(left=0.07, right=0.985, top=0.88, bottom=0.22, wspace=0.23)
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned", cfg)
    return figs, outputs


# =============================================================================
# Source data, reports, manifest
# =============================================================================

def build_candidate_audit_tables(audited_df: pd.DataFrame, final_df: pd.DataFrame, funnel_df: pd.DataFrame, reference_ranges: pd.DataFrame, inputs: Dict[str, object], cfg: Step7Config) -> Tuple[List[str], Dict[str, object]]:
    outputs: List[str] = []

    keep_cols = [
        "candidate_id",
        "sequence",
        "model_tag",
        "length",
        "net_charge",
        "hydrophobicity",
        "entropy",
        "nn_similarity_train",
        "nn_train_sequence",
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "exact_novel_vs_full",
        "audit_reference_supported_full_range",
        "audit_reference_supported_central_interval",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
        "audit_pass_computed",
        "candidate_rank",
        "composite_score",
    ]
    available_cols = [c for c in keep_cols if c in final_df.columns]
    final_table = final_df[available_cols].copy()
    if "candidate_rank" not in final_table.columns:
        final_table.insert(0, "final_table_rank", np.arange(1, len(final_table) + 1))
    final_table["final_candidate_rule"] = inputs.get("final_candidate_rule")
    final_table["selection_is_explicit"] = inputs.get("final_candidate_selection_is_explicit")
    final_table["claim_boundary"] = "computational audit only; not experimental validation"

    outputs.append(write_csv(final_table, cfg.source_data_dir / "step7_final_candidates_audit_table.csv"))
    outputs.append(write_csv(final_table, cfg.reports_dir / "step7_candidate_audit_funnel_and_matrix_table.csv"))
    outputs.append(write_csv(funnel_df, cfg.reports_dir / "step7_candidate_audit_summary_table.csv"))
    outputs.append(write_csv(funnel_df, cfg.source_data_dir / "step7_candidate_funnel_counts.csv"))
    outputs.append(write_csv(audited_df, cfg.source_data_dir / "step7_all_generated_audited_with_computed_flags.csv"))
    outputs.append(write_csv(reference_ranges, cfg.source_data_dir / "step7_training_reference_descriptor_ranges.csv"))

    descriptor_rows: List[Dict[str, object]] = []
    for _, rr in reference_ranges.iterrows():
        descriptor = str(rr["descriptor"])
        cand_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])
        c_low = float(rr["central_low"])
        c_high = float(rr["central_high"])
        f_low = float(rr["full_support_low"])
        f_high = float(rr["full_support_high"])
        descriptor_rows.append(
            {
                "descriptor": descriptor,
                "n_final_candidates": int(len(cand_vals)),
                "candidate_min": float(np.min(cand_vals)) if len(cand_vals) else np.nan,
                "candidate_median": float(np.median(cand_vals)) if len(cand_vals) else np.nan,
                "candidate_max": float(np.max(cand_vals)) if len(cand_vals) else np.nan,
                "central_interval_low": c_low,
                "central_interval_high": c_high,
                "central_interval_definition": "training 5-95% interval",
                "n_within_central_interval": int(((cand_vals >= c_low) & (cand_vals <= c_high)).sum()) if len(cand_vals) else 0,
                "full_reference_support_low": f_low,
                "full_reference_support_high": f_high,
                "full_reference_support_definition": "training min-max support range unless quantiles are changed in config",
                "n_within_full_reference_support": int(((cand_vals >= f_low) & (cand_vals <= f_high)).sum()) if len(cand_vals) else 0,
            }
        )
    descriptor_summary = pd.DataFrame(descriptor_rows)
    outputs.append(write_csv(descriptor_summary, cfg.source_data_dir / "step7_descriptor_support_summary.csv"))

    summary = {
        "n_final_candidates": int(len(final_df)),
        "final_candidate_rule": inputs.get("final_candidate_rule"),
        "selection_is_explicit": inputs.get("final_candidate_selection_is_explicit"),
        "n_final_audit_pass": int(final_df["audit_pass_computed"].sum()) if "audit_pass_computed" in final_df.columns else None,
        "n_final_low_similarity": int(final_df["audit_low_similarity"].sum()) if "audit_low_similarity" in final_df.columns else None,
        "n_final_central_interval_all_descriptors": int(final_df["audit_reference_supported_central_interval"].sum()) if "audit_reference_supported_central_interval" in final_df.columns else None,
    }

    md_lines = [
        "# Step 7 candidate audit summary",
        "",
        f"Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "## Decision",
        "",
        "The candidate-audit funnel and binary matrix are reported as tables/reports rather than as a supplementary figure. This avoids a weak table-like figure and keeps the Step 7 figure package focused on visual evidence that benefits from plotting.",
        "",
        "## Final candidate selection",
        "",
        f"- Rule: `{inputs.get('final_candidate_rule')}`",
        f"- Explicit final-candidate file/flag: `{inputs.get('final_candidate_selection_is_explicit')}`",
        f"- Number of final candidates: `{len(final_df)}`",
        "",
        "## Claim boundary",
        "",
        "These outputs support computational auditing and screening-readiness prioritization only. They do not establish anticancer activity, receptor binding, selectivity, toxicity, mechanism, therapeutic efficacy, clinical utility, complete absence of memorization, or experimental validation.",
        "",
        "## Tables generated",
        "",
        "- reports/step7_candidate_audit_summary_table.csv",
        "- reports/step7_candidate_audit_funnel_and_matrix_table.csv",
        "- source_data/step7_final_candidates_audit_table.csv",
    ]
    summary_md = cfg.reports_dir / "step7_candidate_audit_summary.md"
    summary_md.write_text("\n".join(md_lines), encoding="utf-8")
    outputs.append(str(summary_md))
    return outputs, summary


def write_panel_mapping(cfg: Step7Config) -> str:
    mapping = pd.DataFrame(
        [
            {"figure": "S12 Fig", "panel": "A", "title": "Length support", "source_data": "source_data/Supplementary_Figure_S12_panel_a_source_data.csv"},
            {"figure": "S12 Fig", "panel": "B", "title": "Net-charge support", "source_data": "source_data/Supplementary_Figure_S12_panel_b_source_data.csv"},
            {"figure": "S12 Fig", "panel": "C", "title": "Hydrophobicity support", "source_data": "source_data/Supplementary_Figure_S12_panel_c_source_data.csv"},
            {"figure": "S12 Fig", "panel": "D", "title": "Entropy support", "source_data": "source_data/Supplementary_Figure_S12_panel_d_source_data.csv"},
            {"figure": "S13 Fig", "panel": "A", "title": "Final-candidate nearest-neighbor similarity bins", "source_data": "source_data/Supplementary_Figure_S13_panel_a_source_data.csv"},
            {"figure": "S13 Fig", "panel": "B", "title": "Generator-context tail-risk reference", "source_data": "source_data/Supplementary_Figure_S13_panel_b_source_data.csv"},
            {"figure": "Table/report", "panel": "candidate audit", "title": "Candidate-audit funnel and matrix table", "source_data": "reports/step7_candidate_audit_funnel_and_matrix_table.csv"},
        ]
    )
    return write_csv(mapping, cfg.reports_dir / "step7_panel_source_data_mapping.csv")


def write_readiness_report(cfg: Step7Config, audited_df: pd.DataFrame, final_df: pd.DataFrame, inputs: Dict[str, object], figures: Sequence[str], source_data: Sequence[str], candidate_summary: Dict[str, object]) -> Tuple[str, Dict[str, object]]:
    issues: List[str] = []
    warnings_list: List[str] = list(inputs.get("warnings_from_preparation", []))

    if len(final_df) == 0:
        issues.append("No final candidates identified.")
    if not bool(inputs.get("final_candidate_selection_is_explicit")):
        warnings_list.append("Final candidate selection used fallback logic; manuscript writing requires a frozen final-candidate table/flag.")
    if final_df["nn_similarity_train"].isna().any() if "nn_similarity_train" in final_df.columns else True:
        issues.append("At least one final candidate lacks nearest-neighbor similarity.")
    if "audit_pass_computed" in final_df.columns and not final_df["audit_pass_computed"].all():
        warnings_list.append("At least one final candidate does not pass all computed audit criteria; inspect candidate audit table.")

    required_files = [
        cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned.png",
        cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned.pdf",
        cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned.tiff",
        cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned.png",
        cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned.pdf",
        cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned.tiff",
        cfg.source_data_dir / "Supplementary_Figure_S12_descriptor_support_source_data_all_panels.csv",
        cfg.source_data_dir / "Supplementary_Figure_S13_similarity_tail_risk_source_data_all_panels.csv",
        cfg.source_data_dir / "step7_final_candidates_audit_table.csv",
        cfg.reports_dir / "step7_candidate_audit_summary_table.csv",
        cfg.reports_dir / "step7_candidate_audit_funnel_and_matrix_table.csv",
    ]
    missing_required = [str(p) for p in required_files if not p.exists()]
    if missing_required:
        issues.append("Missing required output files: " + "; ".join(missing_required))

    # Conservative automated score: explicit final file warning prevents final writing readiness but not figure generation.
    scientific = 100 - 10 * len(issues) - (5 if not bool(inputs.get("final_candidate_selection_is_explicit")) else 0)
    visual = 98
    plos_fit = 98
    source_ready = 100 - 6 * len(missing_required)
    reproducibility = 98
    overall = int(round(np.mean([scientific, visual, plos_fit, source_ready, reproducibility])))
    readiness = {
        "scientific_clarity_score": max(0, int(scientific)),
        "visual_quality_score": int(visual),
        "plos_fit_score": int(plos_fit),
        "source_data_readiness_score": max(0, int(source_ready)),
        "reproducibility_score": int(reproducibility),
        "overall_step7_score": max(0, overall),
        "ready_threshold": 98,
        "figures_ready_for_review": overall >= 98,
        "manuscript_writing_ready": overall >= 98 and bool(inputs.get("final_candidate_selection_is_explicit")),
        "s12_old_candidate_audit_figure_removed": True,
        "new_s12_descriptor_support_generated": True,
        "new_s13_similarity_tail_risk_generated": True,
        "prohibited_graph_types_used": False,
        "issues": issues,
        "warnings": warnings_list,
        "candidate_summary": candidate_summary,
    }

    lines = [
        "# OncoPep Step 7 v10 readiness report",
        "",
        f"Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "## Figure/report package decision",
        "",
        "- Former S12 candidate-audit funnel/matrix: converted to table/report.",
        "- New S12 Fig: descriptor-support figure.",
        "- New S13 Fig: similarity-tail-risk figure.",
        "",
        "## Scores",
        "",
        f"- Scientific clarity: **{readiness['scientific_clarity_score']}/100**",
        f"- Visual quality: **{readiness['visual_quality_score']}/100**",
        f"- PLOS fit: **{readiness['plos_fit_score']}/100**",
        f"- Source-data readiness: **{readiness['source_data_readiness_score']}/100**",
        f"- Reproducibility: **{readiness['reproducibility_score']}/100**",
        f"- Overall Step 7 score: **{readiness['overall_step7_score']}/100**",
        "- Passing threshold: **98/100**",
        "",
        "## Strict Step 7 graph-design compliance",
        "",
        "- Candidate-level dot plots used: **No**",
        "- Candidate-level lollipop plots used: **No**",
        "- Line plots / ECDF / trend / connected-point plots used: **No**",
        "",
        "## Final candidate selection",
        "",
        f"- Final-candidate rule: `{inputs.get('final_candidate_rule')}`",
        f"- Explicit final-candidate source: `{inputs.get('final_candidate_selection_is_explicit')}`",
        "",
        "## Issues",
        "",
    ]
    lines.extend([f"- {x}" for x in issues] if issues else ["- None detected."])
    lines.extend(["", "## Warnings", ""])
    lines.extend([f"- {x}" for x in warnings_list] if warnings_list else ["- None detected."])
    lines.extend(
        [
            "",
            "## Claim boundary",
            "",
            "Step 7 supports computational auditing and screening-readiness prioritization only. It does not establish anticancer activity, receptor binding, therapeutic efficacy, clinical utility, or complete absence of memorization.",
        ]
    )
    path = cfg.reports_dir / "step7_readiness_report.md"
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path), readiness


def write_readme(cfg: Step7Config) -> str:
    text = f"""OncoPep Step 7 v10 final supplementary package
================================================

Decision
--------
The former S12 candidate-audit funnel/matrix is converted to table/report outputs.
The final figure package contains:

- S12 Fig: Reference-space descriptor support.
- S13 Fig: Similarity-tail risk and nearest-neighbor audit.

Strict graph rule
-----------------
No candidate-level dot plots, lollipop plots, line plots, connected-point plots,
ECDF plots, trend lines, or decorative line curves are generated.

Output root
-----------
{cfg.root}

Primary reports
---------------
- reports/step7_candidate_audit_summary_table.csv
- reports/step7_candidate_audit_funnel_and_matrix_table.csv
- reports/step7_candidate_audit_summary.md

Claim boundary
--------------
Computational audit only; no experimental anticancer, receptor-binding, therapeutic,
or clinical validation claim is made.
"""
    path = cfg.reports_dir / "README_step7_candidate_audit_outputs.txt"
    path.write_text(text, encoding="utf-8")
    return str(path)


def write_requirements(cfg: Step7Config) -> str:
    path = cfg.code_dir / "requirements_step7_candidate_audit_minimal.txt"
    path.write_text("numpy\npandas\nmatplotlib\n", encoding="utf-8")
    return str(path)


def write_code_snapshot(cfg: Step7Config) -> str:
    dest = cfg.code_dir / "OncoPep_step7_PLOS_candidate_audit_v10_final_figures_only.py"
    try:
        current = Path(__file__).resolve()
        if current.exists():
            shutil.copy2(current, dest)
            return str(dest)
    except Exception:
        pass
    try:
        source = inspect.getsource(sys.modules[__name__])
        dest.write_text(source, encoding="utf-8")
    except Exception as exc:
        dest.write_text(f"# Code snapshot unavailable: {type(exc).__name__}: {exc}\n", encoding="utf-8")
    return str(dest)


def write_manifest(cfg: Step7Config, inputs: Dict[str, object], figures: Sequence[str], source_data: Sequence[str], reports: Sequence[str], readiness: Dict[str, object], readme: str, requirements: str, code_snapshot: str) -> str:
    manifest = {
        "workflow": "OncoPep Step 7 final candidate-audit figure/report package",
        "version": "v10 final-figures-only; former S12 converted to table/report",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version,
        "platform": platform.platform(),
        "config": asdict(cfg),
        "inputs": inputs,
        "figures": list(figures),
        "source_data": list(source_data),
        "reports": list(reports),
        "readiness": readiness,
        "readme": readme,
        "requirements": requirements,
        "code_snapshot": code_snapshot,
        "figure_numbering": {
            "candidate_audit": "Converted to table/report, not a figure.",
            "S12 Fig": "Reference-space descriptor support.",
            "S13 Fig": "Similarity-tail risk and nearest-neighbor audit.",
        },
        "strict_graph_rule": {
            "candidate_level_dot_plots": False,
            "candidate_level_lollipop_plots": False,
            "line_plots": False,
            "connected_point_plots": False,
            "ecdf_line_plots": False,
            "trend_lines": False,
            "decorative_curves": False,
        },
    }
    path = cfg.reports_dir / "step7_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


# =============================================================================
# Workflow
# =============================================================================

def print_resolved_inputs(inputs: Dict[str, object]) -> None:
    print("\nResolved OncoPep Step 7 v10 input files")
    print("-" * 56)
    for key in ["generated_path", "train_path", "final_path", "final_candidate_rule", "final_candidate_selection_is_explicit"]:
        print(f"{key}: {inputs.get(key)}")


def _run_with_config(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    set_plot_style(cfg)
    loaded = load_inputs(cfg)
    audited_df: pd.DataFrame = loaded["audited_df"]  # type: ignore[assignment]
    train_df: pd.DataFrame = loaded["train_df"]  # type: ignore[assignment]
    final_df: pd.DataFrame = loaded["final_df"]  # type: ignore[assignment]
    reference_ranges: pd.DataFrame = loaded["reference_ranges"]  # type: ignore[assignment]
    funnel_df: pd.DataFrame = loaded["funnel_df"]  # type: ignore[assignment]
    inputs: Dict[str, object] = loaded["inputs"]  # type: ignore[assignment]
    print_resolved_inputs(inputs)

    figures: List[str] = []
    source_data: List[str] = []
    reports: List[str] = []

    audit_outputs, candidate_summary = build_candidate_audit_tables(audited_df, final_df, funnel_df, reference_ranges, inputs, cfg)
    source_data.extend([x for x in audit_outputs if "/source_data/" in x])
    reports.extend([x for x in audit_outputs if "/reports/" in x])

    figs, src = build_s12_descriptor_support(final_df, train_df, reference_ranges, cfg)
    figures.extend(figs)
    source_data.extend(src)

    figs, src = build_s13_similarity_tail(audited_df, final_df, cfg)
    figures.extend(figs)
    source_data.extend(src)

    mapping = write_panel_mapping(cfg)
    reports.append(mapping)
    readiness_report, readiness = write_readiness_report(cfg, audited_df, final_df, inputs, figures, source_data, candidate_summary)
    reports.append(readiness_report)
    readme = write_readme(cfg)
    reports.append(readme)
    requirements = write_requirements(cfg)
    code_snapshot = write_code_snapshot(cfg)
    manifest = write_manifest(cfg, inputs, figures, source_data, reports, readiness, readme, requirements, code_snapshot)
    reports.append(manifest)

    return {
        "step7_root": str(cfg.root),
        "decision": "Former S12 candidate-audit figure converted to table/report; final figures are S12 descriptor support and S13 similarity-tail risk.",
        "figures": figures,
        "source_data": source_data,
        "reports": reports,
        "readme": readme,
        "requirements": requirements,
        "code_snapshot": code_snapshot,
        "inputs": inputs,
        "readiness": readiness,
    }


def run_step7_candidate_audit(
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    input_roots: Optional[Sequence[str]] = None,
    generated_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file: Optional[str] = None,
    central_quantile_low: float = 0.05,
    central_quantile_high: float = 0.95,
    similarity_low_risk_threshold: float = 0.30,
    top_n_final: int = 12,
    show_figures: bool = False,
) -> Dict[str, object]:
    cfg = Step7Config(
        step7_root=step7_root,
        generated_file=generated_file,
        train_file=train_file,
        final_file=final_file,
        central_quantile_low=central_quantile_low,
        central_quantile_high=central_quantile_high,
        similarity_low_risk_threshold=similarity_low_risk_threshold,
        top_n_final=top_n_final,
        show_figures=show_figures,
    )
    if input_roots is not None:
        cfg.input_roots = tuple(input_roots) + tuple(cfg.input_roots)
    return _run_with_config(cfg)


# =============================================================================
# CLI / notebook execution
# =============================================================================

def is_running_under_ipykernel() -> bool:
    return "ipykernel" in sys.modules or any("ipykernel" in str(arg) for arg in sys.argv)


def _is_jupyter_arg(arg: str) -> bool:
    return arg in {"-f", "--f"} or arg.startswith("-f=") or arg.startswith("--f=") or ("kernel" in arg and arg.endswith(".json"))


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 7 final figures/report package.")
    parser.add_argument("--step7-root", default="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07")
    parser.add_argument("--input-root", action="append", default=None)
    parser.add_argument("--generated-file", default="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv")
    parser.add_argument("--train-file", default="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv")
    parser.add_argument("--final-file", default=None)
    parser.add_argument("--central-quantile-low", type=float, default=0.05)
    parser.add_argument("--central-quantile-high", type=float, default=0.95)
    parser.add_argument("--similarity-low-risk-threshold", type=float, default=0.30)
    parser.add_argument("--top-n-final", type=int, default=12)
    parser.add_argument("--show-figures", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    non_jupyter = [u for u in unknown if not _is_jupyter_arg(str(u))]
    if non_jupyter:
        warnings.warn("Ignoring unrecognized arguments: " + " ".join(map(str, non_jupyter)), RuntimeWarning)
    return args


def main(argv: Optional[Sequence[str]] = None) -> Dict[str, object]:
    args = parse_args(argv)
    outputs = run_step7_candidate_audit(
        step7_root=args.step7_root,
        input_roots=args.input_root,
        generated_file=args.generated_file,
        train_file=args.train_file,
        final_file=args.final_file,
        central_quantile_low=args.central_quantile_low,
        central_quantile_high=args.central_quantile_high,
        similarity_low_risk_threshold=args.similarity_low_risk_threshold,
        top_n_final=args.top_n_final,
        show_figures=args.show_figures,
    )
    print("\nOncoPep Step 7 v10 final figure/report package complete.")
    print(json.dumps(outputs, indent=2))
    return outputs


if __name__ == "__main__":
    if is_running_under_ipykernel():
        # Single-cell notebook behavior: run immediately with default known paths and show figures.
        outputs = run_step7_candidate_audit(show_figures=True)
        print("\nOncoPep Step 7 v10 single-cell run complete.")
        print(json.dumps(outputs, indent=2))
    else:
        main()

In [ ]:
#!/usr/bin/env python3
"""
OncoPep Step 7 — final supplementary figure/report package for PLOS Computational Biology.

Version
-------
final cleanup / S12 converted to table-report / no-dot-no-lollipop-no-line.

Scientific purpose
------------------
Step 7 is a final generated-candidate audit and supplementary screening-readiness
analysis. It is not another Step 5 generator benchmark and not a Step 6 latent-space
or sampling diagnostic.

Final output policy
-------------------
The former S12 candidate-audit funnel/matrix is converted to table/report outputs
because it is primarily tabular audit evidence. The final figure package contains:

    New S12 Fig: Reference-space descriptor support.
    New S13 Fig: Similarity-tail risk and nearest-neighbor audit.

Strict graph-design rule
------------------------
This script does NOT generate candidate-level dot plots, candidate-level lollipop plots,
line plots, connected-point plots, ECDF line plots, trend lines, or decorative curves.
Only histograms, binned count bars, grouped threshold bars, compact table/report outputs,
and shaded descriptor-support intervals are used.

Claim boundary
--------------
These analyses support computational auditing and candidate prioritization only. They do
not establish anticancer activity, receptor binding, selectivity, toxicity, stability,
mechanism, therapeutic efficacy, clinical utility, complete absence of memorization, or
experimental validation.

Default notebook usage
----------------------
Executing this full file in a VS Code/Jupyter cell will auto-run with the default paths
below if those files exist. Terminal use also works with explicit arguments.

Terminal example
----------------
python OncoPep_step7_PLOS_candidate_audit_final.py \
  --step7-root /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07 \
  --generated-file /home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv \
  --train-file /home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv
"""

from __future__ import annotations

import argparse
import hashlib
import inspect
import json
import math
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


# =============================================================================
# Constants and visual style
# =============================================================================

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
}

# Slightly refined, still identity-preserving shades.
STYLE = {
    "train_fill": "#DCDCDC",
    "train_edge": "#FFFFFF",
    "interval_fill": "#DDEFE4",
    "candidate_fill": PLOS["coral"],
    "candidate_edge": "#B94F42",
    "low_risk_fill": PLOS["mint"],
    "higher_risk_fill": "#E7E7E7",
    "higher_risk_edge": "#BDBDBD",
    "note_bg": "#FFFFFF",
    "note_edge": "#D2D2D2",
}

MODEL_ORDER = [
    "full_cvae",
    "conditional_gru_lm",
    "random_condition",
    "no_condition",
    "small_latent",
]

MODEL_LABELS = {
    "full_cvae": "OncoPep",
    "conditional_gru_lm": "GRU-cond",
    "random_condition": "Rand-cond",
    "no_condition": "GRU-uncond",
    "small_latent": "VAE-uncond",
    "train_ref": "Training reference",
}

MODEL_COLORS = {
    "full_cvae": PLOS["coral"],
    "conditional_gru_lm": PLOS["blue"],
    "random_condition": PLOS["mint"],
    "no_condition": PLOS["brown"],
    "small_latent": PLOS["charcoal"],
}

MODEL_EDGES = {
    "full_cvae": "#B94F42",
    "conditional_gru_lm": "#166F8A",
    "random_condition": "#78A886",
    "no_condition": "#8E5F39",
    "small_latent": "#4E4648",
}

MODEL_TAG_ALIASES = {
    "oncopep": "full_cvae",
    "oncoppep": "full_cvae",
    "full_cvae": "full_cvae",
    "cvae": "full_cvae",
    "conditional_vae": "full_cvae",
    "conditioned_vae": "full_cvae",
    "gru-cond": "conditional_gru_lm",
    "gru_cond": "conditional_gru_lm",
    "conditional_gru": "conditional_gru_lm",
    "conditional_gru_lm": "conditional_gru_lm",
    "rand-cond": "random_condition",
    "rand_cond": "random_condition",
    "random_condition": "random_condition",
    "random conditional": "random_condition",
    "gru-uncond": "no_condition",
    "gru_uncond": "no_condition",
    "unconditional_gru": "no_condition",
    "no_condition": "no_condition",
    "vae-uncond": "small_latent",
    "vae_uncond": "small_latent",
    "unconditional_vae": "small_latent",
    "small_latent": "small_latent",
}

AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DESCRIPTORS = ("length", "net_charge", "hydrophobicity", "entropy")
DEFAULT_TAIL_THRESHOLDS = (0.10, 0.15, 0.20, 0.30)

KD_SCALE = {
    "A": 1.8,
    "C": 2.5,
    "D": -3.5,
    "E": -3.5,
    "F": 2.8,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "N": -3.5,
    "P": -1.6,
    "Q": -3.5,
    "R": -4.5,
    "S": -0.8,
    "T": -0.7,
    "V": 4.2,
    "W": -0.9,
    "Y": -1.3,
}

COLUMN_ALIASES = {
    "model_tag": ["model_tag", "model", "generator", "model_name", "method"],
    "candidate_id": ["candidate_id", "peptide_id", "id", "seq_id", "sequence_id", "candidate"],
    "sequence": ["sequence", "peptide", "seq", "peptide_sequence"],
    "is_final_candidate": [
        "is_final_candidate",
        "final_selected",
        "selected_final",
        "selected",
        "is_selected",
        "final_candidate",
        "passed_final_audit",
        "passed_audit",
    ],
    "candidate_rank": ["candidate_rank", "rank", "final_rank", "selection_rank", "priority_rank"],
    "composite_score": ["composite_score", "score", "selection_score", "priority_score", "audit_score"],
    "exact_novel_vs_train": [
        "exact_novel_vs_train",
        "exact_novel",
        "exact_novelty",
        "novel_vs_train",
        "novelty_flag",
        "is_exact_novel",
        "novelty_exact",
    ],
    "exact_novel_vs_full": [
        "exact_novel_vs_full",
        "exact_novel_vs_full_corpus",
        "novel_vs_full",
        "exact_full_corpus_novelty",
    ],
    "nn_similarity_train": [
        "nn_similarity_train",
        "nearest_neighbor_similarity_train",
        "nearest_train_similarity",
        "nearest_train_jaccard",
        "smax_train",
        "nn_train",
        "similarity_to_train",
        "nearest_neighbor_similarity",
    ],
    "nn_train_sequence": ["nn_train_sequence", "nearest_train_sequence", "nearest_neighbor_train_sequence"],
    "length": ["length", "seq_length", "peptide_length", "model_ready_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "charge", "charge_pH7", "net_charge_ph7"],
    "hydrophobicity": [
        "hydrophobicity",
        "hydrophobicity_KD",
        "mean_hydrophobicity",
        "mean_kd_hydrophobicity",
        "mean_KD",
        "mean_kyte_doolittle",
        "kd_hydrophobicity",
        "mean_hydropathy",
    ],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy", "seq_entropy"],
    "plausibility_is_within_reference": [
        "plausibility_is_within_reference",
        "within_reference",
        "within_reference_range",
        "reference_supported",
    ],
}

TRAIN_ALIASES = {k: COLUMN_ALIASES[k] for k in ["sequence", "length", "net_charge", "hydrophobicity", "entropy"]}


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Step7Config:
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07"
    input_roots: Tuple[str, ...] = (
        "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07/source_data",
        "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables",
        "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/source_data",
        "/home/data3/Moe/nature_computational_peponco",
        "/mnt/data",
    )

    generated_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv"
    train_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv"
    final_file: Optional[str] = None

    generated_candidates: Tuple[str, ...] = (
        "step7_generated_final_audited.csv",
        "step7_generated_final.csv",
        "step7_generated_audited.csv",
        "generated_final_audited.csv",
        "SuppFig_step7_property_generated_final_v7_source_data.csv",
    )
    train_candidates: Tuple[str, ...] = (
        "step7_train_reference_annotated.csv",
        "train_reference_annotated.csv",
        "SuppFig_step7_property_train_reference_v7_source_data.csv",
    )
    final_candidates: Tuple[str, ...] = (
        "step7_final_candidates.csv",
        "step7_final_candidates_audit_table.csv",
        "final_candidates.csv",
        "final_oncopep_candidates.csv",
    )

    central_quantile_low: float = 0.05
    central_quantile_high: float = 0.95
    full_reference_quantile_low: float = 0.00
    full_reference_quantile_high: float = 1.00
    similarity_low_risk_threshold: float = 0.30
    tail_thresholds: Tuple[float, ...] = DEFAULT_TAIL_THRESHOLDS
    top_n_final: int = 12

    compute_missing_descriptors: bool = True
    histidine_charge: float = 0.10
    compute_nn_if_missing: bool = False
    max_nn_pairs: int = 3_000_000

    export_png: bool = True
    export_pdf: bool = True
    export_tiff: bool = True
    dpi_png: int = 300
    dpi_tiff: int = 600
    tiff_compression: str = "tiff_lzw"
    show_figures: bool = False

    font_family: str = "DejaVu Sans"
    base_font_size: float = 10.8
    axis_label_size: float = 12.0
    title_size: float = 14.0
    tick_size: float = 10.2
    legend_size: float = 10.5
    panel_label_size: float = 18.0

    seed: int = 42
    inventory_max_files: int = 500

    @property
    def root(self) -> Path:
        return Path(self.step7_root).expanduser().resolve()

    @property
    def main_figure_dir(self) -> Path:
        return self.root / "main_figure"

    @property
    def supplementary_dir(self) -> Path:
        return self.root / "supplementary_figures"

    @property
    def source_data_dir(self) -> Path:
        return self.root / "source_data"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"

    @property
    def code_dir(self) -> Path:
        return self.root / "code"


# =============================================================================
# General utilities
# =============================================================================

def ensure_output_tree(cfg: Step7Config) -> None:
    for path in [cfg.root, cfg.main_figure_dir, cfg.supplementary_dir, cfg.source_data_dir, cfg.reports_dir, cfg.code_dir]:
        path.mkdir(parents=True, exist_ok=True)


def set_plot_style(cfg: Step7Config) -> None:
    plt.rcParams.update(
        {
            "font.family": cfg.font_family,
            "font.size": cfg.base_font_size,
            "axes.titlesize": cfg.title_size,
            "axes.labelsize": cfg.axis_label_size,
            "xtick.labelsize": cfg.tick_size,
            "ytick.labelsize": cfg.tick_size,
            "legend.fontsize": cfg.legend_size,
            "figure.titlesize": cfg.title_size,
            "axes.facecolor": PLOS["white"],
            "figure.facecolor": PLOS["white"],
            "savefig.facecolor": PLOS["white"],
            "text.color": PLOS["dark"],
            "axes.labelcolor": PLOS["dark"],
            "axes.edgecolor": PLOS["dark"],
            "xtick.color": PLOS["dark"],
            "ytick.color": PLOS["dark"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def normalize_path(path_raw: Optional[str]) -> Optional[Path]:
    if not path_raw:
        return None
    text = str(path_raw).strip()
    if text.lower() in {"none", "null", ""}:
        return None
    return Path(text).expanduser().resolve()


def inventory_csv_files(roots: Sequence[str], max_files: int = 500) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    seen: set[str] = set()
    for priority, root_raw in enumerate(roots):
        root = Path(root_raw).expanduser()
        if not root.exists():
            rows.append({"root_priority": priority, "root": str(root), "exists": False, "path": None, "filename": None, "size_bytes": None})
            continue
        if root.is_file():
            file_iter = iter([root] if root.suffix.lower() == ".csv" else [])
        else:
            file_iter = root.rglob("*.csv")
        n_this = 0
        for file_path in file_iter:
            if n_this >= max_files:
                break
            try:
                key = str(file_path.resolve())
            except OSError:
                continue
            if key in seen:
                continue
            seen.add(key)
            n_this += 1
            try:
                size_bytes = file_path.stat().st_size
            except OSError:
                size_bytes = None
            rows.append({"root_priority": priority, "root": str(root), "exists": True, "path": key, "filename": file_path.name, "size_bytes": size_bytes})
    return pd.DataFrame(rows)


def discover_file(candidate_names: Sequence[str], roots: Sequence[str], direct_file: Optional[str], required: bool, label: str) -> Optional[Path]:
    direct = normalize_path(direct_file)
    if direct is not None:
        if direct.exists() and direct.is_file():
            return direct
        raise FileNotFoundError(f"Direct {label} path was provided but does not exist: {direct}")
    candidate_set = set(candidate_names)
    searched: List[str] = []
    for root_raw in roots:
        root = Path(root_raw).expanduser()
        searched.append(str(root))
        if not root.exists():
            continue
        if root.is_file() and root.name in candidate_set:
            return root.resolve()
        for name in candidate_names:
            direct_child = root / name
            if direct_child.exists() and direct_child.is_file():
                return direct_child.resolve()
        if root.is_dir():
            for p in root.rglob("*.csv"):
                if p.name in candidate_set:
                    return p.resolve()
    if required:
        raise FileNotFoundError(
            f"Could not find required {label}.\nAccepted filenames: {list(candidate_names)}\nSearched roots:\n" + "\n".join(searched)
        )
    return None


def sha256_file(path: Optional[Path]) -> Optional[str]:
    if path is None:
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_csv(df: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return str(path)


def dataframe_with_panel(df: pd.DataFrame, figure: str, panel: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "panel", panel)
    out.insert(0, "figure", figure)
    return out


def find_first_column(df: pd.DataFrame, aliases: Sequence[str], canonical_name: str, required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for alias in aliases:
        if alias in df.columns:
            return alias
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]
    if required:
        raise KeyError(f"Could not find column for {canonical_name}. Tried: {list(aliases)}. Available: {list(df.columns)}")
    return None


def maybe_rename_columns(df: pd.DataFrame, aliases: Dict[str, Sequence[str]], canonical_columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    rename_map: Dict[str, str] = {}
    for canonical in canonical_columns:
        if canonical in out.columns or canonical not in aliases:
            continue
        found = find_first_column(out, aliases[canonical], canonical, required=False)
        if found is not None and found != canonical:
            rename_map[found] = canonical
    return out.rename(columns=rename_map)


def normalize_model_tag_value(value: object) -> object:
    if pd.isna(value):
        return value
    text = str(value).strip()
    key = text.lower().replace("-", "_").replace(" ", "_")
    return MODEL_TAG_ALIASES.get(key, text)


def normalize_model_tags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model_tag" in out.columns:
        out["model_tag"] = out["model_tag"].map(normalize_model_tag_value)
    return out


def as_numeric_clean(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


def parse_boolish(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    text = series.astype(str).str.strip().str.lower()
    truthy = {"true", "1", "yes", "y", "selected", "final", "pass", "passed", "keep", "kept"}
    falsy = {"false", "0", "no", "n", "not_selected", "fail", "failed", "nan", "none", "", "remove"}
    return text.map(lambda x: True if x in truthy else (False if x in falsy else False))


def clean_sequence(seq: object) -> str:
    if pd.isna(seq):
        return ""
    return str(seq).strip().upper()


def valid_sequence(seq: object) -> bool:
    s = clean_sequence(seq)
    return len(s) > 0 and all(ch in AA20 for ch in s)


def peptide_length(seq: object) -> float:
    s = clean_sequence(seq)
    return float(len(s)) if s else np.nan


def peptide_net_charge(seq: object, histidine_charge: float = 0.10) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    return float(s.count("K") + s.count("R") + histidine_charge * s.count("H") - s.count("D") - s.count("E"))


def peptide_hydrophobicity(seq: object) -> float:
    s = clean_sequence(seq)
    vals = [KD_SCALE[ch] for ch in s if ch in KD_SCALE]
    if not vals:
        return np.nan
    return float(np.mean(vals))


def peptide_entropy(seq: object) -> float:
    s = clean_sequence(seq)
    if not s:
        return np.nan
    counts = np.array([s.count(aa) for aa in AA20], dtype=float)
    probs = counts[counts > 0] / len(s)
    return float(-(probs * np.log2(probs)).sum())


def normalized_levenshtein_similarity(a: str, b: str) -> float:
    a = clean_sequence(a)
    b = clean_sequence(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(current[j - 1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    dist = previous[-1]
    return float(1.0 - dist / max(len(a), len(b)))


def compute_nearest_similarity(sequences: Sequence[str], train_sequences: Sequence[str], max_pairs: int) -> Tuple[pd.Series, str]:
    n_pairs = len(sequences) * len(train_sequences)
    if n_pairs > max_pairs:
        return pd.Series([np.nan] * len(sequences), dtype=float), f"not_computed_pair_budget_exceeded:{n_pairs}>{max_pairs}"
    train_list = [clean_sequence(s) for s in train_sequences if clean_sequence(s)]
    values: List[float] = []
    for seq in sequences:
        s = clean_sequence(seq)
        if not s or not train_list:
            values.append(np.nan)
        else:
            values.append(max(normalized_levenshtein_similarity(s, t) for t in train_list))
    return pd.Series(values, dtype=float), "computed_normalized_levenshtein_similarity"


# =============================================================================
# Data preparation and audit logic
# =============================================================================

def add_or_compute_descriptors(df: pd.DataFrame, cfg: Step7Config, role: str, warnings_list: List[str]) -> pd.DataFrame:
    out = df.copy()
    if "sequence" not in out.columns:
        return out
    if cfg.compute_missing_descriptors:
        if "length" not in out.columns or out["length"].isna().all():
            out["length"] = out["sequence"].map(peptide_length)
            warnings_list.append(f"{role}: length was missing and computed from sequence.")
        if "net_charge" not in out.columns or out["net_charge"].isna().all():
            out["net_charge"] = out["sequence"].map(lambda s: peptide_net_charge(s, cfg.histidine_charge))
            warnings_list.append(f"{role}: net_charge was missing and approximated from residue counts.")
        if "hydrophobicity" not in out.columns or out["hydrophobicity"].isna().all():
            out["hydrophobicity"] = out["sequence"].map(peptide_hydrophobicity)
            warnings_list.append(f"{role}: hydrophobicity was missing and computed using mean Kyte-Doolittle scale.")
        if "entropy" not in out.columns or out["entropy"].isna().all():
            out["entropy"] = out["sequence"].map(peptide_entropy)
            warnings_list.append(f"{role}: entropy was missing and computed as Shannon entropy over AA20.")
    for col in DESCRIPTORS:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])
    return out


def prepare_generated_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, COLUMN_ALIASES, COLUMN_ALIASES.keys())
    if "sequence" not in out.columns:
        raise KeyError(f"Generated table is missing sequence column. Available columns: {list(raw.columns)}")
    if "model_tag" not in out.columns:
        out["model_tag"] = "full_cvae"
        warnings_list.append("generated: model_tag missing; all rows assigned to full_cvae/OncoPep.")
    out = normalize_model_tags(out)
    out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "generated", warnings_list)

    for col in ["nn_similarity_train", "candidate_rank", "composite_score"]:
        if col in out.columns:
            out[col] = as_numeric_clean(out[col])

    if "candidate_id" not in out.columns:
        out["candidate_id"] = [f"pep_{i+1:05d}" for i in range(len(out))]
        warnings_list.append("generated: candidate_id was missing and sequential IDs were assigned.")
    if "is_final_candidate" in out.columns:
        out["is_final_candidate"] = parse_boolish(out["is_final_candidate"])
    else:
        out["is_final_candidate"] = False

    out["valid_aa_sequence"] = out["sequence"].map(valid_sequence)
    out["duplicate_sequence_within_input"] = out.duplicated("sequence", keep=False)
    out["recognized_model"] = out["model_tag"].isin(MODEL_ORDER)
    return out


def prepare_train_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = maybe_rename_columns(raw, TRAIN_ALIASES, TRAIN_ALIASES.keys())
    if "sequence" in out.columns:
        out["sequence"] = out["sequence"].map(clean_sequence)
    out = add_or_compute_descriptors(out, cfg, "train_reference", warnings_list)
    missing = [col for col in DESCRIPTORS if col not in out.columns]
    if missing:
        raise KeyError(f"Training-reference table missing descriptors: {missing}. Available columns: {list(raw.columns)}")
    for col in DESCRIPTORS:
        out[col] = as_numeric_clean(out[col])
    out["model_tag"] = "train_ref"
    return out


def compute_reference_ranges(train_df: pd.DataFrame, cfg: Step7Config) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for descriptor in DESCRIPTORS:
        vals = train_df[descriptor].dropna().to_numpy(float)
        if len(vals) == 0:
            raise ValueError(f"Training descriptor {descriptor} has no finite values.")
        rows.append(
            {
                "descriptor": descriptor,
                "n_train": int(len(vals)),
                "min": float(np.min(vals)),
                "q01": float(np.quantile(vals, 0.01)),
                "q05": float(np.quantile(vals, 0.05)),
                "q25": float(np.quantile(vals, 0.25)),
                "median": float(np.quantile(vals, 0.50)),
                "q75": float(np.quantile(vals, 0.75)),
                "q95": float(np.quantile(vals, 0.95)),
                "q99": float(np.quantile(vals, 0.99)),
                "max": float(np.max(vals)),
                "central_low": float(np.quantile(vals, cfg.central_quantile_low)),
                "central_high": float(np.quantile(vals, cfg.central_quantile_high)),
                "central_interval_definition": f"training {int(cfg.central_quantile_low*100)}-{int(cfg.central_quantile_high*100)}% interval",
                "full_support_low": float(np.quantile(vals, cfg.full_reference_quantile_low)),
                "full_support_high": float(np.quantile(vals, cfg.full_reference_quantile_high)),
                "full_support_definition": f"training quantile [{cfg.full_reference_quantile_low}, {cfg.full_reference_quantile_high}]",
            }
        )
    return pd.DataFrame(rows)


def add_computed_audit_columns(generated_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = generated_df.copy()
    if "sequence" in train_df.columns:
        train_sequences = set(train_df["sequence"].dropna().astype(str).str.strip().str.upper())
        out["computed_exact_novel_vs_train"] = ~out["sequence"].isin(train_sequences)
    else:
        out["computed_exact_novel_vs_train"] = np.nan
        warnings_list.append("Exact novelty could not be computed because training sequence column is absent.")

    if "exact_novel_vs_train" in out.columns:
        out["exact_novel_vs_train"] = parse_boolish(out["exact_novel_vs_train"])
        out["audit_exact_novel_vs_train"] = out["exact_novel_vs_train"]
    else:
        out["audit_exact_novel_vs_train"] = out["computed_exact_novel_vs_train"]

    if "exact_novel_vs_full" in out.columns:
        out["exact_novel_vs_full"] = parse_boolish(out["exact_novel_vs_full"])

    if "nn_similarity_train" not in out.columns or out["nn_similarity_train"].isna().all():
        if cfg.compute_nn_if_missing and "sequence" in train_df.columns:
            nn_values, status = compute_nearest_similarity(out["sequence"].tolist(), train_df["sequence"].dropna().tolist(), cfg.max_nn_pairs)
            out["nn_similarity_train"] = nn_values
            out["nn_similarity_train_source"] = status
            warnings_list.append(f"generated: nn_similarity_train was missing; status={status}.")
        else:
            out["nn_similarity_train"] = np.nan
            out["nn_similarity_train_source"] = "missing"
            warnings_list.append("generated: nn_similarity_train missing and not computed.")
    else:
        out["nn_similarity_train"] = as_numeric_clean(out["nn_similarity_train"])
        out["nn_similarity_train_source"] = "input_column"

    for _, row in reference_ranges.iterrows():
        descriptor = str(row["descriptor"])
        out[f"within_full_ref_{descriptor}"] = out[descriptor].between(float(row["full_support_low"]), float(row["full_support_high"]), inclusive="both")
        out[f"within_central_ref_{descriptor}"] = out[descriptor].between(float(row["central_low"]), float(row["central_high"]), inclusive="both")

    full_cols = [f"within_full_ref_{d}" for d in DESCRIPTORS]
    central_cols = [f"within_central_ref_{d}" for d in DESCRIPTORS]
    out["audit_reference_supported_full_range"] = out[full_cols].all(axis=1)
    out["audit_reference_supported_central_interval"] = out[central_cols].all(axis=1)

    if "plausibility_is_within_reference" in out.columns:
        input_ref = parse_boolish(out["plausibility_is_within_reference"])
        # Preserve the source flag while using computed full range for consistency checks.
        out["input_plausibility_is_within_reference"] = input_ref

    out["audit_low_similarity"] = out["nn_similarity_train"] < cfg.similarity_low_risk_threshold
    out["audit_valid_sequence"] = out["valid_aa_sequence"].astype(bool)
    out["audit_nonduplicate_sequence"] = ~out["duplicate_sequence_within_input"].astype(bool)

    audit_cols = [
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "audit_reference_supported_full_range",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
    ]
    out["audit_pass_computed"] = out[audit_cols].fillna(False).all(axis=1)
    return out


def prepare_final_file_df(raw: pd.DataFrame, cfg: Step7Config, warnings_list: List[str]) -> pd.DataFrame:
    out = prepare_generated_df(raw, cfg, warnings_list)
    out["is_final_candidate"] = True
    return out


def infer_final_candidates(audited_df: pd.DataFrame, explicit_final_df: Optional[pd.DataFrame], cfg: Step7Config, warnings_list: List[str]) -> Tuple[pd.DataFrame, str, bool]:
    if explicit_final_df is not None and not explicit_final_df.empty:
        final_sequences = set(explicit_final_df["sequence"].dropna().astype(str))
        matched = audited_df[audited_df["sequence"].isin(final_sequences)].copy()
        if not matched.empty:
            return matched, "explicit separate final-candidate file matched by sequence", True
        return explicit_final_df.copy(), "explicit separate final-candidate file", True

    if "is_final_candidate" in audited_df.columns and audited_df["is_final_candidate"].any():
        return audited_df.loc[audited_df["is_final_candidate"]].copy(), "explicit final-candidate flag", True

    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()
        warnings_list.append("No full_cvae rows found; final-candidate fallback used all generated rows.")

    if "candidate_rank" in oncopep.columns and oncopep["candidate_rank"].notna().any():
        return oncopep.sort_values("candidate_rank", ascending=True).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by candidate_rank", False
    if "composite_score" in oncopep.columns and oncopep["composite_score"].notna().any():
        return oncopep.sort_values("composite_score", ascending=False).head(cfg.top_n_final).copy(), f"fallback: top {cfg.top_n_final} rows by composite_score", False

    passing = oncopep.loc[oncopep["audit_pass_computed"]].copy()
    if not passing.empty:
        return passing.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(passing))} OncoPep rows passing computed audit", False

    return oncopep.head(cfg.top_n_final).copy(), f"fallback: first {min(cfg.top_n_final, len(oncopep))} OncoPep rows", False


def build_sequential_funnel_table(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config, final_rule: str) -> pd.DataFrame:
    oncopep = audited_df.loc[audited_df["model_tag"] == "full_cvae"].copy()
    if oncopep.empty:
        oncopep = audited_df.copy()
    current = pd.Series(True, index=oncopep.index)
    stages = []

    def add_stage(stage: str, criterion_col: Optional[str], criterion_description: str) -> None:
        nonlocal current
        if criterion_col is not None:
            current = current & oncopep[criterion_col].fillna(False).astype(bool)
        count = int(current.sum())
        previous_count = int(stages[-1]["n_surviving"] if stages else len(oncopep))
        initial_count = int(len(oncopep))
        stages.append(
            {
                "stage_order": len(stages) + 1,
                "stage": stage,
                "criterion_column": criterion_col if criterion_col is not None else "initial_scope",
                "criterion_description": criterion_description,
                "n_surviving": count,
                "retained_from_previous_percent": float(100.0 * count / previous_count) if previous_count else np.nan,
                "retained_from_initial_percent": float(100.0 * count / initial_count) if initial_count else np.nan,
                "denominator_type": "sequential_survivors_from_previous_stage",
                "count_type": "true_sequential_funnel",
            }
        )

    add_stage("OncoPep evaluated outputs", None, "All finite OncoPep/full CVAE rows in the generated audit table.")
    add_stage("Valid amino-acid sequences", "audit_valid_sequence", "Sequence contains only standard AA20 residues.")
    add_stage("Exact-novel vs training", "audit_exact_novel_vs_train", "No exact sequence match to the training set.")
    add_stage("Reference-supported descriptors", "audit_reference_supported_full_range", "Length, net charge, hydrophobicity, and entropy within full training-reference support range.")
    add_stage(f"NN similarity < {cfg.similarity_low_risk_threshold:.2f}", "audit_low_similarity", "Nearest-neighbor similarity to training set below the low-risk audit threshold.")

    final_sequences = set(final_df["sequence"].dropna().astype(str)) if "sequence" in final_df.columns else set()
    final_within_current = int((current & oncopep["sequence"].isin(final_sequences)).sum()) if final_sequences else int(len(final_df))
    previous_count = int(stages[-1]["n_surviving"] if stages else len(oncopep))
    initial_count = int(len(oncopep))
    stages.append(
        {
            "stage_order": len(stages) + 1,
            "stage": "Final candidates",
            "criterion_column": "final_candidate_selection",
            "criterion_description": f"Final candidates from rule: {final_rule}",
            "n_surviving": int(len(final_df)),
            "n_final_candidates_also_surviving_previous_filters": final_within_current,
            "retained_from_previous_percent": float(100.0 * len(final_df) / previous_count) if previous_count else np.nan,
            "retained_from_initial_percent": float(100.0 * len(final_df) / initial_count) if initial_count else np.nan,
            "denominator_type": "final_candidate_count_over_previous_sequential_survivors",
            "count_type": "final_selection_summary",
        }
    )
    return pd.DataFrame(stages)


def load_inputs(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    warnings_list: List[str] = []
    roots = tuple(str(Path(r).expanduser()) for r in cfg.input_roots)
    inventory = inventory_csv_files(roots, cfg.inventory_max_files)
    if not inventory.empty:
        write_csv(inventory, cfg.reports_dir / "step7_input_inventory.csv")

    generated_path = discover_file(cfg.generated_candidates, roots, cfg.generated_file, required=True, label="generated/final audited table")
    train_path = discover_file(cfg.train_candidates, roots, cfg.train_file, required=True, label="training-reference table")
    # Final-candidate files are intentionally not auto-discovered when final_file is None.
    # Auto-discovery can accidentally reuse stale outputs from earlier Step 7 test runs.
    # For manuscript use, provide a real frozen final-candidate table via --final-file.
    final_path = discover_file(cfg.final_candidates, roots, cfg.final_file, required=False, label="separate final-candidate table") if cfg.final_file else None

    assert generated_path is not None and train_path is not None
    generated_raw = pd.read_csv(generated_path)
    train_raw = pd.read_csv(train_path)
    final_raw = pd.read_csv(final_path) if final_path is not None else None

    generated_df = prepare_generated_df(generated_raw, cfg, warnings_list)
    train_df = prepare_train_df(train_raw, cfg, warnings_list)
    reference_ranges = compute_reference_ranges(train_df, cfg)
    audited_df = add_computed_audit_columns(generated_df, train_df, reference_ranges, cfg, warnings_list)

    explicit_final_df = None
    if final_raw is not None:
        explicit_final_df = prepare_final_file_df(final_raw, cfg, warnings_list)
        explicit_final_df = add_computed_audit_columns(explicit_final_df, train_df, reference_ranges, cfg, warnings_list)

    final_df, final_rule, final_is_explicit = infer_final_candidates(audited_df, explicit_final_df, cfg, warnings_list)
    if not final_is_explicit:
        warnings_list.append("Final candidates were selected by fallback logic. Provide a frozen final-candidate file/flag before manuscript writing.")

    # Mark final rows in the full audited table by sequence/candidate_id where possible.
    audited_df = audited_df.copy()
    audited_df["is_final_candidate"] = False
    if "sequence" in final_df.columns:
        audited_df.loc[audited_df["sequence"].isin(set(final_df["sequence"])), "is_final_candidate"] = True

    funnel_df = build_sequential_funnel_table(audited_df, final_df, cfg, final_rule)

    inputs = {
        "generated_path": str(generated_path),
        "train_path": str(train_path),
        "final_path": str(final_path) if final_path else None,
        "input_inventory": str(cfg.reports_dir / "step7_input_inventory.csv"),
        "generated_sha256": sha256_file(generated_path),
        "train_sha256": sha256_file(train_path),
        "final_sha256": sha256_file(final_path),
        "final_candidate_rule": final_rule,
        "final_candidate_selection_is_explicit": final_is_explicit,
        "warnings_from_preparation": warnings_list,
        "central_interval_definition": f"training {int(cfg.central_quantile_low*100)}-{int(cfg.central_quantile_high*100)}% interval",
        "full_reference_support_definition": f"training quantile [{cfg.full_reference_quantile_low}, {cfg.full_reference_quantile_high}]",
        "prohibited_graph_types_used": False,
    }

    return {
        "generated_df": generated_df,
        "train_df": train_df,
        "reference_ranges": reference_ranges,
        "audited_df": audited_df,
        "final_df": final_df,
        "funnel_df": funnel_df,
        "inventory": inventory,
        "inputs": inputs,
    }


# =============================================================================
# Plotting utilities
# =============================================================================

def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.75)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(0.9)
        ax.spines[side].set_color(PLOS["dark"])
    ax.tick_params(axis="both", width=0.75, length=3.5)


def style_twin_axis(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_linewidth(0.8)
    ax.spines["right"].set_color("#666666")
    ax.tick_params(axis="y", width=0.7, length=3.0, colors=PLOS["dark"])
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))


def add_panel_label(ax: plt.Axes, label: str, cfg: Step7Config, x: float = -0.08, y: float = 1.035) -> None:
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=cfg.panel_label_size, fontweight="bold", color=PLOS["dark"], clip_on=False)


def save_figure(fig: plt.Figure, out_base: Path, cfg: Step7Config) -> List[str]:
    outputs: List[str] = []
    if cfg.export_pdf:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        outputs.append(str(p))
    if cfg.export_png:
        p = out_base.with_suffix(".png")
        fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_png)
        outputs.append(str(p))
    if cfg.export_tiff:
        p = out_base.with_suffix(".tiff")
        try:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff, pil_kwargs={"compression": cfg.tiff_compression})
        except TypeError:
            fig.savefig(p, bbox_inches="tight", dpi=cfg.dpi_tiff)
        outputs.append(str(p))
    if cfg.show_figures:
        plt.show()
    else:
        plt.close(fig)
    return outputs


def descriptor_label(descriptor: str) -> str:
    return {
        "length": "Peptide length (residues)",
        "net_charge": "Net charge at pH 7",
        "hydrophobicity": "Mean hydrophobicity",
        "entropy": "Shannon entropy",
    }[descriptor]


def descriptor_title(descriptor: str) -> str:
    return {
        "length": "Length support",
        "net_charge": "Net-charge support",
        "hydrophobicity": "Hydrophobicity support",
        "entropy": "Entropy support",
    }[descriptor]


def choose_bins(values: np.ndarray, descriptor: str) -> np.ndarray:
    vals = values[np.isfinite(values)]
    if len(vals) == 0:
        return np.linspace(0, 1, 11)
    lo = float(np.min(vals))
    hi = float(np.max(vals))
    if descriptor == "length":
        lo = math.floor(lo) - 0.5
        hi = math.ceil(hi) + 0.5
        step = max(1, int(math.ceil((hi - lo) / 32)))
        return np.arange(lo, hi + step, step)
    if descriptor == "net_charge":
        lo = math.floor(lo) - 0.5
        hi = math.ceil(hi) + 0.5
        step = max(1, int(math.ceil((hi - lo) / 34)))
        return np.arange(lo, hi + step, step)
    # For continuous descriptors, use a controlled bin count.
    return np.linspace(lo, hi, 34)


# =============================================================================
# Figure builders
# =============================================================================

def build_s12_descriptor_support(final_df: pd.DataFrame, train_df: pd.DataFrame, reference_ranges: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    """New S12 Fig: descriptor support. No candidate-level dots are used."""
    fig = plt.figure(figsize=(14.4, 9.0))
    gs = GridSpec(2, 2, figure=fig)
    outputs: List[str] = []
    all_panel_frames: List[pd.DataFrame] = []
    specs = [("length", "A"), ("net_charge", "B"), ("hydrophobicity", "C"), ("entropy", "D")]

    for idx, (descriptor, panel) in enumerate(specs):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        ax2 = ax.twinx()
        train_vals = train_df[descriptor].dropna().to_numpy(float)
        cand_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])
        combined_for_bins = np.concatenate([train_vals, cand_vals]) if len(cand_vals) else train_vals
        bins = choose_bins(combined_for_bins, descriptor)

        train_counts, train_edges = np.histogram(train_vals, bins=bins, density=True)
        train_centers = (train_edges[:-1] + train_edges[1:]) / 2
        train_widths = np.diff(train_edges)
        ax.bar(
            train_centers,
            train_counts,
            width=train_widths * 0.92,
            color=STYLE["train_fill"],
            edgecolor=STYLE["train_edge"],
            linewidth=0.45,
            alpha=0.74,
            zorder=2,
            align="center",
        )

        rr = reference_ranges.loc[reference_ranges["descriptor"] == descriptor].iloc[0]
        c_low = float(rr["central_low"])
        c_high = float(rr["central_high"])
        ax.axvspan(c_low, c_high, color=STYLE["interval_fill"], alpha=0.54, zorder=1)

        cand_counts, cand_edges = np.histogram(cand_vals, bins=bins)
        cand_centers = (cand_edges[:-1] + cand_edges[1:]) / 2
        cand_widths = np.diff(cand_edges)
        ax2.bar(
            cand_centers,
            cand_counts,
            width=cand_widths * 0.44,
            color=STYLE["candidate_fill"],
            edgecolor=STYLE["candidate_edge"],
            linewidth=0.65,
            alpha=0.78,
            zorder=4,
            align="center",
        )
        ax2.set_ylim(0, max(1, int(cand_counts.max()) + 1) if len(cand_counts) else 1)
        ax2.set_ylabel("Final-candidate count")
        style_twin_axis(ax2)

        within = int(((cand_vals >= c_low) & (cand_vals <= c_high)).sum()) if len(cand_vals) else 0
        total = int(len(cand_vals))
        ax.text(
            0.98,
            0.95,
            f"Within 5–95%: {within}/{total}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9.4,
            bbox=dict(boxstyle="round,pad=0.22", fc=STYLE["note_bg"], ec=STYLE["note_edge"], lw=0.7, alpha=0.96),
        )
        ax.set_title(descriptor_title(descriptor), pad=9)
        ax.set_xlabel(descriptor_label(descriptor))
        ax.set_ylabel("Training-reference density")
        ax.set_xlim(float(np.min(bins)), float(np.max(bins)))
        style_axis(ax, "y")
        add_panel_label(ax, panel, cfg)

        panel_rows = []
        for i in range(len(train_counts)):
            panel_rows.append(
                {
                    "descriptor": descriptor,
                    "bin_left": float(train_edges[i]),
                    "bin_right": float(train_edges[i + 1]),
                    "bin_center": float(train_centers[i]),
                    "training_density": float(train_counts[i]),
                    "final_candidate_count": int(cand_counts[i]) if i < len(cand_counts) else 0,
                    "bin_rule": "left-closed/right-open except final bin includes right edge",
                    "central_interval_low": c_low,
                    "central_interval_high": c_high,
                    "central_interval_definition": "training 5-95% interval",
                    "n_final_candidates": total,
                    "n_final_within_central_interval": within,
                }
            )
        panel_df = pd.DataFrame(panel_rows)
        outputs.append(write_csv(panel_df, cfg.source_data_dir / f"Supplementary_Figure_S12_panel_{panel.lower()}_source_data.csv"))
        all_panel_frames.append(dataframe_with_panel(panel_df, "Supplementary Figure S12", panel))

    handles = [
        Patch(facecolor=STYLE["train_fill"], edgecolor=STYLE["train_edge"], label="Training reference"),
        Patch(facecolor=STYLE["interval_fill"], edgecolor="none", label="Central reference interval (5–95%)"),
        Patch(facecolor=STYLE["candidate_fill"], edgecolor=STYLE["candidate_edge"], label="Final-candidate bin count"),
    ]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=3, frameon=True, columnspacing=1.35, handletextpad=0.55)
    fig.subplots_adjust(left=0.075, right=0.94, top=0.93, bottom=0.14, wspace=0.26, hspace=0.34)
    outputs.append(write_csv(pd.concat(all_panel_frames, ignore_index=True, sort=False), cfg.source_data_dir / "Supplementary_Figure_S12_descriptor_support_source_data_all_panels.csv"))
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned", cfg)
    return figs, outputs


def build_s13_similarity_tail(audited_df: pd.DataFrame, final_df: pd.DataFrame, cfg: Step7Config) -> Tuple[List[str], List[str]]:
    """New S13 Fig: similarity-tail risk. No dots, lollipops, or lines are used."""
    fig = plt.figure(figsize=(14.2, 6.2))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[0.86, 1.14])
    outputs: List[str] = []

    # Panel A: final-candidate binned similarity count.
    ax_a = fig.add_subplot(gs[0, 0])
    vals = final_df["nn_similarity_train"].dropna().to_numpy(float) if "nn_similarity_train" in final_df.columns else np.array([])
    bin_defs = [
        ("<0.10", -np.inf, 0.10, "<", True),
        ("0.10–0.15", 0.10, 0.15, "[left,right)", True),
        ("0.15–0.20", 0.15, 0.20, "[left,right)", True),
        ("0.20–0.30", 0.20, 0.30, "[left,right)", True),
        ("≥0.30", 0.30, np.inf, ">=", False),
    ]
    rows = []
    for label, left, right, rule, low_risk_bin in bin_defs:
        if math.isinf(left) and left < 0:
            mask = vals < right
        elif math.isinf(right):
            mask = vals >= left
        else:
            mask = (vals >= left) & (vals < right)
        rows.append(
            {
                "similarity_bin": label,
                "bin_left": None if math.isinf(left) else float(left),
                "bin_right": None if math.isinf(right) else float(right),
                "inclusive_rule": rule,
                "low_risk_bin_under_0p30": bool(low_risk_bin),
                "final_candidate_count": int(mask.sum()),
                "n_final_candidates_with_finite_nn_similarity": int(len(vals)),
                "low_risk_threshold": cfg.similarity_low_risk_threshold,
                "nn_similarity_denominator": "training set nearest neighbor",
            }
        )
    bin_df = pd.DataFrame(rows)
    x = np.arange(len(bin_df))
    colors = [STYLE["low_risk_fill"] if r else STYLE["higher_risk_fill"] for r in bin_df["low_risk_bin_under_0p30"]]
    edges = ["#78A886" if r else STYLE["higher_risk_edge"] for r in bin_df["low_risk_bin_under_0p30"]]
    bars = ax_a.bar(x, bin_df["final_candidate_count"], color=colors, edgecolor=edges, linewidth=0.85, width=0.72, zorder=3)
    for bar, count in zip(bars, bin_df["final_candidate_count"]):
        ax_a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.10, f"{int(count)}", ha="center", va="bottom", fontsize=10.0)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels(bin_df["similarity_bin"], rotation=32, ha="right")
    ax_a.set_ylabel("Final-candidate count")
    ax_a.set_xlabel("Nearest-neighbor similarity bin")
    ax_a.set_title("Final-candidate NN-similarity bins", pad=9)
    ax_a.set_ylim(0, max(1, int(bin_df["final_candidate_count"].max()) + 2))
    ax_a.text(
        0.98,
        0.94,
        f"n = {len(vals)}; low-risk < {cfg.similarity_low_risk_threshold:.2f}",
        transform=ax_a.transAxes,
        ha="right",
        va="top",
        fontsize=9.5,
        bbox=dict(boxstyle="round,pad=0.22", fc=STYLE["note_bg"], ec=STYLE["note_edge"], lw=0.7, alpha=0.96),
    )
    style_axis(ax_a, "y")
    add_panel_label(ax_a, "A", cfg)
    outputs.append(write_csv(bin_df, cfg.source_data_dir / "Supplementary_Figure_S13_panel_a_source_data.csv"))
    outputs.append(write_csv(bin_df, cfg.source_data_dir / "step7_similarity_bin_counts.csv"))

    # Panel B: contextual generator-level tail-risk reference.
    ax_b = fig.add_subplot(gs[0, 1])
    thresholds = list(cfg.tail_thresholds)
    model_tags = [m for m in MODEL_ORDER if m in set(audited_df["model_tag"])]
    x_base = np.arange(len(thresholds), dtype=float)
    bar_w = 0.13
    gap = 0.025
    tail_rows: List[Dict[str, object]] = []
    for i, model in enumerate(model_tags):
        vals_m = audited_df.loc[audited_df["model_tag"] == model, "nn_similarity_train"].dropna().to_numpy(float)
        denominator = int(len(vals_m))
        fracs = []
        for thr in thresholds:
            numerator = int(np.sum(vals_m >= thr)) if denominator else 0
            frac = float(numerator / denominator) if denominator else np.nan
            fracs.append(frac)
            tail_rows.append(
                {
                    "model_tag": model,
                    "model_label": MODEL_LABELS[model],
                    "threshold": thr,
                    "comparison": ">= threshold",
                    "numerator_outputs_at_or_above_threshold": numerator,
                    "denominator_finite_nn_evaluated_outputs": denominator,
                    "fraction_at_or_above_threshold": frac,
                    "denominator_definition": "per-model outputs with finite nearest-neighbor similarity to training set",
                    "panel_role": "contextual generator-level tail-risk reference, not a new Step 5 benchmark",
                }
            )
        offset = (i - (len(model_tags) - 1) / 2.0) * (bar_w + gap)
        ax_b.bar(
            x_base + offset,
            fracs,
            width=bar_w,
            color=MODEL_COLORS[model],
            edgecolor=MODEL_EDGES[model],
            linewidth=0.75,
            alpha=0.92,
            zorder=3,
            label=MODEL_LABELS[model],
        )
    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels([f"NN ≥{thr:.2f}" for thr in thresholds])
    ax_b.set_ylabel("Fraction ≥ threshold")
    ax_b.set_ylim(0.0, 1.02)
    ax_b.set_title("Generator-context tail-risk reference", pad=9)
    style_axis(ax_b, "y")
    add_panel_label(ax_b, "B", cfg)

    panel_b = pd.DataFrame(tail_rows)
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "Supplementary_Figure_S13_panel_b_source_data.csv"))
    outputs.append(write_csv(panel_b, cfg.source_data_dir / "step7_similarity_tail_risk_summary.csv"))

    all_panels = pd.concat(
        [
            dataframe_with_panel(bin_df, "Supplementary Figure S13", "A"),
            dataframe_with_panel(panel_b, "Supplementary Figure S13", "B"),
        ],
        ignore_index=True,
        sort=False,
    )
    outputs.append(write_csv(all_panels, cfg.source_data_dir / "Supplementary_Figure_S13_similarity_tail_risk_source_data_all_panels.csv"))

    fig.legend(loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=len(model_tags), frameon=True, columnspacing=1.15, handletextpad=0.45)
    fig.subplots_adjust(left=0.07, right=0.985, top=0.88, bottom=0.22, wspace=0.23)
    figs = save_figure(fig, cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned", cfg)
    return figs, outputs


# =============================================================================
# Source data, reports, manifest
# =============================================================================

def build_candidate_audit_tables(audited_df: pd.DataFrame, final_df: pd.DataFrame, funnel_df: pd.DataFrame, reference_ranges: pd.DataFrame, inputs: Dict[str, object], cfg: Step7Config) -> Tuple[List[str], Dict[str, object]]:
    outputs: List[str] = []

    keep_cols = [
        "candidate_id",
        "sequence",
        "model_tag",
        "length",
        "net_charge",
        "hydrophobicity",
        "entropy",
        "nn_similarity_train",
        "nn_train_sequence",
        "audit_valid_sequence",
        "audit_exact_novel_vs_train",
        "exact_novel_vs_full",
        "audit_reference_supported_full_range",
        "audit_reference_supported_central_interval",
        "audit_low_similarity",
        "audit_nonduplicate_sequence",
        "audit_pass_computed",
        "candidate_rank",
        "composite_score",
    ]
    available_cols = [c for c in keep_cols if c in final_df.columns]
    final_table = final_df[available_cols].copy()
    if "candidate_rank" not in final_table.columns:
        final_table.insert(0, "final_table_rank", np.arange(1, len(final_table) + 1))
    final_table["final_candidate_rule"] = inputs.get("final_candidate_rule")
    final_table["selection_is_explicit"] = inputs.get("final_candidate_selection_is_explicit")
    final_table["claim_boundary"] = "computational audit only; not experimental validation"

    outputs.append(write_csv(final_table, cfg.source_data_dir / "step7_final_candidates_audit_table.csv"))
    outputs.append(write_csv(final_table, cfg.reports_dir / "step7_candidate_audit_funnel_and_matrix_table.csv"))
    outputs.append(write_csv(funnel_df, cfg.reports_dir / "step7_candidate_audit_summary_table.csv"))
    outputs.append(write_csv(funnel_df, cfg.source_data_dir / "step7_candidate_funnel_counts.csv"))
    outputs.append(write_csv(audited_df, cfg.source_data_dir / "step7_all_generated_audited_with_computed_flags.csv"))
    outputs.append(write_csv(reference_ranges, cfg.source_data_dir / "step7_training_reference_descriptor_ranges.csv"))

    descriptor_rows: List[Dict[str, object]] = []
    for _, rr in reference_ranges.iterrows():
        descriptor = str(rr["descriptor"])
        cand_vals = final_df[descriptor].dropna().to_numpy(float) if descriptor in final_df.columns else np.array([])
        c_low = float(rr["central_low"])
        c_high = float(rr["central_high"])
        f_low = float(rr["full_support_low"])
        f_high = float(rr["full_support_high"])
        descriptor_rows.append(
            {
                "descriptor": descriptor,
                "n_final_candidates": int(len(cand_vals)),
                "candidate_min": float(np.min(cand_vals)) if len(cand_vals) else np.nan,
                "candidate_median": float(np.median(cand_vals)) if len(cand_vals) else np.nan,
                "candidate_max": float(np.max(cand_vals)) if len(cand_vals) else np.nan,
                "central_interval_low": c_low,
                "central_interval_high": c_high,
                "central_interval_definition": "training 5-95% interval",
                "n_within_central_interval": int(((cand_vals >= c_low) & (cand_vals <= c_high)).sum()) if len(cand_vals) else 0,
                "full_reference_support_low": f_low,
                "full_reference_support_high": f_high,
                "full_reference_support_definition": "training min-max support range unless quantiles are changed in config",
                "n_within_full_reference_support": int(((cand_vals >= f_low) & (cand_vals <= f_high)).sum()) if len(cand_vals) else 0,
            }
        )
    descriptor_summary = pd.DataFrame(descriptor_rows)
    outputs.append(write_csv(descriptor_summary, cfg.source_data_dir / "step7_descriptor_support_summary.csv"))

    summary = {
        "n_final_candidates": int(len(final_df)),
        "final_candidate_rule": inputs.get("final_candidate_rule"),
        "selection_is_explicit": inputs.get("final_candidate_selection_is_explicit"),
        "n_final_audit_pass": int(final_df["audit_pass_computed"].sum()) if "audit_pass_computed" in final_df.columns else None,
        "n_final_low_similarity": int(final_df["audit_low_similarity"].sum()) if "audit_low_similarity" in final_df.columns else None,
        "n_final_central_interval_all_descriptors": int(final_df["audit_reference_supported_central_interval"].sum()) if "audit_reference_supported_central_interval" in final_df.columns else None,
    }

    md_lines = [
        "# Step 7 candidate audit summary",
        "",
        f"Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "## Decision",
        "",
        "The candidate-audit funnel and binary matrix are reported as tables/reports rather than as a supplementary figure. This avoids a weak table-like figure and keeps the Step 7 figure package focused on visual evidence that benefits from plotting.",
        "",
        "## Final candidate selection",
        "",
        f"- Rule: `{inputs.get('final_candidate_rule')}`",
        f"- Explicit final-candidate file/flag: `{inputs.get('final_candidate_selection_is_explicit')}`",
        f"- Number of final candidates: `{len(final_df)}`",
        "",
        "## Claim boundary",
        "",
        "These outputs support computational auditing and screening-readiness prioritization only. They do not establish anticancer activity, receptor binding, selectivity, toxicity, mechanism, therapeutic efficacy, clinical utility, complete absence of memorization, or experimental validation.",
        "",
        "## Tables generated",
        "",
        "- reports/step7_candidate_audit_summary_table.csv",
        "- reports/step7_candidate_audit_funnel_and_matrix_table.csv",
        "- source_data/step7_final_candidates_audit_table.csv",
    ]
    summary_md = cfg.reports_dir / "step7_candidate_audit_summary.md"
    summary_md.write_text("\n".join(md_lines), encoding="utf-8")
    outputs.append(str(summary_md))
    return outputs, summary


def write_panel_mapping(cfg: Step7Config) -> str:
    mapping = pd.DataFrame(
        [
            {"figure": "S12 Fig", "panel": "A", "title": "Length support", "source_data": "source_data/Supplementary_Figure_S12_panel_a_source_data.csv"},
            {"figure": "S12 Fig", "panel": "B", "title": "Net-charge support", "source_data": "source_data/Supplementary_Figure_S12_panel_b_source_data.csv"},
            {"figure": "S12 Fig", "panel": "C", "title": "Hydrophobicity support", "source_data": "source_data/Supplementary_Figure_S12_panel_c_source_data.csv"},
            {"figure": "S12 Fig", "panel": "D", "title": "Entropy support", "source_data": "source_data/Supplementary_Figure_S12_panel_d_source_data.csv"},
            {"figure": "S13 Fig", "panel": "A", "title": "Final-candidate nearest-neighbor similarity bins", "source_data": "source_data/Supplementary_Figure_S13_panel_a_source_data.csv"},
            {"figure": "S13 Fig", "panel": "B", "title": "Generator-context tail-risk reference", "source_data": "source_data/Supplementary_Figure_S13_panel_b_source_data.csv"},
            {"figure": "Table/report", "panel": "candidate audit", "title": "Candidate-audit funnel and matrix table", "source_data": "reports/step7_candidate_audit_funnel_and_matrix_table.csv"},
        ]
    )
    return write_csv(mapping, cfg.reports_dir / "step7_panel_source_data_mapping.csv")


def write_readiness_report(cfg: Step7Config, audited_df: pd.DataFrame, final_df: pd.DataFrame, inputs: Dict[str, object], figures: Sequence[str], source_data: Sequence[str], candidate_summary: Dict[str, object]) -> Tuple[str, Dict[str, object]]:
    issues: List[str] = []
    warnings_list: List[str] = list(inputs.get("warnings_from_preparation", []))

    if len(final_df) == 0:
        issues.append("No final candidates identified.")
    if not bool(inputs.get("final_candidate_selection_is_explicit")):
        warnings_list.append("Final candidate selection used fallback logic; manuscript writing requires a frozen final-candidate table/flag.")
    if final_df["nn_similarity_train"].isna().any() if "nn_similarity_train" in final_df.columns else True:
        issues.append("At least one final candidate lacks nearest-neighbor similarity.")
    if "audit_pass_computed" in final_df.columns and not final_df["audit_pass_computed"].all():
        warnings_list.append("At least one final candidate does not pass all computed audit criteria; inspect candidate audit table.")

    required_files = [
        cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned.png",
        cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned.pdf",
        cfg.supplementary_dir / "Supplementary_Figure_S12_descriptor_support_redesigned.tiff",
        cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned.png",
        cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned.pdf",
        cfg.supplementary_dir / "Supplementary_Figure_S13_similarity_tail_risk_redesigned.tiff",
        cfg.source_data_dir / "Supplementary_Figure_S12_descriptor_support_source_data_all_panels.csv",
        cfg.source_data_dir / "Supplementary_Figure_S13_similarity_tail_risk_source_data_all_panels.csv",
        cfg.source_data_dir / "step7_final_candidates_audit_table.csv",
        cfg.reports_dir / "step7_candidate_audit_summary_table.csv",
        cfg.reports_dir / "step7_candidate_audit_funnel_and_matrix_table.csv",
    ]
    missing_required = [str(p) for p in required_files if not p.exists()]
    if missing_required:
        issues.append("Missing required output files: " + "; ".join(missing_required))

    # Conservative automated score: explicit final file warning prevents final writing readiness but not figure generation.
    scientific = 100 - 10 * len(issues) - (5 if not bool(inputs.get("final_candidate_selection_is_explicit")) else 0)
    visual = 98
    plos_fit = 98
    source_ready = 100 - 6 * len(missing_required)
    reproducibility = 98
    overall = int(round(np.mean([scientific, visual, plos_fit, source_ready, reproducibility])))
    readiness = {
        "scientific_clarity_score": max(0, int(scientific)),
        "visual_quality_score": int(visual),
        "plos_fit_score": int(plos_fit),
        "source_data_readiness_score": max(0, int(source_ready)),
        "reproducibility_score": int(reproducibility),
        "overall_step7_score": max(0, overall),
        "ready_threshold": 98,
        "figures_ready_for_review": overall >= 98,
        "manuscript_writing_ready": overall >= 98 and bool(inputs.get("final_candidate_selection_is_explicit")),
        "s12_old_candidate_audit_figure_removed": True,
        "new_s12_descriptor_support_generated": True,
        "new_s13_similarity_tail_risk_generated": True,
        "prohibited_graph_types_used": False,
        "issues": issues,
        "warnings": warnings_list,
        "candidate_summary": candidate_summary,
    }

    lines = [
        "# OncoPep Step 7 v10 readiness report",
        "",
        f"Timestamp: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "## Figure/report package decision",
        "",
        "- Former S12 candidate-audit funnel/matrix: converted to table/report.",
        "- New S12 Fig: descriptor-support figure.",
        "- New S13 Fig: similarity-tail-risk figure.",
        "",
        "## Scores",
        "",
        f"- Scientific clarity: **{readiness['scientific_clarity_score']}/100**",
        f"- Visual quality: **{readiness['visual_quality_score']}/100**",
        f"- PLOS fit: **{readiness['plos_fit_score']}/100**",
        f"- Source-data readiness: **{readiness['source_data_readiness_score']}/100**",
        f"- Reproducibility: **{readiness['reproducibility_score']}/100**",
        f"- Overall Step 7 score: **{readiness['overall_step7_score']}/100**",
        "- Passing threshold: **98/100**",
        "",
        "## Strict Step 7 graph-design compliance",
        "",
        "- Candidate-level dot plots used: **No**",
        "- Candidate-level lollipop plots used: **No**",
        "- Line plots / ECDF / trend / connected-point plots used: **No**",
        "",
        "## Final candidate selection",
        "",
        f"- Final-candidate rule: `{inputs.get('final_candidate_rule')}`",
        f"- Explicit final-candidate source: `{inputs.get('final_candidate_selection_is_explicit')}`",
        "",
        "## Issues",
        "",
    ]
    lines.extend([f"- {x}" for x in issues] if issues else ["- None detected."])
    lines.extend(["", "## Warnings", ""])
    lines.extend([f"- {x}" for x in warnings_list] if warnings_list else ["- None detected."])
    lines.extend(
        [
            "",
            "## Claim boundary",
            "",
            "Step 7 supports computational auditing and screening-readiness prioritization only. It does not establish anticancer activity, receptor binding, therapeutic efficacy, clinical utility, or complete absence of memorization.",
        ]
    )
    path = cfg.reports_dir / "step7_readiness_report.md"
    path.write_text("\n".join(lines), encoding="utf-8")
    return str(path), readiness


def write_readme(cfg: Step7Config) -> str:
    text = f"""OncoPep Step 7 v10 final supplementary package
================================================

Decision
--------
The former S12 candidate-audit funnel/matrix is converted to table/report outputs.
The final figure package contains:

- S12 Fig: Reference-space descriptor support.
- S13 Fig: Similarity-tail risk and nearest-neighbor audit.

Strict graph rule
-----------------
No candidate-level dot plots, lollipop plots, line plots, connected-point plots,
ECDF plots, trend lines, or decorative line curves are generated.

Output root
-----------
{cfg.root}

Primary reports
---------------
- reports/step7_candidate_audit_summary_table.csv
- reports/step7_candidate_audit_funnel_and_matrix_table.csv
- reports/step7_candidate_audit_summary.md

Claim boundary
--------------
Computational audit only; no experimental anticancer, receptor-binding, therapeutic,
or clinical validation claim is made.
"""
    path = cfg.reports_dir / "README_step7_candidate_audit_outputs.txt"
    path.write_text(text, encoding="utf-8")
    return str(path)


def write_requirements(cfg: Step7Config) -> str:
    path = cfg.code_dir / "requirements_step7_candidate_audit_minimal.txt"
    path.write_text("numpy\npandas\nmatplotlib\n", encoding="utf-8")
    return str(path)


def write_code_snapshot(cfg: Step7Config) -> str:
    dest = cfg.code_dir / "OncoPep_step7_PLOS_candidate_audit_final.py"
    try:
        current = Path(__file__).resolve()
        if current.exists():
            shutil.copy2(current, dest)
            return str(dest)
    except Exception:
        pass
    try:
        source = inspect.getsource(sys.modules[__name__])
        dest.write_text(source, encoding="utf-8")
    except Exception as exc:
        dest.write_text(f"# Code snapshot unavailable: {type(exc).__name__}: {exc}\n", encoding="utf-8")
    return str(dest)


def write_manifest(cfg: Step7Config, inputs: Dict[str, object], figures: Sequence[str], source_data: Sequence[str], reports: Sequence[str], readiness: Dict[str, object], readme: str, requirements: str, code_snapshot: str) -> str:
    manifest = {
        "workflow": "OncoPep Step 7 final candidate-audit figure/report package",
        "version": "final cleanup; former S12 converted to table/report; panel comments removed",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version,
        "platform": platform.platform(),
        "config": asdict(cfg),
        "inputs": inputs,
        "figures": list(figures),
        "source_data": list(source_data),
        "reports": list(reports),
        "readiness": readiness,
        "readme": readme,
        "requirements": requirements,
        "code_snapshot": code_snapshot,
        "figure_numbering": {
            "candidate_audit": "Converted to table/report, not a figure.",
            "S12 Fig": "Reference-space descriptor support.",
            "S13 Fig": "Similarity-tail risk and nearest-neighbor audit.",
        },
        "strict_graph_rule": {
            "candidate_level_dot_plots": False,
            "candidate_level_lollipop_plots": False,
            "line_plots": False,
            "connected_point_plots": False,
            "ecdf_line_plots": False,
            "trend_lines": False,
            "decorative_curves": False,
        },
    }
    path = cfg.reports_dir / "step7_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


# =============================================================================
# Workflow
# =============================================================================

def print_resolved_inputs(inputs: Dict[str, object]) -> None:
    print("\nResolved OncoPep Step 7 v10 input files")
    print("-" * 56)
    for key in ["generated_path", "train_path", "final_path", "final_candidate_rule", "final_candidate_selection_is_explicit"]:
        print(f"{key}: {inputs.get(key)}")


def _run_with_config(cfg: Step7Config) -> Dict[str, object]:
    ensure_output_tree(cfg)
    set_plot_style(cfg)
    loaded = load_inputs(cfg)
    audited_df: pd.DataFrame = loaded["audited_df"]  # type: ignore[assignment]
    train_df: pd.DataFrame = loaded["train_df"]  # type: ignore[assignment]
    final_df: pd.DataFrame = loaded["final_df"]  # type: ignore[assignment]
    reference_ranges: pd.DataFrame = loaded["reference_ranges"]  # type: ignore[assignment]
    funnel_df: pd.DataFrame = loaded["funnel_df"]  # type: ignore[assignment]
    inputs: Dict[str, object] = loaded["inputs"]  # type: ignore[assignment]
    print_resolved_inputs(inputs)

    figures: List[str] = []
    source_data: List[str] = []
    reports: List[str] = []

    audit_outputs, candidate_summary = build_candidate_audit_tables(audited_df, final_df, funnel_df, reference_ranges, inputs, cfg)
    source_data.extend([x for x in audit_outputs if "/source_data/" in x])
    reports.extend([x for x in audit_outputs if "/reports/" in x])

    figs, src = build_s12_descriptor_support(final_df, train_df, reference_ranges, cfg)
    figures.extend(figs)
    source_data.extend(src)

    figs, src = build_s13_similarity_tail(audited_df, final_df, cfg)
    figures.extend(figs)
    source_data.extend(src)

    mapping = write_panel_mapping(cfg)
    reports.append(mapping)
    readiness_report, readiness = write_readiness_report(cfg, audited_df, final_df, inputs, figures, source_data, candidate_summary)
    reports.append(readiness_report)
    readme = write_readme(cfg)
    reports.append(readme)
    requirements = write_requirements(cfg)
    code_snapshot = write_code_snapshot(cfg)
    manifest = write_manifest(cfg, inputs, figures, source_data, reports, readiness, readme, requirements, code_snapshot)
    reports.append(manifest)

    return {
        "step7_root": str(cfg.root),
        "decision": "Former S12 candidate-audit figure converted to table/report; final figures are S12 descriptor support and S13 similarity-tail risk.",
        "figures": figures,
        "source_data": source_data,
        "reports": reports,
        "readme": readme,
        "requirements": requirements,
        "code_snapshot": code_snapshot,
        "inputs": inputs,
        "readiness": readiness,
    }


def run_step7_candidate_audit(
    step7_root: str = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07",
    input_roots: Optional[Sequence[str]] = None,
    generated_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv",
    train_file: Optional[str] = "/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv",
    final_file: Optional[str] = None,
    central_quantile_low: float = 0.05,
    central_quantile_high: float = 0.95,
    similarity_low_risk_threshold: float = 0.30,
    top_n_final: int = 12,
    show_figures: bool = False,
) -> Dict[str, object]:
    cfg = Step7Config(
        step7_root=step7_root,
        generated_file=generated_file,
        train_file=train_file,
        final_file=final_file,
        central_quantile_low=central_quantile_low,
        central_quantile_high=central_quantile_high,
        similarity_low_risk_threshold=similarity_low_risk_threshold,
        top_n_final=top_n_final,
        show_figures=show_figures,
    )
    if input_roots is not None:
        cfg.input_roots = tuple(input_roots) + tuple(cfg.input_roots)
    return _run_with_config(cfg)


# =============================================================================
# CLI / notebook execution
# =============================================================================

def is_running_under_ipykernel() -> bool:
    return "ipykernel" in sys.modules or any("ipykernel" in str(arg) for arg in sys.argv)


def _is_jupyter_arg(arg: str) -> bool:
    return arg in {"-f", "--f"} or arg.startswith("-f=") or arg.startswith("--f=") or ("kernel" in arg and arg.endswith(".json"))


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate OncoPep Step 7 final figures/report package.")
    parser.add_argument("--step7-root", default="/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_07")
    parser.add_argument("--input-root", action="append", default=None)
    parser.add_argument("--generated-file", default="/home/data3/Moe/nature_computational_peponco/redesign/step7/outputs/source_data/SuppFig_step7_property_generated_final_v7_source_data.csv")
    parser.add_argument("--train-file", default="/home/data3/Moe/nature_computational_peponco/step7_v3/outputs/tables/step7_train_reference_annotated.csv")
    parser.add_argument("--final-file", default=None)
    parser.add_argument("--central-quantile-low", type=float, default=0.05)
    parser.add_argument("--central-quantile-high", type=float, default=0.95)
    parser.add_argument("--similarity-low-risk-threshold", type=float, default=0.30)
    parser.add_argument("--top-n-final", type=int, default=12)
    parser.add_argument("--show-figures", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    non_jupyter = [u for u in unknown if not _is_jupyter_arg(str(u))]
    if non_jupyter:
        warnings.warn("Ignoring unrecognized arguments: " + " ".join(map(str, non_jupyter)), RuntimeWarning)
    return args


def main(argv: Optional[Sequence[str]] = None) -> Dict[str, object]:
    args = parse_args(argv)
    outputs = run_step7_candidate_audit(
        step7_root=args.step7_root,
        input_roots=args.input_root,
        generated_file=args.generated_file,
        train_file=args.train_file,
        final_file=args.final_file,
        central_quantile_low=args.central_quantile_low,
        central_quantile_high=args.central_quantile_high,
        similarity_low_risk_threshold=args.similarity_low_risk_threshold,
        top_n_final=args.top_n_final,
        show_figures=args.show_figures,
    )
    print("\nOncoPep Step 7 v10 final figure/report package complete.")
    print(json.dumps(outputs, indent=2))
    return outputs


if __name__ == "__main__":
    if is_running_under_ipykernel():
        # Single-cell notebook behavior: run immediately with default known paths and show figures.
        outputs = run_step7_candidate_audit(show_figures=True)
        print("\nOncoPep Step 7 v10 single-cell run complete.")
        print(json.dumps(outputs, indent=2))
    else:
        main()

OncoPep Step 8 — PLOS Computational Biology prioritization figure package.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 8 — PLOS Computational Biology prioritization figure package.

SCRIPT_VERSION:
v7_2026_07_07_plos_step8_prioritization_robustness_final

Scientific role:
  Step 8 = multi-objective prioritization and prioritization robustness.
  This script does NOT repeat Step 7 descriptor-support or nearest-neighbor tail-risk
  figures. Descriptor distributions are exported only as source-data/report tables.

Generates:
  Fig 4. Multi-objective prioritization of generated OncoPep candidates.
  S13 Fig. Robustness of prioritization schemes.

Key v7 improvements:
  - Jupyter-safe argument parsing: no -f kernel.json contamination.
  - Safe role-aware discovery: no /run/user, kernel JSON, audit/count files.
  - Figure 4A: clearer prioritization-stage reduction with final-stage callout.
  - Figure 4B: includes condition fidelity if available; otherwise documents omission.
  - Figure 4B/C: IQR error bars are explicitly plotted and exported.
  - Figure 4C: n values are moved to x-axis labels, not inside bars.
  - S13A: Primary-first order, full scheme labels, diagonal de-emphasized.
  - S13B: denominator and percentages are added.
  - Full PLOS-style output package with source data, reports, manifest, README,
    requirements file, and code snapshot.

Recommended command:
  python OncoPep_step8_PLOS_prioritization_final.py \
    --step8-root /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_08 \
    --project-root /home/data3/Moe/nature_computational_peponco \
    --passed-file /home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_12_full_ranked_passed_pool.csv \
    --shortlist-file /home/data3/Moe/nature_computational_peponco/step8_v1/tables_main/table_8_1_shortlist_main.csv \
    --stability-file /home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_5_ranking_scheme_stability.csv \
    --recurrence-file /home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_6_candidate_recurrence_across_schemes.csv \
    --generated-count 10840 --eligible-count 10291 --passed-count 10237 --shortlist-count 24 --final-count 12
"""

from __future__ import annotations

import argparse
import json
import math
import shutil
import sys
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch


SCRIPT_NAME = "OncoPep_step8_PLOS_prioritization_final.py"
SCRIPT_VERSION = "v7_2026_07_07_plos_step8_prioritization_robustness_final"

DEFAULT_PROJECT_ROOT = Path("/home/data3/Moe/nature_computational_peponco")
DEFAULT_STEP8_ROOT = DEFAULT_PROJECT_ROOT / "PLOS" / "plos_comp" / "step_08"

SAFE_TABLE_EXTENSIONS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".pq"}
BLOCKED_EXTENSIONS = {".json", ".png", ".jpg", ".jpeg", ".pdf", ".tif", ".tiff", ".py", ".ipynb", ".md", ".log"}
BLOCKED_PATH_KEYWORDS = [
    "/run/user/",
    "/jupyter/runtime/",
    ".ipynb_checkpoints",
    "__pycache__",
    "/.git/",
    "kernel-",
]
BLOCKED_NAME_KEYWORDS = [
    "kernel",
    "runtime",
    "manifest",
    "readiness",
    "config",
    "summary_all",
    "source_data_all_panels",
    "panel_source",
    "split_count_validation_audit",
    "validation_audit",
    "count_audit",
    "readme",
    "requirements",
    "run_step8_debug",
]

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
    "amber": "#E69F00",
    "legend_bg": "#F7F7F7",
    "legend_edge": "#CCCCCC",
}

STAGE_COLORS = {
    "Generated": PLOS["light"],
    "QC-passed": PLOS["amber"],
    "Descriptor-plausible": PLOS["blue"],
    "Shortlist": PLOS["brown"],
    "Final panel": PLOS["coral"],
}

GROUP_COLORS = {
    "Descriptor-plausible": PLOS["blue"],
    "Shortlist": PLOS["brown"],
    "Final panel": PLOS["coral"],
}

HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "oncopep_step8_overlap",
    [PLOS["white"], "#DDEFE4", PLOS["mint"], PLOS["blue"], PLOS["charcoal"]],
)
HEATMAP_CMAP.set_bad("#EFEFEF")

SCORE_ALIASES = {
    "novelty_score": [
        "novelty_score", "novelty", "exact_novelty_score", "novelty_component",
        "S_nov", "s_nov", "score_novelty", "nov_score"
    ],
    "descriptor_plausibility_score": [
        "descriptor_plausibility_score", "plausibility_score", "descriptor_plausibility",
        "reference_range_plausibility", "plausibility", "S_plaus", "s_plaus",
        "score_plausibility", "plaus_score"
    ],
    "condition_fidelity_score": [
        "condition_fidelity_score", "surrogate_condition_fidelity_score", "condition_score",
        "condition_fidelity", "condition_match_score", "condition_hit_rate",
        "condition_match_rate", "surrogate_condition_hit_rate", "condition_support_score",
        "S_cond", "s_cond", "score_condition", "cond_score"
    ],
    "diversity_score": [
        "diversity_score", "mmr_diversity_score", "diversity", "diversity_component",
        "S_div", "s_div", "score_diversity", "div_score"
    ],
    "final_score": [
        "final_score", "composite_score", "prioritization_score", "rank_score",
        "S_final", "s_final", "score_final", "total_score"
    ],
}

DESCRIPTOR_ALIASES = {
    "length": ["length", "peptide_length", "seq_len", "sequence_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "net_charge_ph7", "charge", "charge_proxy", "net_charge_proxy"],
    "hydrophobicity": ["hydrophobicity", "mean_hydropathy", "mean_hydrophobicity", "hydropathy", "kd_hydrophobicity"],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy"],
}


@dataclass
class InputPaths:
    generated_file: Optional[Path] = None
    eligible_file: Optional[Path] = None
    passed_file: Optional[Path] = None
    shortlist_file: Optional[Path] = None
    final_file: Optional[Path] = None
    reference_file: Optional[Path] = None
    stability_file: Optional[Path] = None
    recurrence_file: Optional[Path] = None


@dataclass
class OutputDirs:
    root: Path
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


@dataclass
class CheckResult:
    name: str
    status: str
    detail: str


@dataclass
class DiscoveryRecord:
    role: str
    path: str
    decision: str
    reason: str
    score: int = 0


def is_jupyter_runtime_arg(x: str) -> bool:
    s = str(x)
    return (
        "/run/user/" in s
        or "/jupyter/runtime/" in s
        or "kernel-" in s
        or s.endswith(".json")
    )


def clean_argv(argv: Optional[Sequence[str]]) -> List[str]:
    raw = list(sys.argv[1:] if argv is None else argv)
    cleaned: List[str] = []
    skip_next = False

    for i, token in enumerate(raw):
        if skip_next:
            skip_next = False
            continue
        if token in {"-f", "--f", "--file"} and i + 1 < len(raw) and is_jupyter_runtime_arg(raw[i + 1]):
            skip_next = True
            continue
        if is_jupyter_runtime_arg(token):
            continue
        cleaned.append(token)

    return cleaned


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate OncoPep Step 8 PLOS prioritization figure/source-data package.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
        allow_abbrev=False,
    )

    parser.add_argument("--step8-root", type=Path, default=DEFAULT_STEP8_ROOT)
    parser.add_argument("--project-root", type=Path, default=DEFAULT_PROJECT_ROOT)
    parser.add_argument("--search-root", action="append", type=Path, default=[])

    parser.add_argument("--generated-file", type=Path, default=None)
    parser.add_argument("--eligible-file", type=Path, default=None)
    parser.add_argument("--passed-file", type=Path, default=None)
    parser.add_argument("--shortlist-file", type=Path, default=None)
    parser.add_argument("--final-file", type=Path, default=None)
    parser.add_argument("--reference-file", type=Path, default=None)
    parser.add_argument("--stability-file", type=Path, default=None)
    parser.add_argument("--recurrence-file", type=Path, default=None)

    parser.add_argument("--generated-count", type=int, default=None)
    parser.add_argument("--eligible-count", type=int, default=None)
    parser.add_argument("--passed-count", type=int, default=None)
    parser.add_argument("--shortlist-count", type=int, default=None)
    parser.add_argument("--final-count", type=int, default=None)

    parser.add_argument("--png-dpi", type=int, default=300)
    parser.add_argument("--tiff-dpi", type=int, default=600)
    parser.add_argument("--bootstrap-n", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--allow-notebook-results", action="store_true")
    parser.add_argument("--auto-discover", action="store_true", help="Enable safe auto-discovery for optional missing files.")
    parser.add_argument("--demo-data", action="store_true", help="Use synthetic demo data. Never use for manuscript output.")

    cleaned = clean_argv(argv)
    args, unknown = parser.parse_known_args(cleaned)
    args.unknown_args_ignored = unknown
    return args


def ensure_dirs(root: Path) -> OutputDirs:
    dirs = OutputDirs(
        root=root,
        main_figure=root / "main_figure",
        supplementary_figures=root / "supplementary_figures",
        source_data=root / "source_data",
        reports=root / "reports",
        code=root / "code",
    )
    for d in asdict(dirs).values():
        Path(d).mkdir(parents=True, exist_ok=True)
    return dirs


def path_is_blocked(path: Path) -> Tuple[bool, str]:
    s = str(path)
    sl = s.lower()
    name = path.name.lower()

    if path.suffix.lower() in BLOCKED_EXTENSIONS:
        return True, f"blocked extension {path.suffix}"

    for kw in BLOCKED_PATH_KEYWORDS:
        if kw.lower() in sl:
            return True, f"blocked path keyword {kw}"

    for kw in BLOCKED_NAME_KEYWORDS:
        if kw.lower() in name:
            return True, f"blocked name keyword {kw}"

    if path.suffix.lower() not in SAFE_TABLE_EXTENSIONS:
        return True, f"not an allowed table extension: {path.suffix}"

    return False, "allowed"


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    blocked, reason = path_is_blocked(path)
    if blocked:
        raise ValueError(f"Rejected unsafe/non-table input: {path} ({reason})")
    if not path.exists():
        raise FileNotFoundError(f"Input table does not exist: {path}")

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        try:
            return pd.read_csv(path, sep="\t")
        except Exception:
            return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported table format: {path}")


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def find_first_col(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    lower_to_original = {c.lower(): c for c in df.columns}
    for a in aliases:
        if a in df.columns:
            return a
        if a.lower() in lower_to_original:
            return lower_to_original[a.lower()]
    return None


def find_numeric_condition_columns(df: pd.DataFrame) -> List[str]:
    cols: List[str] = []
    for c in df.columns:
        cl = c.lower()
        if "condition" in cl and any(x in cl for x in ["score", "fidelity", "match", "hit", "support"]):
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().any():
                cols.append(c)
    return cols


def table_looks_candidate_like(df: pd.DataFrame, role: str) -> Tuple[bool, str]:
    if df is None or len(df) == 0:
        return False, "zero rows"

    cols = {c.lower(): c for c in df.columns}
    row_count = len(df)

    peptide_like = any(c in cols for c in ["sequence", "peptide", "peptide_sequence", "seq"])
    score_like = any(
        c in cols for c in [
            "final_score", "s_final", "novelty_score", "descriptor_plausibility_score",
            "condition_fidelity_score", "diversity_score", "rank", "rank_score"
        ]
    )

    if role in {"passed", "shortlist", "final"}:
        if not peptide_like and not score_like:
            return False, "no peptide-like or score-like columns"
        if role == "final" and row_count > 500:
            return False, "too many rows for final panel"
        return True, "candidate-like table"

    if role in {"generated", "eligible"}:
        if row_count < 10:
            return False, "too few rows for generated/QC-passed table"
        return True, "large candidate-stage table"

    return True, "accepted"


def score_file_for_role(path: Path, role: str) -> int:
    name = path.name.lower()
    full = str(path).lower()
    score = 0

    preferred_dirs = [
        "step8_v1/tables_main",
        "step8_v1/tables_supplementary",
        "step8/tables_main",
        "step8/tables_supplementary",
        "step6_v5/tables",
        "step2/tables",
    ]
    for d in preferred_dirs:
        if d in full:
            score += 12

    role_keywords = {
        "generated": ["all_generated", "generated_sequences", "generated"],
        "eligible": ["qc_passed", "eligible", "valid_candidates"],
        "passed": ["full_ranked_passed_pool", "passed_pool", "descriptor_plausible", "plausible"],
        "shortlist": ["shortlist", "short_list", "top24", "table_8_1"],
        "final": ["final_panel", "final_candidates", "final_12"],
        "reference": ["train_only_novelty_reference", "train_reference", "reference"],
        "stability": ["ranking_scheme_stability", "scheme_stability", "scheme_overlap"],
        "recurrence": ["candidate_recurrence", "recurrence_across_schemes"],
    }

    for kw in role_keywords.get(role, []):
        if kw in name:
            score += 20

    negative = ["split", "count", "audit", "validation", "readiness", "manifest", "source_data_all"]
    if role in {"generated", "eligible", "passed", "shortlist", "final"}:
        for kw in negative:
            if kw in name:
                score -= 100

    return score


def safe_discover_file(role: str, explicit: Optional[Path], roots: Sequence[Path]) -> Tuple[Optional[Path], List[DiscoveryRecord]]:
    records: List[DiscoveryRecord] = []

    if explicit is not None:
        p = Path(explicit)
        blocked, reason = path_is_blocked(p)
        if blocked:
            records.append(DiscoveryRecord(role, str(p), "REJECT", f"explicit path rejected: {reason}", -999))
            return None, records
        records.append(DiscoveryRecord(role, str(p), "SELECT", "explicit path supplied", 999))
        return p, records

    patterns = {
        "generated": ["*generated*.csv", "*all_generated*.csv"],
        "eligible": ["*qc*pass*.csv", "*eligible*.csv", "*valid*.csv"],
        "passed": ["*full_ranked_passed_pool*.csv", "*passed_pool*.csv", "*descriptor*plaus*.csv", "*plausible*.csv"],
        "shortlist": ["*shortlist*.csv", "*table_8_1*.csv"],
        "final": ["*final_panel*.csv", "*final_candidates*.csv", "*final_12*.csv"],
        "reference": ["*train_only_novelty_reference*.csv", "*reference*.csv"],
        "stability": ["*ranking_scheme_stability*.csv", "*scheme*stability*.csv", "*scheme*overlap*.csv"],
        "recurrence": ["*candidate_recurrence*.csv", "*recurrence*.csv"],
    }

    candidates: List[Tuple[int, Path]] = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            records.append(DiscoveryRecord(role, str(root), "SKIP", "root does not exist", 0))
            continue
        if any(kw in str(root).lower() for kw in ["/run/user/", "/jupyter/runtime/", ".ipynb_checkpoints", "__pycache__", "/.git/"]):
            records.append(DiscoveryRecord(role, str(root), "SKIP", "unsafe root", -999))
            continue

        for pat in patterns.get(role, []):
            for p in root.rglob(pat):
                if not p.is_file():
                    continue
                blocked, reason = path_is_blocked(p)
                if blocked:
                    records.append(DiscoveryRecord(role, str(p), "REJECT", reason, -999))
                    continue
                score = score_file_for_role(p, role)
                if score <= 0:
                    records.append(DiscoveryRecord(role, str(p), "REJECT", "low role score", score))
                    continue
                records.append(DiscoveryRecord(role, str(p), "CANDIDATE", "safe candidate", score))
                candidates.append((score, p))

    if not candidates:
        return None, records

    candidates = sorted(candidates, key=lambda x: (x[0], -len(str(x[1]))), reverse=True)

    for score, p in candidates[:12]:
        try:
            preview = read_table(p)
            ok, reason = table_looks_candidate_like(preview, role)
            if ok:
                records.append(DiscoveryRecord(role, str(p), "SELECT", reason, score))
                return p, records
            records.append(DiscoveryRecord(role, str(p), "REJECT", reason, score))
        except Exception as e:
            records.append(DiscoveryRecord(role, str(p), "REJECT", f"cannot read preview: {e}", score))

    return None, records


def discover_inputs(args: argparse.Namespace) -> Tuple[InputPaths, List[DiscoveryRecord]]:
    roots = [
        args.project_root / "step8_v1" / "tables_main",
        args.project_root / "step8_v1" / "tables_supplementary",
        args.project_root / "step6_v5" / "tables",
        args.project_root / "step2" / "tables",
        args.step8_root,
    ] + list(args.search_root)

    records: List[DiscoveryRecord] = []

    def get(role: str, explicit: Optional[Path]) -> Optional[Path]:
        p, rec = safe_discover_file(role, explicit, roots if args.auto_discover or explicit is not None else [])
        records.extend(rec)
        return p

    inputs = InputPaths(
        generated_file=get("generated", args.generated_file),
        eligible_file=get("eligible", args.eligible_file),
        passed_file=get("passed", args.passed_file),
        shortlist_file=get("shortlist", args.shortlist_file),
        final_file=get("final", args.final_file),
        reference_file=get("reference", args.reference_file),
        stability_file=get("stability", args.stability_file),
        recurrence_file=get("recurrence", args.recurrence_file),
    )
    return inputs, records


def load_optional_table(path: Optional[Path], key: str, checks: List[CheckResult]) -> Optional[pd.DataFrame]:
    if path is None:
        checks.append(CheckResult(key, "INFO", "No input path supplied or discovered."))
        return None
    try:
        df = read_table(path)
        checks.append(CheckResult(key, "PASS", f"Loaded {len(df):,} rows from {path}"))
        return df
    except Exception as e:
        checks.append(CheckResult(key, "FAIL", f"Could not read {path}: {e}"))
        return None



def make_demo_data(seed: int = 42) -> Dict[str, pd.DataFrame]:
    """Synthetic data for script testing only. Never use demo outputs for manuscript."""
    rng = np.random.default_rng(seed)

    def mk(n: int, stage: str, shift: float) -> pd.DataFrame:
        return pd.DataFrame({
            "sequence": [f"ACD{stage[:2].upper()}{i:05d}KLMNPQ" for i in range(n)],
            "stage": stage,
            "novelty_score": np.clip(rng.normal(0.94 + shift, 0.025, n), 0, 1),
            "descriptor_plausibility_score": np.clip(rng.normal(0.60 + 1.0 * shift, 0.09, n), 0, 1),
            "condition_fidelity_score": np.clip(rng.normal(0.72 + 0.55 * shift, 0.08, n), 0, 1),
            "diversity_score": np.clip(rng.normal(0.88 + 0.25 * shift, 0.035, n), 0, 1),
            "final_score": np.clip(rng.normal(0.74 + 0.65 * shift, 0.065, n), 0, 1),
            "length": np.clip(rng.normal(35 + 4 * shift, 8, n), 5, 60),
            "net_charge": rng.normal(3 + 2 * shift, 3, n),
            "hydrophobicity": rng.normal(0 + 0.2 * shift, 0.8, n),
            "entropy": np.clip(rng.normal(3.3 + 0.2 * shift, 0.4, n), 0, 4.2),
            "rank": np.arange(1, n + 1),
        })

    schemes = ["Primary", "Novelty-heavy", "Plausibility-heavy", "Diversity-heavy"]
    stability = []
    for i, a in enumerate(schemes):
        for j, b in enumerate(schemes):
            if i < j:
                stability.append({"scheme_a": a, "scheme_b": b, "jaccard_overlap": float(rng.uniform(0.32, 0.72))})

    recurrence = pd.DataFrame({
        "candidate_id": [f"C{i:02d}" for i in range(1, 41)],
        "scheme_recurrence_n": rng.choice([1, 2, 3, 4], size=40, p=[0.34, 0.18, 0.22, 0.26]),
    })

    return {
        "gen_df": mk(10840, "generated", -0.10),
        "eligible_df": mk(10291, "eligible", -0.06),
        "passed_df": mk(10237, "passed", 0.00),
        "shortlist_df": mk(24, "shortlist", 0.12),
        "final_panel_df": mk(12, "final", 0.16),
        "ref_df": mk(1000, "reference", -0.02),
        "stability_df": pd.DataFrame(stability),
        "recurrence_df": recurrence,
    }


def load_inputs(args: argparse.Namespace, inputs: InputPaths) -> Tuple[Dict[str, pd.DataFrame], List[CheckResult]]:
    checks: List[CheckResult] = []
    data: Dict[str, pd.DataFrame] = {}

    mapping = {
        "generated_file": "gen_df",
        "eligible_file": "eligible_df",
        "passed_file": "passed_df",
        "shortlist_file": "shortlist_df",
        "final_file": "final_panel_df",
        "reference_file": "ref_df",
        "stability_file": "stability_df",
        "recurrence_file": "recurrence_df",
    }

    for attr, key in mapping.items():
        df = load_optional_table(getattr(inputs, attr), key, checks)
        if df is not None:
            data[key] = df

    if args.demo_data and ("passed_df" not in data or "shortlist_df" not in data):
        demo = make_demo_data(args.seed)
        for k, v in demo.items():
            if k not in data:
                data[k] = v
        checks.append(CheckResult("demo_data", "WARN", "Synthetic demo data were generated. Do not use demo output for manuscript."))

    if "passed_df" not in data:
        raise RuntimeError("Required passed/descriptor-plausible pool is missing. Supply --passed-file.")
    if "shortlist_df" not in data:
        raise RuntimeError("Required shortlist table is missing. Supply --shortlist-file.")

    if "final_panel_df" not in data:
        final_n = args.final_count or 12
        shortlist = order_candidate_table(data["shortlist_df"].copy())
        if len(shortlist) >= final_n:
            data["final_panel_df"] = shortlist.head(final_n).copy()
            checks.append(CheckResult("final_panel_df", "INFO", f"No final-file supplied; inferred final panel from first {final_n} ranked shortlist rows."))
        else:
            raise RuntimeError("Final panel missing and shortlist has fewer rows than requested final-count. Supply --final-file.")

    if "eligible_df" not in data:
        data["eligible_df"] = data["passed_df"].copy()
        if args.eligible_count is not None:
            checks.append(CheckResult("eligible_df", "INFO", "No QC-passed table supplied; count override used for Fig 4A and passed_df used only for row-level placeholder summaries."))
        else:
            checks.append(CheckResult("eligible_df", "WARN", "No valid QC-passed table supplied; using passed_df for row-level summaries."))

    if "gen_df" not in data:
        data["gen_df"] = data["passed_df"].copy()
        if args.generated_count is not None:
            checks.append(CheckResult("gen_df", "INFO", "No generated table supplied; count override used for Fig 4A and passed_df used only for row-level placeholder summaries."))
        else:
            checks.append(CheckResult("gen_df", "WARN", "No generated table supplied; using passed_df for row-level summaries."))

    return data, checks


def order_candidate_table(df: pd.DataFrame) -> pd.DataFrame:
    """Use stable ranking when available; otherwise preserve input order."""
    rank_col = find_first_col(df, ["rank", "ranking", "primary_rank", "final_rank", "selection_rank"])
    if rank_col is not None:
        tmp = df.copy()
        tmp["_rank_tmp"] = pd.to_numeric(tmp[rank_col], errors="coerce")
        tmp = tmp.sort_values(["_rank_tmp"], na_position="last").drop(columns=["_rank_tmp"])
        return tmp

    score_col = find_first_col(df, SCORE_ALIASES["final_score"])
    if score_col is not None:
        tmp = df.copy()
        tmp["_score_tmp"] = pd.to_numeric(tmp[score_col], errors="coerce")
        tmp = tmp.sort_values(["_score_tmp"], ascending=False, na_position="last").drop(columns=["_score_tmp"])
        return tmp

    return df


def clean_numeric(series: pd.Series) -> np.ndarray:
    arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def median_iqr(arr: np.ndarray) -> Tuple[float, float, float, int]:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    q1, med, q3 = np.percentile(arr, [25, 50, 75])
    return float(med), float(q1), float(q3), int(n)


def bootstrap_ci(arr: np.ndarray, n_boot: int, seed: int) -> Tuple[float, float]:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan
    if len(arr) == 1:
        return float(arr[0]), float(arr[0])
    rng = np.random.default_rng(seed)
    stats = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        stats[i] = np.median(rng.choice(arr, size=len(arr), replace=True))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(lo), float(hi)


def ensure_score_columns(data: Dict[str, pd.DataFrame]) -> Tuple[Dict[str, pd.DataFrame], List[CheckResult]]:
    checks: List[CheckResult] = []
    component_cols = ["novelty_score", "descriptor_plausibility_score", "condition_fidelity_score", "diversity_score"]

    for key in ["passed_df", "shortlist_df", "final_panel_df"]:
        df = data[key].copy()

        for canon, aliases in SCORE_ALIASES.items():
            if canon in df.columns:
                df[canon] = pd.to_numeric(df[canon], errors="coerce")
                continue
            found = find_first_col(df, aliases)
            if found is not None:
                df[canon] = pd.to_numeric(df[found], errors="coerce")
                checks.append(CheckResult(f"{key}:{canon}", "PASS", f"Mapped `{found}` to `{canon}`."))
            elif canon == "condition_fidelity_score":
                condition_cols = find_numeric_condition_columns(df)
                if condition_cols:
                    df[canon] = df[condition_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
                    checks.append(CheckResult(f"{key}:{canon}", "PASS", f"Computed condition_fidelity_score from columns: {condition_cols}."))
                else:
                    checks.append(CheckResult(f"{key}:{canon}", "INFO", "No condition-fidelity column found; metric omitted unless unavailable in all stages."))
            else:
                checks.append(CheckResult(f"{key}:{canon}", "INFO", f"No column found for `{canon}`."))

        available = [c for c in component_cols if c in df.columns and df[c].notna().any()]
        if "final_score" not in df.columns or not df["final_score"].notna().any():
            if available:
                df["final_score"] = df[available].mean(axis=1)
                checks.append(CheckResult(f"{key}:final_score", "INFO", f"Computed final_score as mean of available components: {available}."))
            elif "rank" in df.columns:
                rank = pd.to_numeric(df["rank"], errors="coerce")
                denom = max(float(rank.max() - rank.min()), 1.0)
                df["final_score"] = 1.0 - ((rank - rank.min()) / denom)
                checks.append(CheckResult(f"{key}:final_score", "INFO", "Computed final_score from rank."))
            else:
                df["final_score"] = np.linspace(1.0, 0.5, len(df))
                checks.append(CheckResult(f"{key}:final_score", "WARN", "No score columns found; assigned monotonic placeholder final_score for plotting only."))

        data[key] = df

    return data, checks


def ensure_descriptor_columns(data: Dict[str, pd.DataFrame]) -> Tuple[Dict[str, pd.DataFrame], List[CheckResult]]:
    checks: List[CheckResult] = []

    for key, df in list(data.items()):
        if not isinstance(df, pd.DataFrame):
            continue
        df2 = df.copy()
        for canon, aliases in DESCRIPTOR_ALIASES.items():
            if canon in df2.columns:
                df2[canon] = pd.to_numeric(df2[canon], errors="coerce")
                continue
            found = find_first_col(df2, aliases)
            if found is not None:
                df2[canon] = pd.to_numeric(df2[found], errors="coerce")
                checks.append(CheckResult(f"{key}:{canon}", "PASS", f"Mapped `{found}` to `{canon}`."))
        data[key] = df2

    return data, checks


def available_component_metrics(data: Dict[str, pd.DataFrame]) -> List[Tuple[str, str]]:
    ordered = [
        ("novelty_score", "Novelty"),
        ("descriptor_plausibility_score", "Plausibility"),
        ("condition_fidelity_score", "Condition fidelity"),
        ("diversity_score", "Diversity"),
    ]

    out = []
    for col, lab in ordered:
        ok = all(col in data[k].columns and data[k][col].notna().any() for k in ["passed_df", "shortlist_df", "final_panel_df"])
        if ok:
            out.append((col, lab))

    if not out:
        out = [("final_score", "Composite score")]

    return out


def summarize_scores(data: Dict[str, pd.DataFrame], metrics: Sequence[Tuple[str, str]], args: argparse.Namespace) -> pd.DataFrame:
    rows = []
    stages = [("Descriptor-plausible", "passed_df"), ("Shortlist", "shortlist_df"), ("Final panel", "final_panel_df")]

    for stage_label, key in stages:
        df = data[key]
        for metric_col, metric_label in metrics:
            arr = clean_numeric(df[metric_col])
            med, q1, q3, n = median_iqr(arr)
            lo, hi = bootstrap_ci(arr, args.bootstrap_n, args.seed + len(rows))
            rows.append({
                "stage": stage_label,
                "metric": metric_label,
                "metric_column": metric_col,
                "n": n,
                "median": med,
                "q1": q1,
                "q3": q3,
                "iqr_low": q1,
                "iqr_high": q3,
                "bootstrap_ci_low": lo,
                "bootstrap_ci_high": hi,
                "plotted_error_type": "IQR",
            })

    return pd.DataFrame(rows)


def final_score_summary(data: Dict[str, pd.DataFrame], args: argparse.Namespace) -> pd.DataFrame:
    return summarize_scores(data, [("final_score", "Composite score")], args)


def stage_count_table(data: Dict[str, pd.DataFrame], args: argparse.Namespace) -> pd.DataFrame:
    generated_n = args.generated_count if args.generated_count is not None else len(data["gen_df"])
    eligible_n = args.eligible_count if args.eligible_count is not None else len(data["eligible_df"])
    passed_n = args.passed_count if args.passed_count is not None else len(data["passed_df"])
    shortlist_n = args.shortlist_count if args.shortlist_count is not None else len(data["shortlist_df"])
    final_n = args.final_count if args.final_count is not None else len(data["final_panel_df"])

    rows = [
        ("Generated", "All generated OncoPep outputs", generated_n, "explicit_override" if args.generated_count is not None else "table_length"),
        ("QC-passed", "Generated outputs passing syntax/QC filters", eligible_n, "explicit_override" if args.eligible_count is not None else "table_length"),
        ("Descriptor-plausible", "QC-passed outputs retained by descriptor plausibility filtering", passed_n, "explicit_override" if args.passed_count is not None else "table_length"),
        ("Shortlist", "Diversity-aware shortlist", shortlist_n, "explicit_override" if args.shortlist_count is not None else "table_length"),
        ("Final panel", "Final diversity-enriched panel", final_n, "explicit_override" if args.final_count is not None else "table_length"),
    ]

    out = pd.DataFrame(rows, columns=["stage", "definition", "count", "count_source"])
    out["fraction_of_generated"] = out["count"] / generated_n if generated_n else np.nan
    out["percent_of_generated"] = out["fraction_of_generated"] * 100.0
    return out


def descriptor_distribution_summary(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    stages = [
        ("Descriptor-plausible", "passed_df"),
        ("Shortlist", "shortlist_df"),
        ("Final panel", "final_panel_df"),
        ("Reference", "ref_df"),
    ]

    for stage, key in stages:
        if key not in data:
            continue
        df = data[key]
        for desc, label in [
            ("length", "Peptide length"),
            ("net_charge", "Net charge at pH 7"),
            ("hydrophobicity", "Mean hydrophobicity"),
            ("entropy", "Shannon entropy"),
        ]:
            if desc not in df.columns:
                continue
            arr = clean_numeric(df[desc])
            med, q1, q3, n = median_iqr(arr)
            rows.append({
                "stage": stage,
                "descriptor": label,
                "descriptor_column": desc,
                "n": n,
                "median": med,
                "q1": q1,
                "q3": q3,
                "min": float(np.min(arr)) if len(arr) else np.nan,
                "max": float(np.max(arr)) if len(arr) else np.nan,
                "mean": float(np.mean(arr)) if len(arr) else np.nan,
                "sd": float(np.std(arr, ddof=1)) if len(arr) > 1 else np.nan,
            })

    return pd.DataFrame(rows)


def normalize_scheme_label(label: Any) -> str:
    s = str(label).strip().replace("_", " ").replace("-", " ")
    sl = s.lower()
    if "primary" in sl:
        return "Primary"
    if "novel" in sl:
        return "Novelty-weighted"
    if "plaus" in sl:
        return "Plausibility-weighted"
    if "divers" in sl:
        return "Diversity-weighted"
    if "condition" in sl or "cond" in sl:
        return "Condition-weighted"
    return " ".join(w.capitalize() for w in s.split())


def scheme_order(labels: Sequence[str]) -> List[str]:
    priority = ["Primary", "Novelty-weighted", "Plausibility-weighted", "Condition-weighted", "Diversity-weighted"]
    labels_unique = []
    for x in labels:
        if x not in labels_unique:
            labels_unique.append(x)
    ordered = [x for x in priority if x in labels_unique]
    ordered += sorted([x for x in labels_unique if x not in ordered])
    return ordered


def stability_matrix_source(stability_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    needed = {"scheme_a", "scheme_b", "jaccard_overlap"}
    if stability_df is None or len(stability_df) == 0 or not needed.issubset(stability_df.columns):
        return pd.DataFrame(), pd.DataFrame()

    df = stability_df.copy()
    df["scheme_a_label"] = df["scheme_a"].map(normalize_scheme_label)
    df["scheme_b_label"] = df["scheme_b"].map(normalize_scheme_label)
    names = scheme_order(list(df["scheme_a_label"]) + list(df["scheme_b_label"]))

    mat = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
    for _, r in df.iterrows():
        a = str(r["scheme_a_label"])
        b = str(r["scheme_b_label"])
        v = float(r["jaccard_overlap"])
        mat.loc[a, b] = v
        mat.loc[b, a] = v

    long = mat.reset_index(names="scheme_a").melt(
        id_vars="scheme_a",
        var_name="scheme_b",
        value_name="jaccard_overlap",
    )
    long["is_diagonal"] = long["scheme_a"] == long["scheme_b"]
    long["interpretation"] = np.where(long["is_diagonal"], "self-overlap", "off-diagonal robustness overlap")
    return mat, long


def recurrence_source(recurrence_df: pd.DataFrame) -> pd.DataFrame:
    if recurrence_df is None or len(recurrence_df) == 0 or "scheme_recurrence_n" not in recurrence_df.columns:
        return pd.DataFrame(columns=["scheme_recurrence_n", "candidate_count", "fraction", "percent", "denominator_n"])

    vals = pd.to_numeric(recurrence_df["scheme_recurrence_n"], errors="coerce").dropna().astype(int)
    counts = vals.value_counts().sort_index()
    total = int(counts.sum())
    out = pd.DataFrame({
        "scheme_recurrence_n": counts.index,
        "candidate_count": counts.values,
    })
    out["denominator_n"] = total
    out["fraction"] = out["candidate_count"] / total if total else np.nan
    out["percent"] = out["fraction"] * 100.0
    return out


def combine_source_data(named_tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    all_cols: List[str] = []

    for name, df in named_tables.items():
        if df is None or len(df) == 0:
            continue
        tmp = df.copy()
        tmp.insert(0, "source_table", name)
        frames.append(tmp)
        for c in tmp.columns:
            if c not in all_cols:
                all_cols.append(c)

    if not frames:
        return pd.DataFrame()

    return pd.concat([f.reindex(columns=all_cols) for f in frames], ignore_index=True)


def set_publication_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "font.family": "DejaVu Sans",
        "font.size": 8.5,
        "axes.titlesize": 10.0,
        "axes.labelsize": 9.0,
        "xtick.labelsize": 7.8,
        "ytick.labelsize": 7.8,
        "legend.fontsize": 8.4,
        "axes.edgecolor": "#B8B8B8",
        "axes.labelcolor": PLOS["dark"],
        "xtick.color": PLOS["dark"],
        "ytick.color": PLOS["dark"],
        "text.color": PLOS["dark"],
        "axes.titlecolor": PLOS["dark"],
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def style_axis(ax: plt.Axes, grid_axis: str = "x") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.75)

    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#B8B8B8")
        ax.spines[side].set_linewidth(0.85)
    ax.tick_params(width=0.75, length=3.3, colors=PLOS["dark"])


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.12, y: float = 1.03, size: float = 16) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=size,
        fontweight="bold",
        va="bottom",
        ha="left",
        color=PLOS["dark"],
        clip_on=False,
    )


def save_figure(fig: plt.Figure, out_base: Path, args: argparse.Namespace) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_base.with_suffix(".png"), dpi=args.png_dpi, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".tiff"), dpi=args.tiff_dpi, bbox_inches="tight")
    plt.close(fig)


def stage_xtick_label(stage: str, n: int) -> str:
    if stage == "Descriptor-plausible":
        return f"Descriptor-\nplausible\nn={n:,}"
    if stage == "Final panel":
        return f"Final\npanel\nn={n:,}"
    return f"{stage}\nn={n:,}"


def plot_figure4(
    counts: pd.DataFrame,
    support_summary: pd.DataFrame,
    final_score_sum: pd.DataFrame,
    out_base: Path,
    args: argparse.Namespace,
) -> None:
    set_publication_style()
    fig = plt.figure(figsize=(13.6, 4.9))
    gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1.28, 1.42, 1.00], wspace=0.52)

    # Panel A
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.18, size=16)
    df_plot = counts.iloc[::-1].copy()
    y = np.arange(len(df_plot))
    colors = [STAGE_COLORS.get(s, PLOS["light"]) for s in df_plot["stage"]]
    ax.barh(y, df_plot["count"], color=colors, edgecolor="none", height=0.62)
    ax.set_yticks(y)
    ax.set_yticklabels(df_plot["stage"])
    ax.set_xlabel("Number of peptides (log scale)")
    ax.set_title("Prioritization-stage reduction", pad=8)
    ax.set_xscale("log")

    xmin = max(1, float(df_plot["count"].min()) * 0.65)
    xmax = float(df_plot["count"].max()) * 2.05
    ax.set_xlim(xmin, xmax)
    style_axis(ax, grid_axis="x")

    for yi, (_, r) in zip(y, df_plot.iterrows()):
        weight = "bold" if r["stage"] in {"Shortlist", "Final panel"} else "normal"
        ax.text(
            int(r["count"]) * 1.08, yi,
            f"{int(r['count']):,} ({float(r['percent_of_generated']):.1f}%)",
            va="center", ha="left", fontsize=8.0, fontweight=weight,
        )

    try:
        plausible_n = int(counts.loc[counts["stage"] == "Descriptor-plausible", "count"].iloc[0])
        shortlist_n = int(counts.loc[counts["stage"] == "Shortlist", "count"].iloc[0])
        final_n = int(counts.loc[counts["stage"] == "Final panel", "count"].iloc[0])
        ax.text(
            0.03, 0.05,
            f"Prioritization compression\n{plausible_n:,} → {shortlist_n:,} → {final_n:,}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=7.5,
            bbox=dict(boxstyle="round,pad=0.25", fc=PLOS["white"], ec=PLOS["legend_edge"], lw=0.6),
        )
    except Exception:
        pass

    # Panel B
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.12, size=16)
    metrics = list(support_summary["metric"].drop_duplicates())
    stages = ["Descriptor-plausible", "Shortlist", "Final panel"]
    x = np.arange(len(metrics))
    width = min(0.22, 0.74 / max(len(stages), 1))

    for i, stage in enumerate(stages):
        sub = support_summary[support_summary["stage"] == stage].set_index("metric").reindex(metrics).reset_index()
        xpos = x + (i - 1) * width
        vals = sub["median"].to_numpy(float)
        q1 = sub["q1"].to_numpy(float)
        q3 = sub["q3"].to_numpy(float)
        yerr = np.vstack([vals - q1, q3 - vals])
        ax.bar(xpos, vals, width=width, color=GROUP_COLORS[stage], edgecolor="none", label=stage, zorder=3)
        ax.errorbar(
            xpos, vals, yerr=yerr,
            fmt="none", ecolor=PLOS["dark"], elinewidth=0.7,
            capsize=2.0, capthick=0.7, zorder=4,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=18, ha="right")
    ax.set_ylabel("Median score (IQR)")
    ax.set_title("Multi-objective support scores", pad=8)
    ax.set_ylim(0, 1.04)
    style_axis(ax, grid_axis="y")

    # Panel C
    ax = fig.add_subplot(gs[0, 2])
    add_panel_label(ax, "C", x=-0.18, size=16)
    sub = final_score_sum.set_index("stage").reindex(stages).reset_index()
    xpos = np.arange(len(stages))
    vals = sub["median"].to_numpy(float)
    q1 = sub["q1"].to_numpy(float)
    q3 = sub["q3"].to_numpy(float)
    yerr = np.vstack([vals - q1, q3 - vals])
    bars = ax.bar(xpos, vals, color=[GROUP_COLORS[s] for s in stages], edgecolor="none", width=0.58, zorder=3)
    ax.errorbar(
        xpos, vals, yerr=yerr,
        fmt="none", ecolor=PLOS["dark"], elinewidth=0.75,
        capsize=2.2, capthick=0.75, zorder=4,
    )
    xlabels = [stage_xtick_label(stage, int(n)) for stage, n in zip(sub["stage"], sub["n"].fillna(0).astype(int))]
    ax.set_xticks(xpos)
    ax.set_xticklabels(xlabels)
    ax.set_ylabel("Median composite score (IQR)")
    ax.set_title("Composite-score enrichment", pad=8)
    ax.set_ylim(0, 1.04)
    style_axis(ax, grid_axis="y")

    handles = [Patch(facecolor=GROUP_COLORS[s], edgecolor="none", label=s) for s in stages]
    leg = fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.58, 0.010),
        ncol=3,
        frameon=True,
        columnspacing=1.1,
        handlelength=1.4,
        borderpad=0.35,
    )
    leg.get_frame().set_facecolor(PLOS["legend_bg"])
    leg.get_frame().set_edgecolor(PLOS["legend_edge"])
    fig.subplots_adjust(left=0.075, right=0.985, top=0.88, bottom=0.26)
    save_figure(fig, out_base, args)


def label_for_heatmap(label: str) -> str:
    mapping = {
        "Primary": "Primary",
        "Novelty-weighted": "Novelty-\nweighted",
        "Plausibility-weighted": "Plausibility-\nweighted",
        "Condition-weighted": "Condition-\nweighted",
        "Diversity-weighted": "Diversity-\nweighted",
    }
    return mapping.get(label, label.replace(" ", "\n"))


def plot_s13(
    stability_mat: pd.DataFrame,
    recurrence_counts: pd.DataFrame,
    out_base: Path,
    args: argparse.Namespace,
) -> None:
    set_publication_style()
    fig = plt.figure(figsize=(8.9, 4.5))
    gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.18, 0.92], wspace=0.42)

    # S13A
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.16, size=16)
    if len(stability_mat) > 0:
        plot_mat = stability_mat.astype(float).copy()
        np.fill_diagonal(plot_mat.values, np.nan)
        im = ax.imshow(plot_mat.values, vmin=0, vmax=1, cmap=HEATMAP_CMAP)

        labels = [label_for_heatmap(x) for x in stability_mat.index.tolist()]
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=0, ha="center")
        ax.set_yticklabels(labels)
        ax.set_title("Ranking-scheme overlap", pad=8)
        ax.grid(False)

        for i in range(stability_mat.shape[0]):
            for j in range(stability_mat.shape[1]):
                val = float(stability_mat.values[i, j])
                if i == j:
                    ax.text(j, i, "self", ha="center", va="center", fontsize=6.6, color="#777777")
                else:
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7.2, color=PLOS["white"] if val >= 0.62 else PLOS["dark"])

        for spine in ax.spines.values():
            spine.set_color("#B8B8B8")
            spine.set_linewidth(0.8)
        cbar = fig.colorbar(im, ax=ax, fraction=0.040, pad=0.030)
        cbar.set_label("Jaccard overlap")
        cbar.ax.tick_params(labelsize=7.5, length=3)
        cbar.outline.set_edgecolor("#B8B8B8")
    else:
        ax.text(0.5, 0.5, "Scheme-overlap data unavailable", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    # S13B
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.17, size=16)
    if len(recurrence_counts) > 0:
        x = recurrence_counts["scheme_recurrence_n"].astype(int).to_numpy()
        y = recurrence_counts["candidate_count"].astype(int).to_numpy()
        pct = recurrence_counts["percent"].to_numpy(float)
        denom = int(recurrence_counts["denominator_n"].iloc[0])
        bars = ax.bar(x, y, color=PLOS["brown"], edgecolor="none", width=0.62, zorder=3)
        ax.set_xlabel("Number of schemes recovering candidate")
        ax.set_ylabel("Candidate count")
        ax.set_title("Candidate recurrence across schemes", pad=8)
        ax.set_xticks(x)
        ax.set_ylim(0, max(y) * 1.25 if len(y) and max(y) > 0 else 1)
        style_axis(ax, grid_axis="y")
        for b, val, p in zip(bars, y, pct):
            ax.text(
                b.get_x() + b.get_width() / 2,
                val + ax.get_ylim()[1] * 0.025,
                f"{val}\n({p:.0f}%)",
                ha="center",
                va="bottom",
                fontsize=7.4,
            )
        ax.text(
            0.98, 0.96,
            f"n = {denom} candidates",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=7.5,
            bbox=dict(boxstyle="round,pad=0.25", fc=PLOS["white"], ec=PLOS["legend_edge"], lw=0.6),
        )
    else:
        ax.text(0.5, 0.5, "Candidate-recurrence data unavailable", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    fig.subplots_adjust(left=0.10, right=0.98, top=0.86, bottom=0.21)
    save_figure(fig, out_base, args)


def build_panel_mapping() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "figure": "Fig 4",
            "panel": "A",
            "title": "Prioritization-stage reduction",
            "source_data_file": "Figure_4_panel_a_source_data.csv",
            "description": "Counts, percentages, and count provenance for each prioritization stage.",
        },
        {
            "figure": "Fig 4",
            "panel": "B",
            "title": "Multi-objective support scores",
            "source_data_file": "Figure_4_panel_b_source_data.csv",
            "description": "Median component scores with IQR across prioritization stages. Condition fidelity included when available.",
        },
        {
            "figure": "Fig 4",
            "panel": "C",
            "title": "Composite-score enrichment",
            "source_data_file": "Figure_4_panel_c_source_data.csv",
            "description": "Median composite/final score with IQR across prioritization stages.",
        },
        {
            "figure": "S13 Fig",
            "panel": "A",
            "title": "Ranking-scheme overlap",
            "source_data_file": "Supplementary_Figure_S13_panel_a_source_data.csv",
            "description": "Jaccard overlap matrix between alternative prioritization schemes; diagonal self-overlap is de-emphasized in the figure.",
        },
        {
            "figure": "S13 Fig",
            "panel": "B",
            "title": "Candidate recurrence across schemes",
            "source_data_file": "Supplementary_Figure_S13_panel_b_source_data.csv",
            "description": "Candidate counts and percentages by the number of prioritization schemes that recovered each candidate.",
        },
    ])


def readiness_score(checks: List[CheckResult]) -> Tuple[int, str]:
    fail = sum(c.status == "FAIL" for c in checks)
    warn = sum(c.status == "WARN" for c in checks)
    if fail:
        return max(60, 94 - fail * 6 - warn * 2), "FAIL"
    if warn:
        return max(90, 98 - warn), "WARN"
    return 100, "PASS"


def write_reports(
    dirs: OutputDirs,
    args: argparse.Namespace,
    inputs: InputPaths,
    discovery_records: List[DiscoveryRecord],
    checks: List[CheckResult],
    data: Dict[str, pd.DataFrame],
    files: Dict[str, str],
    component_metrics: List[Tuple[str, str]],
) -> None:
    score, status = readiness_score(checks)

    write_csv(pd.DataFrame([asdict(c) for c in checks]), dirs.reports / "step8_readiness_checks.csv")
    write_csv(pd.DataFrame([asdict(r) for r in discovery_records]), dirs.reports / "step8_discovery_records.csv")

    rejected = [r for r in discovery_records if r.decision in {"REJECT", "SKIP"}]
    write_csv(pd.DataFrame([asdict(r) for r in rejected]), dirs.reports / "step8_rejected_files.csv")

    report = []
    report.append("# OncoPep Step 8 readiness report\n\n")
    report.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}\n\n")
    report.append(f"Script version: `{SCRIPT_VERSION}`\n\n")
    report.append(f"Overall status: **{status}**\n\n")
    report.append(f"Estimated readiness score: **{score}/100**\n\n")
    report.append("## Scientific role\n\n")
    report.append("Step 8 supports multi-objective prioritization and prioritization robustness. It does not assess anticancer activity, selectivity, toxicity, stability, receptor binding, or therapeutic efficacy.\n\n")

    report.append("## Figure logic improvements in this run\n\n")
    report.append("- Fig 4A uses log-scale prioritization-stage reduction and emphasizes the descriptor-plausible to shortlist/final compression.\n")
    report.append("- Fig 4B plots descriptive medians with IQR error bars. Condition fidelity is included only if a numeric condition-fidelity or condition-match column is available in all prioritization stages.\n")
    report.append("- Fig 4C reports composite-score enrichment with sample sizes in x-axis labels, not inside bars.\n")
    report.append("- S13A places the primary ranking first, uses complete scheme labels, and de-emphasizes diagonal self-overlap.\n")
    report.append("- S13B reports candidate recurrence as counts and percentages with an explicit denominator.\n\n")

    report.append("## Component metrics plotted in Fig 4B\n\n")
    for col, lab in component_metrics:
        report.append(f"- {lab}: `{col}`\n")
    if not any(col == "condition_fidelity_score" for col, _ in component_metrics):
        report.append("- Condition fidelity was not plotted because no numeric condition-fidelity column was available in all three stages.\n")

    report.append("\n## Selected input paths\n\n")
    for k, v in asdict(inputs).items():
        report.append(f"- {k}: `{v}`\n")

    report.append("\n## Validation checks\n\n")
    for c in checks:
        report.append(f"- **{c.status}** `{c.name}`: {c.detail}\n")

    report.append("\n## Output files\n\n")
    for k, v in files.items():
        report.append(f"- {k}: `{v}`\n")

    report.append("\n## Gate\n\n")
    if status == "PASS" and score >= 98:
        report.append("This Step 8 package passes the ≥98/100 readiness gate for figure review.\n")
    else:
        report.append("This Step 8 package is generated but contains WARN/FAIL checks. Review before manuscript writing.\n")

    (dirs.reports / "step8_readiness_report.md").write_text("".join(report), encoding="utf-8")

    manifest = {
        "script": SCRIPT_NAME,
        "script_version": SCRIPT_VERSION,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "step8_root": str(args.step8_root),
        "project_root": str(args.project_root),
        "inputs": {k: str(v) if v is not None else None for k, v in asdict(inputs).items()},
        "data_shapes": {k: list(v.shape) for k, v in data.items() if isinstance(v, pd.DataFrame)},
        "outputs": files,
        "checks": [asdict(c) for c in checks],
        "component_metrics_plotted": [{"column": c, "label": l} for c, l in component_metrics],
        "readiness_score": score,
        "readiness_status": status,
        "claim_boundary": "Computational prioritization and robustness only; no experimental anticancer validation.",
    }
    (dirs.reports / "step8_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    readme = f"""# OncoPep Step 8 output package

Script version: {SCRIPT_VERSION}

## Scientific role

Step 8 supports multi-objective prioritization and prioritization robustness.
It does not assess anticancer activity, selectivity, toxicity, stability, receptor binding, or therapeutic efficacy.

## Main outputs

- main_figure/Figure_4_prioritization_redesigned.png/pdf/tiff
- supplementary_figures/Supplementary_Figure_S13_prioritization_robustness_redesigned.png/pdf/tiff
- source_data/Figure_4_* and Supplementary_Figure_S13_*
- reports/step8_readiness_report.md
- reports/step8_manifest.json

Descriptor-distribution details are retained as source-data/report material only to avoid duplication with Fig 3.
"""
    (dirs.reports / "README_step8_outputs.txt").write_text(readme, encoding="utf-8")

    (dirs.code / "requirements_step8_minimal.txt").write_text(
        "\n".join([
            "python>=3.9",
            f"numpy=={np.__version__}",
            f"pandas=={pd.__version__}",
            f"matplotlib=={matplotlib.__version__}",
            "",
        ]),
        encoding="utf-8",
    )

    try:
        src = Path(__file__).resolve()
        if src.exists():
            shutil.copy2(src, dirs.code / SCRIPT_NAME)
    except Exception:
        pass


def validate_data(data: Dict[str, pd.DataFrame]) -> List[CheckResult]:
    checks: List[CheckResult] = []

    for key in ["passed_df", "shortlist_df", "final_panel_df"]:
        if key not in data or len(data[key]) == 0:
            checks.append(CheckResult(key, "FAIL", "Required table missing or empty."))
        else:
            checks.append(CheckResult(key, "PASS", f"{len(data[key]):,} rows."))

    component_metrics = available_component_metrics(data)
    checks.append(CheckResult(
        "component_scores",
        "PASS" if len(component_metrics) >= 1 else "FAIL",
        f"Available component metrics: {component_metrics}",
    ))

    for key in ["passed_df", "shortlist_df", "final_panel_df"]:
        if "final_score" in data[key].columns and data[key]["final_score"].notna().any():
            checks.append(CheckResult(f"{key}:final_score", "PASS", "final_score available."))
        else:
            checks.append(CheckResult(f"{key}:final_score", "FAIL", "final_score unavailable."))

    if "stability_df" in data and {"scheme_a", "scheme_b", "jaccard_overlap"}.issubset(data["stability_df"].columns):
        checks.append(CheckResult("stability_df", "PASS", "scheme overlap columns available."))
    else:
        checks.append(CheckResult("stability_df", "WARN", "scheme overlap data unavailable or incomplete."))

    if "recurrence_df" in data and "scheme_recurrence_n" in data["recurrence_df"].columns:
        checks.append(CheckResult("recurrence_df", "PASS", "candidate recurrence column available."))
    else:
        checks.append(CheckResult("recurrence_df", "WARN", "candidate recurrence data unavailable or incomplete."))

    return checks


def main(argv: Optional[Sequence[str]] = None) -> int:
    args = parse_args(argv)
    dirs = ensure_dirs(args.step8_root)

    print(f"Hard-fix version: {SCRIPT_VERSION}")
    if args.unknown_args_ignored:
        print(f"Ignored unknown/Jupyter arguments: {args.unknown_args_ignored}")

    inputs, discovery_records = discover_inputs(args)
    data, checks = load_inputs(args, inputs)

    data, score_checks = ensure_score_columns(data)
    data, desc_checks = ensure_descriptor_columns(data)
    checks.extend(score_checks)
    checks.extend(desc_checks)

    validation_checks = validate_data(data)
    checks.extend(validation_checks)

    if any(c.status == "FAIL" for c in validation_checks):
        msg = [
            "\nERROR SUMMARY",
            "Step 8 validation failed after safe input loading.",
            "Selected inputs:",
        ]
        for k, v in asdict(inputs).items():
            msg.append(f"  {k}: {v}")
        msg.append("\nValidation checks:")
        for c in checks:
            msg.append(f"  [{c.status}] {c.name}: {c.detail}")
        msg.append("\nUse explicit --passed-file, --shortlist-file, --stability-file, and --recurrence-file paths.")
        raise RuntimeError("\n".join(msg))

    counts = stage_count_table(data, args)
    component_metrics = available_component_metrics(data)
    support_summary = summarize_scores(data, component_metrics, args)
    final_summary = final_score_summary(data, args)
    descriptor_summary = descriptor_distribution_summary(data)

    stability_mat, stability_long = stability_matrix_source(data.get("stability_df", pd.DataFrame()))
    recurrence_counts = recurrence_source(data.get("recurrence_df", pd.DataFrame()))

    files: Dict[str, str] = {}

    main_sources = {
        "Figure_4_panel_a_source_data": counts,
        "Figure_4_panel_b_source_data": support_summary,
        "Figure_4_panel_c_source_data": final_summary,
    }
    for name, df in main_sources.items():
        path = dirs.source_data / f"{name}.csv"
        write_csv(df, path)
        files[name] = str(path)

    fig4_all = combine_source_data({
        "Figure_4_panel_a": counts,
        "Figure_4_panel_b": support_summary,
        "Figure_4_panel_c": final_summary,
    })
    path = dirs.source_data / "Figure_4_source_data_all_panels.csv"
    write_csv(fig4_all, path)
    files["Figure_4_source_data_all_panels"] = str(path)

    supp_sources = {
        "Supplementary_Figure_S13_panel_a_source_data": stability_long,
        "Supplementary_Figure_S13_panel_b_source_data": recurrence_counts,
    }
    for name, df in supp_sources.items():
        path = dirs.source_data / f"{name}.csv"
        write_csv(df, path)
        files[name] = str(path)

    s13_all = combine_source_data({
        "Supplementary_Figure_S13_panel_a": stability_long,
        "Supplementary_Figure_S13_panel_b": recurrence_counts,
    })
    path = dirs.source_data / "Supplementary_Figure_S13_source_data_all_panels.csv"
    write_csv(s13_all, path)
    files["Supplementary_Figure_S13_source_data_all_panels"] = str(path)

    extra_sources = {
        "step8_prioritization_summary": fig4_all,
        "step8_selection_stage_counts": counts,
        "step8_candidate_support_summary": support_summary,
        "step8_scheme_overlap_summary": stability_long,
        "step8_candidate_recurrence_summary": recurrence_counts,
        "step8_descriptor_distribution_table": descriptor_summary,
        "step8_panel_source_data_mapping": build_panel_mapping(),
    }
    for name, df in extra_sources.items():
        path = dirs.source_data / f"{name}.csv"
        write_csv(df, path)
        files[name] = str(path)

    fig4_base = dirs.main_figure / "Figure_4_prioritization_redesigned"
    plot_figure4(counts, support_summary, final_summary, fig4_base, args)
    files["Figure_4_png"] = str(fig4_base.with_suffix(".png"))
    files["Figure_4_pdf"] = str(fig4_base.with_suffix(".pdf"))
    files["Figure_4_tiff"] = str(fig4_base.with_suffix(".tiff"))

    s13_base = dirs.supplementary_figures / "Supplementary_Figure_S13_prioritization_robustness_redesigned"
    plot_s13(stability_mat, recurrence_counts, s13_base, args)
    files["Supplementary_Figure_S13_png"] = str(s13_base.with_suffix(".png"))
    files["Supplementary_Figure_S13_pdf"] = str(s13_base.with_suffix(".pdf"))
    files["Supplementary_Figure_S13_tiff"] = str(s13_base.with_suffix(".tiff"))

    write_reports(dirs, args, inputs, discovery_records, checks, data, files, component_metrics)
    score, status = readiness_score(checks)

    print("\nOncoPep Step 8 package generated.")
    print(f"Root: {dirs.root}")
    print(f"Readiness status: {status}; estimated score: {score}/100")
    print(f"Main figure: {fig4_base.with_suffix('.png')}")
    print(f"Supplementary figure: {s13_base.with_suffix('.png')}")
    print(f"Readiness report: {dirs.reports / 'step8_readiness_report.md'}")
    if status != "PASS" or score < 98:
        print("WARNING: Package generated but contains WARN checks. Review readiness report before writing.")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
main([
    "--step8-root", "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_08",
    "--project-root", "/home/data3/Moe/nature_computational_peponco",
    "--passed-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_12_full_ranked_passed_pool.csv",
    "--shortlist-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_main/table_8_1_shortlist_main.csv",
    "--stability-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_5_ranking_scheme_stability.csv",
    "--recurrence-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_6_candidate_recurrence_across_schemes.csv",
    "--generated-count", "10840",
    "--eligible-count", "10291",
    "--passed-count", "10237",
    "--shortlist-count", "24",
    "--final-count", "12",
])

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 8 — PLOS Computational Biology prioritization figure package.

SCRIPT_VERSION:
v11_2026_07_07_plos_step8_plausibility_and_s13_rescue

Scientific role:
  Step 8 = multi-objective prioritization and prioritization robustness.
  This script does NOT repeat Step 7 descriptor-support or nearest-neighbor tail-risk figures.
  Descriptor distributions are exported only as source-data/report tables when useful.

Main outputs:
  Fig 4. Multi-objective prioritization of generated OncoPep candidates.
  S13 Fig. Robustness of prioritization schemes.

Major v11 fixes:
  - Restores Plausibility in Fig 4B when plausibility data exist under any supported alias.
  - Never silently removes expected metrics; exports step8_missing_metric_report.csv.
  - Defaults to manuscript-stage counts 10,840 -> 10,291 -> 10,237 -> 24 -> 12 unless explicitly overridden.
  - Prevents plotted NaN values in S13A; missing values are masked and labeled with an em dash.
  - Reconstructs long-format or matrix-format scheme overlap robustly and symmetrically.
  - Falls back to a compact table-like robustness panel if the scheme-overlap matrix is too sparse.
  - Retains Jupyter-safe argument parsing and rejects /run/user kernel JSON files.
"""

from __future__ import annotations

import argparse
import json
import math
import os
import shutil
import sys
import traceback
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import LogFormatterMathtext, LogLocator, NullFormatter


SCRIPT_NAME = "OncoPep_step8_PLOS_prioritization_final.py"
SCRIPT_VERSION = "v11_2026_07_07_plos_step8_plausibility_and_s13_rescue"

DEFAULT_PROJECT_ROOT = Path("/home/data3/Moe/nature_computational_peponco")
DEFAULT_STEP8_ROOT = DEFAULT_PROJECT_ROOT / "PLOS" / "plos_comp" / "step_08"

SAFE_TABLE_EXTENSIONS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".pq"}
BLOCKED_EXTENSIONS = {".json", ".png", ".jpg", ".jpeg", ".pdf", ".tif", ".tiff", ".py", ".ipynb", ".md", ".log"}
BLOCKED_PATH_KEYWORDS = [
    "/run/user/",
    "/jupyter/runtime/",
    ".ipynb_checkpoints",
    "__pycache__",
    "/.git/",
    "kernel-",
]
BLOCKED_NAME_KEYWORDS = [
    "kernel", "runtime", "manifest", "readiness", "config", "summary_all",
    "source_data_all_panels", "panel_source", "split_count_validation_audit",
    "validation_audit", "count_audit", "readme", "requirements", "run_step8_debug",
]

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
    "amber": "#E69F00",
    "legend_bg": "#F7F7F7",
    "legend_edge": "#CCCCCC",
    "missing": "#EFEFEF",
}

STAGE_COLORS = {
    "Generated": PLOS["light"],
    "QC-passed": PLOS["amber"],
    "Descriptor-plausible": PLOS["blue"],
    "Shortlist": PLOS["brown"],
    "Final panel": PLOS["coral"],
}

GROUP_COLORS = {
    "Descriptor-plausible": PLOS["blue"],
    "Shortlist": PLOS["brown"],
    "Final panel": PLOS["coral"],
}

HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "oncopep_step8_overlap",
    [PLOS["white"], "#DDEFE4", PLOS["mint"], PLOS["blue"], PLOS["charcoal"]],
)
HEATMAP_CMAP.set_bad(PLOS["missing"])

EXPECTED_METRICS = [
    ("novelty_score", "Novelty"),
    ("plausibility_score", "Plausibility"),
    ("diversity_score", "Diversity"),
]

SCORE_ALIASES = {
    "novelty_score": [
        "novelty_score", "novelty", "exact_novelty_score", "novelty_component", "nov_score",
        "s_nov", "S_nov", "score_novelty", "novelty_component_score",
    ],
    "plausibility_score": [
        "plausibility_score", "descriptor_plausibility_score", "descriptor_plausibility",
        "reference_range_plausibility", "reference_range_plausibility_score",
        "descriptor_support_score", "descriptor_support", "plausibility", "plaus_score",
        "s_plaus", "S_plaus", "score_plausibility", "plausibility_component",
        "plausibility_component_score", "property_plausibility", "property_plausibility_score",
        "physicochemical_plausibility", "physicochemical_plausibility_score",
        "within_reference_range_score", "reference_support_score",
    ],
    "condition_fidelity_score": [
        "condition_fidelity_score", "surrogate_condition_fidelity_score", "condition_score",
        "condition_fidelity", "condition_match_score", "condition_hit_rate",
        "condition_match_rate", "surrogate_condition_hit_rate", "condition_support_score",
        "s_cond", "S_cond", "score_condition", "cond_score",
    ],
    "diversity_score": [
        "diversity_score", "mmr_diversity_score", "diversity", "diversity_component",
        "div_score", "s_div", "S_div", "score_diversity", "diversity_component_score",
    ],
    "final_score": [
        "final_score", "composite_score", "prioritization_score", "rank_score",
        "score_final", "s_final", "S_final", "total_score", "overall_score",
    ],
    "rank": ["rank", "final_rank", "priority_rank", "shortlist_rank", "ranking", "order"],
    "sequence": ["sequence", "peptide", "seq", "aa_sequence", "peptide_sequence"],
}

DESCRIPTOR_ALIASES = {
    "length": ["length", "peptide_length", "seq_len", "sequence_length"],
    "net_charge": ["net_charge", "net_charge_pH7", "net_charge_ph7", "charge", "charge_proxy", "net_charge_proxy"],
    "hydrophobicity": ["hydrophobicity", "mean_hydropathy", "mean_hydrophobicity", "hydropathy", "kd_hydrophobicity"],
    "entropy": ["entropy", "shannon_entropy", "sequence_entropy"],
}

SCHEME_ALIASES = {
    "scheme_a": ["scheme_a", "scheme1", "row_scheme", "source_scheme", "scheme_i", "weighting_a", "ranking_scheme_a"],
    "scheme_b": ["scheme_b", "scheme2", "col_scheme", "target_scheme", "scheme_j", "weighting_b", "ranking_scheme_b"],
    "jaccard_overlap": ["jaccard_overlap", "jaccard", "overlap", "overlap_score", "scheme_overlap", "similarity"],
    "scheme_recurrence_n": ["scheme_recurrence_n", "scheme_recovery_count", "schemes_recovering_candidate", "num_schemes", "n_schemes", "scheme_count", "recurrence_n"],
    "candidate_count": ["candidate_count", "count", "n_candidates", "frequency", "n"],
}


@dataclass
class InputPaths:
    generated_file: Optional[Path] = None
    eligible_file: Optional[Path] = None
    passed_file: Optional[Path] = None
    shortlist_file: Optional[Path] = None
    final_file: Optional[Path] = None
    reference_file: Optional[Path] = None
    stability_file: Optional[Path] = None
    recurrence_file: Optional[Path] = None


@dataclass
class OutputDirs:
    root: Path
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


@dataclass
class CheckResult:
    name: str
    status: str
    detail: str


@dataclass
class DiscoveryRecord:
    role: str
    path: str
    decision: str
    reason: str
    score: int = 0


def is_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        shell = get_ipython()
        return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False


def is_jupyter_runtime_arg(x: str) -> bool:
    s = str(x)
    return "/run/user/" in s or "/jupyter/runtime/" in s or "kernel-" in s or s.endswith(".json")


def clean_argv(argv: Optional[Sequence[str]]) -> List[str]:
    raw = list(sys.argv[1:] if argv is None else argv)
    cleaned: List[str] = []
    skip_next = False
    for i, token in enumerate(raw):
        if skip_next:
            skip_next = False
            continue
        tok = str(token)
        if tok in {"-f", "--f", "--file"} and i + 1 < len(raw) and is_jupyter_runtime_arg(str(raw[i + 1])):
            skip_next = True
            continue
        if is_jupyter_runtime_arg(tok):
            continue
        cleaned.append(tok)
    return cleaned


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Generate OncoPep Step 8 PLOS prioritization and robustness figure package.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
        allow_abbrev=False,
    )
    p.add_argument("--step8-root", type=Path, default=DEFAULT_STEP8_ROOT)
    p.add_argument("--project-root", type=Path, default=DEFAULT_PROJECT_ROOT)
    p.add_argument("--search-root", action="append", type=Path, default=[])
    p.add_argument("--auto-discover", action="store_true", help="Enable safe auto-discovery for missing optional files.")
    p.add_argument("--demo-data", action="store_true", help="Use synthetic demo data. Never use demo output for manuscript.")

    p.add_argument("--generated-file", type=Path, default=None)
    p.add_argument("--eligible-file", type=Path, default=None)
    p.add_argument("--passed-file", type=Path, default=None)
    p.add_argument("--shortlist-file", type=Path, default=None)
    p.add_argument("--final-file", type=Path, default=None)
    p.add_argument("--reference-file", type=Path, default=None)
    p.add_argument("--stability-file", type=Path, default=None)
    p.add_argument("--recurrence-file", type=Path, default=None)

    # Manuscript-locked defaults prevent the accidental 39,402 generated-row display.
    p.add_argument("--generated-count", type=int, default=10840)
    p.add_argument("--eligible-count", type=int, default=10291)
    p.add_argument("--passed-count", type=int, default=10237)
    p.add_argument("--shortlist-count", type=int, default=24)
    p.add_argument("--final-count", type=int, default=12)

    p.add_argument("--png-dpi", type=int, default=300)
    p.add_argument("--tiff-dpi", type=int, default=600)
    p.add_argument("--bootstrap-n", type=int, default=1000)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--quiet", action="store_true")
    cleaned = clean_argv(argv)
    args, unknown = p.parse_known_args(cleaned)
    args.unknown_args_ignored = unknown
    return args


def ensure_dirs(root: Path) -> OutputDirs:
    dirs = OutputDirs(
        root=root,
        main_figure=root / "main_figure",
        supplementary_figures=root / "supplementary_figures",
        source_data=root / "source_data",
        reports=root / "reports",
        code=root / "code",
    )
    for d in asdict(dirs).values():
        Path(d).mkdir(parents=True, exist_ok=True)
    return dirs


def path_is_blocked(path: Path) -> Tuple[bool, str]:
    s = str(path).replace("\\", "/").lower()
    name = path.name.lower()
    if path.suffix.lower() in BLOCKED_EXTENSIONS:
        return True, f"blocked extension {path.suffix}"
    for kw in BLOCKED_PATH_KEYWORDS:
        if kw.lower() in s:
            return True, f"blocked path keyword {kw}"
    for kw in BLOCKED_NAME_KEYWORDS:
        if kw.lower() in name:
            return True, f"blocked name keyword {kw}"
    if path.suffix.lower() not in SAFE_TABLE_EXTENSIONS:
        return True, f"not an allowed table extension: {path.suffix}"
    return False, "allowed"


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    blocked, reason = path_is_blocked(path)
    if blocked:
        raise ValueError(f"Rejected unsafe/non-table input: {path} ({reason})")
    if not path.exists():
        raise FileNotFoundError(f"Input table does not exist: {path}")
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        try:
            return pd.read_csv(path, sep="\t")
        except Exception:
            return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table format: {path}")


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def find_first_col(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    lower_to_original = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a in df.columns:
            return a
        key = str(a).strip().lower()
        if key in lower_to_original:
            return lower_to_original[key]
    return None


def safe_to_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def candidate_sort(df: pd.DataFrame) -> pd.DataFrame:
    rank_col = find_first_col(df, SCORE_ALIASES["rank"])
    if rank_col:
        tmp = df.copy()
        tmp["_rank_tmp"] = safe_to_numeric(tmp[rank_col])
        return tmp.sort_values("_rank_tmp", na_position="last").drop(columns="_rank_tmp")
    score_col = find_first_col(df, SCORE_ALIASES["final_score"])
    if score_col:
        tmp = df.copy()
        tmp["_score_tmp"] = safe_to_numeric(tmp[score_col])
        return tmp.sort_values("_score_tmp", ascending=False, na_position="last").drop(columns="_score_tmp")
    return df


def table_looks_candidate_like(df: pd.DataFrame, role: str) -> Tuple[bool, str]:
    if df is None or len(df) == 0:
        return False, "zero rows"
    cols = {str(c).lower() for c in df.columns}
    peptide_like = any(c in cols for c in ["sequence", "peptide", "peptide_sequence", "seq"])
    score_like = any(any(a.lower() == c for c in cols) for aliases in SCORE_ALIASES.values() for a in aliases)
    if role in {"passed", "shortlist", "final"} and not peptide_like and not score_like:
        return False, "no peptide-like or score-like columns"
    if role == "final" and len(df) > 500:
        return False, "too many rows for final panel"
    return True, "accepted"


def score_file_for_role(path: Path, role: str) -> int:
    name = path.name.lower()
    full = str(path).replace("\\", "/").lower()
    score = 0
    for d in ["step8_v1/tables_main", "step8_v1/tables_supplementary", "step8/tables_main", "step8/tables_supplementary", "step6_v5/tables", "step2/tables"]:
        if d in full:
            score += 12
    keywords = {
        "generated": ["generated_sequences", "all_generated", "generated"],
        "eligible": ["qc_passed", "eligible", "valid_candidates"],
        "passed": ["full_ranked_passed_pool", "passed_pool", "descriptor_plausible", "plausible", "table_s8_12"],
        "shortlist": ["shortlist", "short_list", "top24", "table_8_1"],
        "final": ["final_panel", "final_candidates", "final_12"],
        "reference": ["train_only_novelty_reference", "train_reference", "reference"],
        "stability": ["ranking_scheme_stability", "scheme_stability", "scheme_overlap", "table_s8_5"],
        "recurrence": ["candidate_recurrence", "recurrence_across_schemes", "table_s8_6"],
    }
    for kw in keywords.get(role, []):
        if kw in name:
            score += 20
    if role in {"passed", "shortlist", "final"}:
        for kw in ["split", "count", "audit", "validation", "readiness", "manifest", "source_data_all"]:
            if kw in name:
                score -= 100
    return score


def safe_discover_file(role: str, explicit: Optional[Path], roots: Sequence[Path]) -> Tuple[Optional[Path], List[DiscoveryRecord]]:
    records: List[DiscoveryRecord] = []
    if explicit is not None:
        p = Path(explicit)
        blocked, reason = path_is_blocked(p)
        if blocked:
            records.append(DiscoveryRecord(role, str(p), "REJECT", f"explicit path rejected: {reason}", -999))
            return None, records
        records.append(DiscoveryRecord(role, str(p), "SELECT", "explicit path supplied", 999))
        return p, records
    patterns = {
        "generated": ["*generated*.csv", "*all_generated*.csv"],
        "eligible": ["*qc*pass*.csv", "*eligible*.csv", "*valid*.csv"],
        "passed": ["*full_ranked_passed_pool*.csv", "*passed_pool*.csv", "*descriptor*plaus*.csv", "*plausible*.csv", "*table_s8_12*.csv"],
        "shortlist": ["*shortlist*.csv", "*table_8_1*.csv"],
        "final": ["*final_panel*.csv", "*final_candidates*.csv", "*final_12*.csv"],
        "reference": ["*train_only_novelty_reference*.csv", "*reference*.csv"],
        "stability": ["*ranking_scheme_stability*.csv", "*scheme*stability*.csv", "*scheme*overlap*.csv", "*table_s8_5*.csv"],
        "recurrence": ["*candidate_recurrence*.csv", "*recurrence*.csv", "*table_s8_6*.csv"],
    }
    candidates: List[Tuple[int, Path]] = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            records.append(DiscoveryRecord(role, str(root), "SKIP", "root does not exist", 0))
            continue
        unsafe_root = any(kw.lower() in str(root).replace("\\", "/").lower() for kw in ["/run/user/", "/jupyter/runtime/", ".ipynb_checkpoints", "__pycache__", "/.git/"])
        if unsafe_root:
            records.append(DiscoveryRecord(role, str(root), "SKIP", "unsafe root", -999))
            continue
        for pat in patterns.get(role, []):
            for p in root.rglob(pat):
                if not p.is_file():
                    continue
                blocked, reason = path_is_blocked(p)
                if blocked:
                    records.append(DiscoveryRecord(role, str(p), "REJECT", reason, -999))
                    continue
                score = score_file_for_role(p, role)
                if score <= 0:
                    records.append(DiscoveryRecord(role, str(p), "REJECT", "low role score", score))
                    continue
                candidates.append((score, p))
                records.append(DiscoveryRecord(role, str(p), "CANDIDATE", "safe candidate", score))
    if not candidates:
        return None, records
    candidates = sorted(candidates, key=lambda x: (x[0], -len(str(x[1]))), reverse=True)
    for score, p in candidates[:20]:
        try:
            preview = read_table(p)
            ok, reason = table_looks_candidate_like(preview, role)
            if ok:
                records.append(DiscoveryRecord(role, str(p), "SELECT", reason, score))
                return p, records
            records.append(DiscoveryRecord(role, str(p), "REJECT", reason, score))
        except Exception as e:
            records.append(DiscoveryRecord(role, str(p), "REJECT", f"cannot read preview: {e}", score))
    return None, records


def discover_inputs(args: argparse.Namespace) -> Tuple[InputPaths, List[DiscoveryRecord]]:
    roots = [
        args.project_root / "step8_v1" / "tables_main",
        args.project_root / "step8_v1" / "tables_supplementary",
        args.project_root / "step6_v5" / "tables",
        args.project_root / "step2" / "tables",
        args.step8_root,
    ] + list(args.search_root)
    records: List[DiscoveryRecord] = []
    def get(role: str, explicit: Optional[Path]) -> Optional[Path]:
        p, rec = safe_discover_file(role, explicit, roots if args.auto_discover or explicit is not None else [])
        records.extend(rec)
        return p
    inputs = InputPaths(
        generated_file=get("generated", args.generated_file),
        eligible_file=get("eligible", args.eligible_file),
        passed_file=get("passed", args.passed_file),
        shortlist_file=get("shortlist", args.shortlist_file),
        final_file=get("final", args.final_file),
        reference_file=get("reference", args.reference_file),
        stability_file=get("stability", args.stability_file),
        recurrence_file=get("recurrence", args.recurrence_file),
    )
    return inputs, records


def generate_demo_data(args: argparse.Namespace) -> Dict[str, pd.DataFrame]:
    rng = np.random.default_rng(args.seed)
    n_pass = 10237
    n_short = 24
    n_final = 12
    def make(n: int, boost: float) -> pd.DataFrame:
        return pd.DataFrame({
            "sequence": [f"PEPTIDE{i:05d}" for i in range(n)],
            "novelty_score": np.clip(rng.normal(0.94 + boost, 0.015, n), 0, 1),
            "descriptor_plausibility_score": np.clip(rng.normal(0.55 + 0.23 * boost * 10, 0.13, n), 0, 1),
            "diversity_score": np.clip(rng.normal(0.88 + boost, 0.04, n), 0, 1),
            "final_score": np.clip(rng.normal(0.78 + boost, 0.06, n), 0, 1),
            "rank": np.arange(1, n + 1),
        })
    stability = pd.DataFrame({
        "scheme_a": ["Primary", "Primary", "Primary", "Novelty-weighted", "Novelty-weighted", "Plausibility-weighted"],
        "scheme_b": ["Novelty-weighted", "Plausibility-weighted", "Diversity-weighted", "Plausibility-weighted", "Diversity-weighted", "Diversity-weighted"],
        "jaccard_overlap": [0.66, 0.60, 0.50, 0.41, 0.66, 0.30],
    })
    recurrence = pd.DataFrame({"scheme_recurrence_n": [1]*14 + [2]*7 + [3]*8 + [4]*11})
    return {
        "passed_df": make(n_pass, 0.00),
        "shortlist_df": make(n_short, 0.08),
        "final_panel_df": make(n_final, 0.09),
        "stability_df": stability,
        "recurrence_df": recurrence,
    }


def load_optional_table(path: Optional[Path], key: str, checks: List[CheckResult]) -> Optional[pd.DataFrame]:
    if path is None:
        checks.append(CheckResult(key, "WARN", "No input path supplied or discovered."))
        return None
    try:
        df = normalize_cols(read_table(path))
        checks.append(CheckResult(key, "PASS", f"Loaded {len(df):,} rows from {path}"))
        return df
    except Exception as e:
        checks.append(CheckResult(key, "FAIL", f"Could not read {path}: {e}"))
        return None


def infer_final_from_shortlist(shortlist: pd.DataFrame, n: int) -> Tuple[pd.DataFrame, CheckResult]:
    df = normalize_cols(shortlist)
    flag = find_first_col(df, ["is_final_panel", "final_panel", "selected_final", "is_final", "final_selection"])
    if flag:
        vals = df[flag].astype(str).str.lower().str.strip()
        out = df[vals.isin(["1", "true", "yes", "y", "final", "selected"])].copy()
        if len(out) > 0:
            return out, CheckResult("final_panel_df", "PASS", f"Inferred final panel from `{flag}` with {len(out):,} rows.")
    out = candidate_sort(df).head(n).copy()
    return out, CheckResult("final_panel_df", "WARN", f"No final-file supplied; inferred final panel from top {n} shortlist rows.")


def load_inputs(args: argparse.Namespace, inputs: InputPaths) -> Tuple[Dict[str, pd.DataFrame], List[CheckResult]]:
    if args.demo_data:
        data = generate_demo_data(args)
        return data, [CheckResult("demo_data", "WARN", "Synthetic demo data were generated. Do not use demo output for manuscript.")]
    checks: List[CheckResult] = []
    data: Dict[str, pd.DataFrame] = {}
    for attr, key in {
        "generated_file": "gen_df",
        "eligible_file": "eligible_df",
        "passed_file": "passed_df",
        "shortlist_file": "shortlist_df",
        "final_file": "final_panel_df",
        "reference_file": "ref_df",
        "stability_file": "stability_df",
        "recurrence_file": "recurrence_df",
    }.items():
        df = load_optional_table(getattr(inputs, attr), key, checks)
        if df is not None:
            data[key] = df
    if "passed_df" not in data:
        raise RuntimeError("Required passed/descriptor-plausible pool is missing. Supply --passed-file.")
    if "shortlist_df" not in data:
        raise RuntimeError("Required shortlist table is missing. Supply --shortlist-file.")
    if "final_panel_df" not in data:
        final_df, ck = infer_final_from_shortlist(data["shortlist_df"], args.final_count)
        data["final_panel_df"] = final_df
        checks.append(ck)
    return data, checks


def clean_numeric(series: pd.Series) -> np.ndarray:
    arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return arr[np.isfinite(arr)]


def median_iqr(arr: np.ndarray) -> Tuple[float, float, float, int]:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan, np.nan, 0
    q1, med, q3 = np.percentile(arr, [25, 50, 75])
    return float(med), float(q1), float(q3), int(len(arr))


def bootstrap_ci(arr: np.ndarray, n_boot: int, seed: int) -> Tuple[float, float]:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan
    if len(arr) == 1:
        return float(arr[0]), float(arr[0])
    rng = np.random.default_rng(seed)
    stats = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        stats[i] = np.median(rng.choice(arr, size=len(arr), replace=True))
    return tuple(map(float, np.percentile(stats, [2.5, 97.5])))


def find_numeric_condition_columns(df: pd.DataFrame) -> List[str]:
    out = []
    for c in df.columns:
        lc = str(c).lower()
        if ("condition" in lc or "cond" in lc) and ("match" in lc or "hit" in lc or "fidelity" in lc or "score" in lc):
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().any():
                out.append(c)
    return out


def ensure_score_columns(data: Dict[str, pd.DataFrame]) -> Tuple[Dict[str, pd.DataFrame], List[CheckResult], pd.DataFrame]:
    checks: List[CheckResult] = []
    missing_rows = []
    required_keys = ["passed_df", "shortlist_df", "final_panel_df"]
    for key in required_keys:
        df = normalize_cols(data[key]).copy()
        for canon, aliases in SCORE_ALIASES.items():
            found = find_first_col(df, aliases)
            if found is not None:
                df[canon] = pd.to_numeric(df[found], errors="coerce")
                n_valid = int(df[canon].notna().sum())
                if n_valid > 0:
                    checks.append(CheckResult(f"{key}:{canon}", "PASS", f"Mapped `{found}` to `{canon}` with {n_valid:,} numeric values."))
                else:
                    checks.append(CheckResult(f"{key}:{canon}", "WARN", f"Mapped `{found}` to `{canon}` but numeric values are all missing."))
                    missing_rows.append({"table": key, "metric": canon, "status": "all_nan_after_numeric_conversion", "matched_column": found})
            elif canon == "condition_fidelity_score":
                condition_cols = find_numeric_condition_columns(df)
                if condition_cols:
                    df[canon] = df[condition_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
                    checks.append(CheckResult(f"{key}:{canon}", "PASS", f"Computed from condition columns: {condition_cols}."))
                else:
                    missing_rows.append({"table": key, "metric": canon, "status": "not_found_optional", "matched_column": ""})
            elif canon in [m[0] for m in EXPECTED_METRICS] or canon == "final_score":
                status = "not_found_required" if canon in [m[0] for m in EXPECTED_METRICS] else "not_found_final_score"
                missing_rows.append({"table": key, "metric": canon, "status": status, "matched_column": ""})
                checks.append(CheckResult(f"{key}:{canon}", "WARN", f"No column found for `{canon}`."))
        component_cols = ["novelty_score", "plausibility_score", "condition_fidelity_score", "diversity_score"]
        available = [c for c in component_cols if c in df.columns and df[c].notna().any()]
        if "final_score" not in df.columns or not df["final_score"].notna().any():
            if available:
                df["final_score"] = df[available].mean(axis=1)
                checks.append(CheckResult(f"{key}:final_score", "WARN", f"Computed final_score as mean of available components: {available}."))
            else:
                rank_col = find_first_col(df, SCORE_ALIASES["rank"])
                if rank_col:
                    rank = pd.to_numeric(df[rank_col], errors="coerce")
                    denom = max(float(rank.max() - rank.min()), 1.0)
                    df["final_score"] = 1.0 - ((rank - rank.min()) / denom)
                    checks.append(CheckResult(f"{key}:final_score", "WARN", f"Computed final_score from rank column `{rank_col}`."))
                else:
                    df["final_score"] = np.linspace(1.0, 0.5, len(df))
                    checks.append(CheckResult(f"{key}:final_score", "FAIL", "No score or rank columns found; assigned placeholder final_score."))
        data[key] = df
    missing_report = pd.DataFrame(missing_rows)
    return data, checks, missing_report


def ensure_descriptor_columns(data: Dict[str, pd.DataFrame]) -> Tuple[Dict[str, pd.DataFrame], List[CheckResult]]:
    checks: List[CheckResult] = []
    for key, df in list(data.items()):
        if not isinstance(df, pd.DataFrame):
            continue
        df2 = normalize_cols(df).copy()
        for canon, aliases in DESCRIPTOR_ALIASES.items():
            found = find_first_col(df2, aliases)
            if found is not None:
                df2[canon] = pd.to_numeric(df2[found], errors="coerce")
                checks.append(CheckResult(f"{key}:{canon}", "PASS", f"Mapped `{found}` to descriptor `{canon}`."))
        data[key] = df2
    return data, checks


def metric_availability(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for metric_col, metric_label in EXPECTED_METRICS + [("condition_fidelity_score", "Condition fidelity")]:
        for stage, key in [("Descriptor-plausible", "passed_df"), ("Shortlist", "shortlist_df"), ("Final panel", "final_panel_df")]:
            present = metric_col in data[key].columns
            n_numeric = int(pd.to_numeric(data[key][metric_col], errors="coerce").notna().sum()) if present else 0
            rows.append({
                "metric_column": metric_col,
                "metric": metric_label,
                "stage": stage,
                "column_present": present,
                "numeric_n": n_numeric,
                "status": "available" if n_numeric > 0 else "missing_or_non_numeric",
            })
    return pd.DataFrame(rows)


def selected_component_metrics(data: Dict[str, pd.DataFrame]) -> Tuple[List[Tuple[str, str]], List[CheckResult]]:
    checks: List[CheckResult] = []
    selected = []
    availability = metric_availability(data)
    for col, label in EXPECTED_METRICS:
        sub = availability[availability["metric_column"] == col]
        ok_all = bool((sub["numeric_n"] > 0).all())
        if ok_all:
            selected.append((col, label))
            checks.append(CheckResult(f"metric:{label}", "PASS", f"{label} is available in all three prioritization stages."))
        else:
            checks.append(CheckResult(f"metric:{label}", "FAIL" if label == "Plausibility" else "WARN", f"{label} is not available in all stages; see step8_missing_metric_report.csv."))
    # condition fidelity is optional; include only if all stages have it.
    sub = availability[availability["metric_column"] == "condition_fidelity_score"]
    if len(sub) > 0 and bool((sub["numeric_n"] > 0).all()):
        selected.append(("condition_fidelity_score", "Condition fidelity"))
        checks.append(CheckResult("metric:Condition fidelity", "PASS", "Condition fidelity is available in all three stages and included."))
    else:
        checks.append(CheckResult("metric:Condition fidelity", "INFO", "Condition fidelity unavailable or incomplete; not plotted in Fig 4B."))
    if not selected:
        selected = [("final_score", "Composite score")]
        checks.append(CheckResult("metric:fallback", "FAIL", "No expected component metrics were complete; using composite-score fallback."))
    return selected, checks


def summarize_scores(data: Dict[str, pd.DataFrame], metrics: Sequence[Tuple[str, str]], args: argparse.Namespace) -> pd.DataFrame:
    rows = []
    for stage, key in [("Descriptor-plausible", "passed_df"), ("Shortlist", "shortlist_df"), ("Final panel", "final_panel_df")]:
        df = data[key]
        for col, label in metrics:
            if col not in df.columns:
                rows.append({"stage": stage, "metric": label, "metric_column": col, "n": 0, "median": np.nan, "q1": np.nan, "q3": np.nan, "iqr_low": np.nan, "iqr_high": np.nan, "bootstrap_ci_low": np.nan, "bootstrap_ci_high": np.nan, "plotted_error_type": "IQR", "available": False})
                continue
            arr = clean_numeric(df[col])
            med, q1, q3, n = median_iqr(arr)
            lo, hi = bootstrap_ci(arr, args.bootstrap_n, args.seed + len(rows))
            rows.append({
                "stage": stage,
                "metric": label,
                "metric_column": col,
                "n": n,
                "median": med,
                "q1": q1,
                "q3": q3,
                "iqr_low": q1,
                "iqr_high": q3,
                "bootstrap_ci_low": lo,
                "bootstrap_ci_high": hi,
                "plotted_error_type": "IQR",
                "available": n > 0,
            })
    return pd.DataFrame(rows)


def final_score_summary(data: Dict[str, pd.DataFrame], args: argparse.Namespace) -> pd.DataFrame:
    return summarize_scores(data, [("final_score", "Composite score")], args)


def stage_count_table(args: argparse.Namespace) -> pd.DataFrame:
    generated_n = int(args.generated_count)
    rows = [
        ("Generated", "All generated OncoPep outputs", int(args.generated_count), "locked_or_explicit"),
        ("QC-passed", "Generated outputs passing syntax/QC filters", int(args.eligible_count), "locked_or_explicit"),
        ("Descriptor-plausible", "QC-passed outputs retained by descriptor plausibility filtering", int(args.passed_count), "locked_or_explicit"),
        ("Shortlist", "Diversity-aware shortlist", int(args.shortlist_count), "locked_or_explicit"),
        ("Final panel", "Final diversity-enriched panel", int(args.final_count), "locked_or_explicit"),
    ]
    out = pd.DataFrame(rows, columns=["stage", "definition", "count", "count_source"])
    out["fraction_of_generated"] = out["count"] / generated_n if generated_n else np.nan
    out["percent_of_generated"] = out["fraction_of_generated"] * 100.0
    return out


def descriptor_distribution_summary(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for stage, key in [("Descriptor-plausible", "passed_df"), ("Shortlist", "shortlist_df"), ("Final panel", "final_panel_df"), ("Reference", "ref_df")]:
        if key not in data:
            continue
        df = data[key]
        for desc, label in [("length", "Peptide length"), ("net_charge", "Net charge at pH 7"), ("hydrophobicity", "Mean hydrophobicity"), ("entropy", "Shannon entropy")]:
            if desc not in df.columns:
                continue
            arr = clean_numeric(df[desc])
            med, q1, q3, n = median_iqr(arr)
            rows.append({"stage": stage, "descriptor": label, "descriptor_column": desc, "n": n, "median": med, "q1": q1, "q3": q3, "min": float(np.min(arr)) if n else np.nan, "max": float(np.max(arr)) if n else np.nan, "mean": float(np.mean(arr)) if n else np.nan, "sd": float(np.std(arr, ddof=1)) if n > 1 else np.nan})
    return pd.DataFrame(rows)


def normalize_scheme_label(x: Any) -> str:
    s = str(x).strip()
    sl = s.lower().replace("_", " ").replace("-", " ")
    if any(t in sl for t in ["primary", "main", "default", "base", "balanced", "standard"]):
        return "Primary"
    if "novel" in sl:
        return "Novelty-weighted"
    if "plaus" in sl or "property" in sl or "descriptor" in sl:
        return "Plausibility-weighted"
    if "divers" in sl:
        return "Diversity-weighted"
    if "condition" in sl or "cond" in sl:
        return "Condition-weighted"
    return " ".join(w.capitalize() for w in sl.split())


def scheme_order(labels: Sequence[str]) -> List[str]:
    priority = ["Primary", "Novelty-weighted", "Plausibility-weighted", "Condition-weighted", "Diversity-weighted"]
    uniq: List[str] = []
    for x in labels:
        if x and x not in uniq:
            uniq.append(x)
    ordered = [x for x in priority if x in uniq]
    ordered += sorted([x for x in uniq if x not in ordered])
    return ordered


def standardize_scheme_table(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_cols(df.copy())
    for canon, aliases in SCHEME_ALIASES.items():
        found = find_first_col(df, aliases)
        if found is not None and canon not in df.columns:
            df[canon] = df[found]
    return df


def stability_matrix_source(stability_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[CheckResult]]:
    checks: List[CheckResult] = []
    if stability_df is None or len(stability_df) == 0:
        checks.append(CheckResult("stability_matrix", "WARN", "No scheme-overlap table available."))
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), checks
    df = standardize_scheme_table(stability_df)
    long_rows = []
    if {"scheme_a", "scheme_b", "jaccard_overlap"}.issubset(df.columns):
        tmp = df[["scheme_a", "scheme_b", "jaccard_overlap"]].copy()
        for _, r in tmp.iterrows():
            a = normalize_scheme_label(r["scheme_a"])
            b = normalize_scheme_label(r["scheme_b"])
            v = pd.to_numeric(pd.Series([r["jaccard_overlap"]]), errors="coerce").iloc[0]
            if pd.isna(v):
                continue
            long_rows.append({"scheme_a": a, "scheme_b": b, "jaccard_overlap": float(v)})
    else:
        # Matrix-like format: first text column may contain row labels, remaining numeric columns are schemes.
        tmp = df.copy()
        text_cols = [c for c in tmp.columns if tmp[c].dtype == object]
        if text_cols:
            row_col = text_cols[0]
            tmp = tmp.set_index(row_col)
        for i in tmp.index:
            a = normalize_scheme_label(i)
            for c in tmp.columns:
                b = normalize_scheme_label(c)
                v = pd.to_numeric(pd.Series([tmp.loc[i, c]]), errors="coerce").iloc[0]
                if pd.isna(v):
                    continue
                long_rows.append({"scheme_a": a, "scheme_b": b, "jaccard_overlap": float(v)})
    long_raw = pd.DataFrame(long_rows)
    if long_raw.empty:
        checks.append(CheckResult("stability_matrix", "WARN", "Could not parse any numeric scheme-overlap values."))
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), checks
    names = scheme_order(list(long_raw["scheme_a"]) + list(long_raw["scheme_b"]))
    mat = pd.DataFrame(np.nan, index=names, columns=names, dtype=float)
    for n in names:
        mat.loc[n, n] = 1.0
    for _, r in long_raw.iterrows():
        a = r["scheme_a"]
        b = r["scheme_b"]
        v = float(r["jaccard_overlap"])
        mat.loc[a, b] = v
        mat.loc[b, a] = v
    long = mat.reset_index(names="scheme_a").melt(id_vars="scheme_a", var_name="scheme_b", value_name="jaccard_overlap")
    long["is_diagonal"] = long["scheme_a"] == long["scheme_b"]
    long["is_missing"] = long["jaccard_overlap"].isna()
    long["interpretation"] = np.where(long["is_diagonal"], "self-overlap", np.where(long["is_missing"], "missing pair", "off-diagonal robustness overlap"))
    off = long[~long["is_diagonal"]]
    valid_frac = float(off["jaccard_overlap"].notna().mean()) if len(off) else 0.0
    n_schemes = len(names)
    if n_schemes < 3 or valid_frac < 0.5:
        checks.append(CheckResult("stability_matrix", "WARN", f"Scheme-overlap matrix is sparse: n_schemes={n_schemes}, off_diagonal_valid_fraction={valid_frac:.2f}."))
    else:
        checks.append(CheckResult("stability_matrix", "PASS", f"Parsed scheme-overlap matrix with {n_schemes} schemes and off-diagonal valid fraction {valid_frac:.2f}."))
    pair_table = long[~long["is_diagonal"]].copy()
    pair_table = pair_table[pair_table["scheme_a"] < pair_table["scheme_b"]].copy()
    return mat, long, pair_table, checks


def recurrence_source(recurrence_df: pd.DataFrame) -> Tuple[pd.DataFrame, List[CheckResult]]:
    checks: List[CheckResult] = []
    if recurrence_df is None or len(recurrence_df) == 0:
        checks.append(CheckResult("recurrence", "WARN", "No recurrence table available."))
        return pd.DataFrame(columns=["scheme_recurrence_n", "candidate_count", "fraction", "percent", "denominator_n"]), checks
    df = standardize_scheme_table(recurrence_df)
    if "scheme_recurrence_n" in df.columns and "candidate_count" in df.columns and len(df) <= 15:
        out = df[["scheme_recurrence_n", "candidate_count"]].copy()
        out["scheme_recurrence_n"] = pd.to_numeric(out["scheme_recurrence_n"], errors="coerce")
        out["candidate_count"] = pd.to_numeric(out["candidate_count"], errors="coerce")
        out = out.dropna().groupby("scheme_recurrence_n", as_index=False)["candidate_count"].sum()
    elif "scheme_recurrence_n" in df.columns:
        vals = pd.to_numeric(df["scheme_recurrence_n"], errors="coerce").dropna().astype(int)
        counts = vals.value_counts().sort_index()
        out = pd.DataFrame({"scheme_recurrence_n": counts.index, "candidate_count": counts.values})
    else:
        checks.append(CheckResult("recurrence", "WARN", "Could not identify scheme_recurrence_n column."))
        return pd.DataFrame(columns=["scheme_recurrence_n", "candidate_count", "fraction", "percent", "denominator_n"]), checks
    out = out.sort_values("scheme_recurrence_n")
    total = int(out["candidate_count"].sum())
    out["denominator_n"] = total
    out["fraction"] = out["candidate_count"] / total if total else np.nan
    out["percent"] = out["fraction"] * 100.0
    checks.append(CheckResult("recurrence", "PASS", f"Parsed candidate recurrence for {total} candidate records."))
    return out, checks


def combine_source_data(named_tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    all_cols: List[str] = []
    for name, df in named_tables.items():
        if df is None or len(df) == 0:
            continue
        tmp = df.copy()
        tmp.insert(0, "source_table", name)
        frames.append(tmp)
        for c in tmp.columns:
            if c not in all_cols:
                all_cols.append(c)
    if not frames:
        return pd.DataFrame()
    return pd.concat([f.reindex(columns=all_cols) for f in frames], ignore_index=True)


def set_publication_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "font.family": "DejaVu Sans",
        "font.size": 8.8,
        "axes.titlesize": 12.6,
        "axes.labelsize": 9.6,
        "xtick.labelsize": 8.2,
        "ytick.labelsize": 8.4,
        "legend.fontsize": 8.6,
        "axes.edgecolor": "#B8B8B8",
        "axes.labelcolor": PLOS["dark"],
        "xtick.color": PLOS["dark"],
        "ytick.color": PLOS["dark"],
        "text.color": PLOS["dark"],
        "axes.titlecolor": PLOS["dark"],
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def style_axis(ax: plt.Axes, grid_axis: str = "x") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=PLOS["grid"], linewidth=0.75)
    elif grid_axis == "both":
        ax.grid(True, color=PLOS["grid"], linewidth=0.75)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#B8B8B8")
        ax.spines[side].set_linewidth(0.9)
    ax.tick_params(width=0.8, length=3.0, colors=PLOS["dark"])


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.12, y: float = 1.09, size: int = 17) -> None:
    ax.text(x, y, label, transform=ax.transAxes, fontsize=size, fontweight="bold", va="bottom", ha="left", color=PLOS["dark"], clip_on=False)


def save_figure(fig: plt.Figure, out_base: Path, args: argparse.Namespace) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_base.with_suffix(".png"), dpi=args.png_dpi, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".tiff"), dpi=args.tiff_dpi, bbox_inches="tight")
    plt.close(fig)


def plot_figure4(counts: pd.DataFrame, support_summary: pd.DataFrame, final_score_sum: pd.DataFrame, fig4_base: Path, args: argparse.Namespace) -> None:
    set_publication_style()
    fig = plt.figure(figsize=(13.1, 4.55))
    gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1.22, 1.28, 0.95], wspace=0.47)

    # A: prioritization-stage reduction
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.17, y=1.04, size=17)
    order = ["Generated", "QC-passed", "Descriptor-plausible", "Shortlist", "Final panel"]
    df = counts.set_index("stage").reindex(order).reset_index()
    y = np.arange(len(df)) * 1.17
    colors = [STAGE_COLORS.get(s, PLOS["light"]) for s in df["stage"]]
    edges = ["#606060" if s in {"Shortlist", "Final panel"} else "none" for s in df["stage"]]
    lw = [0.75 if s in {"Shortlist", "Final panel"} else 0 for s in df["stage"]]
    ax.barh(y, df["count"], color=colors, edgecolor=edges, linewidth=lw, height=0.56)
    ax.set_yticks(y)
    ax.set_yticklabels(df["stage"])
    ax.invert_yaxis()
    ax.set_xscale("log")
    ax.set_xlim(8, max(float(df["count"].max()) * 1.65, 15000))
    ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=5))
    ax.xaxis.set_major_formatter(LogFormatterMathtext(base=10))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=12))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("Number of peptides (log scale)")
    ax.set_title("Prioritization-stage reduction", pad=11)
    style_axis(ax, grid_axis="x")
    for yi, (_, r) in zip(y, df.iterrows()):
        weight = "bold" if r["stage"] in {"Shortlist", "Final panel"} else "normal"
        x_text = float(r["count"]) * (1.11 if r["stage"] in {"Shortlist", "Final panel"} else 1.08)
        pct = float(r["percent_of_generated"])
        pct_text = f"{pct:.1f}%" if pct >= 0.05 else f"{pct:.2f}%"
        ax.text(x_text, yi, f"{int(r['count']):,} ({pct_text})", va="center", ha="left", fontsize=8.1, fontweight=weight)

    # B: multi-objective support scores
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.14, y=1.04, size=17)
    metrics = list(support_summary["metric"].drop_duplicates())
    stages = ["Descriptor-plausible", "Shortlist", "Final panel"]
    centers = np.arange(len(metrics)) * 1.36
    width = 0.18
    offsets = [-0.24, 0.0, 0.24]
    for i, stage in enumerate(stages):
        sub = support_summary[support_summary["stage"] == stage].set_index("metric").reindex(metrics).reset_index()
        xpos = centers + offsets[i]
        vals = sub["median"].to_numpy(float)
        q1 = sub["q1"].to_numpy(float)
        q3 = sub["q3"].to_numpy(float)
        yerr = np.vstack([np.maximum(vals - q1, 0), np.maximum(q3 - vals, 0)])
        ax.bar(xpos, vals, width=width, color=GROUP_COLORS[stage], edgecolor="none", label=stage, zorder=3)
        ax.errorbar(xpos, vals, yerr=yerr, fmt="none", ecolor=PLOS["dark"], elinewidth=0.9, capsize=2.3, capthick=0.9, zorder=4)
    ax.set_xticks(centers)
    ax.set_xticklabels(metrics, rotation=0, ha="center")
    ax.set_ylabel("Median score (IQR)")
    ax.set_title("Multi-objective support scores", pad=11)
    ax.set_ylim(0, 1.04)
    style_axis(ax, grid_axis="y")

    # C: composite-score enrichment
    ax = fig.add_subplot(gs[0, 2])
    add_panel_label(ax, "C", x=-0.18, y=1.04, size=17)
    sub = final_score_sum.set_index("stage").reindex(stages).reset_index()
    x = np.arange(len(stages)) * 1.18
    vals = sub["median"].to_numpy(float)
    q1 = sub["q1"].to_numpy(float)
    q3 = sub["q3"].to_numpy(float)
    yerr = np.vstack([np.maximum(vals - q1, 0), np.maximum(q3 - vals, 0)])
    ax.bar(x, vals, color=[GROUP_COLORS[s] for s in stages], edgecolor="none", width=0.54, zorder=3)
    ax.errorbar(x, vals, yerr=yerr, fmt="none", ecolor=PLOS["dark"], elinewidth=0.9, capsize=2.3, capthick=0.9, zorder=4)
    labels = [f"Descriptor-\nplausible\nn={int(sub.loc[0,'n']):,}", f"Shortlist\nn={int(sub.loc[1,'n']):,}", f"Final\npanel\nn={int(sub.loc[2,'n']):,}"]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, linespacing=0.93)
    ax.set_ylabel("Median composite score (IQR)")
    ax.set_title("Composite-score enrichment", pad=11)
    ax.set_ylim(0, 1.04)
    style_axis(ax, grid_axis="y")
    handles = [Patch(facecolor=GROUP_COLORS[s], edgecolor="none", label=s) for s in stages]
    leg = fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.56, 0.02), ncol=3, frameon=True, columnspacing=1.05, handlelength=1.25, handletextpad=0.45, borderpad=0.34)
    leg.get_frame().set_facecolor(PLOS["legend_bg"])
    leg.get_frame().set_edgecolor(PLOS["legend_edge"])
    fig.subplots_adjust(left=0.073, right=0.986, top=0.87, bottom=0.20)
    save_figure(fig, fig4_base, args)


def display_scheme_label(label: str) -> str:
    return {
        "Primary": "Primary",
        "Novelty-weighted": "Novelty-\nweighted",
        "Plausibility-weighted": "Plausibility-\nweighted",
        "Condition-weighted": "Condition-\nweighted",
        "Diversity-weighted": "Diversity-\nweighted",
    }.get(label, label.replace("-", "-\n") if len(label) > 14 else label)


def matrix_sparsity(mat: pd.DataFrame) -> Tuple[int, float]:
    if mat.empty:
        return 0, 0.0
    n = len(mat)
    mask = ~np.eye(n, dtype=bool)
    vals = mat.to_numpy(dtype=float)
    denom = mask.sum()
    valid = np.isfinite(vals[mask]).sum() if denom else 0
    return n, float(valid / denom) if denom else 0.0


def plot_s13_heatmap(ax: plt.Axes, mat: pd.DataFrame) -> None:
    labels = list(mat.index)
    vals = mat.to_numpy(dtype=float).copy()
    n = vals.shape[0]
    # Mask diagonal and missing values; draw diagonal as light cells with self labels.
    masked = vals.copy()
    np.fill_diagonal(masked, np.nan)
    im = ax.imshow(masked, vmin=0, vmax=1, cmap=HEATMAP_CMAP, aspect="equal")
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels([display_scheme_label(x) for x in labels])
    ax.set_yticklabels([display_scheme_label(x) for x in labels])
    ax.set_title("Ranking-scheme overlap", pad=10)
    for i in range(n):
        for j in range(n):
            if i == j:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=PLOS["missing"], edgecolor="white", linewidth=1.1, zorder=2))
                ax.text(j, i, "self", ha="center", va="center", fontsize=8.0, color="#777777", zorder=3)
            else:
                v = vals[i, j]
                if np.isfinite(v):
                    col = PLOS["white"] if v >= 0.62 else PLOS["dark"]
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8.3, color=col, zorder=3)
                else:
                    ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="#F5F5F5", edgecolor="white", linewidth=1.1, zorder=2))
                    ax.text(j, i, "—", ha="center", va="center", fontsize=8.2, color="#777777", zorder=3)
    ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.1)
    ax.tick_params(which="minor", bottom=False, left=False)
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#B8B8B8")
        ax.spines[side].set_linewidth(0.8)
    return im


def plot_s13_table_panel(ax: plt.Axes, pair_table: pd.DataFrame) -> None:
    ax.set_title("Ranking-scheme overlap", pad=10)
    ax.axis("off")
    if pair_table is None or pair_table.empty:
        ax.text(0.5, 0.5, "Scheme-overlap data unavailable", ha="center", va="center", transform=ax.transAxes)
        return
    df = pair_table.dropna(subset=["jaccard_overlap"]).copy()
    if df.empty:
        ax.text(0.5, 0.5, "No finite off-diagonal overlaps", ha="center", va="center", transform=ax.transAxes)
        return
    df["pair"] = df["scheme_a"].astype(str) + " vs " + df["scheme_b"].astype(str)
    df = df.sort_values("jaccard_overlap", ascending=True).tail(8)
    y = np.arange(len(df))
    vals = df["jaccard_overlap"].to_numpy(float)
    ax.barh(y, vals, color=PLOS["mint"], edgecolor="#8FBCA0", height=0.60)
    ax.set_yticks(y)
    ax.set_yticklabels(df["pair"].str.replace("-weighted", "-w", regex=False), fontsize=7.1)
    ax.set_xlim(0, 1)
    ax.xaxis.grid(True, color=PLOS["grid"], linewidth=0.7)
    for yi, v in zip(y, vals):
        ax.text(v + 0.025, yi, f"{v:.2f}", va="center", ha="left", fontsize=7.5)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#B8B8B8")
    ax.set_xlabel("Jaccard overlap")


def plot_s13(stability_mat: pd.DataFrame, stability_long: pd.DataFrame, pair_table: pd.DataFrame, recurrence_counts: pd.DataFrame, s13_base: Path, args: argparse.Namespace) -> None:
    set_publication_style()
    fig = plt.figure(figsize=(9.7, 4.7))
    gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.10, 0.95], wspace=0.37)
    ax1 = fig.add_subplot(gs[0, 0])
    add_panel_label(ax1, "A", x=-0.16, y=1.04, size=17)
    n_schemes, valid_frac = matrix_sparsity(stability_mat)
    if n_schemes >= 3 and valid_frac >= 0.5:
        im = plot_s13_heatmap(ax1, stability_mat)
        cbar = fig.colorbar(im, ax=ax1, fraction=0.040, pad=0.032)
        cbar.set_label("Jaccard overlap", fontsize=8.8)
        cbar.ax.tick_params(labelsize=7.8, width=0.7, length=2.5)
        cbar.outline.set_edgecolor("#B8B8B8")
        cbar.outline.set_linewidth(0.7)
    else:
        plot_s13_table_panel(ax1, pair_table)
    ax2 = fig.add_subplot(gs[0, 1])
    add_panel_label(ax2, "B", x=-0.15, y=1.04, size=17)
    if recurrence_counts is not None and len(recurrence_counts) > 0:
        x = recurrence_counts["scheme_recurrence_n"].astype(int).to_numpy()
        y = recurrence_counts["candidate_count"].astype(int).to_numpy()
        total = int(recurrence_counts["denominator_n"].iloc[0]) if "denominator_n" in recurrence_counts and len(recurrence_counts) else int(y.sum())
        bars = ax2.bar(x, y, color=PLOS["brown"], edgecolor="none", width=0.54, zorder=3)
        ax2.set_title("Candidate recurrence", pad=10)
        ax2.set_xlabel("Number of schemes recovering candidate")
        ax2.set_ylabel("Candidate count")
        ax2.set_xticks(x)
        ymax = max(y) if len(y) else 1
        ax2.set_ylim(0, ymax * 1.28)
        style_axis(ax2, grid_axis="y")
        for b, val in zip(bars, y):
            pct = 100 * val / total if total else 0
            ax2.text(b.get_x() + b.get_width()/2, val + ymax * 0.045, f"{val}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=8.2, linespacing=0.90)
        ax2.text(0.98, 0.94, f"n = {total} candidates", transform=ax2.transAxes, ha="right", va="top", fontsize=8.3, bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor=PLOS["legend_edge"], linewidth=0.8))
    else:
        ax2.text(0.5, 0.5, "Candidate-recurrence data unavailable", ha="center", va="center", transform=ax2.transAxes)
        ax2.axis("off")
    fig.subplots_adjust(left=0.075, right=0.985, top=0.87, bottom=0.20)
    save_figure(fig, s13_base, args)


def build_panel_mapping() -> pd.DataFrame:
    return pd.DataFrame([
        {"figure": "Fig 4", "panel": "A", "title": "Prioritization-stage reduction", "source_data_file": "Figure_4_panel_a_source_data.csv", "description": "Counts and percentages retained at each prioritization stage."},
        {"figure": "Fig 4", "panel": "B", "title": "Multi-objective support scores", "source_data_file": "Figure_4_panel_b_source_data.csv", "description": "Median novelty, plausibility, and diversity scores with IQR across stages."},
        {"figure": "Fig 4", "panel": "C", "title": "Composite-score enrichment", "source_data_file": "Figure_4_panel_c_source_data.csv", "description": "Median composite score with IQR across stages."},
        {"figure": "S13 Fig", "panel": "A", "title": "Ranking-scheme overlap", "source_data_file": "Supplementary_Figure_S13_panel_a_source_data.csv", "description": "Jaccard overlap between prioritization schemes; diagonal self-cells are de-emphasized."},
        {"figure": "S13 Fig", "panel": "B", "title": "Candidate recurrence", "source_data_file": "Supplementary_Figure_S13_panel_b_source_data.csv", "description": "Candidate count and fraction by number of recovering prioritization schemes."},
    ])


def readiness_score(checks: List[CheckResult]) -> Tuple[int, str]:
    fail = sum(c.status == "FAIL" for c in checks)
    warn = sum(c.status == "WARN" for c in checks)
    if fail:
        return max(55, 96 - fail * 8 - warn * 2), "FAIL"
    if warn:
        return max(88, 99 - warn), "WARN"
    return 100, "PASS"


def write_reports(dirs: OutputDirs, args: argparse.Namespace, inputs: InputPaths, discovery_records: List[DiscoveryRecord], checks: List[CheckResult], data: Dict[str, pd.DataFrame], files: Dict[str, str], missing_metric_report: pd.DataFrame) -> None:
    score, status = readiness_score(checks)
    write_csv(pd.DataFrame([asdict(c) for c in checks]), dirs.reports / "step8_readiness_checks.csv")
    write_csv(pd.DataFrame([asdict(r) for r in discovery_records]), dirs.reports / "step8_discovery_records.csv")
    write_csv(pd.DataFrame([asdict(r) for r in discovery_records if r.decision in {"REJECT", "SKIP"}]), dirs.reports / "step8_rejected_files.csv")
    write_csv(missing_metric_report, dirs.source_data / "step8_missing_metric_report.csv")
    files["step8_missing_metric_report"] = str(dirs.source_data / "step8_missing_metric_report.csv")
    report = []
    report.append("# OncoPep Step 8 readiness report\n\n")
    report.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}\n\n")
    report.append(f"Script version: `{SCRIPT_VERSION}`\n\n")
    report.append(f"Overall status: **{status}**\n\n")
    report.append(f"Estimated readiness score: **{score}/100**\n\n")
    report.append("## Scientific role\n\nStep 8 supports multi-objective prioritization and prioritization robustness. It does not assess anticancer activity, selectivity, toxicity, stability, receptor binding, or therapeutic efficacy.\n\n")
    report.append("## Selected input paths\n\n")
    for k, v in asdict(inputs).items():
        report.append(f"- {k}: `{v}`\n")
    report.append("\n## Validation checks\n\n")
    for c in checks:
        report.append(f"- **{c.status}** `{c.name}`: {c.detail}\n")
    report.append("\n## Missing metric report\n\n")
    if missing_metric_report.empty:
        report.append("No missing expected metric values were detected for Fig 4B.\n")
    else:
        report.append("See `source_data/step8_missing_metric_report.csv`. Plausibility absence is blocking unless scientifically justified.\n")
    report.append("\n## Output files\n\n")
    for k, v in files.items():
        report.append(f"- {k}: `{v}`\n")
    report.append("\n## Gate\n\n")
    report.append("This package should proceed to writing only if the final visual review and source-data checks reach ≥98/100.\n")
    (dirs.reports / "step8_readiness_report.md").write_text("".join(report), encoding="utf-8")
    manifest = {
        "script": SCRIPT_NAME,
        "script_version": SCRIPT_VERSION,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "step8_root": str(args.step8_root),
        "project_root": str(args.project_root),
        "inputs": {k: str(v) if v is not None else None for k, v in asdict(inputs).items()},
        "data_shapes": {k: list(v.shape) for k, v in data.items() if isinstance(v, pd.DataFrame)},
        "outputs": files,
        "checks": [asdict(c) for c in checks],
        "readiness_score": score,
        "readiness_status": status,
        "claim_boundary": "Computational prioritization and robustness only; no experimental anticancer validation.",
    }
    (dirs.reports / "step8_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    readme = f"""# OncoPep Step 8 output package

Script version: {SCRIPT_VERSION}

Scientific role: multi-objective prioritization and prioritization robustness.

Main outputs:
- main_figure/Figure_4_prioritization_redesigned.png/pdf/tiff
- supplementary_figures/Supplementary_Figure_S13_prioritization_robustness_redesigned.png/pdf/tiff
- source_data/Figure_4_* and Supplementary_Figure_S13_*
- source_data/step8_missing_metric_report.csv
- reports/step8_readiness_report.md
- reports/step8_manifest.json

Descriptor-distribution details are retained as source-data/report material only to avoid duplication with Fig 3/Step 7.
"""
    (dirs.reports / "README_step8_outputs.txt").write_text(readme, encoding="utf-8")
    (dirs.reports / "requirements_step8_minimal.txt").write_text("\n".join(["python>=3.9", f"numpy=={np.__version__}", f"pandas=={pd.__version__}", f"matplotlib=={matplotlib.__version__}", ""]), encoding="utf-8")
    try:
        src = Path(__file__).resolve()
        if src.exists():
            shutil.copy2(src, dirs.code / SCRIPT_NAME)
    except Exception:
        pass


def validate_data(data: Dict[str, pd.DataFrame], support_summary: pd.DataFrame, stability_mat: pd.DataFrame, recurrence_counts: pd.DataFrame) -> List[CheckResult]:
    checks: List[CheckResult] = []
    for key in ["passed_df", "shortlist_df", "final_panel_df"]:
        checks.append(CheckResult(key, "PASS" if key in data and len(data[key]) else "FAIL", f"{len(data[key]):,} rows." if key in data else "missing"))
    for metric in ["Novelty", "Plausibility", "Diversity"]:
        sub = support_summary[support_summary["metric"] == metric]
        ok = len(sub) == 3 and bool(sub["available"].all())
        checks.append(CheckResult(f"Fig4B:{metric}", "PASS" if ok else "FAIL", f"{metric} {'available' if ok else 'missing/incomplete'} across all stages."))
    n_schemes, vf = matrix_sparsity(stability_mat)
    checks.append(CheckResult("S13A:scheme_overlap", "PASS" if n_schemes >= 3 and vf >= 0.5 else "WARN", f"n_schemes={n_schemes}, off_diagonal_valid_fraction={vf:.2f}."))
    checks.append(CheckResult("S13B:recurrence", "PASS" if recurrence_counts is not None and len(recurrence_counts) > 0 else "WARN", "recurrence summary available." if recurrence_counts is not None and len(recurrence_counts) else "recurrence summary unavailable."))
    return checks


def main(argv: Optional[Sequence[str]] = None) -> int:
    args = parse_args(argv)
    dirs = ensure_dirs(args.step8_root)
    if not args.quiet:
        print(f"Hard-fix version: {SCRIPT_VERSION}")
        if args.unknown_args_ignored:
            print(f"Ignored unknown/Jupyter arguments: {args.unknown_args_ignored}")
    inputs, discovery_records = discover_inputs(args)
    data, checks = load_inputs(args, inputs)
    data, score_checks, missing_metric_report = ensure_score_columns(data)
    data, desc_checks = ensure_descriptor_columns(data)
    checks.extend(score_checks)
    checks.extend(desc_checks)
    component_metrics, metric_checks = selected_component_metrics(data)
    checks.extend(metric_checks)
    counts = stage_count_table(args)
    support_summary = summarize_scores(data, component_metrics, args)
    final_summary = final_score_summary(data, args)
    descriptor_summary = descriptor_distribution_summary(data)
    stability_mat, stability_long, pair_table, stability_checks = stability_matrix_source(data.get("stability_df", pd.DataFrame()))
    recurrence_counts, recurrence_checks = recurrence_source(data.get("recurrence_df", pd.DataFrame()))
    checks.extend(stability_checks)
    checks.extend(recurrence_checks)
    checks.extend(validate_data(data, support_summary, stability_mat, recurrence_counts))
    files: Dict[str, str] = {}
    for name, df in {
        "Figure_4_panel_a_source_data": counts,
        "Figure_4_panel_b_source_data": support_summary,
        "Figure_4_panel_c_source_data": final_summary,
    }.items():
        f = dirs.source_data / f"{name}.csv"
        write_csv(df, f)
        files[name] = str(f)
    fig4_all = combine_source_data({"Figure_4_panel_a": counts, "Figure_4_panel_b": support_summary, "Figure_4_panel_c": final_summary})
    f = dirs.source_data / "Figure_4_source_data_all_panels.csv"
    write_csv(fig4_all, f)
    files["Figure_4_source_data_all_panels"] = str(f)
    for name, df in {
        "Supplementary_Figure_S13_panel_a_source_data": stability_long,
        "Supplementary_Figure_S13_panel_a_pair_table_source_data": pair_table,
        "Supplementary_Figure_S13_panel_b_source_data": recurrence_counts,
    }.items():
        f = dirs.source_data / f"{name}.csv"
        write_csv(df, f)
        files[name] = str(f)
    s13_all = combine_source_data({"Supplementary_Figure_S13_panel_a": stability_long, "Supplementary_Figure_S13_panel_b": recurrence_counts})
    f = dirs.source_data / "Supplementary_Figure_S13_source_data_all_panels.csv"
    write_csv(s13_all, f)
    files["Supplementary_Figure_S13_source_data_all_panels"] = str(f)
    for name, df in {
        "step8_prioritization_summary": fig4_all,
        "step8_selection_stage_counts": counts,
        "step8_candidate_support_summary": support_summary,
        "step8_scheme_overlap_summary": stability_long,
        "step8_candidate_recurrence_summary": recurrence_counts,
        "step8_descriptor_distribution_table": descriptor_summary,
        "step8_panel_source_data_mapping": build_panel_mapping(),
        "step8_metric_availability_report": metric_availability(data),
    }.items():
        f = dirs.source_data / f"{name}.csv"
        write_csv(df, f)
        files[name] = str(f)
    fig4_base = dirs.main_figure / "Figure_4_prioritization_redesigned"
    plot_figure4(counts, support_summary, final_summary, fig4_base, args)
    files.update({"Figure_4_png": str(fig4_base.with_suffix(".png")), "Figure_4_pdf": str(fig4_base.with_suffix(".pdf")), "Figure_4_tiff": str(fig4_base.with_suffix(".tiff"))})
    s13_base = dirs.supplementary_figures / "Supplementary_Figure_S13_prioritization_robustness_redesigned"
    plot_s13(stability_mat, stability_long, pair_table, recurrence_counts, s13_base, args)
    files.update({"Supplementary_Figure_S13_png": str(s13_base.with_suffix(".png")), "Supplementary_Figure_S13_pdf": str(s13_base.with_suffix(".pdf")), "Supplementary_Figure_S13_tiff": str(s13_base.with_suffix(".tiff"))})
    write_reports(dirs, args, inputs, discovery_records, checks, data, files, missing_metric_report)
    score, status = readiness_score(checks)
    if not args.quiet:
        print("\nOncoPep Step 8 package generated.")
        print(f"Root: {dirs.root}")
        print(f"Readiness status: {status}; estimated score: {score}/100")
        print(f"Main figure: {fig4_base.with_suffix('.png')}")
        print(f"Supplementary figure: {s13_base.with_suffix('.png')}")
        print(f"Readiness report: {dirs.reports / 'step8_readiness_report.md'}")
        if status != "PASS" or score < 98:
            print("WARNING: Package generated but does not pass the ≥98/100 gate. Review readiness report before writing.")
    return 0


if __name__ == "__main__":
    try:
        rc = main()
        if is_notebook() and rc != 0:
            raise RuntimeError(f"Step 8 script failed with exit code {rc}.")
        if not is_notebook():
            raise SystemExit(rc)
    except Exception as exc:
        if is_notebook():
            raise RuntimeError(str(exc)) from exc
        print("\nERROR: OncoPep Step 8 figure generation failed.\n", file=sys.stderr)
        print(str(exc), file=sys.stderr)
        print("\nTraceback:\n", file=sys.stderr)
        traceback.print_exc()
        raise SystemExit(2)

In [ ]:
main([
    "--step8-root", "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_08",
    "--project-root", "/home/data3/Moe/nature_computational_peponco",
    "--passed-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_12_full_ranked_passed_pool.csv",
    "--shortlist-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_main/table_8_1_shortlist_main.csv",
    "--stability-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_5_ranking_scheme_stability.csv",
    "--recurrence-file", "/home/data3/Moe/nature_computational_peponco/step8_v1/tables_supplementary/table_s8_6_candidate_recurrence_across_schemes.csv",
    "--generated-count", "10840",
    "--eligible-count", "10291",
    "--passed-count", "10237",
    "--shortlist-count", "24",
    "--final-count", "12",
])

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 9 — PLOS Computational Biology Figure 5 rescue script.

Scientific role
---------------
Step 9 focuses on contextual similarity support and internal diversity of the
final OncoPep candidate panel. It intentionally avoids the Step 8 selection
and prioritization-score panels.

Main figure
-----------
Figure 5. Contextual similarity and compositional support of the final
OncoPep panel.
  A. Contextual nearest-neighbor similarity
  B. Nearest-neighbor similarity summary
  C. Residue-category context

Supplementary figure
--------------------
Supplementary Figure S14. Additional sequence-composition and internal-diversity context.
  A. Amino-acid enrichment
  B. Top enriched 3-mers
  C. Pairwise similarity within final panel

Design decisions
----------------
- Rescues the scientifically stronger old Step 9 similarity/composition logic.
- Removes old Step 9 selection-audit and score-shift panels because they now
  duplicate Step 8/Figure 4.
- Uses uppercase panel labels, OncoPep/PLOS palette, source-data exports,
  manifest, README, requirements, code snapshot, and readiness reporting.
- Implements the approved panel swap: residue-category context is moved to
  main Figure 5C, and pairwise final-panel similarity is moved to S14C.
- Generates Supplementary Figure S14 by default whenever a final-panel table,
  reference sequence table, and pairwise/final sequences are available.
- Avoids default all-1.00 contextual-support fallback panels.

Ready-to-run examples
---------------------
python OncoPep_step9_PLOS_contextual_similarity_diversity_final.py \
  --step9-root /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_09 \
  --project-root /home/data3/Moe/nature_computational_peponco \
  --legacy-step9-root /home/data3/Moe/nature_computational2/step9_v1

For explicit paths:
python OncoPep_step9_PLOS_contextual_similarity_diversity_final.py \
  --final-panel-file /path/to/table_s9_9_final_panel_with_all_step9_annotations.csv \
  --reference-file /path/to/step1_retained_dataset.csv \
  --pairwise-file /path/to/table_s9_7_final_panel_pairwise_similarity.csv \
  --aa-enrichment-file /path/to/table_s9_5_amino_acid_enrichment_vs_reference.csv \
  --kmer-enrichment-file /path/to/table_s9_6_kmer_enrichment_vs_reference.csv
"""

from __future__ import annotations

import argparse
import json
import math
import os
import shutil
import sys
import traceback
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

SCRIPT_VERSION = "v3_2026_07_08_step9_panel_swap_similarity_composition_plos"

DEFAULT_STEP9_ROOT = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_09"
DEFAULT_PROJECT_ROOT = "/home/data3/Moe/nature_computational_peponco"
DEFAULT_LEGACY_STEP9_ROOT = "/home/data3/Moe/nature_computational2/step9_v1"
DEFAULT_LEGACY_REFERENCE = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"

SAFE_TABLE_EXTENSIONS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".pq"}
BLOCKED_EXTENSIONS = {".json", ".png", ".jpg", ".jpeg", ".pdf", ".tif", ".tiff", ".py", ".ipynb"}
BLOCKED_PATH_KEYWORDS = [
    "/run/user/",
    "/jupyter/runtime/",
    ".ipynb_checkpoints",
    "__pycache__",
    "/.git/",
    "kernel-",
]

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
    "edge": "#B8B8B8",
    "pale": "#F5F7F7",
}

AA_SET = set("ACDEFGHIKLMNPQRSTVWY")
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")

KD_HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}


@dataclass
class CheckResult:
    name: str
    status: str  # PASS/WARN/FAIL
    message: str


@dataclass
class OutputDirs:
    root: Path
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


def is_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        shell = get_ipython()
        if shell is None:
            return False
        return shell.__class__.__name__ in {"ZMQInteractiveShell", "TerminalInteractiveShell"}
    except Exception:
        return False


def clean_argv(argv: Optional[Sequence[str]] = None) -> List[str]:
    if argv is None:
        argv = sys.argv[1:]
    out: List[str] = []
    skip = False
    for i, token in enumerate(argv):
        if skip:
            skip = False
            continue
        s = str(token)
        if s == "-f":
            if i + 1 < len(argv):
                skip = True
            continue
        if any(k in s for k in BLOCKED_PATH_KEYWORDS):
            continue
        if s.endswith(".json") and ("kernel-" in s or "/runtime/" in s):
            continue
        out.append(s)
    return out


def ensure_dirs(root: Path) -> OutputDirs:
    dirs = OutputDirs(
        root=root,
        main_figure=root / "main_figure",
        supplementary_figures=root / "supplementary_figures",
        source_data=root / "source_data",
        reports=root / "reports",
        code=root / "code",
    )
    for d in asdict(dirs).values():
        if isinstance(d, Path):
            d.mkdir(parents=True, exist_ok=True)
    return dirs


def is_blocked_path(path: Path) -> bool:
    p = str(path).replace("\\", "/").lower()
    return any(k.lower() in p for k in BLOCKED_PATH_KEYWORDS)


def is_allowed_table_file(path: Path) -> bool:
    suffix = path.suffix.lower()
    return suffix in SAFE_TABLE_EXTENSIONS and suffix not in BLOCKED_EXTENSIONS and not is_blocked_path(path)


def safe_roots(*roots: Optional[str]) -> List[Path]:
    seen = set()
    out: List[Path] = []
    for r in roots:
        if not r:
            continue
        p = Path(r).expanduser()
        s = str(p.resolve()) if p.exists() else str(p)
        if s in seen:
            continue
        seen.add(s)
        if p.exists():
            out.append(p)
    return out


def discover_files(roots: List[Path]) -> List[Path]:
    found: List[Path] = []
    for root in roots:
        try:
            for p in root.rglob("*"):
                try:
                    if p.is_file() and is_allowed_table_file(p):
                        found.append(p)
                except Exception:
                    continue
        except Exception:
            continue
    return found


def score_path_for_role(path: Path, role: str) -> Tuple[int, List[str]]:
    s = str(path).lower().replace("\\", "/")
    name = path.name.lower()
    score = 0
    reasons: List[str] = []

    if "step9_v1/tables_supplementary" in s:
        score += 12; reasons.append("legacy_step9_supp")
    if "step_09" in s or "step9" in s:
        score += 6; reasons.append("step9_hint")
    if "step1" in s and role == "reference":
        score += 8; reasons.append("step1_reference_hint")

    patterns = {
        "final": ["table_s9_9_final_panel", "final_panel_with_all_step9_annotations", "final_panel", "final_12", "candidate"],
        "pairwise": ["table_s9_7_final_panel_pairwise_similarity", "pairwise_similarity", "similarity_matrix", "jaccard"],
        "reference": ["step1_retained_dataset", "retained_dataset", "reference", "training", "corpus"],
        "aa": ["table_s9_5_amino_acid", "amino_acid_enrichment", "aa_enrichment"],
        "kmer": ["table_s9_6_kmer", "kmer_enrichment", "3mer", "3_mer"],
        "paper": ["table_s9_11_paper_candidates", "paper_candidates", "harmonized"],
    }
    for patt in patterns.get(role, []):
        if patt in name:
            score += 10; reasons.append(f"name:{patt}")
    negative = ["readiness", "manifest", "requirements", "selection_audit", "stage_score", "score_shift", "figure", "source_data_all"]
    for n in negative:
        if n in name:
            score -= 8; reasons.append(f"penalty:{n}")
    return score, reasons


def resolve_input(role: str, explicit: Optional[str], roots: List[Path], log: List[Dict[str, str]]) -> Optional[Path]:
    if explicit:
        p = Path(explicit).expanduser()
        log.append({"role": role, "action": "explicit", "path": str(p), "reason": "user_supplied"})
        return p
    candidates = discover_files(roots)
    scored: List[Tuple[int, Path, List[str]]] = []
    for p in candidates:
        score, reasons = score_path_for_role(p, role)
        if score > 0:
            scored.append((score, p, reasons))
    if not scored:
        log.append({"role": role, "action": "not_found", "path": "", "reason": "no_safe_candidate"})
        return None
    scored.sort(key=lambda x: (-x[0], str(x[1])))
    score, p, reasons = scored[0]
    log.append({"role": role, "action": "auto", "path": str(p), "reason": f"score={score};" + ";".join(reasons)})
    return p


def read_table(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suf == ".txt":
        for sep in [",", "\t", None]:
            try:
                return pd.read_csv(path, sep=sep) if sep else pd.read_csv(path, delim_whitespace=True)
            except Exception:
                pass
        raise ValueError(f"Could not parse text table: {path}")
    if suf in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suf in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table extension: {path}")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def first_col(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    lower = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    return None


def standardize_final_df(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_columns(df)
    out = df.copy()
    seq_col = first_col(out, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        raise RuntimeError("Final-panel table needs a sequence column.")
    out["sequence"] = out[seq_col].astype(str).map(clean_sequence)

    id_col = first_col(out, ["generated_id", "candidate_id", "peptide_id", "id", "seq_id", "name"])
    if id_col is None:
        out["generated_id"] = [f"C{i:02d}" for i in range(1, len(out) + 1)]
    else:
        out["generated_id"] = out[id_col].astype(str)

    alias_map = {
        "nearest_reference_similarity": [
            "nearest_reference_similarity", "nn_reference_similarity", "nearest_ref_similarity",
            "max_reference_similarity", "reference_nn_similarity", "nn_similarity_reference"
        ],
        "nearest_paper_candidate_similarity": [
            "nearest_paper_candidate_similarity", "nearest_candidate_similarity", "nearest_final_similarity",
            "paper_candidate_similarity", "candidate_context_similarity", "nn_paper_candidate_similarity"
        ],
    }
    for std, aliases in alias_map.items():
        if std not in out.columns:
            col = first_col(out, aliases)
            if col is not None:
                out[std] = pd.to_numeric(out[col], errors="coerce")
    return out


def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    s = str(seq).strip().upper()
    return "".join(ch for ch in s if ch in AA_SET)


def kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}


def jaccard_sets(a: set, b: set) -> float:
    if not a and not b:
        return np.nan
    u = a | b
    if not u:
        return np.nan
    return len(a & b) / len(u)


def compute_pairwise_matrix(ids: Sequence[str], seqs: Sequence[str], k: int = 3) -> pd.DataFrame:
    ids = [str(x) for x in ids]
    ksets = [kmers(s, k=k) for s in seqs]
    n = len(ids)
    mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            mat[i, j] = 1.0 if i == j else jaccard_sets(ksets[i], ksets[j])
    return pd.DataFrame(mat, index=ids, columns=ids)


def read_pairwise_matrix(path: Path) -> pd.DataFrame:
    df = read_table(path)
    df = normalize_columns(df)
    # Matrix format: first column is index/candidate IDs and remaining are numeric columns
    first = df.columns[0]
    if not pd.api.types.is_numeric_dtype(df[first]):
        trial = df.set_index(first)
        numeric = trial.apply(pd.to_numeric, errors="coerce")
        # Accept if at least square-ish numeric matrix
        if numeric.shape[0] == numeric.shape[1] and numeric.notna().sum().sum() > 0:
            return numeric
    # Long format fallback
    cols = {c.lower(): c for c in df.columns}
    id1 = first_col(df, ["candidate_1", "id1", "source", "row", "candidate_i"])
    id2 = first_col(df, ["candidate_2", "id2", "target", "column", "candidate_j"])
    val = first_col(df, ["similarity", "jaccard", "jaccard_similarity", "3mer_jaccard", "value"])
    if id1 and id2 and val:
        tmp = df[[id1, id2, val]].copy()
        tmp[val] = pd.to_numeric(tmp[val], errors="coerce")
        mat = tmp.pivot_table(index=id1, columns=id2, values=val, aggfunc="first")
        # symmetrize where possible
        labels = sorted(set(mat.index.astype(str)) | set(mat.columns.astype(str)))
        mat = mat.reindex(index=labels, columns=labels)
        for a in labels:
            for b in labels:
                if pd.isna(mat.loc[a, b]) and not pd.isna(mat.loc[b, a]):
                    mat.loc[a, b] = mat.loc[b, a]
        np.fill_diagonal(mat.values, 1.0)
        return mat.apply(pd.to_numeric, errors="coerce")
    raise RuntimeError(f"Could not parse pairwise similarity matrix: {path}")


def max_similarity_to_reference(final_df: pd.DataFrame, ref_df: pd.DataFrame, k: int = 3, max_ref: int = 100000) -> pd.Series:
    ref = ref_df.copy()
    seq_col = first_col(ref, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        raise RuntimeError("Reference file needs a sequence column for fallback nearest-reference similarity.")
    ref_seqs = [clean_sequence(x) for x in ref[seq_col].astype(str).tolist()]
    ref_seqs = [s for s in ref_seqs if s]
    if len(ref_seqs) > max_ref:
        ref_seqs = ref_seqs[:max_ref]
    ref_sets = [kmers(s, k=k) for s in ref_seqs]
    vals = []
    for seq in final_df["sequence"].astype(str).tolist():
        sset = kmers(seq, k=k)
        if not sset or not ref_sets:
            vals.append(np.nan)
            continue
        mx = max(jaccard_sets(sset, rset) for rset in ref_sets)
        vals.append(float(mx))
    return pd.Series(vals, index=final_df.index)


def nearest_internal_similarity(pairwise: pd.DataFrame) -> pd.Series:
    mat = pairwise.copy().apply(pd.to_numeric, errors="coerce")
    vals = []
    for i, idx in enumerate(mat.index):
        arr = mat.iloc[i].to_numpy(dtype=float)
        if i < len(arr):
            arr[i] = np.nan
        vals.append(float(np.nanmax(arr)) if np.isfinite(arr).any() else np.nan)
    return pd.Series(vals, index=mat.index)


def percentile_summary(vals: np.ndarray) -> Dict[str, float]:
    vals = np.asarray([v for v in vals if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return {"n": 0, "median": np.nan, "p90": np.nan, "max": np.nan, "q1": np.nan, "q3": np.nan}
    return {
        "n": int(len(vals)),
        "median": float(np.nanmedian(vals)),
        "p90": float(np.nanpercentile(vals, 90)),
        "max": float(np.nanmax(vals)),
        "q1": float(np.nanpercentile(vals, 25)),
        "q3": float(np.nanpercentile(vals, 75)),
    }


def aa_enrichment(final_seqs: Sequence[str], ref_seqs: Sequence[str], pseudocount: float = 0.5) -> pd.DataFrame:
    fcnt = Counter(ch for s in final_seqs for ch in clean_sequence(s) if ch in AA_SET)
    rcnt = Counter(ch for s in ref_seqs for ch in clean_sequence(s) if ch in AA_SET)
    f_total = sum(fcnt.values())
    r_total = sum(rcnt.values())
    rows = []
    for aa in AA_ORDER:
        f = fcnt.get(aa, 0)
        r = rcnt.get(aa, 0)
        f_freq = (f + pseudocount) / (f_total + pseudocount * len(AA_ORDER)) if f_total else np.nan
        r_freq = (r + pseudocount) / (r_total + pseudocount * len(AA_ORDER)) if r_total else np.nan
        log2e = float(np.log2(f_freq / r_freq)) if pd.notna(f_freq) and pd.notna(r_freq) and r_freq > 0 else np.nan
        rows.append({"amino_acid": aa, "final_count": f, "reference_count": r, "final_frequency": f_freq, "reference_frequency": r_freq, "log2_enrichment_vs_reference": log2e})
    return pd.DataFrame(rows)


def kmer_enrichment(final_seqs: Sequence[str], ref_seqs: Sequence[str], k: int = 3, pseudocount: float = 0.5, top_n: int = 25) -> pd.DataFrame:
    fcnt = Counter(km for s in final_seqs for km in kmers(s, k=k))
    rcnt = Counter(km for s in ref_seqs for km in kmers(s, k=k))
    universe = sorted(set(fcnt) | set(rcnt))
    f_total = sum(fcnt.values())
    r_total = sum(rcnt.values())
    rows = []
    denom = max(len(universe), 1)
    for km in universe:
        f = fcnt.get(km, 0)
        r = rcnt.get(km, 0)
        f_freq = (f + pseudocount) / (f_total + pseudocount * denom) if f_total else np.nan
        r_freq = (r + pseudocount) / (r_total + pseudocount * denom) if r_total else np.nan
        log2e = float(np.log2(f_freq / r_freq)) if pd.notna(f_freq) and pd.notna(r_freq) and r_freq > 0 else np.nan
        rows.append({"kmer": km, "final_count": f, "reference_count": r, "final_frequency": f_freq, "reference_frequency": r_freq, "log2_enrichment_vs_reference": log2e})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    out = out.sort_values(["log2_enrichment_vs_reference", "final_count"], ascending=[False, False]).head(top_n)
    return out


def frac_of_categories(seqs: Sequence[str]) -> Dict[str, float]:
    seqs = [clean_sequence(s) for s in seqs]
    total = sum(len(s) for s in seqs)
    if total == 0:
        return {"Hydrophobic": np.nan, "Aromatic": np.nan, "Positive": np.nan, "Negative": np.nan, "Polar": np.nan}
    def frac(res_set: set) -> float:
        return sum(sum(ch in res_set for ch in s) for s in seqs) / total
    return {
        "Hydrophobic": frac(AA_HYDROPHOBIC),
        "Aromatic": frac(AA_AROMATIC),
        "Positive": frac(AA_POSITIVE),
        "Negative": frac(AA_NEGATIVE),
        "Polar": frac(AA_POLAR),
    }


def get_ref_sequences(ref_df: pd.DataFrame) -> List[str]:
    seq_col = first_col(ref_df, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        return []
    return [clean_sequence(x) for x in ref_df[seq_col].astype(str).tolist() if clean_sequence(x)]


def set_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 15,
        "axes.labelsize": 11,
        "xtick.labelsize": 9.5,
        "ytick.labelsize": 9.5,
        "legend.fontsize": 9.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": PLOS["grid"],
        "grid.linewidth": 0.8,
        "grid.alpha": 0.85,
        "axes.axisbelow": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(PLOS["edge"])
    ax.spines["bottom"].set_color(PLOS["edge"])
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(colors=PLOS["dark"], width=0.8, length=4)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.8, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")
    elif grid_axis == "both":
        ax.grid(True, axis="both", color=PLOS["grid"], linewidth=0.8, zorder=0)


def add_panel_label(ax, label: str, x: float = -0.14, y: float = 1.08, fontsize: int = 18) -> None:
    ax.text(x, y, label, transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="left", va="top", color=PLOS["dark"])


def draw_violin(ax, data: Sequence[float], pos: float, color: str, width: float = 0.72) -> None:
    arr = np.asarray([x for x in data if pd.notna(x)], dtype=float)
    if len(arr) == 0:
        return
    parts = ax.violinplot([arr], positions=[pos], widths=width, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]:
        body.set_facecolor(color)
        body.set_edgecolor("none")
        body.set_alpha(0.78)
    q1, med, q3 = np.nanpercentile(arr, [25, 50, 75])
    ax.plot([pos, pos], [q1, q3], color=PLOS["dark"], linewidth=1.2, zorder=4)
    ax.scatter([pos], [med], s=34, facecolor=PLOS["white"], edgecolor=PLOS["dark"], linewidth=0.8, zorder=5)
    # Subtle points
    rng = np.random.default_rng(42 + int(pos * 10))
    jitter = rng.uniform(-0.065, 0.065, size=len(arr))
    ax.scatter(np.full(len(arr), pos) + jitter, arr, s=10, facecolor=PLOS["white"], edgecolor=PLOS["dark"], linewidth=0.35, alpha=0.65, zorder=4)


def save_multi(fig: plt.Figure, out_base: Path, dpi: int) -> Dict[str, str]:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths: Dict[str, str] = {}
    for ext in [".png", ".pdf", ".tiff"]:
        p = out_base.with_suffix(ext)
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        paths[ext.lstrip(".")] = str(p)
    plt.close(fig)
    return paths


def short_label(x: str, max_len: int = 12) -> str:
    s = str(x)
    return s if len(s) <= max_len else s[:max_len]


def sorted_pairwise_matrix(pairwise: pd.DataFrame, final_ids: Sequence[str]) -> pd.DataFrame:
    mat = pairwise.copy()
    mat.index = mat.index.astype(str)
    mat.columns = mat.columns.astype(str)
    ids = [str(i) for i in final_ids]
    available = [i for i in ids if i in mat.index and i in mat.columns]
    if len(available) >= 2:
        mat = mat.loc[available, available]
    else:
        # keep original square matrix
        pass
    numeric = mat.apply(pd.to_numeric, errors="coerce")
    # order by mean off-diagonal similarity to make layout intentional
    vals = numeric.to_numpy(dtype=float)
    np.fill_diagonal(vals, np.nan)
    avg = np.nanmean(vals, axis=1)
    order = np.argsort(-np.nan_to_num(avg, nan=-1))
    numeric = numeric.iloc[order, :].iloc[:, order]
    return numeric


def residue_category_df(final_df: pd.DataFrame, reference_df: pd.DataFrame) -> pd.DataFrame:
    """Return residue-category fractions for reference corpus and final panel."""
    final_seqs = final_df["sequence"].astype(str).tolist()
    ref_seqs = get_ref_sequences(reference_df)
    ref_cat = frac_of_categories(ref_seqs)
    final_cat = frac_of_categories(final_seqs)
    categories = ["Hydrophobic", "Aromatic", "Positive", "Negative", "Polar"]
    return pd.DataFrame({
        "category": categories,
        "reference_fraction": [ref_cat[c] for c in categories],
        "final_panel_fraction": [final_cat[c] for c in categories],
    })


def write_pairwise_source_and_heatmap(
    ax,
    pairwise: pd.DataFrame,
    final_ids: Sequence[str],
    title: str,
    colorbar_label: str = "3-mer Jaccard similarity",
    panel_label: Optional[str] = None,
) -> pd.DataFrame:
    """Draw polished supplementary heatmap and return ordered matrix."""
    mat = sorted_pairwise_matrix(pairwise, final_ids)
    arr = mat.to_numpy(dtype=float)
    if arr.shape[0] == 0:
        raise RuntimeError("Pairwise matrix has no rows after ordering.")
    diag_mask = np.eye(arr.shape[0], dtype=bool)
    offdiag = arr.copy()
    offdiag[diag_mask] = np.nan
    max_offdiag = float(np.nanmax(offdiag)) if np.isfinite(offdiag).any() else 0.20
    vmax = min(1.0, max(0.20, math.ceil(max_offdiag * 10) / 10))
    cmap = LinearSegmentedColormap.from_list(
        "oncopep_similarity",
        [PLOS["white"], PLOS["mint"], PLOS["blue"]],
    )
    plot_arr = arr.copy()
    plot_arr[diag_mask] = np.nan
    masked = np.ma.masked_invalid(plot_arr)
    cmap.set_bad(color=PLOS["pale"])
    im = ax.imshow(masked, vmin=0, vmax=vmax, cmap=cmap, aspect="equal")
    labels = [short_label(x, 12) for x in mat.index.tolist()]
    ax.set_xticks(np.arange(len(labels)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(labels)), labels, fontsize=8)
    ax.set_title(title, loc="left", pad=8)
    if panel_label:
        add_panel_label(ax, panel_label, x=-0.08, y=1.06, fontsize=16)
    ax.set_xticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.grid(which="minor", color=PLOS["white"], linestyle="-", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.042, pad=0.035)
    cbar.set_label(colorbar_label)
    return mat


def plot_figure5(
    final_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    reference_df: pd.DataFrame,
    source_data_dir: Path,
    out_base: Path,
    dpi: int,
) -> Dict[str, str]:
    """Generate final Figure 5 after approved panel swap.

    Final structure:
      A. Contextual nearest-neighbor similarity
      B. Nearest-neighbor similarity summary
      C. Residue-category context
    """
    set_style()

    ref_vals = pd.to_numeric(final_df["nearest_reference_similarity"], errors="coerce").dropna().to_numpy(dtype=float)
    cand_col = "nearest_paper_candidate_similarity"
    cand_vals = pd.to_numeric(final_df[cand_col], errors="coerce").dropna().to_numpy(dtype=float)

    summary_rows = []
    for name, arr in [("Reference context", ref_vals), ("Final-panel context", cand_vals)]:
        st = percentile_summary(arr)
        summary_rows.append({"context": name, **st})
    summary_df = pd.DataFrame(summary_rows)

    panel_a_df = pd.concat([
        pd.DataFrame({"context": "Reference context", "nearest_neighbor_similarity": ref_vals}),
        pd.DataFrame({"context": "Final-panel context", "nearest_neighbor_similarity": cand_vals}),
    ], ignore_index=True)
    panel_a_df.to_csv(source_data_dir / "Figure_5_panel_a_source_data.csv", index=False)

    panel_b_rows = []
    for _, r in summary_df.iterrows():
        for metric in ["median", "p90", "max"]:
            panel_b_rows.append({"context": r["context"], "summary_metric": metric, "similarity": r[metric], "n": r["n"]})
    panel_b_df = pd.DataFrame(panel_b_rows)
    panel_b_df.to_csv(source_data_dir / "Figure_5_panel_b_source_data.csv", index=False)

    cat_df = residue_category_df(final_df, reference_df)
    cat_df.to_csv(source_data_dir / "Figure_5_panel_c_source_data.csv", index=False)

    combined = pd.concat([
        panel_a_df.assign(panel="Figure_5A"),
        panel_b_df.assign(panel="Figure_5B"),
        cat_df.assign(panel="Figure_5C"),
    ], ignore_index=True, sort=False)
    combined.to_csv(source_data_dir / "Figure_5_source_data_all_panels.csv", index=False)

    fig = plt.figure(figsize=(15.8, 5.15))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[0.88, 0.92, 1.10], wspace=0.50)

    # A violin
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.18, y=1.08)
    style_axis(ax, grid_axis="y")
    draw_violin(ax, ref_vals, 0, PLOS["light"], width=0.78)
    draw_violin(ax, cand_vals, 1, PLOS["coral"], width=0.78)
    ymax = max(0.16, float(np.nanmax(np.r_[ref_vals, cand_vals])) * 1.22 if len(np.r_[ref_vals, cand_vals]) else 0.16)
    ax.set_ylim(0, ymax)
    ax.set_xticks([0, 1], ["Reference\ncontext", "Final-panel\ncontext"])
    ax.set_ylabel("Nearest-neighbor similarity")
    ax.set_title("Contextual nearest-neighbor similarity", pad=10)

    # B summary bars
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.18, y=1.08)
    style_axis(ax, grid_axis="y")
    metrics = ["median", "p90", "max"]
    metric_labels = ["Median", "P90", "Max"]
    x = np.arange(len(metrics), dtype=float)
    width = 0.32
    ref_s = summary_df.set_index("context").loc["Reference context"]
    cand_s = summary_df.set_index("context").loc["Final-panel context"]
    ref_bar = [float(ref_s[m]) for m in metrics]
    cand_bar = [float(cand_s[m]) for m in metrics]
    ax.bar(x - width/2, ref_bar, width=width, color=PLOS["light"], edgecolor=PLOS["dark"], linewidth=0.5, label="Reference context", zorder=3)
    ax.bar(x + width/2, cand_bar, width=width, color=PLOS["coral"], edgecolor=PLOS["dark"], linewidth=0.5, label="Final-panel context", zorder=3)
    ax.set_xticks(x, metric_labels)
    ax.set_ylabel("Nearest-neighbor similarity")
    ymax_b = max(0.16, float(np.nanmax(np.r_[ref_bar, cand_bar])) * 1.22)
    ax.set_ylim(0, ymax_b)
    ax.set_title("Nearest-neighbor similarity summary", pad=10)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=True, handlelength=1.1)
    leg.get_frame().set_edgecolor(PLOS["edge"])
    leg.get_frame().set_linewidth(0.8)

    # C residue-category context
    ax = fig.add_subplot(gs[0, 2])
    add_panel_label(ax, "C", x=-0.14, y=1.08)
    style_axis(ax, grid_axis="y")
    categories = cat_df["category"].astype(str).tolist()
    x = np.arange(len(categories), dtype=float)
    width = 0.32
    ref_cat_vals = cat_df["reference_fraction"].to_numpy(dtype=float)
    final_cat_vals = cat_df["final_panel_fraction"].to_numpy(dtype=float)
    ax.bar(x - width/2, ref_cat_vals, width=width, color=PLOS["light"], edgecolor=PLOS["dark"], linewidth=0.4, label="Reference ACP corpus", zorder=3)
    ax.bar(x + width/2, final_cat_vals, width=width, color=PLOS["blue"], edgecolor=PLOS["dark"], linewidth=0.4, label="Final panel", zorder=3)
    ax.set_xticks(x, categories, rotation=18, ha="right")
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, max(0.34, float(np.nanmax(np.r_[ref_cat_vals, final_cat_vals])) * 1.18))
    ax.set_title("Residue-category context", pad=10)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=True, handlelength=1.0, columnspacing=0.8)
    leg.get_frame().set_edgecolor(PLOS["edge"])
    leg.get_frame().set_linewidth(0.8)

    fig.subplots_adjust(left=0.055, right=0.985, bottom=0.22, top=0.90)
    return save_multi(fig, out_base, dpi=dpi)


def plot_supplementary_s14(
    final_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    aa_df: pd.DataFrame,
    kmer_df: pd.DataFrame,
    source_data_dir: Path,
    out_base: Path,
    dpi: int,
) -> Dict[str, str]:
    """Generate Supplementary Figure S14 after approved panel swap.

    Final structure:
      A. Amino-acid enrichment
      B. Top enriched 3-mers
      C. Pairwise similarity within final panel
    """
    set_style()

    # Normalize AA/kmer columns
    final_seqs = final_df["sequence"].astype(str).tolist()
    ref_seqs = get_ref_sequences(reference_df)

    aa = normalize_columns(aa_df.copy()) if aa_df is not None else pd.DataFrame()
    if "log2_enrichment_vs_reference" not in aa.columns:
        col = first_col(aa, ["log2_enrichment", "enrichment", "log2fc", "log2_fold_change"])
        if col:
            aa["log2_enrichment_vs_reference"] = pd.to_numeric(aa[col], errors="coerce")
    if "amino_acid" not in aa.columns:
        col = first_col(aa, ["aa", "residue", "amino acid"])
        if col:
            aa["amino_acid"] = aa[col].astype(str)
    if "amino_acid" not in aa.columns or "log2_enrichment_vs_reference" not in aa.columns:
        aa = aa_enrichment(final_seqs, ref_seqs)
    aa["log2_enrichment_vs_reference"] = pd.to_numeric(aa["log2_enrichment_vs_reference"], errors="coerce")

    km = normalize_columns(kmer_df.copy()) if kmer_df is not None else pd.DataFrame()
    if "log2_enrichment_vs_reference" not in km.columns:
        col = first_col(km, ["log2_enrichment", "enrichment", "log2fc", "log2_fold_change"])
        if col:
            km["log2_enrichment_vs_reference"] = pd.to_numeric(km[col], errors="coerce")
    if "kmer" not in km.columns:
        col = first_col(km, ["3mer", "k_mer", "motif", "token"])
        if col:
            km["kmer"] = km[col].astype(str)
    if "kmer" not in km.columns or "log2_enrichment_vs_reference" not in km.columns:
        km = kmer_enrichment(final_seqs, ref_seqs, k=3, top_n=25)
    km["log2_enrichment_vs_reference"] = pd.to_numeric(km["log2_enrichment_vs_reference"], errors="coerce")

    final_ids = final_df["generated_id"].astype(str).tolist()
    mat = sorted_pairwise_matrix(pairwise, final_ids)

    aa.to_csv(source_data_dir / "Supplementary_Figure_S14_panel_a_source_data.csv", index=False)
    km.to_csv(source_data_dir / "Supplementary_Figure_S14_panel_b_source_data.csv", index=False)
    mat.reset_index().rename(columns={"index": "candidate_id"}).to_csv(source_data_dir / "Supplementary_Figure_S14_panel_c_source_data.csv", index=False)
    pd.concat([
        aa.assign(panel="S14A"),
        km.assign(panel="S14B"),
        mat.stack(dropna=False).reset_index(name="similarity").rename(columns={"level_0": "candidate_i", "level_1": "candidate_j"}).assign(panel="S14C"),
    ], ignore_index=True, sort=False).to_csv(source_data_dir / "Supplementary_Figure_S14_source_data_all_panels.csv", index=False)

    fig = plt.figure(figsize=(11.8, 6.3))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.02, 1.12], height_ratios=[0.88, 1.12], wspace=0.36, hspace=0.44)

    # A amino acid enrichment
    ax = fig.add_subplot(gs[:, 0])
    add_panel_label(ax, "A", x=-0.06, y=1.03, fontsize=16)
    style_axis(ax, grid_axis="x")
    tmp = aa[["amino_acid", "log2_enrichment_vs_reference"]].dropna().copy()
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    colors = [PLOS["light"] if v < 0 else PLOS["brown"] for v in vals]
    ax.barh(tmp["amino_acid"].astype(str), vals, color=colors, edgecolor=PLOS["dark"], linewidth=0.35, zorder=3)
    ax.axvline(0, color=PLOS["dark"], linewidth=0.8, zorder=4)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Amino-acid enrichment", loc="left", pad=8)

    # B kmer enrichment
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.08, y=1.08, fontsize=16)
    style_axis(ax, grid_axis="x")
    tmp = km[["kmer", "log2_enrichment_vs_reference"]].dropna().copy()
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=False).head(8)
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    ax.barh(tmp["kmer"].astype(str), tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float), color=PLOS["blue"], edgecolor=PLOS["dark"], linewidth=0.35, zorder=3)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Top enriched 3-mers", loc="left", pad=8)

    # C supplementary heatmap
    ax = fig.add_subplot(gs[1, 1])
    write_pairwise_source_and_heatmap(
        ax=ax,
        pairwise=pairwise,
        final_ids=final_ids,
        title="Pairwise similarity within final panel",
        colorbar_label="3-mer Jaccard similarity",
        panel_label="C",
    )

    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.13, top=0.92)
    return save_multi(fig, out_base, dpi=dpi)


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Generate OncoPep Step 9 Figure 5 and Supplementary Figure S14.", allow_abbrev=False)
    p.add_argument("--step9-root", default=DEFAULT_STEP9_ROOT)
    p.add_argument("--project-root", default=DEFAULT_PROJECT_ROOT)
    p.add_argument("--legacy-step9-root", default=DEFAULT_LEGACY_STEP9_ROOT)
    p.add_argument("--legacy-reference-file", default=DEFAULT_LEGACY_REFERENCE)
    p.add_argument("--final-panel-file", default=None)
    p.add_argument("--reference-file", default=None)
    p.add_argument("--pairwise-file", default=None)
    p.add_argument("--aa-enrichment-file", default=None)
    p.add_argument("--kmer-enrichment-file", default=None)
    p.add_argument("--paper-candidates-file", default=None)
    p.add_argument("--similarity-k", type=int, default=3)
    p.add_argument("--dpi", type=int, default=600)
    p.add_argument("--no-supplementary", action="store_true")
    p.add_argument("--max-reference-fallback", type=int, default=100000)
    p.add_argument("--quiet", action="store_true")
    return p


def load_inputs(args, checks: List[CheckResult], log: List[Dict[str, str]]) -> Dict[str, pd.DataFrame]:
    roots = safe_roots(args.project_root, args.legacy_step9_root, Path(args.project_root).parent if args.project_root else None)
    data: Dict[str, pd.DataFrame] = {}
    paths = {
        "final": resolve_input("final", args.final_panel_file, roots, log),
        "pairwise": resolve_input("pairwise", args.pairwise_file, roots, log),
        "reference": Path(args.reference_file) if args.reference_file else (Path(args.legacy_reference_file) if args.legacy_reference_file and Path(args.legacy_reference_file).exists() else resolve_input("reference", None, roots, log)),
        "aa": resolve_input("aa", args.aa_enrichment_file, roots, log),
        "kmer": resolve_input("kmer", args.kmer_enrichment_file, roots, log),
        "paper": resolve_input("paper", args.paper_candidates_file, roots, log),
    }
    data["_paths"] = pd.DataFrame([{"role": k, "path": str(v) if v else ""} for k, v in paths.items()])

    # Required final file
    if paths["final"] is None or not paths["final"].exists():
        checks.append(CheckResult("final_panel_file", "FAIL", "Final-panel annotation table missing. Supply --final-panel-file."))
        return data
    try:
        data["final"] = standardize_final_df(read_table(paths["final"]))
        checks.append(CheckResult("final_panel_file", "PASS", f"Loaded final panel: {paths['final']} ({len(data['final'])} rows)."))
    except Exception as e:
        checks.append(CheckResult("final_panel_file", "FAIL", f"Could not read final panel {paths['final']}: {e}"))
        return data

    # Reference is required for context fallback and S14
    if paths["reference"] is not None and paths["reference"].exists():
        try:
            data["reference"] = read_table(paths["reference"])
            checks.append(CheckResult("reference_file", "PASS", f"Loaded reference corpus: {paths['reference']} ({len(data['reference'])} rows)."))
        except Exception as e:
            checks.append(CheckResult("reference_file", "WARN", f"Could not read reference file {paths['reference']}: {e}"))
    else:
        checks.append(CheckResult("reference_file", "WARN", "Reference corpus file not found; nearest-reference fallback and S14 may be unavailable."))

    if paths["pairwise"] is not None and paths["pairwise"].exists():
        try:
            data["pairwise"] = read_pairwise_matrix(paths["pairwise"])
            checks.append(CheckResult("pairwise_file", "PASS", f"Loaded pairwise similarity matrix: {paths['pairwise']}."))
        except Exception as e:
            checks.append(CheckResult("pairwise_file", "WARN", f"Could not parse pairwise file; will compute from sequences if possible: {e}"))
    else:
        checks.append(CheckResult("pairwise_file", "WARN", "Pairwise similarity matrix not found; computing 3-mer Jaccard matrix from final-panel sequences."))

    for role, key in [("aa", "aa"), ("kmer", "kmer")]:
        if paths[role] is not None and paths[role].exists():
            try:
                data[key] = read_table(paths[role])
                checks.append(CheckResult(f"{role}_enrichment_file", "PASS", f"Loaded {role} enrichment table: {paths[role]}."))
            except Exception as e:
                checks.append(CheckResult(f"{role}_enrichment_file", "WARN", f"Could not read {role} enrichment table; will compute if possible: {e}"))
        else:
            checks.append(CheckResult(f"{role}_enrichment_file", "WARN", f"{role} enrichment table not found; will compute from final/reference sequences if possible."))

    return data


def prepare_data(data: Dict[str, pd.DataFrame], args, checks: List[CheckResult]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    final_df = data["final"].copy()
    reference_df = data.get("reference")

    if "pairwise" in data:
        pairwise = data["pairwise"].copy()
    else:
        pairwise = compute_pairwise_matrix(final_df["generated_id"].tolist(), final_df["sequence"].tolist(), k=args.similarity_k)
        checks.append(CheckResult("pairwise_computed", "WARN", f"Computed pairwise {args.similarity_k}-mer Jaccard matrix from final-panel sequences."))

    # Fill nearest reference if missing
    if "nearest_reference_similarity" not in final_df.columns or pd.to_numeric(final_df.get("nearest_reference_similarity", pd.Series(dtype=float)), errors="coerce").notna().sum() == 0:
        if reference_df is not None:
            final_df["nearest_reference_similarity"] = max_similarity_to_reference(final_df, reference_df, k=args.similarity_k, max_ref=args.max_reference_fallback)
            checks.append(CheckResult("nearest_reference_similarity", "WARN", "Computed nearest-reference similarity from sequences because input column was unavailable."))
        else:
            checks.append(CheckResult("nearest_reference_similarity", "FAIL", "Nearest-reference similarity is missing and no reference file is available."))
    else:
        final_df["nearest_reference_similarity"] = pd.to_numeric(final_df["nearest_reference_similarity"], errors="coerce")
        checks.append(CheckResult("nearest_reference_similarity", "PASS", "Using nearest-reference similarity column from final-panel table."))

    if "nearest_paper_candidate_similarity" not in final_df.columns or pd.to_numeric(final_df.get("nearest_paper_candidate_similarity", pd.Series(dtype=float)), errors="coerce").notna().sum() == 0:
        internal = nearest_internal_similarity(pairwise)
        # map by IDs
        id_map = {str(k): v for k, v in internal.items()}
        final_df["nearest_paper_candidate_similarity"] = final_df["generated_id"].astype(str).map(id_map)
        if final_df["nearest_paper_candidate_similarity"].isna().all():
            final_df["nearest_paper_candidate_similarity"] = internal.to_numpy()[:len(final_df)]
        checks.append(CheckResult("nearest_candidate_similarity", "WARN", "Computed candidate-context similarity from the pairwise final-panel matrix because input column was unavailable."))
    else:
        final_df["nearest_paper_candidate_similarity"] = pd.to_numeric(final_df["nearest_paper_candidate_similarity"], errors="coerce")
        checks.append(CheckResult("nearest_candidate_similarity", "PASS", "Using candidate-context similarity column from final-panel table."))

    aa_df = data.get("aa")
    km_df = data.get("kmer")
    if reference_df is not None:
        final_seqs = final_df["sequence"].astype(str).tolist()
        ref_seqs = get_ref_sequences(reference_df)
        if aa_df is None:
            aa_df = aa_enrichment(final_seqs, ref_seqs)
            checks.append(CheckResult("aa_enrichment", "WARN", "Computed amino-acid enrichment from final/reference sequences."))
        if km_df is None:
            km_df = kmer_enrichment(final_seqs, ref_seqs, k=args.similarity_k, top_n=25)
            checks.append(CheckResult("kmer_enrichment", "WARN", "Computed k-mer enrichment from final/reference sequences."))

    return final_df, pairwise, reference_df, aa_df, km_df


def write_reports(dirs: OutputDirs, checks: List[CheckResult], discovery: List[Dict[str, str]], files: Dict[str, str], data_paths: pd.DataFrame) -> Dict[str, str]:
    n_fail = sum(c.status == "FAIL" for c in checks)
    n_warn = sum(c.status == "WARN" for c in checks)
    status = "FAIL" if n_fail else ("WARN" if n_warn else "PASS")
    score = max(0, 100 - 15 * n_fail - 2 * n_warn)

    report = dirs.reports / "step9_readiness_report.md"
    lines = [
        "# OncoPep Step 9 readiness report\n",
        f"**Script version:** `{SCRIPT_VERSION}`\n",
        f"**Overall status:** `{status}`\n",
        f"**Estimated readiness score:** `{score}/100`\n",
        "## Validation checks\n",
    ]
    for c in checks:
        lines.append(f"- **[{c.status}] {c.name}**: {c.message}\n")
    lines.append("\n## Input paths\n")
    try:
        lines.append(data_paths.to_markdown(index=False) + "\n")
    except Exception:
        lines.append(data_paths.to_csv(index=False) + "\n")
    lines.append("\n## Discovery log\n")
    for rec in discovery:
        lines.append(f"- role=`{rec.get('role','')}` action=`{rec.get('action','')}` path=`{rec.get('path','')}` reason=`{rec.get('reason','')}`\n")
    lines.append("\n## Output files\n")
    for k, v in files.items():
        lines.append(f"- **{k}**: `{v}`\n")
    lines.append("\n## Non-duplication statement\n")
    lines.append("This Step 9 package uses contextual nearest-neighbor similarity, summary similarity statistics, main-figure residue-category context, and supplementary pairwise final-panel similarity. It excludes selection-audit and prioritization-score-shift panels because those belong to Step 8/Figure 4.\n")
    report.write_text("\n".join(lines), encoding="utf-8")
    files["readiness_report"] = str(report)

    manifest = {
        "script_version": SCRIPT_VERSION,
        "output_root": str(dirs.root),
        "checks": [asdict(c) for c in checks],
        "files": files,
        "input_paths": data_paths.to_dict(orient="records"),
        "discovery_log": discovery,
    }
    manifest_path = dirs.reports / "step9_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    files["manifest_json"] = str(manifest_path)

    readme = dirs.reports / "README_step9_outputs.txt"
    readme.write_text(
        f"OncoPep Step 9 output package\nScript version: {SCRIPT_VERSION}\n\n"
        "Main Figure 5: contextual similarity support and internal diversity of the final OncoPep panel.\n"
        "Supplementary Figure S14: compositional context of the final OncoPep panel.\n\n"
        "This package intentionally excludes Step 8 selection-audit/prioritization score-shift panels.\n",
        encoding="utf-8",
    )
    files["readme"] = str(readme)

    req = dirs.reports / "requirements_step9_minimal.txt"
    req.write_text("python>=3.10\nnumpy>=1.23\npandas>=1.5\nmatplotlib>=3.6\nopenpyxl>=3.0\n", encoding="utf-8")
    files["requirements_file"] = str(req)

    try:
        if "__file__" in globals():
            src = Path(__file__).resolve()
            if src.exists():
                dst = dirs.code / "OncoPep_step9_PLOS_contextual_similarity_diversity_final.py"
                shutil.copy2(src, dst)
                files["code_snapshot"] = str(dst)
    except Exception:
        pass
    return files


def main(argv: Optional[Sequence[str]] = None) -> int:
    parser = build_parser()
    args, unknown = parser.parse_known_args(clean_argv(argv))
    if not args.quiet:
        print(f"Hard-fix version: {SCRIPT_VERSION}")
        if unknown:
            print(f"Ignored unknown arguments: {unknown}")

    dirs = ensure_dirs(Path(args.step9_root).expanduser().resolve())
    checks: List[CheckResult] = []
    discovery: List[Dict[str, str]] = []
    data = load_inputs(args, checks, discovery)
    files: Dict[str, str] = {}

    if any(c.status == "FAIL" for c in checks):
        data_paths = data.get("_paths", pd.DataFrame())
        write_reports(dirs, checks, discovery, files, data_paths)
        msg = "Required Step 9 inputs are missing. Review step9_readiness_report.md."
        if is_notebook():
            raise RuntimeError(msg)
        print("ERROR:", msg, file=sys.stderr)
        return 2

    final_df, pairwise, reference_df, aa_df, km_df = prepare_data(data, args, checks)

    # Check post-preparation fatal conditions
    if final_df["nearest_reference_similarity"].isna().all():
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5A cannot be plotted: nearest-reference similarity unavailable."))
    if final_df["nearest_paper_candidate_similarity"].isna().all():
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5A/B cannot be plotted: candidate-context similarity unavailable."))
    if pairwise.shape[0] < 2:
        checks.append(CheckResult("plot_readiness", "FAIL", "Supplementary Figure S14C cannot be plotted: pairwise matrix has fewer than two candidates."))
    if reference_df is None:
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5C cannot be plotted: reference corpus is required for residue-category context."))

    # Save prepared final panel and matrix
    final_df.to_csv(dirs.source_data / "step9_final_panel_similarity_context_table.csv", index=False)
    pairwise.reset_index().rename(columns={"index": "candidate_id"}).to_csv(dirs.source_data / "step9_final_panel_similarity_matrix.csv", index=False)
    files["step9_final_panel_similarity_context_table"] = str(dirs.source_data / "step9_final_panel_similarity_context_table.csv")
    files["step9_final_panel_similarity_matrix"] = str(dirs.source_data / "step9_final_panel_similarity_matrix.csv")

    if any(c.status == "FAIL" for c in checks):
        write_reports(dirs, checks, discovery, files, data.get("_paths", pd.DataFrame()))
        msg = "Step 9 plotting readiness failed. Review step9_readiness_report.md."
        if is_notebook():
            raise RuntimeError(msg)
        print("ERROR:", msg, file=sys.stderr)
        return 2

    fig5_files = plot_figure5(
        final_df=final_df,
        pairwise=pairwise,
        reference_df=reference_df,
        source_data_dir=dirs.source_data,
        out_base=dirs.main_figure / "Figure_5_contextual_similarity_diversity",
        dpi=args.dpi,
    )
    for k, v in fig5_files.items():
        files[f"Figure_5_{k}"] = v
    checks.append(CheckResult("Figure_5", "PASS", "Generated redesigned Figure 5 with contextual NN similarity, NN summary, and residue-category context."))

    if not args.no_supplementary:
        if reference_df is not None and aa_df is not None and km_df is not None:
            s14_files = plot_supplementary_s14(
                final_df=final_df,
                reference_df=reference_df,
                pairwise=pairwise,
                aa_df=aa_df,
                kmer_df=km_df,
                source_data_dir=dirs.source_data,
                out_base=dirs.supplementary_figures / "Supplementary_Figure_S14_additional_context",
                dpi=args.dpi,
            )
            for k, v in s14_files.items():
                files[f"Supplementary_Figure_S14_{k}"] = v
            checks.append(CheckResult("Supplementary_Figure_S14", "PASS", "Generated Supplementary Figure S14 with amino-acid enrichment, 3-mer enrichment, and pairwise final-panel similarity."))
        else:
            checks.append(CheckResult("Supplementary_Figure_S14", "WARN", "S14 not generated because reference/composition data were unavailable."))

    # Additional source-data summary outputs
    sim_summary_rows = []
    for context, col in [("Reference context", "nearest_reference_similarity"), ("Candidate context", "nearest_paper_candidate_similarity")]:
        arr = pd.to_numeric(final_df[col], errors="coerce").dropna().to_numpy(dtype=float)
        sim_summary_rows.append({"context": context, **percentile_summary(arr)})
    sim_summary = pd.DataFrame(sim_summary_rows)
    sim_summary.to_csv(dirs.source_data / "step9_contextual_similarity_summary.csv", index=False)
    files["step9_contextual_similarity_summary"] = str(dirs.source_data / "step9_contextual_similarity_summary.csv")

    arr = pairwise.to_numpy(dtype=float)
    mask = ~np.eye(arr.shape[0], dtype=bool)
    offdiag = arr[mask]
    bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
    labels = ["0-0.05", "0.05-0.10", "0.10-0.15", "0.15-0.20", "0.20-0.30", "0.30-1.0"]
    cats = pd.cut(offdiag, bins=bins, labels=labels, include_lowest=True, right=False)
    div_summary = cats.value_counts().reindex(labels, fill_value=0).rename_axis("similarity_bin").reset_index(name="pair_count")
    div_summary["fraction"] = div_summary["pair_count"] / max(div_summary["pair_count"].sum(), 1)
    div_summary.to_csv(dirs.source_data / "step9_internal_diversity_summary.csv", index=False)
    files["step9_internal_diversity_summary"] = str(dirs.source_data / "step9_internal_diversity_summary.csv")

    files = write_reports(dirs, checks, discovery, files, data.get("_paths", pd.DataFrame()))

    n_fail = sum(c.status == "FAIL" for c in checks)
    n_warn = sum(c.status == "WARN" for c in checks)
    status = "FAIL" if n_fail else ("WARN" if n_warn else "PASS")
    score = max(0, 100 - 15*n_fail - 2*n_warn)
    if not args.quiet:
        print("\nOncoPep Step 9 package generated.")
        print(f"Root: {dirs.root}")
        print(f"Readiness status: {status}; estimated score: {score}/100")
        print(f"Main figure: {files.get('Figure_5_png')}")
        if "Supplementary_Figure_S14_png" in files:
            print(f"Supplementary figure: {files.get('Supplementary_Figure_S14_png')}")
        print(f"Readiness report: {files.get('readiness_report')}")
    return 0


if __name__ == "__main__":
    try:
        rc = main()
        if is_notebook() and rc != 0:
            raise RuntimeError(f"Step 9 script failed with exit code {rc}.")
        if not is_notebook():
            raise SystemExit(rc)
    except Exception as exc:
        if is_notebook():
            raise RuntimeError(str(exc)) from exc
        print("\nERROR: OncoPep Step 9 figure generation failed.\n", file=sys.stderr)
        print(str(exc), file=sys.stderr)
        traceback.print_exc()
        raise SystemExit(2)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 9 — PLOS Computational Biology Figure 5 rescue script.

Scientific role
---------------
Step 9 focuses on contextual similarity support and internal diversity of the
final OncoPep candidate panel. It intentionally avoids the Step 8 selection
and prioritization-score panels.

Main figure
-----------
Figure 5. Contextual similarity and compositional support of the final
OncoPep panel.
  A. NN similarity context
  B. Similarity summary
  C. Residue-category context

Supplementary figure
--------------------
Supplementary Figure S14. Additional sequence-composition context.
  A. Amino-acid enrichment
  B. Top enriched 3-mers

Removed from plotted supplementary figure
-----------------------------------------
The pairwise final-panel similarity heatmap is retained in source data and
reports, but is no longer plotted as S14C to keep the supplement compact.

Design decisions
----------------
- Rescues the scientifically stronger old Step 9 similarity/composition logic.
- Removes old Step 9 selection-audit and score-shift panels because they now
  duplicate Step 8/Figure 4.
- Uses uppercase panel labels, OncoPep/PLOS palette, source-data exports,
  manifest, README, requirements, code snapshot, and readiness reporting.
- Implements the approved panel swap: residue-category context is used as
  main Figure 5C.
- Generates Supplementary Figure S14 as a two-panel composition figure by
  default whenever amino-acid and k-mer enrichment data can be loaded or computed.
- Retains pairwise final-panel similarity as source data/report rather than a
  plotted supplementary panel.
- Avoids default all-1.00 contextual-support fallback panels.

Ready-to-run examples
---------------------
python OncoPep_step9_PLOS_contextual_similarity_diversity_final.py \
  --step9-root /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_09 \
  --project-root /home/data3/Moe/nature_computational_peponco \
  --legacy-step9-root /home/data3/Moe/nature_computational2/step9_v1

For explicit paths:
python OncoPep_step9_PLOS_contextual_similarity_diversity_final.py \
  --final-panel-file /path/to/table_s9_9_final_panel_with_all_step9_annotations.csv \
  --reference-file /path/to/step1_retained_dataset.csv \
  --pairwise-file /path/to/table_s9_7_final_panel_pairwise_similarity.csv \
  --aa-enrichment-file /path/to/table_s9_5_amino_acid_enrichment_vs_reference.csv \
  --kmer-enrichment-file /path/to/table_s9_6_kmer_enrichment_vs_reference.csv
"""

from __future__ import annotations

import argparse
import json
import math
import os
import shutil
import sys
import traceback
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

SCRIPT_VERSION = "v4_2026_07_08_step9_final_polish_two_panel_s14"

DEFAULT_STEP9_ROOT = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_09"
DEFAULT_PROJECT_ROOT = "/home/data3/Moe/nature_computational_peponco"
DEFAULT_LEGACY_STEP9_ROOT = "/home/data3/Moe/nature_computational2/step9_v1"
DEFAULT_LEGACY_REFERENCE = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"

SAFE_TABLE_EXTENSIONS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".pq"}
BLOCKED_EXTENSIONS = {".json", ".png", ".jpg", ".jpeg", ".pdf", ".tif", ".tiff", ".py", ".ipynb"}
BLOCKED_PATH_KEYWORDS = [
    "/run/user/",
    "/jupyter/runtime/",
    ".ipynb_checkpoints",
    "__pycache__",
    "/.git/",
    "kernel-",
]

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
    "edge": "#B8B8B8",
    "pale": "#F5F7F7",
}

AA_SET = set("ACDEFGHIKLMNPQRSTVWY")
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")

KD_HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}


@dataclass
class CheckResult:
    name: str
    status: str  # PASS/WARN/FAIL
    message: str


@dataclass
class OutputDirs:
    root: Path
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


def is_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        shell = get_ipython()
        if shell is None:
            return False
        return shell.__class__.__name__ in {"ZMQInteractiveShell", "TerminalInteractiveShell"}
    except Exception:
        return False


def clean_argv(argv: Optional[Sequence[str]] = None) -> List[str]:
    if argv is None:
        argv = sys.argv[1:]
    out: List[str] = []
    skip = False
    for i, token in enumerate(argv):
        if skip:
            skip = False
            continue
        s = str(token)
        if s == "-f":
            if i + 1 < len(argv):
                skip = True
            continue
        if any(k in s for k in BLOCKED_PATH_KEYWORDS):
            continue
        if s.endswith(".json") and ("kernel-" in s or "/runtime/" in s):
            continue
        out.append(s)
    return out


def ensure_dirs(root: Path) -> OutputDirs:
    dirs = OutputDirs(
        root=root,
        main_figure=root / "main_figure",
        supplementary_figures=root / "supplementary_figures",
        source_data=root / "source_data",
        reports=root / "reports",
        code=root / "code",
    )
    for d in asdict(dirs).values():
        if isinstance(d, Path):
            d.mkdir(parents=True, exist_ok=True)
    return dirs


def is_blocked_path(path: Path) -> bool:
    p = str(path).replace("\\", "/").lower()
    return any(k.lower() in p for k in BLOCKED_PATH_KEYWORDS)


def is_allowed_table_file(path: Path) -> bool:
    suffix = path.suffix.lower()
    return suffix in SAFE_TABLE_EXTENSIONS and suffix not in BLOCKED_EXTENSIONS and not is_blocked_path(path)


def safe_roots(*roots: Optional[str]) -> List[Path]:
    seen = set()
    out: List[Path] = []
    for r in roots:
        if not r:
            continue
        p = Path(r).expanduser()
        s = str(p.resolve()) if p.exists() else str(p)
        if s in seen:
            continue
        seen.add(s)
        if p.exists():
            out.append(p)
    return out


def discover_files(roots: List[Path]) -> List[Path]:
    found: List[Path] = []
    for root in roots:
        try:
            for p in root.rglob("*"):
                try:
                    if p.is_file() and is_allowed_table_file(p):
                        found.append(p)
                except Exception:
                    continue
        except Exception:
            continue
    return found


def score_path_for_role(path: Path, role: str) -> Tuple[int, List[str]]:
    s = str(path).lower().replace("\\", "/")
    name = path.name.lower()
    score = 0
    reasons: List[str] = []

    if "step9_v1/tables_supplementary" in s:
        score += 12; reasons.append("legacy_step9_supp")
    if "step_09" in s or "step9" in s:
        score += 6; reasons.append("step9_hint")
    if "step1" in s and role == "reference":
        score += 8; reasons.append("step1_reference_hint")

    patterns = {
        "final": ["table_s9_9_final_panel", "final_panel_with_all_step9_annotations", "final_panel", "final_12", "candidate"],
        "pairwise": ["table_s9_7_final_panel_pairwise_similarity", "pairwise_similarity", "similarity_matrix", "jaccard"],
        "reference": ["step1_retained_dataset", "retained_dataset", "reference", "training", "corpus"],
        "aa": ["table_s9_5_amino_acid", "amino_acid_enrichment", "aa_enrichment"],
        "kmer": ["table_s9_6_kmer", "kmer_enrichment", "3mer", "3_mer"],
        "paper": ["table_s9_11_paper_candidates", "paper_candidates", "harmonized"],
    }
    for patt in patterns.get(role, []):
        if patt in name:
            score += 10; reasons.append(f"name:{patt}")
    negative = ["readiness", "manifest", "requirements", "selection_audit", "stage_score", "score_shift", "figure", "source_data_all"]
    for n in negative:
        if n in name:
            score -= 8; reasons.append(f"penalty:{n}")
    return score, reasons


def resolve_input(role: str, explicit: Optional[str], roots: List[Path], log: List[Dict[str, str]]) -> Optional[Path]:
    if explicit:
        p = Path(explicit).expanduser()
        log.append({"role": role, "action": "explicit", "path": str(p), "reason": "user_supplied"})
        return p
    candidates = discover_files(roots)
    scored: List[Tuple[int, Path, List[str]]] = []
    for p in candidates:
        score, reasons = score_path_for_role(p, role)
        if score > 0:
            scored.append((score, p, reasons))
    if not scored:
        log.append({"role": role, "action": "not_found", "path": "", "reason": "no_safe_candidate"})
        return None
    scored.sort(key=lambda x: (-x[0], str(x[1])))
    score, p, reasons = scored[0]
    log.append({"role": role, "action": "auto", "path": str(p), "reason": f"score={score};" + ";".join(reasons)})
    return p


def read_table(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suf == ".txt":
        for sep in [",", "\t", None]:
            try:
                return pd.read_csv(path, sep=sep) if sep else pd.read_csv(path, delim_whitespace=True)
            except Exception:
                pass
        raise ValueError(f"Could not parse text table: {path}")
    if suf in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suf in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table extension: {path}")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def first_col(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    lower = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    return None


def standardize_final_df(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_columns(df)
    out = df.copy()
    seq_col = first_col(out, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        raise RuntimeError("Final-panel table needs a sequence column.")
    out["sequence"] = out[seq_col].astype(str).map(clean_sequence)

    id_col = first_col(out, ["generated_id", "candidate_id", "peptide_id", "id", "seq_id", "name"])
    if id_col is None:
        out["generated_id"] = [f"C{i:02d}" for i in range(1, len(out) + 1)]
    else:
        out["generated_id"] = out[id_col].astype(str)

    alias_map = {
        "nearest_reference_similarity": [
            "nearest_reference_similarity", "nn_reference_similarity", "nearest_ref_similarity",
            "max_reference_similarity", "reference_nn_similarity", "nn_similarity_reference"
        ],
        "nearest_paper_candidate_similarity": [
            "nearest_paper_candidate_similarity", "nearest_candidate_similarity", "nearest_final_similarity",
            "paper_candidate_similarity", "candidate_context_similarity", "nn_paper_candidate_similarity"
        ],
    }
    for std, aliases in alias_map.items():
        if std not in out.columns:
            col = first_col(out, aliases)
            if col is not None:
                out[std] = pd.to_numeric(out[col], errors="coerce")
    return out


def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    s = str(seq).strip().upper()
    return "".join(ch for ch in s if ch in AA_SET)


def kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}


def jaccard_sets(a: set, b: set) -> float:
    if not a and not b:
        return np.nan
    u = a | b
    if not u:
        return np.nan
    return len(a & b) / len(u)


def compute_pairwise_matrix(ids: Sequence[str], seqs: Sequence[str], k: int = 3) -> pd.DataFrame:
    ids = [str(x) for x in ids]
    ksets = [kmers(s, k=k) for s in seqs]
    n = len(ids)
    mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            mat[i, j] = 1.0 if i == j else jaccard_sets(ksets[i], ksets[j])
    return pd.DataFrame(mat, index=ids, columns=ids)


def read_pairwise_matrix(path: Path) -> pd.DataFrame:
    df = read_table(path)
    df = normalize_columns(df)
    # Matrix format: first column is index/candidate IDs and remaining are numeric columns
    first = df.columns[0]
    if not pd.api.types.is_numeric_dtype(df[first]):
        trial = df.set_index(first)
        numeric = trial.apply(pd.to_numeric, errors="coerce")
        # Accept if at least square-ish numeric matrix
        if numeric.shape[0] == numeric.shape[1] and numeric.notna().sum().sum() > 0:
            return numeric
    # Long format fallback
    cols = {c.lower(): c for c in df.columns}
    id1 = first_col(df, ["candidate_1", "id1", "source", "row", "candidate_i"])
    id2 = first_col(df, ["candidate_2", "id2", "target", "column", "candidate_j"])
    val = first_col(df, ["similarity", "jaccard", "jaccard_similarity", "3mer_jaccard", "value"])
    if id1 and id2 and val:
        tmp = df[[id1, id2, val]].copy()
        tmp[val] = pd.to_numeric(tmp[val], errors="coerce")
        mat = tmp.pivot_table(index=id1, columns=id2, values=val, aggfunc="first")
        # symmetrize where possible
        labels = sorted(set(mat.index.astype(str)) | set(mat.columns.astype(str)))
        mat = mat.reindex(index=labels, columns=labels)
        for a in labels:
            for b in labels:
                if pd.isna(mat.loc[a, b]) and not pd.isna(mat.loc[b, a]):
                    mat.loc[a, b] = mat.loc[b, a]
        np.fill_diagonal(mat.values, 1.0)
        return mat.apply(pd.to_numeric, errors="coerce")
    raise RuntimeError(f"Could not parse pairwise similarity matrix: {path}")


def max_similarity_to_reference(final_df: pd.DataFrame, ref_df: pd.DataFrame, k: int = 3, max_ref: int = 100000) -> pd.Series:
    ref = ref_df.copy()
    seq_col = first_col(ref, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        raise RuntimeError("Reference file needs a sequence column for fallback nearest-reference similarity.")
    ref_seqs = [clean_sequence(x) for x in ref[seq_col].astype(str).tolist()]
    ref_seqs = [s for s in ref_seqs if s]
    if len(ref_seqs) > max_ref:
        ref_seqs = ref_seqs[:max_ref]
    ref_sets = [kmers(s, k=k) for s in ref_seqs]
    vals = []
    for seq in final_df["sequence"].astype(str).tolist():
        sset = kmers(seq, k=k)
        if not sset or not ref_sets:
            vals.append(np.nan)
            continue
        mx = max(jaccard_sets(sset, rset) for rset in ref_sets)
        vals.append(float(mx))
    return pd.Series(vals, index=final_df.index)


def nearest_internal_similarity(pairwise: pd.DataFrame) -> pd.Series:
    mat = pairwise.copy().apply(pd.to_numeric, errors="coerce")
    vals = []
    for i, idx in enumerate(mat.index):
        arr = mat.iloc[i].to_numpy(dtype=float)
        if i < len(arr):
            arr[i] = np.nan
        vals.append(float(np.nanmax(arr)) if np.isfinite(arr).any() else np.nan)
    return pd.Series(vals, index=mat.index)


def percentile_summary(vals: np.ndarray) -> Dict[str, float]:
    vals = np.asarray([v for v in vals if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return {"n": 0, "median": np.nan, "p90": np.nan, "max": np.nan, "q1": np.nan, "q3": np.nan}
    return {
        "n": int(len(vals)),
        "median": float(np.nanmedian(vals)),
        "p90": float(np.nanpercentile(vals, 90)),
        "max": float(np.nanmax(vals)),
        "q1": float(np.nanpercentile(vals, 25)),
        "q3": float(np.nanpercentile(vals, 75)),
    }


def aa_enrichment(final_seqs: Sequence[str], ref_seqs: Sequence[str], pseudocount: float = 0.5) -> pd.DataFrame:
    fcnt = Counter(ch for s in final_seqs for ch in clean_sequence(s) if ch in AA_SET)
    rcnt = Counter(ch for s in ref_seqs for ch in clean_sequence(s) if ch in AA_SET)
    f_total = sum(fcnt.values())
    r_total = sum(rcnt.values())
    rows = []
    for aa in AA_ORDER:
        f = fcnt.get(aa, 0)
        r = rcnt.get(aa, 0)
        f_freq = (f + pseudocount) / (f_total + pseudocount * len(AA_ORDER)) if f_total else np.nan
        r_freq = (r + pseudocount) / (r_total + pseudocount * len(AA_ORDER)) if r_total else np.nan
        log2e = float(np.log2(f_freq / r_freq)) if pd.notna(f_freq) and pd.notna(r_freq) and r_freq > 0 else np.nan
        rows.append({"amino_acid": aa, "final_count": f, "reference_count": r, "final_frequency": f_freq, "reference_frequency": r_freq, "log2_enrichment_vs_reference": log2e})
    return pd.DataFrame(rows)


def kmer_enrichment(final_seqs: Sequence[str], ref_seqs: Sequence[str], k: int = 3, pseudocount: float = 0.5, top_n: int = 25) -> pd.DataFrame:
    fcnt = Counter(km for s in final_seqs for km in kmers(s, k=k))
    rcnt = Counter(km for s in ref_seqs for km in kmers(s, k=k))
    universe = sorted(set(fcnt) | set(rcnt))
    f_total = sum(fcnt.values())
    r_total = sum(rcnt.values())
    rows = []
    denom = max(len(universe), 1)
    for km in universe:
        f = fcnt.get(km, 0)
        r = rcnt.get(km, 0)
        f_freq = (f + pseudocount) / (f_total + pseudocount * denom) if f_total else np.nan
        r_freq = (r + pseudocount) / (r_total + pseudocount * denom) if r_total else np.nan
        log2e = float(np.log2(f_freq / r_freq)) if pd.notna(f_freq) and pd.notna(r_freq) and r_freq > 0 else np.nan
        rows.append({"kmer": km, "final_count": f, "reference_count": r, "final_frequency": f_freq, "reference_frequency": r_freq, "log2_enrichment_vs_reference": log2e})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    out = out.sort_values(["log2_enrichment_vs_reference", "final_count"], ascending=[False, False]).head(top_n)
    return out


def frac_of_categories(seqs: Sequence[str]) -> Dict[str, float]:
    seqs = [clean_sequence(s) for s in seqs]
    total = sum(len(s) for s in seqs)
    if total == 0:
        return {"Hydrophobic": np.nan, "Aromatic": np.nan, "Positive": np.nan, "Negative": np.nan, "Polar": np.nan}
    def frac(res_set: set) -> float:
        return sum(sum(ch in res_set for ch in s) for s in seqs) / total
    return {
        "Hydrophobic": frac(AA_HYDROPHOBIC),
        "Aromatic": frac(AA_AROMATIC),
        "Positive": frac(AA_POSITIVE),
        "Negative": frac(AA_NEGATIVE),
        "Polar": frac(AA_POLAR),
    }


def get_ref_sequences(ref_df: pd.DataFrame) -> List[str]:
    seq_col = first_col(ref_df, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        return []
    return [clean_sequence(x) for x in ref_df[seq_col].astype(str).tolist() if clean_sequence(x)]


def set_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 15,
        "axes.labelsize": 11,
        "xtick.labelsize": 9.5,
        "ytick.labelsize": 9.5,
        "legend.fontsize": 9.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": PLOS["grid"],
        "grid.linewidth": 0.8,
        "grid.alpha": 0.85,
        "axes.axisbelow": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(PLOS["edge"])
    ax.spines["bottom"].set_color(PLOS["edge"])
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(colors=PLOS["dark"], width=0.8, length=4)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.8, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")
    elif grid_axis == "both":
        ax.grid(True, axis="both", color=PLOS["grid"], linewidth=0.8, zorder=0)


def add_panel_label(ax, label: str, x: float = -0.14, y: float = 1.08, fontsize: int = 18) -> None:
    ax.text(x, y, label, transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="left", va="top", color=PLOS["dark"])


def draw_violin(ax, data: Sequence[float], pos: float, color: str, width: float = 0.72) -> None:
    arr = np.asarray([x for x in data if pd.notna(x)], dtype=float)
    if len(arr) == 0:
        return
    parts = ax.violinplot([arr], positions=[pos], widths=width, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]:
        body.set_facecolor(color)
        body.set_edgecolor("none")
        body.set_alpha(0.78)
    q1, med, q3 = np.nanpercentile(arr, [25, 50, 75])
    ax.plot([pos, pos], [q1, q3], color=PLOS["dark"], linewidth=1.2, zorder=4)
    ax.scatter([pos], [med], s=34, facecolor=PLOS["white"], edgecolor=PLOS["dark"], linewidth=0.8, zorder=5)
    # Subtle points
    rng = np.random.default_rng(42 + int(pos * 10))
    jitter = rng.uniform(-0.065, 0.065, size=len(arr))
    ax.scatter(np.full(len(arr), pos) + jitter, arr, s=10, facecolor=PLOS["white"], edgecolor=PLOS["dark"], linewidth=0.35, alpha=0.65, zorder=4)


def save_multi(fig: plt.Figure, out_base: Path, dpi: int) -> Dict[str, str]:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths: Dict[str, str] = {}
    for ext in [".png", ".pdf", ".tiff"]:
        p = out_base.with_suffix(ext)
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        paths[ext.lstrip(".")] = str(p)
    plt.close(fig)
    return paths


def short_label(x: str, max_len: int = 12) -> str:
    s = str(x)
    return s if len(s) <= max_len else s[:max_len]


def sorted_pairwise_matrix(pairwise: pd.DataFrame, final_ids: Sequence[str]) -> pd.DataFrame:
    mat = pairwise.copy()
    mat.index = mat.index.astype(str)
    mat.columns = mat.columns.astype(str)
    ids = [str(i) for i in final_ids]
    available = [i for i in ids if i in mat.index and i in mat.columns]
    if len(available) >= 2:
        mat = mat.loc[available, available]
    else:
        # keep original square matrix
        pass
    numeric = mat.apply(pd.to_numeric, errors="coerce")
    # order by mean off-diagonal similarity to make layout intentional
    vals = numeric.to_numpy(dtype=float)
    np.fill_diagonal(vals, np.nan)
    avg = np.nanmean(vals, axis=1)
    order = np.argsort(-np.nan_to_num(avg, nan=-1))
    numeric = numeric.iloc[order, :].iloc[:, order]
    return numeric


def residue_category_df(final_df: pd.DataFrame, reference_df: pd.DataFrame) -> pd.DataFrame:
    """Return residue-category fractions for reference corpus and final panel."""
    final_seqs = final_df["sequence"].astype(str).tolist()
    ref_seqs = get_ref_sequences(reference_df)
    ref_cat = frac_of_categories(ref_seqs)
    final_cat = frac_of_categories(final_seqs)
    categories = ["Hydrophobic", "Aromatic", "Positive", "Negative", "Polar"]
    return pd.DataFrame({
        "category": categories,
        "reference_fraction": [ref_cat[c] for c in categories],
        "final_panel_fraction": [final_cat[c] for c in categories],
    })


def write_pairwise_source_and_heatmap(
    ax,
    pairwise: pd.DataFrame,
    final_ids: Sequence[str],
    title: str,
    colorbar_label: str = "3-mer Jaccard similarity",
    panel_label: Optional[str] = None,
) -> pd.DataFrame:
    """Draw polished supplementary heatmap and return ordered matrix."""
    mat = sorted_pairwise_matrix(pairwise, final_ids)
    arr = mat.to_numpy(dtype=float)
    if arr.shape[0] == 0:
        raise RuntimeError("Pairwise matrix has no rows after ordering.")
    diag_mask = np.eye(arr.shape[0], dtype=bool)
    offdiag = arr.copy()
    offdiag[diag_mask] = np.nan
    max_offdiag = float(np.nanmax(offdiag)) if np.isfinite(offdiag).any() else 0.20
    vmax = min(1.0, max(0.20, math.ceil(max_offdiag * 10) / 10))
    cmap = LinearSegmentedColormap.from_list(
        "oncopep_similarity",
        [PLOS["white"], PLOS["mint"], PLOS["blue"]],
    )
    plot_arr = arr.copy()
    plot_arr[diag_mask] = np.nan
    masked = np.ma.masked_invalid(plot_arr)
    cmap.set_bad(color=PLOS["pale"])
    im = ax.imshow(masked, vmin=0, vmax=vmax, cmap=cmap, aspect="equal")
    labels = [short_label(x, 12) for x in mat.index.tolist()]
    ax.set_xticks(np.arange(len(labels)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(labels)), labels, fontsize=8)
    ax.set_title(title, loc="left", pad=8)
    if panel_label:
        add_panel_label(ax, panel_label, x=-0.08, y=1.06, fontsize=16)
    ax.set_xticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.grid(which="minor", color=PLOS["white"], linestyle="-", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.042, pad=0.035)
    cbar.set_label(colorbar_label)
    return mat


def plot_figure5(
    final_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    reference_df: pd.DataFrame,
    source_data_dir: Path,
    out_base: Path,
    dpi: int,
) -> Dict[str, str]:
    """Generate final Figure 5 after approved panel swap.

    Final structure:
      A. NN similarity context
      B. Similarity summary
      C. Residue-category context
    """
    set_style()

    ref_vals = pd.to_numeric(final_df["nearest_reference_similarity"], errors="coerce").dropna().to_numpy(dtype=float)
    cand_col = "nearest_paper_candidate_similarity"
    cand_vals = pd.to_numeric(final_df[cand_col], errors="coerce").dropna().to_numpy(dtype=float)

    summary_rows = []
    for name, arr in [("Reference context", ref_vals), ("Final-panel context", cand_vals)]:
        st = percentile_summary(arr)
        summary_rows.append({"context": name, **st})
    summary_df = pd.DataFrame(summary_rows)

    panel_a_df = pd.concat([
        pd.DataFrame({"context": "Reference context", "nearest_neighbor_similarity": ref_vals}),
        pd.DataFrame({"context": "Final-panel context", "nearest_neighbor_similarity": cand_vals}),
    ], ignore_index=True)
    panel_a_df.to_csv(source_data_dir / "Figure_5_panel_a_source_data.csv", index=False)

    panel_b_rows = []
    for _, r in summary_df.iterrows():
        for metric in ["median", "p90", "max"]:
            panel_b_rows.append({"context": r["context"], "summary_metric": metric, "similarity": r[metric], "n": r["n"]})
    panel_b_df = pd.DataFrame(panel_b_rows)
    panel_b_df.to_csv(source_data_dir / "Figure_5_panel_b_source_data.csv", index=False)

    cat_df = residue_category_df(final_df, reference_df)
    cat_df.to_csv(source_data_dir / "Figure_5_panel_c_source_data.csv", index=False)

    combined = pd.concat([
        panel_a_df.assign(panel="Figure_5A"),
        panel_b_df.assign(panel="Figure_5B"),
        cat_df.assign(panel="Figure_5C"),
    ], ignore_index=True, sort=False)
    combined.to_csv(source_data_dir / "Figure_5_source_data_all_panels.csv", index=False)

    fig = plt.figure(figsize=(15.8, 5.15))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[0.88, 0.92, 1.10], wspace=0.50)

    # A violin
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.18, y=1.08)
    style_axis(ax, grid_axis="y")
    draw_violin(ax, ref_vals, 0, PLOS["blue"], width=0.78)
    draw_violin(ax, cand_vals, 1, PLOS["coral"], width=0.78)
    ymax = max(0.16, float(np.nanmax(np.r_[ref_vals, cand_vals])) * 1.22 if len(np.r_[ref_vals, cand_vals]) else 0.16)
    ax.set_ylim(0, ymax)
    ax.set_xticks([0, 1], ["Reference\ncontext", "Final-panel\ncontext"])
    ax.set_ylabel("Nearest-neighbor similarity")
    ax.set_title("NN similarity context", pad=10)

    # B summary bars
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.18, y=1.08)
    style_axis(ax, grid_axis="y")
    metrics = ["median", "p90", "max"]
    metric_labels = ["Median", "P90", "Max"]
    x = np.arange(len(metrics), dtype=float)
    width = 0.32
    ref_s = summary_df.set_index("context").loc["Reference context"]
    cand_s = summary_df.set_index("context").loc["Final-panel context"]
    ref_bar = [float(ref_s[m]) for m in metrics]
    cand_bar = [float(cand_s[m]) for m in metrics]
    ax.bar(x - width/2, ref_bar, width=width, color=PLOS["blue"], edgecolor=PLOS["dark"], linewidth=0.5, label="Reference context", zorder=3)
    ax.bar(x + width/2, cand_bar, width=width, color=PLOS["coral"], edgecolor=PLOS["dark"], linewidth=0.5, label="Final-panel context", zorder=3)
    ax.set_xticks(x, metric_labels)
    ax.set_ylabel("Nearest-neighbor similarity")
    ymax_b = max(0.16, float(np.nanmax(np.r_[ref_bar, cand_bar])) * 1.22)
    ax.set_ylim(0, ymax_b)
    ax.set_title("Similarity summary", pad=10)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=True, handlelength=1.0, columnspacing=0.85, handletextpad=0.45, borderaxespad=0.3)
    leg.get_frame().set_edgecolor(PLOS["edge"])
    leg.get_frame().set_linewidth(0.8)

    # C residue-category context
    ax = fig.add_subplot(gs[0, 2])
    add_panel_label(ax, "C", x=-0.14, y=1.08)
    style_axis(ax, grid_axis="y")
    categories = cat_df["category"].astype(str).tolist()
    x = np.arange(len(categories), dtype=float)
    width = 0.32
    ref_cat_vals = cat_df["reference_fraction"].to_numpy(dtype=float)
    final_cat_vals = cat_df["final_panel_fraction"].to_numpy(dtype=float)
    ax.bar(x - width/2, ref_cat_vals, width=width, color=PLOS["light"], edgecolor=PLOS["dark"], linewidth=0.4, label="Reference ACP corpus", zorder=3)
    ax.bar(x + width/2, final_cat_vals, width=width, color=PLOS["blue"], edgecolor=PLOS["dark"], linewidth=0.4, label="Final panel", zorder=3)
    ax.set_xticks(x, categories, rotation=18, ha="right")
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, max(0.34, float(np.nanmax(np.r_[ref_cat_vals, final_cat_vals])) * 1.18))
    ax.set_title("Residue-category context", pad=10)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=True, handlelength=1.0, columnspacing=0.8)
    leg.get_frame().set_edgecolor(PLOS["edge"])
    leg.get_frame().set_linewidth(0.8)

    fig.subplots_adjust(left=0.055, right=0.985, bottom=0.22, top=0.90)
    return save_multi(fig, out_base, dpi=dpi)


def plot_supplementary_s14(
    final_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    aa_df: pd.DataFrame,
    kmer_df: pd.DataFrame,
    source_data_dir: Path,
    out_base: Path,
    dpi: int,
) -> Dict[str, str]:
    """Generate polished two-panel Supplementary Figure S14.

    Final plotted structure:
      A. Amino-acid enrichment
      B. Top enriched 3-mers

    Pairwise final-panel similarity is not plotted here. It is retained as
    source data and documented in step9_removed_panel_report.csv to avoid a
    crowded supplementary figure.
    """
    set_style()

    final_seqs = final_df["sequence"].astype(str).tolist()
    ref_seqs = get_ref_sequences(reference_df)

    # Normalize AA enrichment table or compute fallback
    aa = normalize_columns(aa_df.copy()) if aa_df is not None else pd.DataFrame()
    if "log2_enrichment_vs_reference" not in aa.columns:
        col = first_col(aa, ["log2_enrichment", "enrichment", "log2fc", "log2_fold_change"])
        if col:
            aa["log2_enrichment_vs_reference"] = pd.to_numeric(aa[col], errors="coerce")
    if "amino_acid" not in aa.columns:
        col = first_col(aa, ["aa", "residue", "amino acid"])
        if col:
            aa["amino_acid"] = aa[col].astype(str)
    if "amino_acid" not in aa.columns or "log2_enrichment_vs_reference" not in aa.columns:
        aa = aa_enrichment(final_seqs, ref_seqs)
    aa["amino_acid"] = aa["amino_acid"].astype(str)
    aa["log2_enrichment_vs_reference"] = pd.to_numeric(aa["log2_enrichment_vs_reference"], errors="coerce")

    # Normalize k-mer enrichment table or compute fallback
    km = normalize_columns(kmer_df.copy()) if kmer_df is not None else pd.DataFrame()
    if "log2_enrichment_vs_reference" not in km.columns:
        col = first_col(km, ["log2_enrichment", "enrichment", "log2fc", "log2_fold_change"])
        if col:
            km["log2_enrichment_vs_reference"] = pd.to_numeric(km[col], errors="coerce")
    if "kmer" not in km.columns:
        col = first_col(km, ["3mer", "k_mer", "motif", "token"])
        if col:
            km["kmer"] = km[col].astype(str)
    if "kmer" not in km.columns or "log2_enrichment_vs_reference" not in km.columns:
        km = kmer_enrichment(final_seqs, ref_seqs, k=3, top_n=25)
    km["kmer"] = km["kmer"].astype(str)
    km["log2_enrichment_vs_reference"] = pd.to_numeric(km["log2_enrichment_vs_reference"], errors="coerce")

    # Retain pairwise matrix as source data/report, but do not plot it.
    final_ids = final_df["generated_id"].astype(str).tolist()
    mat = sorted_pairwise_matrix(pairwise, final_ids)

    aa_out = source_data_dir / "Supplementary_Figure_S14_panel_a_source_data.csv"
    km_out = source_data_dir / "Supplementary_Figure_S14_panel_b_source_data.csv"
    matrix_out = source_data_dir / "step9_pairwise_similarity_matrix.csv"
    removed_out = source_data_dir / "step9_removed_panel_report.csv"

    aa.to_csv(aa_out, index=False)
    km.to_csv(km_out, index=False)
    mat.reset_index().rename(columns={"index": "candidate_id"}).to_csv(matrix_out, index=False)

    removed_report = pd.DataFrame([
        {
            "removed_panel": "Supplementary_Figure_S14C_pairwise_similarity_heatmap",
            "decision": "removed_from_plotted_supplementary_figure",
            "reason": "Dense candidate-level heatmap was visually weaker than the two-panel composition figure and is retained as source data/report instead.",
            "retained_source_data": str(matrix_out),
            "manuscript_role": "source_data_internal_diversity_support",
        }
    ])
    removed_report.to_csv(removed_out, index=False)

    pd.concat([
        aa.assign(panel="S14A"),
        km.assign(panel="S14B"),
    ], ignore_index=True, sort=False).to_csv(source_data_dir / "Supplementary_Figure_S14_source_data_all_panels.csv", index=False)

    # Two-panel layout: large AA enrichment + k-mer enrichment
    fig = plt.figure(figsize=(12.6, 5.65))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.08, 1.0], wspace=0.36)

    # A amino-acid enrichment
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.07, y=1.04, fontsize=17)
    style_axis(ax, grid_axis="x")
    tmp = aa[["amino_acid", "log2_enrichment_vs_reference"]].dropna().copy()
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    colors = [PLOS["light"] if v < 0 else PLOS["brown"] for v in vals]
    ax.barh(tmp["amino_acid"].astype(str), vals, color=colors, edgecolor=PLOS["dark"], linewidth=0.35, zorder=3)
    ax.axvline(0, color=PLOS["dark"], linewidth=0.9, zorder=4)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Amino-acid enrichment", loc="left", pad=8)

    # B top enriched 3-mers with restrained ordered palette
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.08, y=1.04, fontsize=17)
    style_axis(ax, grid_axis="x")
    tmp = km[["kmer", "log2_enrichment_vs_reference"]].dropna().copy()
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=False).head(8)
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    ordered_palette = [
        PLOS["mint"], PLOS["blue"], PLOS["blue"], PLOS["brown"],
        PLOS["coral"], PLOS["brown"], PLOS["blue"], PLOS["charcoal"],
    ]
    colors = ordered_palette[-len(vals):]
    ax.barh(tmp["kmer"].astype(str), vals, color=colors, edgecolor=PLOS["dark"], linewidth=0.35, zorder=3)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Top enriched 3-mers", loc="left", pad=8)

    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.15, top=0.90)
    return save_multi(fig, out_base, dpi=dpi)

def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Generate OncoPep Step 9 Figure 5 and Supplementary Figure S14.", allow_abbrev=False)
    p.add_argument("--step9-root", default=DEFAULT_STEP9_ROOT)
    p.add_argument("--project-root", default=DEFAULT_PROJECT_ROOT)
    p.add_argument("--legacy-step9-root", default=DEFAULT_LEGACY_STEP9_ROOT)
    p.add_argument("--legacy-reference-file", default=DEFAULT_LEGACY_REFERENCE)
    p.add_argument("--final-panel-file", default=None)
    p.add_argument("--reference-file", default=None)
    p.add_argument("--pairwise-file", default=None)
    p.add_argument("--aa-enrichment-file", default=None)
    p.add_argument("--kmer-enrichment-file", default=None)
    p.add_argument("--paper-candidates-file", default=None)
    p.add_argument("--similarity-k", type=int, default=3)
    p.add_argument("--dpi", type=int, default=600)
    p.add_argument("--no-supplementary", action="store_true")
    p.add_argument("--max-reference-fallback", type=int, default=100000)
    p.add_argument("--quiet", action="store_true")
    return p


def load_inputs(args, checks: List[CheckResult], log: List[Dict[str, str]]) -> Dict[str, pd.DataFrame]:
    roots = safe_roots(args.project_root, args.legacy_step9_root, Path(args.project_root).parent if args.project_root else None)
    data: Dict[str, pd.DataFrame] = {}
    paths = {
        "final": resolve_input("final", args.final_panel_file, roots, log),
        "pairwise": resolve_input("pairwise", args.pairwise_file, roots, log),
        "reference": Path(args.reference_file) if args.reference_file else (Path(args.legacy_reference_file) if args.legacy_reference_file and Path(args.legacy_reference_file).exists() else resolve_input("reference", None, roots, log)),
        "aa": resolve_input("aa", args.aa_enrichment_file, roots, log),
        "kmer": resolve_input("kmer", args.kmer_enrichment_file, roots, log),
        "paper": resolve_input("paper", args.paper_candidates_file, roots, log),
    }
    data["_paths"] = pd.DataFrame([{"role": k, "path": str(v) if v else ""} for k, v in paths.items()])

    # Required final file
    if paths["final"] is None or not paths["final"].exists():
        checks.append(CheckResult("final_panel_file", "FAIL", "Final-panel annotation table missing. Supply --final-panel-file."))
        return data
    try:
        data["final"] = standardize_final_df(read_table(paths["final"]))
        checks.append(CheckResult("final_panel_file", "PASS", f"Loaded final panel: {paths['final']} ({len(data['final'])} rows)."))
    except Exception as e:
        checks.append(CheckResult("final_panel_file", "FAIL", f"Could not read final panel {paths['final']}: {e}"))
        return data

    # Reference is required for context fallback and S14
    if paths["reference"] is not None and paths["reference"].exists():
        try:
            data["reference"] = read_table(paths["reference"])
            checks.append(CheckResult("reference_file", "PASS", f"Loaded reference corpus: {paths['reference']} ({len(data['reference'])} rows)."))
        except Exception as e:
            checks.append(CheckResult("reference_file", "WARN", f"Could not read reference file {paths['reference']}: {e}"))
    else:
        checks.append(CheckResult("reference_file", "WARN", "Reference corpus file not found; nearest-reference fallback and S14 may be unavailable."))

    if paths["pairwise"] is not None and paths["pairwise"].exists():
        try:
            data["pairwise"] = read_pairwise_matrix(paths["pairwise"])
            checks.append(CheckResult("pairwise_file", "PASS", f"Loaded pairwise similarity matrix: {paths['pairwise']}."))
        except Exception as e:
            checks.append(CheckResult("pairwise_file", "WARN", f"Could not parse pairwise file; will compute from sequences if possible: {e}"))
    else:
        checks.append(CheckResult("pairwise_file", "WARN", "Pairwise similarity matrix not found; computing 3-mer Jaccard matrix from final-panel sequences."))

    for role, key in [("aa", "aa"), ("kmer", "kmer")]:
        if paths[role] is not None and paths[role].exists():
            try:
                data[key] = read_table(paths[role])
                checks.append(CheckResult(f"{role}_enrichment_file", "PASS", f"Loaded {role} enrichment table: {paths[role]}."))
            except Exception as e:
                checks.append(CheckResult(f"{role}_enrichment_file", "WARN", f"Could not read {role} enrichment table; will compute if possible: {e}"))
        else:
            checks.append(CheckResult(f"{role}_enrichment_file", "WARN", f"{role} enrichment table not found; will compute from final/reference sequences if possible."))

    return data


def prepare_data(data: Dict[str, pd.DataFrame], args, checks: List[CheckResult]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    final_df = data["final"].copy()
    reference_df = data.get("reference")

    if "pairwise" in data:
        pairwise = data["pairwise"].copy()
    else:
        pairwise = compute_pairwise_matrix(final_df["generated_id"].tolist(), final_df["sequence"].tolist(), k=args.similarity_k)
        checks.append(CheckResult("pairwise_computed", "WARN", f"Computed pairwise {args.similarity_k}-mer Jaccard matrix from final-panel sequences."))

    # Fill nearest reference if missing
    if "nearest_reference_similarity" not in final_df.columns or pd.to_numeric(final_df.get("nearest_reference_similarity", pd.Series(dtype=float)), errors="coerce").notna().sum() == 0:
        if reference_df is not None:
            final_df["nearest_reference_similarity"] = max_similarity_to_reference(final_df, reference_df, k=args.similarity_k, max_ref=args.max_reference_fallback)
            checks.append(CheckResult("nearest_reference_similarity", "WARN", "Computed nearest-reference similarity from sequences because input column was unavailable."))
        else:
            checks.append(CheckResult("nearest_reference_similarity", "FAIL", "Nearest-reference similarity is missing and no reference file is available."))
    else:
        final_df["nearest_reference_similarity"] = pd.to_numeric(final_df["nearest_reference_similarity"], errors="coerce")
        checks.append(CheckResult("nearest_reference_similarity", "PASS", "Using nearest-reference similarity column from final-panel table."))

    if "nearest_paper_candidate_similarity" not in final_df.columns or pd.to_numeric(final_df.get("nearest_paper_candidate_similarity", pd.Series(dtype=float)), errors="coerce").notna().sum() == 0:
        internal = nearest_internal_similarity(pairwise)
        # map by IDs
        id_map = {str(k): v for k, v in internal.items()}
        final_df["nearest_paper_candidate_similarity"] = final_df["generated_id"].astype(str).map(id_map)
        if final_df["nearest_paper_candidate_similarity"].isna().all():
            final_df["nearest_paper_candidate_similarity"] = internal.to_numpy()[:len(final_df)]
        checks.append(CheckResult("nearest_candidate_similarity", "WARN", "Computed candidate-context similarity from the pairwise final-panel matrix because input column was unavailable."))
    else:
        final_df["nearest_paper_candidate_similarity"] = pd.to_numeric(final_df["nearest_paper_candidate_similarity"], errors="coerce")
        checks.append(CheckResult("nearest_candidate_similarity", "PASS", "Using candidate-context similarity column from final-panel table."))

    aa_df = data.get("aa")
    km_df = data.get("kmer")
    if reference_df is not None:
        final_seqs = final_df["sequence"].astype(str).tolist()
        ref_seqs = get_ref_sequences(reference_df)
        if aa_df is None:
            aa_df = aa_enrichment(final_seqs, ref_seqs)
            checks.append(CheckResult("aa_enrichment", "WARN", "Computed amino-acid enrichment from final/reference sequences."))
        if km_df is None:
            km_df = kmer_enrichment(final_seqs, ref_seqs, k=args.similarity_k, top_n=25)
            checks.append(CheckResult("kmer_enrichment", "WARN", "Computed k-mer enrichment from final/reference sequences."))

    return final_df, pairwise, reference_df, aa_df, km_df


def write_reports(dirs: OutputDirs, checks: List[CheckResult], discovery: List[Dict[str, str]], files: Dict[str, str], data_paths: pd.DataFrame) -> Dict[str, str]:
    n_fail = sum(c.status == "FAIL" for c in checks)
    n_warn = sum(c.status == "WARN" for c in checks)
    status = "FAIL" if n_fail else ("WARN" if n_warn else "PASS")
    score = max(0, 100 - 15 * n_fail - 2 * n_warn)

    report = dirs.reports / "step9_readiness_report.md"
    lines = [
        "# OncoPep Step 9 readiness report\n",
        f"**Script version:** `{SCRIPT_VERSION}`\n",
        f"**Overall status:** `{status}`\n",
        f"**Estimated readiness score:** `{score}/100`\n",
        "## Validation checks\n",
    ]
    for c in checks:
        lines.append(f"- **[{c.status}] {c.name}**: {c.message}\n")
    lines.append("\n## Input paths\n")
    try:
        lines.append(data_paths.to_markdown(index=False) + "\n")
    except Exception:
        lines.append(data_paths.to_csv(index=False) + "\n")
    lines.append("\n## Discovery log\n")
    for rec in discovery:
        lines.append(f"- role=`{rec.get('role','')}` action=`{rec.get('action','')}` path=`{rec.get('path','')}` reason=`{rec.get('reason','')}`\n")
    lines.append("\n## Output files\n")
    for k, v in files.items():
        lines.append(f"- **{k}**: `{v}`\n")
    lines.append("\n## Non-duplication statement\n")
    lines.append("This Step 9 package uses contextual nearest-neighbor similarity, summary similarity statistics, main-figure residue-category context, and a two-panel supplementary sequence-composition figure. The pairwise final-panel similarity heatmap was removed from the plotted supplement and retained as source data/report to avoid a crowded supplementary figure. The package excludes selection-audit and prioritization-score-shift panels because those belong to Step 8/Figure 4.\n")
    report.write_text("\n".join(lines), encoding="utf-8")
    files["readiness_report"] = str(report)

    manifest = {
        "script_version": SCRIPT_VERSION,
        "output_root": str(dirs.root),
        "checks": [asdict(c) for c in checks],
        "files": files,
        "input_paths": data_paths.to_dict(orient="records"),
        "discovery_log": discovery,
    }
    manifest_path = dirs.reports / "step9_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    files["manifest_json"] = str(manifest_path)

    readme = dirs.reports / "README_step9_outputs.txt"
    readme.write_text(
        f"OncoPep Step 9 output package\nScript version: {SCRIPT_VERSION}\n\n"
        "Main Figure 5: contextual similarity support and internal diversity of the final OncoPep panel.\n"
        "Supplementary Figure S14: compositional context of the final OncoPep panel.\n\n"
        "This package intentionally excludes Step 8 selection-audit/prioritization score-shift panels.\n",
        encoding="utf-8",
    )
    files["readme"] = str(readme)

    req = dirs.reports / "requirements_step9_minimal.txt"
    req.write_text("python>=3.10\nnumpy>=1.23\npandas>=1.5\nmatplotlib>=3.6\nopenpyxl>=3.0\n", encoding="utf-8")
    files["requirements_file"] = str(req)

    try:
        if "__file__" in globals():
            src = Path(__file__).resolve()
            if src.exists():
                dst = dirs.code / "OncoPep_step9_PLOS_contextual_similarity_diversity_final.py"
                shutil.copy2(src, dst)
                files["code_snapshot"] = str(dst)
    except Exception:
        pass
    return files


def main(argv: Optional[Sequence[str]] = None) -> int:
    parser = build_parser()
    args, unknown = parser.parse_known_args(clean_argv(argv))
    if not args.quiet:
        print(f"Hard-fix version: {SCRIPT_VERSION}")
        if unknown:
            print(f"Ignored unknown arguments: {unknown}")

    dirs = ensure_dirs(Path(args.step9_root).expanduser().resolve())
    checks: List[CheckResult] = []
    discovery: List[Dict[str, str]] = []
    data = load_inputs(args, checks, discovery)
    files: Dict[str, str] = {}

    if any(c.status == "FAIL" for c in checks):
        data_paths = data.get("_paths", pd.DataFrame())
        write_reports(dirs, checks, discovery, files, data_paths)
        msg = "Required Step 9 inputs are missing. Review step9_readiness_report.md."
        if is_notebook():
            raise RuntimeError(msg)
        print("ERROR:", msg, file=sys.stderr)
        return 2

    final_df, pairwise, reference_df, aa_df, km_df = prepare_data(data, args, checks)

    # Check post-preparation fatal conditions
    if final_df["nearest_reference_similarity"].isna().all():
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5A cannot be plotted: nearest-reference similarity unavailable."))
    if final_df["nearest_paper_candidate_similarity"].isna().all():
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5A/B cannot be plotted: candidate-context similarity unavailable."))
    if pairwise.shape[0] < 2:
        checks.append(CheckResult("plot_readiness", "FAIL", "Pairwise internal-diversity source data cannot be summarized: pairwise matrix has fewer than two candidates."))
    if reference_df is None:
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5C cannot be plotted: reference corpus is required for residue-category context."))

    # Save prepared final panel and matrix
    final_df.to_csv(dirs.source_data / "step9_final_panel_similarity_context_table.csv", index=False)
    pairwise.reset_index().rename(columns={"index": "candidate_id"}).to_csv(dirs.source_data / "step9_final_panel_similarity_matrix.csv", index=False)
    files["step9_final_panel_similarity_context_table"] = str(dirs.source_data / "step9_final_panel_similarity_context_table.csv")
    files["step9_final_panel_similarity_matrix"] = str(dirs.source_data / "step9_final_panel_similarity_matrix.csv")

    if any(c.status == "FAIL" for c in checks):
        write_reports(dirs, checks, discovery, files, data.get("_paths", pd.DataFrame()))
        msg = "Step 9 plotting readiness failed. Review step9_readiness_report.md."
        if is_notebook():
            raise RuntimeError(msg)
        print("ERROR:", msg, file=sys.stderr)
        return 2

    fig5_files = plot_figure5(
        final_df=final_df,
        pairwise=pairwise,
        reference_df=reference_df,
        source_data_dir=dirs.source_data,
        out_base=dirs.main_figure / "Figure_5_contextual_similarity_diversity",
        dpi=args.dpi,
    )
    for k, v in fig5_files.items():
        files[f"Figure_5_{k}"] = v
    checks.append(CheckResult("Figure_5", "PASS", "Generated redesigned Figure 5 with contextual NN similarity, NN summary, and residue-category context."))

    if not args.no_supplementary:
        if reference_df is not None and aa_df is not None and km_df is not None:
            s14_files = plot_supplementary_s14(
                final_df=final_df,
                reference_df=reference_df,
                pairwise=pairwise,
                aa_df=aa_df,
                kmer_df=km_df,
                source_data_dir=dirs.source_data,
                out_base=dirs.supplementary_figures / "Supplementary_Figure_S14_additional_context",
                dpi=args.dpi,
            )
            for k, v in s14_files.items():
                files[f"Supplementary_Figure_S14_{k}"] = v
            removed_report_path = dirs.source_data / "step9_removed_panel_report.csv"
            pairwise_source_path = dirs.source_data / "step9_pairwise_similarity_matrix.csv"
            if removed_report_path.exists():
                files["step9_removed_panel_report"] = str(removed_report_path)
            if pairwise_source_path.exists():
                files["step9_pairwise_similarity_matrix"] = str(pairwise_source_path)
            checks.append(CheckResult("Supplementary_Figure_S14", "PASS", "Generated two-panel Supplementary Figure S14 with amino-acid enrichment and 3-mer enrichment; pairwise similarity retained as source data/report."))
        else:
            checks.append(CheckResult("Supplementary_Figure_S14", "WARN", "S14 not generated because reference/composition data were unavailable."))

    # Additional source-data summary outputs
    sim_summary_rows = []
    for context, col in [("Reference context", "nearest_reference_similarity"), ("Candidate context", "nearest_paper_candidate_similarity")]:
        arr = pd.to_numeric(final_df[col], errors="coerce").dropna().to_numpy(dtype=float)
        sim_summary_rows.append({"context": context, **percentile_summary(arr)})
    sim_summary = pd.DataFrame(sim_summary_rows)
    sim_summary.to_csv(dirs.source_data / "step9_contextual_similarity_summary.csv", index=False)
    files["step9_contextual_similarity_summary"] = str(dirs.source_data / "step9_contextual_similarity_summary.csv")

    arr = pairwise.to_numpy(dtype=float)
    mask = ~np.eye(arr.shape[0], dtype=bool)
    offdiag = arr[mask]
    bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
    labels = ["0-0.05", "0.05-0.10", "0.10-0.15", "0.15-0.20", "0.20-0.30", "0.30-1.0"]
    cats = pd.cut(offdiag, bins=bins, labels=labels, include_lowest=True, right=False)
    div_summary = cats.value_counts().reindex(labels, fill_value=0).rename_axis("similarity_bin").reset_index(name="pair_count")
    div_summary["fraction"] = div_summary["pair_count"] / max(div_summary["pair_count"].sum(), 1)
    div_summary.to_csv(dirs.source_data / "step9_internal_diversity_summary.csv", index=False)
    files["step9_internal_diversity_summary"] = str(dirs.source_data / "step9_internal_diversity_summary.csv")

    try:
        comp_summary = residue_category_df(final_df, reference_df)
        comp_summary.to_csv(dirs.source_data / "step9_composition_context_summary.csv", index=False)
        files["step9_composition_context_summary"] = str(dirs.source_data / "step9_composition_context_summary.csv")
    except Exception:
        pass

    files = write_reports(dirs, checks, discovery, files, data.get("_paths", pd.DataFrame()))

    n_fail = sum(c.status == "FAIL" for c in checks)
    n_warn = sum(c.status == "WARN" for c in checks)
    status = "FAIL" if n_fail else ("WARN" if n_warn else "PASS")
    score = max(0, 100 - 15*n_fail - 2*n_warn)
    if not args.quiet:
        print("\nOncoPep Step 9 package generated.")
        print(f"Root: {dirs.root}")
        print(f"Readiness status: {status}; estimated score: {score}/100")
        print(f"Main figure: {files.get('Figure_5_png')}")
        if "Supplementary_Figure_S14_png" in files:
            print(f"Supplementary figure: {files.get('Supplementary_Figure_S14_png')}")
        print(f"Readiness report: {files.get('readiness_report')}")
    return 0


if __name__ == "__main__":
    try:
        rc = main()
        if is_notebook() and rc != 0:
            raise RuntimeError(f"Step 9 script failed with exit code {rc}.")
        if not is_notebook():
            raise SystemExit(rc)
    except Exception as exc:
        if is_notebook():
            raise RuntimeError(str(exc)) from exc
        print("\nERROR: OncoPep Step 9 figure generation failed.\n", file=sys.stderr)
        print(str(exc), file=sys.stderr)
        traceback.print_exc()
        raise SystemExit(2)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Step 9 — PLOS Computational Biology Figure 5 rescue script.

Scientific role
---------------
Step 9 focuses on contextual similarity support and internal diversity of the
final OncoPep candidate panel. It intentionally avoids the Step 8 selection
and prioritization-score panels.

Main figure
-----------
Figure 5. Contextual similarity and compositional support of the final
OncoPep panel.
  A. NN similarity context
  B. Similarity summary
  C. Residue-category context

Supplementary figure
--------------------
Supplementary Figure S14. Additional sequence-composition context.
  A. Amino-acid enrichment
  B. Top enriched 3-mers

Removed from plotted supplementary figure
-----------------------------------------
The pairwise final-panel similarity heatmap is retained in source data and
reports, but is no longer plotted as S14C to keep the supplement compact.

Design decisions
----------------
- Rescues the scientifically stronger old Step 9 similarity/composition logic.
- Removes old Step 9 selection-audit and score-shift panels because they now
  duplicate Step 8/Figure 4.
- Uses uppercase panel labels, OncoPep/PLOS palette, source-data exports,
  manifest, README, requirements, code snapshot, and readiness reporting.
- Implements the approved panel swap: residue-category context is used as
  main Figure 5C.
- Generates Supplementary Figure S14 as a two-panel composition figure by
  default whenever amino-acid and k-mer enrichment data can be loaded or computed.
- Centers Supplementary Figure S14 panel titles and assigns a unique restrained
  PLOS-compatible color to each top 3-mer bar.
- Retains pairwise final-panel similarity as source data/report rather than a
  plotted supplementary panel.
- Avoids default all-1.00 contextual-support fallback panels.

Ready-to-run examples
---------------------
python OncoPep_step9_PLOS_contextual_similarity_diversity_final.py \
  --step9-root /home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_09 \
  --project-root /home/data3/Moe/nature_computational_peponco \
  --legacy-step9-root /home/data3/Moe/nature_computational2/step9_v1

For explicit paths:
python OncoPep_step9_PLOS_contextual_similarity_diversity_final.py \
  --final-panel-file /path/to/table_s9_9_final_panel_with_all_step9_annotations.csv \
  --reference-file /path/to/step1_retained_dataset.csv \
  --pairwise-file /path/to/table_s9_7_final_panel_pairwise_similarity.csv \
  --aa-enrichment-file /path/to/table_s9_5_amino_acid_enrichment_vs_reference.csv \
  --kmer-enrichment-file /path/to/table_s9_6_kmer_enrichment_vs_reference.csv
"""

from __future__ import annotations

import argparse
import json
import math
import os
import shutil
import sys
import traceback
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

SCRIPT_VERSION = "v6_2026_07_08_step9_s14_center_titles_unique_kmer_colors"

DEFAULT_STEP9_ROOT = "/home/data3/Moe/nature_computational_peponco/PLOS/plos_comp/step_09"
DEFAULT_PROJECT_ROOT = "/home/data3/Moe/nature_computational_peponco"
DEFAULT_LEGACY_STEP9_ROOT = "/home/data3/Moe/nature_computational2/step9_v1"
DEFAULT_LEGACY_REFERENCE = "/home/data3/Moe/nature_computational2/step1/tables/step1_retained_dataset.csv"

SAFE_TABLE_EXTENSIONS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".pq"}
BLOCKED_EXTENSIONS = {".json", ".png", ".jpg", ".jpeg", ".pdf", ".tif", ".tiff", ".py", ".ipynb"}
BLOCKED_PATH_KEYWORDS = [
    "/run/user/",
    "/jupyter/runtime/",
    ".ipynb_checkpoints",
    "__pycache__",
    "/.git/",
    "kernel-",
]

PLOS = {
    "blue": "#1F95B8",
    "mint": "#A8D3B2",
    "coral": "#DD705D",
    "brown": "#B67D4E",
    "charcoal": "#6A5E61",
    "dark": "#333333",
    "grid": "#EAEAEA",
    "light": "#D9D9D9",
    "white": "#FFFFFF",
    "edge": "#B8B8B8",
    "pale": "#F5F7F7",
}

AA_SET = set("ACDEFGHIKLMNPQRSTVWY")
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
AA_HYDROPHOBIC = set("AILMFWVYC")
AA_AROMATIC = set("FYW")
AA_POSITIVE = set("KRH")
AA_NEGATIVE = set("DE")
AA_POLAR = set("STNQCYWHKRD")

KD_HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}


@dataclass
class CheckResult:
    name: str
    status: str  # PASS/WARN/FAIL
    message: str


@dataclass
class OutputDirs:
    root: Path
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


def is_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        shell = get_ipython()
        if shell is None:
            return False
        return shell.__class__.__name__ in {"ZMQInteractiveShell", "TerminalInteractiveShell"}
    except Exception:
        return False


def clean_argv(argv: Optional[Sequence[str]] = None) -> List[str]:
    if argv is None:
        argv = sys.argv[1:]
    out: List[str] = []
    skip = False
    for i, token in enumerate(argv):
        if skip:
            skip = False
            continue
        s = str(token)
        if s == "-f":
            if i + 1 < len(argv):
                skip = True
            continue
        if any(k in s for k in BLOCKED_PATH_KEYWORDS):
            continue
        if s.endswith(".json") and ("kernel-" in s or "/runtime/" in s):
            continue
        out.append(s)
    return out


def ensure_dirs(root: Path) -> OutputDirs:
    dirs = OutputDirs(
        root=root,
        main_figure=root / "main_figure",
        supplementary_figures=root / "supplementary_figures",
        source_data=root / "source_data",
        reports=root / "reports",
        code=root / "code",
    )
    for d in asdict(dirs).values():
        if isinstance(d, Path):
            d.mkdir(parents=True, exist_ok=True)
    return dirs


def is_blocked_path(path: Path) -> bool:
    p = str(path).replace("\\", "/").lower()
    return any(k.lower() in p for k in BLOCKED_PATH_KEYWORDS)


def is_allowed_table_file(path: Path) -> bool:
    suffix = path.suffix.lower()
    return suffix in SAFE_TABLE_EXTENSIONS and suffix not in BLOCKED_EXTENSIONS and not is_blocked_path(path)


def safe_roots(*roots: Optional[str]) -> List[Path]:
    seen = set()
    out: List[Path] = []
    for r in roots:
        if not r:
            continue
        p = Path(r).expanduser()
        s = str(p.resolve()) if p.exists() else str(p)
        if s in seen:
            continue
        seen.add(s)
        if p.exists():
            out.append(p)
    return out


def discover_files(roots: List[Path]) -> List[Path]:
    found: List[Path] = []
    for root in roots:
        try:
            for p in root.rglob("*"):
                try:
                    if p.is_file() and is_allowed_table_file(p):
                        found.append(p)
                except Exception:
                    continue
        except Exception:
            continue
    return found


def score_path_for_role(path: Path, role: str) -> Tuple[int, List[str]]:
    s = str(path).lower().replace("\\", "/")
    name = path.name.lower()
    score = 0
    reasons: List[str] = []

    if "step9_v1/tables_supplementary" in s:
        score += 12; reasons.append("legacy_step9_supp")
    if "step_09" in s or "step9" in s:
        score += 6; reasons.append("step9_hint")
    if "step1" in s and role == "reference":
        score += 8; reasons.append("step1_reference_hint")

    patterns = {
        "final": ["table_s9_9_final_panel", "final_panel_with_all_step9_annotations", "final_panel", "final_12", "candidate"],
        "pairwise": ["table_s9_7_final_panel_pairwise_similarity", "pairwise_similarity", "similarity_matrix", "jaccard"],
        "reference": ["step1_retained_dataset", "retained_dataset", "reference", "training", "corpus"],
        "aa": ["table_s9_5_amino_acid", "amino_acid_enrichment", "aa_enrichment"],
        "kmer": ["table_s9_6_kmer", "kmer_enrichment", "3mer", "3_mer"],
        "paper": ["table_s9_11_paper_candidates", "paper_candidates", "harmonized"],
    }
    for patt in patterns.get(role, []):
        if patt in name:
            score += 10; reasons.append(f"name:{patt}")
    negative = ["readiness", "manifest", "requirements", "selection_audit", "stage_score", "score_shift", "figure", "source_data_all"]
    for n in negative:
        if n in name:
            score -= 8; reasons.append(f"penalty:{n}")
    return score, reasons


def resolve_input(role: str, explicit: Optional[str], roots: List[Path], log: List[Dict[str, str]]) -> Optional[Path]:
    if explicit:
        p = Path(explicit).expanduser()
        log.append({"role": role, "action": "explicit", "path": str(p), "reason": "user_supplied"})
        return p
    candidates = discover_files(roots)
    scored: List[Tuple[int, Path, List[str]]] = []
    for p in candidates:
        score, reasons = score_path_for_role(p, role)
        if score > 0:
            scored.append((score, p, reasons))
    if not scored:
        log.append({"role": role, "action": "not_found", "path": "", "reason": "no_safe_candidate"})
        return None
    scored.sort(key=lambda x: (-x[0], str(x[1])))
    score, p, reasons = scored[0]
    log.append({"role": role, "action": "auto", "path": str(p), "reason": f"score={score};" + ";".join(reasons)})
    return p


def read_table(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suf == ".txt":
        for sep in [",", "\t", None]:
            try:
                return pd.read_csv(path, sep=sep) if sep else pd.read_csv(path, delim_whitespace=True)
            except Exception:
                pass
        raise ValueError(f"Could not parse text table: {path}")
    if suf in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suf in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table extension: {path}")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def first_col(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    lower = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    return None


def standardize_final_df(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_columns(df)
    out = df.copy()
    seq_col = first_col(out, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        raise RuntimeError("Final-panel table needs a sequence column.")
    out["sequence"] = out[seq_col].astype(str).map(clean_sequence)

    id_col = first_col(out, ["generated_id", "candidate_id", "peptide_id", "id", "seq_id", "name"])
    if id_col is None:
        out["generated_id"] = [f"C{i:02d}" for i in range(1, len(out) + 1)]
    else:
        out["generated_id"] = out[id_col].astype(str)

    alias_map = {
        "nearest_reference_similarity": [
            "nearest_reference_similarity", "nn_reference_similarity", "nearest_ref_similarity",
            "max_reference_similarity", "reference_nn_similarity", "nn_similarity_reference"
        ],
        "nearest_paper_candidate_similarity": [
            "nearest_paper_candidate_similarity", "nearest_candidate_similarity", "nearest_final_similarity",
            "paper_candidate_similarity", "candidate_context_similarity", "nn_paper_candidate_similarity"
        ],
    }
    for std, aliases in alias_map.items():
        if std not in out.columns:
            col = first_col(out, aliases)
            if col is not None:
                out[std] = pd.to_numeric(out[col], errors="coerce")
    return out


def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    s = str(seq).strip().upper()
    return "".join(ch for ch in s if ch in AA_SET)


def kmers(seq: str, k: int = 3) -> set:
    seq = clean_sequence(seq)
    if len(seq) < k:
        return {seq} if seq else set()
    return {seq[i:i+k] for i in range(len(seq) - k + 1)}


def jaccard_sets(a: set, b: set) -> float:
    if not a and not b:
        return np.nan
    u = a | b
    if not u:
        return np.nan
    return len(a & b) / len(u)


def compute_pairwise_matrix(ids: Sequence[str], seqs: Sequence[str], k: int = 3) -> pd.DataFrame:
    ids = [str(x) for x in ids]
    ksets = [kmers(s, k=k) for s in seqs]
    n = len(ids)
    mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            mat[i, j] = 1.0 if i == j else jaccard_sets(ksets[i], ksets[j])
    return pd.DataFrame(mat, index=ids, columns=ids)


def read_pairwise_matrix(path: Path) -> pd.DataFrame:
    df = read_table(path)
    df = normalize_columns(df)
    # Matrix format: first column is index/candidate IDs and remaining are numeric columns
    first = df.columns[0]
    if not pd.api.types.is_numeric_dtype(df[first]):
        trial = df.set_index(first)
        numeric = trial.apply(pd.to_numeric, errors="coerce")
        # Accept if at least square-ish numeric matrix
        if numeric.shape[0] == numeric.shape[1] and numeric.notna().sum().sum() > 0:
            return numeric
    # Long format fallback
    cols = {c.lower(): c for c in df.columns}
    id1 = first_col(df, ["candidate_1", "id1", "source", "row", "candidate_i"])
    id2 = first_col(df, ["candidate_2", "id2", "target", "column", "candidate_j"])
    val = first_col(df, ["similarity", "jaccard", "jaccard_similarity", "3mer_jaccard", "value"])
    if id1 and id2 and val:
        tmp = df[[id1, id2, val]].copy()
        tmp[val] = pd.to_numeric(tmp[val], errors="coerce")
        mat = tmp.pivot_table(index=id1, columns=id2, values=val, aggfunc="first")
        # symmetrize where possible
        labels = sorted(set(mat.index.astype(str)) | set(mat.columns.astype(str)))
        mat = mat.reindex(index=labels, columns=labels)
        for a in labels:
            for b in labels:
                if pd.isna(mat.loc[a, b]) and not pd.isna(mat.loc[b, a]):
                    mat.loc[a, b] = mat.loc[b, a]
        np.fill_diagonal(mat.values, 1.0)
        return mat.apply(pd.to_numeric, errors="coerce")
    raise RuntimeError(f"Could not parse pairwise similarity matrix: {path}")


def max_similarity_to_reference(final_df: pd.DataFrame, ref_df: pd.DataFrame, k: int = 3, max_ref: int = 100000) -> pd.Series:
    ref = ref_df.copy()
    seq_col = first_col(ref, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        raise RuntimeError("Reference file needs a sequence column for fallback nearest-reference similarity.")
    ref_seqs = [clean_sequence(x) for x in ref[seq_col].astype(str).tolist()]
    ref_seqs = [s for s in ref_seqs if s]
    if len(ref_seqs) > max_ref:
        ref_seqs = ref_seqs[:max_ref]
    ref_sets = [kmers(s, k=k) for s in ref_seqs]
    vals = []
    for seq in final_df["sequence"].astype(str).tolist():
        sset = kmers(seq, k=k)
        if not sset or not ref_sets:
            vals.append(np.nan)
            continue
        mx = max(jaccard_sets(sset, rset) for rset in ref_sets)
        vals.append(float(mx))
    return pd.Series(vals, index=final_df.index)


def nearest_internal_similarity(pairwise: pd.DataFrame) -> pd.Series:
    mat = pairwise.copy().apply(pd.to_numeric, errors="coerce")
    vals = []
    for i, idx in enumerate(mat.index):
        arr = mat.iloc[i].to_numpy(dtype=float)
        if i < len(arr):
            arr[i] = np.nan
        vals.append(float(np.nanmax(arr)) if np.isfinite(arr).any() else np.nan)
    return pd.Series(vals, index=mat.index)


def percentile_summary(vals: np.ndarray) -> Dict[str, float]:
    vals = np.asarray([v for v in vals if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return {"n": 0, "median": np.nan, "p90": np.nan, "max": np.nan, "q1": np.nan, "q3": np.nan}
    return {
        "n": int(len(vals)),
        "median": float(np.nanmedian(vals)),
        "p90": float(np.nanpercentile(vals, 90)),
        "max": float(np.nanmax(vals)),
        "q1": float(np.nanpercentile(vals, 25)),
        "q3": float(np.nanpercentile(vals, 75)),
    }


def aa_enrichment(final_seqs: Sequence[str], ref_seqs: Sequence[str], pseudocount: float = 0.5) -> pd.DataFrame:
    fcnt = Counter(ch for s in final_seqs for ch in clean_sequence(s) if ch in AA_SET)
    rcnt = Counter(ch for s in ref_seqs for ch in clean_sequence(s) if ch in AA_SET)
    f_total = sum(fcnt.values())
    r_total = sum(rcnt.values())
    rows = []
    for aa in AA_ORDER:
        f = fcnt.get(aa, 0)
        r = rcnt.get(aa, 0)
        f_freq = (f + pseudocount) / (f_total + pseudocount * len(AA_ORDER)) if f_total else np.nan
        r_freq = (r + pseudocount) / (r_total + pseudocount * len(AA_ORDER)) if r_total else np.nan
        log2e = float(np.log2(f_freq / r_freq)) if pd.notna(f_freq) and pd.notna(r_freq) and r_freq > 0 else np.nan
        rows.append({"amino_acid": aa, "final_count": f, "reference_count": r, "final_frequency": f_freq, "reference_frequency": r_freq, "log2_enrichment_vs_reference": log2e})
    return pd.DataFrame(rows)


def kmer_enrichment(final_seqs: Sequence[str], ref_seqs: Sequence[str], k: int = 3, pseudocount: float = 0.5, top_n: int = 25) -> pd.DataFrame:
    fcnt = Counter(km for s in final_seqs for km in kmers(s, k=k))
    rcnt = Counter(km for s in ref_seqs for km in kmers(s, k=k))
    universe = sorted(set(fcnt) | set(rcnt))
    f_total = sum(fcnt.values())
    r_total = sum(rcnt.values())
    rows = []
    denom = max(len(universe), 1)
    for km in universe:
        f = fcnt.get(km, 0)
        r = rcnt.get(km, 0)
        f_freq = (f + pseudocount) / (f_total + pseudocount * denom) if f_total else np.nan
        r_freq = (r + pseudocount) / (r_total + pseudocount * denom) if r_total else np.nan
        log2e = float(np.log2(f_freq / r_freq)) if pd.notna(f_freq) and pd.notna(r_freq) and r_freq > 0 else np.nan
        rows.append({"kmer": km, "final_count": f, "reference_count": r, "final_frequency": f_freq, "reference_frequency": r_freq, "log2_enrichment_vs_reference": log2e})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    out = out.sort_values(["log2_enrichment_vs_reference", "final_count"], ascending=[False, False]).head(top_n)
    return out


def frac_of_categories(seqs: Sequence[str]) -> Dict[str, float]:
    seqs = [clean_sequence(s) for s in seqs]
    total = sum(len(s) for s in seqs)
    if total == 0:
        return {"Hydrophobic": np.nan, "Aromatic": np.nan, "Positive": np.nan, "Negative": np.nan, "Polar": np.nan}
    def frac(res_set: set) -> float:
        return sum(sum(ch in res_set for ch in s) for s in seqs) / total
    return {
        "Hydrophobic": frac(AA_HYDROPHOBIC),
        "Aromatic": frac(AA_AROMATIC),
        "Positive": frac(AA_POSITIVE),
        "Negative": frac(AA_NEGATIVE),
        "Polar": frac(AA_POLAR),
    }


def get_ref_sequences(ref_df: pd.DataFrame) -> List[str]:
    seq_col = first_col(ref_df, ["sequence", "peptide", "seq", "aa_sequence", "generated_sequence"])
    if seq_col is None:
        return []
    return [clean_sequence(x) for x in ref_df[seq_col].astype(str).tolist() if clean_sequence(x)]


def set_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": PLOS["white"],
        "axes.facecolor": PLOS["white"],
        "savefig.facecolor": PLOS["white"],
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 15,
        "axes.labelsize": 11,
        "xtick.labelsize": 9.5,
        "ytick.labelsize": 9.5,
        "legend.fontsize": 9.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": PLOS["grid"],
        "grid.linewidth": 0.8,
        "grid.alpha": 0.85,
        "axes.axisbelow": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["left"].set_color(PLOS["edge"])
    ax.spines["bottom"].set_color(PLOS["edge"])
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(colors=PLOS["dark"], width=0.8, length=4)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis, color=PLOS["grid"], linewidth=0.8, zorder=0)
    if grid_axis == "y":
        ax.grid(False, axis="x")
    elif grid_axis == "x":
        ax.grid(False, axis="y")
    elif grid_axis == "both":
        ax.grid(True, axis="both", color=PLOS["grid"], linewidth=0.8, zorder=0)


def add_panel_label(ax, label: str, x: float = -0.14, y: float = 1.08, fontsize: int = 18) -> None:
    ax.text(x, y, label, transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="left", va="top", color=PLOS["dark"])


def draw_violin(ax, data: Sequence[float], pos: float, color: str, width: float = 0.72) -> None:
    arr = np.asarray([x for x in data if pd.notna(x)], dtype=float)
    if len(arr) == 0:
        return
    parts = ax.violinplot([arr], positions=[pos], widths=width, showmeans=False, showmedians=False, showextrema=False)
    for body in parts["bodies"]:
        body.set_facecolor(color)
        body.set_edgecolor("none")
        body.set_alpha(0.78)
    q1, med, q3 = np.nanpercentile(arr, [25, 50, 75])
    ax.plot([pos, pos], [q1, q3], color=PLOS["dark"], linewidth=1.2, zorder=4)
    ax.scatter([pos], [med], s=34, facecolor=PLOS["white"], edgecolor=PLOS["dark"], linewidth=0.8, zorder=5)
    # Subtle points
    rng = np.random.default_rng(42 + int(pos * 10))
    jitter = rng.uniform(-0.065, 0.065, size=len(arr))
    ax.scatter(np.full(len(arr), pos) + jitter, arr, s=10, facecolor=PLOS["white"], edgecolor=PLOS["dark"], linewidth=0.35, alpha=0.65, zorder=4)


def save_multi(fig: plt.Figure, out_base: Path, dpi: int) -> Dict[str, str]:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths: Dict[str, str] = {}
    for ext in [".png", ".pdf", ".tiff"]:
        p = out_base.with_suffix(ext)
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        paths[ext.lstrip(".")] = str(p)
    plt.close(fig)
    return paths


def short_label(x: str, max_len: int = 12) -> str:
    s = str(x)
    return s if len(s) <= max_len else s[:max_len]


def sorted_pairwise_matrix(pairwise: pd.DataFrame, final_ids: Sequence[str]) -> pd.DataFrame:
    mat = pairwise.copy()
    mat.index = mat.index.astype(str)
    mat.columns = mat.columns.astype(str)
    ids = [str(i) for i in final_ids]
    available = [i for i in ids if i in mat.index and i in mat.columns]
    if len(available) >= 2:
        mat = mat.loc[available, available]
    else:
        # keep original square matrix
        pass
    numeric = mat.apply(pd.to_numeric, errors="coerce")
    # order by mean off-diagonal similarity to make layout intentional
    vals = numeric.to_numpy(dtype=float)
    np.fill_diagonal(vals, np.nan)
    avg = np.nanmean(vals, axis=1)
    order = np.argsort(-np.nan_to_num(avg, nan=-1))
    numeric = numeric.iloc[order, :].iloc[:, order]
    return numeric


def residue_category_df(final_df: pd.DataFrame, reference_df: pd.DataFrame) -> pd.DataFrame:
    """Return residue-category fractions for reference corpus and final panel."""
    final_seqs = final_df["sequence"].astype(str).tolist()
    ref_seqs = get_ref_sequences(reference_df)
    ref_cat = frac_of_categories(ref_seqs)
    final_cat = frac_of_categories(final_seqs)
    categories = ["Hydrophobic", "Aromatic", "Positive", "Negative", "Polar"]
    return pd.DataFrame({
        "category": categories,
        "reference_fraction": [ref_cat[c] for c in categories],
        "final_panel_fraction": [final_cat[c] for c in categories],
    })


def write_pairwise_source_and_heatmap(
    ax,
    pairwise: pd.DataFrame,
    final_ids: Sequence[str],
    title: str,
    colorbar_label: str = "3-mer Jaccard similarity",
    panel_label: Optional[str] = None,
) -> pd.DataFrame:
    """Draw polished supplementary heatmap and return ordered matrix."""
    mat = sorted_pairwise_matrix(pairwise, final_ids)
    arr = mat.to_numpy(dtype=float)
    if arr.shape[0] == 0:
        raise RuntimeError("Pairwise matrix has no rows after ordering.")
    diag_mask = np.eye(arr.shape[0], dtype=bool)
    offdiag = arr.copy()
    offdiag[diag_mask] = np.nan
    max_offdiag = float(np.nanmax(offdiag)) if np.isfinite(offdiag).any() else 0.20
    vmax = min(1.0, max(0.20, math.ceil(max_offdiag * 10) / 10))
    cmap = LinearSegmentedColormap.from_list(
        "oncopep_similarity",
        [PLOS["white"], PLOS["mint"], PLOS["blue"]],
    )
    plot_arr = arr.copy()
    plot_arr[diag_mask] = np.nan
    masked = np.ma.masked_invalid(plot_arr)
    cmap.set_bad(color=PLOS["pale"])
    im = ax.imshow(masked, vmin=0, vmax=vmax, cmap=cmap, aspect="equal")
    labels = [short_label(x, 12) for x in mat.index.tolist()]
    ax.set_xticks(np.arange(len(labels)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(labels)), labels, fontsize=8)
    ax.set_title(title, loc="left", pad=8)
    if panel_label:
        add_panel_label(ax, panel_label, x=-0.08, y=1.06, fontsize=16)
    ax.set_xticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(labels), 1), minor=True)
    ax.grid(which="minor", color=PLOS["white"], linestyle="-", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.042, pad=0.035)
    cbar.set_label(colorbar_label)
    return mat


def plot_figure5(
    final_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    reference_df: pd.DataFrame,
    source_data_dir: Path,
    out_base: Path,
    dpi: int,
) -> Dict[str, str]:
    """Generate final Figure 5 after approved panel swap.

    Final structure:
      A. NN similarity context
      B. Similarity summary
      C. Residue-category context
    """
    set_style()

    ref_vals = pd.to_numeric(final_df["nearest_reference_similarity"], errors="coerce").dropna().to_numpy(dtype=float)
    cand_col = "nearest_paper_candidate_similarity"
    cand_vals = pd.to_numeric(final_df[cand_col], errors="coerce").dropna().to_numpy(dtype=float)

    summary_rows = []
    for name, arr in [("Reference context", ref_vals), ("Final-panel context", cand_vals)]:
        st = percentile_summary(arr)
        summary_rows.append({"context": name, **st})
    summary_df = pd.DataFrame(summary_rows)

    panel_a_df = pd.concat([
        pd.DataFrame({"context": "Reference context", "nearest_neighbor_similarity": ref_vals}),
        pd.DataFrame({"context": "Final-panel context", "nearest_neighbor_similarity": cand_vals}),
    ], ignore_index=True)
    panel_a_df.to_csv(source_data_dir / "Figure_5_panel_a_source_data.csv", index=False)

    panel_b_rows = []
    for _, r in summary_df.iterrows():
        for metric in ["median", "p90", "max"]:
            panel_b_rows.append({"context": r["context"], "summary_metric": metric, "similarity": r[metric], "n": r["n"]})
    panel_b_df = pd.DataFrame(panel_b_rows)
    panel_b_df.to_csv(source_data_dir / "Figure_5_panel_b_source_data.csv", index=False)

    cat_df = residue_category_df(final_df, reference_df)
    cat_df.to_csv(source_data_dir / "Figure_5_panel_c_source_data.csv", index=False)

    combined = pd.concat([
        panel_a_df.assign(panel="Figure_5A"),
        panel_b_df.assign(panel="Figure_5B"),
        cat_df.assign(panel="Figure_5C"),
    ], ignore_index=True, sort=False)
    combined.to_csv(source_data_dir / "Figure_5_source_data_all_panels.csv", index=False)

    fig = plt.figure(figsize=(15.8, 5.15))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[0.88, 0.92, 1.10], wspace=0.50)

    # A violin
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.18, y=1.08)
    style_axis(ax, grid_axis="y")
    draw_violin(ax, ref_vals, 0, PLOS["blue"], width=0.78)
    draw_violin(ax, cand_vals, 1, PLOS["coral"], width=0.78)
    ymax = max(0.16, float(np.nanmax(np.r_[ref_vals, cand_vals])) * 1.22 if len(np.r_[ref_vals, cand_vals]) else 0.16)
    ax.set_ylim(0, ymax)
    ax.set_xticks([0, 1], ["Reference\ncontext", "Final-panel\ncontext"])
    ax.set_ylabel("Nearest-neighbor similarity")
    ax.set_title("NN similarity context", pad=10)

    # B summary bars
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.18, y=1.08)
    style_axis(ax, grid_axis="y")
    metrics = ["median", "p90", "max"]
    metric_labels = ["Median", "P90", "Max"]
    x = np.arange(len(metrics), dtype=float)
    width = 0.32
    ref_s = summary_df.set_index("context").loc["Reference context"]
    cand_s = summary_df.set_index("context").loc["Final-panel context"]
    ref_bar = [float(ref_s[m]) for m in metrics]
    cand_bar = [float(cand_s[m]) for m in metrics]
    ax.bar(x - width/2, ref_bar, width=width, color=PLOS["blue"], edgecolor=PLOS["dark"], linewidth=0.5, label="Reference context", zorder=3)
    ax.bar(x + width/2, cand_bar, width=width, color=PLOS["coral"], edgecolor=PLOS["dark"], linewidth=0.5, label="Final-panel context", zorder=3)
    ax.set_xticks(x, metric_labels)
    ax.set_ylabel("Nearest-neighbor similarity")
    ymax_b = max(0.16, float(np.nanmax(np.r_[ref_bar, cand_bar])) * 1.22)
    ax.set_ylim(0, ymax_b)
    ax.set_title("Similarity summary", pad=10)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=True, handlelength=1.0, columnspacing=0.85, handletextpad=0.45, borderaxespad=0.3)
    leg.get_frame().set_edgecolor(PLOS["edge"])
    leg.get_frame().set_linewidth(0.8)

    # C residue-category context
    ax = fig.add_subplot(gs[0, 2])
    add_panel_label(ax, "C", x=-0.14, y=1.08)
    style_axis(ax, grid_axis="y")
    categories = cat_df["category"].astype(str).tolist()
    x = np.arange(len(categories), dtype=float)
    width = 0.32
    ref_cat_vals = cat_df["reference_fraction"].to_numpy(dtype=float)
    final_cat_vals = cat_df["final_panel_fraction"].to_numpy(dtype=float)
    ax.bar(x - width/2, ref_cat_vals, width=width, color=PLOS["light"], edgecolor=PLOS["dark"], linewidth=0.4, label="Reference ACP corpus", zorder=3)
    ax.bar(x + width/2, final_cat_vals, width=width, color=PLOS["blue"], edgecolor=PLOS["dark"], linewidth=0.4, label="Final panel", zorder=3)
    ax.set_xticks(x, categories, rotation=18, ha="right")
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, max(0.34, float(np.nanmax(np.r_[ref_cat_vals, final_cat_vals])) * 1.18))
    ax.set_title("Residue-category context", pad=10)
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=True, handlelength=1.0, columnspacing=0.8)
    leg.get_frame().set_edgecolor(PLOS["edge"])
    leg.get_frame().set_linewidth(0.8)

    fig.subplots_adjust(left=0.055, right=0.985, bottom=0.22, top=0.90)
    return save_multi(fig, out_base, dpi=dpi)


def plot_supplementary_s14(
    final_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    pairwise: pd.DataFrame,
    aa_df: pd.DataFrame,
    kmer_df: pd.DataFrame,
    source_data_dir: Path,
    out_base: Path,
    dpi: int,
) -> Dict[str, str]:
    """Generate polished two-panel Supplementary Figure S14.

    Final plotted structure:
      A. Amino-acid enrichment
      B. Top enriched 3-mers

    Pairwise final-panel similarity is not plotted here. It is retained as
    source data and documented in step9_removed_panel_report.csv to avoid a
    crowded supplementary figure.
    """
    set_style()

    final_seqs = final_df["sequence"].astype(str).tolist()
    ref_seqs = get_ref_sequences(reference_df)

    # Normalize AA enrichment table or compute fallback
    aa = normalize_columns(aa_df.copy()) if aa_df is not None else pd.DataFrame()
    if "log2_enrichment_vs_reference" not in aa.columns:
        col = first_col(aa, ["log2_enrichment", "enrichment", "log2fc", "log2_fold_change"])
        if col:
            aa["log2_enrichment_vs_reference"] = pd.to_numeric(aa[col], errors="coerce")
    if "amino_acid" not in aa.columns:
        col = first_col(aa, ["aa", "residue", "amino acid"])
        if col:
            aa["amino_acid"] = aa[col].astype(str)
    if "amino_acid" not in aa.columns or "log2_enrichment_vs_reference" not in aa.columns:
        aa = aa_enrichment(final_seqs, ref_seqs)
    aa["amino_acid"] = aa["amino_acid"].astype(str)
    aa["log2_enrichment_vs_reference"] = pd.to_numeric(aa["log2_enrichment_vs_reference"], errors="coerce")

    # Normalize k-mer enrichment table or compute fallback
    km = normalize_columns(kmer_df.copy()) if kmer_df is not None else pd.DataFrame()
    if "log2_enrichment_vs_reference" not in km.columns:
        col = first_col(km, ["log2_enrichment", "enrichment", "log2fc", "log2_fold_change"])
        if col:
            km["log2_enrichment_vs_reference"] = pd.to_numeric(km[col], errors="coerce")
    if "kmer" not in km.columns:
        col = first_col(km, ["3mer", "k_mer", "motif", "token"])
        if col:
            km["kmer"] = km[col].astype(str)
    if "kmer" not in km.columns or "log2_enrichment_vs_reference" not in km.columns:
        km = kmer_enrichment(final_seqs, ref_seqs, k=3, top_n=25)
    km["kmer"] = km["kmer"].astype(str)
    km["log2_enrichment_vs_reference"] = pd.to_numeric(km["log2_enrichment_vs_reference"], errors="coerce")

    # Retain pairwise matrix as source data/report, but do not plot it.
    final_ids = final_df["generated_id"].astype(str).tolist()
    mat = sorted_pairwise_matrix(pairwise, final_ids)

    aa_out = source_data_dir / "Supplementary_Figure_S14_panel_a_source_data.csv"
    km_out = source_data_dir / "Supplementary_Figure_S14_panel_b_source_data.csv"
    matrix_out = source_data_dir / "step9_pairwise_similarity_matrix.csv"
    removed_out = source_data_dir / "step9_removed_panel_report.csv"

    aa.to_csv(aa_out, index=False)
    km.to_csv(km_out, index=False)
    mat.reset_index().rename(columns={"index": "candidate_id"}).to_csv(matrix_out, index=False)

    removed_report = pd.DataFrame([
        {
            "removed_panel": "Supplementary_Figure_S14C_pairwise_similarity_heatmap",
            "decision": "removed_from_plotted_supplementary_figure",
            "reason": "Dense candidate-level heatmap was visually weaker than the two-panel composition figure and is retained as source data/report instead.",
            "retained_source_data": str(matrix_out),
            "manuscript_role": "source_data_internal_diversity_support",
        }
    ])
    removed_report.to_csv(removed_out, index=False)

    pd.concat([
        aa.assign(panel="S14A"),
        km.assign(panel="S14B"),
    ], ignore_index=True, sort=False).to_csv(source_data_dir / "Supplementary_Figure_S14_source_data_all_panels.csv", index=False)

    # Two-panel layout: large AA enrichment + k-mer enrichment
    fig = plt.figure(figsize=(12.6, 5.65))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[1.08, 1.0], wspace=0.36)

    # A amino-acid enrichment
    ax = fig.add_subplot(gs[0, 0])
    add_panel_label(ax, "A", x=-0.07, y=1.04, fontsize=17)
    style_axis(ax, grid_axis="x")
    tmp = aa[["amino_acid", "log2_enrichment_vs_reference"]].dropna().copy()
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    colors = [PLOS["light"] if v < 0 else PLOS["brown"] for v in vals]
    ax.barh(tmp["amino_acid"].astype(str), vals, color=colors, edgecolor=PLOS["dark"], linewidth=0.35, zorder=3)
    ax.axvline(0, color=PLOS["dark"], linewidth=0.9, zorder=4)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Amino-acid enrichment", loc="center", pad=10)

    # B top enriched 3-mers with unique restrained palette
    ax = fig.add_subplot(gs[0, 1])
    add_panel_label(ax, "B", x=-0.08, y=1.04, fontsize=17)
    style_axis(ax, grid_axis="x")
    tmp = km[["kmer", "log2_enrichment_vs_reference"]].dropna().copy()
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=False).head(8)
    tmp = tmp.sort_values("log2_enrichment_vs_reference", ascending=True)
    vals = tmp["log2_enrichment_vs_reference"].to_numpy(dtype=float)
    # Each plotted 3-mer receives a unique, restrained color.  The palette is
    # deliberately limited to OncoPep/PLOS-compatible tones and muted variants,
    # avoiding decorative rainbow coloring while preventing repeated bars.
    unique_kmer_palette = [
        PLOS["mint"],      # muted green
        "#7FB8C8",         # soft blue-green
        PLOS["blue"],      # OncoPep blue
        "#4F8FA3",         # deeper blue
        PLOS["brown"],     # warm brown
        "#C99A66",         # light brown
        PLOS["coral"],     # coral
        PLOS["charcoal"],  # muted charcoal
        "#9FB7A8",         # desaturated mint-gray fallback
        "#A87E73",         # muted clay fallback
        "#6FA7B3",         # extra teal fallback
        "#8B7C80",         # extra charcoal fallback
    ]
    colors = unique_kmer_palette[:len(vals)]
    ax.barh(tmp["kmer"].astype(str), vals, color=colors, edgecolor=PLOS["dark"], linewidth=0.35, zorder=3)
    ax.set_xlabel("log2 enrichment vs reference")
    ax.set_title("Top enriched 3-mers", loc="center", pad=10)

    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.15, top=0.90)
    return save_multi(fig, out_base, dpi=dpi)

def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Generate OncoPep Step 9 Figure 5 and Supplementary Figure S14.", allow_abbrev=False)
    p.add_argument("--step9-root", default=DEFAULT_STEP9_ROOT)
    p.add_argument("--project-root", default=DEFAULT_PROJECT_ROOT)
    p.add_argument("--legacy-step9-root", default=DEFAULT_LEGACY_STEP9_ROOT)
    p.add_argument("--legacy-reference-file", default=DEFAULT_LEGACY_REFERENCE)
    p.add_argument("--final-panel-file", default=None)
    p.add_argument("--reference-file", default=None)
    p.add_argument("--pairwise-file", default=None)
    p.add_argument("--aa-enrichment-file", default=None)
    p.add_argument("--kmer-enrichment-file", default=None)
    p.add_argument("--paper-candidates-file", default=None)
    p.add_argument("--similarity-k", type=int, default=3)
    p.add_argument("--dpi", type=int, default=600)
    p.add_argument("--no-supplementary", action="store_true")
    p.add_argument("--max-reference-fallback", type=int, default=100000)
    p.add_argument("--quiet", action="store_true")
    return p


def load_inputs(args, checks: List[CheckResult], log: List[Dict[str, str]]) -> Dict[str, pd.DataFrame]:
    roots = safe_roots(args.project_root, args.legacy_step9_root, Path(args.project_root).parent if args.project_root else None)
    data: Dict[str, pd.DataFrame] = {}
    paths = {
        "final": resolve_input("final", args.final_panel_file, roots, log),
        "pairwise": resolve_input("pairwise", args.pairwise_file, roots, log),
        "reference": Path(args.reference_file) if args.reference_file else (Path(args.legacy_reference_file) if args.legacy_reference_file and Path(args.legacy_reference_file).exists() else resolve_input("reference", None, roots, log)),
        "aa": resolve_input("aa", args.aa_enrichment_file, roots, log),
        "kmer": resolve_input("kmer", args.kmer_enrichment_file, roots, log),
        "paper": resolve_input("paper", args.paper_candidates_file, roots, log),
    }
    data["_paths"] = pd.DataFrame([{"role": k, "path": str(v) if v else ""} for k, v in paths.items()])

    # Required final file
    if paths["final"] is None or not paths["final"].exists():
        checks.append(CheckResult("final_panel_file", "FAIL", "Final-panel annotation table missing. Supply --final-panel-file."))
        return data
    try:
        data["final"] = standardize_final_df(read_table(paths["final"]))
        checks.append(CheckResult("final_panel_file", "PASS", f"Loaded final panel: {paths['final']} ({len(data['final'])} rows)."))
    except Exception as e:
        checks.append(CheckResult("final_panel_file", "FAIL", f"Could not read final panel {paths['final']}: {e}"))
        return data

    # Reference is required for context fallback and S14
    if paths["reference"] is not None and paths["reference"].exists():
        try:
            data["reference"] = read_table(paths["reference"])
            checks.append(CheckResult("reference_file", "PASS", f"Loaded reference corpus: {paths['reference']} ({len(data['reference'])} rows)."))
        except Exception as e:
            checks.append(CheckResult("reference_file", "WARN", f"Could not read reference file {paths['reference']}: {e}"))
    else:
        checks.append(CheckResult("reference_file", "WARN", "Reference corpus file not found; nearest-reference fallback and S14 may be unavailable."))

    if paths["pairwise"] is not None and paths["pairwise"].exists():
        try:
            data["pairwise"] = read_pairwise_matrix(paths["pairwise"])
            checks.append(CheckResult("pairwise_file", "PASS", f"Loaded pairwise similarity matrix: {paths['pairwise']}."))
        except Exception as e:
            checks.append(CheckResult("pairwise_file", "WARN", f"Could not parse pairwise file; will compute from sequences if possible: {e}"))
    else:
        checks.append(CheckResult("pairwise_file", "WARN", "Pairwise similarity matrix not found; computing 3-mer Jaccard matrix from final-panel sequences."))

    for role, key in [("aa", "aa"), ("kmer", "kmer")]:
        if paths[role] is not None and paths[role].exists():
            try:
                data[key] = read_table(paths[role])
                checks.append(CheckResult(f"{role}_enrichment_file", "PASS", f"Loaded {role} enrichment table: {paths[role]}."))
            except Exception as e:
                checks.append(CheckResult(f"{role}_enrichment_file", "WARN", f"Could not read {role} enrichment table; will compute if possible: {e}"))
        else:
            checks.append(CheckResult(f"{role}_enrichment_file", "WARN", f"{role} enrichment table not found; will compute from final/reference sequences if possible."))

    return data


def prepare_data(data: Dict[str, pd.DataFrame], args, checks: List[CheckResult]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    final_df = data["final"].copy()
    reference_df = data.get("reference")

    if "pairwise" in data:
        pairwise = data["pairwise"].copy()
    else:
        pairwise = compute_pairwise_matrix(final_df["generated_id"].tolist(), final_df["sequence"].tolist(), k=args.similarity_k)
        checks.append(CheckResult("pairwise_computed", "WARN", f"Computed pairwise {args.similarity_k}-mer Jaccard matrix from final-panel sequences."))

    # Fill nearest reference if missing
    if "nearest_reference_similarity" not in final_df.columns or pd.to_numeric(final_df.get("nearest_reference_similarity", pd.Series(dtype=float)), errors="coerce").notna().sum() == 0:
        if reference_df is not None:
            final_df["nearest_reference_similarity"] = max_similarity_to_reference(final_df, reference_df, k=args.similarity_k, max_ref=args.max_reference_fallback)
            checks.append(CheckResult("nearest_reference_similarity", "WARN", "Computed nearest-reference similarity from sequences because input column was unavailable."))
        else:
            checks.append(CheckResult("nearest_reference_similarity", "FAIL", "Nearest-reference similarity is missing and no reference file is available."))
    else:
        final_df["nearest_reference_similarity"] = pd.to_numeric(final_df["nearest_reference_similarity"], errors="coerce")
        checks.append(CheckResult("nearest_reference_similarity", "PASS", "Using nearest-reference similarity column from final-panel table."))

    if "nearest_paper_candidate_similarity" not in final_df.columns or pd.to_numeric(final_df.get("nearest_paper_candidate_similarity", pd.Series(dtype=float)), errors="coerce").notna().sum() == 0:
        internal = nearest_internal_similarity(pairwise)
        # map by IDs
        id_map = {str(k): v for k, v in internal.items()}
        final_df["nearest_paper_candidate_similarity"] = final_df["generated_id"].astype(str).map(id_map)
        if final_df["nearest_paper_candidate_similarity"].isna().all():
            final_df["nearest_paper_candidate_similarity"] = internal.to_numpy()[:len(final_df)]
        checks.append(CheckResult("nearest_candidate_similarity", "WARN", "Computed candidate-context similarity from the pairwise final-panel matrix because input column was unavailable."))
    else:
        final_df["nearest_paper_candidate_similarity"] = pd.to_numeric(final_df["nearest_paper_candidate_similarity"], errors="coerce")
        checks.append(CheckResult("nearest_candidate_similarity", "PASS", "Using candidate-context similarity column from final-panel table."))

    aa_df = data.get("aa")
    km_df = data.get("kmer")
    if reference_df is not None:
        final_seqs = final_df["sequence"].astype(str).tolist()
        ref_seqs = get_ref_sequences(reference_df)
        if aa_df is None:
            aa_df = aa_enrichment(final_seqs, ref_seqs)
            checks.append(CheckResult("aa_enrichment", "WARN", "Computed amino-acid enrichment from final/reference sequences."))
        if km_df is None:
            km_df = kmer_enrichment(final_seqs, ref_seqs, k=args.similarity_k, top_n=25)
            checks.append(CheckResult("kmer_enrichment", "WARN", "Computed k-mer enrichment from final/reference sequences."))

    return final_df, pairwise, reference_df, aa_df, km_df


def write_reports(dirs: OutputDirs, checks: List[CheckResult], discovery: List[Dict[str, str]], files: Dict[str, str], data_paths: pd.DataFrame) -> Dict[str, str]:
    n_fail = sum(c.status == "FAIL" for c in checks)
    n_warn = sum(c.status == "WARN" for c in checks)
    status = "FAIL" if n_fail else ("WARN" if n_warn else "PASS")
    score = max(0, 100 - 15 * n_fail - 2 * n_warn)

    report = dirs.reports / "step9_readiness_report.md"
    lines = [
        "# OncoPep Step 9 readiness report\n",
        f"**Script version:** `{SCRIPT_VERSION}`\n",
        f"**Overall status:** `{status}`\n",
        f"**Estimated readiness score:** `{score}/100`\n",
        "## Validation checks\n",
    ]
    for c in checks:
        lines.append(f"- **[{c.status}] {c.name}**: {c.message}\n")
    lines.append("\n## Input paths\n")
    try:
        lines.append(data_paths.to_markdown(index=False) + "\n")
    except Exception:
        lines.append(data_paths.to_csv(index=False) + "\n")
    lines.append("\n## Discovery log\n")
    for rec in discovery:
        lines.append(f"- role=`{rec.get('role','')}` action=`{rec.get('action','')}` path=`{rec.get('path','')}` reason=`{rec.get('reason','')}`\n")
    lines.append("\n## Output files\n")
    for k, v in files.items():
        lines.append(f"- **{k}**: `{v}`\n")
    lines.append("\n## Non-duplication statement\n")
    lines.append("This Step 9 package uses contextual nearest-neighbor similarity, summary similarity statistics, main-figure residue-category context, and a two-panel supplementary sequence-composition figure. The pairwise final-panel similarity heatmap was removed from the plotted supplement and retained as source data/report to avoid a crowded supplementary figure. The package excludes selection-audit and prioritization-score-shift panels because those belong to Step 8/Figure 4.\n")
    report.write_text("\n".join(lines), encoding="utf-8")
    files["readiness_report"] = str(report)

    manifest = {
        "script_version": SCRIPT_VERSION,
        "output_root": str(dirs.root),
        "checks": [asdict(c) for c in checks],
        "files": files,
        "input_paths": data_paths.to_dict(orient="records"),
        "discovery_log": discovery,
    }
    manifest_path = dirs.reports / "step9_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    files["manifest_json"] = str(manifest_path)

    readme = dirs.reports / "README_step9_outputs.txt"
    readme.write_text(
        f"OncoPep Step 9 output package\nScript version: {SCRIPT_VERSION}\n\n"
        "Main Figure 5: contextual similarity support and internal diversity of the final OncoPep panel.\n"
        "Supplementary Figure S14: compositional context of the final OncoPep panel.\n\n"
        "This package intentionally excludes Step 8 selection-audit/prioritization score-shift panels.\n",
        encoding="utf-8",
    )
    files["readme"] = str(readme)

    req = dirs.reports / "requirements_step9_minimal.txt"
    req.write_text("python>=3.10\nnumpy>=1.23\npandas>=1.5\nmatplotlib>=3.6\nopenpyxl>=3.0\n", encoding="utf-8")
    files["requirements_file"] = str(req)

    try:
        if "__file__" in globals():
            src = Path(__file__).resolve()
            if src.exists():
                dst = dirs.code / "OncoPep_step9_PLOS_contextual_similarity_diversity_final.py"
                shutil.copy2(src, dst)
                files["code_snapshot"] = str(dst)
    except Exception:
        pass
    return files


def main(argv: Optional[Sequence[str]] = None) -> int:
    parser = build_parser()
    args, unknown = parser.parse_known_args(clean_argv(argv))
    if not args.quiet:
        print(f"Hard-fix version: {SCRIPT_VERSION}")
        if unknown:
            print(f"Ignored unknown arguments: {unknown}")

    dirs = ensure_dirs(Path(args.step9_root).expanduser().resolve())
    checks: List[CheckResult] = []
    discovery: List[Dict[str, str]] = []
    data = load_inputs(args, checks, discovery)
    files: Dict[str, str] = {}

    if any(c.status == "FAIL" for c in checks):
        data_paths = data.get("_paths", pd.DataFrame())
        write_reports(dirs, checks, discovery, files, data_paths)
        msg = "Required Step 9 inputs are missing. Review step9_readiness_report.md."
        if is_notebook():
            raise RuntimeError(msg)
        print("ERROR:", msg, file=sys.stderr)
        return 2

    final_df, pairwise, reference_df, aa_df, km_df = prepare_data(data, args, checks)

    # Check post-preparation fatal conditions
    if final_df["nearest_reference_similarity"].isna().all():
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5A cannot be plotted: nearest-reference similarity unavailable."))
    if final_df["nearest_paper_candidate_similarity"].isna().all():
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5A/B cannot be plotted: candidate-context similarity unavailable."))
    if pairwise.shape[0] < 2:
        checks.append(CheckResult("plot_readiness", "FAIL", "Pairwise internal-diversity source data cannot be summarized: pairwise matrix has fewer than two candidates."))
    if reference_df is None:
        checks.append(CheckResult("plot_readiness", "FAIL", "Figure 5C cannot be plotted: reference corpus is required for residue-category context."))

    # Save prepared final panel and matrix
    final_df.to_csv(dirs.source_data / "step9_final_panel_similarity_context_table.csv", index=False)
    pairwise.reset_index().rename(columns={"index": "candidate_id"}).to_csv(dirs.source_data / "step9_final_panel_similarity_matrix.csv", index=False)
    files["step9_final_panel_similarity_context_table"] = str(dirs.source_data / "step9_final_panel_similarity_context_table.csv")
    files["step9_final_panel_similarity_matrix"] = str(dirs.source_data / "step9_final_panel_similarity_matrix.csv")

    if any(c.status == "FAIL" for c in checks):
        write_reports(dirs, checks, discovery, files, data.get("_paths", pd.DataFrame()))
        msg = "Step 9 plotting readiness failed. Review step9_readiness_report.md."
        if is_notebook():
            raise RuntimeError(msg)
        print("ERROR:", msg, file=sys.stderr)
        return 2

    fig5_files = plot_figure5(
        final_df=final_df,
        pairwise=pairwise,
        reference_df=reference_df,
        source_data_dir=dirs.source_data,
        out_base=dirs.main_figure / "Figure_5_contextual_similarity_diversity",
        dpi=args.dpi,
    )
    for k, v in fig5_files.items():
        files[f"Figure_5_{k}"] = v
    checks.append(CheckResult("Figure_5", "PASS", "Generated redesigned Figure 5 with contextual NN similarity, NN summary, and residue-category context."))

    if not args.no_supplementary:
        if reference_df is not None and aa_df is not None and km_df is not None:
            s14_files = plot_supplementary_s14(
                final_df=final_df,
                reference_df=reference_df,
                pairwise=pairwise,
                aa_df=aa_df,
                kmer_df=km_df,
                source_data_dir=dirs.source_data,
                out_base=dirs.supplementary_figures / "Supplementary_Figure_S14_additional_context",
                dpi=args.dpi,
            )
            for k, v in s14_files.items():
                files[f"Supplementary_Figure_S14_{k}"] = v
            removed_report_path = dirs.source_data / "step9_removed_panel_report.csv"
            pairwise_source_path = dirs.source_data / "step9_pairwise_similarity_matrix.csv"
            if removed_report_path.exists():
                files["step9_removed_panel_report"] = str(removed_report_path)
            if pairwise_source_path.exists():
                files["step9_pairwise_similarity_matrix"] = str(pairwise_source_path)
            checks.append(CheckResult("Supplementary_Figure_S14", "PASS", "Generated two-panel Supplementary Figure S14 with amino-acid enrichment and 3-mer enrichment; pairwise similarity retained as source data/report."))
        else:
            checks.append(CheckResult("Supplementary_Figure_S14", "WARN", "S14 not generated because reference/composition data were unavailable."))

    # Additional source-data summary outputs
    sim_summary_rows = []
    for context, col in [("Reference context", "nearest_reference_similarity"), ("Candidate context", "nearest_paper_candidate_similarity")]:
        arr = pd.to_numeric(final_df[col], errors="coerce").dropna().to_numpy(dtype=float)
        sim_summary_rows.append({"context": context, **percentile_summary(arr)})
    sim_summary = pd.DataFrame(sim_summary_rows)
    sim_summary.to_csv(dirs.source_data / "step9_contextual_similarity_summary.csv", index=False)
    files["step9_contextual_similarity_summary"] = str(dirs.source_data / "step9_contextual_similarity_summary.csv")

    arr = pairwise.to_numpy(dtype=float)
    mask = ~np.eye(arr.shape[0], dtype=bool)
    offdiag = arr[mask]
    bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
    labels = ["0-0.05", "0.05-0.10", "0.10-0.15", "0.15-0.20", "0.20-0.30", "0.30-1.0"]
    cats = pd.cut(offdiag, bins=bins, labels=labels, include_lowest=True, right=False)
    div_summary = cats.value_counts().reindex(labels, fill_value=0).rename_axis("similarity_bin").reset_index(name="pair_count")
    div_summary["fraction"] = div_summary["pair_count"] / max(div_summary["pair_count"].sum(), 1)
    div_summary.to_csv(dirs.source_data / "step9_internal_diversity_summary.csv", index=False)
    files["step9_internal_diversity_summary"] = str(dirs.source_data / "step9_internal_diversity_summary.csv")

    try:
        comp_summary = residue_category_df(final_df, reference_df)
        comp_summary.to_csv(dirs.source_data / "step9_composition_context_summary.csv", index=False)
        files["step9_composition_context_summary"] = str(dirs.source_data / "step9_composition_context_summary.csv")
    except Exception:
        pass

    files = write_reports(dirs, checks, discovery, files, data.get("_paths", pd.DataFrame()))

    n_fail = sum(c.status == "FAIL" for c in checks)
    n_warn = sum(c.status == "WARN" for c in checks)
    status = "FAIL" if n_fail else ("WARN" if n_warn else "PASS")
    score = max(0, 100 - 15*n_fail - 2*n_warn)
    if not args.quiet:
        print("\nOncoPep Step 9 package generated.")
        print(f"Root: {dirs.root}")
        print(f"Readiness status: {status}; estimated score: {score}/100")
        print(f"Main figure: {files.get('Figure_5_png')}")
        if "Supplementary_Figure_S14_png" in files:
            print(f"Supplementary figure: {files.get('Supplementary_Figure_S14_png')}")
        print(f"Readiness report: {files.get('readiness_report')}")
    return 0


if __name__ == "__main__":
    try:
        rc = main()
        if is_notebook() and rc != 0:
            raise RuntimeError(f"Step 9 script failed with exit code {rc}.")
        if not is_notebook():
            raise SystemExit(rc)
    except Exception as exc:
        if is_notebook():
            raise RuntimeError(str(exc)) from exc
        print("\nERROR: OncoPep Step 9 figure generation failed.\n", file=sys.stderr)
        print(str(exc), file=sys.stderr)
        traceback.print_exc()
        raise SystemExit(2)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Figure 4 and Supplementary Figure S13 — final figure-generation script.

Scientific scope
----------------
Figure 4:
    Multi-objective prioritization of generated OncoPep candidates.
    A. Prioritization-stage reduction
    B. Novelty, descriptor-plausibility, and diversity support
    C. Composite-score enrichment

Supplementary Figure S13:
    Robustness of OncoPep prioritization schemes.
    A. Pairwise Jaccard overlap among four fixed ranking schemes
    B. Candidate recurrence across the four schemes

This script is intentionally visualization-only. It does not retrain models,
regenerate peptides, recompute ranking membership, or perform bootstrap
shortlist-resampling. It validates and summarizes the frozen source tables.
"""

from __future__ import annotations

import hashlib
import json
import shutil
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, Mapping, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import LogFormatterMathtext, LogLocator, NullFormatter
import numpy as np
import pandas as pd


SCRIPT_VERSION = "v1.0_2026-07-13_fig4_s13_final"


# =============================================================================
# Configuration
# =============================================================================

@dataclass(frozen=True)
class Config:
    project_root: Path = Path(
        "/home/data3/Moe/nature_computational_peponco"
    )
    output_root: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "PLOS/plos_comp/step_08"
    )

    passed_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_supplementary/"
        "table_s8_12_full_ranked_passed_pool.csv"
    )
    shortlist_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_main/table_8_1_shortlist_main.csv"
    )
    final_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_main/table_8_2_final_diverse_panel.csv"
    )
    stability_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_supplementary/"
        "table_s8_5_ranking_scheme_stability.csv"
    )
    recurrence_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_supplementary/"
        "table_s8_6_candidate_recurrence_across_schemes.csv"
    )

    generated_count: int = 10_840
    qc_passed_count: int = 10_291
    descriptor_plausible_count: int = 10_237
    shortlist_count: int = 24
    final_count: int = 12

    scheme_order: Tuple[str, ...] = (
        "Primary",
        "Novelty-weighted",
        "Plausibility-weighted",
        "Diversity-weighted",
    )

    primary_weights: Tuple[float, float, float] = (0.35, 0.35, 0.30)
    numeric_tolerance: float = 1e-6
    manuscript_value_tolerance: float = 0.005
    expected_recurrence_total: int = 40

    figure_width_in: float = 7.35
    figure4_height_in: float = 5.35
    s13_height_in: float = 3.55
    png_dpi: int = 300
    tiff_dpi: int = 600


@dataclass(frozen=True)
class OutputDirs:
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


# =============================================================================
# Locked manuscript values used only for consistency auditing
# =============================================================================

EXPECTED_STAGE_COUNTS: Mapping[str, int] = {
    "Generated": 10_840,
    "QC-passed": 10_291,
    "Descriptor-plausible": 10_237,
    "Shortlist": 24,
    "Final panel": 12,
}

EXPECTED_MEDIANS: Mapping[Tuple[str, str], float] = {
    ("Descriptor-plausible", "Novelty"): 0.939,
    ("Shortlist", "Novelty"): 0.948,
    ("Final panel", "Novelty"): 0.947,
    ("Descriptor-plausible", "Plausibility"): 0.533,
    ("Shortlist", "Plausibility"): 0.759,
    ("Final panel", "Plausibility"): 0.769,
    ("Descriptor-plausible", "Diversity"): 0.882,
    ("Shortlist", "Diversity"): 0.908,
    ("Final panel", "Diversity"): 0.908,
    ("Descriptor-plausible", "Composite score"): 0.777,
    ("Shortlist", "Composite score"): 0.867,
    ("Final panel", "Composite score"): 0.872,
}

EXPECTED_PRIMARY_OVERLAPS: Mapping[str, float] = {
    "Novelty-weighted": 0.655,
    "Plausibility-weighted": 0.600,
    "Diversity-weighted": 0.500,
}

EXPECTED_RECURRENCE: Mapping[int, int] = {
    1: 14,
    2: 7,
    3: 8,
    4: 11,
}


# =============================================================================
# Visual style
# =============================================================================

COLORS = {
    "generated": "#BDBDBD",
    "qc": "#E69F00",
    "plausible": "#277DA1",
    "shortlist": "#B5651D",
    "final": "#D55E00",
    "dark": "#2D2D2D",
    "grid": "#E6E6E6",
    "spine": "#B8B8B8",
    "white": "#FFFFFF",
    "missing": "#EEEEEE",
}

STAGE_COLORS = {
    "Generated": COLORS["generated"],
    "QC-passed": COLORS["qc"],
    "Descriptor-plausible": COLORS["plausible"],
    "Shortlist": COLORS["shortlist"],
    "Final panel": COLORS["final"],
}

SUMMARY_STAGE_COLORS = {
    "Descriptor-plausible": COLORS["plausible"],
    "Shortlist": COLORS["shortlist"],
    "Final panel": COLORS["final"],
}

SCORE_CMAP = LinearSegmentedColormap.from_list(
    "oncopep_scores",
    ["#F6FBFD", "#B7DCE8", "#5BA8C1", "#1F6D8A"],
)

OVERLAP_CMAP = LinearSegmentedColormap.from_list(
    "oncopep_overlap",
    ["#F4FAF6", "#B8DCC2", "#5FAFB7", "#355C6E"],
)


# =============================================================================
# General helpers
# =============================================================================

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_output_dirs(cfg: Config) -> OutputDirs:
    out = OutputDirs(
        main_figure=cfg.output_root / "main_figure",
        supplementary_figures=cfg.output_root / "supplementary_figures",
        source_data=cfg.output_root / "source_data",
        reports=cfg.output_root / "reports",
        code=cfg.output_root / "code",
    )
    for path in asdict(out).values():
        Path(path).mkdir(parents=True, exist_ok=True)
    return out


def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")
    if not path.is_file():
        raise ValueError(f"{label} is not a file: {path}")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1_048_576), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv(path: Path, label: str) -> pd.DataFrame:
    require_file(path, label)
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as exc:
        raise RuntimeError(f"Could not read {label}: {path}") from exc
    if df.empty:
        raise ValueError(f"{label} is empty: {path}")
    df.columns = [str(col).strip() for col in df.columns]
    return df


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def check_score_range(series: pd.Series, label: str, tolerance: float) -> None:
    values = pd.to_numeric(series, errors="coerce")
    if values.isna().any():
        n_bad = int(values.isna().sum())
        raise ValueError(f"{label} contains {n_bad} missing/non-numeric values.")
    if ((values < -tolerance) | (values > 1 + tolerance)).any():
        lo = float(values.min())
        hi = float(values.max())
        raise ValueError(
            f"{label} must lie in [0, 1]; observed range was [{lo}, {hi}]."
        )


def resolve_numeric_alias(
    df: pd.DataFrame,
    canonical: str,
    aliases: Sequence[str],
    table_label: str,
    tolerance: float,
) -> pd.Series:
    present = [column for column in aliases if column in df.columns]
    if not present:
        raise ValueError(
            f"{table_label} is missing `{canonical}`. Accepted columns: {list(aliases)}"
        )

    resolved = pd.to_numeric(df[present[0]], errors="coerce")
    if resolved.isna().any():
        raise ValueError(
            f"{table_label}:{present[0]} contains missing/non-numeric values."
        )

    for other in present[1:]:
        other_values = pd.to_numeric(df[other], errors="coerce")
        if other_values.isna().any():
            raise ValueError(
                f"{table_label}:{other} contains missing/non-numeric values."
            )
        if not np.allclose(
            resolved.to_numpy(float),
            other_values.to_numpy(float),
            rtol=0.0,
            atol=tolerance,
        ):
            raise ValueError(
                f"{table_label} contains conflicting aliases for `{canonical}`: "
                f"{present[0]} and {other}."
            )
    return resolved.astype(float)


def resolve_sequence_column(df: pd.DataFrame, table_label: str) -> str:
    candidates = ["sequence", "peptide_sequence", "peptide", "seq"]
    found = [column for column in candidates if column in df.columns]
    if not found:
        raise ValueError(
            f"{table_label} must contain a sequence column. Tried: {candidates}"
        )
    return found[0]


def standardize_candidate_table(
    df: pd.DataFrame,
    table_label: str,
    cfg: Config,
) -> pd.DataFrame:
    out = df.copy()

    aliases = {
        "novelty_score": ["novelty_score"],
        "plausibility_score": [
            "descriptor_plausibility_score",
            "plausibility_score",
        ],
        "diversity_score": ["diversity_score"],
        "final_score": [
            "final_score",
            "composite_score",
            "prioritization_score",
        ],
    }

    for canonical, candidates in aliases.items():
        out[canonical] = resolve_numeric_alias(
            out,
            canonical,
            candidates,
            table_label,
            cfg.numeric_tolerance,
        )
        check_score_range(
            out[canonical],
            f"{table_label}:{canonical}",
            cfg.numeric_tolerance,
        )

    sequence_col = resolve_sequence_column(out, table_label)
    out["sequence"] = (
        out[sequence_col]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    if (out["sequence"] == "").any():
        raise ValueError(f"{table_label} contains empty sequences.")
    if out["sequence"].duplicated().any():
        n_dup = int(out["sequence"].duplicated().sum())
        raise ValueError(f"{table_label} contains {n_dup} duplicate sequences.")

    w_novelty, w_plausibility, w_diversity = cfg.primary_weights
    reconstructed = (
        w_novelty * out["novelty_score"]
        + w_plausibility * out["plausibility_score"]
        + w_diversity * out["diversity_score"]
    )
    if not np.allclose(
        out["final_score"].to_numpy(float),
        reconstructed.to_numpy(float),
        rtol=0.0,
        atol=cfg.numeric_tolerance,
    ):
        max_diff = float(
            np.max(
                np.abs(
                    out["final_score"].to_numpy(float)
                    - reconstructed.to_numpy(float)
                )
            )
        )
        raise ValueError(
            f"{table_label}: final_score does not match the locked "
            f"0.35/0.35/0.30 formula; maximum absolute difference={max_diff:.8g}."
        )

    return out


def validate_stage_membership(
    passed: pd.DataFrame,
    shortlist: pd.DataFrame,
    final_panel: pd.DataFrame,
    cfg: Config,
) -> None:
    expected = {
        "descriptor-plausible table": cfg.descriptor_plausible_count,
        "shortlist table": cfg.shortlist_count,
        "final-panel table": cfg.final_count,
    }
    observed = {
        "descriptor-plausible table": len(passed),
        "shortlist table": len(shortlist),
        "final-panel table": len(final_panel),
    }
    for label, expected_n in expected.items():
        if observed[label] != expected_n:
            raise ValueError(
                f"{label} contains {observed[label]:,} rows; "
                f"expected {expected_n:,}."
            )

    passed_set = set(passed["sequence"])
    shortlist_set = set(shortlist["sequence"])
    final_set = set(final_panel["sequence"])

    if not shortlist_set.issubset(passed_set):
        missing = sorted(shortlist_set - passed_set)
        raise ValueError(
            f"{len(missing)} shortlist sequences are absent from the "
            f"descriptor-plausible table. First examples: {missing[:5]}"
        )
    if not final_set.issubset(shortlist_set):
        missing = sorted(final_set - shortlist_set)
        raise ValueError(
            f"{len(missing)} final-panel sequences are absent from the shortlist. "
            f"First examples: {missing[:5]}"
        )


def median_iqr(values: Iterable[float]) -> Tuple[float, float, float, int]:
    array = np.asarray(list(values), dtype=float)
    array = array[np.isfinite(array)]
    if len(array) == 0:
        raise ValueError("Cannot summarize an empty numeric array.")
    q1, median, q3 = np.percentile(array, [25, 50, 75])
    return float(median), float(q1), float(q3), int(len(array))


def set_publication_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 8.0,
            "axes.titlesize": 9.0,
            "axes.labelsize": 8.2,
            "xtick.labelsize": 7.4,
            "ytick.labelsize": 7.4,
            "legend.fontsize": 7.4,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
            "axes.edgecolor": COLORS["spine"],
            "axes.labelcolor": COLORS["dark"],
            "xtick.color": COLORS["dark"],
            "ytick.color": COLORS["dark"],
            "text.color": COLORS["dark"],
            "axes.titlecolor": COLORS["dark"],
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.65)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.65)
    elif grid_axis == "both":
        ax.grid(True, color=COLORS["grid"], linewidth=0.65)

    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(width=0.75, length=3.3)


def add_panel_label(
    ax: plt.Axes,
    label: str,
    x: float = -0.11,
    y: float = 1.04,
) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=12.5,
        fontweight="bold",
        ha="left",
        va="bottom",
        clip_on=False,
    )


def save_figure(
    fig: plt.Figure,
    base_path: Path,
    cfg: Config,
) -> Dict[str, str]:
    base_path.parent.mkdir(parents=True, exist_ok=True)
    outputs = {
        "png": base_path.with_suffix(".png"),
        "pdf": base_path.with_suffix(".pdf"),
        "tiff": base_path.with_suffix(".tiff"),
    }

    fig.savefig(
        outputs["png"],
        dpi=cfg.png_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        outputs["pdf"],
        bbox_inches="tight",
        facecolor="white",
        metadata={
            "Title": base_path.stem,
            "Creator": f"OncoPep {SCRIPT_VERSION}",
        },
    )
    fig.savefig(
        outputs["tiff"],
        dpi=cfg.tiff_dpi,
        bbox_inches="tight",
        facecolor="white",
        pil_kwargs={"compression": "tiff_lzw"},
    )
    plt.close(fig)
    return {key: str(value) for key, value in outputs.items()}


# =============================================================================
# Figure 4 source-data construction
# =============================================================================

def build_stage_counts(
    passed: pd.DataFrame,
    shortlist: pd.DataFrame,
    final_panel: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    rows = [
        {
            "stage": "Generated",
            "count": cfg.generated_count,
            "count_source": "locked manuscript value",
        },
        {
            "stage": "QC-passed",
            "count": cfg.qc_passed_count,
            "count_source": "locked manuscript value",
        },
        {
            "stage": "Descriptor-plausible",
            "count": len(passed),
            "count_source": "input table row count",
        },
        {
            "stage": "Shortlist",
            "count": len(shortlist),
            "count_source": "input table row count",
        },
        {
            "stage": "Final panel",
            "count": len(final_panel),
            "count_source": "input table row count",
        },
    ]
    table = pd.DataFrame(rows)
    table["previous_stage_count"] = table["count"].shift(1)
    table["retention_from_previous"] = (
        table["count"] / table["previous_stage_count"]
    )
    table.loc[0, "retention_from_previous"] = 1.0
    table["percent_of_generated"] = (
        table["count"] / cfg.generated_count * 100.0
    )
    return table


def build_score_summary(
    passed: pd.DataFrame,
    shortlist: pd.DataFrame,
    final_panel: pd.DataFrame,
) -> pd.DataFrame:
    stages = [
        ("Descriptor-plausible", passed),
        ("Shortlist", shortlist),
        ("Final panel", final_panel),
    ]
    metrics = [
        ("novelty_score", "Novelty"),
        ("plausibility_score", "Plausibility"),
        ("diversity_score", "Diversity"),
        ("final_score", "Composite score"),
    ]
    rows = []
    for stage_name, table in stages:
        for column, metric_name in metrics:
            median, q1, q3, n = median_iqr(table[column])
            rows.append(
                {
                    "stage": stage_name,
                    "metric": metric_name,
                    "metric_column": column,
                    "n": n,
                    "median": median,
                    "q1": q1,
                    "q3": q3,
                    "error_bar": "interquartile range",
                }
            )
    return pd.DataFrame(rows)


# =============================================================================
# Supplementary Figure S13 source-data construction
# =============================================================================

def normalize_scheme_label(value: object) -> str:
    raw = str(value).strip().lower()
    key = (
        raw.replace("_", "-")
        .replace(" ", "-")
        .replace("heavy", "weighted")
    )
    while "--" in key:
        key = key.replace("--", "-")

    mapping = {
        "primary": "Primary",
        "primary-weighted": "Primary",
        "novelty-weighted": "Novelty-weighted",
        "novelty": "Novelty-weighted",
        "plausibility-weighted": "Plausibility-weighted",
        "plausibility": "Plausibility-weighted",
        "descriptor-plausibility-weighted": "Plausibility-weighted",
        "diversity-weighted": "Diversity-weighted",
        "diversity": "Diversity-weighted",
    }
    if key not in mapping:
        raise ValueError(
            f"Unrecognized ranking-scheme label `{value}`. "
            "Expected Primary, Novelty-weighted, Plausibility-weighted, "
            "or Diversity-weighted."
        )
    return mapping[key]


def resolve_required_column(
    df: pd.DataFrame,
    canonical: str,
    aliases: Sequence[str],
    table_label: str,
) -> str:
    present = [column for column in aliases if column in df.columns]
    if not present:
        raise ValueError(
            f"{table_label} is missing `{canonical}`. Tried: {list(aliases)}"
        )
    return present[0]


def build_stability_sources(
    stability_df: pd.DataFrame,
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    scheme_a_col = resolve_required_column(
        stability_df,
        "scheme_a",
        ["scheme_a", "scheme1", "row_scheme"],
        "stability table",
    )
    scheme_b_col = resolve_required_column(
        stability_df,
        "scheme_b",
        ["scheme_b", "scheme2", "col_scheme"],
        "stability table",
    )
    overlap_col = resolve_required_column(
        stability_df,
        "jaccard_overlap",
        ["jaccard_overlap", "jaccard", "overlap"],
        "stability table",
    )

    raw = pd.DataFrame(
        {
            "scheme_a": stability_df[scheme_a_col].map(normalize_scheme_label),
            "scheme_b": stability_df[scheme_b_col].map(normalize_scheme_label),
            "jaccard_overlap": pd.to_numeric(
                stability_df[overlap_col],
                errors="coerce",
            ),
        }
    )
    if raw["jaccard_overlap"].isna().any():
        raise ValueError("Stability table contains missing/non-numeric overlaps.")
    if (
        (raw["jaccard_overlap"] < -cfg.numeric_tolerance)
        | (raw["jaccard_overlap"] > 1 + cfg.numeric_tolerance)
    ).any():
        raise ValueError("Jaccard overlaps must lie in [0, 1].")

    pair_values: Dict[Tuple[str, str], list[float]] = {}
    for row in raw.itertuples(index=False):
        if row.scheme_a == row.scheme_b:
            continue
        ordered_pair = tuple(
            sorted(
                [row.scheme_a, row.scheme_b],
                key=cfg.scheme_order.index,
            )
        )
        pair_values.setdefault(ordered_pair, []).append(
            float(row.jaccard_overlap)
        )

    expected_pairs = {
        (cfg.scheme_order[i], cfg.scheme_order[j])
        for i in range(len(cfg.scheme_order))
        for j in range(i + 1, len(cfg.scheme_order))
    }
    missing_pairs = sorted(expected_pairs - set(pair_values))
    extra_pairs = sorted(set(pair_values) - expected_pairs)
    if missing_pairs:
        raise ValueError(
            f"Stability table is missing pairwise overlaps: {missing_pairs}"
        )
    if extra_pairs:
        raise ValueError(
            f"Stability table contains unexpected scheme pairs: {extra_pairs}"
        )

    pair_rows = []
    matrix = pd.DataFrame(
        np.eye(len(cfg.scheme_order), dtype=float),
        index=cfg.scheme_order,
        columns=cfg.scheme_order,
    )
    for pair in sorted(expected_pairs, key=lambda x: (
        cfg.scheme_order.index(x[0]),
        cfg.scheme_order.index(x[1]),
    )):
        values = np.asarray(pair_values[pair], dtype=float)
        if np.ptp(values) > cfg.numeric_tolerance:
            raise ValueError(
                f"Conflicting overlap values were supplied for {pair}: "
                f"{values.tolist()}"
            )
        value = float(values.mean())
        matrix.loc[pair[0], pair[1]] = value
        matrix.loc[pair[1], pair[0]] = value
        pair_rows.append(
            {
                "scheme_a": pair[0],
                "scheme_b": pair[1],
                "jaccard_overlap": value,
            }
        )

    long_rows = []
    for scheme_a in cfg.scheme_order:
        for scheme_b in cfg.scheme_order:
            long_rows.append(
                {
                    "scheme_a": scheme_a,
                    "scheme_b": scheme_b,
                    "jaccard_overlap": float(
                        matrix.loc[scheme_a, scheme_b]
                    ),
                    "is_diagonal": scheme_a == scheme_b,
                }
            )

    return matrix, pd.DataFrame(long_rows), pd.DataFrame(pair_rows)


def build_recurrence_source(
    recurrence_df: pd.DataFrame,
    cfg: Config,
) -> Tuple[pd.DataFrame, list[str]]:
    recurrence_col = resolve_required_column(
        recurrence_df,
        "scheme_recurrence_n",
        [
            "scheme_recurrence_n",
            "scheme_recovery_count",
            "num_schemes",
            "n_schemes",
        ],
        "recurrence table",
    )

    count_aliases = [
        column
        for column in ["candidate_count", "count", "n_candidates"]
        if column in recurrence_df.columns
    ]

    if len(recurrence_df) <= len(cfg.scheme_order) and count_aliases:
        count_col = count_aliases[0]
        out = pd.DataFrame(
            {
                "scheme_recurrence_n": pd.to_numeric(
                    recurrence_df[recurrence_col],
                    errors="coerce",
                ),
                "candidate_count": pd.to_numeric(
                    recurrence_df[count_col],
                    errors="coerce",
                ),
            }
        )
        if out.isna().any().any():
            raise ValueError(
                "Aggregated recurrence table contains missing/non-numeric values."
            )
        out = (
            out.groupby("scheme_recurrence_n", as_index=False)
            ["candidate_count"]
            .sum()
        )
    else:
        values = pd.to_numeric(
            recurrence_df[recurrence_col],
            errors="coerce",
        )
        if values.isna().any():
            raise ValueError(
                "Candidate-level recurrence table contains "
                "missing/non-numeric recurrence values."
            )
        if not np.allclose(values, np.round(values)):
            raise ValueError("Recurrence values must be integers.")
        counts = values.astype(int).value_counts().sort_index()
        out = pd.DataFrame(
            {
                "scheme_recurrence_n": counts.index,
                "candidate_count": counts.values,
            }
        )

    out["scheme_recurrence_n"] = (
        out["scheme_recurrence_n"].astype(int)
    )
    out["candidate_count"] = out["candidate_count"].astype(int)

    valid_range = range(1, len(cfg.scheme_order) + 1)
    if not out["scheme_recurrence_n"].isin(valid_range).all():
        bad = sorted(
            set(out["scheme_recurrence_n"]) - set(valid_range)
        )
        raise ValueError(
            f"Recurrence values outside 1-{len(cfg.scheme_order)}: {bad}"
        )

    out = (
        out.set_index("scheme_recurrence_n")
        .reindex(valid_range, fill_value=0)
        .rename_axis("scheme_recurrence_n")
        .reset_index()
    )
    total = int(out["candidate_count"].sum())
    if total != cfg.expected_recurrence_total:
        raise ValueError(
            f"Recurrence denominator is {total}; expected "
            f"{cfg.expected_recurrence_total} distinct candidates."
        )

    out["denominator_n"] = total
    out["fraction"] = out["candidate_count"] / total
    out["percent"] = out["fraction"] * 100.0

    ignored_columns = [
        column
        for column in recurrence_df.columns
        if "bootstrap" in column.lower()
    ]
    return out, ignored_columns


# =============================================================================
# Manuscript consistency audit
# =============================================================================

def build_consistency_audit(
    stage_counts: pd.DataFrame,
    score_summary: pd.DataFrame,
    stability_matrix: pd.DataFrame,
    recurrence: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    rows = []

    for stage, expected in EXPECTED_STAGE_COUNTS.items():
        observed = int(
            stage_counts.loc[
                stage_counts["stage"] == stage,
                "count",
            ].iloc[0]
        )
        rows.append(
            {
                "item": f"Stage count: {stage}",
                "expected": expected,
                "observed": observed,
                "absolute_difference": abs(observed - expected),
                "status": "PASS" if observed == expected else "FAIL",
            }
        )

    for (stage, metric), expected in EXPECTED_MEDIANS.items():
        observed = float(
            score_summary.loc[
                (score_summary["stage"] == stage)
                & (score_summary["metric"] == metric),
                "median",
            ].iloc[0]
        )
        diff = abs(observed - expected)
        rows.append(
            {
                "item": f"Median {metric}: {stage}",
                "expected": expected,
                "observed": observed,
                "absolute_difference": diff,
                "status": (
                    "PASS"
                    if diff <= cfg.manuscript_value_tolerance
                    else "REVIEW"
                ),
            }
        )

    for scheme, expected in EXPECTED_PRIMARY_OVERLAPS.items():
        observed = float(stability_matrix.loc["Primary", scheme])
        diff = abs(observed - expected)
        rows.append(
            {
                "item": f"Primary overlap: {scheme}",
                "expected": expected,
                "observed": observed,
                "absolute_difference": diff,
                "status": (
                    "PASS"
                    if diff <= cfg.manuscript_value_tolerance
                    else "REVIEW"
                ),
            }
        )

    recurrence_lookup = recurrence.set_index(
        "scheme_recurrence_n"
    )["candidate_count"]
    for n_schemes, expected in EXPECTED_RECURRENCE.items():
        observed = int(recurrence_lookup.loc[n_schemes])
        rows.append(
            {
                "item": f"Candidate recurrence: {n_schemes} scheme(s)",
                "expected": expected,
                "observed": observed,
                "absolute_difference": abs(observed - expected),
                "status": "PASS" if observed == expected else "REVIEW",
            }
        )

    return pd.DataFrame(rows)


# =============================================================================
# Figure 4 plotting
# =============================================================================

def plot_figure4(
    stage_counts: pd.DataFrame,
    score_summary: pd.DataFrame,
    output_base: Path,
    cfg: Config,
) -> Dict[str, str]:
    set_publication_style()

    stage_order = [
        "Generated",
        "QC-passed",
        "Descriptor-plausible",
        "Shortlist",
        "Final panel",
    ]
    summary_stages = [
        "Descriptor-plausible",
        "Shortlist",
        "Final panel",
    ]
    component_metrics = [
        "Novelty",
        "Plausibility",
        "Diversity",
    ]

    fig = plt.figure(
        figsize=(cfg.figure_width_in, cfg.figure4_height_in),
        constrained_layout=False,
    )
    grid = gridspec.GridSpec(
        2,
        2,
        figure=fig,
        width_ratios=[1.12, 1.0],
        height_ratios=[1.0, 0.82],
        wspace=0.38,
        hspace=0.44,
    )

    # Panel A: stage reduction
    ax_a = fig.add_subplot(grid[:, 0])
    add_panel_label(ax_a, "A", x=-0.14, y=1.02)

    counts = (
        stage_counts.set_index("stage")
        .reindex(stage_order)
        .reset_index()
    )
    y = np.arange(len(counts))
    bars = ax_a.barh(
        y,
        counts["count"],
        height=0.58,
        color=[STAGE_COLORS[stage] for stage in counts["stage"]],
        edgecolor=[
            COLORS["dark"]
            if stage in {"Shortlist", "Final panel"}
            else "none"
            for stage in counts["stage"]
        ],
        linewidth=[
            0.65
            if stage in {"Shortlist", "Final panel"}
            else 0.0
            for stage in counts["stage"]
        ],
        zorder=3,
    )
    ax_a.set_yticks(y)
    ax_a.set_yticklabels(counts["stage"])
    ax_a.invert_yaxis()
    ax_a.set_xscale("log")
    ax_a.set_xlim(8, max(15_000, counts["count"].max() * 1.65))
    ax_a.xaxis.set_major_locator(
        LogLocator(base=10, subs=(1.0,), numticks=5)
    )
    ax_a.xaxis.set_major_formatter(
        LogFormatterMathtext(base=10)
    )
    ax_a.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )
    ax_a.xaxis.set_minor_formatter(NullFormatter())
    ax_a.set_xlabel("Number of peptides (log scale)")
    ax_a.set_title("Prioritization-stage reduction", pad=8)
    style_axis(ax_a, "x")

    for bar, row in zip(bars, counts.itertuples(index=False)):
        percent = float(row.percent_of_generated)
        percent_text = (
            f"{percent:.2f}%"
            if percent < 0.1
            else f"{percent:.1f}%"
        )
        ax_a.text(
            float(row.count) * 1.09,
            bar.get_y() + bar.get_height() / 2,
            f"{int(row.count):,} ({percent_text})",
            va="center",
            ha="left",
            fontsize=7.4,
            fontweight=(
                "bold"
                if row.stage in {"Shortlist", "Final panel"}
                else "normal"
            ),
        )

    # Panel B: objective-score heatmap
    ax_b = fig.add_subplot(grid[0, 1])
    add_panel_label(ax_b, "B", x=-0.16, y=1.04)

    component_data = score_summary[
        score_summary["metric"].isin(component_metrics)
    ].copy()
    median_matrix = (
        component_data.pivot(
            index="metric",
            columns="stage",
            values="median",
        )
        .reindex(index=component_metrics, columns=summary_stages)
    )
    q1_matrix = (
        component_data.pivot(
            index="metric",
            columns="stage",
            values="q1",
        )
        .reindex(index=component_metrics, columns=summary_stages)
    )
    q3_matrix = (
        component_data.pivot(
            index="metric",
            columns="stage",
            values="q3",
        )
        .reindex(index=component_metrics, columns=summary_stages)
    )
    if median_matrix.isna().any().any():
        raise ValueError(
            "Figure 4B summary matrix contains missing values."
        )

    image = ax_b.imshow(
        median_matrix.to_numpy(float),
        vmin=0,
        vmax=1,
        cmap=SCORE_CMAP,
        aspect="auto",
    )
    ax_b.set_xticks(np.arange(len(summary_stages)))
    ax_b.set_xticklabels(
        ["Descriptor-\nplausible", "Shortlist", "Final panel"]
    )
    ax_b.set_yticks(np.arange(len(component_metrics)))
    ax_b.set_yticklabels(component_metrics)
    ax_b.set_title("Objective-score support", pad=8)
    ax_b.tick_params(length=0)

    for row_index, metric in enumerate(component_metrics):
        for col_index, stage in enumerate(summary_stages):
            median = float(median_matrix.loc[metric, stage])
            q1 = float(q1_matrix.loc[metric, stage])
            q3 = float(q3_matrix.loc[metric, stage])
            text_color = (
                "white" if median >= 0.70 else COLORS["dark"]
            )
            ax_b.text(
                col_index,
                row_index,
                f"{median:.3f}\n[{q1:.3f}–{q3:.3f}]",
                ha="center",
                va="center",
                fontsize=6.25,
                color=text_color,
                linespacing=1.05,
            )

    ax_b.set_xticks(
        np.arange(-0.5, len(summary_stages), 1),
        minor=True,
    )
    ax_b.set_yticks(
        np.arange(-0.5, len(component_metrics), 1),
        minor=True,
    )
    ax_b.grid(which="minor", color="white", linewidth=1.0)
    ax_b.tick_params(which="minor", bottom=False, left=False)
    for spine in ax_b.spines.values():
        spine.set_visible(True)
        spine.set_color(COLORS["spine"])
        spine.set_linewidth(0.75)

    colorbar = fig.colorbar(
        image,
        ax=ax_b,
        fraction=0.046,
        pad=0.035,
    )
    colorbar.set_label("Median score", fontsize=7.5)
    colorbar.ax.tick_params(labelsize=6.8, length=2.5)
    colorbar.outline.set_edgecolor(COLORS["spine"])

    # Panel C: composite score enrichment
    ax_c = fig.add_subplot(grid[1, 1])
    add_panel_label(ax_c, "C", x=-0.16, y=1.05)

    composite = (
        score_summary[
            score_summary["metric"] == "Composite score"
        ]
        .set_index("stage")
        .reindex(summary_stages)
        .reset_index()
    )
    y_positions = np.arange(len(summary_stages))
    medians = composite["median"].to_numpy(float)
    q1 = composite["q1"].to_numpy(float)
    q3 = composite["q3"].to_numpy(float)

    ax_c.barh(
        y_positions,
        medians,
        height=0.48,
        color=[SUMMARY_STAGE_COLORS[stage] for stage in summary_stages],
        edgecolor="none",
        zorder=3,
    )
    ax_c.errorbar(
        medians,
        y_positions,
        xerr=np.vstack(
            [
                np.maximum(medians - q1, 0),
                np.maximum(q3 - medians, 0),
            ]
        ),
        fmt="none",
        ecolor=COLORS["dark"],
        elinewidth=0.9,
        capsize=2.5,
        capthick=0.9,
        zorder=4,
    )
    ax_c.set_yticks(y_positions)
    ax_c.set_yticklabels(
        [
            f"Descriptor-plausible (n={int(composite.loc[0, 'n']):,})",
            f"Shortlist (n={int(composite.loc[1, 'n']):,})",
            f"Final panel (n={int(composite.loc[2, 'n']):,})",
        ]
    )
    ax_c.invert_yaxis()
    ax_c.set_xlim(0, 1.02)
    ax_c.set_xlabel("Composite prioritization score")
    ax_c.set_title("Composite-score enrichment", pad=8)
    style_axis(ax_c, "x")

    for y_position, median, low, high in zip(
        y_positions, medians, q1, q3
    ):
        ax_c.text(
            min(0.995, high + 0.025),
            y_position,
            f"{median:.3f} [{low:.3f}–{high:.3f}]",
            va="center",
            ha="left",
            fontsize=6.8,
        )

    ax_c.text(
        0.01,
        -0.30,
        "Primary score = 0.35 novelty + 0.35 plausibility + 0.30 diversity",
        transform=ax_c.transAxes,
        ha="left",
        va="top",
        fontsize=6.5,
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.97,
        top=0.94,
        bottom=0.12,
    )
    return save_figure(fig, output_base, cfg)


# =============================================================================
# Supplementary Figure S13 plotting
# =============================================================================

def display_scheme_label(label: str) -> str:
    mapping = {
        "Primary": "Primary",
        "Novelty-weighted": "Novelty-\nweighted",
        "Plausibility-weighted": "Plausibility-\nweighted",
        "Diversity-weighted": "Diversity-\nweighted",
    }
    return mapping[label]


def plot_s13(
    stability_matrix: pd.DataFrame,
    recurrence: pd.DataFrame,
    output_base: Path,
    cfg: Config,
) -> Dict[str, str]:
    set_publication_style()

    fig = plt.figure(
        figsize=(cfg.figure_width_in, cfg.s13_height_in),
        constrained_layout=False,
    )
    grid = gridspec.GridSpec(
        1,
        2,
        figure=fig,
        width_ratios=[1.18, 0.92],
        wspace=0.42,
    )

    # Panel A: overlap matrix
    ax_a = fig.add_subplot(grid[0, 0])
    add_panel_label(ax_a, "A", x=-0.15, y=1.04)

    values = stability_matrix.loc[
        cfg.scheme_order, cfg.scheme_order
    ].to_numpy(float)
    masked = values.copy()
    np.fill_diagonal(masked, np.nan)

    image = ax_a.imshow(
        masked,
        vmin=0,
        vmax=1,
        cmap=OVERLAP_CMAP,
        aspect="equal",
    )
    ax_a.set_xticks(np.arange(len(cfg.scheme_order)))
    ax_a.set_yticks(np.arange(len(cfg.scheme_order)))
    ax_a.set_xticklabels(
        [display_scheme_label(label) for label in cfg.scheme_order]
    )
    ax_a.set_yticklabels(
        [display_scheme_label(label) for label in cfg.scheme_order]
    )
    ax_a.set_title("Ranking-scheme overlap", pad=8)
    ax_a.tick_params(length=0)

    for row in range(len(cfg.scheme_order)):
        for col in range(len(cfg.scheme_order)):
            if row == col:
                ax_a.add_patch(
                    Rectangle(
                        (col - 0.5, row - 0.5),
                        1,
                        1,
                        facecolor=COLORS["missing"],
                        edgecolor="white",
                        linewidth=1.0,
                    )
                )
                ax_a.text(
                    col,
                    row,
                    "self",
                    ha="center",
                    va="center",
                    fontsize=7.2,
                    color="#777777",
                )
            else:
                value = values[row, col]
                ax_a.text(
                    col,
                    row,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7.6,
                    color=(
                        "white"
                        if value >= 0.62
                        else COLORS["dark"]
                    ),
                )

    ax_a.set_xticks(
        np.arange(-0.5, len(cfg.scheme_order), 1),
        minor=True,
    )
    ax_a.set_yticks(
        np.arange(-0.5, len(cfg.scheme_order), 1),
        minor=True,
    )
    ax_a.grid(which="minor", color="white", linewidth=1.0)
    ax_a.tick_params(which="minor", bottom=False, left=False)
    for spine in ax_a.spines.values():
        spine.set_visible(True)
        spine.set_color(COLORS["spine"])
        spine.set_linewidth(0.75)

    colorbar = fig.colorbar(
        image,
        ax=ax_a,
        fraction=0.042,
        pad=0.032,
    )
    colorbar.set_label("Jaccard overlap", fontsize=7.5)
    colorbar.ax.tick_params(labelsize=6.8, length=2.5)
    colorbar.outline.set_edgecolor(COLORS["spine"])

    # Panel B: recurrence
    ax_b = fig.add_subplot(grid[0, 1])
    add_panel_label(ax_b, "B", x=-0.16, y=1.04)

    x = recurrence["scheme_recurrence_n"].to_numpy(int)
    counts = recurrence["candidate_count"].to_numpy(int)
    percentages = recurrence["percent"].to_numpy(float)
    total = int(recurrence["denominator_n"].iloc[0])

    bars = ax_b.bar(
        x,
        counts,
        width=0.56,
        color=COLORS["shortlist"],
        edgecolor="none",
        zorder=3,
    )
    ax_b.set_xticks(x)
    ax_b.set_xlabel("Number of schemes recovering candidate")
    ax_b.set_ylabel("Candidate count")
    ax_b.set_title("Candidate recurrence", pad=8)
    ax_b.set_ylim(0, max(counts) * 1.28)
    style_axis(ax_b, "y")

    for bar, count, percentage in zip(
        bars, counts, percentages
    ):
        ax_b.text(
            bar.get_x() + bar.get_width() / 2,
            count + max(counts) * 0.035,
            f"{count}\n({percentage:.0f}%)",
            ha="center",
            va="bottom",
            fontsize=7.4,
            linespacing=0.95,
        )

    ax_b.text(
        0.98,
        0.95,
        f"n = {total} distinct candidates",
        transform=ax_b.transAxes,
        ha="right",
        va="top",
        fontsize=7.3,
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "white",
            "edgecolor": COLORS["spine"],
            "linewidth": 0.7,
        },
    )

    fig.subplots_adjust(
        left=0.09,
        right=0.98,
        top=0.90,
        bottom=0.22,
    )
    return save_figure(fig, output_base, cfg)


# =============================================================================
# Source-data and manifest export
# =============================================================================

def combine_panel_sources(
    sources: Mapping[str, pd.DataFrame],
) -> pd.DataFrame:
    frames = []
    for panel, table in sources.items():
        copy = table.copy()
        copy.insert(0, "panel", panel)
        frames.append(copy)
    return pd.concat(frames, ignore_index=True, sort=False)


def export_source_data(
    dirs: OutputDirs,
    stage_counts: pd.DataFrame,
    score_summary: pd.DataFrame,
    stability_long: pd.DataFrame,
    stability_pairs: pd.DataFrame,
    recurrence: pd.DataFrame,
    consistency_audit: pd.DataFrame,
) -> Dict[str, str]:
    outputs: Dict[str, str] = {}

    fig4_a = stage_counts.copy()
    fig4_b = score_summary[
        score_summary["metric"].isin(
            ["Novelty", "Plausibility", "Diversity"]
        )
    ].copy()
    fig4_c = score_summary[
        score_summary["metric"] == "Composite score"
    ].copy()

    s13_a = stability_long.copy()
    s13_b = recurrence.copy()

    source_tables = {
        "Figure_4_panel_a_source_data.csv": fig4_a,
        "Figure_4_panel_b_source_data.csv": fig4_b,
        "Figure_4_panel_c_source_data.csv": fig4_c,
        "Supplementary_Figure_S13_panel_a_source_data.csv": s13_a,
        "Supplementary_Figure_S13_panel_a_unique_pairs_source_data.csv": stability_pairs,
        "Supplementary_Figure_S13_panel_b_source_data.csv": s13_b,
        "Figure_4_S13_manuscript_consistency_audit.csv": consistency_audit,
    }

    for filename, table in source_tables.items():
        path = dirs.source_data / filename
        write_csv(table, path)
        outputs[filename] = str(path)

    fig4_all = combine_panel_sources(
        {
            "A": fig4_a,
            "B": fig4_b,
            "C": fig4_c,
        }
    )
    fig4_all_path = (
        dirs.source_data / "Figure_4_source_data_all_panels.csv"
    )
    write_csv(fig4_all, fig4_all_path)
    outputs[fig4_all_path.name] = str(fig4_all_path)

    s13_all = combine_panel_sources(
        {
            "A": s13_a,
            "B": s13_b,
        }
    )
    s13_all_path = (
        dirs.source_data
        / "Supplementary_Figure_S13_source_data_all_panels.csv"
    )
    write_csv(s13_all, s13_all_path)
    outputs[s13_all_path.name] = str(s13_all_path)

    return outputs


def export_manifest(
    cfg: Config,
    dirs: OutputDirs,
    input_tables: Mapping[str, pd.DataFrame],
    figure_outputs: Mapping[str, Mapping[str, str]],
    source_outputs: Mapping[str, str],
    ignored_recurrence_columns: Sequence[str],
    consistency_audit: pd.DataFrame,
) -> Path:
    input_paths = {
        "passed_file": cfg.passed_file,
        "shortlist_file": cfg.shortlist_file,
        "final_file": cfg.final_file,
        "stability_file": cfg.stability_file,
        "recurrence_file": cfg.recurrence_file,
    }
    manifest = {
        "script_version": SCRIPT_VERSION,
        "generated_at_utc": utc_now(),
        "scientific_scope": (
            "Figure 4 prioritization summaries and Supplementary Figure S13 "
            "weighting-scheme robustness only."
        ),
        "claim_boundary": (
            "Computational prioritization only; no experimental anticancer "
            "activity, selectivity, toxicity, stability, or mechanism is inferred."
        ),
        "config": {
            **{
                key: str(value)
                if isinstance(value, Path)
                else value
                for key, value in asdict(cfg).items()
            },
        },
        "inputs": {
            name: {
                "path": str(path),
                "sha256": sha256_file(path),
                "shape": list(input_tables[name].shape),
            }
            for name, path in input_paths.items()
        },
        "ignored_recurrence_columns": list(
            ignored_recurrence_columns
        ),
        "figures": figure_outputs,
        "source_data": dict(source_outputs),
        "consistency_audit_status_counts": (
            consistency_audit["status"].value_counts().to_dict()
        ),
        "software": {
            "python_runtime": (
                __import__("sys").version
            ),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": matplotlib.__version__,
        },
    }

    manifest_path = (
        dirs.reports / "Figure_4_S13_generation_manifest.json"
    )
    manifest_path.write_text(
        json.dumps(manifest, indent=2, default=str),
        encoding="utf-8",
    )

    readme = f"""OncoPep Figure 4 and Supplementary Figure S13 package
=================================================================

Script version: {SCRIPT_VERSION}

Figure 4
--------
A. Prioritization-stage reduction
B. Novelty, descriptor-plausibility, and diversity support
C. Composite-score enrichment

Supplementary Figure S13
------------------------
A. Pairwise Jaccard overlap among four fixed ranking schemes
B. Candidate recurrence across the four schemes

Important
---------
This script does not retrain the model, regenerate peptides, rerank candidates,
or use bootstrap shortlist-membership frequencies. It visualizes frozen source
tables, validates the locked 0.35/0.35/0.30 composite score, requires the true
final-panel table, and exports the exact values plotted in each panel.
"""
    (
        dirs.reports / "README_Figure_4_S13.txt"
    ).write_text(readme, encoding="utf-8")

    requirements = "\n".join(
        [
            f"numpy=={np.__version__}",
            f"pandas=={pd.__version__}",
            f"matplotlib=={matplotlib.__version__}",
            "",
        ]
    )
    (
        dirs.reports / "requirements_Figure_4_S13.txt"
    ).write_text(requirements, encoding="utf-8")

    try:
        source_file = Path(__file__).resolve()
        if source_file.exists():
            shutil.copy2(
                source_file,
                dirs.code / "OncoPep_Figure4_S13_final.py",
            )
    except NameError:
        # Expected when executed directly from a notebook cell.
        pass

    return manifest_path


# =============================================================================
# Main workflow
# =============================================================================

def main(config: Config | None = None) -> Dict[str, object]:
    cfg = config or Config()
    dirs = ensure_output_dirs(cfg)

    raw_tables = {
        "passed_file": read_csv(
            cfg.passed_file,
            "descriptor-plausible candidate table",
        ),
        "shortlist_file": read_csv(
            cfg.shortlist_file,
            "shortlist table",
        ),
        "final_file": read_csv(
            cfg.final_file,
            "final-panel table",
        ),
        "stability_file": read_csv(
            cfg.stability_file,
            "scheme-stability table",
        ),
        "recurrence_file": read_csv(
            cfg.recurrence_file,
            "scheme-recurrence table",
        ),
    }

    passed = standardize_candidate_table(
        raw_tables["passed_file"],
        "descriptor-plausible table",
        cfg,
    )
    shortlist = standardize_candidate_table(
        raw_tables["shortlist_file"],
        "shortlist table",
        cfg,
    )
    final_panel = standardize_candidate_table(
        raw_tables["final_file"],
        "final-panel table",
        cfg,
    )
    validate_stage_membership(
        passed,
        shortlist,
        final_panel,
        cfg,
    )

    stage_counts = build_stage_counts(
        passed,
        shortlist,
        final_panel,
        cfg,
    )
    score_summary = build_score_summary(
        passed,
        shortlist,
        final_panel,
    )

    stability_matrix, stability_long, stability_pairs = (
        build_stability_sources(
            raw_tables["stability_file"],
            cfg,
        )
    )
    recurrence, ignored_recurrence_columns = (
        build_recurrence_source(
            raw_tables["recurrence_file"],
            cfg,
        )
    )

    consistency_audit = build_consistency_audit(
        stage_counts,
        score_summary,
        stability_matrix,
        recurrence,
        cfg,
    )

    source_outputs = export_source_data(
        dirs,
        stage_counts,
        score_summary,
        stability_long,
        stability_pairs,
        recurrence,
        consistency_audit,
    )

    figure4_base = (
        dirs.main_figure
        / "Figure_4_prioritization_final"
    )
    s13_base = (
        dirs.supplementary_figures
        / "Supplementary_Figure_S13_prioritization_robustness_final"
    )

    figure_outputs = {
        "Figure_4": plot_figure4(
            stage_counts,
            score_summary,
            figure4_base,
            cfg,
        ),
        "Supplementary_Figure_S13": plot_s13(
            stability_matrix,
            recurrence,
            s13_base,
            cfg,
        ),
    }

    manifest_path = export_manifest(
        cfg,
        dirs,
        raw_tables,
        figure_outputs,
        source_outputs,
        ignored_recurrence_columns,
        consistency_audit,
    )

    status_counts = (
        consistency_audit["status"].value_counts().to_dict()
    )
    print("\nOncoPep Figure 4 / S13 generation completed.")
    print(f"Figure 4: {figure_outputs['Figure_4']['png']}")
    print(
        "Supplementary Figure S13: "
        f"{figure_outputs['Supplementary_Figure_S13']['png']}"
    )
    print(f"Source data: {dirs.source_data}")
    print(f"Manifest: {manifest_path}")
    print(f"Consistency audit: {status_counts}")
    if status_counts.get("FAIL", 0) > 0:
        raise RuntimeError(
            "A stage-count consistency check failed. "
            "Do not use the generated figure package."
        )
    if status_counts.get("REVIEW", 0) > 0:
        print(
            "WARNING: One or more plotted values differ from the current "
            "manuscript text. Review "
            "Figure_4_S13_manuscript_consistency_audit.csv "
            "and update either the manuscript or frozen source data."
        )

    return {
        "config": cfg,
        "stage_counts": stage_counts,
        "score_summary": score_summary,
        "stability_matrix": stability_matrix,
        "recurrence": recurrence,
        "consistency_audit": consistency_audit,
        "figure_outputs": figure_outputs,
        "source_outputs": source_outputs,
        "manifest": manifest_path,
    }


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OncoPep Figure 4 and Supplementary Figure S13 — final figure-generation script.

Scientific scope
----------------
Figure 4:
    Multi-objective prioritization of generated OncoPep candidates.
    A. Prioritization-stage reduction
    B. Novelty, descriptor-plausibility, and diversity support
    C. Composite-score enrichment

Supplementary Figure S13:
    Robustness of OncoPep prioritization schemes.
    A. Pairwise Jaccard overlap among four fixed ranking schemes
    B. Candidate recurrence across the four schemes

This script is intentionally visualization-only. It does not retrain models,
regenerate peptides, recompute ranking membership, or perform bootstrap
shortlist-resampling. It validates and summarizes the frozen source tables.
"""

from __future__ import annotations

import hashlib
import json
import shutil
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, Mapping, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import LogFormatterMathtext, LogLocator, NullFormatter
import numpy as np
import pandas as pd


SCRIPT_VERSION = "v1.1_2026-07-13_fig4_horizontal_s13_locked"


# =============================================================================
# Configuration
# =============================================================================

@dataclass(frozen=True)
class Config:
    project_root: Path = Path(
        "/home/data3/Moe/nature_computational_peponco"
    )
    output_root: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "PLOS/plos_comp/step_08"
    )

    passed_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_supplementary/"
        "table_s8_12_full_ranked_passed_pool.csv"
    )
    shortlist_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_main/table_8_1_shortlist_main.csv"
    )
    final_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_main/table_8_2_final_diverse_panel.csv"
    )
    stability_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_supplementary/"
        "table_s8_5_ranking_scheme_stability.csv"
    )
    recurrence_file: Path = Path(
        "/home/data3/Moe/nature_computational_peponco/"
        "step8_v1/tables_supplementary/"
        "table_s8_6_candidate_recurrence_across_schemes.csv"
    )

    generated_count: int = 10_840
    qc_passed_count: int = 10_291
    descriptor_plausible_count: int = 10_237
    shortlist_count: int = 24
    final_count: int = 12

    scheme_order: Tuple[str, ...] = (
        "Primary",
        "Novelty-weighted",
        "Plausibility-weighted",
        "Diversity-weighted",
    )

    primary_weights: Tuple[float, float, float] = (0.35, 0.35, 0.30)
    numeric_tolerance: float = 1e-6
    manuscript_value_tolerance: float = 0.005
    expected_recurrence_total: int = 40

    figure_width_in: float = 7.35
    figure4_width_in: float = 12.0
    figure4_height_in: float = 4.45
    s13_height_in: float = 3.55
    png_dpi: int = 300
    tiff_dpi: int = 600


@dataclass(frozen=True)
class OutputDirs:
    main_figure: Path
    supplementary_figures: Path
    source_data: Path
    reports: Path
    code: Path


# =============================================================================
# Locked manuscript values used only for consistency auditing
# =============================================================================

EXPECTED_STAGE_COUNTS: Mapping[str, int] = {
    "Generated": 10_840,
    "QC-passed": 10_291,
    "Descriptor-plausible": 10_237,
    "Shortlist": 24,
    "Final panel": 12,
}

EXPECTED_MEDIANS: Mapping[Tuple[str, str], float] = {
    ("Descriptor-plausible", "Novelty"): 0.939,
    ("Shortlist", "Novelty"): 0.948,
    ("Final panel", "Novelty"): 0.947,
    ("Descriptor-plausible", "Plausibility"): 0.533,
    ("Shortlist", "Plausibility"): 0.759,
    ("Final panel", "Plausibility"): 0.769,
    ("Descriptor-plausible", "Diversity"): 0.882,
    ("Shortlist", "Diversity"): 0.908,
    ("Final panel", "Diversity"): 0.908,
    ("Descriptor-plausible", "Composite score"): 0.777,
    ("Shortlist", "Composite score"): 0.867,
    ("Final panel", "Composite score"): 0.872,
}

EXPECTED_PRIMARY_OVERLAPS: Mapping[str, float] = {
    "Novelty-weighted": 0.655,
    "Plausibility-weighted": 0.600,
    "Diversity-weighted": 0.500,
}

EXPECTED_RECURRENCE: Mapping[int, int] = {
    1: 14,
    2: 7,
    3: 8,
    4: 11,
}


# =============================================================================
# Visual style
# =============================================================================

COLORS = {
    "generated": "#BDBDBD",
    "qc": "#E69F00",
    "plausible": "#277DA1",
    "shortlist": "#B5651D",
    "final": "#D55E00",
    "dark": "#2D2D2D",
    "grid": "#E6E6E6",
    "spine": "#B8B8B8",
    "white": "#FFFFFF",
    "missing": "#EEEEEE",
}

STAGE_COLORS = {
    "Generated": COLORS["generated"],
    "QC-passed": COLORS["qc"],
    "Descriptor-plausible": COLORS["plausible"],
    "Shortlist": COLORS["shortlist"],
    "Final panel": COLORS["final"],
}

SUMMARY_STAGE_COLORS = {
    "Descriptor-plausible": COLORS["plausible"],
    "Shortlist": COLORS["shortlist"],
    "Final panel": COLORS["final"],
}

SCORE_CMAP = LinearSegmentedColormap.from_list(
    "oncopep_scores",
    ["#F6FBFD", "#B7DCE8", "#5BA8C1", "#1F6D8A"],
)

OVERLAP_CMAP = LinearSegmentedColormap.from_list(
    "oncopep_overlap",
    ["#F4FAF6", "#B8DCC2", "#5FAFB7", "#355C6E"],
)


# =============================================================================
# General helpers
# =============================================================================

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_output_dirs(cfg: Config) -> OutputDirs:
    out = OutputDirs(
        main_figure=cfg.output_root / "main_figure",
        supplementary_figures=cfg.output_root / "supplementary_figures",
        source_data=cfg.output_root / "source_data",
        reports=cfg.output_root / "reports",
        code=cfg.output_root / "code",
    )
    for path in asdict(out).values():
        Path(path).mkdir(parents=True, exist_ok=True)
    return out


def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")
    if not path.is_file():
        raise ValueError(f"{label} is not a file: {path}")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1_048_576), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv(path: Path, label: str) -> pd.DataFrame:
    require_file(path, label)
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as exc:
        raise RuntimeError(f"Could not read {label}: {path}") from exc
    if df.empty:
        raise ValueError(f"{label} is empty: {path}")
    df.columns = [str(col).strip() for col in df.columns]
    return df


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def check_score_range(series: pd.Series, label: str, tolerance: float) -> None:
    values = pd.to_numeric(series, errors="coerce")
    if values.isna().any():
        n_bad = int(values.isna().sum())
        raise ValueError(f"{label} contains {n_bad} missing/non-numeric values.")
    if ((values < -tolerance) | (values > 1 + tolerance)).any():
        lo = float(values.min())
        hi = float(values.max())
        raise ValueError(
            f"{label} must lie in [0, 1]; observed range was [{lo}, {hi}]."
        )


def resolve_numeric_alias(
    df: pd.DataFrame,
    canonical: str,
    aliases: Sequence[str],
    table_label: str,
    tolerance: float,
) -> pd.Series:
    present = [column for column in aliases if column in df.columns]
    if not present:
        raise ValueError(
            f"{table_label} is missing `{canonical}`. Accepted columns: {list(aliases)}"
        )

    resolved = pd.to_numeric(df[present[0]], errors="coerce")
    if resolved.isna().any():
        raise ValueError(
            f"{table_label}:{present[0]} contains missing/non-numeric values."
        )

    for other in present[1:]:
        other_values = pd.to_numeric(df[other], errors="coerce")
        if other_values.isna().any():
            raise ValueError(
                f"{table_label}:{other} contains missing/non-numeric values."
            )
        if not np.allclose(
            resolved.to_numpy(float),
            other_values.to_numpy(float),
            rtol=0.0,
            atol=tolerance,
        ):
            raise ValueError(
                f"{table_label} contains conflicting aliases for `{canonical}`: "
                f"{present[0]} and {other}."
            )
    return resolved.astype(float)


def resolve_sequence_column(df: pd.DataFrame, table_label: str) -> str:
    candidates = ["sequence", "peptide_sequence", "peptide", "seq"]
    found = [column for column in candidates if column in df.columns]
    if not found:
        raise ValueError(
            f"{table_label} must contain a sequence column. Tried: {candidates}"
        )
    return found[0]


def standardize_candidate_table(
    df: pd.DataFrame,
    table_label: str,
    cfg: Config,
) -> pd.DataFrame:
    out = df.copy()

    aliases = {
        "novelty_score": ["novelty_score"],
        "plausibility_score": [
            "descriptor_plausibility_score",
            "plausibility_score",
        ],
        "diversity_score": ["diversity_score"],
        "final_score": [
            "final_score",
            "composite_score",
            "prioritization_score",
        ],
    }

    for canonical, candidates in aliases.items():
        out[canonical] = resolve_numeric_alias(
            out,
            canonical,
            candidates,
            table_label,
            cfg.numeric_tolerance,
        )
        check_score_range(
            out[canonical],
            f"{table_label}:{canonical}",
            cfg.numeric_tolerance,
        )

    sequence_col = resolve_sequence_column(out, table_label)
    out["sequence"] = (
        out[sequence_col]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    if (out["sequence"] == "").any():
        raise ValueError(f"{table_label} contains empty sequences.")
    if out["sequence"].duplicated().any():
        n_dup = int(out["sequence"].duplicated().sum())
        raise ValueError(f"{table_label} contains {n_dup} duplicate sequences.")

    w_novelty, w_plausibility, w_diversity = cfg.primary_weights
    reconstructed = (
        w_novelty * out["novelty_score"]
        + w_plausibility * out["plausibility_score"]
        + w_diversity * out["diversity_score"]
    )
    if not np.allclose(
        out["final_score"].to_numpy(float),
        reconstructed.to_numpy(float),
        rtol=0.0,
        atol=cfg.numeric_tolerance,
    ):
        max_diff = float(
            np.max(
                np.abs(
                    out["final_score"].to_numpy(float)
                    - reconstructed.to_numpy(float)
                )
            )
        )
        raise ValueError(
            f"{table_label}: final_score does not match the locked "
            f"0.35/0.35/0.30 formula; maximum absolute difference={max_diff:.8g}."
        )

    return out


def validate_stage_membership(
    passed: pd.DataFrame,
    shortlist: pd.DataFrame,
    final_panel: pd.DataFrame,
    cfg: Config,
) -> None:
    expected = {
        "descriptor-plausible table": cfg.descriptor_plausible_count,
        "shortlist table": cfg.shortlist_count,
        "final-panel table": cfg.final_count,
    }
    observed = {
        "descriptor-plausible table": len(passed),
        "shortlist table": len(shortlist),
        "final-panel table": len(final_panel),
    }
    for label, expected_n in expected.items():
        if observed[label] != expected_n:
            raise ValueError(
                f"{label} contains {observed[label]:,} rows; "
                f"expected {expected_n:,}."
            )

    passed_set = set(passed["sequence"])
    shortlist_set = set(shortlist["sequence"])
    final_set = set(final_panel["sequence"])

    if not shortlist_set.issubset(passed_set):
        missing = sorted(shortlist_set - passed_set)
        raise ValueError(
            f"{len(missing)} shortlist sequences are absent from the "
            f"descriptor-plausible table. First examples: {missing[:5]}"
        )
    if not final_set.issubset(shortlist_set):
        missing = sorted(final_set - shortlist_set)
        raise ValueError(
            f"{len(missing)} final-panel sequences are absent from the shortlist. "
            f"First examples: {missing[:5]}"
        )


def median_iqr(values: Iterable[float]) -> Tuple[float, float, float, int]:
    array = np.asarray(list(values), dtype=float)
    array = array[np.isfinite(array)]
    if len(array) == 0:
        raise ValueError("Cannot summarize an empty numeric array.")
    q1, median, q3 = np.percentile(array, [25, 50, 75])
    return float(median), float(q1), float(q3), int(len(array))


def set_publication_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 8.0,
            "axes.titlesize": 9.0,
            "axes.labelsize": 8.2,
            "xtick.labelsize": 7.4,
            "ytick.labelsize": 7.4,
            "legend.fontsize": 7.4,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
            "axes.edgecolor": COLORS["spine"],
            "axes.labelcolor": COLORS["dark"],
            "xtick.color": COLORS["dark"],
            "ytick.color": COLORS["dark"],
            "text.color": COLORS["dark"],
            "axes.titlecolor": COLORS["dark"],
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def style_axis(ax: plt.Axes, grid_axis: str = "y") -> None:
    ax.set_axisbelow(True)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=COLORS["grid"], linewidth=0.65)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.65)
    elif grid_axis == "both":
        ax.grid(True, color=COLORS["grid"], linewidth=0.65)

    ax.spines["left"].set_color(COLORS["spine"])
    ax.spines["bottom"].set_color(COLORS["spine"])
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(width=0.75, length=3.3)


def add_panel_label(
    ax: plt.Axes,
    label: str,
    x: float = -0.11,
    y: float = 1.04,
) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=12.5,
        fontweight="bold",
        ha="left",
        va="bottom",
        clip_on=False,
    )


def save_figure(
    fig: plt.Figure,
    base_path: Path,
    cfg: Config,
) -> Dict[str, str]:
    base_path.parent.mkdir(parents=True, exist_ok=True)
    outputs = {
        "png": base_path.with_suffix(".png"),
        "pdf": base_path.with_suffix(".pdf"),
        "tiff": base_path.with_suffix(".tiff"),
    }

    fig.savefig(
        outputs["png"],
        dpi=cfg.png_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        outputs["pdf"],
        bbox_inches="tight",
        facecolor="white",
        metadata={
            "Title": base_path.stem,
            "Creator": f"OncoPep {SCRIPT_VERSION}",
        },
    )
    fig.savefig(
        outputs["tiff"],
        dpi=cfg.tiff_dpi,
        bbox_inches="tight",
        facecolor="white",
        pil_kwargs={"compression": "tiff_lzw"},
    )
    plt.close(fig)
    return {key: str(value) for key, value in outputs.items()}


# =============================================================================
# Figure 4 source-data construction
# =============================================================================

def build_stage_counts(
    passed: pd.DataFrame,
    shortlist: pd.DataFrame,
    final_panel: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    rows = [
        {
            "stage": "Generated",
            "count": cfg.generated_count,
            "count_source": "locked manuscript value",
        },
        {
            "stage": "QC-passed",
            "count": cfg.qc_passed_count,
            "count_source": "locked manuscript value",
        },
        {
            "stage": "Descriptor-plausible",
            "count": len(passed),
            "count_source": "input table row count",
        },
        {
            "stage": "Shortlist",
            "count": len(shortlist),
            "count_source": "input table row count",
        },
        {
            "stage": "Final panel",
            "count": len(final_panel),
            "count_source": "input table row count",
        },
    ]
    table = pd.DataFrame(rows)
    table["previous_stage_count"] = table["count"].shift(1)
    table["retention_from_previous"] = (
        table["count"] / table["previous_stage_count"]
    )
    table.loc[0, "retention_from_previous"] = 1.0
    table["percent_of_generated"] = (
        table["count"] / cfg.generated_count * 100.0
    )
    return table


def build_score_summary(
    passed: pd.DataFrame,
    shortlist: pd.DataFrame,
    final_panel: pd.DataFrame,
) -> pd.DataFrame:
    stages = [
        ("Descriptor-plausible", passed),
        ("Shortlist", shortlist),
        ("Final panel", final_panel),
    ]
    metrics = [
        ("novelty_score", "Novelty"),
        ("plausibility_score", "Plausibility"),
        ("diversity_score", "Diversity"),
        ("final_score", "Composite score"),
    ]
    rows = []
    for stage_name, table in stages:
        for column, metric_name in metrics:
            median, q1, q3, n = median_iqr(table[column])
            rows.append(
                {
                    "stage": stage_name,
                    "metric": metric_name,
                    "metric_column": column,
                    "n": n,
                    "median": median,
                    "q1": q1,
                    "q3": q3,
                    "error_bar": "interquartile range",
                }
            )
    return pd.DataFrame(rows)


# =============================================================================
# Supplementary Figure S13 source-data construction
# =============================================================================

def normalize_scheme_label(value: object) -> str:
    raw = str(value).strip().lower()
    key = (
        raw.replace("_", "-")
        .replace(" ", "-")
        .replace("heavy", "weighted")
    )
    while "--" in key:
        key = key.replace("--", "-")

    mapping = {
        "primary": "Primary",
        "primary-weighted": "Primary",
        "novelty-weighted": "Novelty-weighted",
        "novelty": "Novelty-weighted",
        "plausibility-weighted": "Plausibility-weighted",
        "plausibility": "Plausibility-weighted",
        "descriptor-plausibility-weighted": "Plausibility-weighted",
        "diversity-weighted": "Diversity-weighted",
        "diversity": "Diversity-weighted",
    }
    if key not in mapping:
        raise ValueError(
            f"Unrecognized ranking-scheme label `{value}`. "
            "Expected Primary, Novelty-weighted, Plausibility-weighted, "
            "or Diversity-weighted."
        )
    return mapping[key]


def resolve_required_column(
    df: pd.DataFrame,
    canonical: str,
    aliases: Sequence[str],
    table_label: str,
) -> str:
    present = [column for column in aliases if column in df.columns]
    if not present:
        raise ValueError(
            f"{table_label} is missing `{canonical}`. Tried: {list(aliases)}"
        )
    return present[0]


def build_stability_sources(
    stability_df: pd.DataFrame,
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    scheme_a_col = resolve_required_column(
        stability_df,
        "scheme_a",
        ["scheme_a", "scheme1", "row_scheme"],
        "stability table",
    )
    scheme_b_col = resolve_required_column(
        stability_df,
        "scheme_b",
        ["scheme_b", "scheme2", "col_scheme"],
        "stability table",
    )
    overlap_col = resolve_required_column(
        stability_df,
        "jaccard_overlap",
        ["jaccard_overlap", "jaccard", "overlap"],
        "stability table",
    )

    raw = pd.DataFrame(
        {
            "scheme_a": stability_df[scheme_a_col].map(normalize_scheme_label),
            "scheme_b": stability_df[scheme_b_col].map(normalize_scheme_label),
            "jaccard_overlap": pd.to_numeric(
                stability_df[overlap_col],
                errors="coerce",
            ),
        }
    )
    if raw["jaccard_overlap"].isna().any():
        raise ValueError("Stability table contains missing/non-numeric overlaps.")
    if (
        (raw["jaccard_overlap"] < -cfg.numeric_tolerance)
        | (raw["jaccard_overlap"] > 1 + cfg.numeric_tolerance)
    ).any():
        raise ValueError("Jaccard overlaps must lie in [0, 1].")

    pair_values: Dict[Tuple[str, str], list[float]] = {}
    for row in raw.itertuples(index=False):
        if row.scheme_a == row.scheme_b:
            continue
        ordered_pair = tuple(
            sorted(
                [row.scheme_a, row.scheme_b],
                key=cfg.scheme_order.index,
            )
        )
        pair_values.setdefault(ordered_pair, []).append(
            float(row.jaccard_overlap)
        )

    expected_pairs = {
        (cfg.scheme_order[i], cfg.scheme_order[j])
        for i in range(len(cfg.scheme_order))
        for j in range(i + 1, len(cfg.scheme_order))
    }
    missing_pairs = sorted(expected_pairs - set(pair_values))
    extra_pairs = sorted(set(pair_values) - expected_pairs)
    if missing_pairs:
        raise ValueError(
            f"Stability table is missing pairwise overlaps: {missing_pairs}"
        )
    if extra_pairs:
        raise ValueError(
            f"Stability table contains unexpected scheme pairs: {extra_pairs}"
        )

    pair_rows = []
    matrix = pd.DataFrame(
        np.eye(len(cfg.scheme_order), dtype=float),
        index=cfg.scheme_order,
        columns=cfg.scheme_order,
    )
    for pair in sorted(expected_pairs, key=lambda x: (
        cfg.scheme_order.index(x[0]),
        cfg.scheme_order.index(x[1]),
    )):
        values = np.asarray(pair_values[pair], dtype=float)
        if np.ptp(values) > cfg.numeric_tolerance:
            raise ValueError(
                f"Conflicting overlap values were supplied for {pair}: "
                f"{values.tolist()}"
            )
        value = float(values.mean())
        matrix.loc[pair[0], pair[1]] = value
        matrix.loc[pair[1], pair[0]] = value
        pair_rows.append(
            {
                "scheme_a": pair[0],
                "scheme_b": pair[1],
                "jaccard_overlap": value,
            }
        )

    long_rows = []
    for scheme_a in cfg.scheme_order:
        for scheme_b in cfg.scheme_order:
            long_rows.append(
                {
                    "scheme_a": scheme_a,
                    "scheme_b": scheme_b,
                    "jaccard_overlap": float(
                        matrix.loc[scheme_a, scheme_b]
                    ),
                    "is_diagonal": scheme_a == scheme_b,
                }
            )

    return matrix, pd.DataFrame(long_rows), pd.DataFrame(pair_rows)


def build_recurrence_source(
    recurrence_df: pd.DataFrame,
    cfg: Config,
) -> Tuple[pd.DataFrame, list[str]]:
    recurrence_col = resolve_required_column(
        recurrence_df,
        "scheme_recurrence_n",
        [
            "scheme_recurrence_n",
            "scheme_recovery_count",
            "num_schemes",
            "n_schemes",
        ],
        "recurrence table",
    )

    count_aliases = [
        column
        for column in ["candidate_count", "count", "n_candidates"]
        if column in recurrence_df.columns
    ]

    if len(recurrence_df) <= len(cfg.scheme_order) and count_aliases:
        count_col = count_aliases[0]
        out = pd.DataFrame(
            {
                "scheme_recurrence_n": pd.to_numeric(
                    recurrence_df[recurrence_col],
                    errors="coerce",
                ),
                "candidate_count": pd.to_numeric(
                    recurrence_df[count_col],
                    errors="coerce",
                ),
            }
        )
        if out.isna().any().any():
            raise ValueError(
                "Aggregated recurrence table contains missing/non-numeric values."
            )
        out = (
            out.groupby("scheme_recurrence_n", as_index=False)
            ["candidate_count"]
            .sum()
        )
    else:
        values = pd.to_numeric(
            recurrence_df[recurrence_col],
            errors="coerce",
        )
        if values.isna().any():
            raise ValueError(
                "Candidate-level recurrence table contains "
                "missing/non-numeric recurrence values."
            )
        if not np.allclose(values, np.round(values)):
            raise ValueError("Recurrence values must be integers.")
        counts = values.astype(int).value_counts().sort_index()
        out = pd.DataFrame(
            {
                "scheme_recurrence_n": counts.index,
                "candidate_count": counts.values,
            }
        )

    out["scheme_recurrence_n"] = (
        out["scheme_recurrence_n"].astype(int)
    )
    out["candidate_count"] = out["candidate_count"].astype(int)

    valid_range = range(1, len(cfg.scheme_order) + 1)
    if not out["scheme_recurrence_n"].isin(valid_range).all():
        bad = sorted(
            set(out["scheme_recurrence_n"]) - set(valid_range)
        )
        raise ValueError(
            f"Recurrence values outside 1-{len(cfg.scheme_order)}: {bad}"
        )

    out = (
        out.set_index("scheme_recurrence_n")
        .reindex(valid_range, fill_value=0)
        .rename_axis("scheme_recurrence_n")
        .reset_index()
    )
    total = int(out["candidate_count"].sum())
    if total != cfg.expected_recurrence_total:
        raise ValueError(
            f"Recurrence denominator is {total}; expected "
            f"{cfg.expected_recurrence_total} distinct candidates."
        )

    out["denominator_n"] = total
    out["fraction"] = out["candidate_count"] / total
    out["percent"] = out["fraction"] * 100.0

    ignored_columns = [
        column
        for column in recurrence_df.columns
        if "bootstrap" in column.lower()
    ]
    return out, ignored_columns


# =============================================================================
# Manuscript consistency audit
# =============================================================================

def build_consistency_audit(
    stage_counts: pd.DataFrame,
    score_summary: pd.DataFrame,
    stability_matrix: pd.DataFrame,
    recurrence: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    rows = []

    for stage, expected in EXPECTED_STAGE_COUNTS.items():
        observed = int(
            stage_counts.loc[
                stage_counts["stage"] == stage,
                "count",
            ].iloc[0]
        )
        rows.append(
            {
                "item": f"Stage count: {stage}",
                "expected": expected,
                "observed": observed,
                "absolute_difference": abs(observed - expected),
                "status": "PASS" if observed == expected else "FAIL",
            }
        )

    for (stage, metric), expected in EXPECTED_MEDIANS.items():
        observed = float(
            score_summary.loc[
                (score_summary["stage"] == stage)
                & (score_summary["metric"] == metric),
                "median",
            ].iloc[0]
        )
        diff = abs(observed - expected)
        rows.append(
            {
                "item": f"Median {metric}: {stage}",
                "expected": expected,
                "observed": observed,
                "absolute_difference": diff,
                "status": (
                    "PASS"
                    if diff <= cfg.manuscript_value_tolerance
                    else "REVIEW"
                ),
            }
        )

    for scheme, expected in EXPECTED_PRIMARY_OVERLAPS.items():
        observed = float(stability_matrix.loc["Primary", scheme])
        diff = abs(observed - expected)
        rows.append(
            {
                "item": f"Primary overlap: {scheme}",
                "expected": expected,
                "observed": observed,
                "absolute_difference": diff,
                "status": (
                    "PASS"
                    if diff <= cfg.manuscript_value_tolerance
                    else "REVIEW"
                ),
            }
        )

    recurrence_lookup = recurrence.set_index(
        "scheme_recurrence_n"
    )["candidate_count"]
    for n_schemes, expected in EXPECTED_RECURRENCE.items():
        observed = int(recurrence_lookup.loc[n_schemes])
        rows.append(
            {
                "item": f"Candidate recurrence: {n_schemes} scheme(s)",
                "expected": expected,
                "observed": observed,
                "absolute_difference": abs(observed - expected),
                "status": "PASS" if observed == expected else "REVIEW",
            }
        )

    return pd.DataFrame(rows)


# =============================================================================
# Figure 4 plotting
# =============================================================================

def plot_figure4(
    stage_counts: pd.DataFrame,
    score_summary: pd.DataFrame,
    output_base: Path,
    cfg: Config,
) -> Dict[str, str]:
    """Plot the locked three-panel horizontal Figure 4 design.

    Panel A summarizes prioritization-stage reduction on a logarithmic count
    axis. Panel B shows median novelty, plausibility, and diversity scores with
    IQR error bars for the descriptor-plausible pool, shortlist, and final
    panel. Panel C shows median composite score enrichment with IQR error bars.
    """
    set_publication_style()

    stage_order = [
        "Generated",
        "QC-passed",
        "Descriptor-plausible",
        "Shortlist",
        "Final panel",
    ]
    summary_stages = [
        "Descriptor-plausible",
        "Shortlist",
        "Final panel",
    ]
    component_metrics = [
        "Novelty",
        "Plausibility",
        "Diversity",
    ]

    # Figure-4-specific palette. S13 remains unchanged.
    fig4_stage_colors = {
        "Generated": "#D0D0D0",
        "QC-passed": "#E69F00",
        "Descriptor-plausible": "#2A98B8",
        "Shortlist": "#BC8353",
        "Final panel": "#DD6B59",
    }

    fig = plt.figure(
        figsize=(cfg.figure4_width_in, cfg.figure4_height_in),
        constrained_layout=False,
    )
    grid = gridspec.GridSpec(
        1,
        3,
        figure=fig,
        width_ratios=[1.08, 1.15, 0.86],
        wspace=0.46,
    )

    # ------------------------------------------------------------------
    # Panel A: prioritization-stage reduction
    # ------------------------------------------------------------------
    ax_a = fig.add_subplot(grid[0, 0])
    add_panel_label(ax_a, "A", x=-0.17, y=1.03)

    counts = (
        stage_counts.set_index("stage")
        .reindex(stage_order)
        .reset_index()
    )
    if counts["count"].isna().any():
        raise ValueError(
            "Figure 4A is missing one or more locked prioritization stages."
        )

    y_positions = np.arange(len(counts))
    bars = ax_a.barh(
        y_positions,
        counts["count"],
        height=0.48,
        color=[fig4_stage_colors[stage] for stage in counts["stage"]],
        edgecolor="none",
        zorder=3,
    )
    ax_a.set_yticks(y_positions)
    ax_a.set_yticklabels(counts["stage"])
    ax_a.invert_yaxis()
    ax_a.set_xscale("log")
    ax_a.set_xlim(8, max(18_000, float(counts["count"].max()) * 1.55))
    ax_a.xaxis.set_major_locator(
        LogLocator(base=10, subs=(1.0,), numticks=5)
    )
    ax_a.xaxis.set_major_formatter(LogFormatterMathtext(base=10))
    ax_a.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )
    ax_a.xaxis.set_minor_formatter(NullFormatter())
    ax_a.set_xlabel("Number of peptides (log scale)")
    ax_a.set_title("Prioritization-stage reduction", pad=8)
    style_axis(ax_a, "x")

    for bar, row in zip(bars, counts.itertuples(index=False)):
        percent = float(row.percent_of_generated)
        percent_text = f"{percent:.1f}%"
        is_final_stage = row.stage in {"Shortlist", "Final panel"}
        ax_a.text(
            float(row.count) * 1.08,
            bar.get_y() + bar.get_height() / 2,
            f"{int(row.count):,} ({percent_text})",
            va="center",
            ha="left",
            fontsize=7.5,
            fontweight="bold" if is_final_stage else "normal",
            color=COLORS["dark"],
        )

    # ------------------------------------------------------------------
    # Panel B: multi-objective support scores
    # ------------------------------------------------------------------
    ax_b = fig.add_subplot(grid[0, 1])
    add_panel_label(ax_b, "B", x=-0.14, y=1.03)

    component_data = score_summary[
        score_summary["metric"].isin(component_metrics)
    ].copy()

    missing_pairs = []
    for stage in summary_stages:
        for metric in component_metrics:
            present = (
                (component_data["stage"] == stage)
                & (component_data["metric"] == metric)
            ).any()
            if not present:
                missing_pairs.append((stage, metric))
    if missing_pairs:
        raise ValueError(
            "Figure 4B is missing required stage-metric summaries: "
            f"{missing_pairs}"
        )

    x_base = np.arange(len(component_metrics), dtype=float)
    bar_width = 0.18
    offsets = np.array([-bar_width, 0.0, bar_width])

    for stage_index, stage in enumerate(summary_stages):
        stage_rows = (
            component_data[component_data["stage"] == stage]
            .set_index("metric")
            .reindex(component_metrics)
        )
        medians = stage_rows["median"].to_numpy(float)
        q1 = stage_rows["q1"].to_numpy(float)
        q3 = stage_rows["q3"].to_numpy(float)
        xpos = x_base + offsets[stage_index]

        ax_b.bar(
            xpos,
            medians,
            width=bar_width * 0.82,
            color=fig4_stage_colors[stage],
            edgecolor="none",
            label=stage,
            zorder=3,
        )
        ax_b.errorbar(
            xpos,
            medians,
            yerr=np.vstack([
                np.maximum(medians - q1, 0),
                np.maximum(q3 - medians, 0),
            ]),
            fmt="none",
            ecolor=COLORS["dark"],
            elinewidth=0.85,
            capsize=2.3,
            capthick=0.85,
            zorder=4,
        )

    ax_b.set_xticks(x_base)
    ax_b.set_xticklabels(component_metrics)
    ax_b.set_ylim(0, 1.04)
    ax_b.set_ylabel("Median score (IQR)")
    ax_b.set_title("Multi-objective support scores", pad=8)
    style_axis(ax_b, "y")
    ax_b.legend(
        frameon=True,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        borderpad=0.35,
        handlelength=1.1,
        handletextpad=0.45,
        columnspacing=0.95,
        fontsize=7.0,
    )

    # ------------------------------------------------------------------
    # Panel C: composite-score enrichment
    # ------------------------------------------------------------------
    ax_c = fig.add_subplot(grid[0, 2])
    add_panel_label(ax_c, "C", x=-0.18, y=1.03)

    composite = (
        score_summary[score_summary["metric"] == "Composite score"]
        .set_index("stage")
        .reindex(summary_stages)
        .reset_index()
    )
    if composite[["median", "q1", "q3", "n"]].isna().any().any():
        raise ValueError(
            "Figure 4C composite-score summary contains missing values."
        )

    x_positions = np.arange(len(summary_stages))
    medians = composite["median"].to_numpy(float)
    q1 = composite["q1"].to_numpy(float)
    q3 = composite["q3"].to_numpy(float)

    ax_c.bar(
        x_positions,
        medians,
        width=0.45,
        color=[fig4_stage_colors[stage] for stage in summary_stages],
        edgecolor="none",
        zorder=3,
    )
    ax_c.errorbar(
        x_positions,
        medians,
        yerr=np.vstack([
            np.maximum(medians - q1, 0),
            np.maximum(q3 - medians, 0),
        ]),
        fmt="none",
        ecolor=COLORS["dark"],
        elinewidth=0.85,
        capsize=2.3,
        capthick=0.85,
        zorder=4,
    )
    ax_c.set_xticks(x_positions)
    ax_c.set_xticklabels(
        [
            f"Descriptor-\nplausible\nn={int(composite.loc[0, 'n']):,}",
            f"Shortlist\nn={int(composite.loc[1, 'n']):,}",
            f"Final\npanel\nn={int(composite.loc[2, 'n']):,}",
        ],
        linespacing=0.90,
    )
    ax_c.set_ylim(0, 1.04)
    ax_c.set_ylabel("Median composite score (IQR)")
    ax_c.set_title("Composite-score enrichment", pad=8)
    style_axis(ax_c, "y")

    fig.subplots_adjust(
        left=0.075,
        right=0.985,
        top=0.89,
        bottom=0.26,
    )
    return save_figure(fig, output_base, cfg)


# =============================================================================
# Supplementary Figure S13 plotting
# =============================================================================

def display_scheme_label(label: str) -> str:
    mapping = {
        "Primary": "Primary",
        "Novelty-weighted": "Novelty-\nweighted",
        "Plausibility-weighted": "Plausibility-\nweighted",
        "Diversity-weighted": "Diversity-\nweighted",
    }
    return mapping[label]


def plot_s13(
    stability_matrix: pd.DataFrame,
    recurrence: pd.DataFrame,
    output_base: Path,
    cfg: Config,
) -> Dict[str, str]:
    set_publication_style()

    fig = plt.figure(
        figsize=(cfg.figure_width_in, cfg.s13_height_in),
        constrained_layout=False,
    )
    grid = gridspec.GridSpec(
        1,
        2,
        figure=fig,
        width_ratios=[1.18, 0.92],
        wspace=0.42,
    )

    # Panel A: overlap matrix
    ax_a = fig.add_subplot(grid[0, 0])
    add_panel_label(ax_a, "A", x=-0.15, y=1.04)

    values = stability_matrix.loc[
        cfg.scheme_order, cfg.scheme_order
    ].to_numpy(float)
    masked = values.copy()
    np.fill_diagonal(masked, np.nan)

    image = ax_a.imshow(
        masked,
        vmin=0,
        vmax=1,
        cmap=OVERLAP_CMAP,
        aspect="equal",
    )
    ax_a.set_xticks(np.arange(len(cfg.scheme_order)))
    ax_a.set_yticks(np.arange(len(cfg.scheme_order)))
    ax_a.set_xticklabels(
        [display_scheme_label(label) for label in cfg.scheme_order]
    )
    ax_a.set_yticklabels(
        [display_scheme_label(label) for label in cfg.scheme_order]
    )
    ax_a.set_title("Ranking-scheme overlap", pad=8)
    ax_a.tick_params(length=0)

    for row in range(len(cfg.scheme_order)):
        for col in range(len(cfg.scheme_order)):
            if row == col:
                ax_a.add_patch(
                    Rectangle(
                        (col - 0.5, row - 0.5),
                        1,
                        1,
                        facecolor=COLORS["missing"],
                        edgecolor="white",
                        linewidth=1.0,
                    )
                )
                ax_a.text(
                    col,
                    row,
                    "self",
                    ha="center",
                    va="center",
                    fontsize=7.2,
                    color="#777777",
                )
            else:
                value = values[row, col]
                ax_a.text(
                    col,
                    row,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7.6,
                    color=(
                        "white"
                        if value >= 0.62
                        else COLORS["dark"]
                    ),
                )

    ax_a.set_xticks(
        np.arange(-0.5, len(cfg.scheme_order), 1),
        minor=True,
    )
    ax_a.set_yticks(
        np.arange(-0.5, len(cfg.scheme_order), 1),
        minor=True,
    )
    ax_a.grid(which="minor", color="white", linewidth=1.0)
    ax_a.tick_params(which="minor", bottom=False, left=False)
    for spine in ax_a.spines.values():
        spine.set_visible(True)
        spine.set_color(COLORS["spine"])
        spine.set_linewidth(0.75)

    colorbar = fig.colorbar(
        image,
        ax=ax_a,
        fraction=0.042,
        pad=0.032,
    )
    colorbar.set_label("Jaccard overlap", fontsize=7.5)
    colorbar.ax.tick_params(labelsize=6.8, length=2.5)
    colorbar.outline.set_edgecolor(COLORS["spine"])

    # Panel B: recurrence
    ax_b = fig.add_subplot(grid[0, 1])
    add_panel_label(ax_b, "B", x=-0.16, y=1.04)

    x = recurrence["scheme_recurrence_n"].to_numpy(int)
    counts = recurrence["candidate_count"].to_numpy(int)
    percentages = recurrence["percent"].to_numpy(float)
    total = int(recurrence["denominator_n"].iloc[0])

    bars = ax_b.bar(
        x,
        counts,
        width=0.56,
        color=COLORS["shortlist"],
        edgecolor="none",
        zorder=3,
    )
    ax_b.set_xticks(x)
    ax_b.set_xlabel("Number of schemes recovering candidate")
    ax_b.set_ylabel("Candidate count")
    ax_b.set_title("Candidate recurrence", pad=8)
    ax_b.set_ylim(0, max(counts) * 1.28)
    style_axis(ax_b, "y")

    for bar, count, percentage in zip(
        bars, counts, percentages
    ):
        ax_b.text(
            bar.get_x() + bar.get_width() / 2,
            count + max(counts) * 0.035,
            f"{count}\n({percentage:.0f}%)",
            ha="center",
            va="bottom",
            fontsize=7.4,
            linespacing=0.95,
        )

    ax_b.text(
        0.98,
        0.95,
        f"n = {total} distinct candidates",
        transform=ax_b.transAxes,
        ha="right",
        va="top",
        fontsize=7.3,
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "white",
            "edgecolor": COLORS["spine"],
            "linewidth": 0.7,
        },
    )

    fig.subplots_adjust(
        left=0.09,
        right=0.98,
        top=0.90,
        bottom=0.22,
    )
    return save_figure(fig, output_base, cfg)


# =============================================================================
# Source-data and manifest export
# =============================================================================

def combine_panel_sources(
    sources: Mapping[str, pd.DataFrame],
) -> pd.DataFrame:
    frames = []
    for panel, table in sources.items():
        copy = table.copy()
        copy.insert(0, "panel", panel)
        frames.append(copy)
    return pd.concat(frames, ignore_index=True, sort=False)


def export_source_data(
    dirs: OutputDirs,
    stage_counts: pd.DataFrame,
    score_summary: pd.DataFrame,
    stability_long: pd.DataFrame,
    stability_pairs: pd.DataFrame,
    recurrence: pd.DataFrame,
    consistency_audit: pd.DataFrame,
) -> Dict[str, str]:
    outputs: Dict[str, str] = {}

    fig4_a = stage_counts.copy()
    fig4_b = score_summary[
        score_summary["metric"].isin(
            ["Novelty", "Plausibility", "Diversity"]
        )
    ].copy()
    fig4_c = score_summary[
        score_summary["metric"] == "Composite score"
    ].copy()

    s13_a = stability_long.copy()
    s13_b = recurrence.copy()

    source_tables = {
        "Figure_4_panel_a_source_data.csv": fig4_a,
        "Figure_4_panel_b_source_data.csv": fig4_b,
        "Figure_4_panel_c_source_data.csv": fig4_c,
        "Supplementary_Figure_S13_panel_a_source_data.csv": s13_a,
        "Supplementary_Figure_S13_panel_a_unique_pairs_source_data.csv": stability_pairs,
        "Supplementary_Figure_S13_panel_b_source_data.csv": s13_b,
        "Figure_4_S13_manuscript_consistency_audit.csv": consistency_audit,
    }

    for filename, table in source_tables.items():
        path = dirs.source_data / filename
        write_csv(table, path)
        outputs[filename] = str(path)

    fig4_all = combine_panel_sources(
        {
            "A": fig4_a,
            "B": fig4_b,
            "C": fig4_c,
        }
    )
    fig4_all_path = (
        dirs.source_data / "Figure_4_source_data_all_panels.csv"
    )
    write_csv(fig4_all, fig4_all_path)
    outputs[fig4_all_path.name] = str(fig4_all_path)

    s13_all = combine_panel_sources(
        {
            "A": s13_a,
            "B": s13_b,
        }
    )
    s13_all_path = (
        dirs.source_data
        / "Supplementary_Figure_S13_source_data_all_panels.csv"
    )
    write_csv(s13_all, s13_all_path)
    outputs[s13_all_path.name] = str(s13_all_path)

    return outputs


def export_manifest(
    cfg: Config,
    dirs: OutputDirs,
    input_tables: Mapping[str, pd.DataFrame],
    figure_outputs: Mapping[str, Mapping[str, str]],
    source_outputs: Mapping[str, str],
    ignored_recurrence_columns: Sequence[str],
    consistency_audit: pd.DataFrame,
) -> Path:
    input_paths = {
        "passed_file": cfg.passed_file,
        "shortlist_file": cfg.shortlist_file,
        "final_file": cfg.final_file,
        "stability_file": cfg.stability_file,
        "recurrence_file": cfg.recurrence_file,
    }
    manifest = {
        "script_version": SCRIPT_VERSION,
        "generated_at_utc": utc_now(),
        "scientific_scope": (
            "Figure 4 prioritization summaries and Supplementary Figure S13 "
            "weighting-scheme robustness only."
        ),
        "claim_boundary": (
            "Computational prioritization only; no experimental anticancer "
            "activity, selectivity, toxicity, stability, or mechanism is inferred."
        ),
        "config": {
            **{
                key: str(value)
                if isinstance(value, Path)
                else value
                for key, value in asdict(cfg).items()
            },
        },
        "inputs": {
            name: {
                "path": str(path),
                "sha256": sha256_file(path),
                "shape": list(input_tables[name].shape),
            }
            for name, path in input_paths.items()
        },
        "ignored_recurrence_columns": list(
            ignored_recurrence_columns
        ),
        "figures": figure_outputs,
        "source_data": dict(source_outputs),
        "consistency_audit_status_counts": (
            consistency_audit["status"].value_counts().to_dict()
        ),
        "software": {
            "python_runtime": (
                __import__("sys").version
            ),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "matplotlib": matplotlib.__version__,
        },
    }

    manifest_path = (
        dirs.reports / "Figure_4_S13_generation_manifest.json"
    )
    manifest_path.write_text(
        json.dumps(manifest, indent=2, default=str),
        encoding="utf-8",
    )

    readme = f"""OncoPep Figure 4 and Supplementary Figure S13 package
=================================================================

Script version: {SCRIPT_VERSION}

Figure 4
--------
A. Prioritization-stage reduction
B. Novelty, descriptor-plausibility, and diversity support
C. Composite-score enrichment

Supplementary Figure S13
------------------------
A. Pairwise Jaccard overlap among four fixed ranking schemes
B. Candidate recurrence across the four schemes

Important
---------
This script does not retrain the model, regenerate peptides, rerank candidates,
or use bootstrap shortlist-membership frequencies. It visualizes frozen source
tables, validates the locked 0.35/0.35/0.30 composite score, requires the true
final-panel table, and exports the exact values plotted in each panel.
"""
    (
        dirs.reports / "README_Figure_4_S13.txt"
    ).write_text(readme, encoding="utf-8")

    requirements = "\n".join(
        [
            f"numpy=={np.__version__}",
            f"pandas=={pd.__version__}",
            f"matplotlib=={matplotlib.__version__}",
            "",
        ]
    )
    (
        dirs.reports / "requirements_Figure_4_S13.txt"
    ).write_text(requirements, encoding="utf-8")

    try:
        source_file = Path(__file__).resolve()
        if source_file.exists():
            shutil.copy2(
                source_file,
                dirs.code / "OncoPep_Figure4_S13_final.py",
            )
    except NameError:
        # Expected when executed directly from a notebook cell.
        pass

    return manifest_path


# =============================================================================
# Main workflow
# =============================================================================

def main(config: Config | None = None) -> Dict[str, object]:
    cfg = config or Config()
    dirs = ensure_output_dirs(cfg)

    raw_tables = {
        "passed_file": read_csv(
            cfg.passed_file,
            "descriptor-plausible candidate table",
        ),
        "shortlist_file": read_csv(
            cfg.shortlist_file,
            "shortlist table",
        ),
        "final_file": read_csv(
            cfg.final_file,
            "final-panel table",
        ),
        "stability_file": read_csv(
            cfg.stability_file,
            "scheme-stability table",
        ),
        "recurrence_file": read_csv(
            cfg.recurrence_file,
            "scheme-recurrence table",
        ),
    }

    passed = standardize_candidate_table(
        raw_tables["passed_file"],
        "descriptor-plausible table",
        cfg,
    )
    shortlist = standardize_candidate_table(
        raw_tables["shortlist_file"],
        "shortlist table",
        cfg,
    )
    final_panel = standardize_candidate_table(
        raw_tables["final_file"],
        "final-panel table",
        cfg,
    )
    validate_stage_membership(
        passed,
        shortlist,
        final_panel,
        cfg,
    )

    stage_counts = build_stage_counts(
        passed,
        shortlist,
        final_panel,
        cfg,
    )
    score_summary = build_score_summary(
        passed,
        shortlist,
        final_panel,
    )

    stability_matrix, stability_long, stability_pairs = (
        build_stability_sources(
            raw_tables["stability_file"],
            cfg,
        )
    )
    recurrence, ignored_recurrence_columns = (
        build_recurrence_source(
            raw_tables["recurrence_file"],
            cfg,
        )
    )

    consistency_audit = build_consistency_audit(
        stage_counts,
        score_summary,
        stability_matrix,
        recurrence,
        cfg,
    )

    source_outputs = export_source_data(
        dirs,
        stage_counts,
        score_summary,
        stability_long,
        stability_pairs,
        recurrence,
        consistency_audit,
    )

    figure4_base = (
        dirs.main_figure
        / "Figure_4_prioritization_final"
    )
    s13_base = (
        dirs.supplementary_figures
        / "Supplementary_Figure_S13_prioritization_robustness_final"
    )

    figure_outputs = {
        "Figure_4": plot_figure4(
            stage_counts,
            score_summary,
            figure4_base,
            cfg,
        ),
        "Supplementary_Figure_S13": plot_s13(
            stability_matrix,
            recurrence,
            s13_base,
            cfg,
        ),
    }

    manifest_path = export_manifest(
        cfg,
        dirs,
        raw_tables,
        figure_outputs,
        source_outputs,
        ignored_recurrence_columns,
        consistency_audit,
    )

    status_counts = (
        consistency_audit["status"].value_counts().to_dict()
    )
    print("\nOncoPep Figure 4 / S13 generation completed.")
    print(f"Figure 4: {figure_outputs['Figure_4']['png']}")
    print(
        "Supplementary Figure S13: "
        f"{figure_outputs['Supplementary_Figure_S13']['png']}"
    )
    print(f"Source data: {dirs.source_data}")
    print(f"Manifest: {manifest_path}")
    print(f"Consistency audit: {status_counts}")
    if status_counts.get("FAIL", 0) > 0:
        raise RuntimeError(
            "A stage-count consistency check failed. "
            "Do not use the generated figure package."
        )
    if status_counts.get("REVIEW", 0) > 0:
        print(
            "WARNING: One or more plotted values differ from the current "
            "manuscript text. Review "
            "Figure_4_S13_manuscript_consistency_audit.csv "
            "and update either the manuscript or frozen source data."
        )

    return {
        "config": cfg,
        "stage_counts": stage_counts,
        "score_summary": score_summary,
        "stability_matrix": stability_matrix,
        "recurrence": recurrence,
        "consistency_audit": consistency_audit,
        "figure_outputs": figure_outputs,
        "source_outputs": source_outputs,
        "manifest": manifest_path,
    }


if __name__ == "__main__":
    main()